# Action Support and Policy Evaluation in the ICU-Sepsis Benchmark
## Google Colab research notebook (offline RL, exact finite-MDP evaluation)

This notebook is the **revised implementation** of the constraint-aware offline-RL study
(BC, tabular CQL / IQL / BCQ, twin-dueling Deep CQL, hard and soft admissibility-aware CQL,
direct categorical policies, and a critic-guided categorical diffusion policy) on
`Sepsis/ICU-Sepsis-v2` (`icu-sepsis==2.0.1`).

It implements **every code-level repair and every new experiment** of the audit of the original study (the paper's Appendix F) — **Part I**, Sections 0–31 — and a
**cross-environment extension, Part II** (Sections II.1–II.7), which re-runs the central question on environments whose
true dynamics for unsupported actions are known exactly. Sections 32–35 then replay, package, gate and hand off
both parts. 

### What changed relative to the previous notebook (audit ID → where it is fixed)

| Audit ID | Repair implemented here | Section |
|---|---|---|
| **C01** | Mean-equivalent admissible projection of every policy; masked BC, deterministic masked IQL (advantage-weighted scores), uniform-admissible random; exact verification that projected and original policies induce identical mean-rule transition/reward vectors; replay of the *original* Drive artifacts when available | 7, 14, 19, 21 |
| **C02** | Legacy hard AM-CQL kept only as a labelled failure-mode control; two documented repaired designs (A: full-action CQL + masked backup/extraction, B: masked CQL on admissible-logged rows with explicit normalization); TD-row ablation (retain vs exclude inadmissible logged transitions); Huber vs squared TD; conservative-weight sensitivity; separately labelled target-clipping and discounting ablations; Q-quantile / spread / gradient-norm / target / full-buffer Bellman-residual / float32-resolution diagnostics; all dependent diffusion policies recomputed | 10, 16, 23 |
| **C03** | Seeding *before* model construction; named RNG streams (init / minibatch / corruption / evaluation / dataset); run identity = dataset hash + benchmark assets hash + normalizer hash + config hash + code hash + seed + run mode stored and validated in every checkpoint; incompatible checkpoints rejected; normalizer stored inside checkpoints; unconditional dataset-rebuild override removed; resumable checkpoints (optimizer, targets, step, dual variable, RNG states); order / unrelated-draw / resume / cache-invalidation tests | 2, 12, 13 |
| **C04** | Every main and supplementary table is generated here; artifact replay (policy files + checkpoints); release directory with per-seed CSVs, policies, checkpoint metadata, environment lock, artifact inventory with sizes + SHA-256, table-to-command map, zip archive | 21, 32–34 |
| **M01** | Admissible support coverage computed only on observed ∩ admissible pairs, separate observed / admissible / inadmissible counts, per-pair frequency quantiles, corrected histogram with asserted population | 6 |
| **M02** | Released benchmark-support mask vs. support estimated from the simulated training data are separate objects; threshold inequality verified from the benchmark construction code; estimated-support experiment with false exclusions/inclusions, empty rows, fallback frequency | 3, 4, 25 |
| **M03** | Preregistration block; two-sided exact sign-flip tests; Holm over the full family; attainability report; effect sizes + intervals; sign-flip test inversion; TOST equivalence; hierarchical (dataset → training-seed) analysis; replication planning | 1, 27, 30 |
| **M04** | Episode-level train/dev split, normalizer fitted on training episodes only, tabular FQE selector validated against exact returns, locked evaluation on a fresh dataset, tuning-budget disclosure | 26 |
| **M05** | Soft AM-CQL training/extraction alignment study, surrogate vs occupancy metrics, primal/dual residuals, per-state violation distributions, mean/min critic disagreement, margin/tie analysis, tolerance and temperature sensitivity | 10, 24 |
| **M06** | Matched-capacity direct categorical policy, behaviour-prior × critic, advantage-weighted categorical imitation, critic-only, tabular-only / neural-only / mixed guidance, schedule length, initialization, masking and guidance placement, cost accounting (params, wall time, memory, policy construction, lookup latency) | 11–12, 18, 22 |
| **M07** | Independent datasets, smaller data regimes, behaviour mixtures, MDP stress tests (transitions, initial-state distribution, admissible sets) with ranking persistence, severity/support subgroup diagnostics | 27–28 |
| **M08** | Feature provenance/timing table built from the benchmark construction script (action derived from `input_4hourly` and `max_dose_vaso`, which are also clustering features), finite-tensor assertions, representation ablation | 4, 29 |
| **M09** | Strict policy validation (repair is opt-in), conditioning + residual checks, length/occupancy agreement, absorption and return-bound checks, expected violation count and first-violation probability, statewise support failure, fail-loud value iteration, analytic toy tests, termination/mean semantics tested against the pinned environment, action codec verified against the environment mapping, Monte-Carlo cross-check | 8, 14, 20 |
| **M10** | `terminated`, `truncated`, and local `cutoff` stored separately; stepping through the intended wrapper; bootstrapping across censoring boundaries; full trajectory validation; short-cutoff test | 5 |
| **m04–m07** | Pearson r vs Spearman ρ labelled; Matplotlib-version-safe boxplot; pinned benchmark + recorded runtime lock; benchmark-support terminology; execution log with execution counts, timings, versions, GPU | throughout |

**Scope statement.** Exact evaluation removes rollout sampling error *within the
fixed ICU-Sepsis MDP*. It does not establish causal treatment effects, robustness to another hospital,
or clinical safety. "Inadmissible" means *outside the released benchmark-support mask*, not medically unsafe.

### Cross-environment extension (Part II): what it asks and what is verified

Thesis: **unsupported actions are a modelling choice.** An ICU-style benchmark (hospital data → count threshold
τ → admissible mask → estimated rows for supported pairs + a completion rule for the rest) must decide what an
unsupported action does; ICU-Sepsis offers the mean rule and termination. Part II measures what that choice does to
violation costs, optima and method rankings, and — on simulators with known ground truth — whether benchmark
rankings reproduce true-system rankings.

| Research question | Environments | What is verified or measured | Section · switch |
|---|---|---|---|
| RQ1: formal semantics of unsupported actions (propositions P1–P7: projection invariance, optimum under convex imputation, termination gap, exact error decomposition, semantics invariance of admissible policies, hard-masked CQL defect, exact categorical diffusion) | ICU-Sepsis v2, exact Gumbel-max sepsis simulator, Garnet harm family, 20 random absorbing MDPs | per proposition: P1–P3 and P5 on ICU-Sepsis (P1/P2 where the released mean-rule identity holds), the draw-0 Gumbel and Garnet instances and 20 random absorbing MDPs; P4 on the Gumbel and Garnet instances and the random MDPs (no ground truth for ICU-Sepsis); P7 on ICU-Sepsis and the draw-0 benchmark instances; P6 by the Part I gradient tests (Section 13) and closed-form checks. Identities to 1e-9, bounds checked on every learned policy; by-construction facts reported, not counted | II.5 · `xenv_theory` |
| Exact constrained optima | ICU-Sepsis (mean, terminate), Gumbel benchmark and true system, Garnet true systems | CMDP-LP frontier over an expected-violation budget κ; κ = 0 and κ = ∞ endpoints equal the exact constrained / unconstrained optima to 1e-7 | II.5 · `xenv_frontiers` |
| RQ2: benchmark validity | Gumbel-max simulator (Oberst & Sontag, 2019; exact enumeration of the published code) across hospital-data size, τ and physician ε, plus an oracle construction; Garnet absorbing CMDPs with harm *h* of unsupported actions | Kendall τ-b rank fidelity, top-1 agreement and selection regret between benchmark and true values; exact error decomposition (completion semantics vs estimation / coverage); preregistered H1 and H3 | II.3, II.6–II.7 · `xenv_gumbel`, `xenv_garnet` |
| RQ3: what constraint-aware methods buy over admissible projection | all three families (ICU-Sepsis deploy target: terminate rule) | 20 tabular methods with native / masked / projected variants (BC, CQL, IQL, BCQ, VI-LCB, MBS-QI, SPIBB, Lagrangian, CMDP-LP, SfBC, IDQL, FISOR-style, exact categorical diffusion); IQM of normalised scores with stratified bootstrap CIs, performance profiles, probability of improvement, preregistered H2 (sign-flip, Holm, TOST), default vs FQE vs oracle hyper-parameter selection | II.4, II.6–II.7 · `xenv_icu` (+ the two above) |
| Implementation checks | ICU-Sepsis; Gumbel-max simulator | the generic evaluator equals the Part I evaluator to 1e-12; enumerated Gumbel rows match a literal port of the stochastic simulator step (Monte Carlo, chi-square); exact tabular learners agree with the Part I SGD learners | II.3, II.6 · `xenv_tabular_crosscheck` |

The Part II experimental unit is the **dataset draw** (for Gumbel / Garnet a full benchmark-construction
replicate); its learners are deterministic given data. Part II keeps its own result frames and never adds rows to
the Part I registry.

### Version 3 (Section II.8): what this version adds

v3 adds a bracketing theorem with certified
comparisons, and tests outside the designs of II.3–II.6. Every function of Part I and of Sections II.1–II.7 is
unchanged, so all identity-checked caches stay valid. II.8 has its own code hashes and cache keys (`v3/`,
`v3boot/`, `v3harm/`).

| Switch (default on) | Section | What it adds | Main outputs |
|---|---|---|---|
| `xenv_v3_physicians` | II.8b, II.8f | Gumbel cells with physicians that are not near-optimal: a diabetes-blind (confounded) one and an antibiotic-withholding one | `xenv/v3_results_all.csv.gz` |
| `xenv_v3_aggregated` | II.8c, II.8f | Gumbel benchmarks on aggregated observation states with diabetes hidden (and glucose dropped in a second cell): state aggregation plus hidden confounding | `xenv/v3_results_all.csv.gz` |
| `xenv_v3_garnet_random` | II.8d, II.8f | Garnet cells whose unsupported true rows are independent random rows, so no hidden benefit holds or fails at random | `xenv/v3_results_all.csv.gz` |
| `xenv_v3_bootstrap` | II.8g | estimation-aware intervals from B = 10 bootstrap resamples of the hospital data, on 7 II.6 cells | `xenv/v3_bootstrap_*` |
| `xenv_v3_icu_harm` | II.8h | ICU-Sepsis harm curves J(λ), λ ∈ [0, 1], for the Part I main / reference policies (seed 0) and the Part II draw-0 defaults | `xenv/v3_icu_harm_*` |
| (any of the above) | II.8i | bracketing coverage (true rows and benchmark), certified comparisons, rank fidelity, bootstrap coverage | `xenv/v3_bracketing*.csv`, `xenv/v3_certified*.csv`, `xenv/v3_rank_fidelity*.csv`, `xenv/v3_bootstrap_coverage.csv`, `figures/xenv/fig_v3_*` |

The release map (Section 33), the gate (Section 34: switch-aware, closure rows REV-1 … REV-3) and the hand-off
(Section 35) cover every II.8 output. II.8 is exploratory. It is not part of `PREREG_PART2`, and its rows never
enter `XRESULTS` or the preregistered tests H1–H3.

## How to run this notebook on Google Colab

1. **Runtime → Change runtime type → GPU** (T4 or better). CPU works but is slow for the neural parts.
2. Run all cells top to bottom (**Runtime → Run all**). Google Drive is mounted so that every checkpoint
   survives a disconnect. Re-running after a disconnect **resumes**: finished runs are re-used only when their
   stored run identity matches exactly; interrupted neural runs resume from their last resumable checkpoint.
3. `RUN_MODE = "smoke"` (Section 1, or `SEPSIS_RUN_MODE`) runs a short debugging pass of every code path in
   a separate output directory; `RUN_MODE = "paper"` runs the full protocol. `COMPUTE_PROFILE` sets the scale
   of a paper run: `"full"` (default, used for the paper: 10 main seeds, 8 replication datasets × 2 training
   seeds, 50 / 30 / 30 Part II draws per ICU / Gumbel / Garnet cell) or `"lite"` (5 seeds, 5 × 2 replication,
   20 / 15 / 15 draws; its two-sided seed- and dataset-level tests cannot reject at 0.05).
4. Experiment blocks can be switched off in `EXPERIMENTS` (Section 1) or through `SEPSIS_EXPERIMENTS` (JSON;
   unknown keys raise) and run in a later session; the identity-checked cache makes this safe. In the supplied
   v4 notebook the six `xenv_v4_*` switches of Section II.9 are on by default; the final run switched II.9d,
   II.9f and II.9g off.
5. **Clean run:** use a fresh `ROOT` (`SEPSIS_ROOT`) or set `FORCE_RETRAIN = True` (a forced run writes
   `checkpoints/force_retrain_token.json`, so it survives disconnects; delete that file to start another forced
   run). **Artifact replay:** keep `ROOT` and set `ALLOW_TRAINING = False`; a missing artifact then raises, and
   Section 32 compares the replayed results with the previous session's exports. Checkpoint identities do not
   depend on the accelerator, so GPU-trained artifacts can be replayed on a CPU (near-tie greedy actions may
   differ).
6. If the original study's run directory exists at `SEPSIS_LEGACY_ROOT` (default
   `/content/drive/MyDrive/sepsis_fresh_colab`), Section 21 replays its exported policies and checkpoints
   read-only; otherwise Section 21 is skipped.

**Compute.** The final run that produced the paper's numbers (`paper` + `full`, Parts I and II including
II.9b, II.9c and II.9e) ran once, top to bottom, in one session from an empty run folder on a workstation
with one NVIDIA RTX 4070 Ti SUPER, and logged 49.1 h: 22.6 h for Part I and 26.5 h for Part II, of which
about 14 h were the II.9e sensitivity cells. The elapsed time of every cell is logged in
`logs/execution_log.jsonl` and summarised in `tables/compute_summary.csv`.

## 0. Environment setup — pinned benchmark, recorded scientific stack

In [ ]:
# ============================================================
# 0. Environment setup
# ============================================================
# The benchmark is pinned exactly. The Colab-provided scientific stack (NumPy, pandas, SciPy,
# Matplotlib, PyTorch) is NOT reinstalled, because replacing it forces a runtime restart and can
# break CUDA wheels; instead its exact versions are checked against tested minimums and written to
# an environment lock file in Section 33 (audit m05).
import importlib
import importlib.util
import os
import subprocess
import sys

# Deterministic cuBLAS workspace: must be set before CUDA is initialised (audit C03).
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

PINNED_BENCHMARK = {"icu-sepsis": "2.0.1"}
MINIMUM_VERSIONS = {
    "numpy": "1.26",
    "pandas": "2.0",
    "scipy": "1.10",
    "matplotlib": "3.7",
    "torch": "2.1",
    "gymnasium": "0.28.1",
    "tqdm": "4.66",
}


def installed_version(dist_name: str):
    try:
        from importlib.metadata import version
        return version(dist_name)
    except Exception:
        return None


to_install = []
if installed_version("icu-sepsis") != PINNED_BENCHMARK["icu-sepsis"]:
    to_install.append(f"icu-sepsis=={PINNED_BENCHMARK['icu-sepsis']}")
for dist, minimum in [("gymnasium", MINIMUM_VERSIONS["gymnasium"]), ("tqdm", MINIMUM_VERSIONS["tqdm"])]:
    if installed_version(dist) is None:
        to_install.append(f"{dist}>={minimum}")

if to_install:
    print("Installing:", to_install)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
    importlib.invalidate_caches()
else:
    print("Benchmark and helper packages already present.")


def _version_tuple(v: str):
    out = []
    for part in str(v).split("+")[0].split("."):
        digits = "".join(ch for ch in part if ch.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


version_report = {}
for dist, minimum in MINIMUM_VERSIONS.items():
    v = installed_version(dist)
    version_report[dist] = v
    if v is None:
        raise ImportError(f"Required package {dist} is not installed.")
    if _version_tuple(v) < _version_tuple(minimum):
        raise ImportError(f"{dist}=={v} is older than the tested minimum {minimum}.")
version_report["icu-sepsis"] = installed_version("icu-sepsis")
assert version_report["icu-sepsis"] == PINNED_BENCHMARK["icu-sepsis"], version_report
print("Runtime versions:", version_report)

## 1. Imports, configuration, run modes, experiment switches, preregistration, execution log

* `RUN_MODE = "paper"` is the full protocol; `"smoke"` is a fast debugging pass written to a separate
  output directory so smoke checkpoints can never be mistaken for paper checkpoints (audit C03).
* `COMPUTE_PROFILE` (environment variable `SEPSIS_COMPUTE_PROFILE`) sets the replication scale of a paper run.
  `"full"` (default): 10 main training seeds (the smallest two-sided exact sign-flip p is
  2/2¹⁰ ≈ 0.002, so the primary test and the Holm family can reject at 0.05), 8 independent replication datasets × 2
  training seeds (dataset-level two-sided minimum 2/2⁸ ≈ 0.008), and 50 / 30 / 30 Part II draws per ICU / Gumbel /
  Garnet cell. `"lite"`: 5 seeds, 5 × 2 replication, 20 / 15 / 15 draws — a cheaper run whose two-sided seed-level
  and dataset-level tests **cannot** reject (minimum p 0.0625); use it for development, not for the paper. Both
  profiles share `ROOT`: Part I run identities do not contain the seed list, so a later full run re-uses every lite
  Part I checkpoint. Smoke mode keeps its own small overrides.
* `EXPERIMENTS` switches every block on or off (overrides: `SEPSIS_EXPERIMENTS` as a JSON object; unknown keys
  raise). The `xenv_*` switches control Part II and allow splitting the work across Colab sessions.
* `PREREG` declares the primary comparison, endpoint, direction, comparison family, and practical
  effect size **before** confirmatory runs (audit M03/M04). Edit it before the confirmatory study and
  do not edit it after looking at results; its hash is stored in every results manifest.

In [ ]:
# ============================================================
# 1. Imports, configuration and run bookkeeping
# ============================================================
import copy
import dataclasses
import gc
import hashlib
import inspect
import json
import math
import platform
import random
import shutil
import time
import urllib.request
import warnings
import zipfile
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import gymnasium as gym
import icu_sepsis  # noqa: F401  (registers the Sepsis/* environments)
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as sla
import scipy.stats as stats
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import trange

try:
    from IPython.display import display
except Exception:  # pragma: no cover - plain Python fallback
    def display(obj):
        print(obj)

warnings.filterwarnings("once", message=".*does not have a deterministic implementation.*")
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# ---------------------------
# Run mode and global switches
# ---------------------------
RUN_MODE = os.environ.get("SEPSIS_RUN_MODE", "paper").strip().lower()   # "paper" | "smoke"
assert RUN_MODE in {"paper", "smoke"}, RUN_MODE
COMPUTE_PROFILE = os.environ.get("SEPSIS_COMPUTE_PROFILE", "full").strip().lower()   # "full" | "lite"
assert COMPUTE_PROFILE in {"full", "lite"}, COMPUTE_PROFILE

USE_GOOGLE_DRIVE = True        # persist all artifacts on Drive (recommended on Colab)
FORCE_REBUILD_DATA = False     # rebuild datasets even if an identity-matched file exists
FORCE_RETRAIN = False          # ignore identity-matched checkpoints and retrain
ALLOW_TRAINING = True          # False = artifact-replay mode: missing checkpoints raise instead of training
VERIFY_PROVENANCE_ONLINE = True  # fetch the benchmark construction scripts (parsed as text, never executed)
RELEASE_INCLUDE_WEIGHTS = True   # copy checkpoint weights into the release directory
RELEASE_INCLUDE_DATASET = True   # copy frozen datasets into the release (check the benchmark terms first)
SHOW_PROGRESS = True

# Every experiment block can be switched off and run in a later session (identity-checked caching).
EXPERIMENTS = {
    "legacy_artifact_replay": True,      # Phase 2: replay the original run's Drive artifacts read-only
    "reproducibility_tests": True,       # C03 closure tests
    "env_semantics_tests": True,         # M09: termination / mean semantics against the pinned env
    "monte_carlo_crosscheck": True,      # M09: small Monte-Carlo cross-check of exact evaluation
    "main_tabular": True,
    "main_neural_critics": True,
    "main_policy_nets": True,            # M06: matched-capacity direct categorical baselines
    "main_diffusion": True,
    "inference_ablations": True,         # Tables 6-7 + M06 masking/guidance/initialisation variants
    "critic_stability_ablations": True,  # C02 design/TD-row/loss/alpha/clipping ablations
    "include_discounted_ablation": False,  # gamma=0.99 changes the estimand; separately labelled if enabled
    "soft_sensitivity": True,            # M05 tolerance / temperature sensitivity
    "diffusion_schedule_ablation": True, # M06 schedule length (retrains the denoiser)
    "support_estimation": True,          # M02 estimated-support experiment
    "selection_protocol": True,          # M04 episode split + FQE selector validation
    "replication": True,                 # M03/M07 independent datasets x training seeds
    "replication_include_neural": True,  # include repaired critic + diffusion in the replication study
    "data_regimes": True,                # M07 smaller datasets + behaviour mixtures
    "mdp_stress_tests": True,            # M07 transition / initial-state / admissible-set stress tests
    "representation_ablation": True,     # M08 treatment-feature removal
    "release_package": True,             # C04 release directory + zip
    # Part II (cross-environment exact study, Sections II.1-II.7).
    "xenv_theory": True,                 # II.5 propositions P1-P7, numerically verified
    "xenv_frontiers": True,              # II.5 exact constrained (CMDP-LP) frontiers
    "xenv_icu": True,                    # II.6 ICU-Sepsis user-dataset draws
    "xenv_gumbel": True,                 # II.3/II.6 exact Gumbel-max sepsis simulator cells
    "xenv_garnet": True,                 # II.3/II.6 Garnet harm family
    "xenv_tabular_crosscheck": True,     # II.6 exact tabular learners vs Part I SGD learners (needs main_tabular)
    # Part II revision experiments (Section II.8): new cells with their own cache identities (II.6 draws unaffected).
    "xenv_v3_physicians": True,          # II.8 Gumbel physicians that are not near-optimal (confounded, withhold-abx)
    "xenv_v3_aggregated": True,          # II.8 Gumbel benchmarks on aggregated observation states (diabetes hidden)
    "xenv_v3_garnet_random": True,       # II.8 Garnet with independent random true rows for unsupported actions
    "xenv_v3_bootstrap": True,           # II.8g bootstrap-ensemble intervals on II.6 cells (needs xenv_gumbel / xenv_garnet)
    "xenv_v3_icu_harm": True,            # II.8h ICU-Sepsis harm curves J(lambda), Part I seed 0 + Part II draw 0
    # Part II second-revision experiments (Section II.9): new functions and cache keys (II.6 / II.8 draws unaffected).
    "xenv_v4_revision_robust": True,     # II.9b NHB-free certificates, pair types, kappa, II.8g all-policy coverage
    "xenv_v4_confidence": True,          # II.9c L1 confidence-set certificates on the true state space
    "xenv_v4_bootstrap": True,           # II.9d bootstrap B = 200, fixed mask and re-estimated construction
    "xenv_v4_sensitivity": True,         # II.9e timeout hazard, dead-state rule, softer physicians, abstractions
    "xenv_v4_lift": True,                # II.9f physician / uniform lifts and the range of J_true over all lifts
    "xenv_v4_clean_check": True,         # II.9g clean rerun of the Part II draws into an empty cache (ALLOW_TRAINING)
}
if os.environ.get("SEPSIS_EXPERIMENTS"):
    _exp_override = json.loads(os.environ["SEPSIS_EXPERIMENTS"])
    _exp_unknown = sorted(set(_exp_override) - set(EXPERIMENTS))
    assert not _exp_unknown, f"Unknown experiment switches in SEPSIS_EXPERIMENTS: {_exp_unknown}"
    assert all(isinstance(v, bool) for v in _exp_override.values()), "SEPSIS_EXPERIMENTS values must be JSON true/false."
    EXPERIMENTS.update(_exp_override)

# ---------------------------
# Output paths
# ---------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if os.environ.get("SEPSIS_ROOT"):
    ROOT_BASE = Path(os.environ["SEPSIS_ROOT"])
elif IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT_BASE = Path("/content/drive/MyDrive/sepsis_revised_colab")
elif IN_COLAB:
    ROOT_BASE = Path("/content/sepsis_revised_colab")
else:
    ROOT_BASE = Path.cwd() / "sepsis_revised_outputs"

# Separate directory per run mode: a smoke checkpoint can never be loaded by a paper run (C03).
ROOT = ROOT_BASE / RUN_MODE
# The ORIGINAL run directory is only ever read (legacy replay, Section 21); nothing is written there.
LEGACY_ROOT = Path(os.environ.get("SEPSIS_LEGACY_ROOT", "/content/drive/MyDrive/sepsis_fresh_colab"))

DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
POLICY_DIR = ROOT / "policies"
LOG_DIR = ROOT / "logs"
TABLE_DIR = ROOT / "tables"
FIG_DIR = ROOT / "figures"
RELEASE_DIR = ROOT / "release"
for p in [DATA_DIR, CKPT_DIR, POLICY_DIR, LOG_DIR, TABLE_DIR, FIG_DIR, RELEASE_DIR]:
    p.mkdir(parents=True, exist_ok=True)


# ---------------------------
# Configuration
# ---------------------------
@dataclass
class Config:
    # Matched training seeds (main study) and the smaller seed set used for ablations. Ten seeds make the
    # two-sided exact sign-flip test able to reject (minimum p = 2/2^10); the "lite" profile uses 0-4.
    seeds: Tuple[int, ...] = tuple(range(10))
    ablation_seeds: Tuple[int, ...] = (0, 1, 2)

    # Benchmark / frozen offline dataset.
    dataset_seed: int = 2026
    dataset_episodes: int = 10_000
    max_episode_steps: int = 100       # local cutoff (censoring); the benchmark TimeLimit is 500
    behavior: str = "clinician"
    gamma: float = 1.0                 # training discount (evaluation is always the undiscounted return)

    # Generic optimisation.
    batch_size: int = 256
    hidden_dim: int = 256
    lr: float = 3e-4
    weight_decay: float = 1e-5
    grad_clip: float = 10.0
    target_tau: float = 0.005
    log_every: int = 500
    diag_every: int = 5_000            # full-buffer Bellman residual + full Q-table diagnostics
    resume_every: int = 5_000          # resumable-checkpoint interval for neural trainers

    # Tabular baselines.
    tab_cql_steps: int = 25_000
    tab_cql_lr: float = 3e-3
    tab_cql_alpha: float = 1.0
    iql_steps: int = 25_000
    iql_lr: float = 3e-3
    iql_expectile: float = 0.7
    iql_beta: float = 3.0
    iql_max_weight: float = 100.0
    bcq_steps: int = 25_000
    bcq_lr: float = 3e-3
    bcq_threshold: float = 0.3         # discrete-BCQ relative behaviour-probability threshold

    # Deep CQL / AM-CQL.
    deep_steps: int = 30_000
    cql_alpha: float = 1.0
    cql_temperature: float = 1.0

    # Soft AM-CQL.
    soft_initial_lambda: float = 1.0
    soft_dual_lr: float = 1e-2
    soft_target_inadmissible_mass: float = 0.0
    soft_lambda_max: float = 100.0
    soft_policy_temperature: float = 1.0

    # Categorical diffusion behaviour prior.
    diffusion_steps: int = 20_000
    diffusion_lr: float = 3e-4
    diffusion_T: int = 20
    diffusion_beta_start: float = 0.02
    diffusion_beta_end: float = 0.25
    diffusion_action_embed_dim: int = 32
    diffusion_time_embed_dim: int = 32
    diffusion_guidance_scale: float = 1.5
    diffusion_deterministic_policy: bool = True
    guidance_grid: Tuple[float, ...] = (0.0, 0.5, 1.5, 3.0)
    diffusion_T_grid: Tuple[int, ...] = (5, 50)

    # Direct categorical policies (matched budget with the diffusion prior).
    policy_net_steps: int = 20_000
    policy_net_lr: float = 3e-4
    aw_beta: float = 3.0
    aw_max_weight: float = 100.0

    # Statistics.
    bootstrap_samples: int = 10_000
    significance_alpha: float = 0.05

    # Robustness / selection. The dataset is the replication unit: 8 independent datasets x 2 nested training
    # seeds make a dataset-level two-sided test attainable (minimum p = 2/2^8); 2026 is the main dataset.
    replication_dataset_seeds: Tuple[int, ...] = tuple(range(2026, 2034))
    replication_train_seeds: Tuple[int, ...] = (0, 1)
    data_size_regimes: Tuple[int, ...] = (1_000, 3_000)
    behavior_mixtures: Tuple[str, ...] = ("clinician+uniform:0.25", "clinician+uniform_admissible:0.25")
    regime_train_seeds: Tuple[int, ...] = (0, 1, 2)
    regime_dataset_draws: int = 2      # independent dataset draws per reduced size (not prefixes of the main data)
    stress_seeds: Tuple[int, ...] = (0, 1, 2)
    mc_episodes: int = 10_000
    dev_fraction: float = 0.2
    fresh_eval_dataset_seed: int = 3030


def apply_profile(cfg: Config, profile: str) -> Config:
    """Replication scale of a paper run: 'full' keeps the defaults; 'lite' halves the seed and dataset counts."""
    if profile == "lite":
        cfg.seeds = (0, 1, 2, 3, 4)
        cfg.replication_dataset_seeds = (2026, 2027, 2028, 2029, 2030)
        cfg.replication_train_seeds = (0, 1)
    elif profile != "full":
        raise ValueError(f"unknown compute profile {profile!r}")
    return cfg


def apply_run_mode(cfg: Config, mode: str) -> Config:
    if mode == "smoke":
        cfg.seeds = (0, 1)
        cfg.ablation_seeds = (0, 1)
        cfg.dataset_episodes = 400
        cfg.tab_cql_steps = cfg.iql_steps = cfg.bcq_steps = 300
        cfg.deep_steps = 300
        cfg.diffusion_steps = 300
        cfg.policy_net_steps = 300
        cfg.log_every = 50
        cfg.diag_every = 150
        cfg.resume_every = 100
        cfg.diffusion_T_grid = (5,)
        cfg.bootstrap_samples = 500
        cfg.replication_dataset_seeds = (2026, 2027)
        cfg.replication_train_seeds = (0, 1)
        cfg.data_size_regimes = (150,)
        cfg.behavior_mixtures = ("clinician+uniform:0.25",)
        cfg.regime_train_seeds = (0, 1)
        cfg.regime_dataset_draws = 1
        cfg.stress_seeds = (0,)
        cfg.mc_episodes = 400
    return cfg


# The profile is applied first; smoke mode then imposes its own small overrides.
CFG = apply_run_mode(apply_profile(Config(), COMPUTE_PROFILE), RUN_MODE)
# On the main dataset the replication study registers its runs in the "main" group (Section 27), so its training
# seeds must be main seeds; otherwise main-study pairs would become unbalanced across methods.
assert set(CFG.replication_train_seeds) <= set(CFG.seeds), (CFG.replication_train_seeds, CFG.seeds)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def configure_determinism() -> None:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


configure_determinism()

# ---------------------------
# Preregistration (audit M03 / M04) — edit BEFORE confirmatory runs, never after seeing results.
# ---------------------------
PRIMARY_DIFFUSION = "Diffusion-AM-CQL [primary critic B]"
PREREG = {
    "status": "DRAFT - lock (set status to LOCKED with a date) before the confirmatory Phase-4 run",
    "primary_question": (
        "Does critic-guided categorical diffusion improve exact return over the mean-equivalent "
        "admissible projection of tabular IQL when both use the same released benchmark-support mask?"
    ),
    "primary_treatment": PRIMARY_DIFFUSION,
    "primary_comparator": "Tabular IQL [projected]",
    "primary_endpoint": {"metric": "return", "rule": "terminate"},
    "alternative": "two-sided",
    "alpha": 0.05,
    "practical_effect": 0.005,       # absolute exact-return difference (0.5 percentage points)
    "equivalence_margin": 0.005,     # TOST margin (same scale)
    "primary_hard_critic": "hard_admrows",  # repaired design B guides the primary diffusion policy
    "secondary_comparators": [
        "Tabular IQL [masked greedy]",
        "Behavior Cloning [masked greedy]",
        "Tabular CQL [masked greedy]",
        "Masked critic ensemble [critic B + tabular CQL]",
        "Neural BC prior x critic [critic B + tabular CQL]",
        "AM-CQL (Hard, design B: admissible-row CQL)",
        "AM-CQL (Hard, design A: full-action CQL)",
        "Deep CQL [masked greedy]",
        "Diffusion-AM-CQL [legacy critic]",
    ],
    "secondary_rules": ["mean", "terminate"],
    "multiplicity": ("Holm over the distinct hypotheses of the secondary family (comparator x rule). When the treatment "
                     "and a comparator both have zero inadmissible mass, their mean- and terminate-rule returns coincide "
                     "by construction, so that comparison is counted once."),
    "experimental_unit_main": (
        f"training seed, conditional on one frozen dataset; {len(CFG.seeds)} seeds {list(CFG.seeds)} "
        f"(compute profile '{COMPUTE_PROFILE}', run mode '{RUN_MODE}'; the full paper profile uses seeds 0-9, "
        "so the smallest two-sided exact sign-flip p is 2/2^10)"),
    "experimental_unit_hierarchical": (
        f"dataset-generation seed, training seeds nested within dataset; {len(CFG.replication_dataset_seeds)} datasets "
        f"{list(CFG.replication_dataset_seeds)} x {len(CFG.replication_train_seeds)} training seeds "
        f"{list(CFG.replication_train_seeds)} (full profile: 8 x 2, smallest dataset-level two-sided p 2/2^8)"),
    "selection": (
        "Main-study configurations are fixed a priori (Table 3 settings). Inference sweeps are exploratory "
        "and never replace the declared configuration."
    ),
}

# ---------------------------
# Execution log (audit m07): execution counts, timings, success flags, section names
# ---------------------------
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


class ExecutionLog:
    def __init__(self, path: Path):
        self.path = path
        self.section = "startup"
        self._t0 = None
        self._start = None
        self.session_start = utc_now()

    def mark(self, section: str) -> None:
        self.section = section
        print(f"\n=== {section} ===")

    def pre(self, *args, **kwargs) -> None:
        self._t0 = time.perf_counter()
        self._start = utc_now()

    def post(self, result=None, *args, **kwargs) -> None:
        rec = {
            "session_start": self.session_start,
            "section": self.section,
            "execution_count": getattr(result, "execution_count", None),
            "success": bool(getattr(result, "success", True)),
            "error": (repr(getattr(result, "error_in_exec", None))
                      if getattr(result, "error_in_exec", None) is not None else None),
            "start_utc": self._start,
            "elapsed_s": (time.perf_counter() - self._t0) if self._t0 is not None else None,
        }
        try:
            with open(self.path, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(rec) + "\n")
        except Exception:
            pass


EXECLOG = ExecutionLog(LOG_DIR / "execution_log.jsonl")
try:
    _ip = get_ipython()  # noqa: F821
    for _ev in ("pre_run_cell", "post_run_cell"):
        for _cb in list(_ip.events.callbacks.get(_ev, [])):
            if getattr(_cb, "__self__", None).__class__.__name__ == "ExecutionLog":
                _ip.events.unregister(_ev, _cb)
    _ip.events.register("pre_run_cell", EXECLOG.pre)
    _ip.events.register("post_run_cell", EXECLOG.post)
except Exception:
    _ip = None

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
RUNTIME_INFO = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": GPU_NAME,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": installed_version("scipy"),
    "matplotlib": matplotlib.__version__,
    "gymnasium": gym.__version__,
    "icu_sepsis": installed_version("icu-sepsis"),
    "in_colab": IN_COLAB,
    "session_start_utc": EXECLOG.session_start,
    "run_mode": RUN_MODE,
    "compute_profile": COMPUTE_PROFILE,
}
# Replication design actually used by this session (read by Sections 30, 34 and the paper hand-off).
RUN_PROFILE = {
    "run_mode": RUN_MODE,
    "compute_profile": COMPUTE_PROFILE,
    "main_seeds": list(CFG.seeds),
    "ablation_seeds": list(CFG.ablation_seeds),
    "replication_dataset_seeds": list(CFG.replication_dataset_seeds),
    "replication_train_seeds": list(CFG.replication_train_seeds),
    "min_two_sided_signflip_p_main_seeds": 2.0 / 2 ** len(CFG.seeds),
    "min_two_sided_signflip_p_dataset_level": 2.0 / 2 ** len(CFG.replication_dataset_seeds),
    "paper_scale": bool(RUN_MODE == "paper" and COMPUTE_PROFILE == "full"),
}

print("Run mode:", RUN_MODE, "| compute profile:", COMPUTE_PROFILE,
      f"| main seeds: {len(CFG.seeds)} | replication: {len(CFG.replication_dataset_seeds)} datasets x "
      f"{len(CFG.replication_train_seeds)} training seeds")
print("Output root:", ROOT)
print("Device:", DEVICE, "| GPU:", GPU_NAME)
print("Enabled experiments:", [k for k, v in EXPERIMENTS.items() if v])


def _finite_json(o):
    if isinstance(o, dict):
        return {k: _finite_json(v) for k, v in o.items()}
    if isinstance(o, list):
        return [_finite_json(v) for v in o]
    if isinstance(o, float):
        return o if math.isfinite(o) else None   # strict JSON: NaN / +-Infinity -> null
    return o


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    clean = _finite_json(json.loads(json.dumps(obj, default=_json_default)))
    path.write_text(json.dumps(clean, indent=2, allow_nan=False), encoding="utf-8")


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    if isinstance(o, (set, frozenset)):
        return sorted(o)
    if dataclasses.is_dataclass(o):
        return asdict(o)
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    raise TypeError(f"Not JSON serialisable: {type(o)}")


save_json(asdict(CFG), TABLE_DIR / "fresh_config.json")
save_json(PREREG, TABLE_DIR / "preregistration.json")
save_json(EXPERIMENTS, TABLE_DIR / "experiment_switches.json")
save_json(RUN_PROFILE, TABLE_DIR / "run_profile.json")
save_json(RUNTIME_INFO, LOG_DIR / f"runtime_{EXECLOG.session_start.replace(':', '')}.json")

## 2. Provenance: hashing, named RNG streams, run identity, identity-checked and resumable checkpoints (audit C03 / C04)

* **Named RNG streams.** Every stochastic component draws from its own stream derived from
  `(stream name, keys…)`: model initialisation, minibatch indices, diffusion corruption, evaluation /
  Monte-Carlo, dataset generation. Seeding happens *immediately before* model construction, so
  initial weights never depend on earlier notebook execution or on run order.
* **Common random numbers.** For a given training seed, all twin-critic variants share the same
  initial weights and the same minibatch index sequence on the same dataset; differences between
  critic variants are therefore attributable to their objectives rather than to different random inputs.
  Pairing across *different* model families (e.g. tabular IQL seed 3 vs diffusion seed 3) shares only the
  minibatch stream and the dataset; it is a documented pairing convention, not identical randomness.
* **Run identity.** dataset content hash, benchmark assets hash + version, normalizer hash,
  algorithm configuration hash, code hash (source of the trainer and its helpers), training seed and
  run mode. The identity is stored in every checkpoint and validated on load; a mismatch raises
  instead of silently loading. Checkpoint file names contain the identity hash, so changing the dataset,
  objective, hyperparameters, code (functions *and* classes, hashed member by member), or run mode
  automatically invalidates the cache. The accelerator type is recorded as metadata, not as identity, so
  GPU-trained checkpoints can be replayed on a CPU runtime (resume states are device-specific).

In [ ]:
# ============================================================
# 2. Provenance utilities
# ============================================================
EXECLOG.mark("2. Provenance utilities")


def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            block = fh.read(chunk)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def canonical_json(obj) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=_json_default)


def obj_hash(obj) -> str:
    return sha256_bytes(canonical_json(obj).encode("utf-8"))


def array_dict_hash(arrays: Dict[str, np.ndarray]) -> str:
    h = hashlib.sha256()
    for k in sorted(arrays):
        a = np.ascontiguousarray(np.asarray(arrays[k]))
        h.update(k.encode())
        h.update(str(a.dtype).encode())
        h.update(str(a.shape).encode())
        h.update(a.tobytes())
    return h.hexdigest()


def array_hash(a: np.ndarray) -> str:
    return array_dict_hash({"a": a})


def _const_repr(c) -> str:
    """Process-independent representation of a code constant (set/frozenset order depends on PYTHONHASHSEED)."""
    if inspect.iscode(c):
        return _code_signature(c)
    if isinstance(c, (set, frozenset)):
        return f"{type(c).__name__}({sorted(_const_repr(x) for x in c)!r})"
    if isinstance(c, tuple):
        return "(" + ", ".join(_const_repr(x) for x in c) + ",)"
    return repr(c)


def _code_signature(code) -> str:
    """Address-free, hash-seed-independent signature of a code object (fallback when no source is available)."""
    return repr((code.co_code.hex(), code.co_names, tuple(_const_repr(c) for c in code.co_consts)))


_GENERATED_FILES = {"<string>", getattr(dataclasses, "__file__", "")}


def _is_generated(fn) -> bool:
    """Methods synthesised by @dataclass (exec'd from '<string>' or defined in dataclasses.py)."""
    base = inspect.unwrap(fn)
    return (getattr(fn, "__module__", None) == "dataclasses"
            or getattr(base.__code__, "co_filename", "") in _GENERATED_FILES
            or getattr(fn.__code__, "co_filename", "") in _GENERATED_FILES)


def _source_text(o) -> str:
    """Source text used for hashing. Classes are hashed member by member, because inside IPython/Colab
    `inspect.getsource(<class defined in a cell>)` raises OSError while functions and methods work.
    Dataclass-generated methods are skipped (the field list covers them), which keeps the hash identical
    across Python processes and versions."""
    if inspect.isclass(o):
        parts = [o.__qualname__, repr([b.__qualname__ for b in o.__mro__])]
        is_dc = dataclasses.is_dataclass(o)
        if is_dc:
            parts.append(repr([(f.name, str(f.type),
                                None if f.default is dataclasses.MISSING else repr(f.default),
                                f.default_factory is not dataclasses.MISSING) for f in dataclasses.fields(o)]))
        for k, v in sorted(vars(o).items()):
            fn = v.fget if isinstance(v, property) else getattr(v, "__func__", v)
            if inspect.isfunction(fn):
                if is_dc and _is_generated(fn):
                    continue
                try:
                    parts.append(f"{k}:" + inspect.getsource(fn))
                except (OSError, TypeError):
                    parts.append(f"{k}:" + _code_signature(fn.__code__))
            elif not k.startswith("__") and " at 0x" not in repr(v):
                parts.append(f"{k}={v!r}")
        return "\n".join(parts)
    try:
        return inspect.getsource(o)
    except (OSError, TypeError):
        code = getattr(o, "__code__", None)
        if code is None:
            raise TypeError(f"code_hash cannot hash {o!r}")
        return repr((getattr(o, "__qualname__", ""), _code_signature(code)))


def code_hash(*objs) -> str:
    """Hash of the source code of trainers, classes and helpers (audit C03 'code revision')."""
    h = hashlib.sha256()
    for o in objs:
        h.update(_source_text(o).encode("utf-8", errors="replace"))
    return h.hexdigest()


class _CodeHashProbe:
    def f(self):
        return 1


assert code_hash(_CodeHashProbe) != sha256_bytes(repr(("_CodeHashProbe", b"", ())).encode()), \
    "code_hash fell back to a name-only hash for a class."


@dataclass(frozen=True)
class _FrozenProbe:
    a: int = 0
    b: int = 0
    c: int = 0


assert "frozenset(" not in _source_text(_FrozenProbe) and "__setattr__" not in _source_text(_FrozenProbe), \
    "class hash would depend on PYTHONHASHSEED (dataclass-generated methods leaked into the hash)."


def stream_seed(*keys) -> int:
    """Deterministic 62-bit seed for a named RNG stream."""
    digest = hashlib.sha256(canonical_json(["icu-sepsis-revised-v1", *keys]).encode()).digest()
    return int.from_bytes(digest[:8], "little") & ((1 << 62) - 1)


def np_rng(*keys) -> np.random.Generator:
    return np.random.default_rng(stream_seed(*keys))


def torch_generator(seed: int, device: torch.device) -> torch.Generator:
    gen = torch.Generator(device="cuda" if device.type == "cuda" else "cpu")
    gen.manual_seed(int(seed) % (2 ** 63 - 1))
    return gen


def seed_for_init(seed: int) -> None:
    """Seed the global torch RNG immediately before module construction (initialisation stream)."""
    torch.manual_seed(int(seed) % (2 ** 63 - 1))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed) % (2 ** 63 - 1))


def init_model(builder: Callable[[], nn.Module], arch_key: str, seed: int) -> nn.Module:
    """Construct a model from the named initialisation stream, independent of prior RNG state.

    The global torch RNG state is saved and restored, so construction also does not perturb any
    later code that (incorrectly) relies on the global stream.
    """
    cpu_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        seed_for_init(stream_seed("init", arch_key, int(seed)))
        model = builder()
    finally:
        torch.random.set_rng_state(cpu_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)
    return model.to(DEVICE)


def state_dict_hash(module: nn.Module) -> str:
    h = hashlib.sha256()
    for k, v in sorted(module.state_dict().items()):
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def count_parameters(module: nn.Module, trainable_only: bool = True) -> int:
    return int(sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only)))


def peak_gpu_memory_mb() -> Optional[float]:
    if torch.cuda.is_available():
        return float(torch.cuda.max_memory_allocated() / 2 ** 20)
    return None


def reset_peak_gpu_memory() -> None:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def slug(name: str) -> str:
    keep = []
    for ch in str(name):
        if ch.isalnum() or ch in "-_.":
            keep.append(ch)
        elif ch in " /:[]()+,=":
            keep.append("_")
    s = "".join(keep)
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")[:120]


class CheckpointIdentityError(RuntimeError):
    pass


class MissingArtifactError(RuntimeError):
    pass


def load_torch_payload(path: Path, map_location="cpu"):
    # Our own checkpoints contain plain-Python metadata, so the full payload is requested explicitly.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:  # older PyTorch without weights_only
        return torch.load(path, map_location=map_location)


def atomic_torch_save(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(payload, tmp)
    try:
        os.replace(tmp, path)
    except OSError:
        shutil.move(str(tmp), str(path))


# FORCE_RETRAIN persists across Colab disconnects through a token: checkpoints written by the current
# forced run carry the token and are re-used after a reconnect; older checkpoints are ignored.
# Delete checkpoints/force_retrain_token.json to start a new forced run; set FORCE_RETRAIN=False afterwards.
FORCE_TOKEN_FILE = CKPT_DIR / "force_retrain_token.json"
if FORCE_RETRAIN and not FORCE_TOKEN_FILE.exists():
    save_json({"token": os.urandom(16).hex(), "created_utc": utc_now()}, FORCE_TOKEN_FILE)
FORCE_TOKEN = json.loads(FORCE_TOKEN_FILE.read_text())["token"] if FORCE_RETRAIN else None


class CheckpointStore:
    """Identity-addressed checkpoints with validation and resumable training state."""

    def __init__(self, root: Path, allow_training: Optional[bool] = None):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.allow_training = allow_training      # None -> follow the global ALLOW_TRAINING switch
        self.session_hashes: set = set()          # identities written in this Python session
        self.loaded_hashes: set = set()           # identities loaded from earlier sessions

    @staticmethod
    def identity_hash(identity: dict) -> str:
        return obj_hash(identity)

    def _paths(self, label: str, identity: dict):
        h = self.identity_hash(identity)
        d = self.root / slug(label)
        return d / f"final_{h[:20]}.pt", d / f"resume_{h[:20]}.pt", h

    def stale_checkpoints(self, label: str, identity: dict) -> List[str]:
        final, _, _ = self._paths(label, identity)
        if not final.parent.exists():
            return []
        return sorted(p.name for p in final.parent.glob("final_*.pt") if p.name != final.name)

    @staticmethod
    def _force_rejects(payload_token, respect_force: bool) -> bool:
        return bool(FORCE_RETRAIN and respect_force and payload_token != FORCE_TOKEN)

    def load_final(self, label: str, identity: dict, map_location="cpu", respect_force: bool = True):
        final, _, h = self._paths(label, identity)
        if not final.exists():
            return None
        sidecar = final.with_suffix(".identity.json")
        if FORCE_RETRAIN and respect_force and sidecar.exists():
            if json.loads(sidecar.read_text()).get("force_token") != FORCE_TOKEN:
                return None
        payload = load_torch_payload(final, map_location=map_location)
        if self._force_rejects(payload.get("force_token"), respect_force):
            return None
        stored = payload.get("identity")
        if payload.get("identity_hash") != h or stored is None or obj_hash(stored) != h:
            raise CheckpointIdentityError(
                f"Checkpoint {final} does not match the requested run identity; refusing to load it."
            )
        if h not in self.session_hashes:
            self.loaded_hashes.add(h)
        return payload

    def save_final(self, label: str, identity: dict, payload: dict) -> Path:
        final, resume, h = self._paths(label, identity)
        payload = dict(payload)
        payload.update(identity=identity, identity_hash=h, saved_utc=utc_now(), runtime=RUNTIME_INFO,
                       force_token=FORCE_TOKEN, device_type=DEVICE.type)
        atomic_torch_save(payload, final)
        self.session_hashes.add(h)
        save_json({"label": label, "identity_hash": h, "identity": identity, "saved_utc": payload["saved_utc"],
                   "file": final.name, "force_token": FORCE_TOKEN, "device_type": DEVICE.type, "gpu": GPU_NAME},
                  final.with_suffix(".identity.json"))
        if resume.exists():
            resume.unlink()
        return final

    def load_resume(self, label: str, identity: dict, map_location="cpu"):
        _, resume, h = self._paths(label, identity)
        if not resume.exists():
            return None
        payload = load_torch_payload(resume, map_location=map_location)
        if self._force_rejects(payload.get("force_token"), True):
            return None
        if payload.get("identity_hash") != h:
            raise CheckpointIdentityError(f"Resume state {resume} has a different identity.")
        if payload.get("device_type") != DEVICE.type:
            print(f"  ignoring resume state {resume.name}: saved on {payload.get('device_type')}, now on {DEVICE.type}")
            return None
        return payload

    def save_resume(self, label: str, identity: dict, payload: dict) -> None:
        _, resume, h = self._paths(label, identity)
        payload = dict(payload)
        payload.update(identity=identity, identity_hash=h, saved_utc=utc_now(), force_token=FORCE_TOKEN,
                       device_type=DEVICE.type)
        atomic_torch_save(payload, resume)

    def require_training_allowed(self, label: str) -> None:
        allowed = ALLOW_TRAINING if self.allow_training is None else bool(self.allow_training)
        if not allowed:
            raise MissingArtifactError(
                f"ALLOW_TRAINING=False (artifact-replay mode) but no identity-matched checkpoint "
                f"exists for '{label}'."
            )


STORE = CheckpointStore(CKPT_DIR)


def generator_state(gen: torch.Generator) -> torch.Tensor:
    return gen.get_state().clone()


def restore_generator(gen: torch.Generator, state: torch.Tensor) -> None:
    gen.set_state(state.cpu() if state.device.type != "cpu" else state)


print("Provenance utilities ready. Checkpoint root:", CKPT_DIR)

## 3. Benchmark adapter, action codec verified against the environment, and the two support objects

Terminology used everywhere below (audit m06 / M02):

* **Released benchmark-support mask** `SUPPORT_MASK[s, a]` — read from the benchmark's
  `admissible_actions`. In the benchmark construction code an action is admissible when its source
  transition count is **strictly greater than** the threshold (`metadata.threshold = 20`; verified in
  Section 4). It is *privileged side information* relative to a purely logged-data learner, and it is
  not ground-truth clinical safety. "Inadmissible" = outside this mask.
* **Training-dataset support estimate** — computed from counts in the simulated offline dataset
  (Sections 9 and 25). It answers a different question from the released mask.
* `OPERATIONAL_MASK` equals the released mask on live (non-terminal) states and is all-True on the
  three absorbing states, only so that tensor operations on terminal rows are defined.

In [ ]:
# ============================================================
# 3. ICU-Sepsis benchmark adapter
# ============================================================
EXECLOG.mark("3. Benchmark adapter")

from icu_sepsis.utils import constants as ICU_CONSTANTS


class ActionCodec:
    """Flat 25-action index <-> environment action, verified against the environment's own mapping."""

    def __init__(self, action_space, num_actions: int):
        self.num_actions = int(num_actions)
        self.levels = int(round(math.sqrt(self.num_actions)))
        assert self.levels * self.levels == self.num_actions
        if hasattr(action_space, "n"):
            self.kind = "discrete"
            assert int(action_space.n) == self.num_actions
        elif hasattr(action_space, "nvec"):
            self.kind = "multidiscrete"
            assert int(np.prod(action_space.nvec)) == self.num_actions
        else:
            raise TypeError(f"Unsupported action space: {action_space}")

    def components(self, action_idx: int) -> Tuple[int, int]:
        """(fluid level, vasopressor level) — fluid-major, as in the benchmark construction code."""
        a = int(action_idx)
        return a // self.levels, a % self.levels

    def to_env(self, action_idx: int):
        a = int(action_idx)
        if not 0 <= a < self.num_actions:
            raise ValueError(f"Invalid flat action index {a}")
        return a if self.kind == "discrete" else np.asarray(self.components(a), dtype=np.int64)


def find_wrapper(env, class_name: str):
    e = env
    while e is not None:
        if type(e).__name__ == class_name:
            return e
        e = getattr(e, "env", None)
    return None


class ICUSepsisBenchmark:
    ENV_ID = "Sepsis/ICU-Sepsis-v2"

    def __init__(self):
        self.env_mean = gym.make(self.ENV_ID, inadmissible_action_strategy="mean")
        self.env_terminate = gym.make(self.ENV_ID, inadmissible_action_strategy="terminate")
        self.base = self.env_mean.unwrapped
        self.num_states = int(self.base.num_states)
        self.num_actions = int(self.base.num_actions)
        self.codec = ActionCodec(self.env_mean.action_space, self.num_actions)

        self.state_features = np.asarray(self.base.state_cluster_centers, dtype=np.float32)
        self.sofa_scores = np.asarray(self.base.sofa_scores, dtype=np.float64)
        self.expert_policy = np.asarray(self.base.expert_policy, dtype=np.float64)
        self.env_metadata = dict(self.base.env_metadata)

        self.terminal_states = sorted(int(x) for x in ICU_CONSTANTS.STATES_TERMINAL)
        self.death_state = int(ICU_CONSTANTS.STATE_DEATH)
        self.survival_state = int(ICU_CONSTANTS.STATE_SURVIVAL)
        self.sinf_state = int(ICU_CONSTANTS.STATE_S_INF)
        self.live = np.ones(self.num_states, dtype=bool)
        self.live[self.terminal_states] = False

        admissible = self.base.dynamics["admissible_actions"]
        self.support_mask = np.zeros((self.num_states, self.num_actions), dtype=bool)
        for s, acts in enumerate(admissible):
            self.support_mask[s, np.asarray(list(acts), dtype=np.int64)] = True
        self.operational_mask = self.support_mask.copy()
        self.operational_mask[~self.live] = True
        self.feature_dim = int(self.state_features.shape[1])

        self.version = installed_version("icu-sepsis")
        asset_dir = Path(inspect.getfile(type(self.base))).parent / "assets"
        self.asset_hashes = {p.name: sha256_file(p) for p in sorted(asset_dir.glob("*")) if p.is_file()}
        self.assets_hash = obj_hash(self.asset_hashes)
        self.identity = {"env_id": self.ENV_ID, "icu_sepsis": self.version, "assets_hash": self.assets_hash}

    def exact_mdp(self) -> Dict[str, np.ndarray]:
        d = self.base.dynamics
        return {
            "tx_mat": np.array(d["tx_mat"], dtype=np.float64, copy=True),
            "r_mat": np.array(d["r_mat"], dtype=np.float64, copy=True),
            "d_0": np.array(d["d_0"], dtype=np.float64, copy=True),
        }

    def verify_action_codec(self) -> Dict[str, Any]:
        """Check the flat index against the environment's own wrapper + base-env mapping (audit M09)."""
        flat = find_wrapper(self.env_mean, "FlattenActionWrapper")
        rows = []
        for a in range(self.num_actions):
            env_pair = tuple(int(x) for x in flat.action(a)) if flat is not None else self.codec.components(a)
            base_idx = int(self.base._get_action_idx(np.asarray(env_pair, dtype=np.int64)))
            rows.append({"flat": a, "env_pair": env_pair, "base_index": base_idx,
                         "codec_pair": self.codec.components(a)})
        ok = all(r["base_index"] == r["flat"] and tuple(r["codec_pair"]) == tuple(r["env_pair"]) for r in rows)
        return {"wrapper_found": flat is not None, "action_space": str(self.env_mean.action_space),
                "all_25_actions_consistent": bool(ok),
                "convention": "flat = levels * fluid_level + vasopressor_level (fluid-major)"}


BENCH = ICUSepsisBenchmark()
S_N, A_N = BENCH.num_states, BENCH.num_actions
LIVE = BENCH.live
SUPPORT_MASK = BENCH.support_mask          # released benchmark-support mask
OPERATIONAL_MASK = BENCH.operational_mask  # released mask on live rows, all-True on absorbing rows

CODEC_CHECK = BENCH.verify_action_codec()
assert CODEC_CHECK["all_25_actions_consistent"], CODEC_CHECK
assert S_N == 716 and A_N == 25
assert SUPPORT_MASK[LIVE].any(axis=1).all(), "A live state has an empty released-support row."
assert np.isfinite(BENCH.state_features).all(), "Non-finite state-centroid features."
assert np.isfinite(BENCH.sofa_scores).all(), "Non-finite SOFA scores."

# ---------- Mean-rule identity audit (prerequisite for the admissible projection, audit C01) ----------
_EX = BENCH.exact_mdp()
_P, _R = _EX["tx_mat"], _EX["r_mat"]
_r_sa = np.einsum("san,san->sa", _P, _R, optimize=True)
max_row_dev, max_r_dev, n_bad_pairs = 0.0, 0.0, 0
for s in np.flatnonzero(LIVE):
    m = SUPPORT_MASK[s]
    if (~m).any():
        n_bad_pairs += int((~m).sum())
        max_row_dev = max(max_row_dev, float(np.abs(_P[s, ~m] - _P[s, m].mean(axis=0)).max()))
        max_r_dev = max(max_r_dev, float(np.abs(_r_sa[s, ~m] - _r_sa[s, m].mean()).max()))
reward_dest_only = bool(np.allclose(_R, _R[:1, :1, :], atol=0.0))
MEAN_RULE_IDENTITY = {
    "inadmissible_live_pairs": n_bad_pairs,
    "max_abs_dev_inadmissible_row_vs_mean_admissible_rows": max_row_dev,
    "max_abs_dev_expected_reward": max_r_dev,
    "reward_depends_only_on_destination_state": reward_dest_only,
    "reward_values": np.unique(_R).tolist(),
    "rewarding_destinations": np.unique(np.argwhere(_R != 0)[:, 2]).tolist(),
    "benchmark_metadata": BENCH.env_metadata,
}
MEAN_RULE_IDENTITY_HOLDS = bool(max_row_dev <= 1e-12 and max_r_dev <= 1e-12)
MEAN_RULE_IDENTITY["identity_holds_to_1e-12"] = MEAN_RULE_IDENTITY_HOLDS
del _EX, _P, _R, _r_sa
gc.collect()

print(json.dumps({"codec": CODEC_CHECK, "mean_rule_identity": MEAN_RULE_IDENTITY}, indent=2, default=_json_default))
if not MEAN_RULE_IDENTITY_HOLDS:
    print("WARNING: inadmissible transition rows are NOT the uniform mean of admissible rows in this build; "
          "the mean-equivalence of the admissible projection (Section 19) will be reported, not assumed.")
save_json({"codec": CODEC_CHECK, "mean_rule_identity": MEAN_RULE_IDENTITY, "benchmark_identity": BENCH.identity},
          TABLE_DIR / "benchmark_semantics_audit.json")

## 4. Benchmark audit and construction provenance (audit M02, M08, m04)

* Admissible-set statistics and the SOFA association are reported with **Pearson r** and
  **Spearman ρ** labelled separately (the previous notebook printed an unlabelled Pearson value while
  the original study quoted Spearman; both are legitimate but different statistics).
* The benchmark's construction scripts are fetched from the public repository at a pinned commit and
  **parsed as text / AST only (never executed)** to verify: the support threshold inequality, the
  inadmissible-row mapping method, the action definition, the 47-feature order, and whether
  treatment variables are also clustering features. If the fetch fails (offline), an embedded
  transcription made on 17 Oct 2026 from the same commit is used and flagged as such.

In [ ]:
# ============================================================
# 4. Benchmark audit + construction provenance
# ============================================================
EXECLOG.mark("4. Benchmark audit and provenance")

admissible_sizes = SUPPORT_MASK.sum(axis=1)
live_sizes = admissible_sizes[LIVE]
pearson_r, pearson_p = stats.pearsonr(BENCH.sofa_scores[LIVE], live_sizes)
spearman_rho, spearman_p = stats.spearmanr(BENCH.sofa_scores[LIVE], live_sizes)

BENCHMARK_AUDIT = {
    "num_states": S_N,
    "num_actions": A_N,
    "feature_dim": BENCH.feature_dim,
    "terminal_states": {"death": BENCH.death_state, "survival": BENCH.survival_state, "s_inf": BENCH.sinf_state},
    "live_states": int(LIVE.sum()),
    "admissible_pairs_live": int(SUPPORT_MASK[LIVE].sum()),
    "admissible_per_state_min": int(live_sizes.min()),
    "admissible_per_state_max": int(live_sizes.max()),
    "admissible_per_state_mean": float(live_sizes.mean()),
    "states_with_single_admissible_action": int((live_sizes == 1).sum()),
    "sofa_vs_admissible_count_pearson_r": float(pearson_r),
    "sofa_vs_admissible_count_pearson_p": float(pearson_p),
    "sofa_vs_admissible_count_spearman_rho": float(spearman_rho),
    "sofa_vs_admissible_count_spearman_p": float(spearman_p),
    "benchmark_identity": BENCH.identity,
    "env_metadata": BENCH.env_metadata,
}
print(json.dumps(BENCHMARK_AUDIT, indent=2, default=_json_default))
save_json(BENCHMARK_AUDIT, TABLE_DIR / "benchmark_audit.json")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(live_sizes, bins=np.arange(live_sizes.min(), live_sizes.max() + 2) - 0.5)
ax.set_xlabel("Admissible actions in the released benchmark-support mask")
ax.set_ylabel("Live (non-terminal) states")
ax.set_title("Released benchmark-support set sizes (713 live states)")
fig.tight_layout()
fig.savefig(FIG_DIR / "benchmark_support_set_sizes.png", dpi=200, bbox_inches="tight")
plt.show()

# ---------- Construction provenance (parsed, never executed) ----------
import ast
import re

PROVENANCE_REF = {
    "repository": "icu-sepsis/icu-sepsis",
    "commit": "c9a6d19b85d34883304afaf7cbe1ff02ede9e12b",
    "path": "packages/icu_sepsis_helpers/icu_sepsis_helpers/mdp_creation",
    "sha256": {
        "create_rl_table.py": "7f3c83b4845cec29dcc123d4cc8321223283a8827449b6e894357cc6aebea305",
        "create_matrices.py": "cb4db6738c70a245cbffe90cc792be0e03d89ca8954be0442ad2a5083564d4cd",
        "parse_matrices.py": "6bd716ab2bf502656d85c8292588f4e7999dde69fe3fbce3f2baf1ecb75f8a73",
    },
    "note": ("The helper package is not shipped in the icu-sepsis 2.0.1 wheel. The commit is the repository "
             "HEAD at audit time (2024-10-23); confirm it corresponds to the released 2.0.1 assets."),
}

# Embedded transcription (fallback when offline) of create_rl_table.py at the commit above.
EMBEDDED_FEATURE_ORDER = {
    "colbin": ["gender", "mechvent", "max_dose_vaso", "re_admission"],
    "colnorm": ["age", "Weight_kg", "GCS", "HR", "SysBP", "MeanBP", "DiaBP", "RR", "Temp_C", "FiO2_1",
                "Potassium", "Sodium", "Chloride", "Glucose", "Magnesium", "Calcium", "Hb", "WBC_count",
                "Platelets_count", "PTT", "PT", "Arterial_pH", "paO2", "paCO2", "Arterial_BE", "HCO3",
                "Arterial_lactate", "SOFA", "SIRS", "Shock_Index", "PaO2_FiO2", "cumulated_balance"],
    "collog": ["SpO2", "BUN", "Creatinine", "SGOT", "SGPT", "Total_bili", "INR",
               "input_total", "input_4hourly", "output_total", "output_4hourly"],
}
EMBEDDED_FACTS = {
    "threshold_rule_in_code": "s_a_mat <= transition_threshold  -> zeroed (admissible iff count > threshold)",
    "action_definition": "action = n_action_levels * level(input_4hourly) + level(max_dose_vaso)",
    "treatment_columns_are_clustering_features": True,
    "inadmissible_row_method": "uniform_unweighted (mean of admissible rows)",
    "feature_transforms": "binary: x-0.5; normal: z-score; log: z-score(log(x+0.1))",
    "construction_quirks": [
        "all_cols[3] ('re_admission') receives +0.6 after normalisation (code comment says 'max dose?')",
        "all_cols[44] ('input_4hourly') is multiplied by 2 after normalisation ('increase weight')",
    ],
}


def fetch_text(url: str, timeout: float = 15.0) -> Optional[str]:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            return resp.read().decode("utf-8")
    except Exception as exc:  # offline or blocked
        print("  fetch failed:", url, "->", type(exc).__name__)
        return None


def parse_list_assignments(source: str, names: Sequence[str]) -> Dict[str, List[str]]:
    out = {}
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Assign) and isinstance(node.value, ast.List):
            for tgt in node.targets:
                if isinstance(tgt, ast.Name) and tgt.id in names and tgt.id not in out:
                    out[tgt.id] = list(ast.literal_eval(node.value))
    return out


def nospace(s: str) -> str:
    return re.sub(r"\s+", "", s)


PROVENANCE = {"reference": PROVENANCE_REF, "source": "embedded transcription", "checks": {}}
fetched = {}
if VERIFY_PROVENANCE_ONLINE:
    base_url = f"https://raw.githubusercontent.com/{PROVENANCE_REF['repository']}/{PROVENANCE_REF['commit']}/{PROVENANCE_REF['path']}"
    for fname in PROVENANCE_REF["sha256"]:
        txt = fetch_text(f"{base_url}/{fname}")
        if txt is not None:
            fetched[fname] = txt

if len(fetched) == len(PROVENANCE_REF["sha256"]):
    PROVENANCE["source"] = "fetched at runtime from the pinned commit"
    PROVENANCE["fetched_sha256"] = {k: sha256_bytes(v.encode("utf-8")) for k, v in fetched.items()}
    PROVENANCE["sha256_matches_reference"] = {
        k: PROVENANCE["fetched_sha256"][k] == PROVENANCE_REF["sha256"][k] for k in fetched}
    lists = parse_list_assignments(fetched["create_rl_table.py"], ["colbin", "colnorm", "collog"])
    rl = nospace(fetched["create_rl_table.py"])
    mx = nospace(fetched["create_matrices.py"])
    pm = nospace(fetched["parse_matrices.py"])
    PROVENANCE["checks"] = {
        "feature_lists_match_embedded": lists == EMBEDDED_FEATURE_ORDER,
        "threshold_rule_count_le_threshold_removed": "below_thresh=np.where(s_a_mat<=transition_threshold)" in mx,
        "fluid_level_from_input_4hourly": "ac1=get_action_nums(Dataset_Raw,'input_4hourly',n_action_levels)" in rl,
        "vaso_level_from_max_dose_vaso": "ac2=get_action_nums(Dataset_Raw,'max_dose_vaso',n_action_levels)" in rl,
        "action_is_fluid_major": "Dataset['action']=n_action_levels*ac1+ac2" in rl,
        "states_clustered_on_all_47_features": ("cluster_states(Dataset,cols" in rl
                                                 and "n_action_levels,all_cols,n_clustering" in rl),
        "uniform_unweighted_mapping_implemented": "elifmethod=='uniform_unweighted':" in pm,
        "readmission_plus_0.6_quirk": "Dataset[all_cols[4-1]]=Dataset[all_cols[4-1]]+0.6" in rl,
        "input_4hourly_times_2_quirk": "Dataset[all_cols[45-1]]=Dataset[all_cols[45-1]]*2" in rl,
    }
    FEATURE_ORDER = lists if lists else EMBEDDED_FEATURE_ORDER
else:
    PROVENANCE["checks"] = {"note": "offline: embedded transcription used; verify manually", **EMBEDDED_FACTS}
    FEATURE_ORDER = EMBEDDED_FEATURE_ORDER
PROVENANCE["embedded_facts"] = EMBEDDED_FACTS
PROVENANCE["metadata_threshold"] = BENCH.env_metadata.get("threshold")
PROVENANCE["metadata_action_map_method"] = BENCH.env_metadata.get("action_map_method")
print(json.dumps(PROVENANCE, indent=2, default=_json_default))
save_json(PROVENANCE, TABLE_DIR / "benchmark_construction_provenance.json")

FEATURE_NAMES = FEATURE_ORDER["colbin"] + FEATURE_ORDER["colnorm"] + FEATURE_ORDER["collog"]
assert len(FEATURE_NAMES) == BENCH.feature_dim == 47, (len(FEATURE_NAMES), BENCH.feature_dim)

TREATMENT_RELATION = {
    "input_4hourly": ("defines the logged fluid level of the same 4-h bloc", "high"),
    "max_dose_vaso": ("defines the logged vasopressor level of the same 4-h bloc", "high"),
    "input_total": ("cumulative fluid input, may include the current bloc", "high"),
    "cumulated_balance": ("cumulative fluid balance, may include the current bloc", "medium"),
    "output_4hourly": ("urine output of the bloc, plausibly treatment-affected", "medium"),
    "output_total": ("cumulative urine output, plausibly treatment-affected", "medium"),
    "mechvent": ("concurrent intervention indicator", "medium"),
}
centroids_live = BENCH.state_features[LIVE].astype(np.float64)
feat_rows = []
for j, name in enumerate(FEATURE_NAMES):
    group = ("binary (x - 0.5)" if name in FEATURE_ORDER["colbin"]
             else "z-score" if name in FEATURE_ORDER["colnorm"] else "z-score(log(x + 0.1))")
    rel, risk = TREATMENT_RELATION.get(name, ("physiological / demographic", "low (verify measurement time within bloc)"))
    feat_rows.append({
        "index_0based": j,
        "index_1based": j + 1,
        "feature": name,
        "transform_group": group,
        "treatment_relation": rel,
        "timing_risk": risk,
        "is_clustering_feature": True,
        "centroid_mean_live": float(centroids_live[:, j].mean()),
        "centroid_sd_live": float(centroids_live[:, j].std()),
        "centroid_min_live": float(centroids_live[:, j].min()),
        "centroid_max_live": float(centroids_live[:, j].max()),
    })
FEATURE_PROVENANCE = pd.DataFrame(feat_rows)
FEATURE_PROVENANCE.to_csv(TABLE_DIR / "feature_provenance_timing.csv", index=False)
display(FEATURE_PROVENANCE[["index_1based", "feature", "transform_group", "treatment_relation", "timing_risk"]])

TREATMENT_FEATURES_DIRECT = tuple(FEATURE_NAMES.index(n) for n in ("input_4hourly", "max_dose_vaso", "input_total"))
TREATMENT_FEATURES_RELATED = tuple(sorted(set(TREATMENT_FEATURES_DIRECT) | {
    FEATURE_NAMES.index(n) for n in ("cumulated_balance", "output_4hourly", "output_total", "mechvent")}))
print("Centroids are in the benchmark's transformed units (see transform_group); they are not raw clinical units.")
print("Treatment-direct feature indices:", TREATMENT_FEATURES_DIRECT, "| treatment-related:", TREATMENT_FEATURES_RELATED)

## 5. Freeze / load the offline dataset with explicit episode semantics (audit M10, C03)

* The environment is stepped **through its registered wrapper chain** (`TimeLimit → OrderEnforcing →
  PassiveEnvChecker → FlattenActionWrapper → ICUSepsisEnv`) with the flat action index; the previous
  notebook reset the wrapper but stepped the unwrapped object.
* Three end-of-episode events are stored separately: `terminated` (absorbing state reached),
  `truncated` (benchmark `TimeLimit`, 500 steps) and `cutoff` (this notebook's local maximum length).
  Learners bootstrap with `1 - terminal`, where `terminal = terminated`; a censored (truncated / cutoff)
  transition is **not** converted into death.
* The dataset file name and manifest contain a hash of the full generation specification (behaviour,
  episodes, seed, cutoff, environment id, benchmark assets, generator code). A cached file is used only if
  its recomputed content hash equals its manifest; the old unconditional `FORCE_REBUILD_DATA = True`
  override is gone.
* The clinician dataset uses the same generation stream as the previous notebook (`default_rng(2026)`),
  so the main dataset is expected to be bit-identical to the original 93,563-transition dataset; Section 21
  verifies this against the original file when it is available.

In [ ]:
# ============================================================
# 5. Offline dataset specification, generation and validation
# ============================================================
EXECLOG.mark("5. Offline dataset")


@dataclass(frozen=True)
class DatasetSpec:
    n_episodes: int
    seed: int
    max_steps: int
    behavior: str = "clinician"
    strategy: str = "mean"

    def as_dict(self) -> dict:
        return {**asdict(self), "env_id": ICUSepsisBenchmark.ENV_ID, "benchmark": BENCH.identity}


class DatasetIntegrityError(RuntimeError):
    pass


def _validated_expert_rows() -> np.ndarray:
    ep = np.array(BENCH.expert_policy, dtype=np.float64, copy=True)
    live_rows = ep[LIVE]
    if not (np.isfinite(live_rows).all() and (live_rows >= 0).all() and (live_rows.sum(axis=1) > 0).all()):
        raise DatasetIntegrityError("Benchmark expert policy has invalid live rows.")
    return ep


EXPERT_ROWS = _validated_expert_rows()


def make_behavior(behavior: str) -> Callable[[int, np.random.Generator], int]:
    """'clinician' | 'uniform' | 'uniform_admissible' | 'clinician+uniform:EPS' | 'clinician+uniform_admissible:EPS'."""
    def clinician(s, rng):
        p = EXPERT_ROWS[int(s)].copy()
        p /= p.sum()
        return int(rng.choice(A_N, p=p))

    def uniform(s, rng):
        return int(rng.integers(0, A_N))

    def uniform_admissible(s, rng):
        return int(rng.choice(np.flatnonzero(SUPPORT_MASK[int(s)])))

    components = {"clinician": clinician, "uniform": uniform, "uniform_admissible": uniform_admissible}
    if "+" not in behavior:
        return components[behavior]
    base_name, rest = behavior.split("+", 1)
    mix_name, eps = rest.split(":")
    eps = float(eps)
    base_fn, mix_fn = components[base_name], components[mix_name]

    def mixture(s, rng):
        return mix_fn(s, rng) if rng.random() < eps else base_fn(s, rng)
    return mixture


def build_offline_dataset(spec: DatasetSpec) -> Dict[str, np.ndarray]:
    env = gym.make(ICUSepsisBenchmark.ENV_ID, inadmissible_action_strategy=spec.strategy)
    behavior = make_behavior(spec.behavior)
    # Dataset-generation stream: default_rng(seed), as in the original notebook (main dataset reproducible).
    rng = np.random.default_rng(spec.seed)
    cols = {k: [] for k in ["state", "action", "reward", "next_state", "terminated", "truncated", "cutoff",
                            "episode_id", "timestep"]}
    for ep in trange(spec.n_episodes, desc=f"Generating {spec.behavior} data (seed {spec.seed})",
                     leave=False, disable=not SHOW_PROGRESS):
        reset_seed = int(rng.integers(0, 2 ** 31 - 1))
        state, _ = env.reset(seed=reset_seed)
        state = int(state)
        for t in range(spec.max_steps):
            a = behavior(state, rng)
            next_state, reward, terminated, truncated, _ = env.step(BENCH.codec.to_env(a))
            next_state = int(next_state)
            terminated, truncated = bool(terminated), bool(truncated)
            cutoff = (t == spec.max_steps - 1) and not (terminated or truncated)
            cols["state"].append(state)
            cols["action"].append(a)
            cols["reward"].append(float(reward))
            cols["next_state"].append(next_state)
            cols["terminated"].append(terminated)
            cols["truncated"].append(truncated)
            cols["cutoff"].append(cutoff)
            cols["episode_id"].append(ep)
            cols["timestep"].append(t)
            state = next_state
            if terminated or truncated:
                break
    env.close()
    data = {
        "state": np.asarray(cols["state"], dtype=np.int64),
        "action": np.asarray(cols["action"], dtype=np.int64),
        "reward": np.asarray(cols["reward"], dtype=np.float32),
        "next_state": np.asarray(cols["next_state"], dtype=np.int64),
        "terminated": np.asarray(cols["terminated"], dtype=bool),
        "truncated": np.asarray(cols["truncated"], dtype=bool),
        "cutoff": np.asarray(cols["cutoff"], dtype=bool),
        "episode_id": np.asarray(cols["episode_id"], dtype=np.int32),
        "timestep": np.asarray(cols["timestep"], dtype=np.int32),
    }
    data["terminal"] = data["terminated"].astype(np.float32)          # bootstrapping mask: 1 - terminal
    data["logged_action_admissible"] = SUPPORT_MASK[data["state"], data["action"]]
    return data


REQUIRED_FIELDS = ("state", "action", "reward", "next_state", "terminated", "truncated", "cutoff",
                   "terminal", "episode_id", "timestep", "logged_action_admissible")


def validate_dataset(data: Dict[str, np.ndarray], spec: Optional[DatasetSpec] = None) -> Dict[str, Any]:
    """Strict integrity gate (audit M10). Raises DatasetIntegrityError on any violation."""
    def check(cond, msg):
        if not bool(cond):
            raise DatasetIntegrityError(msg)

    missing = set(REQUIRED_FIELDS).difference(data)
    check(not missing, f"Missing dataset fields: {sorted(missing)}")
    n = len(data["state"])
    check(n > 0 and all(len(data[k]) == n for k in REQUIRED_FIELDS), "Dataset arrays have unequal lengths.")
    s, a, ns = data["state"], data["action"], data["next_state"]
    r = data["reward"].astype(np.float64)
    term, trunc, cut = data["terminated"], data["truncated"], data["cutoff"]
    ep, ts = data["episode_id"].astype(np.int64), data["timestep"].astype(np.int64)

    check(np.isfinite(r).all(), "Non-finite rewards.")
    check(((s >= 0) & (s < S_N)).all() and ((ns >= 0) & (ns < S_N)).all(), "State index out of range.")
    check(((a >= 0) & (a < A_N)).all(), "Action index out of range.")
    check(LIVE[s].all(), "A transition starts from an absorbing (terminal) state.")
    check(np.array_equal(data["logged_action_admissible"], SUPPORT_MASK[s, a]), "Admissibility flags disagree with the mask.")
    check(np.array_equal(data["terminal"].astype(bool), term), "terminal != terminated.")

    starts = np.r_[True, ep[1:] != ep[:-1]]
    ends = np.r_[ep[1:] != ep[:-1], True]
    check((np.diff(ep) >= 0).all(), "Episode ids are not grouped / sorted.")
    n_ep = int(starts.sum())
    check(np.array_equal(ep[starts], np.arange(n_ep)), "Episode ids are not contiguous 0..E-1.")
    start_pos = np.flatnonzero(starts)
    first_index_of_row = start_pos[np.cumsum(starts) - 1]
    check(np.array_equal(ts, np.arange(n) - first_index_of_row), "Timesteps are not 0..L-1 within episodes.")
    inner = ~ends
    check(np.array_equal(ns[inner], s[np.flatnonzero(inner) + 1]), "next_state does not continue the episode.")
    check(not (term & ~ends).any(), "A terminal event occurs before the end of an episode.")
    check(not ((trunc | cut) & ~ends).any(), "A censoring event occurs before the end of an episode.")
    check((term | trunc | cut)[ends].all(), "An episode ends without a terminal or censoring event.")
    check(not (cut & (term | trunc)).any(), "cutoff overlaps with terminated/truncated.")
    is_terminal_next = ~LIVE[ns]
    check(np.array_equal(is_terminal_next, term), "terminated flag disagrees with the absorbing next state.")
    check(np.isin(r, [0.0, 1.0]).all(), "Rewards outside {0, 1}.")
    check(np.array_equal(r == 1.0, term & (ns == BENCH.survival_state)), "Reward/outcome inconsistency.")
    ep_return = np.bincount(ep, weights=r, minlength=n_ep)
    check(np.isin(ep_return, [0.0, 1.0]).all(), "Episode return outside {0, 1}.")
    lengths = np.bincount(ep, minlength=n_ep)
    if spec is not None:
        check(n_ep == spec.n_episodes, f"Expected {spec.n_episodes} episodes, found {n_ep}.")
        check(lengths.max() <= spec.max_steps, "Episode longer than the local cutoff.")

    final_next = ns[ends]
    return {
        "transitions": int(n),
        "episodes": n_ep,
        "terminated_episodes": int(term[ends].sum()),
        "survival_episodes": int((term[ends] & (final_next == BENCH.survival_state)).sum()),
        "death_episodes": int((term[ends] & (final_next == BENCH.death_state)).sum()),
        "s_inf_episodes": int((term[ends] & (final_next == BENCH.sinf_state)).sum()),
        "truncated_episodes": int(trunc[ends].sum()),
        "cutoff_episodes": int(cut[ends].sum()),
        "censored_episodes": int((trunc | cut)[ends].sum()),
        "mean_length": float(lengths.mean()),
        "median_length": float(np.median(lengths)),
        "max_length": int(lengths.max()),
        "inadmissible_logged_transitions": int((~data["logged_action_admissible"]).sum()),
        "inadmissible_logged_fraction": float((~data["logged_action_admissible"]).mean()),
    }


DATASET_CODE_HASH = code_hash(build_offline_dataset, make_behavior, validate_dataset, DatasetSpec)


def dataset_identity(spec: DatasetSpec) -> dict:
    return {**spec.as_dict(), "generator_code": DATASET_CODE_HASH}


def load_or_build_dataset(spec: DatasetSpec, save: bool = True) -> Tuple[Dict[str, np.ndarray], dict]:
    ident = dataset_identity(spec)
    h = obj_hash(ident)
    path = DATA_DIR / f"offline_{slug(spec.behavior)}_{spec.n_episodes}eps_seed{spec.seed}_max{spec.max_steps}_{h[:10]}.npz"
    manifest_path = path.with_suffix(".manifest.json")
    if save and path.exists() and manifest_path.exists() and not FORCE_REBUILD_DATA:
        manifest = json.loads(manifest_path.read_text())
        with np.load(path, allow_pickle=False) as z:
            data = {k: z[k] for k in z.files}
        content = array_dict_hash(data)
        if content != manifest["content_sha256"] or manifest["spec_hash"] != h:
            raise DatasetIntegrityError(f"Cached dataset {path} fails its manifest hash check.")
        validate_dataset(data, spec)
        print("Loaded identity-matched dataset:", path.name)
        return data, manifest
    if not ALLOW_TRAINING and save:
        raise MissingArtifactError(f"Artifact-replay mode: dataset {path.name} is missing.")
    data = build_offline_dataset(spec)
    report = validate_dataset(data, spec)
    manifest = {
        "spec": ident,
        "spec_hash": h,
        "content_sha256": array_dict_hash(data),
        "file": path.name,
        "created_utc": utc_now(),
        "report": report,
    }
    if save:
        np.savez_compressed(path, **data)
        manifest["file_sha256"] = sha256_file(path)
        save_json(manifest, manifest_path)
        print("Saved dataset:", path.name)
    return data, manifest


# ---------- Cutoff-path test with an intentionally short maximum length (audit M10 closure) ----------
_cut_spec = DatasetSpec(n_episodes=60, seed=11, max_steps=3)
_cut_data = build_offline_dataset(_cut_spec)
CUTOFF_TEST = validate_dataset(_cut_data, _cut_spec)
assert CUTOFF_TEST["cutoff_episodes"] > 0, "Short-cutoff test did not exercise the censoring path."
assert not _cut_data["terminal"][_cut_data["cutoff"]].any(), "Censored rows must not be marked terminal."
print("Short-cutoff test:", CUTOFF_TEST)
del _cut_data

# ---------- Main frozen dataset ----------
MAIN_SPEC = DatasetSpec(n_episodes=CFG.dataset_episodes, seed=CFG.dataset_seed,
                        max_steps=CFG.max_episode_steps, behavior=CFG.behavior)
OFFLINE, OFFLINE_MANIFEST = load_or_build_dataset(MAIN_SPEC)
DATASET_REPORT = validate_dataset(OFFLINE, MAIN_SPEC)
DATASET_PATH = DATA_DIR / OFFLINE_MANIFEST["file"]
print(json.dumps(DATASET_REPORT, indent=2))
save_json({"main": OFFLINE_MANIFEST, "report": DATASET_REPORT, "short_cutoff_test": CUTOFF_TEST},
          TABLE_DIR / "dataset_manifest_main.json")

## 6. Exploratory data analysis of the frozen dataset, with corrected support coverage (audit M01, m04, m05)

The ICU-Sepsis states are processed **cluster centroids** in transformed units, so every figure
describes the *processed benchmark and the simulated clinician-policy dataset*, not individual
MIMIC-III patients, and nothing here is a causal treatment-effect estimate.

**Support-coverage repair (M01).** The previous notebook divided *all* observed state-action pairs
(6,738, including inadmissible ones) by the number of admissible pairs (2,238), giving 301.07%, and its
histogram silently dropped ratios above one. Here observed pairs are first intersected with the released
mask on live states; observed admissible, observed inadmissible, and total observed pairs are separate
fields; per-pair frequency quantiles are reported because "observed at least once" does not imply
reliable estimation; and the histogram population is asserted to equal the 713 live states.

In [ ]:
# ============================================================
# 6. EDA of the frozen offline dataset
# ============================================================
EXECLOG.mark("6. EDA")

EDA_DIR = FIG_DIR / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)
LEVELS = BENCH.codec.levels


def save_figure(fig, path_stem: Path, dpi: int = 300) -> None:
    fig.tight_layout()
    fig.savefig(path_stem.with_suffix(".png"), dpi=dpi, bbox_inches="tight")
    fig.savefig(path_stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def boxplot_compat(ax, data, labels, **kwargs):
    """Matplotlib >= 3.9 uses tick_labels; older versions use labels (audit m05)."""
    try:
        return ax.boxplot(data, tick_labels=labels, **kwargs)
    except TypeError:
        return ax.boxplot(data, labels=labels, **kwargs)


TRANSITIONS = pd.DataFrame({
    "state": OFFLINE["state"],
    "action": OFFLINE["action"],
    "reward": OFFLINE["reward"].astype(np.float64),
    "next_state": OFFLINE["next_state"],
    "terminated": OFFLINE["terminated"],
    "truncated": OFFLINE["truncated"],
    "cutoff": OFFLINE["cutoff"],
    "episode_id": OFFLINE["episode_id"].astype(np.int64),
    "timestep": OFFLINE["timestep"].astype(np.int64),
    "admissible": OFFLINE["logged_action_admissible"],
})
TRANSITIONS["fluid_level"] = TRANSITIONS["action"] // LEVELS
TRANSITIONS["vasopressor_level"] = TRANSITIONS["action"] % LEVELS
TRANSITIONS["sofa"] = BENCH.sofa_scores[TRANSITIONS["state"].to_numpy()]

EPISODES = (
    TRANSITIONS.groupby("episode_id", sort=True)
    .agg(length=("state", "size"), episode_return=("reward", "sum"), initial_state=("state", "first"),
         final_next_state=("next_state", "last"), terminated=("terminated", "last"),
         censored_trunc=("truncated", "last"), censored_cutoff=("cutoff", "last"),
         initial_sofa=("sofa", "first"), mean_sofa=("sofa", "mean"), max_sofa=("sofa", "max"))
    .reset_index()
)


def episode_outcome(final_state: int, terminated: bool) -> str:
    if not terminated:
        return "Censored (cutoff / time limit)"
    if final_state == BENCH.survival_state:
        return "Survival"
    if final_state == BENCH.death_state:
        return "Death"
    if final_state == BENCH.sinf_state:
        return "s_inf terminal"
    return "Other terminal"


EPISODES["outcome"] = [episode_outcome(int(s), bool(t)) for s, t in zip(EPISODES["final_next_state"], EPISODES["terminated"])]
TRANSITIONS = TRANSITIONS.merge(EPISODES[["episode_id", "outcome"]], on="episode_id", how="left", validate="many_to_one")

# ---------- Corrected support audit (M01) ----------
LOGGED_COUNTS = np.zeros((S_N, A_N), dtype=np.int64)
np.add.at(LOGGED_COUNTS, (OFFLINE["state"], OFFLINE["action"]), 1)
allowed = SUPPORT_MASK & LIVE[:, None]
observed = LOGGED_COUNTS > 0
covered = observed & allowed
admissible_pair_coverage = covered.sum() / allowed.sum()
per_state_coverage = covered[LIVE].sum(axis=1) / allowed[LIVE].sum(axis=1)
per_state_inadm_frac = np.divide(
    (LOGGED_COUNTS * ~SUPPORT_MASK)[LIVE].sum(axis=1), LOGGED_COUNTS[LIVE].sum(axis=1),
    out=np.zeros(int(LIVE.sum())), where=LOGGED_COUNTS[LIVE].sum(axis=1) > 0)
assert 0.0 <= admissible_pair_coverage <= 1.0
assert np.all((per_state_coverage >= 0.0) & (per_state_coverage <= 1.0))
assert np.all(covered[LIVE].sum(axis=1) <= allowed[LIVE].sum(axis=1))
hist_cov, hist_edges = np.histogram(per_state_coverage, bins=np.linspace(0, 1, 21))
assert hist_cov.sum() == LIVE.sum(), "Coverage histogram does not contain every live state."

pair_freq = LOGGED_COUNTS[allowed]
quantile_levels = [0.0, 0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.99, 1.0]
SUPPORT_AUDIT = {
    "live_states": int(LIVE.sum()),
    "observed_live_states": int((LOGGED_COUNTS[LIVE].sum(axis=1) > 0).sum()),
    "total_unique_observed_pairs": int(observed.sum()),
    "observed_admissible_pairs": int(covered.sum()),
    "observed_inadmissible_pairs": int((observed & ~SUPPORT_MASK & LIVE[:, None]).sum()),
    "admissible_pairs": int(allowed.sum()),
    "admissible_pair_coverage": float(admissible_pair_coverage),
    "states_with_full_admissible_coverage": int((per_state_coverage == 1.0).sum()),
    "admissible_pair_count_quantiles": {f"q{int(100 * q):02d}": float(np.quantile(pair_freq, q)) for q in quantile_levels},
    "admissible_pairs_with_fewer_than": {str(k): int((pair_freq < k).sum()) for k in (1, 5, 10, 21, 50, 100)},
    "logged_transitions": int(len(OFFLINE["state"])),
    "inadmissible_logged_transitions": int((~OFFLINE["logged_action_admissible"]).sum()),
    "inadmissible_logged_fraction": float((~OFFLINE["logged_action_admissible"]).mean()),
    "states_with_any_inadmissible_logged_action": int(((LOGGED_COUNTS * ~SUPPORT_MASK)[LIVE].sum(axis=1) > 0).sum()),
    "coverage_histogram_counts": hist_cov.tolist(),
    "coverage_histogram_edges": hist_edges.tolist(),
    "note": ("Coverage counts a pair once it is observed; per-pair counts show estimation support. "
             "Inadmissible transitions are retained in the data (not deleted to make a statistic valid)."),
}
save_json(SUPPORT_AUDIT, TABLE_DIR / "corrected_support_audit.json")
print(json.dumps(SUPPORT_AUDIT, indent=2))

STATE_SUPPORT = pd.DataFrame({
    "state": np.arange(S_N),
    "is_live": LIVE,
    "visits": LOGGED_COUNTS.sum(axis=1),
    "admissible_actions": SUPPORT_MASK.sum(axis=1) * LIVE,
    "observed_admissible_actions": covered.sum(axis=1),
    "observed_inadmissible_actions": (observed & ~SUPPORT_MASK & LIVE[:, None]).sum(axis=1),
    "min_count_over_admissible_actions": np.where(allowed, LOGGED_COUNTS, np.iinfo(np.int64).max).min(axis=1) * LIVE,
    "sofa": BENCH.sofa_scores,
})
STATE_SUPPORT["admissible_coverage"] = np.where(LIVE, STATE_SUPPORT["observed_admissible_actions"] / np.maximum(STATE_SUPPORT["admissible_actions"], 1), np.nan)
STATE_SUPPORT.to_csv(TABLE_DIR / "eda_state_support.csv", index=False)

ACTION_COUNTS = pd.DataFrame({
    "action": np.arange(A_N),
    "fluid_level": np.arange(A_N) // LEVELS,
    "vasopressor_level": np.arange(A_N) % LEVELS,
    "count": LOGGED_COUNTS.sum(axis=0),
})
ACTION_COUNTS["percentage"] = 100.0 * ACTION_COUNTS["count"] / ACTION_COUNTS["count"].sum()

EDA_SUMMARY = pd.DataFrame([
    ("Transitions", DATASET_REPORT["transitions"]),
    ("Episodes", DATASET_REPORT["episodes"]),
    ("Mean episode length", DATASET_REPORT["mean_length"]),
    ("Median episode length", DATASET_REPORT["median_length"]),
    ("Maximum episode length", DATASET_REPORT["max_length"]),
    ("Survival episodes", DATASET_REPORT["survival_episodes"]),
    ("Death episodes", DATASET_REPORT["death_episodes"]),
    ("s_inf terminal episodes", DATASET_REPORT["s_inf_episodes"]),
    ("Censored episodes (cutoff or time limit)", DATASET_REPORT["censored_episodes"]),
    ("Observed live states", SUPPORT_AUDIT["observed_live_states"]),
    ("Live-state coverage (%)", 100.0 * SUPPORT_AUDIT["observed_live_states"] / SUPPORT_AUDIT["live_states"]),
    ("Unique observed state-action pairs (all)", SUPPORT_AUDIT["total_unique_observed_pairs"]),
    ("Observed admissible pairs", SUPPORT_AUDIT["observed_admissible_pairs"]),
    ("Observed inadmissible pairs", SUPPORT_AUDIT["observed_inadmissible_pairs"]),
    ("Admissible pairs (released mask)", SUPPORT_AUDIT["admissible_pairs"]),
    ("Admissible-pair coverage (%)", 100.0 * SUPPORT_AUDIT["admissible_pair_coverage"]),
    ("Median logged count per admissible pair", SUPPORT_AUDIT["admissible_pair_count_quantiles"]["q50"]),
    ("Admissible pairs with < 21 logged transitions", SUPPORT_AUDIT["admissible_pairs_with_fewer_than"]["21"]),
    ("Logged inadmissible transitions", SUPPORT_AUDIT["inadmissible_logged_transitions"]),
    ("Logged inadmissible fraction (%)", 100.0 * SUPPORT_AUDIT["inadmissible_logged_fraction"]),
], columns=["metric", "value"])
display(EDA_SUMMARY)
EDA_SUMMARY.to_csv(TABLE_DIR / "eda_dataset_summary.csv", index=False)
EPISODES.to_csv(TABLE_DIR / "eda_episode_summary.csv", index=False)
ACTION_COUNTS.to_csv(TABLE_DIR / "eda_action_counts.csv", index=False)
TRANSITIONS.to_csv(TABLE_DIR / "eda_transitions_enriched.csv.gz", index=False, compression="gzip")
print(f"Table 1 reconciliation: admissible pairs = {SUPPORT_AUDIT['admissible_pairs']}, "
      f"admissible-pair coverage = {100 * SUPPORT_AUDIT['admissible_pair_coverage']:.3f}% "
      f"(observed admissible pairs = {SUPPORT_AUDIT['observed_admissible_pairs']}).")
# Values reported in the manuscript's Table 1 / the original notebook output (paper mode only is comparable).
MANUSCRIPT_TABLE1 = {"logged_transitions": 93563, "episodes": 10000, "total_unique_observed_pairs": 6738,
                     "admissible_pairs": 2238, "inadmissible_logged_transitions": 11106, "admissible_pair_coverage": 1.0}
_regen = {**SUPPORT_AUDIT, "episodes": DATASET_REPORT["episodes"]}
TABLE1_RECON = pd.DataFrame([{"quantity": k, "regenerated": float(_regen[k]), "manuscript_or_original": float(v),
                              "difference": float(_regen[k]) - float(v)} for k, v in MANUSCRIPT_TABLE1.items()])
TABLE1_RECON.to_csv(TABLE_DIR / "table1_reconciliation.csv", index=False)
display(TABLE1_RECON)
if RUN_MODE == "paper" and (TABLE1_RECON["difference"].abs() > 1e-12).any():
    print("WARNING: regenerated dataset statistics differ from manuscript Table 1 / the original run "
          "(see table1_reconciliation.csv).")

# ---------- Figures ----------
outcome_order = ["Survival", "Death", "s_inf terminal", "Censored (cutoff / time limit)", "Other terminal"]
oc = EPISODES["outcome"].value_counts()
labels_present = [x for x in outcome_order if x in oc.index]
fig, ax = plt.subplots(figsize=(7.5, 4.8))
bars = ax.bar(labels_present, [int(oc[x]) for x in labels_present])
for b, x in zip(bars, labels_present):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(oc[x]):,}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Episodes")
ax.set_title("Frozen offline dataset: episode outcomes")
ax.tick_params(axis="x", rotation=15)
save_figure(fig, EDA_DIR / "01_outcome_distribution")

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.hist(EPISODES["length"], bins=np.arange(1, int(EPISODES["length"].max()) + 2) - 0.5)
ax.axvline(EPISODES["length"].median(), linestyle="--", label=f"Median = {EPISODES['length'].median():.1f}")
ax.set_xlabel("Episode length (decisions)")
ax.set_ylabel("Episodes")
ax.set_title("Clinician-policy trajectory length")
ax.legend()
save_figure(fig, EDA_DIR / "02_episode_length_distribution")

grid_pct = 100.0 * LOGGED_COUNTS.sum(axis=0).reshape(LEVELS, LEVELS) / LOGGED_COUNTS.sum()
fig, ax = plt.subplots(figsize=(7.0, 5.8))
im = ax.imshow(grid_pct, origin="lower", aspect="auto", cmap="viridis")
ax.set_xlabel("Vasopressor level")
ax.set_ylabel("IV-fluid level")
ax.set_xticks(range(LEVELS))
ax.set_yticks(range(LEVELS))
ax.set_title("Logged clinician-policy treatment frequency (%)")
fig.colorbar(im, ax=ax).set_label("Transitions (%)")
for i in range(LEVELS):
    for j in range(LEVELS):
        ax.text(j, i, f"{grid_pct[i, j]:.1f}", ha="center", va="center", fontsize=8, color="w")
save_figure(fig, EDA_DIR / "03_clinician_action_grid")

box_labels, box_values = [], []
for label in ["Survival", "Death", "s_inf terminal"]:
    v = EPISODES.loc[EPISODES["outcome"] == label, "initial_sofa"].to_numpy(dtype=np.float64)
    if len(v):
        box_labels.append(label)
        box_values.append(v)
fig, ax = plt.subplots(figsize=(7.5, 4.8))
boxplot_compat(ax, box_values, box_labels, showfliers=False)
ax.set_ylabel("Initial-state mean SOFA")
ax.set_title("Initial severity by simulated outcome")
save_figure(fig, EDA_DIR / "04_initial_sofa_by_outcome")

fig, ax = plt.subplots(figsize=(8.0, 5.0))
traj_rows = []
for label in ["Survival", "Death"]:
    sub = TRANSITIONS[TRANSITIONS["outcome"] == label]
    if sub.empty:
        continue
    g = sub.groupby("timestep")["sofa"].agg(["mean", "std", "count"]).reset_index()
    g = g[g["count"] >= max(20, int(0.01 * max((EPISODES["outcome"] == label).sum(), 1)))].copy()
    if g.empty:
        continue
    g["ci95"] = 1.96 * g["std"] / np.sqrt(g["count"])
    traj_rows.append(g.assign(outcome=label))
    ax.plot(g["timestep"], g["mean"], label=label)
    ax.fill_between(g["timestep"].to_numpy(), (g["mean"] - g["ci95"]).to_numpy(), (g["mean"] + g["ci95"]).to_numpy(), alpha=0.18)
ax.set_xlabel("Timestep")
ax.set_ylabel("Mean state SOFA")
ax.set_title("SOFA along simulated clinician-policy episodes")
ax.legend()
save_figure(fig, EDA_DIR / "05_sofa_trajectory_by_outcome")
if traj_rows:
    pd.concat(traj_rows, ignore_index=True).to_csv(TABLE_DIR / "eda_sofa_trajectory_by_outcome.csv", index=False)

sev = TRANSITIONS.copy()
sev["sofa_quartile"] = pd.qcut(sev["sofa"], q=4, duplicates="drop")
INTENSITY_BY_SOFA = (sev.groupby("sofa_quartile", observed=True)
                     .agg(transitions=("action", "size"), mean_fluid_level=("fluid_level", "mean"),
                          mean_vasopressor_level=("vasopressor_level", "mean"), mean_sofa=("sofa", "mean"))
                     .reset_index())
INTENSITY_BY_SOFA["severity_group"] = [f"Q{i + 1}" for i in range(len(INTENSITY_BY_SOFA))]
INTENSITY_BY_SOFA.astype({"sofa_quartile": str}).to_csv(TABLE_DIR / "eda_treatment_intensity_by_sofa.csv", index=False)
fig, ax = plt.subplots(figsize=(7.5, 4.8))
x = np.arange(len(INTENSITY_BY_SOFA))
ax.bar(x - 0.18, INTENSITY_BY_SOFA["mean_fluid_level"], 0.36, label="IV fluid")
ax.bar(x + 0.18, INTENSITY_BY_SOFA["mean_vasopressor_level"], 0.36, label="Vasopressor")
ax.set_xticks(x)
ax.set_xticklabels(INTENSITY_BY_SOFA["severity_group"])
ax.set_xlabel("State SOFA quartile (Q1 lower to Q4 higher)")
ax.set_ylabel("Mean discretised treatment level")
ax.set_title("Logged treatment intensity across severity strata")
ax.legend()
save_figure(fig, EDA_DIR / "06_treatment_intensity_by_sofa")

top = STATE_SUPPORT[STATE_SUPPORT["visits"] > 0].sort_values("visits", ascending=False).head(25).sort_values("visits")
fig, ax = plt.subplots(figsize=(8.0, 7.0))
ax.barh(top["state"].astype(str), top["visits"])
ax.set_xlabel("Logged transitions")
ax.set_ylabel("State ID")
ax.set_title("Top 25 most visited states")
save_figure(fig, EDA_DIR / "07_top_state_visitation")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
axes[0].hist(per_state_coverage, bins=np.linspace(0, 1, 21))
axes[0].set_xlabel("Observed admissible actions / admissible actions")
axes[0].set_ylabel("Live states (n = 713)")
axes[0].set_title("(a) Admissible-pair coverage per state")
axes[1].hist(np.log10(np.maximum(pair_freq, 1)), bins=40)
axes[1].axvline(np.log10(21), linestyle="--", color="k", label="21 transitions")
axes[1].set_xlabel("log10 logged transitions per admissible pair")
axes[1].set_ylabel("Admissible pairs")
axes[1].set_title("(b) Estimation support per admissible pair")
axes[1].legend()
axes[2].hist(per_state_inadm_frac, bins=np.linspace(0, 1, 21))
axes[2].set_xlabel("Fraction of logged transitions that are inadmissible")
axes[2].set_ylabel("Live states")
axes[2].set_title("(c) Inadmissible logged actions per state")
save_figure(fig, EDA_DIR / "08_corrected_support_coverage")

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.scatter(BENCH.sofa_scores[LIVE], live_sizes, alpha=0.5, s=18)
ax.set_xlabel("State mean SOFA")
ax.set_ylabel("Admissible actions (released mask)")
ax.set_title(f"Severity vs support-set size (Spearman ρ = {spearman_rho:.3f}; Pearson r = {pearson_r:.3f})")
save_figure(fig, EDA_DIR / "09_sofa_vs_admissible_actions")
pd.DataFrame([{"spearman_rho": spearman_rho, "spearman_p": spearman_p, "pearson_r": pearson_r,
               "pearson_p": pearson_p, "n_states": int(LIVE.sum())}]).to_csv(
    TABLE_DIR / "eda_sofa_admissibility_association.csv", index=False)

Xc = BENCH.state_features[LIVE].astype(np.float64)
Xc = Xc - Xc.mean(axis=0, keepdims=True)
_, sv, vt = np.linalg.svd(Xc, full_matrices=False)
coords = Xc @ vt[:2].T
expl = sv ** 2 / (sv ** 2).sum()
visits_live = STATE_SUPPORT.loc[LIVE, "visits"].to_numpy(dtype=np.float64)
fig, ax = plt.subplots(figsize=(8.0, 6.2))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=BENCH.sofa_scores[LIVE], cmap="viridis", alpha=0.72,
                s=16 + 70 * np.log1p(visits_live) / max(np.log1p(visits_live).max(), 1.0))
ax.set_xlabel(f"PC1 ({100 * expl[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({100 * expl[1]:.1f}% variance)")
ax.set_title("State-centroid PCA (mean-centred, benchmark-transformed units)")
fig.colorbar(sc, ax=ax).set_label("Mean SOFA")
save_figure(fig, EDA_DIR / "10_state_centroid_pca")
pd.DataFrame({"state": np.flatnonzero(LIVE), "pc1": coords[:, 0], "pc2": coords[:, 1],
              "sofa": BENCH.sofa_scores[LIVE], "visits": visits_live.astype(int)}).to_csv(
    TABLE_DIR / "eda_state_pca_coordinates.csv", index=False)

fvar = BENCH.state_features[LIVE].var(axis=0)
top_idx = np.argsort(fvar)[-12:]
corr = np.corrcoef(BENCH.state_features[LIVE][:, top_idx].astype(np.float64), rowvar=False)
fig, ax = plt.subplots(figsize=(8.5, 7.0))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
lab = [FEATURE_NAMES[j] for j in top_idx]
ax.set_xticks(range(len(lab)))
ax.set_yticks(range(len(lab)))
ax.set_xticklabels(lab, rotation=45, ha="right")
ax.set_yticklabels(lab)
ax.set_title("Pearson correlation among the 12 highest-variance centroid features")
fig.colorbar(im, ax=ax).set_label("Pearson r")
save_figure(fig, EDA_DIR / "11_top_feature_correlation")
pd.DataFrame(corr, index=lab, columns=lab).to_csv(TABLE_DIR / "eda_top_feature_correlation.csv")

# ---------- Manuscript Figures 2 and 3 (previously produced by unreleased scripts) ----------
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
axes[0, 0].hist(EPISODES["length"], bins=30)
axes[0, 0].set_title("(a) Trajectory length")
axes[0, 0].set_xlabel("Episode length (steps)")
axes[0, 0].set_ylabel("Simulated episodes")
im = axes[0, 1].imshow(grid_pct, origin="lower", cmap="Blues")
for i in range(LEVELS):
    for j in range(LEVELS):
        axes[0, 1].text(j, i, f"{grid_pct[i, j]:.1f}", ha="center", va="center", fontsize=7)
axes[0, 1].set_title("(b) Logged treatment frequency (%)")
axes[0, 1].set_xlabel("Vasopressor level")
axes[0, 1].set_ylabel("Fluid level")
ranked = np.sort(LOGGED_COUNTS[LIVE].sum(axis=1))[::-1]
axes[1, 0].plot(np.arange(len(ranked)), np.maximum(ranked, 1))
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("(c) Uneven state visitation")
axes[1, 0].set_xlabel("State rank by visitation")
axes[1, 0].set_ylabel("Logged transitions")
axes[1, 1].hist(live_sizes, bins=np.arange(1, live_sizes.max() + 2) - 0.5)
axes[1, 1].set_title("(d) Released support-set sizes")
axes[1, 1].set_xlabel("Admissible actions per state")
axes[1, 1].set_ylabel("Live states")
save_figure(fig, FIG_DIR / "paper_fig2_dataset_distributions")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
bins = np.linspace(0, 20, 26)
for label in ["Survival", "Death"]:
    v = EPISODES.loc[EPISODES["outcome"] == label, "initial_sofa"]
    axes[0].hist(v, bins=bins, density=True, histtype="step", linewidth=1.5, label=f"{label} ({len(v):,})")
axes[0].set_title("(a) Initial severity by simulated outcome")
axes[0].set_xlabel("Initial state mean SOFA")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[1].scatter(BENCH.sofa_scores[LIVE], live_sizes, s=8, alpha=0.5)
axes[1].set_title(f"(b) Severity vs support-set size (Spearman ρ = {spearman_rho:.3f})")
axes[1].set_xlabel("State mean SOFA")
axes[1].set_ylabel("Admissible action count")
save_figure(fig, FIG_DIR / "paper_fig3_severity")
print("EDA complete:", EDA_DIR)

## 7. Shared utilities: normalizer bound to data, tensor buffer, strict policy validation, admissible projection, policy registry

* The feature normalizer is visit-weighted over the *training* states of a data bundle (all
  transitions in the main study; training episodes only when an episode split is used, Section 26).
  Its hash is part of every run identity and it is stored inside every neural checkpoint.
* `validate_policy` rejects malformed policies (shape, non-finite, negative, row sums). Repair is
  an explicit opt-in diagnostic (`repair_policy_rows`) and is never applied silently (audit M09).
* `project_to_support` is the **mean-equivalent admissible projection** (audit C01):
  `π̃(a|s) = m(s,a)·[π(a|s) + b(s)/k(s)]`, where `b(s)` is the probability π puts on inadmissible
  actions and `k(s)` the number of admissible actions. Under the benchmark's mean rule every inadmissible
  action already uses the uniform mixture of admissible transition rows, so π and π̃ induce identical
  mean-rule dynamics and rewards, while π̃ never selects an inadmissible action. This equivalence is
  stated in Choudhary et al. (2024, §4.3); it is used here as a control, not claimed as new.
  It differs from `renormalize_on_support`, which conditions π on admissibility.

In [ ]:
# ============================================================
# 7. Shared utilities
# ============================================================
EXECLOG.mark("7. Shared utilities")


class FeatureNormalizer:
    def __init__(self, mean: np.ndarray, std: np.ndarray, keep: Sequence[int]):
        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)
        self.keep = np.asarray(keep, dtype=np.int64)

    @classmethod
    def fit(cls, features: np.ndarray, state_indices: np.ndarray, drop: Sequence[int] = ()):
        keep = [j for j in range(features.shape[1]) if j not in set(drop)]
        x = np.asarray(features[np.asarray(state_indices)][:, keep], dtype=np.float32)  # visit-weighted
        mean = x.mean(axis=0)
        std = x.std(axis=0)
        std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
        return cls(mean, std, keep)

    def transform(self, features: np.ndarray) -> np.ndarray:
        out = (np.asarray(features, dtype=np.float32)[:, self.keep] - self.mean) / self.std
        if not np.isfinite(out).all():
            raise FloatingPointError("Non-finite normalised features.")
        return out.astype(np.float32)

    def to_dict(self) -> dict:
        return {"mean": self.mean.tolist(), "std": self.std.tolist(), "keep": self.keep.tolist()}

    @property
    def hash(self) -> str:
        return array_dict_hash({"mean": self.mean, "std": self.std, "keep": self.keep})


class OfflineTensorBuffer:
    def __init__(self, data: Dict[str, np.ndarray], all_x: np.ndarray, device: torch.device):
        t = lambda a, dt: torch.as_tensor(np.asarray(a), dtype=dt, device=device)
        self.device = device
        self.state = t(data["state"], torch.long)
        self.action = t(data["action"], torch.long)
        self.reward = t(data["reward"], torch.float32)
        self.next_state = t(data["next_state"], torch.long)
        self.terminal = t(data["terminal"], torch.float32)
        self.adm = t(data["logged_action_admissible"], torch.bool)
        self.all_x = t(all_x, torch.float32)
        self.mask = t(OPERATIONAL_MASK, torch.bool)
        self.n = int(len(data["state"]))
        assert torch.isfinite(self.all_x).all(), "Non-finite feature tensor."
        assert torch.isfinite(self.reward).all()

    def batch_from_index(self, idx: torch.Tensor) -> Dict[str, torch.Tensor]:
        s, ns = self.state[idx], self.next_state[idx]
        return {"s_idx": s, "a": self.action[idx], "r": self.reward[idx], "ns_idx": ns,
                "terminal": self.terminal[idx], "adm": self.adm[idx], "x": self.all_x[s], "nx": self.all_x[ns],
                "mask": self.mask[s], "next_mask": self.mask[ns]}

    def sample(self, batch_size: int, generator: torch.Generator) -> Dict[str, torch.Tensor]:
        idx = torch.randint(0, self.n, (batch_size,), generator=generator, device=self.device)
        return self.batch_from_index(idx)

    def chunks(self, size: int = 8192):
        for start in range(0, self.n, size):
            yield self.batch_from_index(torch.arange(start, min(start + size, self.n), device=self.device))


@dataclass
class DataBundle:
    name: str
    data: Dict[str, np.ndarray]
    manifest: dict
    normalizer: FeatureNormalizer
    feature_drop: Tuple[int, ...] = ()

    def __post_init__(self):
        self.data_hash = array_dict_hash(self.data)
        self.all_x = self.normalizer.transform(BENCH.state_features)
        self.counts = np.zeros((S_N, A_N), dtype=np.int64)
        np.add.at(self.counts, (self.data["state"], self.data["action"]), 1)
        self._buffer = None

    @property
    def input_dim(self) -> int:
        return int(self.all_x.shape[1])

    def identity(self) -> dict:
        return {"dataset_hash": self.data_hash, "normalizer_hash": self.normalizer.hash,
                "feature_drop": list(self.feature_drop), "transitions": int(len(self.data["state"]))}

    def buffer(self) -> OfflineTensorBuffer:
        if self._buffer is None:
            self._buffer = OfflineTensorBuffer(self.data, self.all_x, DEVICE)
        return self._buffer

    def release(self) -> None:
        self._buffer = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def subset_rows(data: Dict[str, np.ndarray], rows: np.ndarray) -> Dict[str, np.ndarray]:
    return {k: v[rows] for k, v in data.items()}


def make_bundle(name: str, data: Dict[str, np.ndarray], manifest: dict, rows: Optional[np.ndarray] = None,
                feature_drop: Sequence[int] = ()) -> DataBundle:
    d = data if rows is None else subset_rows(data, rows)
    norm = FeatureNormalizer.fit(BENCH.state_features, d["state"], drop=feature_drop)
    return DataBundle(name=name, data=d, manifest=manifest, normalizer=norm, feature_drop=tuple(feature_drop))


MAIN = make_bundle("main", OFFLINE, OFFLINE_MANIFEST)
np.savez(DATA_DIR / "feature_normalizer_main.npz", mean=MAIN.normalizer.mean, std=MAIN.normalizer.std,
         keep=MAIN.normalizer.keep)
print("Main bundle:", MAIN.identity())

# Feature provenance (M08): terminal-state encodings and the notebook's additional standardisation.
for _lab, _s in (("death", BENCH.death_state), ("survival", BENCH.survival_state), ("s_inf", BENCH.sinf_state)):
    FEATURE_PROVENANCE[f"{_lab}_state_{_s}_centroid_value"] = BENCH.state_features[_s].astype(np.float64)
_mu = np.full(BENCH.feature_dim, np.nan)
_sd = np.full(BENCH.feature_dim, np.nan)
_mu[MAIN.normalizer.keep] = MAIN.normalizer.mean
_sd[MAIN.normalizer.keep] = MAIN.normalizer.std
FEATURE_PROVENANCE["notebook_norm_mean"] = _mu
FEATURE_PROVENANCE["notebook_norm_std"] = _sd
FEATURE_PROVENANCE["notebook_norm_fit_on"] = "visit-weighted current (pre-action) states of the main dataset"
FEATURE_PROVENANCE.to_csv(TABLE_DIR / "feature_provenance_timing.csv", index=False)
print("Absorbing-state centroid vectors are all zeros:", bool(np.all(BENCH.state_features[~LIVE] == 0)))

# ---------- Policy validation, extraction and projection ----------
FIXED = -1  # seed label for fixed (non-learned) policies


class PolicyValidationError(ValueError):
    pass


def validate_policy(pi: np.ndarray, name: str = "policy", atol: float = 1e-8,
                    shape: Optional[Tuple[int, int]] = None) -> np.ndarray:
    pi = np.asarray(pi, dtype=np.float64)
    problems = []
    shape = (S_N, A_N) if shape is None else tuple(shape)
    if pi.shape != shape:
        raise PolicyValidationError(f"{name}: shape {pi.shape} != {shape}")
    if not np.isfinite(pi).all():
        problems.append(f"{int((~np.isfinite(pi)).sum())} non-finite entries")
    if (pi < -1e-12).any():
        problems.append(f"{int((pi < -1e-12).sum())} negative entries")
    rs = pi.sum(axis=1)
    if np.isfinite(rs).all() and not np.allclose(rs, 1.0, atol=atol):
        problems.append(f"{int((np.abs(rs - 1.0) > atol).sum())} rows do not sum to one")
    if problems:
        raise PolicyValidationError(f"{name}: " + "; ".join(problems))
    return np.clip(pi, 0.0, None)


def repair_policy_rows(pi: np.ndarray, fallback_mask: np.ndarray = OPERATIONAL_MASK) -> Tuple[np.ndarray, dict]:
    """Explicit opt-in diagnostic repair. Never used silently by the evaluator."""
    raw = np.asarray(pi, dtype=np.float64)
    out = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    out = np.clip(out, 0.0, None)
    rs = out.sum(axis=1)
    empty = rs <= 0
    for s in np.flatnonzero(empty):
        out[s, fallback_mask[s]] = 1.0 / fallback_mask[s].sum()
    out = out / out.sum(axis=1, keepdims=True)
    report = {"nonfinite_entries": int((~np.isfinite(raw)).sum()), "negative_entries": int((raw < 0).sum()),
              "empty_rows_replaced": int(empty.sum())}
    return out, report


def one_hot_policy(actions: np.ndarray, num_actions: int = A_N) -> np.ndarray:
    actions = np.asarray(actions, dtype=np.int64)
    pi = np.zeros((len(actions), num_actions), dtype=np.float64)
    pi[np.arange(len(actions)), actions] = 1.0
    return pi


def masked_argmax(scores: np.ndarray, mask: np.ndarray = OPERATIONAL_MASK) -> np.ndarray:
    """Argmax restricted to `mask`; ties resolve to the lowest admissible index (deterministic)."""
    m = np.asarray(mask, dtype=bool)
    if not m.any(axis=1).all():
        raise ValueError("masked_argmax: a row has no allowed action.")
    return np.where(m, np.asarray(scores, dtype=np.float64), -np.inf).argmax(axis=1)


def greedy_policy(scores: np.ndarray, masked: bool) -> np.ndarray:
    if masked:
        return one_hot_policy(masked_argmax(scores, OPERATIONAL_MASK))
    return one_hot_policy(np.asarray(scores, dtype=np.float64).argmax(axis=1))


def inadmissible_mass(pi: np.ndarray, mask: np.ndarray = SUPPORT_MASK) -> np.ndarray:
    b = (np.asarray(pi) * (~mask)).sum(axis=1)
    b[~LIVE] = 0.0
    return b


def project_to_support(pi: np.ndarray, mask: np.ndarray = SUPPORT_MASK) -> np.ndarray:
    """Mean-equivalent admissible projection (C01). Terminal rows are left unchanged."""
    pi = validate_policy(pi, "projection input")
    m = np.asarray(mask, dtype=bool)[LIVE]
    if not m.any(axis=1).all():
        raise ValueError("Projection requires a non-empty admissible set in every live state.")
    out = pi.copy()
    live_pi = pi[LIVE]
    bad = (live_pi * ~m).sum(axis=1, keepdims=True)
    out[LIVE] = m * (live_pi + bad / m.sum(axis=1, keepdims=True))
    assert np.allclose(out.sum(axis=1), 1.0, atol=1e-10)
    assert np.all(out[LIVE][~m] == 0.0)
    return out


def renormalize_on_support(pi: np.ndarray, mask: np.ndarray = SUPPORT_MASK) -> Tuple[np.ndarray, int]:
    """Condition π on admissibility; rows with zero admissible mass fall back to uniform-admissible."""
    pi = validate_policy(pi, "renormalisation input")
    m = np.asarray(mask, dtype=bool)
    out = pi.copy()
    num = (pi * m)[LIVE]
    z = num.sum(axis=1, keepdims=True)
    fallback = (z[:, 0] <= 0)
    num[fallback] = m[LIVE][fallback]
    z = num.sum(axis=1, keepdims=True)
    out[LIVE] = num / z
    return out, int(fallback.sum())


def uniform_admissible_policy(mask: np.ndarray = SUPPORT_MASK) -> np.ndarray:
    out = np.full((S_N, A_N), 1.0 / A_N)
    m = mask[LIVE].astype(np.float64)
    out[LIVE] = m / m.sum(axis=1, keepdims=True)
    return out


def tie_and_margin_report(scores: np.ndarray, name: str, tol: float = 1e-9) -> Dict[str, Any]:
    """Greedy-admissibility diagnostics (audit M05): margins and ties between admissible and inadmissible."""
    q = np.asarray(scores, dtype=np.float64)[LIVE]
    m = SUPPORT_MASK[LIVE]
    has_bad = (~m).any(axis=1)
    best_adm = np.where(m, q, -np.inf).max(axis=1)
    best_bad = np.where(~m, q, -np.inf).max(axis=1)
    margin = (best_adm - best_bad)[has_bad]
    q_adm_sorted = np.sort(np.where(m, q, -np.inf), axis=1)
    multi = m.sum(axis=1) >= 2
    adm_ties = int((np.abs(q_adm_sorted[multi, -1] - q_adm_sorted[multi, -2]) <= tol * np.maximum(1.0, np.abs(q_adm_sorted[multi, -1]))).sum())
    return {
        "policy": name,
        "states_with_inadmissible_actions": int(has_bad.sum()),
        "min_margin_admissible_minus_inadmissible": float(margin.min()) if margin.size else float("nan"),
        "states_inadmissible_ranked_first": int((margin < 0).sum()),
        "states_tied_admissible_vs_inadmissible": int((np.abs(margin) <= tol).sum()),
        "states_tied_among_admissible": adm_ties,
    }


# ---------- Policy registry (every evaluated policy is a named, documented object) ----------
@dataclass
class PolicyEntry:
    name: str
    family: str
    extraction: str
    stochastic: bool
    mask_source: str          # "none" | "released benchmark mask" | "dataset-estimated (...)"
    mask_use: str             # "none" | "training" | "extraction" | "training+extraction" | "projection" | ...
    information: str          # data and side information used
    group: str = "main"
    dataset: str = "main"
    parent: str = ""
    critic: str = ""
    notes: str = ""
    by_seed: Dict[int, np.ndarray] = field(default_factory=dict)
    scores: Dict[int, np.ndarray] = field(default_factory=dict)

    def meta(self) -> dict:
        return {k: getattr(self, k) for k in ("name", "family", "extraction", "stochastic", "mask_source",
                                                 "mask_use", "information", "group", "dataset", "parent",
                                                 "critic", "notes")}


REGISTRY: Dict[str, PolicyEntry] = {}
POLICY_FILES: List[dict] = []

# Snapshot of the previous session's exported results and policy files BEFORE anything is overwritten, so
# Section 32 can perform a genuine cross-session replay comparison (audit C04), not only a file round-trip.
_SESSION_TAG = EXECLOG.session_start.replace(":", "")
PREV_SESSION = {"exact_results": None, "policy_hashes": {}}
_prev_results = TABLE_DIR / "exact_results_all.csv"
if _prev_results.exists():
    shutil.copy(_prev_results, LOG_DIR / f"prev_exact_results_{_SESSION_TAG}.csv")
    PREV_SESSION["exact_results"] = pd.read_csv(_prev_results)
for _f in POLICY_DIR.rglob("*.npz"):
    try:
        with np.load(_f) as _z:
            PREV_SESSION["policy_hashes"][str(_f.relative_to(ROOT))] = array_hash(np.asarray(_z["policy"], dtype=np.float64))
    except Exception:
        pass
if PREV_SESSION["policy_hashes"]:
    save_json(PREV_SESSION["policy_hashes"], LOG_DIR / f"prev_policy_hashes_{_SESSION_TAG}.json")
print("Previous-session artifacts found:", PREV_SESSION["exact_results"] is not None,
      "| policy files:", len(PREV_SESSION["policy_hashes"]))


def register_policy(name: str, seed: int, pi: np.ndarray, scores: Optional[np.ndarray] = None,
                    save: bool = True, **meta) -> np.ndarray:
    pi = validate_policy(pi, f"{name} seed={seed}")
    if name not in REGISTRY:
        required = {"family", "extraction", "stochastic", "mask_source", "mask_use", "information"}
        missing = required.difference(meta)
        if missing:
            raise KeyError(f"First registration of {name} needs metadata {sorted(missing)}")
        REGISTRY[name] = PolicyEntry(name=name, **meta)
    entry = REGISTRY[name]
    entry.by_seed[int(seed)] = pi
    if scores is not None:
        entry.scores[int(seed)] = np.asarray(scores, dtype=np.float64)
    if save:
        out = POLICY_DIR / slug(entry.group) / f"{slug(name)}__seed{int(seed)}.npz"
        out.parent.mkdir(parents=True, exist_ok=True)
        payload = {"policy": pi}
        if scores is not None:
            payload["scores"] = np.asarray(scores, dtype=np.float64)
        np.savez_compressed(out, **payload)
        POLICY_FILES.append({"policy": name, "seed": int(seed), "file": str(out.relative_to(ROOT)),
                             "policy_sha256": array_hash(pi)})
    return pi


def registry_frame() -> pd.DataFrame:
    return pd.DataFrame([{**e.meta(), "seeds": sorted(e.by_seed)} for e in REGISTRY.values()])


def entry_seeds(name: str) -> List[int]:
    return sorted(REGISTRY[name].by_seed)


print("Utilities ready.")

## 8. Exact finite-MDP evaluator with strict validation and interpretable violation metrics (audit M09)

The transition model is used **only** for evaluation, oracles, and stress tests — no `train_*` function
receives `P`, `R`, or `d0`. For a policy π and handling rule h ∈ {mean, terminate}, with transient set 𝒯 and
`F = I − P_π,𝒯𝒯`, the evaluator solves (one LU factorisation, three solves):

* value `F v = r_π`, return `J = d0ᵀ v`;
* expected decisions `F ℓ = 1`, `L = d0ᵀ ℓ`;
* occupancy `Fᵀ μ = d0`, with the invariants `L = Σ μ`, absorption probabilities summing to one,
  `J = P(absorb in survival)` (reward 1 only on entering survival), and `0 ≤ J ≤ 1`.

Violation metrics (all relative to the released benchmark-support mask, never "safety"):
pooled step-occupancy rate `U = μᵀb / Σμ`; expected violation count `μᵀb`; probability of at least one
violation (identical under both rules because pre-violation dynamics coincide; under termination it equals
the expected count); number of live states with positive inadmissible mass; maximum and unweighted mean
statewise inadmissible mass. Singular (improper / non-absorbing) systems, large residuals, failed
invariants, malformed policies, and non-converged value iteration all raise.

In [ ]:
# ============================================================
# 8. Exact finite-MDP evaluator
# ============================================================
EXECLOG.mark("8. Exact evaluator")


class ImproperPolicyError(RuntimeError):
    pass


class EvaluationInvariantError(RuntimeError):
    pass


class ExactMDPEvaluator:
    def __init__(self, tx_mat, r_mat, d0, support_mask, terminal_states, death_state, survival_state,
                 r_sa: Optional[np.ndarray] = None, terminal_entry_rewards: Optional[Dict[int, float]] = None,
                 terminate_reward: float = 0.0):
        # terminal_entry_rewards=None keeps the ICU-Sepsis invariants (J = P(absorb in survival), 0 <= J <= 1).
        # Otherwise rewards are paid on entering absorbing states and J = sum_t P(absorb in t) * R_t is checked
        # (Part II environments). terminate_reward is the reward of a forced jump to death_state (0 for ICU-Sepsis).
        P = np.asarray(tx_mat, dtype=np.float64)
        self.d0 = np.asarray(d0, dtype=np.float64)
        self.S, self.A, s2 = P.shape
        if r_mat is None:
            # Expected immediate rewards supplied directly (used for perturbed / empirical MDPs).
            assert r_sa is not None and np.asarray(r_sa).shape == (self.S, self.A)
            R = None
        else:
            R = np.asarray(r_mat, dtype=np.float64)
            assert R.shape == P.shape and np.isfinite(R).all()
        assert self.S == s2 and self.d0.shape == (self.S,)
        assert np.isfinite(P).all() and np.isfinite(self.d0).all()
        assert (P >= -1e-12).all() and (self.d0 >= -1e-12).all()
        assert np.allclose(P.sum(axis=2), 1.0, atol=1e-8), "Transition rows do not sum to one."
        assert np.isclose(self.d0.sum(), 1.0, atol=1e-8)
        self.terminals = np.asarray(sorted(set(int(x) for x in terminal_states)), dtype=np.int64)
        self.live = np.ones(self.S, dtype=bool)
        self.live[self.terminals] = False
        assert np.allclose(self.d0[~self.live], 0.0), "Initial distribution puts mass on absorbing states."
        self.T = np.flatnonzero(self.live)
        self.death, self.survival = int(death_state), int(survival_state)
        self.mask = np.asarray(support_mask, dtype=bool).copy()
        assert self.mask.shape == (self.S, self.A)
        assert self.mask[self.live].any(axis=1).all()
        self.m_eff = self.mask.copy()
        self.m_eff[~self.live] = True
        self.P = P
        self.r_sa_mean = (np.einsum("san,san->sa", P, R, optimize=True) if R is not None
                          else np.asarray(r_sa, dtype=np.float64).copy())
        self.terminate_reward = float(terminate_reward)
        self.terminal_entry_rewards = (None if terminal_entry_rewards is None else
                                       {int(k): float(v) for k, v in terminal_entry_rewards.items()})
        if self.terminal_entry_rewards is not None:
            assert set(self.terminal_entry_rewards) == set(int(t) for t in self.terminals)
            assert np.isclose(self.terminal_entry_rewards[self.death], self.terminate_reward)
        # inadmissible -> death; the forced jump pays the death entry reward (0 for ICU-Sepsis)
        self.r_sa_term = np.where(self.m_eff, self.r_sa_mean, self.terminate_reward)
        del R

    # ---------- policy-induced chain ----------
    def check_policy(self, policy, validate: bool = True, repair: bool = False):
        if repair:
            pi, report = repair_policy_rows(policy, self.m_eff)
            return pi, report
        if validate:
            return validate_policy(policy, "evaluated policy", shape=(self.S, self.A)), None
        return np.asarray(policy, dtype=np.float64), None

    def induced(self, pi: np.ndarray, strategy: str) -> Tuple[np.ndarray, np.ndarray]:
        if strategy == "mean":
            return np.einsum("sa,san->sn", pi, self.P, optimize=True), (pi * self.r_sa_mean).sum(axis=1)
        if strategy == "terminate":
            pi_adm = pi * self.m_eff
            P_pi = np.einsum("sa,san->sn", pi_adm, self.P, optimize=True)
            b = (pi * ~self.m_eff).sum(axis=1)
            P_pi[:, self.death] += b
            return P_pi, (pi_adm * self.r_sa_mean).sum(axis=1) + b * self.terminate_reward
        raise ValueError("strategy must be 'mean' or 'terminate'")

    def _factor(self, M: np.ndarray):
        with warnings.catch_warnings():
            warnings.simplefilter("error", sla.LinAlgWarning)
            try:
                lu = sla.lu_factor(M, check_finite=True)
            except (sla.LinAlgWarning, np.linalg.LinAlgError, ValueError) as exc:
                raise ImproperPolicyError("Singular Bellman system: improper / non-absorbing policy under gamma=1.") from exc
        if np.min(np.abs(np.diag(lu[0]))) < 1e-13:
            raise ImproperPolicyError("Numerically singular Bellman system (improper policy).")
        return lu

    def evaluate(self, policy: np.ndarray, strategy: str, validate: bool = True, repair: bool = False,
                 condition: bool = False, return_vectors: bool = False) -> Dict[str, Any]:
        pi, repair_report = self.check_policy(policy, validate=validate, repair=repair)
        P_pi, r_pi = self.induced(pi, strategy)
        T = self.T
        P_tt = P_pi[np.ix_(T, T)]
        F_mat = np.eye(len(T)) - P_tt
        lu = self._factor(F_mat)
        rT, d0T, ones = r_pi[T], self.d0[T], np.ones(len(T))
        v = sla.lu_solve(lu, rT)
        ell = sla.lu_solve(lu, ones)
        mu = sla.lu_solve(lu, d0T, trans=1)
        res_v = float(np.abs(F_mat @ v - rT).max())
        res_l = float(np.abs(F_mat @ ell - ones).max())
        res_mu = float(np.abs(F_mat.T @ mu - d0T).max())
        tol = 1e-8
        if not (np.isfinite(v).all() and np.isfinite(ell).all() and np.isfinite(mu).all()):
            raise ImproperPolicyError("Non-finite exact solution.")
        if res_v > tol * (1 + np.abs(rT).max()) or res_l > tol * (1 + ell.max()) or res_mu > tol * (1 + mu.max()):
            raise EvaluationInvariantError(f"Residuals too large: value {res_v:.2e}, length {res_l:.2e}, occupancy {res_mu:.2e}")
        if (ell < -1e-8).any() or (mu < -1e-10).any():
            raise EvaluationInvariantError("Negative expected lengths or occupancies.")
        J = float(d0T @ v)
        L = float(d0T @ ell)
        total_occ = float(mu.sum())
        absorb = mu @ P_pi[np.ix_(T, self.terminals)]
        absorb_by_state = {int(s): float(p) for s, p in zip(self.terminals, absorb)}
        if abs(L - total_occ) > 1e-8 * max(1.0, total_occ):
            raise EvaluationInvariantError(f"Expected length {L} != total occupancy {total_occ}.")
        if abs(absorb.sum() - 1.0) > 1e-8:
            raise EvaluationInvariantError(f"Absorption probabilities sum to {absorb.sum()} (improper policy?).")
        if self.terminal_entry_rewards is None:
            if abs(J - absorb_by_state[self.survival]) > 1e-8:
                raise EvaluationInvariantError("Return differs from the survival absorption probability.")
            if not (-1e-10 <= J <= 1 + 1e-10):
                raise EvaluationInvariantError(f"Return {J} outside [0, 1].")
        else:
            J_abs = sum(absorb_by_state[t] * R for t, R in self.terminal_entry_rewards.items())
            if abs(J - J_abs) > 1e-8 * max(1.0, abs(J)):
                raise EvaluationInvariantError(f"Return {J} differs from sum_t P(absorb t) R_t = {J_abs}.")
            lo, hi = min(self.terminal_entry_rewards.values()), max(self.terminal_entry_rewards.values())
            if not (lo - 1e-10 <= J <= hi + 1e-10):
                raise EvaluationInvariantError(f"Return {J} outside [{lo}, {hi}].")

        b = (pi * ~self.m_eff).sum(axis=1)
        exp_viol = float(mu @ b[T])
        if b[T].max() > 0:
            if strategy == "terminate":
                p_any = exp_viol
            else:
                P_term, _ = self.induced(pi, "terminate")
                lu_t = self._factor(np.eye(len(T)) - P_term[np.ix_(T, T)])
                mu_t = sla.lu_solve(lu_t, d0T, trans=1)
                p_any = float(mu_t @ b[T])
        else:
            p_any = 0.0
        out = {
            "return": J,
            "avg_episode_length": L,
            "inadmissible_rate": exp_viol / total_occ if total_occ > 0 else 0.0,
            "expected_violations": exp_viol,
            "prob_any_violation": p_any,
            "states_with_inadmissible_mass": int((b[T] > 1e-12).sum()),
            "max_state_inadmissible_mass": float(b[T].max()),
            "mean_state_inadmissible_mass": float(b[T].mean()),
            "absorb_death": absorb_by_state.get(self.death, 0.0),
            "absorb_survival": absorb_by_state.get(self.survival, 0.0),
            "absorb_other_terminal": float(sum(p for s, p in absorb_by_state.items() if s not in (self.death, self.survival))),
            "residual_value": res_v,
            "residual_length": res_l,
            "residual_occupancy": res_mu,
        }
        if condition:
            out["condition_number"] = float(np.linalg.cond(F_mat))
        if repair_report is not None:
            out["repair_report"] = repair_report
        if return_vectors:
            v_full = np.zeros(self.S)
            v_full[T] = v
            occ_full = np.zeros(self.S)
            occ_full[T] = mu
            out["value_vector"] = v_full
            out["occupancy_vector"] = occ_full
        return out

    # ---------- dynamic programming oracles (fail loudly) ----------
    def q_values(self, V: np.ndarray, strategy: str) -> np.ndarray:
        cont = np.einsum("san,n->sa", self.P, V, optimize=True)
        q = self.r_sa_mean + cont
        if strategy == "terminate":
            q = np.where(self.m_eff, q, self.terminate_reward + V[self.death])
        return q

    def optimal_policy(self, strategy: str = "mean", constrain_to_admissible: bool = False,
                       tol: float = 1e-12, max_iter: int = 200_000,
                       tie_break: str = "admissible") -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
        V = np.zeros(self.S)
        converged, it = False, 0
        for it in range(1, max_iter + 1):
            q = self.q_values(V, strategy)
            if constrain_to_admissible:
                q = np.where(self.m_eff, q, -np.inf)
            new_V = q.max(axis=1)
            new_V[self.terminals] = 0.0
            delta = float(np.max(np.abs(new_V - V)))
            V = new_V
            if delta < tol:
                converged = True
                break
        if not converged:
            raise RuntimeError(f"Value iteration did not converge in {max_iter} iterations (last delta {delta:.3e}).")
        q = self.q_values(V, strategy)
        if constrain_to_admissible:
            q = np.where(self.m_eff, q, -np.inf)
        # Tie-aware greedy choice that prefers admissible actions: under the mean rule an inadmissible action's
        # q is the mean of admissible q's, so exact ties occur (e.g. in single-admissible-action states); the
        # optimal policy is not unique and is reported with admissible tie-breaking.
        qb = q.max(axis=1, keepdims=True)
        tied = q >= qb - 1e-12 * np.maximum(1.0, np.abs(qb))
        pref = tied & self.m_eff
        act = np.where(pref.any(axis=1), pref.argmax(axis=1), tied.argmax(axis=1))
        if tie_break == "lowest_index":   # previous notebook's behaviour (np.argmax), kept for continuity
            act = q.argmax(axis=1)
        policy = one_hot_policy(act, self.A)
        exact = self.evaluate(policy, strategy)["return"]
        info = {"iterations": it, "dp_value": float(self.d0 @ V), "exact_return_of_greedy": exact,
                "abs_gap": abs(exact - float(self.d0 @ V)),
                "near_tie_live_states": int((tied.sum(axis=1) > 1)[self.live].sum()),
                "inadmissible_live_states_in_greedy": int(((policy * ~self.m_eff).sum(axis=1) > 0)[self.live].sum())}
        if info["abs_gap"] > 1e-8:
            raise EvaluationInvariantError(f"Greedy oracle evaluation differs from its DP value: {info}")
        return policy, V, info


# ---------- Analytic toy tests (independent closed forms) ----------
def _toy_mdp(improper: bool = False):
    S, A = 4, 2   # 0,1 transient; 2 death; 3 survival
    P = np.zeros((S, A, S))
    P[0, 0] = [0.1, 0.5, 0.2, 0.2] if not improper else [1.0, 0.0, 0.0, 0.0]
    P[0, 1] = P[0, 0]                     # inadmissible at state 0 -> mean of admissible rows
    P[1, 0] = [0.3, 0.0, 0.3, 0.4]
    P[1, 1] = [0.0, 0.2, 0.5, 0.3]
    for t in (2, 3):
        P[t, :, t] = 1.0
    R = np.zeros((S, A, S))
    R[:, :, 3] = 1.0
    d0 = np.array([1.0, 0.0, 0.0, 0.0])
    mask = np.array([[True, False], [True, True], [True, True], [True, True]])
    return ExactMDPEvaluator(P, R, d0, mask, [2, 3], 2, 3), P


def _inv2(M):
    det = M[0, 0] * M[1, 1] - M[0, 1] * M[1, 0]
    return np.array([[M[1, 1], -M[0, 1]], [-M[1, 0], M[0, 0]]]) / det


def run_evaluator_toy_tests() -> Dict[str, Any]:
    ev, P = _toy_mdp()
    pi = np.array([[0.6, 0.4], [0.5, 0.5], [1.0, 0.0], [1.0, 0.0]])
    # closed forms
    Ptt_mean = np.array([0.6 * P[0, 0, :2] + 0.4 * P[0, 1, :2], 0.5 * P[1, 0, :2] + 0.5 * P[1, 1, :2]])
    r_mean = np.array([P[0, 0, 3], 0.5 * P[1, 0, 3] + 0.5 * P[1, 1, 3]])
    inv_m = _inv2(np.eye(2) - Ptt_mean)
    J_mean = (inv_m @ r_mean)[0]
    L_mean = (inv_m @ np.ones(2))[0]
    Ptt_term = np.array([0.6 * P[0, 0, :2], Ptt_mean[1]])
    r_term = np.array([0.6 * P[0, 0, 3], r_mean[1]])
    inv_t = _inv2(np.eye(2) - Ptt_term)
    J_term = (inv_t @ r_term)[0]
    p_any = (inv_t @ np.array([0.4, 0.0]))[0]
    em, et = ev.evaluate(pi, "mean"), ev.evaluate(pi, "terminate")
    # projection on the toy MDP
    proj = pi.copy()
    proj[0] = [1.0, 0.0]
    pm, pt = ev.evaluate(proj, "mean"), ev.evaluate(proj, "terminate")
    results = {
        "mean_return_error": abs(em["return"] - J_mean),
        "mean_length_error": abs(em["avg_episode_length"] - L_mean),
        "terminate_return_error": abs(et["return"] - J_term),
        "prob_any_violation_error_mean_rule": abs(em["prob_any_violation"] - p_any),
        "prob_any_violation_error_terminate_rule": abs(et["prob_any_violation"] - p_any),
        "projected_equals_original_mean_rule": abs(pm["return"] - em["return"]),
        "projected_equal_across_rules": abs(pm["return"] - pt["return"]),
        "projected_violations": pm["expected_violations"] + pt["expected_violations"],
        "closed_form_mean_return": J_mean,
        "closed_form_terminate_return": J_term,
    }
    for k in ["mean_return_error", "mean_length_error", "terminate_return_error", "prob_any_violation_error_mean_rule",
              "prob_any_violation_error_terminate_rule", "projected_equals_original_mean_rule",
              "projected_equal_across_rules", "projected_violations"]:
        assert results[k] < 1e-12, (k, results[k])
    # improper policy must raise
    ev_bad, _ = _toy_mdp(improper=True)
    try:
        ev_bad.evaluate(np.array([[1.0, 0.0], [1.0, 0.0], [1.0, 0.0], [1.0, 0.0]]), "mean")
        raise AssertionError("Improper policy was not rejected.")
    except ImproperPolicyError:
        results["improper_policy_rejected"] = True
    # malformed policies must raise; repair is opt-in and reported
    for label, bad in {"nan": np.array([[np.nan, 1.0], [0.5, 0.5], [1, 0], [1, 0]]),
                       "negative": np.array([[1.2, -0.2], [0.5, 0.5], [1, 0], [1, 0]]),
                       "empty_row": np.array([[0.0, 0.0], [0.5, 0.5], [1, 0], [1, 0]])}.items():
        try:
            ev.evaluate(bad, "mean")
            raise AssertionError(f"Malformed policy ({label}) was evaluated silently.")
        except PolicyValidationError:
            results[f"malformed_{label}_rejected"] = True
        rep = ev.evaluate(bad, "mean", repair=True)
        results[f"malformed_{label}_opt_in_repair_report"] = rep["repair_report"]
    return results


EVALUATOR_TOY_TESTS = run_evaluator_toy_tests()
print("Analytic evaluator tests passed:", json.dumps(EVALUATOR_TOY_TESTS, indent=1, default=_json_default))

# ---------- Benchmark evaluator (built only after every trainer above is defined) ----------
_EXACT = BENCH.exact_mdp()
EVALUATOR = ExactMDPEvaluator(_EXACT["tx_mat"], _EXACT["r_mat"], _EXACT["d_0"], SUPPORT_MASK,
                              BENCH.terminal_states, BENCH.death_state, BENCH.survival_state)
D0 = _EXACT["d_0"].copy()
del _EXACT
gc.collect()


def safe_evaluate(pi: np.ndarray, rule: str, evaluator: Optional["ExactMDPEvaluator"] = None, **kwargs) -> Optional[Dict[str, Any]]:
    """Exact evaluation that returns None (instead of raising) for improper policies; malformed policies still raise."""
    try:
        return (EVALUATOR if evaluator is None else evaluator).evaluate(pi, rule, **kwargs)
    except (ImproperPolicyError, EvaluationInvariantError) as exc:
        print(f"  evaluation skipped ({rule}): {exc}")
        return None


def evaluate_both(pi: np.ndarray, **kwargs) -> Dict[str, Dict[str, Any]]:
    return {rule: EVALUATOR.evaluate(pi, rule, **kwargs) for rule in ("mean", "terminate")}


def monte_carlo_estimate(pi: np.ndarray, strategy: str, n_episodes: int, key: str) -> Dict[str, float]:
    """Small Monte-Carlo cross-check through the real environment (evaluation RNG stream)."""
    env = gym.make(ICUSepsisBenchmark.ENV_ID, inadmissible_action_strategy=strategy)
    rng = np_rng("evaluation", "monte-carlo", key, strategy)
    cdf = np.cumsum(validate_policy(pi, key), axis=1)
    cdf[:, -1] = 1.0
    G = np.zeros(n_episodes)
    Ls = np.zeros(n_episodes)
    V = np.zeros(n_episodes)
    for ep in range(n_episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 2 ** 31 - 1)))
        for _t in range(10_000):
            a = int(np.searchsorted(cdf[s], rng.random(), side="right"))
            a = min(a, A_N - 1)
            V[ep] += float(not SUPPORT_MASK[s, a])
            s, r, term, trunc, _ = env.step(a)
            G[ep] += float(r)
            Ls[ep] += 1
            if term or trunc:
                break
    env.close()
    return {"mc_return": float(G.mean()), "mc_return_se": float(G.std(ddof=1) / math.sqrt(n_episodes)),
            "mc_length": float(Ls.mean()), "mc_length_se": float(Ls.std(ddof=1) / math.sqrt(n_episodes)),
            "mc_violations": float(V.mean()), "mc_violations_se": float(V.std(ddof=1) / math.sqrt(n_episodes)),
            "mc_episodes": n_episodes}


def env_semantics_tests(n_pairs: int = 40, n_draws: int = 1500, n_mean_pairs: int = 5) -> Dict[str, Any]:
    """Termination / mean semantics of the pinned environment versus the evaluator (audit M09).

    The test sets the environment's internal current state directly (test-only hook) after a reset.
    """
    rng = np_rng("evaluation", "env-semantics")
    pairs = [(int(s), int(a)) for s in np.flatnonzero(LIVE) for a in np.flatnonzero(~SUPPORT_MASK[s])]
    pick = [pairs[i] for i in rng.choice(len(pairs), size=min(n_pairs, len(pairs)), replace=False)]
    env_t = gym.make(ICUSepsisBenchmark.ENV_ID, inadmissible_action_strategy="terminate")
    term_ok = []
    for s, a in pick:
        env_t.reset(seed=int(rng.integers(0, 2 ** 31 - 1)))
        env_t.unwrapped._current_state = s
        ns, r, term, trunc, _ = env_t.step(a)
        one_hot = np.zeros((S_N, A_N))
        one_hot[:, 0] = 1.0
        one_hot[s] = 0.0
        one_hot[s, a] = 1.0
        P_row = EVALUATOR.induced(one_hot, "terminate")[0][s]
        term_ok.append(int(ns) == BENCH.death_state and float(r) == 0.0 and bool(term)
                       and abs(P_row[BENCH.death_state] - 1.0) < 1e-12)
    env_t.close()
    env_m = gym.make(ICUSepsisBenchmark.ENV_ID, inadmissible_action_strategy="mean")
    mean_rows = []
    for s, a in pick[:n_mean_pairs]:
        counts = np.zeros(S_N)
        for _ in range(n_draws):
            env_m.reset(seed=int(rng.integers(0, 2 ** 31 - 1)))
            env_m.unwrapped._current_state = s
            ns, *_ = env_m.step(a)
            counts[int(ns)] += 1
        emp = counts / n_draws
        p = EVALUATOR.P[s, a]
        bound = 5.0 * np.sqrt(p * (1 - p) / n_draws) + 2e-3
        mean_rows.append({"state": s, "action": a, "max_abs_dev": float(np.abs(emp - p).max()),
                          "within_bounds": bool((np.abs(emp - p) <= bound).all())})
    env_m.close()
    out = {"terminate_pairs_tested": len(term_ok), "terminate_all_consistent": bool(all(term_ok)),
           "mean_rule_rows": mean_rows, "mean_rule_all_within_bounds": bool(all(r["within_bounds"] for r in mean_rows))}
    return out


print("Exact evaluator ready (transition model reserved for evaluation, oracles and stress tests).")

## 9. Tabular learners: BC, tabular CQL, tabular IQL (adaptation), tabular discrete BCQ (adaptation), and dataset-estimated support

* **Tabular IQL** is an adaptation of IQL (expectile value regression + advantage-weighted extraction
  from logged action counts), not a reproduction of every detail of neural IQL.
* **Tabular discrete BCQ** follows the discrete-BCQ rule of Fujimoto et al. (2019): actions whose
  behaviour probability relative to the most likely action is below τ (= 0.3) are excluded from both the
  backup and the extraction. The behaviour model here is the count-based maximum-likelihood estimate (the
  tabular analogue of BCQ's learned behaviour network). It is listed separately from simple
  **count-threshold support masks**, which are *not* BCQ.
* Every learner returns its native export plus explicitly labelled masked-greedy variants; the
  admissible projection of every policy is added generically in Section 19.

In [ ]:
# ============================================================
# 9. Tabular learners and dataset-estimated support
# ============================================================
EXECLOG.mark("9. Tabular learners")

COST_ROWS: List[dict] = []   # training / construction cost accounting (audit M06)


def tabular_identity(algo: str, hp: dict, seed: int, bundle: DataBundle, code: str) -> dict:
    return {"algo": algo, "hparams": hp, "seed": int(seed), "run_mode": RUN_MODE, "data": bundle.identity(),
            "benchmark": BENCH.identity, "code": code}


def _tabular_tensors(bundle: DataBundle):
    d = bundle.data
    t = lambda a, dt: torch.as_tensor(np.asarray(a), dtype=dt, device=DEVICE)
    return (t(d["state"], torch.long), t(d["action"], torch.long), t(d["reward"], torch.float32),
            t(d["next_state"], torch.long), t(d["terminal"], torch.float32), len(d["state"]))


def train_behavior_cloning(bundle: DataBundle) -> Dict[str, np.ndarray]:
    """Empirical (maximum-likelihood) behaviour cloning; unseen states use the global action distribution."""
    counts = bundle.counts.astype(np.float64)
    global_pi = counts.sum(axis=0) / max(counts.sum(), 1.0)
    pi = np.tile(global_pi, (S_N, 1))
    seen = counts.sum(axis=1) > 0
    pi[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
    native = one_hot_policy(pi.argmax(axis=1))
    # Masked greedy: most frequent logged admissible action; states without any logged admissible action
    # fall back to the global logged-action frequency over admissible actions (not to the lowest index).
    adm_seen = (counts * OPERATIONAL_MASK).sum(axis=1) > 0
    masked = one_hot_policy(masked_argmax(np.where(adm_seen[:, None], counts, global_pi[None, :]), OPERATIONAL_MASK))
    return {"stochastic": pi, "deterministic": native, "masked_greedy": masked, "counts": counts,
            "masked_fallback_states": int((~adm_seen & LIVE).sum())}


def train_tabular_cql(bundle: DataBundle, seed: int, alpha: float = None, steps: int = None,
                      lr: float = None, tag: str = "tabular_cql") -> Dict[str, Any]:
    alpha = CFG.tab_cql_alpha if alpha is None else alpha
    steps = CFG.tab_cql_steps if steps is None else steps
    lr = CFG.tab_cql_lr if lr is None else lr
    hp = {"alpha": alpha, "steps": steps, "lr": lr, "batch": CFG.batch_size, "gamma": CFG.gamma,
          "temperature": CFG.cql_temperature, "tau": CFG.target_tau, "grad_clip": CFG.grad_clip}
    ident = tabular_identity("tabular_cql", hp, seed, bundle, code_hash(train_tabular_cql, _tabular_tensors, stream_seed, torch_generator))
    label = f"{tag}/{bundle.name}/seed{seed}"
    payload = STORE.load_final(label, ident)
    if payload is not None:
        return {"q": payload["q"].numpy(), "log": pd.DataFrame(payload["log"]), "meta": payload["meta"]}
    STORE.require_training_allowed(label)
    t0 = time.perf_counter()
    s_all, a_all, r_all, ns_all, d_all, n = _tabular_tensors(bundle)
    q = nn.Parameter(torch.zeros(S_N, A_N, device=DEVICE))
    target_q = torch.zeros_like(q.data)
    opt = torch.optim.Adam([q], lr=lr)
    gen = torch_generator(stream_seed("minibatch", bundle.data_hash, seed), DEVICE)
    rows = []
    for step in trange(steps, desc=f"Tabular CQL {bundle.name} s{seed}", leave=False, disable=not SHOW_PROGRESS):
        idx = torch.randint(0, n, (CFG.batch_size,), generator=gen, device=DEVICE)
        s, a, r, ns, d = s_all[idx], a_all[idx], r_all[idx], ns_all[idx], d_all[idx]
        q_sa = q[s, a]
        with torch.no_grad():
            target = r + CFG.gamma * (1.0 - d) * target_q[ns].max(dim=1).values
        td = F.mse_loss(q_sa, target)
        cons = (torch.logsumexp(q[s] / CFG.cql_temperature, dim=1) * CFG.cql_temperature - q_sa).mean()
        loss = td + alpha * cons
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([q], CFG.grad_clip)
        opt.step()
        with torch.no_grad():
            target_q.mul_(1.0 - CFG.target_tau).add_(q.data, alpha=CFG.target_tau)
        if step % CFG.log_every == 0 or step == steps - 1:
            rows.append({"step": step + 1, "loss": float(loss), "td_loss": float(td), "cql_loss": float(cons)})
    q_np = q.detach().cpu().numpy().astype(np.float64)
    meta = {"elapsed_s": time.perf_counter() - t0, "params": S_N * A_N, "steps": steps}
    STORE.save_final(label, ident, {"q": torch.from_numpy(q_np), "log": rows, "meta": meta})
    COST_ROWS.append({"model": "Tabular CQL", "label": label, "seed": seed, "train_s": meta["elapsed_s"],
                      "params": meta["params"], "steps": steps})
    return {"q": q_np, "log": pd.DataFrame(rows), "meta": meta}


def expectile_loss(diff: torch.Tensor, expectile: float) -> torch.Tensor:
    weight = torch.where(diff > 0, expectile, 1.0 - expectile)
    return (weight * diff.square()).mean()


def iql_extraction(q: np.ndarray, v: np.ndarray, counts: np.ndarray, beta: float, max_weight: float):
    """Advantage-weighted extraction from logged counts. Returns AW policy, weighted counts and masked greedy."""
    adv = q - v[:, None]
    weights = np.exp(np.clip(beta * adv, -20.0, math.log(max_weight)))
    weighted = counts * weights
    global_pi = counts.sum(axis=0) / max(counts.sum(), 1.0)
    pi = np.tile(global_pi, (S_N, 1))
    seen = weighted.sum(axis=1) > 0
    pi[seen] = weighted[seen] / weighted[seen].sum(axis=1, keepdims=True)
    # Deterministic masked IQL: argmax of advantage-weighted scores over admissible actions. Q is never
    # updated for unlogged pairs (it stays exactly 0), so states without a logged admissible action fall back
    # to the global logged-action frequency over admissible actions instead of an arbitrary lowest index.
    adm_w = np.where(OPERATIONAL_MASK, weighted, -np.inf)
    no_logged_adm = ~(np.where(OPERATIONAL_MASK, weighted, 0.0) > 0).any(axis=1)
    masked_actions = adm_w.argmax(axis=1)
    masked_actions[no_logged_adm] = masked_argmax(np.broadcast_to(global_pi, q.shape), OPERATIONAL_MASK)[no_logged_adm]
    return pi, weighted, one_hot_policy(masked_actions), int((no_logged_adm & LIVE).sum())


def train_tabular_iql(bundle: DataBundle, seed: int, expectile: float = None, beta: float = None,
                      steps: int = None, lr: float = None, tag: str = "tabular_iql") -> Dict[str, Any]:
    expectile = CFG.iql_expectile if expectile is None else expectile
    beta = CFG.iql_beta if beta is None else beta
    steps = CFG.iql_steps if steps is None else steps
    lr = CFG.iql_lr if lr is None else lr
    hp = {"expectile": expectile, "beta": beta, "steps": steps, "lr": lr, "batch": CFG.batch_size,
          "gamma": CFG.gamma, "max_weight": CFG.iql_max_weight, "grad_clip": CFG.grad_clip}
    ident = tabular_identity("tabular_iql", hp, seed, bundle,
                             code_hash(train_tabular_iql, expectile_loss, iql_extraction, _tabular_tensors,
                                       stream_seed, torch_generator))
    label = f"{tag}/{bundle.name}/seed{seed}"
    payload = STORE.load_final(label, ident)
    if payload is None:
        STORE.require_training_allowed(label)
        t0 = time.perf_counter()
        s_all, a_all, r_all, ns_all, d_all, n = _tabular_tensors(bundle)
        q = nn.Parameter(torch.zeros(S_N, A_N, device=DEVICE))
        v = nn.Parameter(torch.zeros(S_N, device=DEVICE))
        opt_q = torch.optim.Adam([q], lr=lr)
        opt_v = torch.optim.Adam([v], lr=lr)
        gen = torch_generator(stream_seed("minibatch", bundle.data_hash, seed), DEVICE)
        rows = []
        for step in trange(steps, desc=f"Tabular IQL {bundle.name} s{seed}", leave=False, disable=not SHOW_PROGRESS):
            idx = torch.randint(0, n, (CFG.batch_size,), generator=gen, device=DEVICE)
            s, a, r, ns, d = s_all[idx], a_all[idx], r_all[idx], ns_all[idx], d_all[idx]
            v_loss = expectile_loss(q[s, a].detach() - v[s], expectile)
            opt_v.zero_grad(set_to_none=True)
            v_loss.backward()
            opt_v.step()
            with torch.no_grad():
                target = r + CFG.gamma * (1.0 - d) * v[ns]
            q_loss = F.mse_loss(q[s, a], target)
            opt_q.zero_grad(set_to_none=True)
            q_loss.backward()
            torch.nn.utils.clip_grad_norm_([q], CFG.grad_clip)
            opt_q.step()
            if step % CFG.log_every == 0 or step == steps - 1:
                rows.append({"step": step + 1, "q_loss": float(q_loss), "v_loss": float(v_loss)})
        q_np = q.detach().cpu().numpy().astype(np.float64)
        v_np = v.detach().cpu().numpy().astype(np.float64)
        meta = {"elapsed_s": time.perf_counter() - t0, "params": S_N * A_N + S_N, "steps": steps}
        payload = {"q": torch.from_numpy(q_np), "v": torch.from_numpy(v_np), "log": rows, "meta": meta}
        STORE.save_final(label, ident, payload)
        COST_ROWS.append({"model": "Tabular IQL", "label": label, "seed": seed, "train_s": meta["elapsed_s"],
                          "params": meta["params"], "steps": steps})
    q_np, v_np = payload["q"].numpy(), payload["v"].numpy()
    pi, weighted, masked, n_fallback = iql_extraction(q_np, v_np, bundle.counts.astype(np.float64), beta, CFG.iql_max_weight)
    return {"q": q_np, "v": v_np, "aw_policy": pi, "aw_scores": weighted, "deterministic": one_hot_policy(pi.argmax(axis=1)),
            "masked_greedy": masked, "masked_fallback_states": n_fallback, "log": pd.DataFrame(payload["log"]),
            "meta": payload["meta"]}


def behavior_threshold_set(counts: np.ndarray, tau: float) -> np.ndarray:
    counts = counts.astype(np.float64)
    global_pi = counts.sum(axis=0) / max(counts.sum(), 1.0)
    pb = np.tile(global_pi, (S_N, 1))
    seen = counts.sum(axis=1) > 0
    pb[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
    allowed = pb / np.maximum(pb.max(axis=1, keepdims=True), 1e-30) >= tau
    allowed[~LIVE] = True
    return allowed


def train_tabular_bcq(bundle: DataBundle, seed: int, tau: float = None, steps: int = None, lr: float = None,
                      tag: str = "tabular_bcq") -> Dict[str, Any]:
    tau = CFG.bcq_threshold if tau is None else tau
    steps = CFG.bcq_steps if steps is None else steps
    lr = CFG.bcq_lr if lr is None else lr
    hp = {"tau": tau, "steps": steps, "lr": lr, "batch": CFG.batch_size, "gamma": CFG.gamma,
          "target_tau": CFG.target_tau, "grad_clip": CFG.grad_clip}
    ident = tabular_identity("tabular_bcq", hp, seed, bundle,
                             code_hash(train_tabular_bcq, behavior_threshold_set, _tabular_tensors,
                                       stream_seed, torch_generator))
    label = f"{tag}/{bundle.name}/seed{seed}"
    G = behavior_threshold_set(bundle.counts, tau)
    payload = STORE.load_final(label, ident)
    if payload is None:
        STORE.require_training_allowed(label)
        t0 = time.perf_counter()
        s_all, a_all, r_all, ns_all, d_all, n = _tabular_tensors(bundle)
        G_t = torch.as_tensor(G, device=DEVICE)
        q = nn.Parameter(torch.zeros(S_N, A_N, device=DEVICE))
        target_q = torch.zeros_like(q.data)
        opt = torch.optim.Adam([q], lr=lr)
        gen = torch_generator(stream_seed("minibatch", bundle.data_hash, seed), DEVICE)
        rows = []
        for step in trange(steps, desc=f"Tabular BCQ {bundle.name} s{seed}", leave=False, disable=not SHOW_PROGRESS):
            idx = torch.randint(0, n, (CFG.batch_size,), generator=gen, device=DEVICE)
            s, a, r, ns, d = s_all[idx], a_all[idx], r_all[idx], ns_all[idx], d_all[idx]
            with torch.no_grad():
                next_a = q[ns].masked_fill(~G_t[ns], -torch.inf).argmax(dim=1)
                target = r + CFG.gamma * (1.0 - d) * target_q[ns].gather(1, next_a[:, None]).squeeze(1)
            loss = F.mse_loss(q[s, a], target)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([q], CFG.grad_clip)
            opt.step()
            with torch.no_grad():
                target_q.mul_(1.0 - CFG.target_tau).add_(q.data, alpha=CFG.target_tau)
            if step % CFG.log_every == 0 or step == steps - 1:
                rows.append({"step": step + 1, "td_loss": float(loss)})
        q_np = q.detach().cpu().numpy().astype(np.float64)
        meta = {"elapsed_s": time.perf_counter() - t0, "params": S_N * A_N, "steps": steps}
        payload = {"q": torch.from_numpy(q_np), "log": rows, "meta": meta}
        STORE.save_final(label, ident, payload)
        COST_ROWS.append({"model": "Tabular BCQ", "label": label, "seed": seed, "train_s": meta["elapsed_s"],
                          "params": meta["params"], "steps": steps})
    q_np = payload["q"].numpy()
    native = one_hot_policy(masked_argmax(q_np, G))
    both = G & OPERATIONAL_MASK
    both_ok = both.any(axis=1)
    masked_mask = np.where(both_ok[:, None], both, OPERATIONAL_MASK)
    return {"q": q_np, "allowed": G, "deterministic": native,
            "masked_greedy": one_hot_policy(masked_argmax(q_np, masked_mask)),
            "masked_fallback_states": int((~both_ok & LIVE).sum()),
            "log": pd.DataFrame(payload["log"]), "meta": payload["meta"]}


# ---------- Dataset-estimated support (audit M02) ----------
def estimated_support_mask(counts: np.ndarray, rule: str, param: float) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Support estimated from the simulated training data — a different object from the released mask."""
    counts = np.asarray(counts)
    if rule == "count_at_least":
        est = counts >= param
    elif rule == "behavior_ratio_at_least":
        est = behavior_threshold_set(counts, param)
    else:
        raise ValueError(rule)
    est = est.copy()
    est[~LIVE] = True
    empty = LIVE & ~est.any(axis=1)
    for s in np.flatnonzero(empty):   # fallback: most frequently logged action (or all actions if unseen)
        if counts[s].sum() > 0:
            est[s, int(counts[s].argmax())] = True
        else:
            est[s] = True
    rel = SUPPORT_MASK[LIVE]
    e = est[LIVE]
    inter = (rel & e).sum(axis=1)
    union = (rel | e).sum(axis=1)
    report = {
        "rule": rule, "param": param,
        "estimated_pairs_live": int(e.sum()),
        "released_pairs_live": int(rel.sum()),
        "false_exclusions": int((rel & ~e).sum()),
        "false_inclusions": int((~rel & e).sum()),
        "empty_rows_before_fallback": int(empty.sum()),
        "fallback_rows": int(empty.sum()),
        "mean_state_jaccard": float(np.mean(inter / np.maximum(union, 1))),
        "states_identical_to_released": int((e == rel).all(axis=1).sum()),
    }
    return est, report


print("Tabular learners ready.")

## 10. Twin-dueling critics: Deep CQL, legacy hard AM-CQL, two repaired hard designs, soft AM-CQL (audit C02, M05)

All variants share one trainer parameterised by a `CriticSpec`, the same architecture, the same
initial weights for a given seed, and the same minibatch index sequence (common random numbers).

| Variant | CQL regulariser domain | Backup | Extraction | Status |
|---|---|---|---|---|
| Deep CQL | full action set | unmasked | unmasked min-critic greedy | baseline |
| AM-CQL (Hard, legacy) | log-sum-exp over admissible actions **minus the unmasked logged-action Q** (incompatible domain: for an excluded logged action the regulariser's derivative w.r.t. its Q is −1) | masked | masked | **failure-mode control only** |
| AM-CQL (Hard, design A: full-action CQL) | full action set (standard CQL) | masked | masked | repaired comparator |
| AM-CQL (Hard, design B: admissible-row CQL) | admissible log-sum-exp, applied **only to rows whose logged action is admissible**; normalisation = mean over those rows (`conditional`) or sum / batch size (`batch`) | masked | masked | repaired, preregistered primary critic |
| AM-CQL (Soft, legacy) | full | unmasked | unmasked min-critic greedy (training penalty uses the mean-critic softmax) | analysed in Section 24 |

Additional switches used by the ablations: TD rows (`all` vs `admissible_only` — inadmissible logged
transitions carry mean-imputed simulator outcomes, not observations of the forbidden intervention),
TD loss (Huber vs squared), conservative weight α, a separately labelled TD-target clipping ablation
(a stabilisation choice, **not** a repair of the regulariser), a separately labelled discounted
(γ = 0.99, different estimand) ablation, and feature removal (Section 29).

Diagnostics logged during training: TD / regulariser / total loss, gradient norm (pre-clipping), TD-target
mean/min/max, batch admissible-Q quantiles and absolute maximum, within-state admissible action spread,
and every `diag_every` steps the **full-buffer Bellman residual** and full Q-table statistics, including the
ratio of the common per-state offset to the action spread and the number of states whose action
differences are within 64 float32 ulps of the offset (where huge offsets could obscure action differences).

In [ ]:
# ============================================================
# 10. Neural critics
# ============================================================
EXECLOG.mark("10. Neural critics")


class DuelingQNetwork(nn.Module):
    def __init__(self, input_dim: int, num_actions: int, hidden_dim: int = 256):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(),
        )
        self.value = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Linear(hidden_dim // 2, 1))
        self.advantage = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
                                       nn.Linear(hidden_dim // 2, num_actions))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.trunk(x)
        a = self.advantage(h)
        return self.value(h) + a - a.mean(dim=1, keepdim=True)


class TwinDuelingCQL(nn.Module):
    def __init__(self, input_dim: int, num_actions: int, hidden_dim: int = 256):
        super().__init__()
        self.q1 = DuelingQNetwork(input_dim, num_actions, hidden_dim)
        self.q2 = DuelingQNetwork(input_dim, num_actions, hidden_dim)
        self.tq1 = copy.deepcopy(self.q1)
        self.tq2 = copy.deepcopy(self.q2)
        for p in list(self.tq1.parameters()) + list(self.tq2.parameters()):
            p.requires_grad_(False)

    def online(self, x):
        return self.q1(x), self.q2(x)

    @torch.no_grad()
    def target(self, x):
        return self.tq1(x), self.tq2(x)


@torch.no_grad()
def polyak_update(target: nn.Module, source: nn.Module, tau: float) -> None:
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.mul_(1.0 - tau).add_(sp.data, alpha=tau)


def apply_hard_mask(q: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Finite large-negative masking (legacy behaviour); every row of the operational mask is non-empty."""
    return q.masked_fill(~mask, torch.finfo(q.dtype).min)


@dataclass(frozen=True)
class CriticSpec:
    key: str
    display: str
    reg_domain: str = "full"                 # full | masked_legacy | masked_admissible_rows
    reg_normalization: str = "conditional"   # conditional | batch   (masked_admissible_rows only)
    masked_backup: bool = False
    masked_extraction: bool = False
    td_loss: str = "huber"                   # huber | mse
    td_rows: str = "all"                     # all | admissible_only
    alpha: float = 1.0
    temperature: float = 1.0
    target_clip: Optional[Tuple[float, float]] = None
    gamma: float = 1.0
    soft: bool = False
    soft_tolerance: float = 0.0
    soft_temperature: float = 1.0
    soft_aggregation: str = "mean"           # mean | min
    steps: int = 0
    status: str = ""

    def hparams(self) -> dict:
        d = asdict(self)
        d.pop("display")
        d.pop("status")
        d.update(lr=CFG.lr, weight_decay=CFG.weight_decay, batch=CFG.batch_size, hidden=CFG.hidden_dim,
                 grad_clip=CFG.grad_clip, target_tau=CFG.target_tau,
                 soft_initial_lambda=CFG.soft_initial_lambda, soft_dual_lr=CFG.soft_dual_lr,
                 soft_lambda_max=CFG.soft_lambda_max)
        return d


def make_spec(key: str, display: str, **kw) -> CriticSpec:
    kw.setdefault("steps", CFG.deep_steps)
    kw.setdefault("alpha", CFG.cql_alpha)
    kw.setdefault("temperature", CFG.cql_temperature)
    kw.setdefault("gamma", CFG.gamma)
    return CriticSpec(key=key, display=display, **kw)


SPEC_DEEP = make_spec("deep", "Deep CQL", status="baseline")
SPEC_HARD_LEGACY = make_spec("hard_legacy", "AM-CQL (Hard, legacy)", reg_domain="masked_legacy",
                             masked_backup=True, masked_extraction=True, status="failure-mode control")
SPEC_HARD_A = make_spec("hard_fullcql", "AM-CQL (Hard, design A: full-action CQL)", reg_domain="full",
                        masked_backup=True, masked_extraction=True, status="repaired comparator")
SPEC_HARD_B = make_spec("hard_admrows", "AM-CQL (Hard, design B: admissible-row CQL)",
                        reg_domain="masked_admissible_rows", reg_normalization="conditional",
                        masked_backup=True, masked_extraction=True, status="repaired (preregistered primary)")
SPEC_SOFT = make_spec("soft_legacy", "AM-CQL (Soft, legacy)", soft=True,
                      soft_tolerance=CFG.soft_target_inadmissible_mass,
                      soft_temperature=CFG.soft_policy_temperature, status="soft surrogate")
MAIN_CRITIC_SPECS = [SPEC_DEEP, SPEC_HARD_LEGACY, SPEC_HARD_A, SPEC_HARD_B, SPEC_SOFT]
SPEC_BY_KEY = {s.key: s for s in MAIN_CRITIC_SPECS}
assert PREREG["primary_hard_critic"] in SPEC_BY_KEY


def cql_regularizer(q: torch.Tensor, a: torch.Tensor, mask: torch.Tensor, adm: torch.Tensor,
                    spec: CriticSpec) -> torch.Tensor:
    T = spec.temperature
    assert T > 0
    q_log = q.gather(1, a[:, None]).squeeze(1)
    if spec.reg_domain == "full":
        return (T * torch.logsumexp(q / T, dim=1) - q_log).mean()
    if spec.reg_domain == "masked_legacy":
        # Original (defective) objective, retained only as a labelled failure-mode control.
        return (T * torch.logsumexp(apply_hard_mask(q, mask) / T, dim=1) - q_log).mean()
    if spec.reg_domain == "masked_admissible_rows":
        if not bool(adm.any()):
            return q.sum() * 0.0
        q_ok, m_ok = q[adm], mask[adm]
        assert bool(m_ok.any(dim=1).all())
        lse = T * torch.logsumexp(q_ok.masked_fill(~m_ok, -torch.inf) / T, dim=1)
        contrib = lse - q_log[adm]
        if spec.reg_normalization == "conditional":
            return contrib.mean()
        if spec.reg_normalization == "batch":
            return contrib.sum() / q.shape[0]
        raise ValueError(spec.reg_normalization)
    raise ValueError(spec.reg_domain)


def td_elementwise(pred: torch.Tensor, target: torch.Tensor, kind: str) -> torch.Tensor:
    if kind == "huber":
        return F.smooth_l1_loss(pred, target, reduction="none")
    if kind == "mse":
        return F.mse_loss(pred, target, reduction="none")
    raise ValueError(kind)


def soft_policy_logits(q1: torch.Tensor, q2: torch.Tensor, aggregation: str) -> torch.Tensor:
    return 0.5 * (q1 + q2) if aggregation == "mean" else torch.minimum(q1, q2)


def inadmissible_probability_mass(logits: torch.Tensor, mask: torch.Tensor, temperature: float) -> torch.Tensor:
    return (torch.softmax(logits / temperature, dim=1) * (~mask).float()).sum(dim=1)


@torch.no_grad()
def critic_tables(model: TwinDuelingCQL, all_x: np.ndarray, batch: int = 2048,
                  device: Optional[torch.device] = None) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    x = torch.as_tensor(all_x, dtype=torch.float32, device=DEVICE if device is None else device)
    q1s, q2s = [], []
    for i in range(0, len(x), batch):
        a, b = model.online(x[i:i + batch])
        q1s.append(a.double().cpu())
        q2s.append(b.double().cpu())
    return torch.cat(q1s).numpy(), torch.cat(q2s).numpy()


@torch.no_grad()
def full_diagnostics(model: TwinDuelingCQL, buf: OfflineTensorBuffer, bundle: DataBundle, spec: CriticSpec,
                     step: int) -> dict:
    q1, q2 = critic_tables(model, bundle.all_x)
    qmin = np.minimum(q1, q2)[LIVE]
    m = SUPPORT_MASK[LIVE]
    adm_vals = qmin[m]
    q_adm = np.where(m, qmin, np.nan)
    offset = np.nanmean(q_adm, axis=1)
    spread = np.nanmax(q_adm, axis=1) - np.nanmin(q_adm, axis=1)
    multi = m.sum(axis=1) >= 2
    eps32 = float(np.finfo(np.float32).eps)
    near_res = multi & (spread <= 64 * eps32 * np.maximum(np.abs(offset), 1.0))
    sq, n_rows = 0.0, 0
    abs_sum = 0.0
    sq_adm, n_adm, sq_in, n_in = 0.0, 0, 0.0, 0
    model.eval()
    for b in buf.chunks():
        o1, o2 = model.online(b["x"])
        q_sa = torch.minimum(o1, o2).gather(1, b["a"][:, None]).squeeze(1)
        n_on = 0.5 * sum(model.online(b["nx"]))
        if spec.masked_backup:
            n_on = apply_hard_mask(n_on, b["next_mask"])
        na = n_on.argmax(dim=1)
        t1, t2 = model.target(b["nx"])
        tq = torch.minimum(t1, t2)
        if spec.masked_backup:
            tq = apply_hard_mask(tq, b["next_mask"])
        y = b["r"] + spec.gamma * (1.0 - b["terminal"]) * tq.gather(1, na[:, None]).squeeze(1)
        diff = (q_sa - y).double()
        d2 = diff ** 2
        sq += float(d2.sum())
        abs_sum += float(diff.abs().sum())
        n_rows += int(diff.numel())
        adm = b["adm"]
        sq_adm += float(d2[adm].sum())
        n_adm += int(adm.sum())
        sq_in += float(d2[~adm].sum())
        n_in += int((~adm).sum())
    model.train()
    td_only_adm = spec.td_rows == "admissible_only"
    return {
        "step": step,
        "bellman_residual_rmse_full_buffer": math.sqrt(sq / max(n_rows, 1)),
        "bellman_residual_rmse_adm_logged_rows": math.sqrt(sq_adm / max(n_adm, 1)),
        "bellman_residual_rmse_inadm_logged_rows": math.sqrt(sq_in / n_in) if n_in else float("nan"),
        "bellman_residual_rmse_td_objective_rows": (math.sqrt(sq_adm / max(n_adm, 1)) if td_only_adm
                                                    else math.sqrt(sq / max(n_rows, 1))),
        "bellman_residual_mae_full_buffer": abs_sum / max(n_rows, 1),
        "q_adm_max_abs": float(np.abs(adm_vals).max()),
        "q_adm_min": float(adm_vals.min()),
        "q_adm_max": float(adm_vals.max()),
        "q_adm_q01": float(np.quantile(adm_vals, 0.01)),
        "q_adm_q50": float(np.quantile(adm_vals, 0.50)),
        "q_adm_q99": float(np.quantile(adm_vals, 0.99)),
        "q_inadm_max": float(qmin[~m].max()) if (~m).any() else float("nan"),
        "mean_state_spread": float(spread[multi].mean()) if multi.any() else 0.0,
        "median_offset_to_spread_ratio": float(np.median(np.abs(offset[multi]) / np.maximum(spread[multi], 1e-30))) if multi.any() else 0.0,
        "states_spread_within_64_ulp_of_offset": int(near_res.sum()),
        "critic_disagreement_frac": float((q1[LIVE].argmax(1) != q2[LIVE].argmax(1)).mean()),
    }


CRITIC_CODE_HASH = None  # set after all helper definitions below


def critic_identity(spec: CriticSpec, seed: int, bundle: DataBundle) -> dict:
    return {"algo": "twin_dueling_critic", "spec": spec.hparams(), "seed": int(seed), "run_mode": RUN_MODE,
            "data": bundle.identity(), "benchmark": BENCH.identity, "code": CRITIC_CODE_HASH}


def train_critic(spec: CriticSpec, bundle: DataBundle, seed: int, tag: str = "critic",
                 interrupt_at: Optional[int] = None, store: Optional[CheckpointStore] = None) -> Optional[Dict[str, Any]]:
    """Train (or load) one critic. `interrupt_at` simulates a disconnect for the resume test."""
    store = STORE if store is None else store
    ident = critic_identity(spec, seed, bundle)
    label = f"{tag}/{spec.key}/{bundle.name}/seed{seed}"
    payload = store.load_final(label, ident)
    if payload is None:
        store.require_training_allowed(label)
        payload = _train_critic_loop(spec, bundle, seed, ident, label, store, interrupt_at)
        if payload is None:
            return None
    return {"spec": spec, "seed": seed, "bundle": bundle.name, "q1": payload["q1"].numpy(),
            "q2": payload["q2"].numpy(), "log": pd.DataFrame(payload["log"]),
            "diag": pd.DataFrame(payload["diag"]), "meta": payload["meta"], "label": label}


def _train_critic_loop(spec, bundle, seed, ident, label, store, interrupt_at):
    t0 = time.perf_counter()
    reset_peak_gpu_memory()
    buf = bundle.buffer()
    model = init_model(lambda: TwinDuelingCQL(bundle.input_dim, A_N, CFG.hidden_dim),
                       f"twin_dueling_{bundle.input_dim}x{CFG.hidden_dim}", seed)
    params = list(model.q1.parameters()) + list(model.q2.parameters())
    opt = torch.optim.AdamW(params, lr=CFG.lr, weight_decay=CFG.weight_decay)
    gen = torch_generator(stream_seed("minibatch", bundle.data_hash, seed), DEVICE)
    lam = torch.tensor(CFG.soft_initial_lambda if spec.soft else 0.0, device=DEVICE)
    rows, diag_rows, start = [], [], 0
    prior_elapsed, prior_peak, segments = 0.0, 0.0, 1
    resume = store.load_resume(label, ident, map_location=DEVICE)
    if resume is not None:
        model.load_state_dict(resume["model"])
        opt.load_state_dict(resume["optimizer"])
        restore_generator(gen, resume["generator"])
        lam = resume["lambda"].to(device=DEVICE, dtype=torch.float32)
        rows, diag_rows, start = resume["log"], resume["diag"], int(resume["step"])
        prior_elapsed = float(resume.get("elapsed_s", 0.0))
        prior_peak = float(resume.get("peak_gpu_mb") or 0.0)
        segments = int(resume.get("segments", 1)) + 1
        print(f"  resumed {label} at step {start}")
    model.train()
    for step in trange(start, spec.steps, desc=f"{spec.key} {bundle.name} s{seed}", leave=False, disable=not SHOW_PROGRESS):
        b = buf.sample(CFG.batch_size, gen)
        q1, q2 = model.online(b["x"])
        q1_sa = q1.gather(1, b["a"][:, None]).squeeze(1)
        q2_sa = q2.gather(1, b["a"][:, None]).squeeze(1)
        with torch.no_grad():
            n1, n2 = model.online(b["nx"])
            nscore = 0.5 * (n1 + n2)
            if spec.masked_backup:
                nscore = apply_hard_mask(nscore, b["next_mask"])
            na = nscore.argmax(dim=1)
            t1, t2 = model.target(b["nx"])
            tq = torch.minimum(t1, t2)
            if spec.masked_backup:
                tq = apply_hard_mask(tq, b["next_mask"])
            y = b["r"] + spec.gamma * (1.0 - b["terminal"]) * tq.gather(1, na[:, None]).squeeze(1)
            if spec.target_clip is not None:
                y = y.clamp(spec.target_clip[0], spec.target_clip[1])
        w = b["adm"].float() if spec.td_rows == "admissible_only" else torch.ones_like(y)
        wsum = w.sum().clamp_min(1.0)
        td1 = (td_elementwise(q1_sa, y, spec.td_loss) * w).sum() / wsum
        td2 = (td_elementwise(q2_sa, y, spec.td_loss) * w).sum() / wsum
        reg1 = cql_regularizer(q1, b["a"], b["mask"], b["adm"], spec)
        reg2 = cql_regularizer(q2, b["a"], b["mask"], b["adm"], spec)
        loss = td1 + td2 + spec.alpha * (reg1 + reg2)
        mass = torch.zeros((), device=DEVICE)
        if spec.soft:
            logits = soft_policy_logits(q1, q2, spec.soft_aggregation)
            mass = inadmissible_probability_mass(logits, b["mask"], spec.soft_temperature).mean()
            loss = loss + lam.detach() * (mass - spec.soft_tolerance)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(params, CFG.grad_clip)
        opt.step()
        if spec.soft:
            with torch.no_grad():
                lam.add_(CFG.soft_dual_lr * (mass.detach() - spec.soft_tolerance)).clamp_(0.0, CFG.soft_lambda_max)
        polyak_update(model.tq1, model.q1, CFG.target_tau)
        polyak_update(model.tq2, model.q2, CFG.target_tau)

        if step % CFG.log_every == 0 or step == spec.steps - 1:
            with torch.no_grad():
                qm = torch.minimum(q1, q2)
                adm_q = qm[b["mask"]]
                spread = (qm.masked_fill(~b["mask"], -torch.inf).max(1).values
                          - qm.masked_fill(~b["mask"], torch.inf).min(1).values)
                qs = torch.quantile(adm_q.float(), torch.tensor([0.01, 0.5, 0.99], device=DEVICE))
            rows.append({
                "step": step + 1, "loss": float(loss), "td_loss": float(td1 + td2), "cql_loss": float(reg1 + reg2),
                "grad_norm": float(grad_norm), "inadmissible_mass_surrogate": float(mass),
                "primal_residual": float(mass) - spec.soft_tolerance if spec.soft else 0.0,
                "lambda": float(lam), "target_mean": float(y.mean()), "target_min": float(y.min()),
                "target_max": float(y.max()), "q_adm_q01": float(qs[0]), "q_adm_q50": float(qs[1]),
                "q_adm_q99": float(qs[2]), "q_adm_max_abs": float(adm_q.abs().max()),
                "mean_admissible_spread": float(spread.mean()), "frac_rows_admissible": float(b["adm"].float().mean()),
            })
        if (step + 1) % CFG.diag_every == 0 or step == spec.steps - 1:
            diag_rows.append(full_diagnostics(model, buf, bundle, spec, step + 1))
        if (step + 1) % CFG.resume_every == 0 and step + 1 < spec.steps:
            store.save_resume(label, ident, {"model": model.state_dict(), "optimizer": opt.state_dict(),
                                             "generator": generator_state(gen), "lambda": lam.detach().cpu(),
                                             "step": step + 1, "log": rows, "diag": diag_rows,
                                             "elapsed_s": prior_elapsed + (time.perf_counter() - t0),
                                             "peak_gpu_mb": max(prior_peak, peak_gpu_memory_mb() or 0.0),
                                             "segments": segments})
            if interrupt_at is not None and step + 1 >= interrupt_at:
                return None
    q1t, q2t = critic_tables(model, bundle.all_x)
    meta = {"elapsed_s": prior_elapsed + (time.perf_counter() - t0), "params": count_parameters(model),
            "peak_gpu_mb": (max(prior_peak, peak_gpu_memory_mb() or 0.0) if torch.cuda.is_available() else None),
            "steps": spec.steps, "final_lambda": float(lam), "normalizer": bundle.normalizer.to_dict(),
            "resumed": resume is not None, "segments": segments}
    payload = {"model": model.state_dict(), "q1": torch.from_numpy(q1t), "q2": torch.from_numpy(q2t),
               "log": rows, "diag": diag_rows, "meta": meta}
    store.save_final(label, ident, payload)
    COST_ROWS.append({"model": spec.display, "label": label, "seed": seed, "train_s": meta["elapsed_s"],
                      "params": meta["params"], "peak_gpu_mb": meta["peak_gpu_mb"], "steps": spec.steps})
    return payload


def load_critic_model(result: Dict[str, Any], bundle: DataBundle) -> TwinDuelingCQL:
    """Rebuild a critic network from its identity-validated checkpoint (used by artifact replay)."""
    ident = critic_identity(result["spec"], result["seed"], bundle)
    payload = STORE.load_final(result["label"], ident, map_location=DEVICE, respect_force=False)
    if payload is None:
        raise MissingArtifactError(result["label"])
    model = TwinDuelingCQL(bundle.input_dim, A_N, CFG.hidden_dim).to(DEVICE)
    model.load_state_dict(payload["model"])
    model.eval()
    return model


CRITIC_CODE_HASH = code_hash(DuelingQNetwork, TwinDuelingCQL, polyak_update, apply_hard_mask, CriticSpec, make_spec,
                             cql_regularizer, td_elementwise, soft_policy_logits, inadmissible_probability_mass,
                             critic_tables, full_diagnostics, train_critic, _train_critic_loop, init_model,
                             OfflineTensorBuffer, stream_seed, torch_generator, generator_state, restore_generator)


# ---------- Policy extraction from critic tables ----------
def critic_extractions(res: Dict[str, Any]) -> Dict[str, Tuple[np.ndarray, str, bool]]:
    """Return {suffix: (policy, extraction description, stochastic)} for one trained critic."""
    spec = res["spec"]
    q1, q2 = res["q1"], res["q2"]
    qmin, qmean = np.minimum(q1, q2), 0.5 * (q1 + q2)
    out = {"": (greedy_policy(qmin, masked=spec.masked_extraction),
                "masked min-critic greedy" if spec.masked_extraction else "unmasked min-critic greedy", False)}
    if not spec.masked_extraction:
        out["[masked greedy]"] = (greedy_policy(qmin, masked=True), "masked min-critic greedy (inference mask)", False)
    else:
        # Constraint-placement control: mask used in training only (backup / regulariser), unmasked export.
        out["[unmasked greedy]"] = (greedy_policy(qmin, masked=False),
                                    "unmasked min-critic greedy (mask used in training only)", False)
    if spec.soft:
        logits = soft_policy_logits(torch.from_numpy(q1), torch.from_numpy(q2), spec.soft_aggregation).numpy()
        z = logits / spec.soft_temperature
        z = z - z.max(axis=1, keepdims=True)
        sm = np.exp(z)
        sm /= sm.sum(axis=1, keepdims=True)
        out["[aligned greedy: mean critic]"] = (greedy_policy(qmean, masked=False),
                                                "unmasked greedy on the training aggregation (mean critic)", False)
        out["[aligned stochastic: training softmax]"] = (sm, "training softmax (mean critic, T_p), stochastic", True)
    return out


print("Neural critic code hash:", CRITIC_CODE_HASH[:16])

## 11. Direct categorical policies: matched-capacity neural BC and advantage-weighted categorical imitation (audit M06)

The action space has only 25 categories, so a direct categorical network can represent any per-state
action distribution without a denoising chain. These baselines use the same state encoder as the diffusion
denoiser, the same optimiser, learning rate, batch size, minibatch stream and number of updates as the
diffusion prior. The advantage-weighted variant reweights the cross-entropy by
`exp(clip(β·A(s,a), −20, log w_max))` (batch-normalised) with advantages from tabular IQL of the same seed.

In [ ]:
# ============================================================
# 11. Direct categorical policy networks
# ============================================================
EXECLOG.mark("11. Direct categorical policies")


class CategoricalPolicyNet(nn.Module):
    def __init__(self, state_dim: int, num_actions: int, hidden_dim: int):
        super().__init__()
        self.state_encoder = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
                                           nn.Linear(hidden_dim, hidden_dim), nn.SiLU())
        self.head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
                                  nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, num_actions))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.state_encoder(x))


POLICYNET_CODE_HASH = None


def train_policy_net(bundle: DataBundle, seed: int, mode: str = "bc", advantages: Optional[np.ndarray] = None,
                     tag: str = "policy_net", store: Optional[CheckpointStore] = None) -> Dict[str, Any]:
    assert mode in {"bc", "aw"}
    store = STORE if store is None else store
    hp = {"mode": mode, "steps": CFG.policy_net_steps, "lr": CFG.policy_net_lr, "batch": CFG.batch_size,
          "hidden": CFG.hidden_dim, "weight_decay": CFG.weight_decay, "grad_clip": CFG.grad_clip}
    if mode == "aw":
        hp.update(aw_beta=CFG.aw_beta, aw_max_weight=CFG.aw_max_weight, advantages_hash=array_hash(advantages))
    ident = {"algo": "categorical_policy_net", "hparams": hp, "seed": int(seed), "run_mode": RUN_MODE,
             "data": bundle.identity(), "benchmark": BENCH.identity, "code": POLICYNET_CODE_HASH}
    label = f"{tag}/{mode}/{bundle.name}/seed{seed}"
    payload = store.load_final(label, ident)
    if payload is None:
        store.require_training_allowed(label)
        t0 = time.perf_counter()
        reset_peak_gpu_memory()
        buf = bundle.buffer()
        model = init_model(lambda: CategoricalPolicyNet(bundle.input_dim, A_N, CFG.hidden_dim),
                           f"policy_net_{bundle.input_dim}x{CFG.hidden_dim}", seed)
        opt = torch.optim.AdamW(model.parameters(), lr=CFG.policy_net_lr, weight_decay=CFG.weight_decay)
        gen = torch_generator(stream_seed("minibatch", bundle.data_hash, seed), DEVICE)
        adv_t = None
        if mode == "aw":
            adv_t = torch.as_tensor(advantages, dtype=torch.float32, device=DEVICE)
        rows, start = [], 0
        prior_elapsed, prior_peak, segments = 0.0, 0.0, 1
        resume = store.load_resume(label, ident, map_location=DEVICE)
        if resume is not None:
            model.load_state_dict(resume["model"])
            opt.load_state_dict(resume["optimizer"])
            restore_generator(gen, resume["generator"])
            rows, start = resume["log"], int(resume["step"])
            prior_elapsed = float(resume.get("elapsed_s", 0.0))
            prior_peak = float(resume.get("peak_gpu_mb") or 0.0)
            segments = int(resume.get("segments", 1)) + 1
        model.train()
        for step in trange(start, CFG.policy_net_steps, desc=f"policy-net {mode} {bundle.name} s{seed}",
                           leave=False, disable=not SHOW_PROGRESS):
            b = buf.sample(CFG.batch_size, gen)
            logits = model(b["x"])
            ce = F.cross_entropy(logits, b["a"], reduction="none")
            if mode == "aw":
                w = torch.exp(torch.clamp(CFG.aw_beta * adv_t[b["s_idx"], b["a"]], -20.0, math.log(CFG.aw_max_weight)))
                w = w / w.mean()
                loss = (w * ce).mean()
            else:
                loss = ce.mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            opt.step()
            if step % CFG.log_every == 0 or step == CFG.policy_net_steps - 1:
                rows.append({"step": step + 1, "loss": float(loss),
                             "accuracy": float((logits.argmax(1) == b["a"]).float().mean())})
            if (step + 1) % CFG.resume_every == 0 and step + 1 < CFG.policy_net_steps:
                store.save_resume(label, ident, {"model": model.state_dict(), "optimizer": opt.state_dict(),
                                                 "generator": generator_state(gen), "step": step + 1, "log": rows,
                                                 "elapsed_s": prior_elapsed + (time.perf_counter() - t0),
                                                 "peak_gpu_mb": max(prior_peak, peak_gpu_memory_mb() or 0.0),
                                                 "segments": segments})
        model.eval()
        with torch.no_grad():
            probs = torch.softmax(model(torch.as_tensor(bundle.all_x, device=DEVICE)).double(), dim=1).cpu()
        meta = {"elapsed_s": prior_elapsed + (time.perf_counter() - t0), "params": count_parameters(model),
                "peak_gpu_mb": (max(prior_peak, peak_gpu_memory_mb() or 0.0) if torch.cuda.is_available() else None),
                "steps": CFG.policy_net_steps, "normalizer": bundle.normalizer.to_dict(),
                "resumed": resume is not None, "segments": segments}
        payload = {"model": model.state_dict(), "probs": probs, "log": rows, "meta": meta}
        store.save_final(label, ident, payload)
        COST_ROWS.append({"model": f"Neural categorical ({mode})", "label": label, "seed": seed,
                          "train_s": meta["elapsed_s"], "params": meta["params"],
                          "peak_gpu_mb": meta["peak_gpu_mb"], "steps": CFG.policy_net_steps})
    probs = payload["probs"].numpy()
    probs = probs / probs.sum(axis=1, keepdims=True)
    return {"probs": probs, "log": pd.DataFrame(payload["log"]), "meta": payload["meta"], "label": label}


POLICYNET_CODE_HASH = code_hash(CategoricalPolicyNet, train_policy_net, init_model, OfflineTensorBuffer, stream_seed,
                                torch_generator, generator_state, restore_generator)
print("Direct categorical policy code hash:", POLICYNET_CODE_HASH[:16])

## 12. Categorical diffusion prior and exact critic-guided reverse inference (audit C03, M06)

* **C03 repair:** the denoiser is constructed from its named initialisation stream *inside*
  `init_model`, i.e. after seeding — the previous notebook built the network before seeding, so its
  initial weights depended on earlier notebook execution. Minibatch indices and corruption
  (timestep + noisy-token) draws now use separate streams.
* **Schedule audit:** with β linearly spaced from 0.02 to 0.25 over T = 20 steps,
  `Q̄_T = ᾱ_T I + (1 − ᾱ_T) U` with `ᾱ_T = Π(1 − β_t) ≈ 0.0515`: the terminal forward distribution is
  **not exactly uniform**. Starting inference from a uniform distribution over admissible actions, masking at
  every step and adding critic guidance make the procedure an explicitly *modified inference heuristic*,
  not exact sampling from the learned diffusion model. Intermediate tokens are latent variables, not treatments.
* Inference variants (all computed by exact marginal propagation over the finite state set):
  initialisation (uniform-admissible vs uniform over all 25), masking placement (every step, final step only,
  none), guidance placement (every step, final step only, none), guidance strength, critic pairing
  (hard-only, tabular-only, mixed), deterministic vs stochastic final extraction.
* The deployed policy is a **precomputed lookup table** over the 716 states; reverse-chain cost is a
  policy-construction cost, reported separately from per-decision lookup latency.

In [ ]:
# ============================================================
# 12. Categorical diffusion
# ============================================================
EXECLOG.mark("12. Categorical diffusion")


def sinusoidal_time_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    half = dim // 2
    exponent = -math.log(10_000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / max(half - 1, 1)
    angles = t.float()[:, None] * torch.exp(exponent)[None, :]
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
    return F.pad(emb, (0, 1)) if dim % 2 == 1 else emb


class CategoricalDenoiser(nn.Module):
    """Predict x0 from (state, corrupted treatment x_t, step t); treatment token factorised fluid x vasopressor."""

    def __init__(self, state_dim, num_actions, hidden_dim, action_embed_dim, time_embed_dim):
        super().__init__()
        levels = int(round(math.sqrt(num_actions)))
        assert levels * levels == num_actions
        self.levels, self.time_embed_dim = levels, time_embed_dim
        self.fluid_embedding = nn.Embedding(levels, action_embed_dim)
        self.vaso_embedding = nn.Embedding(levels, action_embed_dim)
        self.state_encoder = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
                                           nn.Linear(hidden_dim, hidden_dim), nn.SiLU())
        fused = hidden_dim + 2 * action_embed_dim + time_embed_dim
        self.head = nn.Sequential(nn.Linear(fused, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
                                  nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, num_actions))

    def forward(self, state_x, xt, t):
        fluid = torch.div(xt, self.levels, rounding_mode="floor")
        vaso = xt % self.levels
        h = torch.cat([self.state_encoder(state_x), self.fluid_embedding(fluid), self.vaso_embedding(vaso),
                       sinusoidal_time_embedding(t, self.time_embed_dim)], dim=1)
        return self.head(h)


class CategoricalDiffusion:
    """Uniform categorical corruption Q_t = (1 - beta_t) I + beta_t U with exact posteriors."""

    def __init__(self, num_actions: int, T: int, beta_start: float, beta_end: float):
        self.K, self.T = int(num_actions), int(T)
        self.betas = np.linspace(beta_start, beta_end, self.T, dtype=np.float64)
        eye = np.eye(self.K)
        U = np.ones((self.K, self.K)) / self.K
        self.Q, self.Qbar = [None], [eye]
        running = eye
        for beta in self.betas:
            qt = (1.0 - beta) * eye + beta * U
            running = running @ qt
            self.Q.append(qt)
            self.Qbar.append(running.copy())
        self.alpha_bar = np.cumprod(1.0 - self.betas)
        self.posterior = [None]
        for t in range(1, self.T + 1):
            post = np.zeros((self.K, self.K, self.K))
            for xt in range(self.K):
                num = self.Qbar[t - 1] * self.Q[t][:, xt][None, :]          # [x0, xprev]
                cond = num / np.maximum(self.Qbar[t][:, xt][:, None], 1e-30)
                post[xt] = cond / np.maximum(cond.sum(axis=1, keepdims=True), 1e-30)
            self.posterior.append(post)
            assert np.allclose(self.Q[t].sum(1), 1) and np.allclose(self.Qbar[t].sum(1), 1)
            assert np.allclose(post.sum(axis=2), 1.0, atol=1e-10)
        self._qbar_cache = {}

    def schedule_audit(self) -> dict:
        abar = float(self.alpha_bar[-1])
        closed = abar * np.eye(self.K) + (1 - abar) * np.ones((self.K, self.K)) / self.K
        return {"T": self.T, "beta_start": float(self.betas[0]), "beta_end": float(self.betas[-1]),
                "alpha_bar_T_initial_token_coefficient": abar,
                "max_abs_dev_Qbar_T_from_closed_form": float(np.abs(self.Qbar[-1] - closed).max()),
                "max_prob_of_initial_token_at_T": float(abar + (1 - abar) / self.K),
                "terminal_forward_exactly_uniform": bool(abar < 1e-12)}

    def sample_xt(self, x0: torch.Tensor, t: torch.Tensor, generator: torch.Generator) -> torch.Tensor:
        key = str(x0.device)
        if key not in self._qbar_cache:
            self._qbar_cache[key] = torch.as_tensor(np.stack(self.Qbar), dtype=torch.float32, device=x0.device)
        return torch.multinomial(self._qbar_cache[key][t, x0], 1, replacement=True, generator=generator).squeeze(1)


DIFFUSION_CODE_HASH = None


def diffusion_identity(bundle: DataBundle, seed: int, T: int) -> dict:
    hp = {"steps": CFG.diffusion_steps, "lr": CFG.diffusion_lr, "T": int(T), "beta_start": CFG.diffusion_beta_start,
          "beta_end": CFG.diffusion_beta_end, "action_embed": CFG.diffusion_action_embed_dim,
          "time_embed": CFG.diffusion_time_embed_dim, "hidden": CFG.hidden_dim, "batch": CFG.batch_size,
          "weight_decay": CFG.weight_decay, "grad_clip": CFG.grad_clip}
    return {"algo": "categorical_diffusion_prior", "hparams": hp, "seed": int(seed), "run_mode": RUN_MODE,
            "data": bundle.identity(), "benchmark": BENCH.identity, "code": DIFFUSION_CODE_HASH}


def build_denoiser(input_dim: int) -> CategoricalDenoiser:
    return CategoricalDenoiser(input_dim, A_N, CFG.hidden_dim, CFG.diffusion_action_embed_dim, CFG.diffusion_time_embed_dim)


def train_diffusion_prior(bundle: DataBundle, seed: int, T: Optional[int] = None, tag: str = "diffusion",
                          store: Optional[CheckpointStore] = None, interrupt_at: Optional[int] = None):
    store = STORE if store is None else store
    T = CFG.diffusion_T if T is None else int(T)
    diffusion = CategoricalDiffusion(A_N, T, CFG.diffusion_beta_start, CFG.diffusion_beta_end)
    ident = diffusion_identity(bundle, seed, T)
    label = f"{tag}/T{T}/{bundle.name}/seed{seed}"
    payload = store.load_final(label, ident, map_location=DEVICE)
    if payload is None:
        store.require_training_allowed(label)
        t0 = time.perf_counter()
        reset_peak_gpu_memory()
        buf = bundle.buffer()
        model = init_model(lambda: build_denoiser(bundle.input_dim), f"denoiser_{bundle.input_dim}x{CFG.hidden_dim}", seed)
        opt = torch.optim.AdamW(model.parameters(), lr=CFG.diffusion_lr, weight_decay=CFG.weight_decay)
        gen_batch = torch_generator(stream_seed("minibatch", bundle.data_hash, seed), DEVICE)
        gen_noise = torch_generator(stream_seed("corruption", bundle.data_hash, seed, T), DEVICE)
        rows, start = [], 0
        prior_elapsed, prior_peak, segments = 0.0, 0.0, 1
        resume = store.load_resume(label, ident, map_location=DEVICE)
        if resume is not None:
            model.load_state_dict(resume["model"])
            opt.load_state_dict(resume["optimizer"])
            restore_generator(gen_batch, resume["gen_batch"])
            restore_generator(gen_noise, resume["gen_noise"])
            rows, start = resume["log"], int(resume["step"])
            prior_elapsed = float(resume.get("elapsed_s", 0.0))
            prior_peak = float(resume.get("peak_gpu_mb") or 0.0)
            segments = int(resume.get("segments", 1)) + 1
            print(f"  resumed {label} at step {start}")
        model.train()
        for step in trange(start, CFG.diffusion_steps, desc=f"diffusion T{T} {bundle.name} s{seed}",
                           leave=False, disable=not SHOW_PROGRESS):
            b = buf.sample(CFG.batch_size, gen_batch)
            x0 = b["a"]
            t = torch.randint(1, T + 1, (CFG.batch_size,), generator=gen_noise, device=DEVICE)
            xt = diffusion.sample_xt(x0, t, gen_noise)
            logits = model(b["x"], xt, t)
            loss = F.cross_entropy(logits, x0)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            opt.step()
            if step % CFG.log_every == 0 or step == CFG.diffusion_steps - 1:
                rows.append({"step": step + 1, "loss": float(loss),
                             "x0_accuracy": float((logits.argmax(1) == x0).float().mean())})
            if (step + 1) % CFG.resume_every == 0 and step + 1 < CFG.diffusion_steps:
                store.save_resume(label, ident, {"model": model.state_dict(), "optimizer": opt.state_dict(),
                                                 "gen_batch": generator_state(gen_batch),
                                                 "gen_noise": generator_state(gen_noise), "step": step + 1, "log": rows,
                                                 "elapsed_s": prior_elapsed + (time.perf_counter() - t0),
                                                 "peak_gpu_mb": max(prior_peak, peak_gpu_memory_mb() or 0.0),
                                                 "segments": segments})
                if interrupt_at is not None and step + 1 >= interrupt_at:
                    return None
        model.eval()
        meta = {"elapsed_s": prior_elapsed + (time.perf_counter() - t0), "params": count_parameters(model),
                "peak_gpu_mb": (max(prior_peak, peak_gpu_memory_mb() or 0.0) if torch.cuda.is_available() else None),
                "steps": CFG.diffusion_steps, "T": T, "normalizer": bundle.normalizer.to_dict(),
                "resumed": resume is not None, "segments": segments}
        payload = {"model": model.state_dict(), "log": rows, "meta": meta}
        store.save_final(label, ident, payload)
        COST_ROWS.append({"model": f"Diffusion prior (T={T})", "label": label, "seed": seed,
                          "train_s": meta["elapsed_s"], "params": meta["params"],
                          "peak_gpu_mb": meta["peak_gpu_mb"], "steps": CFG.diffusion_steps})
    model = build_denoiser(bundle.input_dim).to(DEVICE)
    model.load_state_dict(payload["model"])
    model.eval()
    return {"model": model, "diffusion": diffusion, "log": pd.DataFrame(payload["log"]), "meta": payload["meta"],
            "label": label}


class ReverseCache:
    """Exact base reverse kernels B_t(x_{t-1} | x_t, s) for every state, cached once per trained denoiser."""

    @torch.no_grad()
    def __init__(self, model: CategoricalDenoiser, diffusion: CategoricalDiffusion, all_x: np.ndarray, batch: int = 8192,
                 device: Optional[torch.device] = None):
        dev = DEVICE if device is None else device
        t0 = time.perf_counter()
        model.eval()
        S, K = all_x.shape[0], diffusion.K
        self.T, self.K, self.S = diffusion.T, K, S
        x_all = torch.as_tensor(all_x, dtype=torch.float32, device=dev)
        s_idx = torch.arange(S, device=dev).repeat_interleave(K)
        xt_idx = torch.arange(K, device=dev).repeat(S)
        self.base = {}
        for t in range(self.T, 0, -1):
            probs = []
            for i in range(0, S * K, batch):
                sl = slice(i, i + batch)
                tt = torch.full((len(s_idx[sl]),), t, dtype=torch.long, device=dev)
                probs.append(torch.softmax(model(x_all[s_idx[sl]], xt_idx[sl], tt).double(), dim=1).cpu())
            p_x0 = torch.cat(probs).numpy().reshape(S, K, K)
            # Same floor (1e-30) as the original implementation before taking logarithms.
            self.base[t] = np.clip(np.einsum("ski,kij->skj", p_x0, diffusion.posterior[t], optimize=True), 1e-30, None)
        self.construction_s = time.perf_counter() - t0


def normalize_critic_per_state(q: np.ndarray, mask: np.ndarray = OPERATIONAL_MASK) -> np.ndarray:
    q = np.asarray(q, dtype=np.float64)
    m = np.asarray(mask, dtype=bool)
    cnt = m.sum(axis=1, keepdims=True)
    mu = np.where(m, q, 0.0).sum(axis=1, keepdims=True) / cnt
    sd = np.sqrt(np.where(m, (q - mu) ** 2, 0.0).sum(axis=1, keepdims=True) / cnt)
    return np.where(m, (q - mu) / np.maximum(sd, 1e-6), 0.0)


def critic_ensemble(*q_tables: np.ndarray, mask: np.ndarray = OPERATIONAL_MASK) -> np.ndarray:
    return np.mean([normalize_critic_per_state(q, mask) for q in q_tables], axis=0)


def guided_reverse_policy(cache: ReverseCache, critic_G: Optional[np.ndarray], guidance: float,
                          mask: np.ndarray = OPERATIONAL_MASK, init: str = "uniform_admissible",
                          mask_mode: str = "every_step", guidance_mode: str = "every_step",
                          deterministic: bool = True) -> Tuple[np.ndarray, List[float]]:
    assert init in {"uniform_admissible", "uniform_all"}
    assert mask_mode in {"every_step", "final_only", "none"}
    assert guidance_mode in {"every_step", "final_only", "none"}
    if guidance_mode == "every_step" and guidance != 0 and mask_mode != "every_step":
        raise ValueError("Every-step guidance is only defined together with every-step masking.")
    m = np.asarray(mask, dtype=bool)
    p = m / m.sum(axis=1, keepdims=True) if init == "uniform_admissible" else np.full((cache.S, cache.K), 1.0 / cache.K)
    trace = []
    for t in range(cache.T, 0, -1):
        logits = np.log(cache.base[t])
        g = guidance if (guidance_mode == "every_step" or (guidance_mode == "final_only" and t == 1)) else 0.0
        if g != 0.0 and critic_G is not None:
            logits = logits + g * critic_G[:, None, :]
        masked_now = mask_mode == "every_step" or (mask_mode == "final_only" and t == 1)
        if masked_now:
            logits = np.where(m[:, None, :], logits, -np.inf)
        logits = logits - logits.max(axis=2, keepdims=True)
        kernel = np.exp(logits)
        kernel /= kernel.sum(axis=2, keepdims=True)
        p = np.einsum("sk,skj->sj", p, kernel, optimize=True)
        if masked_now:
            p = np.where(m, p, 0.0)
        p = p / p.sum(axis=1, keepdims=True)
        trace.append(float(np.where(m, 0.0, p)[LIVE].sum(axis=1).max()))
    policy = one_hot_policy(p.argmax(axis=1)) if deterministic else p
    if mask_mode != "none":
        # Invariant with respect to the mask actually supplied (released or dataset-estimated).
        assert np.all(policy[LIVE][~m[LIVE]] <= 1e-14), "Masked reverse inference produced mass outside the mask."
        assert float(np.where(m, 0.0, p)[LIVE].sum(axis=1).max()) <= 1e-12
    return policy, trace


def prior_times_critic(prior: np.ndarray, critic_G: Optional[np.ndarray], guidance: float,
                       mask: np.ndarray = OPERATIONAL_MASK, deterministic: bool = True) -> np.ndarray:
    """pi(a|s) ∝ m(s,a) prior(a|s) exp(g G(s,a)) — the simplest behaviour-prior-times-critic policy."""
    logits = np.log(np.clip(prior, 1e-300, None))
    if critic_G is not None and guidance != 0.0:
        logits = logits + guidance * critic_G
    logits = np.where(mask, logits, -np.inf)
    logits = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    p /= p.sum(axis=1, keepdims=True)
    return one_hot_policy(p.argmax(axis=1)) if deterministic else p


@torch.no_grad()
def ondemand_reverse_latency_ms(model: CategoricalDenoiser, diffusion: CategoricalDiffusion, all_x: np.ndarray,
                                n_states: int = 20) -> float:
    """Wall time to run the full reverse chain for ONE state on demand (not the deployed lookup)."""
    rng = np_rng("evaluation", "latency")
    states = rng.choice(np.flatnonzero(LIVE), size=n_states, replace=False)
    K = diffusion.K
    times = []
    for s in states:
        t0 = time.perf_counter()
        x = torch.as_tensor(all_x[[s] * K], dtype=torch.float32, device=DEVICE)
        xt = torch.arange(K, device=DEVICE)
        p = SUPPORT_MASK[s] / SUPPORT_MASK[s].sum()
        for t in range(diffusion.T, 0, -1):
            p_x0 = torch.softmax(model(x, xt, torch.full((K,), t, device=DEVICE)).double(), 1).cpu().numpy()
            kern = np.einsum("ki,kij->kj", p_x0, diffusion.posterior[t])
            kern = np.where(SUPPORT_MASK[s][None, :], kern, 0.0)
            kern /= kern.sum(axis=1, keepdims=True)
            p = p @ kern
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return 1000.0 * float(np.median(times))


def lookup_latency_us(policy: np.ndarray, n: int = 100_000) -> float:
    table = policy.argmax(axis=1)
    rng = np_rng("evaluation", "lookup")
    qs = rng.integers(0, S_N, size=n)
    t0 = time.perf_counter()
    acc = 0
    for s in qs:
        acc += int(table[s])
    return 1e6 * (time.perf_counter() - t0) / n


DIFFUSION_CODE_HASH = code_hash(sinusoidal_time_embedding, CategoricalDenoiser, CategoricalDiffusion,
                                build_denoiser, train_diffusion_prior, init_model, OfflineTensorBuffer, stream_seed,
                                torch_generator, generator_state, restore_generator)
SCHEDULE_AUDIT = CategoricalDiffusion(A_N, CFG.diffusion_T, CFG.diffusion_beta_start, CFG.diffusion_beta_end).schedule_audit()
print("Diffusion schedule audit:", SCHEDULE_AUDIT)
save_json(SCHEDULE_AUDIT, TABLE_DIR / "diffusion_schedule_audit.json")

## 13. Reproducibility closure tests (audit C03) and regulariser unit tests (audit §6)

**Regulariser unit tests (always run, no training):** finite gradients for the legacy, design-A and design-B
regularisers; the legacy regulariser's derivative with respect to an *excluded* logged action's Q equals
−1/B (the confirmed defect); design B passes zero gradient to excluded rows and to masked entries; an
all-excluded batch returns 0 with zero gradient; the `batch` normalisation equals the `conditional` one times
the fraction of admissible-logged rows.

**Closure tests** (isolated temporary checkpoint stores, short budgets; they also run in artifact-replay mode
because the temporary stores are allowed to train):
1. initial weights for a seed are identical in forward or reverse seed order and after unrelated RNG draws;
2. short critic and denoiser training reproduces after unrelated RNG draws and in a different seed order;
3. an interrupted run resumed from its resumable checkpoint reproduces the uninterrupted run — tested for a
   hard critic, for the **soft critic (dual variable λ restored)**, and for the denoiser;
4. changing the dataset, objective, run mode, seed or *actual class code* changes the run identity, and a
   checkpoint copied under a foreign identity is rejected rather than loaded.

Declared hardware tolerance: bitwise on CPU; on CUDA a maximum absolute difference ≤ 1e-4 is accepted
because some backward kernels (e.g. scatter-add for `gather`) are not deterministic on GPUs.

In [ ]:
# ============================================================
# 13. Regulariser unit tests + reproducibility tests
# ============================================================
EXECLOG.mark("13. Reproducibility tests")
REPRO_TESTS: Dict[str, Any] = {}
HW_TOL = 0.0 if DEVICE.type == "cpu" else 1e-4

# ---------- Regulariser gradient tests (audit §6: test before long training) ----------
_rows = np.flatnonzero(LIVE & ~SUPPORT_MASK.all(axis=1))[:8]
_m = torch.as_tensor(OPERATIONAL_MASK[_rows])
_a = torch.as_tensor([int(np.flatnonzero(~OPERATIONAL_MASK[s])[0]) if i % 2 else int(np.flatnonzero(OPERATIONAL_MASK[s])[0])
                      for i, s in enumerate(_rows)])
_adm = _m.gather(1, _a[:, None]).squeeze(1)
_B = len(_rows)
REG_GRAD_TESTS = {"batch_rows": _B, "admissible_logged_rows": int(_adm.sum())}
for _spec in (SPEC_HARD_LEGACY, SPEC_HARD_A, SPEC_HARD_B, dataclasses.replace(SPEC_HARD_B, reg_normalization="batch")):
    _q = torch.randn(_B, A_N, dtype=torch.float64, generator=torch.Generator().manual_seed(0), requires_grad=True)
    cql_regularizer(_q, _a, _m, _adm, _spec).backward()
    assert torch.isfinite(_q.grad).all(), _spec.key
    if _spec.reg_domain == "masked_legacy":
        _g = _q.grad[~_adm, _a[~_adm]]
        assert torch.allclose(_g, torch.full_like(_g, -1.0 / _B)), "Legacy excluded-row derivative is not -1/B."
        REG_GRAD_TESTS["legacy_excluded_logged_action_derivative"] = float(_g.mean())
    if _spec.reg_domain == "masked_admissible_rows":
        assert float(_q.grad[~_adm].abs().max()) == 0.0, "Design B leaks gradient into excluded rows."
        assert float(_q.grad[_adm][~_m[_adm]].abs().max()) == 0.0, "Design B leaks gradient into masked actions."
_q = torch.randn(_B, A_N, dtype=torch.float64, requires_grad=True)
_r0 = cql_regularizer(_q, _a, _m, torch.zeros_like(_adm), SPEC_HARD_B)
_r0.backward()
assert float(_r0) == 0.0 and float(_q.grad.abs().max()) == 0.0, "All-excluded batch must give 0 with zero gradient."
_q = torch.randn(_B, A_N, dtype=torch.float64)
_rc = cql_regularizer(_q, _a, _m, _adm, SPEC_HARD_B)
_rb = cql_regularizer(_q, _a, _m, _adm, dataclasses.replace(SPEC_HARD_B, reg_normalization="batch"))
assert torch.allclose(_rb, _rc * _adm.double().mean()), "batch != conditional x admissible fraction."
REG_GRAD_TESTS["passed"] = True
REPRO_TESTS["regularizer_unit_tests"] = REG_GRAD_TESTS
save_json(REG_GRAD_TESTS, TABLE_DIR / "regularizer_gradient_tests.json")
print("Regulariser unit tests passed:", REG_GRAD_TESTS)

if EXPERIMENTS["reproducibility_tests"]:
    test_root = ROOT / "_repro_tests"
    shutil.rmtree(test_root, ignore_errors=True)
    saved_cfg = (CFG.resume_every, CFG.diag_every, CFG.log_every, CFG.diffusion_steps)
    try:
        # 1. initialisation independence
        builders = {
            "twin_critic": (lambda: TwinDuelingCQL(MAIN.input_dim, A_N, CFG.hidden_dim), f"twin_dueling_{MAIN.input_dim}x{CFG.hidden_dim}"),
            "denoiser": (lambda: build_denoiser(MAIN.input_dim), f"denoiser_{MAIN.input_dim}x{CFG.hidden_dim}"),
            "policy_net": (lambda: CategoricalPolicyNet(MAIN.input_dim, A_N, CFG.hidden_dim), f"policy_net_{MAIN.input_dim}x{CFG.hidden_dim}"),
        }
        init_rows = {}
        for name, (builder, key) in builders.items():
            ref = state_dict_hash(init_model(builder, key, 3))
            for s in (0, 1, 2):
                init_model(builder, key, s)
            torch.rand(1234)
            np.random.rand(77)
            random.random()
            fwd = state_dict_hash(init_model(builder, key, 3))
            for s in (4, 2, 1, 0):
                init_model(builder, key, s)
            rev = state_dict_hash(init_model(builder, key, 3))
            other = state_dict_hash(init_model(builder, key, 4))
            init_rows[name] = {"same_after_other_seeds_and_draws": ref == fwd, "same_in_reverse_order": ref == rev,
                               "different_seed_differs": ref != other}
            assert all(init_rows[name].values()), (name, init_rows[name])
        REPRO_TESTS["initialisation"] = init_rows

        # 2-3. short training determinism, seed-order independence, resume equivalence
        CFG.resume_every, CFG.diag_every, CFG.log_every = 50, 10_000, 50
        short = dataclasses.replace(SPEC_HARD_B, steps=200)
        short_soft = dataclasses.replace(SPEC_SOFT, steps=200)
        stA, stB, stC = (CheckpointStore(test_root / x, allow_training=True) for x in ("A", "B", "C"))
        rA = train_critic(short, MAIN, 1, tag="repro", store=stA)
        torch.rand(999)
        np.random.rand(5)
        train_critic(short, MAIN, 0, tag="repro", store=stB)
        rB = train_critic(short, MAIN, 1, tag="repro", store=stB)
        assert train_critic(short, MAIN, 1, tag="repro", store=stC, interrupt_at=100) is None, "Interrupt hook failed."
        rC = train_critic(short, MAIN, 1, tag="repro", store=stC)
        sA = train_critic(short_soft, MAIN, 1, tag="repro", store=stA)
        assert train_critic(short_soft, MAIN, 1, tag="repro", store=stC, interrupt_at=100) is None
        sC = train_critic(short_soft, MAIN, 1, tag="repro", store=stC)
        crit = {
            "max_abs_diff_rerun_after_unrelated_draws_and_other_seed": float(np.abs(rA["q1"] - rB["q1"]).max()),
            "max_abs_diff_resumed_vs_uninterrupted": float(np.abs(rA["q1"] - rC["q1"]).max()),
            "resumed_flag": bool(rC["meta"]["resumed"]),
            "soft_max_abs_q_diff_resumed_vs_uninterrupted": float(np.abs(sA["q1"] - sC["q1"]).max()),
            "soft_abs_lambda_diff_resumed_vs_uninterrupted": abs(sA["meta"]["final_lambda"] - sC["meta"]["final_lambda"]),
            "soft_resumed_flag": bool(sC["meta"]["resumed"]),
        }
        crit["passes_declared_tolerance"] = bool(max(
            crit["max_abs_diff_rerun_after_unrelated_draws_and_other_seed"], crit["max_abs_diff_resumed_vs_uninterrupted"],
            crit["soft_max_abs_q_diff_resumed_vs_uninterrupted"], crit["soft_abs_lambda_diff_resumed_vs_uninterrupted"]) <= HW_TOL)
        REPRO_TESTS["critic_training"] = crit

        CFG.diffusion_steps = 200
        dA = train_diffusion_prior(MAIN, 1, tag="repro", store=stA)
        torch.rand(31)
        train_diffusion_prior(MAIN, 0, tag="repro", store=stB)
        dB = train_diffusion_prior(MAIN, 1, tag="repro", store=stB)
        assert train_diffusion_prior(MAIN, 1, tag="repro", store=stC, interrupt_at=100) is None
        dC = train_diffusion_prior(MAIN, 1, tag="repro", store=stC)
        sdA, sdB, sdC = (m["model"].state_dict() for m in (dA, dB, dC))
        diffd = {
            "max_abs_param_diff_rerun": float(max((sdA[k].double() - sdB[k].double()).abs().max() for k in sdA)),
            "max_abs_param_diff_resumed": float(max((sdA[k].double() - sdC[k].double()).abs().max() for k in sdA)),
        }
        diffd["passes_declared_tolerance"] = bool(max(diffd.values()) <= HW_TOL)
        REPRO_TESTS["diffusion_training"] = diffd

        # 4. cache invalidation (including a real class-code edit) and foreign-identity rejection
        def _probe_class(body: str):
            ns = {}
            exec(compile(f"class _Probe:\n    def f(self):\n        return {body}\n", "<probe>", "exec"), ns)
            return ns["_Probe"]
        code_edit_detected = code_hash(_probe_class("1")) != code_hash(_probe_class("2"))
        base_ident = critic_identity(short, 1, MAIN)
        sub = make_bundle("main_subset", OFFLINE, OFFLINE_MANIFEST, rows=np.arange(len(OFFLINE["state"]) - 1))
        variants = {
            "dataset": critic_identity(short, 1, sub),
            "objective": critic_identity(dataclasses.replace(short, alpha=2.0), 1, MAIN),
            "run_mode": {**base_ident, "run_mode": "smoke" if RUN_MODE == "paper" else "paper"},
            "code_hash_value": {**base_ident, "code": "edited-code"},
            "seed": critic_identity(short, 2, MAIN),
        }
        h0 = obj_hash(base_ident)
        inval = {k: obj_hash(v) != h0 for k, v in variants.items()}
        inval["actual_class_code_edit"] = bool(code_edit_detected)
        assert all(inval.values()), inval
        label = "repro/hard_admrows/main/seed1"
        src, _, _ = stA._paths(label, base_ident)
        dst, _, _ = stA._paths(label, variants["objective"])
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        try:
            stA.load_final(label, variants["objective"], respect_force=False)
            rejected = False
        except CheckpointIdentityError:
            rejected = True
        assert rejected, "A checkpoint stored under a foreign identity was loaded."
        REPRO_TESTS["cache_invalidation"] = {**{f"identity_changes_with_{k}": v for k, v in inval.items()},
                                             "foreign_identity_checkpoint_rejected": rejected}
    finally:
        CFG.resume_every, CFG.diag_every, CFG.log_every, CFG.diffusion_steps = saved_cfg
        shutil.rmtree(test_root, ignore_errors=True)
    REPRO_TESTS["hardware_tolerance"] = HW_TOL
    REPRO_TESTS["device"] = str(DEVICE)
    save_json(REPRO_TESTS, TABLE_DIR / "reproducibility_tests.json")
    print(json.dumps(REPRO_TESTS, indent=2, default=_json_default))
    if not (REPRO_TESTS["critic_training"]["passes_declared_tolerance"]
            and REPRO_TESTS["diffusion_training"]["passes_declared_tolerance"]):
        print("WARNING: training reproducibility exceeded the declared hardware tolerance on this device.")
else:
    print("Reproducibility closure tests skipped (EXPERIMENTS['reproducibility_tests'] = False).")

## 14. Reference policies, oracles, and environment-semantics tests (audit C01, M09)

* Behaviour cloning (native argmax, masked greedy, stochastic empirical), uniform random over all 25
  actions, **uniform over admissible actions**, the benchmark clinician policy, and value-iteration
  oracles (unconstrained mean-optimal, admissibility-constrained, terminate-optimal). Oracles use the true
  model and are privileged references, not offline competitors.
* The rounded documentation baselines are kept only as a broad sanity check; the tight checks are the
  analytic tests of Section 8, the evaluator invariants, the environment-semantics test below, and the
  Monte-Carlo cross-check of Section 20.

In [ ]:
# ============================================================
# 14. Reference policies and oracles
# ============================================================
EXECLOG.mark("14. Reference policies")

BC = train_behavior_cloning(MAIN)
register_policy("Behavior Cloning", FIXED, BC["deterministic"], scores=BC["counts"], group="reference",
                family="BC", extraction="argmax of empirical counts (unmasked)", stochastic=False,
                mask_source="none", mask_use="none", information="logged state-action counts")
register_policy("Behavior Cloning [masked greedy]", FIXED, BC["masked_greedy"], group="reference",
                family="BC", extraction="argmax of counts over admissible actions", stochastic=False,
                mask_source="released benchmark mask", mask_use="extraction",
                information="logged counts + released mask", parent="Behavior Cloning")
register_policy("Behavior Cloning [stochastic empirical]", FIXED, BC["stochastic"], group="reference",
                family="BC", extraction="empirical action distribution", stochastic=True,
                mask_source="none", mask_use="none", information="logged state-action counts")
register_policy("Random (uniform over 25)", FIXED, np.full((S_N, A_N), 1.0 / A_N), group="reference",
                family="Random", extraction="uniform", stochastic=True, mask_source="none", mask_use="none",
                information="none")
register_policy("Random (uniform admissible)", FIXED, uniform_admissible_policy(), group="reference",
                family="Random", extraction="uniform over admissible actions", stochastic=True,
                mask_source="released benchmark mask", mask_use="construction", information="released mask")

CLINICIAN_POLICY = EXPERT_ROWS.copy()
CLINICIAN_POLICY[LIVE] = CLINICIAN_POLICY[LIVE] / CLINICIAN_POLICY[LIVE].sum(axis=1, keepdims=True)
CLINICIAN_POLICY[~LIVE] = 1.0 / A_N   # absorbing rows carry no expert probabilities; never used by the evaluator
register_policy("Clinician", FIXED, CLINICIAN_POLICY, group="reference", family="Clinician",
                extraction="benchmark expert policy", stochastic=True, mask_source="none", mask_use="none",
                information="benchmark clinician policy")

ORACLE_INFO = {}
opt_mean, _, ORACLE_INFO["mean_unconstrained"] = EVALUATOR.optimal_policy("mean", constrain_to_admissible=False)
opt_cons, _, ORACLE_INFO["mean_constrained"] = EVALUATOR.optimal_policy("mean", constrain_to_admissible=True)
opt_term, _, ORACLE_INFO["terminate_unconstrained"] = EVALUATOR.optimal_policy("terminate", constrain_to_admissible=False)
opt_mean_low, _, ORACLE_INFO["mean_unconstrained_lowest_index_ties"] = EVALUATOR.optimal_policy(
    "mean", constrain_to_admissible=False, tie_break="lowest_index")
ORACLE_INFO["terminate_optimum_equals_constrained_optimum"] = bool(
    abs(ORACLE_INFO["terminate_unconstrained"]["dp_value"] - ORACLE_INFO["mean_constrained"]["dp_value"]) < 1e-8)
ORACLE_INFO["mean_optimum_equals_constrained_optimum"] = bool(
    abs(ORACLE_INFO["mean_unconstrained"]["dp_value"] - ORACLE_INFO["mean_constrained"]["dp_value"]) < 1e-8)
print("Note: the mean-optimal policy is not unique (exact q ties between admissible and mean-imputed inadmissible "
      "actions). The registered oracle uses admissible tie-breaking; the previous notebook's lowest-index "
      "tie-breaking is kept as a separately labelled policy, and its violations are a tie-breaking artefact.")
for name, pi, note in [("Optimal Oracle (mean-optimal)", opt_mean, "unconstrained value iteration under mean, admissible tie-breaking"),
                       ("Optimal Oracle (mean-optimal, lowest-index ties)", opt_mean_low,
                        "unconstrained value iteration under mean, np.argmax tie-breaking (previous notebook)"),
                       ("Constrained Optimal Oracle", opt_cons, "value iteration over admissible actions"),
                       ("Optimal Oracle (terminate-optimal)", opt_term, "value iteration under terminate")]:
    register_policy(name, FIXED, pi, group="reference", family="Oracle", extraction=note, stochastic=False,
                    mask_source="released benchmark mask" if "Constrained" in name else "none",
                    mask_use="construction" if "Constrained" in name else "none",
                    information="TRUE transition model (privileged)")
print(json.dumps(ORACLE_INFO, indent=2))

reference_baselines = {"Random (uniform over 25)": (0.78, 9.45), "Clinician": (0.78, 9.22),
                       "Optimal Oracle (mean-optimal)": (0.88, 10.99)}
sanity = []
for name, (ref_ret, ref_len) in reference_baselines.items():
    m = EVALUATOR.evaluate(REGISTRY[name].by_seed[FIXED], "mean", condition=True)
    sanity.append({"policy": name, "measured_return": m["return"], "reference_return_rounded": ref_ret,
                   "measured_length": m["avg_episode_length"], "reference_length_rounded": ref_len,
                   "condition_number": m["condition_number"]})
BENCHMARK_SANITY = pd.DataFrame(sanity)
display(BENCHMARK_SANITY)
BENCHMARK_SANITY.to_csv(TABLE_DIR / "benchmark_reference_sanity.csv", index=False)
assert (np.abs(BENCHMARK_SANITY["measured_return"] - BENCHMARK_SANITY["reference_return_rounded"]) <= 0.05).all()
assert (np.abs(BENCHMARK_SANITY["measured_length"] - BENCHMARK_SANITY["reference_length_rounded"]) <= 2.0).all()

ENV_SEMANTICS = {}
if EXPERIMENTS["env_semantics_tests"]:
    ENV_SEMANTICS = env_semantics_tests(n_pairs=40 if RUN_MODE == "paper" else 10,
                                        n_draws=1500 if RUN_MODE == "paper" else 300)
    print(json.dumps(ENV_SEMANTICS, indent=2))
    assert ENV_SEMANTICS["terminate_all_consistent"], "Terminate semantics differ between env and evaluator."
    if not ENV_SEMANTICS["mean_rule_all_within_bounds"]:
        print("WARNING: an empirical mean-rule transition row deviates beyond its 5-sigma bound.")
save_json({"oracles": ORACLE_INFO, "env_semantics": ENV_SEMANTICS, "codec": CODEC_CHECK,
           "evaluator_toy_tests": EVALUATOR_TOY_TESTS}, TABLE_DIR / "numerical_audit.json")

## 15. Main study — tabular learners on the main seeds (0–9 in the full profile, 0–4 in lite)

In [ ]:
# ============================================================
# 15. Main study: tabular learners
# ============================================================
EXECLOG.mark("15. Main study: tabular")
TAB: Dict[Tuple[str, str, int], Dict[str, Any]] = {}


def save_log(name: str, seed: int, df: pd.DataFrame) -> None:
    if df is not None and len(df):
        df.to_csv(LOG_DIR / f"{slug(name)}__seed{seed}.csv", index=False)


def run_tabular_suite(bundle: DataBundle, seeds: Sequence[int], group: str, prefix: str = "",
                      dataset_label: str = "main") -> None:
    for seed in seeds:
        cql = train_tabular_cql(bundle, seed)
        iql = train_tabular_iql(bundle, seed)
        bcq = train_tabular_bcq(bundle, seed)
        TAB[("cql", bundle.name, seed)], TAB[("iql", bundle.name, seed)], TAB[("bcq", bundle.name, seed)] = cql, iql, bcq
        common = dict(group=group, dataset=dataset_label)
        register_policy(f"{prefix}Tabular CQL", seed, greedy_policy(cql["q"], masked=False), scores=cql["q"],
                        family="Tabular CQL", extraction="unmasked greedy", stochastic=False, mask_source="none",
                        mask_use="none", information="logged transitions", **common)
        register_policy(f"{prefix}Tabular CQL [masked greedy]", seed, greedy_policy(cql["q"], masked=True),
                        family="Tabular CQL", extraction="masked greedy (inference mask)", stochastic=False,
                        mask_source="released benchmark mask", mask_use="extraction",
                        information="logged transitions + released mask", parent=f"{prefix}Tabular CQL", **common)
        register_policy(f"{prefix}Tabular IQL", seed, iql["deterministic"], scores=iql["aw_scores"],
                        family="Tabular IQL", extraction="argmax of advantage-weighted counts (unmasked)",
                        stochastic=False, mask_source="none", mask_use="none", information="logged transitions", **common)
        register_policy(f"{prefix}Tabular IQL [masked greedy]", seed, iql["masked_greedy"], family="Tabular IQL",
                        extraction="argmax of advantage-weighted scores over admissible actions", stochastic=False,
                        mask_source="released benchmark mask", mask_use="extraction",
                        information="logged transitions + released mask", parent=f"{prefix}Tabular IQL",
                        notes=f"global-frequency fallback in {iql['masked_fallback_states']} states without a logged admissible action",
                        **common)
        register_policy(f"{prefix}Tabular IQL [stochastic AW]", seed, iql["aw_policy"], family="Tabular IQL",
                        extraction="advantage-weighted count distribution", stochastic=True, mask_source="none",
                        mask_use="none", information="logged transitions", parent=f"{prefix}Tabular IQL", **common)
        register_policy(f"{prefix}Tabular BCQ (tau={CFG.bcq_threshold})", seed, bcq["deterministic"], scores=bcq["q"],
                        family="Tabular BCQ", extraction="greedy over behaviour-thresholded actions", stochastic=False,
                        mask_source="dataset-estimated (behaviour ratio >= tau)", mask_use="training+extraction",
                        information="logged transitions only", **common)
        register_policy(f"{prefix}Tabular BCQ (tau={CFG.bcq_threshold}) [masked greedy]", seed, bcq["masked_greedy"],
                        family="Tabular BCQ", extraction="greedy over (behaviour-thresholded ∩ released) actions",
                        stochastic=False, mask_source="dataset-estimated + released benchmark mask",
                        mask_use="training+extraction", information="logged transitions + released mask",
                        parent=f"{prefix}Tabular BCQ (tau={CFG.bcq_threshold})", **common)
        if group == "main":
            save_log("Tabular CQL", seed, cql["log"])
            save_log("Tabular IQL", seed, iql["log"])
            save_log("Tabular BCQ", seed, bcq["log"])


if EXPERIMENTS["main_tabular"]:
    run_tabular_suite(MAIN, CFG.seeds, group="main")
    print("Tabular main study complete:", [n for n in REGISTRY if n.startswith("Tabular")])

## 16. Main study — Deep CQL, legacy hard AM-CQL, repaired hard designs A and B, soft AM-CQL (main seeds: 0–9 in the full profile, 0–4 in lite)

In [ ]:
# ============================================================
# 16. Main study: neural critics
# ============================================================
EXECLOG.mark("16. Main study: neural critics")
CRITICS: Dict[Tuple[str, str, int], Dict[str, Any]] = {}
MARGIN_ROWS: List[dict] = []


_TAB_TRAINERS = {"cql": train_tabular_cql, "iql": train_tabular_iql, "bcq": train_tabular_bcq}


def get_tab(kind: str, bundle: DataBundle, seed: int) -> Dict[str, Any]:
    """Identity-checked tabular result: in-memory, else loaded from the store (trained only if allowed)."""
    k = (kind, bundle.name, seed)
    if k not in TAB:
        TAB[k] = _TAB_TRAINERS[kind](bundle, seed)
    return TAB[k]


def get_critic(key: str, bundle: DataBundle, seed: int) -> Dict[str, Any]:
    """Identity-checked critic result: in-memory, else loaded from the store (trained only if allowed)."""
    k = (key, bundle.name, seed)
    if k not in CRITICS:
        CRITICS[k] = train_critic(SPEC_BY_KEY[key], bundle, seed)
    return CRITICS[k]


def masked_export_flag(spec: CriticSpec, suffix: str) -> bool:
    return suffix == "[masked greedy]" or (suffix == "" and spec.masked_extraction)


def critic_mask_meta(spec: CriticSpec, suffix: str) -> Tuple[str, str]:
    masked_export = masked_export_flag(spec, suffix)
    if spec.soft:
        return "released benchmark mask", "training penalty" + (" + extraction" if masked_export else "")
    train = [u for u, on in (("backup", spec.masked_backup), ("regulariser", spec.reg_domain != "full")) if on]
    if not train:
        return ("released benchmark mask", "extraction") if masked_export else ("none", "none")
    return "released benchmark mask", ("training+extraction" if masked_export else "training (" + "+".join(train) + ")")


def register_critic(res: Dict[str, Any], group: str, prefix: str = "", dataset_label: str = "main") -> None:
    spec = res["spec"]
    for suffix, (pi, extraction, stochastic) in critic_extractions(res).items():
        name = f"{prefix}{spec.display}{(' ' + suffix) if suffix else ''}"
        src, use = critic_mask_meta(spec, suffix)
        register_policy(name, res["seed"], pi, scores=np.minimum(res["q1"], res["q2"]) if suffix == "" else None,
                        family=f"Neural critic ({spec.key})", extraction=extraction, stochastic=stochastic,
                        mask_source=src, mask_use=use, information="logged transitions + 47-d features"
                        + (" + released mask" if src != "none" else ""), critic=spec.key, group=group,
                        dataset=dataset_label, parent="" if suffix == "" else f"{prefix}{spec.display}",
                        notes=spec.status)
        if not stochastic and not masked_export_flag(spec, suffix):
            scores = 0.5 * (res["q1"] + res["q2"]) if "mean critic" in suffix else np.minimum(res["q1"], res["q2"])
            MARGIN_ROWS.append({**tie_and_margin_report(scores, name), "seed": res["seed"]})


def run_critics(specs: Sequence[CriticSpec], bundle: DataBundle, seeds: Sequence[int], group: str,
                prefix: str = "", tag: str = "critic", dataset_label: str = "main") -> None:
    for seed in seeds:
        for spec in specs:
            res = train_critic(spec, bundle, seed, tag=tag)
            CRITICS[(spec.key, bundle.name, seed)] = res
            register_critic(res, group=group, prefix=prefix, dataset_label=dataset_label)
            if group == "main":
                save_log(spec.display, seed, res["log"])
                res["diag"].assign(variant=spec.key, seed=seed).to_csv(
                    LOG_DIR / f"diag_{slug(spec.key)}__seed{seed}.csv", index=False)


if EXPERIMENTS["main_neural_critics"]:
    run_critics(MAIN_CRITIC_SPECS, MAIN, CFG.seeds, group="main")
    print("Neural critic main study complete.")

## 17. Main study — matched-capacity direct categorical policies (main seeds: 0–9 in the full profile, 0–4 in lite)

In [ ]:
# ============================================================
# 17. Main study: direct categorical policies
# ============================================================
EXECLOG.mark("17. Main study: direct categorical policies")
PNETS: Dict[Tuple[str, str, int], Dict[str, Any]] = {}


def register_policy_net(res: Dict[str, Any], label: str, seed: int, group: str, dataset_label: str = "main",
                        information: str = "logged state-action pairs + 47-d features") -> None:
    probs = res["probs"]
    common = dict(group=group, dataset=dataset_label, family="Direct categorical")
    register_policy(f"{label} [stochastic]", seed, probs, extraction="network softmax", stochastic=True,
                    mask_source="none", mask_use="none", information=information, **common)
    register_policy(f"{label} [greedy]", seed, one_hot_policy(probs.argmax(axis=1)), extraction="unmasked argmax",
                    stochastic=False, mask_source="none", mask_use="none", information=information,
                    parent=f"{label} [stochastic]", **common)
    register_policy(f"{label} [masked greedy]", seed, greedy_policy(probs, masked=True),
                    extraction="argmax over admissible actions", stochastic=False,
                    mask_source="released benchmark mask", mask_use="extraction",
                    information=information + " + released mask", parent=f"{label} [stochastic]", **common)


if EXPERIMENTS["main_policy_nets"]:
    for seed in CFG.seeds:
        bc_net = train_policy_net(MAIN, seed, mode="bc")
        PNETS[("bc", "main", seed)] = bc_net
        register_policy_net(bc_net, "Neural BC (matched capacity)", seed, group="main")
        iql = get_tab("iql", MAIN, seed)
        adv = iql["q"] - iql["v"][:, None]
        aw_net = train_policy_net(MAIN, seed, mode="aw", advantages=adv)
        PNETS[("aw", "main", seed)] = aw_net
        register_policy_net(aw_net, "AW categorical imitation", seed, group="main",
                            information="logged pairs + tabular-IQL advantages + 47-d features")
        # Same critic scores as the primary diffusion policy (M06: preserve the same mask and critic scores).
        rB = get_critic(PREREG["primary_hard_critic"], MAIN, seed)
        G_B_aw = critic_ensemble(np.minimum(rB["q1"], rB["q2"]), get_tab("cql", MAIN, seed)["q"])
        adv_B = np.where(OPERATIONAL_MASK, G_B_aw, -1e3)   # standardised over admissible actions; inadmissible -> weight exp(-20)
        awB = train_policy_net(MAIN, seed, mode="aw", advantages=adv_B, tag="policy_net_awB")
        PNETS[("awB", "main", seed)] = awB
        register_policy_net(awB, "AW categorical imitation [critic B + tabular CQL]", seed, group="main",
                            information="logged pairs + critic-B/tabular-CQL ensemble scores + 47-d features")
        save_log("Neural BC", seed, bc_net["log"])
        save_log("AW categorical imitation", seed, aw_net["log"])
    print("Direct categorical policies complete.")

## 18. Main study — diffusion priors, critic-guided policies, and inference-only ablations (main seeds: 0–9 in the full profile, 0–4 in lite)

The denoiser is trained once per seed; every guided variant reuses its exact reverse kernels, so all
inference ablations (Tables 6–7 of the original study, plus the new masking/initialisation/guidance-placement
controls) are reproducible here rather than in an unreleased script. Variants guided by the legacy hard
critic reproduce the original method and are labelled as such; the preregistered primary policy uses the
repaired design-B critic.

In [ ]:
# ============================================================
# 18. Main study: diffusion + inference ablations
# ============================================================
EXECLOG.mark("18. Main study: diffusion")
DIFFUSION_TRACES: Dict[str, Dict[int, List[float]]] = {}
DIFF_PRIORS: Dict[Tuple[str, int, int], Dict[str, Any]] = {}
G_PRIMARY = CFG.diffusion_guidance_scale


def critic_qmin(key: str, bundle_name: str, seed: int) -> np.ndarray:
    r = CRITICS[(key, bundle_name, seed)]
    return np.minimum(r["q1"], r["q2"])


def register_diffusion(name: str, seed: int, pi: np.ndarray, trace: Optional[List[float]], group: str,
                       extraction: str, critic: str, stochastic: bool = False, mask_use: str = "inference (every reverse step)",
                       mask_source: str = "released benchmark mask", dataset_label: str = "main", notes: str = "") -> None:
    register_policy(name, seed, pi, family="Categorical diffusion", extraction=extraction, stochastic=stochastic,
                    mask_source=mask_source, mask_use=mask_use,
                    information="logged pairs + 47-d features + critics" + (" + released mask" if mask_source != "none" else ""),
                    critic=critic, group=group, dataset=dataset_label, notes=notes)
    if trace is not None:
        DIFFUSION_TRACES.setdefault(name, {})[seed] = trace


def run_diffusion_suite(bundle: DataBundle, seed: int, group: str, prefix: str = "", full_ablations: bool = True,
                        T: Optional[int] = None, dataset_label: str = "main", tab_bundle: Optional[DataBundle] = None,
                        tag: str = "diffusion") -> Dict[str, Any]:
    T = CFG.diffusion_T if T is None else T
    tab_bundle = bundle if tab_bundle is None else tab_bundle
    # Upstream objects are loaded from identity-checked checkpoints when their blocks were switched off.
    get_tab("cql", tab_bundle, seed)
    get_critic(PREREG["primary_hard_critic"], bundle, seed)
    if bundle.name == "main":
        get_critic("hard_legacy", bundle, seed)
        get_critic("hard_fullcql", bundle, seed)
    prior = train_diffusion_prior(bundle, seed, T=T, tag=tag)
    DIFF_PRIORS[(bundle.name, seed, T)] = {k: prior[k] for k in ("log", "meta", "label")}
    cache = ReverseCache(prior["model"], prior["diffusion"], bundle.all_x)
    tab_q = TAB[("cql", tab_bundle.name, seed)]["q"]
    qB = critic_qmin(PREREG["primary_hard_critic"], bundle.name, seed)
    G_B = critic_ensemble(qB, tab_q)
    t0 = time.perf_counter()
    pi_B, tr_B = guided_reverse_policy(cache, G_B, G_PRIMARY)
    construct_s = time.perf_counter() - t0
    name_primary = f"{prefix}{PRIMARY_DIFFUSION}"
    register_diffusion(name_primary, seed, pi_B, tr_B, group, "guided reverse chain, deterministic final",
                       "hard_admrows + tabular CQL", dataset_label=dataset_label)
    out = {"cache_s": cache.construction_s, "construct_s": construct_s}
    if ("hard_legacy", bundle.name, seed) in CRITICS:
        G_L = critic_ensemble(critic_qmin("hard_legacy", bundle.name, seed), tab_q)
        pi_L, tr_L = guided_reverse_policy(cache, G_L, G_PRIMARY)
        register_diffusion(f"{prefix}Diffusion-AM-CQL [legacy critic]", seed, pi_L, tr_L, group,
                           "guided reverse chain, deterministic final (original method)", "hard_legacy + tabular CQL",
                           dataset_label=dataset_label, notes="historical / failure-mode critic")
    if ("hard_fullcql", bundle.name, seed) in CRITICS:
        G_A = critic_ensemble(critic_qmin("hard_fullcql", bundle.name, seed), tab_q)
        pi_A, tr_A = guided_reverse_policy(cache, G_A, G_PRIMARY)
        register_diffusion(f"{prefix}Diffusion-AM-CQL [critic A]", seed, pi_A, tr_A, group,
                           "guided reverse chain, deterministic final", "hard_fullcql + tabular CQL", dataset_label=dataset_label)
    register_policy(f"{prefix}Masked critic ensemble [critic B + tabular CQL]", seed, one_hot_policy(masked_argmax(G_B)),
                    family="Critic only", extraction="masked argmax of the standardised critic ensemble (no diffusion)",
                    stochastic=False, mask_source="released benchmark mask", mask_use="extraction",
                    information="critics + released mask", critic="hard_admrows + tabular CQL", group=group,
                    dataset=dataset_label)
    if ("bc", bundle.name, seed) in PNETS:
        register_policy(f"{prefix}Neural BC prior x critic [critic B + tabular CQL]", seed,
                        prior_times_critic(PNETS[("bc", bundle.name, seed)]["probs"], G_B, G_PRIMARY),
                        family="Prior x critic", extraction="argmax of m * neural-BC prior * exp(g G)", stochastic=False,
                        mask_source="released benchmark mask", mask_use="extraction",
                        information="neural BC + critics + released mask", critic="hard_admrows + tabular CQL",
                        group=group, dataset=dataset_label)
    if full_ablations:
        bc_counts = train_behavior_cloning(bundle)["stochastic"]
        register_policy(f"{prefix}Tabular BC prior x critic [critic B + tabular CQL]", seed,
                        prior_times_critic(bc_counts, G_B, G_PRIMARY), family="Prior x critic",
                        extraction="argmax of m * empirical-BC prior * exp(g G)", stochastic=False,
                        mask_source="released benchmark mask", mask_use="extraction",
                        information="counts + critics + released mask", critic="hard_admrows + tabular CQL", group=group,
                        dataset=dataset_label)
        G_tab = critic_ensemble(tab_q)
        G_Bonly = critic_ensemble(qB)
        variants = {
            "Diffusion [masked, no guidance (g=0)]": dict(G=None, g=0.0, critic="none"),
            "Diffusion [critic B only]": dict(G=G_Bonly, g=G_PRIMARY, critic="hard_admrows"),
            "Diffusion [tabular CQL critic only]": dict(G=G_tab, g=G_PRIMARY, critic="tabular CQL"),
            f"{PRIMARY_DIFFUSION} [stochastic final]": dict(G=G_B, g=G_PRIMARY, critic="hard_admrows + tabular CQL", det=False),
            f"{PRIMARY_DIFFUSION} [init uniform over 25]": dict(G=G_B, g=G_PRIMARY, critic="hard_admrows + tabular CQL", init="uniform_all"),
            f"{PRIMARY_DIFFUSION} [guidance at final step only]": dict(G=G_B, g=G_PRIMARY, critic="hard_admrows + tabular CQL", gmode="final_only"),
            "Diffusion prior [final-step mask, no guidance]": dict(G=None, g=0.0, critic="none", init="uniform_all", mmode="final_only", gmode="none"),
            "Diffusion prior x critic B [final-step mask + guidance]": dict(G=G_B, g=G_PRIMARY, critic="hard_admrows + tabular CQL", init="uniform_all", mmode="final_only", gmode="final_only"),
            "Diffusion prior [unmasked, unguided, stochastic]": dict(G=None, g=0.0, critic="none", init="uniform_all", mmode="none", gmode="none", det=False),
            "Diffusion prior [unmasked, unguided, greedy]": dict(G=None, g=0.0, critic="none", init="uniform_all", mmode="none", gmode="none"),
        }
        if ("hard_legacy", bundle.name, seed) in CRITICS:
            G_L = critic_ensemble(critic_qmin("hard_legacy", bundle.name, seed), tab_q)
            G_Lonly = critic_ensemble(critic_qmin("hard_legacy", bundle.name, seed))
            variants.update({
                "Diffusion [legacy hard critic only]": dict(G=G_Lonly, g=G_PRIMARY, critic="hard_legacy"),
                "Diffusion-AM-CQL [legacy critic] [stochastic final]": dict(G=G_L, g=G_PRIMARY, critic="hard_legacy + tabular CQL", det=False),
            })
            register_policy(f"{prefix}Masked critic ensemble [legacy critic + tabular CQL]", seed,
                            one_hot_policy(masked_argmax(G_L)), family="Critic only",
                            extraction="masked argmax of the standardised critic ensemble (no diffusion)",
                            stochastic=False, mask_source="released benchmark mask", mask_use="extraction",
                            information="critics + released mask", critic="hard_legacy + tabular CQL", group=group,
                            dataset=dataset_label)
            for g in CFG.guidance_grid:
                variants[f"Diffusion-AM-CQL [legacy critic] [g={g}]"] = dict(G=G_L, g=g, critic="hard_legacy + tabular CQL")
        for g in CFG.guidance_grid:
            variants[f"{PRIMARY_DIFFUSION} [g={g}]"] = dict(G=G_B, g=g, critic="hard_admrows + tabular CQL")
        for vname, v in variants.items():
            mmode = v.get("mmode", "every_step")
            pi, tr = guided_reverse_policy(cache, v["G"], v["g"], init=v.get("init", "uniform_admissible"),
                                           mask_mode=mmode, guidance_mode=v.get("gmode", "every_step"),
                                           deterministic=v.get("det", True))
            register_diffusion(f"{prefix}{vname}", seed, pi, tr, group if group != "main" else "inference_ablation",
                               f"init={v.get('init', 'uniform_admissible')}, mask={mmode}, guidance={v.get('gmode', 'every_step')}, g={v['g']}",
                               v["critic"], stochastic=not v.get("det", True),
                               mask_use={"every_step": "inference (every reverse step)", "final_only": "inference (final step only)",
                                         "none": "none"}[mmode],
                               mask_source="none" if mmode == "none" else "released benchmark mask", dataset_label=dataset_label)
    out["lookup_us"] = lookup_latency_us(pi_B)
    if group == "main" and seed == CFG.seeds[0]:
        out["ondemand_ms"] = ondemand_reverse_latency_ms(prior["model"], prior["diffusion"], bundle.all_x)
    del cache
    gc.collect()
    return out


DIFFUSION_COST: List[dict] = []
if EXPERIMENTS["main_diffusion"]:
    for seed in CFG.seeds:
        c = run_diffusion_suite(MAIN, seed, group="main", full_ablations=EXPERIMENTS["inference_ablations"])
        DIFFUSION_COST.append({"seed": seed, **c})
        save_log("Diffusion prior", seed, DIFF_PRIORS[("main", seed, CFG.diffusion_T)]["log"])
        for nm in (PRIMARY_DIFFUSION, "Diffusion-AM-CQL [legacy critic]"):
            if nm in DIFFUSION_TRACES and seed in DIFFUSION_TRACES[nm]:
                save_json({"policy": nm, "seed": seed, "guidance_scale": G_PRIMARY, "T": CFG.diffusion_T,
                           "max_inadmissible_mass_each_reverse_step": DIFFUSION_TRACES[nm][seed]},
                          TABLE_DIR / f"diffusion_reverse_support_{slug(nm)}_seed{seed}.json")
    pd.DataFrame(DIFFUSION_COST).to_csv(TABLE_DIR / "diffusion_policy_construction_cost.csv", index=False)
    print("Diffusion main study complete.")

## 19. Admissible projection and extraction controls — the decisive missing baseline (audit C01)

Every registered policy that places any probability on an inadmissible action receives a
**`[projected]`** counterpart built from the original policy and the *same released mask* available to the
masked learners (so the projection's mask use is disclosed exactly like AM-CQL's). Stochastic natives also get
a **`[mask-renormalized]`** counterpart, which conditions on admissibility and is *not* return-preserving.

Closure checks, per policy and seed: row sums, zero inadmissible probability, and equality of the projected
versus original **policy-induced transition matrix and expected-reward vector under the mean rule**, plus
equality of the projected policy's induced quantities under the two rules, all to numerical tolerance.
Note the projection of a deterministic export is a *stochastic* policy; deterministic masked variants are
evaluated separately (`[masked greedy]`).

In [ ]:
# ============================================================
# 19. Projection controls + equivalence verification
# ============================================================
EXECLOG.mark("19. Projection controls")
PROJECTION_ROWS: List[dict] = []


def add_projection_controls(groups: Sequence[str]) -> None:
    for name in [n for n, e in REGISTRY.items() if e.group in groups and not n.endswith("[projected]")
                 and not n.endswith("[mask-renormalized]")]:
        e = REGISTRY[name]
        if not any(inadmissible_mass(pi).max() > 0 for pi in e.by_seed.values()):
            continue
        done = REGISTRY.get(f"{name} [projected]")
        for seed, pi in e.by_seed.items():
            if done is not None and seed in done.by_seed:
                continue   # already projected (avoids redundant work when a group is revisited)
            proj = project_to_support(pi)
            register_policy(f"{name} [projected]", seed, proj, family=e.family,
                            extraction=f"mean-equivalent admissible projection of: {e.extraction}", stochastic=True,
                            mask_source="released benchmark mask", mask_use="projection",
                            information=e.information + " + released mask", parent=name, critic=e.critic,
                            group=e.group, dataset=e.dataset)
            P_o, r_o = EVALUATOR.induced(pi, "mean")
            P_pm, r_pm = EVALUATOR.induced(proj, "mean")
            P_pt, r_pt = EVALUATOR.induced(proj, "terminate")
            PROJECTION_ROWS.append({
                "policy": name, "seed": seed, "group": e.group,
                "max_inadmissible_mass_original": float(inadmissible_mass(pi).max()),
                "max_inadmissible_mass_projected": float(inadmissible_mass(proj).max()),
                "max_row_sum_error_projected": float(np.abs(proj.sum(axis=1) - 1).max()),
                "max_abs_dP_original_vs_projected_mean": float(np.abs(P_o - P_pm)[LIVE].max()),
                "max_abs_dr_original_vs_projected_mean": float(np.abs(r_o - r_pm)[LIVE].max()),
                "max_abs_dP_projected_mean_vs_terminate": float(np.abs(P_pm - P_pt)[LIVE].max()),
                "max_abs_dr_projected_mean_vs_terminate": float(np.abs(r_pm - r_pt)[LIVE].max()),
            })
            if e.stochastic:
                ren, n_fb = renormalize_on_support(pi)
                register_policy(f"{name} [mask-renormalized]", seed, ren, family=e.family,
                                extraction=f"renormalised on admissible actions: {e.extraction}", stochastic=True,
                                mask_source="released benchmark mask", mask_use="extraction",
                                information=e.information + " + released mask", parent=name, critic=e.critic,
                                group=e.group, dataset=e.dataset, notes=f"{n_fb} zero-mass rows -> uniform admissible")


def finalize_projection_checks(verbose: bool = True) -> bool:
    """Write and assert ALL projection-equivalence rows accumulated so far (called after every projection batch)."""
    global PROJECTION_EQUIVALENCE, PROJECTION_IDENTITY_VERIFIED
    PROJECTION_EQUIVALENCE = (pd.DataFrame(PROJECTION_ROWS).drop_duplicates(["policy", "seed"], keep="last")
                              if PROJECTION_ROWS else pd.DataFrame())
    PROJECTION_EQUIVALENCE.to_csv(TABLE_DIR / "projection_equivalence_checks.csv", index=False)
    if not len(PROJECTION_EQUIVALENCE):
        PROJECTION_IDENTITY_VERIFIED = False
        return False
    assert PROJECTION_EQUIVALENCE["max_inadmissible_mass_projected"].max() == 0.0
    assert PROJECTION_EQUIVALENCE["max_row_sum_error_projected"].max() <= 1e-10
    worst = PROJECTION_EQUIVALENCE[[c for c in PROJECTION_EQUIVALENCE.columns if c.startswith("max_abs")]].max()
    if verbose:
        print(f"Projection equivalence over {len(PROJECTION_EQUIVALENCE)} policy/seed pairs; worst deviations:\n", worst)
    PROJECTION_IDENTITY_VERIFIED = bool(worst.max() <= 1e-10)
    if MEAN_RULE_IDENTITY_HOLDS:
        assert PROJECTION_IDENTITY_VERIFIED, "Projection did not preserve mean-rule dynamics although the identity holds."
    return PROJECTION_IDENTITY_VERIFIED


add_projection_controls(["main", "reference", "inference_ablation"])
finalize_projection_checks()

## 20. Exact evaluation of every registered policy under `mean` and `terminate`, plus a Monte-Carlo cross-check (audit M09)

Every results row carries the policy's family, extraction rule, stochastic/deterministic label,
**mask source and mask use** (audit M02 closure: every table identifies the information each policy used).

In [ ]:
# ============================================================
# 20. Exact evaluation
# ============================================================
EXECLOG.mark("20. Exact evaluation")
RULES = ("mean", "terminate")
RESULT_ROWS: List[dict] = []
EVALUATED: set = set()
METRICS = ["return", "avg_episode_length", "inadmissible_rate", "expected_violations", "prob_any_violation",
           "states_with_inadmissible_mass", "max_state_inadmissible_mass", "mean_state_inadmissible_mass",
           "absorb_death", "absorb_survival"]


def evaluate_registered(names: Optional[Iterable[str]] = None, evaluator: Optional[ExactMDPEvaluator] = None,
                        condition: bool = False) -> pd.DataFrame:
    ev = EVALUATOR if evaluator is None else evaluator
    for name in (list(REGISTRY) if names is None else list(names)):
        e = REGISTRY[name]
        for seed, pi in sorted(e.by_seed.items()):
            if (name, seed) in EVALUATED:
                continue
            for rule in RULES:
                try:
                    m = ev.evaluate(pi, rule, condition=condition and e.group in ("main", "reference"))
                    status = "ok"
                except ImproperPolicyError as exc:
                    m, status = {}, f"improper: {exc}"
                RESULT_ROWS.append({**e.meta(), "seed": int(seed), "rule": rule, "status": status,
                                    **{k: m.get(k, np.nan) for k in METRICS + ["condition_number"]}})
            EVALUATED.add((name, seed))
    return pd.DataFrame(RESULT_ROWS)


def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    ok = df[df["status"] == "ok"]
    agg = {f"{m}_mean": (m, "mean") for m in METRICS}
    agg.update({f"{m}_sd": (m, "std") for m in METRICS})
    out = ok.groupby(["name", "rule"], sort=False).agg(n_seeds=("seed", "nunique"), **agg).reset_index()
    meta_cols = ["name", "family", "extraction", "stochastic", "mask_source", "mask_use", "information", "group",
                 "dataset", "parent", "critic"]
    meta = df[meta_cols].drop_duplicates("name")
    return out.merge(meta, on="name", how="left")


RESULTS = evaluate_registered(condition=True)
SUMMARY = summarize_results(RESULTS)
RESULTS.to_csv(TABLE_DIR / "exact_results_all.csv", index=False)
SUMMARY.to_csv(TABLE_DIR / "exact_results_summary.csv", index=False)
n_bad = int((RESULTS["status"] != "ok").sum())
print(f"Evaluated {RESULTS[['name', 'seed']].drop_duplicates().shape[0]} policy/seed pairs; improper: {n_bad}")

MAIN_TABLE_POLICIES = [
    "Random (uniform over 25)", "Random (uniform admissible)", "Clinician", "Clinician [projected]",
    "Behavior Cloning", "Behavior Cloning [masked greedy]", "Behavior Cloning [projected]",
    "Tabular CQL", "Tabular CQL [masked greedy]", "Tabular CQL [projected]",
    "Tabular IQL", "Tabular IQL [masked greedy]", "Tabular IQL [projected]", "Tabular IQL [stochastic AW] [projected]",
    f"Tabular BCQ (tau={CFG.bcq_threshold})", f"Tabular BCQ (tau={CFG.bcq_threshold}) [masked greedy]",
    "Deep CQL", "Deep CQL [masked greedy]", "Deep CQL [projected]",
    "AM-CQL (Hard, legacy)", "AM-CQL (Hard, design A: full-action CQL)", "AM-CQL (Hard, design B: admissible-row CQL)",
    "AM-CQL (Soft, legacy)", "AM-CQL (Soft, legacy) [masked greedy]", "AM-CQL (Soft, legacy) [projected]",
    "Neural BC (matched capacity) [masked greedy]", "AW categorical imitation [masked greedy]",
    "Masked critic ensemble [critic B + tabular CQL]", "Neural BC prior x critic [critic B + tabular CQL]",
    "Diffusion-AM-CQL [legacy critic]", "Diffusion-AM-CQL [critic A]", PRIMARY_DIFFUSION,
    "Constrained Optimal Oracle", "Optimal Oracle (mean-optimal)",
]


def mask_columns(mask_use: str) -> Dict[str, bool]:
    """Mask availability at training and at inference/extraction (audit M02: every table states it)."""
    mu = str(mask_use)
    return {"mask_at_training": "training" in mu,
            "mask_at_inference": any(t in mu for t in ("extraction", "projection", "inference", "construction"))}


def wide_table(summary: pd.DataFrame, names: Sequence[str], metrics=("return", "inadmissible_rate")) -> pd.DataFrame:
    rows = []
    for n in names:
        sub = summary[summary["name"] == n]
        if sub.empty:
            continue
        row = {"policy": n, "n_seeds": int(sub["n_seeds"].max()), "mask_source": sub["mask_source"].iloc[0],
               "mask_use": sub["mask_use"].iloc[0], **mask_columns(sub["mask_use"].iloc[0]),
               "information": sub["information"].iloc[0], "stochastic": bool(sub["stochastic"].iloc[0])}
        for rule in RULES:
            r = sub[sub["rule"] == rule]
            if r.empty:
                continue
            for m in metrics:
                mean, sd = float(r[f"{m}_mean"].iloc[0]), r[f"{m}_sd"].iloc[0]
                scale = 100.0 if m == "inadmissible_rate" else 1.0
                row[f"{m}_{rule}"] = mean * scale
                row[f"{m}_{rule}_sd"] = (float(sd) * scale) if pd.notna(sd) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


MAIN_TABLE = wide_table(SUMMARY, MAIN_TABLE_POLICIES, metrics=("return", "inadmissible_rate", "prob_any_violation",
                                                               "avg_episode_length"))
MAIN_TABLE.to_csv(TABLE_DIR / "table_main_policy_comparison.csv", index=False)
display(MAIN_TABLE[["policy", "n_seeds", "mask_use", "return_mean", "return_mean_sd", "return_terminate",
                    "return_terminate_sd", "inadmissible_rate_mean", "prob_any_violation_mean"]].round(5))

per_seed = RESULTS[(RESULTS["group"] == "main") & (RESULTS["status"] == "ok")].pivot_table(
    index=["name", "rule"], columns="seed", values="return").reset_index()
per_seed.to_csv(TABLE_DIR / "table_per_seed_returns.csv", index=False)

# ---------- Monte-Carlo cross-check through the real environment ----------
MC_ROWS = []
if EXPERIMENTS["monte_carlo_crosscheck"]:
    mc_targets = [("Clinician", FIXED), ("Random (uniform over 25)", FIXED), ("Tabular IQL", CFG.seeds[0]),
                  ("Tabular IQL [projected]", CFG.seeds[0]), (PRIMARY_DIFFUSION, CFG.seeds[0])]
    for name, seed in mc_targets:
        if name not in REGISTRY or seed not in REGISTRY[name].by_seed:
            continue
        pi = REGISTRY[name].by_seed[seed]
        for rule in RULES:
            exact = safe_evaluate(pi, rule)
            if exact is None:
                continue
            mc = monte_carlo_estimate(pi, rule, CFG.mc_episodes, f"{name}|{seed}")
            z = (mc["mc_return"] - exact["return"]) / max(mc["mc_return_se"], 1e-12)
            zL = (mc["mc_length"] - exact["avg_episode_length"]) / max(mc["mc_length_se"], 1e-12)
            # Poisson-type floor on the SE so that an all-zero Monte-Carlo sample is judged against the exact rate.
            se_v = max(mc["mc_violations_se"], math.sqrt(max(exact["expected_violations"], 1e-12) / CFG.mc_episodes))
            zV = (mc["mc_violations"] - exact["expected_violations"]) / se_v
            MC_ROWS.append({"policy": name, "seed": seed, "rule": rule, "exact_return": exact["return"], **mc,
                            "z_score": z, "exact_length": exact["avg_episode_length"], "z_length": zL,
                            "exact_expected_violations": exact["expected_violations"], "z_violations": zV,
                            "within_4_se": bool(max(abs(z), abs(zL), abs(zV)) <= 4.0)})
    MC_CHECK = pd.DataFrame(MC_ROWS)
    MC_CHECK.to_csv(TABLE_DIR / "monte_carlo_crosscheck.csv", index=False)
    display(MC_CHECK.round(5))
    if len(MC_CHECK) and not MC_CHECK["within_4_se"].all():
        print("WARNING: a Monte-Carlo estimate deviates from exact evaluation by more than 4 standard errors.")

## 21. Legacy artifact replay — the ORIGINAL run's policies and checkpoints (audit Phase 2, C01, C03, C04)

If `LEGACY_ROOT` (default `/content/drive/MyDrive/sepsis_fresh_colab`) exists, its files are read-only inputs:
1. every original exported policy is re-validated and re-evaluated and compared with the saved
   `exact_results_all.csv` rows (reported maximum discrepancy);
2. the admissible projection, deterministic masked IQL (from the original advantage-weighted checkpoint
   scores) and masked BC are evaluated on the **original** artifacts, which directly tests the audit's
   analytical prediction (projected IQL ≈ 0.78997 under both rules versus original diffusion 0.78565);
3. the original dataset is compared with the regenerated one, the saved normalizer is compared with a
   refit (overwrite check), and original critic checkpoints are replayed to recover their Q-scale and to
   check that the exported actions follow from the checkpoints.

In [ ]:
# ============================================================
# 21. Legacy artifact replay (read-only)
# ============================================================
EXECLOG.mark("21. Legacy artifact replay")
LEGACY = {"root": str(LEGACY_ROOT), "found": bool(LEGACY_ROOT.exists()),
          "replay_enabled": bool(EXPERIMENTS["legacy_artifact_replay"])}
LEGACY_NAMES = {"Tabular CQL": "Tabular_CQL", "Tabular IQL": "Tabular_IQL", "Deep CQL": "Deep_CQL",
                "AM-CQL (Hard)": "AM-CQL_(Hard)", "AM-CQL (Soft)": "AM-CQL_(Soft)", "Diffusion-AM-CQL": "Diffusion-AM-CQL"}
LEGACY_FIXED = {"Behavior Cloning": "BC_fixed", "Random": "Random_fixed", "Clinician": "Clinician_fixed",
                "Optimal Oracle": "Optimal_oracle", "Constrained Optimal Oracle": "Constrained_optimal_oracle"}

if EXPERIMENTS["legacy_artifact_replay"] and LEGACY_ROOT.exists():
    lp = LEGACY_ROOT / "policies"
    legacy_rows_path = LEGACY_ROOT / "tables" / "exact_results_all.csv"
    legacy_saved = pd.read_csv(legacy_rows_path) if legacy_rows_path.exists() else None
    replay_rows = []

    def _legacy_register(model: str, seed, pi: np.ndarray, extraction: str, mask_use: str, mask_source: str,
                         stochastic: bool, parent: str = "") -> None:
        register_policy(f"LEGACY {model}", FIXED if seed in ("fixed", "oracle") else int(seed), pi, group="legacy",
                        dataset="original run", family=f"Legacy {model.split(' [')[0]}", extraction=extraction,
                        stochastic=stochastic, mask_source=mask_source, mask_use=mask_use,
                        information="original exported artifact", parent=parent)

    for model, stem in {**LEGACY_FIXED}.items():
        f = lp / f"{stem}.npz"
        if f.exists():
            with np.load(f) as z:
                _legacy_register(model, "fixed", z["policy"], "original export", "see original", "see original",
                                 stochastic=model in ("Random", "Clinician"))
    for model, stem in LEGACY_NAMES.items():
        for seed in range(10):
            f = lp / f"{stem}_seed{seed}.npz"
            if f.exists():
                with np.load(f) as z:
                    _legacy_register(model, seed, z["policy"], "original export",
                                     "training+extraction" if model in ("AM-CQL (Hard)", "Diffusion-AM-CQL") else "none",
                                     "released benchmark mask" if model in ("AM-CQL (Hard)", "Diffusion-AM-CQL", "AM-CQL (Soft)") else "none",
                                     stochastic=False)
    legacy_names = [n for n in REGISTRY if n.startswith("LEGACY ")]
    # deterministic masked IQL from the original advantage-weighted checkpoint scores
    for seed in range(10):
        f = LEGACY_ROOT / "checkpoints" / f"tabular_iql_seed{seed}.pt"
        if f.exists():
            pl = load_torch_payload(f)
            aw = pl["policy"].numpy().astype(np.float64)
            qv = pl["q"].numpy().astype(np.float64)
            adm_aw = np.where(OPERATIONAL_MASK, aw, -np.inf)
            act = adm_aw.argmax(axis=1)
            none_logged = ~(np.where(OPERATIONAL_MASK, aw, 0.0) > 0).any(axis=1)
            act[none_logged] = masked_argmax(qv)[none_logged]
            _legacy_register("Tabular IQL [masked greedy]", seed, one_hot_policy(act),
                             "masked argmax of original AW scores", "extraction", "released benchmark mask", False,
                             parent="LEGACY Tabular IQL")
            _legacy_register("Tabular IQL [stochastic AW]", seed, aw, "original AW distribution", "none", "none", True,
                             parent="LEGACY Tabular IQL")
    if "LEGACY Behavior Cloning" in REGISTRY:
        legacy_data = LEGACY_ROOT / "data" / "offline_expert_10000eps_seed2026.npz"
        if legacy_data.exists():
            with np.load(legacy_data) as z:
                LD = {k: z[k] for k in z.files}
            same = {k: bool(np.array_equal(LD[k], OFFLINE[k])) for k in ("state", "action", "next_state", "episode_id")
                    if k in LD}
            same["reward"] = bool(np.array_equal(LD["reward"].astype(np.float32), OFFLINE["reward"]))
            same["timestep"] = bool(np.array_equal(LD["timestep"].astype(np.int64), OFFLINE["timestep"].astype(np.int64)))
            same["done_equals_terminated_or_truncated"] = bool(np.array_equal(
                LD["done"].astype(bool), OFFLINE["terminated"] | OFFLINE["truncated"])) if len(LD["done"]) == len(OFFLINE["terminated"]) else False
            LEGACY["dataset_identical_to_regenerated"] = same
            cnt = np.zeros((S_N, A_N))
            np.add.at(cnt, (LD["state"], LD["action"]), 1.0)
            _legacy_register("Behavior Cloning [masked greedy]", "fixed", one_hot_policy(masked_argmax(cnt)),
                             "masked argmax of original counts", "extraction", "released benchmark mask", False,
                             parent="LEGACY Behavior Cloning")
            norm_file = LEGACY_ROOT / "data" / "feature_normalizer.npz"
            if norm_file.exists():
                with np.load(norm_file) as z:
                    saved_mean, saved_std = z["mean"], z["std"]
                refit = FeatureNormalizer.fit(BENCH.state_features, LD["state"])
                LEGACY["saved_normalizer_vs_refit_max_abs_diff"] = float(max(np.abs(saved_mean - refit.mean).max(),
                                                                            np.abs(saved_std - refit.std).max()))
                legacy_x = ((BENCH.state_features - saved_mean) / saved_std).astype(np.float32)
                crit_rows = []
                for variant, stem in (("hard", "hard_cql"), ("deep", "deep_cql"), ("soft", "soft_cql")):
                    for seed in range(10):
                        f = LEGACY_ROOT / "checkpoints" / f"{stem}_seed{seed}.pt"
                        if not f.exists():
                            continue
                        pl = load_torch_payload(f, map_location=DEVICE)
                        net = TwinDuelingCQL(BENCH.feature_dim, A_N, 256).to(DEVICE)
                        net.load_state_dict(pl["model"])
                        q1, q2 = critic_tables(net, legacy_x)
                        qmin = np.minimum(q1, q2)
                        exported = REGISTRY.get({"hard": "LEGACY AM-CQL (Hard)", "deep": "LEGACY Deep CQL",
                                                 "soft": "LEGACY AM-CQL (Soft)"}[variant])
                        re_pi = greedy_policy(qmin, masked=(variant == "hard"))
                        agree = (float((re_pi[LIVE].argmax(1) == exported.by_seed[seed][LIVE].argmax(1)).mean())
                                 if exported is not None and seed in exported.by_seed else float("nan"))
                        crit_rows.append({"variant": variant, "seed": seed,
                                          "q_adm_max_abs": float(np.abs(qmin[LIVE][SUPPORT_MASK[LIVE]]).max()),
                                          "q_adm_max": float(qmin[LIVE][SUPPORT_MASK[LIVE]].max()),
                                          "action_agreement_with_exported_policy": agree,
                                          "final_logged_td_loss": float(pd.DataFrame(pl["log"])["td_loss"].iloc[-1]),
                                          "final_logged_cql_loss": float(pd.DataFrame(pl["log"])["cql_loss"].iloc[-1]),
                                          "final_lambda": float(pl["meta"].get("final_lambda", np.nan))})
                        del net
                LEGACY["critic_replay"] = crit_rows
                pd.DataFrame(crit_rows).to_csv(TABLE_DIR / "legacy_critic_diagnostics.csv", index=False)
    add_projection_controls(["legacy"])
    finalize_projection_checks()
    LEGACY_RESULTS = evaluate_registered([n for n in REGISTRY if n.startswith("LEGACY ")])
    LEGACY_RESULTS = LEGACY_RESULTS[LEGACY_RESULTS["group"] == "legacy"]
    if legacy_saved is not None:
        for _, row in legacy_saved.iterrows():
            nm = f"LEGACY {row['model']}"
            seed = FIXED if str(row["seed"]) in ("fixed", "oracle") else int(row["seed"])
            hit = LEGACY_RESULTS[(LEGACY_RESULTS["name"] == nm) & (LEGACY_RESULTS["seed"] == seed)
                                 & (LEGACY_RESULTS["rule"] == row["strategy"])]
            if len(hit):
                replay_rows.append({"model": row["model"], "seed": row["seed"], "rule": row["strategy"],
                                    "saved_return": row["return"], "replayed_return": float(hit["return"].iloc[0]),
                                    "abs_diff_return": abs(row["return"] - float(hit["return"].iloc[0])),
                                    "abs_diff_inadmissible_rate": abs(row["inadmissible_rate"] - float(hit["inadmissible_rate"].iloc[0])),
                                    "abs_diff_length": abs(row["avg_episode_length"] - float(hit["avg_episode_length"].iloc[0]))})
        LEGACY_REPLAY = pd.DataFrame(replay_rows)
        LEGACY_REPLAY.to_csv(TABLE_DIR / "legacy_policy_reevaluation.csv", index=False)
        if len(LEGACY_REPLAY):
            LEGACY["replayed_rows"] = int(len(LEGACY_REPLAY))
            LEGACY["max_abs_discrepancy"] = float(LEGACY_REPLAY[["abs_diff_return", "abs_diff_inadmissible_rate",
                                                                  "abs_diff_length"]].max().max())
    LEGACY_SUMMARY = summarize_results(LEGACY_RESULTS)
    LEGACY_SUMMARY.to_csv(TABLE_DIR / "legacy_projection_controls_summary.csv", index=False)
    show = LEGACY_SUMMARY[LEGACY_SUMMARY["name"].isin([
        "LEGACY Tabular IQL", "LEGACY Tabular IQL [projected]", "LEGACY Tabular IQL [masked greedy]",
        "LEGACY Behavior Cloning [masked greedy]", "LEGACY Tabular CQL [projected]", "LEGACY Deep CQL [projected]",
        "LEGACY AM-CQL (Hard)", "LEGACY Diffusion-AM-CQL"])]
    display(show[["name", "rule", "n_seeds", "return_mean", "return_sd", "inadmissible_rate_mean"]].round(5))
    if "LEGACY Diffusion-AM-CQL" in REGISTRY and "LEGACY Tabular IQL [projected]" in REGISTRY:
        common = sorted(set(REGISTRY["LEGACY Diffusion-AM-CQL"].by_seed) & set(REGISTRY["LEGACY Tabular IQL [projected]"].by_seed))
        dif = [EVALUATOR.evaluate(REGISTRY["LEGACY Diffusion-AM-CQL"].by_seed[s], "terminate")["return"]
               - EVALUATOR.evaluate(REGISTRY["LEGACY Tabular IQL [projected]"].by_seed[s], "terminate")["return"] for s in common]
        LEGACY["original_diffusion_minus_projected_iql_terminate_per_seed"] = dif
        LEGACY["original_diffusion_minus_projected_iql_terminate_mean"] = float(np.mean(dif)) if dif else None
    save_json(LEGACY, TABLE_DIR / "legacy_replay.json")
    print(json.dumps({k: v for k, v in LEGACY.items() if k != "critic_replay"}, indent=2, default=_json_default))
else:
    print("Legacy artifact directory not found (or replay disabled):", LEGACY_ROOT)
    save_json(LEGACY, TABLE_DIR / "legacy_replay.json")

## 22. Inference ablations, guidance sweep, diffusion necessity, schedule length, and cost accounting (audit M06, C04)

Reproduces the original study's Tables 6–7 from code (for both the legacy critic and the repaired primary critic) and
adds the direct categorical alternatives the audit requires. All comparisons here are descriptive (mean ± SD
over seeds and per-seed paired differences); inferential statistics are in Section 30.

In [ ]:
# ============================================================
# 22. Inference ablations / diffusion necessity / cost
# ============================================================
EXECLOG.mark("22. Inference ablations and M06")


def refresh_results() -> None:
    global RESULTS, SUMMARY
    RESULTS = evaluate_registered()
    SUMMARY = summarize_results(RESULTS)
    RESULTS.to_csv(TABLE_DIR / "exact_results_all.csv", index=False)
    SUMMARY.to_csv(TABLE_DIR / "exact_results_summary.csv", index=False)


def paired_frame(a: str, b: str, rule: str, metric: str = "return", df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    df = RESULTS if df is None else df
    x = df[(df["name"] == a) & (df["rule"] == rule) & (df["status"] == "ok")].set_index("seed")[metric]
    y = df[(df["name"] == b) & (df["rule"] == rule) & (df["status"] == "ok")].set_index("seed")[metric]
    if FIXED in y.index and len(y) == 1:
        y = pd.Series(float(y.iloc[0]), index=x.index)
    common = sorted(set(x.index) & set(y.index))
    return pd.DataFrame({"seed": common, "a": x.loc[common].values, "b": y.loc[common].values,
                         "diff": (x.loc[common] - y.loc[common]).values})


def ablation_table(reference: str, rows: Sequence[Tuple[str, str]]) -> pd.DataFrame:
    out = []
    for label, name in rows:
        sub = SUMMARY[(SUMMARY["name"] == name)]
        if sub.empty:
            continue
        rec = {"control": label, "policy": name, "mask_source": sub["mask_source"].iloc[0],
               "mask_use": sub["mask_use"].iloc[0], **mask_columns(sub["mask_use"].iloc[0]),
               "information": sub["information"].iloc[0]}
        for rule in RULES:
            r = sub[sub["rule"] == rule]
            if r.empty:
                continue
            pf = paired_frame(name, reference, rule)
            rec[f"return_{rule}"] = float(r["return_mean"].iloc[0])
            rec[f"return_{rule}_sd"] = float(r["return_sd"].iloc[0]) if pd.notna(r["return_sd"].iloc[0]) else np.nan
            rec[f"delta_J_vs_reference_{rule}"] = float(pf["diff"].mean()) if len(pf) else np.nan
            rec[f"delta_J_sd_{rule}"] = float(pf["diff"].std(ddof=1)) if len(pf) > 1 else np.nan
            rec[f"100U_{rule}"] = 100.0 * float(r["inadmissible_rate_mean"].iloc[0])
        out.append(rec)
    return pd.DataFrame(out)


refresh_results()

TABLE6_LEGACY = ablation_table("Diffusion-AM-CQL [legacy critic]", [
    ("Original diffusion, g = 1.5 (legacy critic)", "Diffusion-AM-CQL [legacy critic]"),
    ("No critic guidance, g = 0", "Diffusion [masked, no guidance (g=0)]"),
    ("No diffusion: masked critic ensemble", "Masked critic ensemble [legacy critic + tabular CQL]"),
    ("Hard critic only", "Diffusion [legacy hard critic only]"),
    ("Tabular critic only", "Diffusion [tabular CQL critic only]"),
    ("Stochastic final policy", "Diffusion-AM-CQL [legacy critic] [stochastic final]"),
    ("Deep CQL with inference mask", "Deep CQL [masked greedy]"),
    ("Tabular CQL with inference mask", "Tabular CQL [masked greedy]"),
    ("Projected tabular IQL (C01 control)", "Tabular IQL [projected]"),
    ("Masked greedy tabular IQL (C01 control)", "Tabular IQL [masked greedy]"),
])
TABLE6_PRIMARY = ablation_table(PRIMARY_DIFFUSION, [
    ("Primary diffusion, g = 1.5 (critic B)", PRIMARY_DIFFUSION),
    ("No critic guidance, g = 0", "Diffusion [masked, no guidance (g=0)]"),
    ("No diffusion: masked critic ensemble", "Masked critic ensemble [critic B + tabular CQL]"),
    ("Neural-only guidance (critic B)", "Diffusion [critic B only]"),
    ("Tabular-only guidance", "Diffusion [tabular CQL critic only]"),
    ("Stochastic final policy", f"{PRIMARY_DIFFUSION} [stochastic final]"),
    ("Initialisation uniform over 25", f"{PRIMARY_DIFFUSION} [init uniform over 25]"),
    ("Guidance at final step only", f"{PRIMARY_DIFFUSION} [guidance at final step only]"),
    ("Prior, final-step mask, no guidance", "Diffusion prior [final-step mask, no guidance]"),
    ("Prior x critic at final step only", "Diffusion prior x critic B [final-step mask + guidance]"),
    ("Unmasked unguided prior (stochastic)", "Diffusion prior [unmasked, unguided, stochastic]"),
    ("Unmasked unguided prior (greedy)", "Diffusion prior [unmasked, unguided, greedy]"),
    ("Direct categorical: neural BC masked greedy", "Neural BC (matched capacity) [masked greedy]"),
    ("Direct categorical: AW imitation (IQL advantages) masked greedy", "AW categorical imitation [masked greedy]"),
    ("Direct categorical: AW imitation (critic-B scores) masked greedy", "AW categorical imitation [critic B + tabular CQL] [masked greedy]"),
    ("Neural BC prior x critic", "Neural BC prior x critic [critic B + tabular CQL]"),
    ("Tabular BC prior x critic", "Tabular BC prior x critic [critic B + tabular CQL]"),
    ("Projected tabular IQL (C01 control)", "Tabular IQL [projected]"),
    ("Masked greedy tabular IQL (C01 control)", "Tabular IQL [masked greedy]"),
    ("Diffusion with design-A critic", "Diffusion-AM-CQL [critic A]"),
    ("Diffusion with legacy critic", "Diffusion-AM-CQL [legacy critic]"),
])
TABLE7 = []
for g in CFG.guidance_grid:
    for critic_label, nm in (("legacy critic", f"Diffusion-AM-CQL [legacy critic] [g={g}]"),
                             ("critic B", f"{PRIMARY_DIFFUSION} [g={g}]")):
        sub = SUMMARY[(SUMMARY["name"] == nm) & (SUMMARY["rule"] == "terminate")]
        if len(sub):
            TABLE7.append({"guidance_g": g, "critic": critic_label, "mask_source": sub["mask_source"].iloc[0],
                           "mask_use": sub["mask_use"].iloc[0], "return_mean": float(sub["return_mean"].iloc[0]),
                           "return_sd": float(sub["return_sd"].iloc[0]),
                           "inadmissible_occupancy_pct": 100 * float(sub["inadmissible_rate_mean"].iloc[0])})
TABLE7 = pd.DataFrame(TABLE7)
TABLE6_LEGACY.to_csv(TABLE_DIR / "table6_inference_ablations_legacy_critic.csv", index=False)
TABLE6_PRIMARY.to_csv(TABLE_DIR / "table6_inference_ablations_primary_critic.csv", index=False)
TABLE7.to_csv(TABLE_DIR / "table7_guidance_sweep.csv", index=False)
pd.concat([TABLE6_LEGACY.assign(reference="legacy critic"), TABLE6_PRIMARY.assign(reference="critic B")]).to_csv(
    TABLE_DIR / "post_training_controls_all.csv", index=False)
display(TABLE6_PRIMARY.round(5))
display(TABLE7.round(5))

# ---------- Schedule length ablation (retrains the denoiser; ablation seeds) ----------
if EXPERIMENTS["diffusion_schedule_ablation"] and EXPERIMENTS["main_diffusion"]:
    for T_alt in CFG.diffusion_T_grid:
        sched = CategoricalDiffusion(A_N, T_alt, CFG.diffusion_beta_start, CFG.diffusion_beta_end).schedule_audit()
        for seed in CFG.ablation_seeds:
            prior = train_diffusion_prior(MAIN, seed, T=T_alt)
            DIFF_PRIORS[("main", seed, T_alt)] = {k: prior[k] for k in ("log", "meta", "label")}
            cache = ReverseCache(prior["model"], prior["diffusion"], MAIN.all_x)
            rB = get_critic(PREREG["primary_hard_critic"], MAIN, seed)
            G_B = critic_ensemble(np.minimum(rB["q1"], rB["q2"]), get_tab("cql", MAIN, seed)["q"])
            pi, tr = guided_reverse_policy(cache, G_B, G_PRIMARY)
            register_diffusion(f"{PRIMARY_DIFFUSION} [T={T_alt}]", seed, pi, tr, "schedule_ablation",
                               f"T={T_alt} (alpha_bar_T={sched['alpha_bar_T_initial_token_coefficient']:.4f})",
                               "hard_admrows + tabular CQL")
            del cache
    refresh_results()
    SCHEDULE_TABLE = SUMMARY[SUMMARY["name"].str.contains(r"\[T=", regex=True) | (SUMMARY["name"] == PRIMARY_DIFFUSION)]
    SCHEDULE_TABLE.to_csv(TABLE_DIR / "table_diffusion_schedule_length.csv", index=False)
    display(SCHEDULE_TABLE[["name", "rule", "n_seeds", "return_mean", "return_sd"]].round(5))

# ---------- Cost accounting (training cost includes upstream critics for guided policies) ----------
cost_rows = []
for (kind, bname, seed), r in TAB.items():
    if bname == "main":
        cost_rows.append({"component": f"tabular_{kind}", "seed": seed, "train_s": r["meta"].get("elapsed_s"),
                          "params": r["meta"].get("params"), "peak_gpu_mb": None})
for (key, bname, seed), r in CRITICS.items():
    if bname == "main":
        cost_rows.append({"component": f"critic_{key}", "seed": seed, "train_s": r["meta"].get("elapsed_s"),
                          "params": r["meta"].get("params"), "peak_gpu_mb": r["meta"].get("peak_gpu_mb")})
for (kind, bname, seed), r in PNETS.items():
    if bname == "main":
        cost_rows.append({"component": f"policy_net_{kind}", "seed": seed, "train_s": r["meta"].get("elapsed_s"),
                          "params": r["meta"].get("params"), "peak_gpu_mb": r["meta"].get("peak_gpu_mb")})
for (bname, seed, T_), r in DIFF_PRIORS.items():
    if bname == "main":
        cost_rows.append({"component": f"diffusion_prior_T{T_}", "seed": seed, "train_s": r["meta"].get("elapsed_s"),
                          "params": r["meta"].get("params"), "peak_gpu_mb": r["meta"].get("peak_gpu_mb")})
COST = pd.DataFrame(cost_rows)
COST.to_csv(TABLE_DIR / "cost_accounting_components.csv", index=False)
if len(COST):
    comp = COST.groupby("component").agg(train_s_mean=("train_s", "mean"), params=("params", "max"),
                                         peak_gpu_mb=("peak_gpu_mb", "max")).reset_index()
    def _t(c):
        v = comp.loc[comp["component"] == c, "train_s_mean"]
        return float(v.iloc[0]) if len(v) and pd.notna(v.iloc[0]) else 0.0
    def _p(c):
        v = comp.loc[comp["component"] == c, "params"]
        return int(v.iloc[0]) if len(v) and pd.notna(v.iloc[0]) else 0
    dcost = pd.DataFrame(DIFFUSION_COST) if DIFFUSION_COST else pd.DataFrame()
    pipeline = [
        {"policy": "Tabular IQL [projected]", "total_train_s": _t("tabular_iql"), "total_params": _p("tabular_iql"),
         "policy_construction_s": 0.0, "deployed": "lookup table"},
        {"policy": "Masked critic ensemble [critic B + tabular CQL]", "total_train_s": _t("critic_hard_admrows") + _t("tabular_cql"),
         "total_params": _p("critic_hard_admrows") + _p("tabular_cql"), "policy_construction_s": 0.0, "deployed": "lookup table"},
        {"policy": "Neural BC prior x critic [critic B + tabular CQL]",
         "total_train_s": _t("policy_net_bc") + _t("critic_hard_admrows") + _t("tabular_cql"),
         "total_params": _p("policy_net_bc") + _p("critic_hard_admrows") + _p("tabular_cql"),
         "policy_construction_s": 0.0, "deployed": "lookup table"},
        {"policy": "Neural BC (matched capacity) [masked greedy]", "total_train_s": _t("policy_net_bc"),
         "total_params": _p("policy_net_bc"), "policy_construction_s": 0.0, "deployed": "lookup table"},
        {"policy": "AW categorical imitation [masked greedy]", "total_train_s": _t("policy_net_aw") + _t("tabular_iql"),
         "total_params": _p("policy_net_aw") + _p("tabular_iql"), "policy_construction_s": 0.0, "deployed": "lookup table"},
        {"policy": "AW categorical imitation [critic B + tabular CQL] [masked greedy]",
         "total_train_s": _t("policy_net_awB") + _t("critic_hard_admrows") + _t("tabular_cql"),
         "total_params": _p("policy_net_awB") + _p("critic_hard_admrows") + _p("tabular_cql"),
         "policy_construction_s": 0.0, "deployed": "lookup table"},
        {"policy": PRIMARY_DIFFUSION,
         "total_train_s": _t(f"diffusion_prior_T{CFG.diffusion_T}") + _t("critic_hard_admrows") + _t("tabular_cql"),
         "total_params": _p(f"diffusion_prior_T{CFG.diffusion_T}") + _p("critic_hard_admrows") + _p("tabular_cql"),
         "policy_construction_s": float((dcost["cache_s"] + dcost["construct_s"]).mean()) if len(dcost) else np.nan,
         "deployed": "lookup table"},
    ]
    COST_SUMMARY = pd.DataFrame(pipeline)
    COST_SUMMARY["deployed_lookup_latency_us"] = float(dcost["lookup_us"].mean()) if len(dcost) else np.nan
    COST_SUMMARY["on_demand_full_reverse_chain_ms_per_state"] = np.where(
        COST_SUMMARY["policy"] == PRIMARY_DIFFUSION,
        float(dcost["ondemand_ms"].dropna().mean()) if len(dcost) and "ondemand_ms" in dcost else np.nan, np.nan)
    PIPE = {"Tabular IQL [projected]": ["tabular_iql"],
            "Masked critic ensemble [critic B + tabular CQL]": ["critic_hard_admrows", "tabular_cql"],
            "Neural BC prior x critic [critic B + tabular CQL]": ["policy_net_bc", "critic_hard_admrows", "tabular_cql"],
            "Neural BC (matched capacity) [masked greedy]": ["policy_net_bc"],
            "AW categorical imitation [masked greedy]": ["policy_net_aw", "tabular_iql"],
            "AW categorical imitation [critic B + tabular CQL] [masked greedy]": ["policy_net_awB", "critic_hard_admrows", "tabular_cql"],
            PRIMARY_DIFFUSION: [f"diffusion_prior_T{CFG.diffusion_T}", "critic_hard_admrows", "tabular_cql"]}
    gpu = pd.to_numeric(comp.set_index("component")["peak_gpu_mb"], errors="coerce")
    COST_SUMMARY["peak_gpu_mb_max_component"] = [gpu.reindex(PIPE.get(p, [])).max() for p in COST_SUMMARY["policy"]]
    COST_SUMMARY["policy_construction_memory_mb"] = np.where(
        COST_SUMMARY["policy"] == PRIMARY_DIFFUSION, CFG.diffusion_T * S_N * A_N * A_N * 8 / 2 ** 20, 0.0)
    COST_SUMMARY.to_csv(TABLE_DIR / "cost_accounting.csv", index=False)
    comp.to_csv(TABLE_DIR / "cost_accounting_component_summary.csv", index=False)
    display(comp)
    display(COST_SUMMARY)
    print("Note: the deployed diffusion policy is a precomputed 716-row lookup table; the reverse chain is a "
          "construction cost, not a per-decision latency.")

## 23. Hard-critic stability study (audit C02)

Equal training budget (same steps, optimiser, batch, minibatch stream and initial weights) on the ablation
seeds. Compared: legacy objective (failure-mode control), design A, design B (both normalisations), retaining
vs excluding inadmissible logged transitions from TD regression, squared vs Huber TD, conservative weight
α ∈ {0.3, 1, 3}, a separately labelled TD-target clipping ablation of the legacy objective (stabilisation only,
not a repair), and — only if enabled — a separately labelled discounted (γ = 0.99) estimand.

The "empirically stable" flag is a declared diagnostic criterion (final max |Q| over admissible actions ≤ 5,
growth of that quantity over the last third of training ≤ 1.5×, Bellman RMSE ≤ 1 on the rows of the TD
objective — all logged rows, or admissible-logged rows only for the TD-admissible ablations; the full-buffer RMSE
and the admissible/inadmissible-row split are reported alongside); it is an
empirical statement about these runs, not a convergence guarantee.

In [ ]:
# ============================================================
# 23. Critic stability ablations
# ============================================================
EXECLOG.mark("23. Critic stability ablations (C02)")
R = dataclasses.replace
ABLATION_SPECS = [
    R(SPEC_HARD_B, key="hard_admrows_batchnorm", display="AM-CQL (Hard, B) [batch-normalised regulariser]", reg_normalization="batch", status="ablation"),
    R(SPEC_HARD_B, key="hard_admrows_tdadm", display="AM-CQL (Hard, B) [TD on admissible-logged rows only]", td_rows="admissible_only", status="ablation"),
    R(SPEC_HARD_A, key="hard_fullcql_tdadm", display="AM-CQL (Hard, A) [TD on admissible-logged rows only]", td_rows="admissible_only", status="ablation"),
    R(SPEC_HARD_B, key="hard_admrows_mse", display="AM-CQL (Hard, B) [squared TD]", td_loss="mse", status="ablation"),
    R(SPEC_HARD_A, key="hard_fullcql_mse", display="AM-CQL (Hard, A) [squared TD]", td_loss="mse", status="ablation"),
    R(SPEC_HARD_B, key="hard_admrows_alpha0.3", display="AM-CQL (Hard, B) [alpha=0.3]", alpha=0.3, status="ablation"),
    R(SPEC_HARD_B, key="hard_admrows_alpha3", display="AM-CQL (Hard, B) [alpha=3]", alpha=3.0, status="ablation"),
    R(SPEC_HARD_LEGACY, key="hard_legacy_mse", display="AM-CQL (Hard, legacy) [squared TD]", td_loss="mse", status="failure-mode ablation"),
    R(SPEC_HARD_LEGACY, key="hard_legacy_tdadm", display="AM-CQL (Hard, legacy) [TD on admissible-logged rows only]", td_rows="admissible_only", status="failure-mode ablation"),
    R(SPEC_HARD_LEGACY, key="hard_legacy_clip01", display="AM-CQL (Hard, legacy) [TD target clipped to [0,1]: stabilisation ablation, not a repair]", target_clip=(0.0, 1.0), status="stabilisation ablation"),
    R(SPEC_HARD_B, key="hard_admrows_regonly", display="AM-CQL (Hard, B-regulariser only: unmasked backup)", masked_backup=False, masked_extraction=False, status="placement ablation"),
]
if EXPERIMENTS["include_discounted_ablation"]:
    ABLATION_SPECS.append(R(SPEC_HARD_B, key="hard_admrows_gamma0.99",
                            display="AM-CQL (Hard, B) [gamma=0.99: discounted-survival estimand]", gamma=0.99,
                            status="different estimand"))

Q_STABLE_BOUND = 5.0


def stability_row(res: Dict[str, Any]) -> dict:
    diag, log = res["diag"], res["log"]
    final = diag.iloc[-1]
    steps = res["spec"].steps
    early = diag[diag["step"] <= int(round(2 * steps / 3))]
    ref = float(early["q_adm_max_abs"].iloc[-1]) if len(early) else float(diag["q_adm_max_abs"].iloc[0])
    growth = float(final["q_adm_max_abs"]) / max(ref, 1e-12)
    row = {"variant": res["spec"].key, "display": res["spec"].display, "status": res["spec"].status,
           "seed": res["seed"], "seed_set": "main" if res["spec"].key in SPEC_BY_KEY else "ablation",
           "final_q_adm_max_abs": float(final["q_adm_max_abs"]),
           "final_q_adm_q01": float(final["q_adm_q01"]), "final_q_adm_q99": float(final["q_adm_q99"]),
           "growth_last_third": growth,
           "final_bellman_rmse": float(final.get("bellman_residual_rmse_td_objective_rows", final["bellman_residual_rmse_full_buffer"])),
           "final_bellman_rmse_full_buffer": float(final["bellman_residual_rmse_full_buffer"]),
           "final_bellman_rmse_adm_logged_rows": float(final.get("bellman_residual_rmse_adm_logged_rows", np.nan)),
           "final_bellman_rmse_inadm_logged_rows": float(final.get("bellman_residual_rmse_inadm_logged_rows", np.nan)),
           "final_mean_state_spread": float(final["mean_state_spread"]),
           "final_offset_to_spread_ratio": float(final["median_offset_to_spread_ratio"]),
           "states_spread_within_64_ulp": int(final["states_spread_within_64_ulp_of_offset"]),
           "final_grad_norm_mean_last5": float(log["grad_norm"].tail(5).mean()),
           "final_td_loss": float(log["td_loss"].iloc[-1]), "final_cql_loss": float(log["cql_loss"].iloc[-1])}
    row["empirically_stable"] = bool(row["final_q_adm_max_abs"] <= Q_STABLE_BOUND and growth <= 1.5
                                     and row["final_bellman_rmse"] <= 1.0)
    return row


if EXPERIMENTS["critic_stability_ablations"] and EXPERIMENTS["main_neural_critics"]:
    run_critics(ABLATION_SPECS, MAIN, CFG.ablation_seeds, group="critic_ablation")
    refresh_results()
_abl_keys = {x.key for x in ABLATION_SPECS}
stab_rows = [stability_row(r) for (k, b, s), r in CRITICS.items() if b == "main" and len(r["diag"])
             and ((k in SPEC_BY_KEY and s in CFG.seeds) or (k in _abl_keys and s in CFG.ablation_seeds))]
CRITIC_STABILITY = pd.DataFrame(stab_rows)
if len(CRITIC_STABILITY):
    ret = RESULTS[(RESULTS["rule"] == "terminate") & (RESULTS["status"] == "ok")][["name", "seed", "return", "inadmissible_rate"]]
    CRITIC_STABILITY = CRITIC_STABILITY.merge(ret.rename(columns={"name": "display", "return": "terminate_return_native",
                                                                  "inadmissible_rate": "native_inadmissible_rate"}),
                                              on=["display", "seed"], how="left")
    CRITIC_STABILITY.to_csv(TABLE_DIR / "critic_stability_ablation.csv", index=False)
    STAB_MAIN_ALL = CRITIC_STABILITY[CRITIC_STABILITY["seed_set"] == "main"].groupby(["variant", "display"]).agg(
        n=("seed", "nunique"), final_q_adm_max_abs=("final_q_adm_max_abs", "median"),
        bellman_rmse=("final_bellman_rmse", "median"), stable_fraction=("empirically_stable", "mean")).reset_index()
    STAB_MAIN_ALL.to_csv(TABLE_DIR / "critic_stability_summary_main_all_seeds.csv", index=False)
    _prim = CRITIC_STABILITY[CRITIC_STABILITY["variant"] == PREREG["primary_hard_critic"]]
    CRITIC_STABILITY_GATE = {"primary_critic": PREREG["primary_hard_critic"], "seeds": sorted(_prim["seed"].tolist()),
                             "stable_on_all_seeds": bool(len(_prim) and _prim["empirically_stable"].all()),
                             "criterion": f"max|Q_adm| <= {Q_STABLE_BOUND}, growth <= 1.5x, TD-objective Bellman RMSE <= 1"}
    save_json(CRITIC_STABILITY_GATE, TABLE_DIR / "critic_stability_gate.json")
    print("Primary critic stability gate:", CRITIC_STABILITY_GATE)
    STAB_SUMMARY = CRITIC_STABILITY[CRITIC_STABILITY["seed"].isin(CFG.ablation_seeds)].groupby(["variant", "display", "status"]).agg(
        n=("seed", "nunique"), final_q_adm_max_abs=("final_q_adm_max_abs", "median"),
        growth_last_third=("growth_last_third", "median"), bellman_rmse=("final_bellman_rmse", "median"),
        stable_fraction=("empirically_stable", "mean"), terminate_return=("terminate_return_native", "mean"),
        terminate_return_sd=("terminate_return_native", "std")).reset_index()
    STAB_SUMMARY.to_csv(TABLE_DIR / "critic_stability_summary.csv", index=False)
    display(STAB_SUMMARY.round(4))
else:
    CRITIC_STABILITY_GATE = {}

# Constraint-placement table (audit §7): extraction, backup and regulariser placement under one architecture.
PLACEMENT_CELLS = [
    (False, False, "full", SPEC_DEEP.display),
    (True, False, "full", SPEC_DEEP.display + " [masked greedy]"),
    (False, True, "full", SPEC_HARD_A.display + " [unmasked greedy]"),
    (True, True, "full", SPEC_HARD_A.display),
    (False, False, "masked_admissible_rows", "AM-CQL (Hard, B-regulariser only: unmasked backup)"),
    (True, False, "masked_admissible_rows", "AM-CQL (Hard, B-regulariser only: unmasked backup) [masked greedy]"),
    (False, True, "masked_admissible_rows", SPEC_HARD_B.display + " [unmasked greedy]"),
    (True, True, "masked_admissible_rows", SPEC_HARD_B.display),
]
_okr = RESULTS[(RESULTS["status"] == "ok") & RESULTS["seed"].isin(CFG.ablation_seeds)]
_pl_rows = []
for ext, bk, reg, nm in PLACEMENT_CELLS:
    sub = _okr[_okr["name"] == nm]
    if sub.empty:
        continue
    row = {"extraction_masked": ext, "backup_masked": bk, "regulariser_domain": reg, "td_rows": "all", "policy": nm,
           "n_seeds": int(sub["seed"].nunique())}
    for rule in RULES:
        r = sub[sub["rule"] == rule]
        row[f"return_{rule}_mean"] = float(r["return"].mean())
        row[f"return_{rule}_sd"] = float(r["return"].std(ddof=1)) if len(r) > 1 else np.nan
        row[f"inadmissible_rate_{rule}"] = float(r["inadmissible_rate"].mean())
    _pl_rows.append(row)
CONSTRAINT_PLACEMENT = pd.DataFrame(_pl_rows)
CONSTRAINT_PLACEMENT.to_csv(TABLE_DIR / "table_constraint_placement.csv", index=False)
if len(CONSTRAINT_PLACEMENT):
    display(CONSTRAINT_PLACEMENT.round(5))

all_diag = [r["diag"].assign(variant=k, seed=s, dataset=b) for (k, b, s), r in CRITICS.items() if len(r["diag"])]
if all_diag:
    pd.concat(all_diag, ignore_index=True).to_csv(TABLE_DIR / "critic_diagnostics.csv", index=False)

## 24. Soft AM-CQL: training constraint versus exported policy (audit M05)

The soft penalty acts on a **mean-critic softmax at temperature T_p, under the logged-state minibatch
distribution**; the legacy export is an **unmasked minimum-critic argmax**. For finite logits the softmax
always assigns positive probability to inadmissible actions (numerical underflow is not structural zero
support), and a finite penalty does not guarantee that inadmissible actions rank below admissible ones.
The table reports, per extraction: surrogate mass under the training distribution, evaluated occupancy rate,
expected violations and first-violation probability, the primal residual (full logged-distribution surrogate − ε), complementary slackness λ·(g − ε), the dual residual (max |Δλ| over the last 10% of logged steps) and the final λ, the per-state
distribution of softmax inadmissible mass, mean/min critic disagreement, and greedy margins/ties.
The original study's statement that the soft variant's zero violations were "achieved by construction" is not supported by these runs.

In [ ]:
# ============================================================
# 24. Soft-constraint alignment analysis
# ============================================================
EXECLOG.mark("24. Soft constraint analysis (M05)")
SOFT_SENS_SPECS = [
    R(SPEC_SOFT, key="soft_eps0.01", display="AM-CQL (Soft) [tolerance eps=0.01]", soft_tolerance=0.01, status="sensitivity"),
    R(SPEC_SOFT, key="soft_T0.5", display="AM-CQL (Soft) [T_p=0.5]", soft_temperature=0.5, status="sensitivity"),
    R(SPEC_SOFT, key="soft_T2", display="AM-CQL (Soft) [T_p=2]", soft_temperature=2.0, status="sensitivity"),
]
if EXPERIMENTS["soft_sensitivity"] and EXPERIMENTS["main_neural_critics"]:
    run_critics(SOFT_SENS_SPECS, MAIN, CFG.ablation_seeds, group="soft_sensitivity")
    add_projection_controls(["soft_sensitivity", "critic_ablation"])
    finalize_projection_checks()
    refresh_results()

soft_rows = []
logged_states = MAIN.data["state"]
for (k, b, seed), res in CRITICS.items():
    spec = res["spec"]
    if b != "main" or not spec.soft:
        continue
    q1, q2 = res["q1"], res["q2"]
    logits = soft_policy_logits(torch.from_numpy(q1), torch.from_numpy(q2), spec.soft_aggregation).numpy() / spec.soft_temperature
    logits -= logits.max(axis=1, keepdims=True)
    sm = np.exp(logits)
    sm /= sm.sum(axis=1, keepdims=True)
    bmass = inadmissible_mass(sm)
    surrogate = float(bmass[logged_states].mean())
    log = res["log"]
    base = dict(variant=k, display=spec.display, seed=seed, tolerance=spec.soft_tolerance, T_p=spec.soft_temperature,
                surrogate_mass_training_distribution=surrogate,
                final_minibatch_surrogate=float(log["inadmissible_mass_surrogate"].iloc[-1]),
                dual_lambda_final=float(res["meta"]["final_lambda"]),
                dual_at_cap=bool(res["meta"]["final_lambda"] >= CFG.soft_lambda_max - 1e-9),
                softmax_state_mass_q50=float(np.quantile(bmass[LIVE], 0.5)),
                softmax_state_mass_q90=float(np.quantile(bmass[LIVE], 0.9)),
                softmax_state_mass_max=float(bmass[LIVE].max()),
                states_softmax_mass_above_1e_minus_6=int((bmass[LIVE] > 1e-6).sum()),
                mean_vs_min_greedy_disagreement=float(np.mean((q1[LIVE] + q2[LIVE]).argmax(1)
                                                              != np.minimum(q1, q2)[LIVE].argmax(1))))
    lam_hist = log["lambda"].to_numpy(dtype=float)
    tail = lam_hist[-max(2, int(math.ceil(0.1 * len(lam_hist)))):]
    base.update(primal_residual_final=surrogate - spec.soft_tolerance,
                primal_residual_last_minibatch=float(log["inadmissible_mass_surrogate"].iloc[-1]) - spec.soft_tolerance,
                complementary_slackness=float(res["meta"]["final_lambda"]) * (surrogate - spec.soft_tolerance),
                dual_residual_max_abs_dlambda_last10pct=float(np.abs(np.diff(tail)).max()) if len(tail) > 1 else np.nan)
    for suffix in ["", "[masked greedy]", "[aligned greedy: mean critic]", "[aligned stochastic: training softmax]",
                   "[aligned stochastic: training softmax] [projected]", "[aligned stochastic: training softmax] [mask-renormalized]"]:
        nm = f"{spec.display}{(' ' + suffix) if suffix else ''}"
        for rule in RULES:
            hit = RESULTS[(RESULTS["name"] == nm) & (RESULTS["seed"] == seed) & (RESULTS["rule"] == rule)]
            if len(hit):
                h = hit.iloc[0]
                soft_rows.append({**base, "extraction": suffix or "legacy: unmasked min-critic greedy", "rule": rule,
                                  "return": h["return"], "evaluated_occupancy_rate": h["inadmissible_rate"],
                                  "expected_violations": h["expected_violations"],
                                  "prob_any_violation": h["prob_any_violation"]})
SOFT_ANALYSIS = pd.DataFrame(soft_rows)
SOFT_ANALYSIS.to_csv(TABLE_DIR / "soft_constraint_analysis.csv", index=False)
MARGINS = pd.DataFrame(MARGIN_ROWS)
MARGINS.to_csv(TABLE_DIR / "greedy_margin_tie_report.csv", index=False)
if len(SOFT_ANALYSIS):
    display(SOFT_ANALYSIS.groupby(["display", "extraction", "rule"]).agg(
        n=("seed", "nunique"), surrogate=("surrogate_mass_training_distribution", "mean"),
        occupancy=("evaluated_occupancy_rate", "mean"), p_any=("prob_any_violation", "mean"),
        ret=("return", "mean"), lam=("dual_lambda_final", "mean")).reset_index().round(5))
if len(MARGINS):
    display(MARGINS.groupby("policy").agg(min_margin=("min_margin_admissible_minus_inadmissible", "min"),
                                          inadm_first=("states_inadmissible_ranked_first", "mean"),
                                          ties=("states_tied_admissible_vs_inadmissible", "mean")).reset_index())

## 25. Released mask versus support estimated from the simulated training data (audit M02)

Two different objects: the released mask encodes source-data transition support (count > 20 in the
original MIMIC-derived table); counts in 10,000 *simulated* clinician episodes answer a different question,
so an action can be frequent here yet unsupported in the source data (the 11.87% logged inadmissible rate is
expected, not a paradox or an action-code bug). This experiment gives learners **only** an estimated mask
(no released-mask access) and evaluates violations against the released mask. The matched-information
comparison in which *every* policy receives the released mask is the projection / masked-greedy set of
Section 19. Estimated masks do not reconstruct the original clinical evidence.

In [ ]:
# ============================================================
# 25. Support estimation experiment
# ============================================================
EXECLOG.mark("25. Support estimation (M02)")
EST_MASK_RULES = [("count_at_least", 1), ("count_at_least", 5), ("count_at_least", 21),
                  ("behavior_ratio_at_least", 0.1), ("behavior_ratio_at_least", 0.3)]
EST_REPORTS = []
EST_MASKS = {}
for rule, param in EST_MASK_RULES:
    est, rep = estimated_support_mask(MAIN.counts, rule, param)
    EST_MASKS[(rule, param)] = est
    EST_REPORTS.append(rep)
SUPPORT_ESTIMATION = pd.DataFrame(EST_REPORTS)
SUPPORT_ESTIMATION.to_csv(TABLE_DIR / "support_estimation_mask_comparison.csv", index=False)
display(SUPPORT_ESTIMATION)

if EXPERIMENTS["support_estimation"] and EXPERIMENTS["main_tabular"]:
    for (rule, param), est in EST_MASKS.items():
        tag = f"est:{rule}>={param}"
        src = f"dataset-estimated ({rule} >= {param})"
        for seed in CFG.seeds:
            cql, iql = get_tab("cql", MAIN, seed), get_tab("iql", MAIN, seed)
            register_policy(f"Tabular CQL [masked greedy, {tag}]", seed, one_hot_policy(masked_argmax(cql["q"], est)),
                            family="Tabular CQL", extraction="masked greedy on an estimated mask", stochastic=False,
                            mask_source=src, mask_use="extraction", information="logged transitions only",
                            group="support_estimation")
            w = np.where(est, iql["aw_scores"], -np.inf)
            act = w.argmax(axis=1)
            none = ~(np.where(est, iql["aw_scores"], 0.0) > 0).any(axis=1)
            act[none] = masked_argmax(np.broadcast_to(MAIN.counts.sum(axis=0).astype(float), iql["q"].shape), est)[none]
            register_policy(f"Tabular IQL [masked greedy, {tag}]", seed, one_hot_policy(act), family="Tabular IQL",
                            extraction="AW-score argmax on an estimated mask", stochastic=False, mask_source=src,
                            mask_use="extraction", information="logged transitions only", group="support_estimation")
            register_policy(f"Tabular IQL [projected onto {tag}]", seed,
                            project_to_support(REGISTRY["Tabular IQL"].by_seed[seed], est), family="Tabular IQL",
                            extraction="uniform redistribution onto an estimated mask (not mean-equivalent)",
                            stochastic=True, mask_source=src, mask_use="projection", information="logged transitions only",
                            group="support_estimation")
        register_policy(f"Behavior Cloning [masked greedy, {tag}]", FIXED, one_hot_policy(masked_argmax(MAIN.counts.astype(float), est)),
                        family="BC", extraction="count argmax on an estimated mask", stochastic=False, mask_source=src,
                        mask_use="extraction", information="logged counts only", group="support_estimation")
    if EXPERIMENTS["main_diffusion"]:
        for seed in CFG.seeds:
            prior = train_diffusion_prior(MAIN, seed)
            cache = ReverseCache(prior["model"], prior["diffusion"], MAIN.all_x)
            for key in [("count_at_least", 21), ("behavior_ratio_at_least", 0.3)]:
                est = EST_MASKS[key]
                G = critic_ensemble(get_tab("cql", MAIN, seed)["q"], mask=est)
                pi, tr = guided_reverse_policy(cache, G, G_PRIMARY, mask=est)
                register_diffusion(f"Diffusion [tabular CQL critic, est:{key[0]}>={key[1]}]", seed, pi, None,
                                   "support_estimation", "guided reverse chain with an estimated mask",
                                   "tabular CQL", mask_source=f"dataset-estimated ({key[0]} >= {key[1]})")
            del cache
            gc.collect()
    refresh_results()
    SUPPORT_EST_RESULTS = SUMMARY[SUMMARY["group"] == "support_estimation"]
    SUPPORT_EST_RESULTS.to_csv(TABLE_DIR / "support_estimation_policy_results.csv", index=False)
    display(SUPPORT_EST_RESULTS[["name", "rule", "n_seeds", "return_mean", "inadmissible_rate_mean",
                                 "prob_any_violation_mean"]].round(5))

## 26. Selection protocol: episode split, training-only normalisation, FQE selector validation, locked fresh evaluation (audit M04)

* Disclosure: the main-study configurations are fixed a priori (Section 1; Table 3 of the original study); exact-evaluation results of the
  previous run were visible during development, and the guidance sweep is exploratory. Every method in the
  main study therefore has a tuning budget of **one** configuration (see `tuning_budget.csv`).
* Offline selection demonstration (tabular families, equal budget of three configurations each; the
  training-only normaliser fit is recorded for protocol completeness, the tabular candidates do not use it): complete
  episodes are split 80/20 with a seeded stream (no trajectory is shared across partitions); the feature
  normaliser is fitted on training episodes only; candidates are trained on the training episodes; a
  **tabular model-based FQE** estimate on the development episodes selects one configuration per family.
  FQE is a candidate selector, not a guaranteed oracle, so its absolute error, rank correlation with exact
  returns and top-1 regret are reported. The selected configurations are then **locked** and retrained on a
  fresh, independently generated dataset for a final untouched exact evaluation. Holding out trajectories
  from the same MDP does not create an independent hospital.

In [ ]:
# ============================================================
# 26. Selection protocol
# ============================================================
EXECLOG.mark("26. Selection protocol (M04)")
TUNING_BUDGET = pd.DataFrame([
    {"method": m, "configurations_tried_main_study": 1, "selection_criterion": "fixed a priori (Table 3 settings)",
     "exact_results_visible_during_development": "yes (previous run); disclose in paper"}
    for m in ["Behavior Cloning", "Tabular CQL", "Tabular IQL", "Tabular BCQ", "Deep CQL", "AM-CQL (Hard, legacy)",
              "AM-CQL (Hard, design A)", "AM-CQL (Hard, design B)", "AM-CQL (Soft)", "Neural BC", "AW imitation",
              "Diffusion prior + guidance (g=1.5 fixed; sweep exploratory)"]])
TUNING_BUDGET.to_csv(TABLE_DIR / "tuning_budget.csv", index=False)


def empirical_mdp_evaluator(dev: Dict[str, np.ndarray], fallback: Dict[str, np.ndarray]) -> Tuple[ExactMDPEvaluator, dict]:
    """Tabular model-based FQE on dev episodes. Each (s, a) row uses dev counts if observed in dev, else the
    training split's counts for that (s, a) if observed there, else the state's action-pooled dev distribution
    (training-split pooled if the state is unseen in dev); states unseen in both splits are pessimistic
    absorbing-death rows."""
    def counts_of(d):
        c = np.zeros((S_N, A_N, S_N), dtype=np.float64)
        np.add.at(c, (d["state"], d["action"], d["next_state"]), 1.0)
        return c
    c_dev, c_tr = counts_of(dev), counts_of(fallback)
    P = np.zeros((S_N, A_N, S_N))
    stats_ = {"rows_from_dev": 0, "rows_from_train": 0, "rows_state_pooled": 0, "states_unobserved": 0}
    for s in np.flatnonzero(LIVE):
        sa_d, sa_t = c_dev[s].sum(axis=1), c_tr[s].sum(axis=1)
        base = c_dev[s] if sa_d.sum() > 0 else c_tr[s]
        if base.sum() == 0:
            P[s, :, BENCH.death_state] = 1.0
            stats_["states_unobserved"] += 1
            continue
        pooled = base.sum(axis=0) / base.sum()
        P[s] = np.where(sa_d[:, None] > 0, c_dev[s] / np.maximum(sa_d[:, None], 1),
                        np.where(sa_t[:, None] > 0, c_tr[s] / np.maximum(sa_t[:, None], 1), pooled[None, :]))
        stats_["rows_from_dev"] += int((sa_d > 0).sum())
        stats_["rows_from_train"] += int(((sa_d == 0) & (sa_t > 0)).sum())
        stats_["rows_state_pooled"] += int(((sa_d == 0) & (sa_t == 0)).sum())
    for t in BENCH.terminal_states:
        P[t, :, t] = 1.0
    d0 = np.bincount(dev["state"][dev["timestep"] == 0], minlength=S_N).astype(np.float64)
    d0 /= d0.sum()
    r_sa = P[:, :, BENCH.survival_state].copy()   # reward 1 on entering survival (verified in Section 3)
    r_sa[~LIVE] = 0.0
    del c_dev, c_tr
    ev = ExactMDPEvaluator(P, None, d0, SUPPORT_MASK, BENCH.terminal_states, BENCH.death_state,
                           BENCH.survival_state, r_sa=r_sa)
    return ev, stats_


def reward_entry_only_on_support() -> Dict[str, Any]:
    """ICU-Sepsis pays rewards on entering a state: on every possible transition out of a live state the reward is
    the entry reward of the destination. Absorbing self-loops are excluded: their reward never enters gamma = 1
    evaluation and is not destination-only in the released r_mat, so the all-rows flag of Section 3 is too strict
    for the model-based FQE below (which estimates r(s, a) = P_hat(survival | s, a) from data only)."""
    R = np.asarray(BENCH.base.dynamics["r_mat"], dtype=np.float64)
    entry = R[int(np.flatnonzero(LIVE)[0]), 0].copy()
    holds = True
    for s in np.flatnonzero(LIVE):      # row by row: no full-size temporaries
        sup = EVALUATOR.P[s] > 0
        holds &= bool(np.all(R[s][sup] == np.broadcast_to(entry, R[s].shape)[sup]))
    return {"holds_on_support": bool(holds), "rewarding_destinations": np.flatnonzero(entry != 0).tolist()}


SELECTION = {}
if EXPERIMENTS["selection_protocol"]:
    REWARD_SUPPORT_CHECK = reward_entry_only_on_support()
    assert REWARD_SUPPORT_CHECK["holds_on_support"] and \
        REWARD_SUPPORT_CHECK["rewarding_destinations"] == [BENCH.survival_state], REWARD_SUPPORT_CHECK
    rng = np_rng("selection", "episode-split", MAIN.data_hash)
    eps = np.unique(OFFLINE["episode_id"])
    dev_eps = rng.choice(eps, size=int(round(CFG.dev_fraction * len(eps))), replace=False)
    dev_rows = np.isin(OFFLINE["episode_id"], dev_eps)
    TRAIN_SPLIT = make_bundle("split_train", OFFLINE, OFFLINE_MANIFEST, rows=np.flatnonzero(~dev_rows))
    DEV_DATA = subset_rows(OFFLINE, np.flatnonzero(dev_rows))
    assert not set(np.unique(TRAIN_SPLIT.data["episode_id"])) & set(np.unique(DEV_DATA["episode_id"]))
    FQE_EVAL, FQE_MODEL_STATS = empirical_mdp_evaluator(DEV_DATA, TRAIN_SPLIT.data)
    sel_seed = CFG.seeds[0]
    grid = ([("Tabular CQL", {"alpha": a}) for a in (0.3, 1.0, 3.0)]
            + [("Tabular IQL", {"expectile": e, "beta": b}) for e, b in ((0.7, 1.0), (0.7, 3.0), (0.9, 3.0))]
            + [("Tabular BCQ", {"tau": t}) for t in (0.1, 0.3, 0.5)])

    def train_candidate(method: str, hp: dict, bundle: DataBundle, seed: int) -> Dict[str, np.ndarray]:
        if method == "Tabular CQL":
            r = train_tabular_cql(bundle, seed, alpha=hp["alpha"], tag="select_cql")
            return {"native": greedy_policy(r["q"], False), "masked greedy": greedy_policy(r["q"], True)}
        if method == "Tabular IQL":
            r = train_tabular_iql(bundle, seed, expectile=hp["expectile"], beta=hp["beta"], tag="select_iql")
            return {"native": r["deterministic"], "masked greedy": r["masked_greedy"]}
        r = train_tabular_bcq(bundle, seed, tau=hp["tau"], tag="select_bcq")
        return {"native": r["deterministic"], "masked greedy": r["masked_greedy"]}

    cand_rows = []
    for method, hp in grid:
        pols = train_candidate(method, hp, TRAIN_SPLIT, sel_seed)
        pols["projected"] = project_to_support(pols["native"])
        for extraction, pi in pols.items():
            for rule in RULES:
                ex = safe_evaluate(pi, rule)
                exact = ex["return"] if ex is not None else np.nan
                fq = safe_evaluate(pi, rule, evaluator=FQE_EVAL)
                fqe = fq["return"] if fq is not None else np.nan
                cand_rows.append({"method": method, "hparams": json.dumps(hp), "extraction": extraction, "rule": rule,
                                  "fqe_dev_return": fqe, "exact_return": exact, "abs_error": abs(fqe - exact)})
    CANDIDATES = pd.DataFrame(cand_rows)
    CANDIDATES.to_csv(TABLE_DIR / "selection_candidates_fqe_vs_exact.csv", index=False)
    endpoint_rule = PREREG["primary_endpoint"]["rule"]
    sel_rows = []
    for method in CANDIDATES["method"].unique():
        sub = CANDIDATES[(CANDIDATES["method"] == method) & (CANDIDATES["rule"] == endpoint_rule)].dropna(subset=["fqe_dev_return"])
        if sub.empty:
            continue
        chosen = sub.loc[sub["fqe_dev_return"].idxmax()]
        best = sub.loc[sub["exact_return"].idxmax()]
        rho = float(stats.spearmanr(sub["fqe_dev_return"], sub["exact_return"])[0]) if len(sub) > 2 else np.nan
        sel_rows.append({"method": method, "selected_hparams": chosen["hparams"], "selected_extraction": chosen["extraction"],
                         "selected_exact_return_on_split_train": chosen["exact_return"],
                         "best_exact_return_in_grid": best["exact_return"], "top1_regret": best["exact_return"] - chosen["exact_return"],
                         "fqe_mean_abs_error": float(sub["abs_error"].mean()), "fqe_spearman_vs_exact": rho})
    SELECTED = pd.DataFrame(sel_rows)
    # Locked evaluation on a fresh, independently generated dataset.
    fresh_spec = DatasetSpec(n_episodes=CFG.dataset_episodes, seed=CFG.fresh_eval_dataset_seed, max_steps=CFG.max_episode_steps)
    fresh_data, fresh_manifest = load_or_build_dataset(fresh_spec)
    FRESH = make_bundle(f"fresh{CFG.fresh_eval_dataset_seed}", fresh_data, fresh_manifest)
    locked = []
    for _, row in SELECTED.iterrows():
        hp = json.loads(row["selected_hparams"])
        default_hp = {"Tabular CQL": {"alpha": CFG.tab_cql_alpha},
                      "Tabular IQL": {"expectile": CFG.iql_expectile, "beta": CFG.iql_beta},
                      "Tabular BCQ": {"tau": CFG.bcq_threshold}}[row["method"]]
        for label, h in (("selected", hp), ("main-study default", default_hp)):
            vals = []
            for seed in CFG.ablation_seeds:
                pols = train_candidate(row["method"], h, FRESH, seed)
                pols["projected"] = project_to_support(pols["native"])
                ev_ = safe_evaluate(pols[row["selected_extraction"]], endpoint_rule)
                vals.append(ev_["return"] if ev_ is not None else np.nan)
            locked.append({"method": row["method"], "config": label, "hparams": json.dumps(h),
                           "extraction": row["selected_extraction"], "rule": endpoint_rule,
                           "fresh_dataset_return_mean": float(np.mean(vals)),
                           "fresh_dataset_return_sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
                           "n_seeds": len(vals)})
    LOCKED = pd.DataFrame(locked)
    SELECTED.to_csv(TABLE_DIR / "selection_fqe_selected.csv", index=False)
    LOCKED.to_csv(TABLE_DIR / "selection_locked_fresh_evaluation.csv", index=False)
    SELECTION = {"dev_episodes": int(len(dev_eps)), "train_episodes": int(len(eps) - len(dev_eps)),
                 "normalizer_fit_on": "training episodes only", "normalizer_hash": TRAIN_SPLIT.normalizer.hash,
                 "fqe": ("tabular empirical-model evaluation on dev episodes; (s,a) unseen in dev use training-split "
                         "counts, else the state's pooled dev distribution; states unseen in both are absorbing-death rows"),
                 "fqe_model_rows": FQE_MODEL_STATS,
                 "n_candidate_evaluations_improper_in_fqe_model": int(CANDIDATES["fqe_dev_return"].isna().sum()),
                 "note": "the training-only normaliser fit is recorded for protocol completeness; tabular candidates do not use features",
                 "stages": ["episode split (seeded, 80/20, complete episodes)", "normaliser fit on training episodes",
                            "train 3 configurations per tabular family on the training split (equal budget)",
                            "tabular FQE on dev episodes selects one configuration + extraction per family",
                            "lock the selected configuration",
                            f"retrain on fresh dataset seed {CFG.fresh_eval_dataset_seed} and evaluate exactly (untouched)"],
                 "endpoint_rule": endpoint_rule, "fresh_dataset": fresh_manifest["file"]}
    save_json(SELECTION, TABLE_DIR / "selection_protocol.json")
    display(SELECTED.round(5))
    display(LOCKED.round(5))
    del FQE_EVAL
    gc.collect()

## 27. Robustness: independent datasets × training seeds, smaller datasets, behaviour mixtures (audit M03, M07)

Independent clinician datasets are generated with different dataset seeds; training seeds are nested within
datasets. The full profile uses **8 datasets × 2 training seeds** (`CFG.replication_dataset_seeds`,
`CFG.replication_train_seeds`): 16 trained runs per method, but only **8 independent units** — the dataset is the
unit of the hierarchical analysis (Section 30), so a dataset-level two-sided sign-flip test can reach
p = 2/2⁸ ≈ 0.008. The lite profile (5 × 2) cannot reject two-sided at 0.05 (minimum p = 2/2⁵ = 0.0625); it is a
development setting. Spending the budget on datasets rather than on training seeds per dataset is deliberate:
the between-dataset variance is the component that a single frozen dataset cannot reveal. The primary dataset
reuses the main-study checkpoints automatically through identical run identities. All of this measures
variability *within one simulator*; it does not measure uncertainty in the simulator itself (Part II addresses
benchmark-construction error on simulators with known ground truth).

In [ ]:
# ============================================================
# 27. Replication and data regimes
# ============================================================
EXECLOG.mark("27. Replication and data regimes (M03/M07)")
REPLICATION_BUNDLES: Dict[int, DataBundle] = {}
REPL_METHODS = ["Tabular CQL", "Tabular CQL [masked greedy]", "Tabular IQL", "Tabular IQL [masked greedy]",
                "Tabular IQL [projected]", f"Tabular BCQ (tau={CFG.bcq_threshold})", "Behavior Cloning [masked greedy]",
                "Masked critic ensemble [critic B + tabular CQL]", "AM-CQL (Hard, design B: admissible-row CQL)", PRIMARY_DIFFUSION]


def rep_prefix(ds: int) -> str:
    return "" if ds == CFG.dataset_seed else f"[D{ds}] "


if EXPERIMENTS["replication"]:
    for ds in CFG.replication_dataset_seeds:
        if ds == CFG.dataset_seed:
            bundle = MAIN
        else:
            spec = DatasetSpec(n_episodes=CFG.dataset_episodes, seed=ds, max_steps=CFG.max_episode_steps)
            data, manifest = load_or_build_dataset(spec)
            bundle = make_bundle(f"rep{ds}", data, manifest)
        REPLICATION_BUNDLES[ds] = bundle
        grp = "main" if ds == CFG.dataset_seed else "replication"
        pre = rep_prefix(ds)
        run_tabular_suite(bundle, CFG.replication_train_seeds, group=grp, prefix=pre, dataset_label=f"dataset {ds}")
        bcp = train_behavior_cloning(bundle)
        register_policy(f"{pre}Behavior Cloning [masked greedy]", FIXED, bcp["masked_greedy"], group=grp,
                        dataset=f"dataset {ds}", family="BC", extraction="argmax of counts over admissible actions",
                        stochastic=False, mask_source="released benchmark mask", mask_use="extraction",
                        information="logged counts + released mask")
        if EXPERIMENTS["replication_include_neural"]:
            run_critics([SPEC_HARD_B], bundle, CFG.replication_train_seeds, group=grp, prefix=pre,
                        dataset_label=f"dataset {ds}")
            for seed in CFG.replication_train_seeds:
                run_diffusion_suite(bundle, seed, group=grp, prefix=pre, full_ablations=False, dataset_label=f"dataset {ds}")
        if ds != CFG.dataset_seed:
            bundle.release()
    add_projection_controls(["replication", "main"])
    finalize_projection_checks(verbose=False)
    refresh_results()

REGIME_ROWS = []
if EXPERIMENTS["data_regimes"]:
    # Independent dataset draws per reduced size (not prefixes of the main dataset).
    regimes = [(f"n={n} draw{k}", DatasetSpec(n_episodes=n, seed=int(stream_seed("dataset", "regime", n, k) % (2 ** 31 - 1)),
                                              max_steps=CFG.max_episode_steps))
               for n in CFG.data_size_regimes for k in range(CFG.regime_dataset_draws)]
    regimes += [(f"behaviour={b}", DatasetSpec(n_episodes=CFG.dataset_episodes, seed=CFG.dataset_seed,
                                               max_steps=CFG.max_episode_steps, behavior=b)) for b in CFG.behavior_mixtures]
    for label, spec in regimes:
        data, manifest = load_or_build_dataset(spec)
        bundle = make_bundle(f"regime_{slug(label)}", data, manifest)
        run_tabular_suite(bundle, CFG.regime_train_seeds, group="data_regime", prefix=f"[{label}] ", dataset_label=label)
        bcp = train_behavior_cloning(bundle)
        register_policy(f"[{label}] Behavior Cloning [masked greedy]", FIXED, bcp["masked_greedy"], group="data_regime",
                        dataset=label, family="BC", extraction="argmax of counts over admissible actions", stochastic=False,
                        mask_source="released benchmark mask", mask_use="extraction", information="logged counts + released mask")
        REGIME_ROWS.append({"regime": label, "dataset_seed": spec.seed, "behavior": spec.behavior,
                            "n_episodes": spec.n_episodes, **validate_dataset(data, spec)})
        bundle.release()
    add_projection_controls(["data_regime"])
    finalize_projection_checks(verbose=False)
    refresh_results()
    REGIMES = pd.DataFrame(REGIME_ROWS)
    REGIMES.to_csv(TABLE_DIR / "data_regime_datasets.csv", index=False)
    reg_sum = SUMMARY[SUMMARY["group"] == "data_regime"]
    reg_sum.to_csv(TABLE_DIR / "data_regime_results.csv", index=False)
    # Per-size summary separating dataset-draw variance from training-seed variance.
    ok_r = RESULTS[(RESULTS["status"] == "ok") & (RESULTS["group"] == "data_regime")]
    size_rows = []
    for n in CFG.data_size_regimes:
        labs = [f"n={n} draw{k}" for k in range(CFG.regime_dataset_draws)]
        methods = sorted({nm.split("] ", 1)[1] for nm in ok_r["name"].unique() if any(nm.startswith(f"[{l}] ") for l in labs)})
        for method in methods:
            for rule in RULES:
                per = [ok_r[(ok_r["name"] == f"[{l}] {method}") & (ok_r["rule"] == rule)]["return"].to_numpy(dtype=float) for l in labs]
                per = [v for v in per if len(v)]
                if not per:
                    continue
                means = np.array([v.mean() for v in per])
                w = [v.var(ddof=1) for v in per if len(v) > 1]
                within = float(np.mean(w)) if w else np.nan
                vm = float(means.var(ddof=1)) if len(per) > 1 else np.nan
                k_h = len(per) / sum(1.0 / len(v) for v in per)
                size_rows.append({"n_episodes": n, "method": method, "rule": rule, "n_draws": len(per),
                                  "mean_of_draw_means": float(means.mean()), "variance_of_draw_means": vm,
                                  "mean_within_draw_variance": within,
                                  "between_draw_variance_component_mom": (max(0.0, vm - within / k_h)
                                                                          if np.isfinite(vm) and np.isfinite(within) else np.nan)})
    REGIME_SIZE_SUMMARY = pd.DataFrame(size_rows)
    REGIME_SIZE_SUMMARY.to_csv(TABLE_DIR / "data_regime_size_summary.csv", index=False)
    display(reg_sum[["name", "rule", "n_seeds", "return_mean", "return_sd", "inadmissible_rate_mean"]].round(5))

## 28. MDP stress tests (locked policies) and severity / support subgroup diagnostics (audit M07)

Policies trained on the original data are evaluated without retraining on perturbed MDPs:
(i) log-normal perturbation of admissible transition rows with inadmissible rows recomputed as the mean of
admissible rows (mean-rule semantics preserved); (ii) a SOFA-tilted initial-state distribution;
(iii) changed admissible sets (removing or adding one action in a random subset of states). These are
**stress tests, not confidence sets**: no calibrated source-count uncertainty is available. Ranking
persistence is summarised by Kendall τ against the unperturbed ranking. Subgroup diagnostics report
exact returns conditional on the initial state's SOFA quartile and support-set size; they are simulation
analyses over state-centroid groups, not patient cohorts. Immediate termination is a benchmark stress rule,
not a prognosis for an unsupported intervention. Under the mean rule, adding an admissible action whose row is the
mean-imputed row leaves the dynamics unchanged, so `mask_add_one` is reported for the terminate rule only.

In [ ]:
# ============================================================
# 28. Stress tests + subgroup diagnostics
# ============================================================
EXECLOG.mark("28. Stress tests and subgroups (M07)")
STRESS_POLICIES = [n for n in MAIN_TABLE_POLICIES if n in REGISTRY and "Oracle" not in n]


def recompute_mean_rows(P: np.ndarray, mask: np.ndarray) -> None:
    for s in np.flatnonzero(LIVE):
        m = mask[s]
        if (~m).any():
            P[s, ~m] = P[s, m].mean(axis=0)


def perturbed_evaluator(kind: str, param: float, seed: int) -> ExactMDPEvaluator:
    rng = np_rng("stress", kind, param, seed)
    P = EVALUATOR.P.copy()
    d0 = D0.copy()
    mask = SUPPORT_MASK.copy()
    if kind == "transition_lognormal":
        noise = np.exp(param * rng.standard_normal(P.shape))
        live_rows = LIVE[:, None] & mask
        P[live_rows] = P[live_rows] * noise[live_rows]
        P[live_rows] /= P[live_rows].sum(axis=1, keepdims=True)
        recompute_mean_rows(P, mask)
    elif kind == "d0_sofa_tilt":
        z = (BENCH.sofa_scores - BENCH.sofa_scores[LIVE].mean()) / BENCH.sofa_scores[LIVE].std()
        d0 = d0 * np.exp(param * z)
        d0[~LIVE] = 0.0
        d0 /= d0.sum()
    elif kind == "mask_remove_one":
        for s in np.flatnonzero(LIVE & (mask.sum(axis=1) >= 2)):
            if rng.random() < param:
                mask[s, rng.choice(np.flatnonzero(mask[s]))] = False
        recompute_mean_rows(P, mask)
    elif kind == "mask_add_one":
        for s in np.flatnonzero(LIVE & (mask.sum(axis=1) < A_N)):
            if rng.random() < param:
                mask[s, rng.choice(np.flatnonzero(~mask[s]))] = True   # its row stays the mean-imputed estimate
        recompute_mean_rows(P, mask)
    else:
        raise ValueError(kind)
    r_sa = P[:, :, BENCH.survival_state].copy()
    r_sa[~LIVE] = 0.0
    return ExactMDPEvaluator(P, None, d0, mask, BENCH.terminal_states, BENCH.death_state, BENCH.survival_state, r_sa=r_sa)


STRESS_ROWS = []
if EXPERIMENTS["mdp_stress_tests"] and STRESS_POLICIES:
    perturbations = ([("transition_lognormal", s) for s in (0.1, 0.3)] + [("d0_sofa_tilt", k) for k in (-0.5, 0.5)]
                     + [("mask_remove_one", 0.2), ("mask_add_one", 0.1)])
    for kind, param in perturbations:
        pseeds = CFG.stress_seeds if kind != "d0_sofa_tilt" else (0,)
        for ps in pseeds:
            ev = perturbed_evaluator(kind, param, ps)
            for name in STRESS_POLICIES:
                for seed, pi in REGISTRY[name].by_seed.items():
                    for rule in RULES:
                        if kind == "mask_add_one" and rule == "mean":
                            continue   # tautological: the added action keeps the mean-imputed row, so P is unchanged
                        try:
                            m = ev.evaluate(pi, rule)
                            STRESS_ROWS.append({"perturbation": kind, "param": param, "perturbation_seed": ps,
                                                "policy": name, "seed": seed, "rule": rule, "return": m["return"],
                                                "inadmissible_rate": m["inadmissible_rate"],
                                                "prob_any_violation": m["prob_any_violation"]})
                        except (ImproperPolicyError, EvaluationInvariantError) as exc:
                            STRESS_ROWS.append({"perturbation": kind, "param": param, "perturbation_seed": ps,
                                                "policy": name, "seed": seed, "rule": rule, "return": np.nan,
                                                "inadmissible_rate": np.nan, "prob_any_violation": np.nan,
                                                "error": str(exc)})
            del ev
            gc.collect()
    STRESS = pd.DataFrame(STRESS_ROWS)
    STRESS.to_csv(TABLE_DIR / "stress_test_results.csv", index=False)
    base = SUMMARY[SUMMARY["name"].isin(STRESS_POLICIES)].set_index(["name", "rule"])["return_mean"]
    rank_rows = []
    for (kind, param, ps, rule), g in STRESS.groupby(["perturbation", "param", "perturbation_seed", "rule"]):
        pm = g.groupby("policy")["return"].mean()
        common = [n for n in pm.index if (n, rule) in base.index and np.isfinite(pm[n])]
        if len(common) < 3:
            continue
        tau = float(stats.kendalltau([base[(n, rule)] for n in common], [pm[n] for n in common])[0])
        top_base = max(common, key=lambda n: base[(n, rule)])
        top_pert = max(common, key=lambda n: pm[n])
        rank_rows.append({"perturbation": kind, "param": param, "perturbation_seed": ps, "rule": rule,
                          "kendall_tau_vs_unperturbed": tau, "top1_unchanged": top_base == top_pert,
                          "top1_unperturbed": top_base, "top1_perturbed": top_pert, "n_policies": len(common)})
    STRESS_RANKING = pd.DataFrame(rank_rows)
    STRESS_RANKING.to_csv(TABLE_DIR / "stress_test_ranking_persistence.csv", index=False)
    display(STRESS_RANKING.groupby(["perturbation", "param", "rule"]).agg(
        kendall_tau=("kendall_tau_vs_unperturbed", "mean"), top1_unchanged=("top1_unchanged", "mean")).reset_index().round(3))

# Subgroup diagnostics on the unperturbed MDP.
SUB_ROWS = []
init_states = np.flatnonzero(D0 > 0)
sofa_q = pd.qcut(pd.Series(BENCH.sofa_scores[init_states]).rank(method="first"), q=4,
                 labels=["Q1 (lowest SOFA)", "Q2", "Q3", "Q4 (highest SOFA)"])
k_init = SUPPORT_MASK[init_states].sum(axis=1)
k_group = np.where(k_init == 1, "k=1", np.where(k_init <= 4, "k=2-4", "k>=5"))
for name in STRESS_POLICIES:
    for seed, pi in REGISTRY[name].by_seed.items():
        for rule in RULES:
            m = safe_evaluate(pi, rule, return_vectors=True)
            if m is None:
                continue
            v = m["value_vector"][init_states]
            w = D0[init_states]
            for gname, labels in (("initial SOFA quartile", np.asarray(sofa_q, dtype=object)), ("initial support size", k_group)):
                for lab in pd.unique(labels):
                    sel = labels == lab
                    SUB_ROWS.append({"policy": name, "seed": seed, "rule": rule, "grouping": gname, "group": lab,
                                     "conditional_return": float((w[sel] * v[sel]).sum() / w[sel].sum()),
                                     "initial_mass": float(w[sel].sum())})
SUBGROUPS = pd.DataFrame(SUB_ROWS)
SUBGROUPS.to_csv(TABLE_DIR / "subgroup_diagnostics.csv", index=False)
if len(SUBGROUPS):
    display(SUBGROUPS.groupby(["grouping", "group", "rule", "policy"])["conditional_return"].mean().unstack("rule").head(40).round(4))

## 29. Representation ablation — removing treatment-related input features (audit M08)

The benchmark construction script defines the logged action from `input_4hourly` and `max_dose_vaso` of the
same 4-hour bloc **and** clusters states on all 47 features including those columns. Dropping neural input
columns therefore cannot remove information already embedded in the state IDs and dynamics; this ablation
only measures how much the neural models *rely on the explicit columns*. A clinically valid representation
would need lagged treatment variables and a rebuilt state clustering. Treatment leakage is a risk requiring
verification at patient level, not a confirmed patient-level leak.

In [ ]:
# ============================================================
# 29. Representation ablation
# ============================================================
EXECLOG.mark("29. Representation ablation (M08)")
REPR_BUNDLES = {}
if EXPERIMENTS["representation_ablation"] and EXPERIMENTS["main_tabular"]:
    for tag_name, drop in (("drop treatment-direct features", TREATMENT_FEATURES_DIRECT),
                           ("drop treatment-related features", TREATMENT_FEATURES_RELATED)):
        bundle = make_bundle(f"main_{slug(tag_name)}", OFFLINE, OFFLINE_MANIFEST, feature_drop=drop)
        REPR_BUNDLES[tag_name] = bundle
        pre = f"[{tag_name}] "
        run_critics([SPEC_HARD_B], bundle, CFG.ablation_seeds, group="representation", prefix=pre, dataset_label="main")
        for seed in CFG.ablation_seeds:
            bc_net = train_policy_net(bundle, seed, mode="bc")
            PNETS[("bc", bundle.name, seed)] = bc_net
            register_policy_net(bc_net, f"{pre}Neural BC (matched capacity)", seed, group="representation")
            if EXPERIMENTS["main_diffusion"]:
                run_diffusion_suite(bundle, seed, group="representation", prefix=pre, full_ablations=False, tab_bundle=MAIN)
        bundle.release()
    add_projection_controls(["representation"])
    finalize_projection_checks(verbose=False)
    refresh_results()
    REPR_RESULTS = SUMMARY[(SUMMARY["group"] == "representation") | (SUMMARY["name"].isin(
        ["AM-CQL (Hard, design B: admissible-row CQL)", "Neural BC (matched capacity) [masked greedy]", PRIMARY_DIFFUSION]))]
    REPR_RESULTS.to_csv(TABLE_DIR / "representation_ablation.csv", index=False)
    display(REPR_RESULTS[["name", "rule", "n_seeds", "return_mean", "return_sd", "inadmissible_rate_mean"]].round(5))

## 30. Statistics redesigned around the experimental unit (audit M03)

* **Primary analysis** (as preregistered): one comparison, one endpoint, two-sided exact sign-flip test on
  the mean paired difference, with the effect size, a percentile bootstrap interval (flagged as fragile for
  few pairs), the confidence set obtained by **inverting the sign-flip test**, a TOST equivalence test at the
  declared margin, and the probability of improvement. The sign-flip test assumes sign-exchangeability
  (symmetry) of the paired differences under the null; equal means alone do not guarantee it.
* **Secondary family:** Holm over the distinct hypotheses (comparator × rule). When the treatment and a
  comparator both have zero inadmissible mass, their mean- and terminate-rule paired differences coincide exactly,
  so that comparison is counted once (`rule_collapsed_comparators`, `family_size_distinct_hypotheses`).
* **Attainability:** with n pairs the smallest two-sided p is 2/2ⁿ (one-sided 1/2ⁿ). The full profile therefore
  uses 10 main seeds: the two-sided minimum is 2/2¹⁰ ≈ 0.002, so the primary test can reject at 0.05 and a first
  Holm rejection is attainable in families of up to 25 distinct hypotheses (2/2¹⁰ ≤ 0.05/25); the 8 replication
  datasets give a dataset-level minimum of 2/2⁸ ≈ 0.008. With 5 seeds (the lite profile and the original study) the two-sided minimum is 0.0625, so no two-sided test can reject at 0.05 and the inverted confidence
  set is unbounded — reported as such, never as "no difference". Seven pairs make a one-sided 5-comparison Holm
  rejection attainable and eight the two-sided analogue; that is a resolution fact, not a power analysis
  (`family_attainability` and `replication_planning.csv` report it for the realised design).
* **Hierarchical analysis** of the replication study: dataset-level paired summaries (training seeds nested
  within datasets), a cluster bootstrap over datasets, and a method-of-moments split into a between-dataset
  component (truncated at 0) and the within-dataset (training-seed) variance; per-method replication summaries.
  This is **estimation, not testing**: with 8 datasets the smallest two-sided p is 2/2⁸ ≈ 0.008, which exceeds
  0.05/7, so no Holm-adjusted rejection is attainable over the (comparator × rule) rows. Holm-adjusted p-values,
  the family size and attainability are reported for completeness, and the 8-cluster bootstrap is flagged as
  fragile (`cluster_bootstrap_fragile_small_n`).
* **Replication planning** from pilot variability of paired differences and the declared practical effect.
* The original study's one-sided analysis is reproduced for the legacy-critic diffusion policy only
  as a historical record: on the original five-seed artifacts, and on fresh retraining with the current main seeds.

In [ ]:
# ============================================================
# 30. Statistics
# ============================================================
EXECLOG.mark("30. Statistics (M03)")
refresh_results()
ALPHA = float(PREREG["alpha"])


_SIGN_CACHE: Dict[int, np.ndarray] = {}


def sign_patterns(n: int) -> np.ndarray:
    if n not in _SIGN_CACHE:
        codes = np.arange(2 ** n, dtype=np.int64)[:, None]
        _SIGN_CACHE[n] = 1.0 - 2.0 * ((codes >> np.arange(n)) & 1).astype(np.float64)
    return _SIGN_CACHE[n]


def sign_flip_test(d: np.ndarray, alternative: str = "two-sided", key: str = "", exact_max_n: int = 16,
                   mc_draws: int = 200_000) -> Dict[str, Any]:
    d = np.asarray(d, dtype=np.float64)
    n = len(d)
    if n < 2:
        return {"p_value": np.nan, "n_pairs": n, "method": "insufficient pairs"}
    if n <= exact_max_n:
        signs, method = sign_patterns(n), "exact enumeration"
    else:
        signs = np_rng("evaluation", "sign-flip", key, n).choice([-1.0, 1.0], size=(mc_draws, n))
        method = f"Monte Carlo ({mc_draws} draws)"
    null = signs @ d / n
    obs = d.mean()
    tol = 1e-12 * max(1.0, abs(obs))
    if alternative == "greater":
        cond = null >= obs - tol
    elif alternative == "less":
        cond = null <= obs + tol
    else:
        cond = np.abs(null) >= abs(obs) - tol
    # Exact enumeration includes the observed sign pattern; the Monte-Carlo estimate uses the +1 correction.
    p = float(cond.mean()) if n <= exact_max_n else float((1 + cond.sum()) / (1 + len(cond)))
    return {"p_value": p, "n_pairs": n, "method": method}


def min_attainable_p(n: int, alternative: str) -> float:
    return (1.0 if alternative != "two-sided" else 2.0) / (2 ** n)


def holm_adjust(p_values: Dict[str, float]) -> Dict[str, float]:
    """Holm step-down over the finite p-values; non-finite p-values are excluded (reported as NaN)."""
    valid = {k: float(v) for k, v in p_values.items() if v is not None and np.isfinite(v)}
    items = sorted(valid.items(), key=lambda kv: kv[1])
    m, running, out = len(items), 0.0, {}
    for rank, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - rank) * p))
        out[name] = running
    for k in p_values:
        out.setdefault(k, np.nan)
    return out


def bootstrap_ci(d: np.ndarray, key: str) -> Tuple[float, float]:
    d = np.asarray(d, dtype=np.float64)
    if len(d) < 2:
        return (np.nan, np.nan)
    idx = np_rng("evaluation", "bootstrap", key).integers(0, len(d), size=(CFG.bootstrap_samples, len(d)))
    return tuple(np.quantile(d[idx].mean(axis=1), [ALPHA / 2, 1 - ALPHA / 2]).tolist())


def sign_flip_confidence_set(d: np.ndarray, key: str) -> Tuple[float, float]:
    d = np.asarray(d, dtype=np.float64)
    if len(d) < 2 or min_attainable_p(len(d), "two-sided") > ALPHA:
        return (-np.inf, np.inf)
    span = max(d.max() - d.min(), 1e-6)
    grid = np.linspace(d.min() - 2 * span, d.max() + 2 * span, 601)
    accept = lambda g: sign_flip_test(d - g, "two-sided", key, mc_draws=20_000)["p_value"] > ALPHA
    keep = [g for g in grid if accept(g)]
    if not keep:
        return (np.nan, np.nan)
    lo, hi = min(keep), max(keep)

    def _bisect(rejected, accepted):
        for _ in range(50):
            mid = 0.5 * (rejected + accepted)
            rejected, accepted = (rejected, mid) if accept(mid) else (mid, accepted)
        return accepted

    i_lo, i_hi = int(np.searchsorted(grid, lo)), int(np.searchsorted(grid, hi))
    lo = _bisect(grid[i_lo - 1], lo) if i_lo > 0 else -np.inf
    hi = _bisect(grid[i_hi + 1], hi) if i_hi < len(grid) - 1 else np.inf
    return (lo, hi)


def tost(d: np.ndarray, margin: float, key: str) -> float:
    d = np.asarray(d, dtype=np.float64)
    if len(d) < 2:
        return np.nan
    p_lower = sign_flip_test(d + margin, "greater", key + "|lo")["p_value"]
    p_upper = sign_flip_test(d - margin, "less", key + "|hi")["p_value"]
    return float(max(p_lower, p_upper))


def probability_of_improvement(x: np.ndarray, y: np.ndarray) -> float:
    x, y = np.asarray(x), np.asarray(y)
    return float(np.mean(x[:, None] > y[None, :]) + 0.5 * np.mean(x[:, None] == y[None, :]))


def compare(a: str, b: str, rule: str, alternative: str = "two-sided", metric: str = "return",
            df: Optional[pd.DataFrame] = None) -> Dict[str, Any]:
    pf = paired_frame(a, b, rule, metric, df)
    d = pf["diff"].to_numpy()
    key = f"{a}|{b}|{rule}|{metric}"
    t = sign_flip_test(d, alternative, key)
    ci = bootstrap_ci(d, key)
    cs = sign_flip_confidence_set(d, key)
    n = len(d)
    return {"treatment": a, "comparator": b, "rule": rule, "metric": metric, "alternative": alternative,
            "treatment_mask_source": REGISTRY[a].mask_source, "treatment_mask_use": REGISTRY[a].mask_use,
            "comparator_mask_source": REGISTRY[b].mask_source, "comparator_mask_use": REGISTRY[b].mask_use,
            "signflip_confidence_unbounded": bool(not (np.isfinite(cs[0]) and np.isfinite(cs[1]))),
            "n_pairs": n, "seeds": pf["seed"].tolist(), "mean_difference": float(d.mean()) if n else np.nan,
            "sd_difference": float(d.std(ddof=1)) if n > 1 else np.nan,
            "bootstrap_ci_low": ci[0], "bootstrap_ci_high": ci[1], "bootstrap_fragile_small_n": bool(n < 10),
            "signflip_confidence_low": cs[0], "signflip_confidence_high": cs[1],
            "p_value": t["p_value"], "test_method": t.get("method"), "min_attainable_p": min_attainable_p(n, alternative) if n else np.nan,
            "rejection_attainable_at_alpha": bool(n and min_attainable_p(n, alternative) <= ALPHA),
            "tost_p": tost(d, PREREG["equivalence_margin"], key), "equivalence_margin": PREREG["equivalence_margin"],
            "prob_improvement": probability_of_improvement(pf["a"].to_numpy(), pf["b"].to_numpy()) if n else np.nan}


STATS = {"prereg_hash": obj_hash(PREREG), "prereg": PREREG}
pt, pc = PREREG["primary_treatment"], PREREG["primary_comparator"]
if pt in REGISTRY and pc in REGISTRY:
    STATS["primary"] = compare(pt, pc, PREREG["primary_endpoint"]["rule"], PREREG["alternative"])
    print(json.dumps(STATS["primary"], indent=2, default=_json_default))

fam_rows = []
for comp in PREREG["secondary_comparators"]:
    if comp in REGISTRY and pt in REGISTRY:
        for rule in PREREG["secondary_rules"]:
            fam_rows.append(compare(pt, comp, rule, PREREG["alternative"]))
FAMILY = pd.DataFrame(fam_rows)
if len(FAMILY):
    def _same_across_rules(c):
        a_ = paired_frame(pt, c, "mean")["diff"].to_numpy()
        b_ = paired_frame(pt, c, "terminate")["diff"].to_numpy()
        return len(a_) == len(b_) and np.array_equal(a_, b_)
    same = {c: _same_across_rules(c) for c in FAMILY["comparator"].unique()}
    FAMILY["hypothesis"] = [r.comparator if same[r.comparator] else f"{r.comparator}|{r.rule}" for r in FAMILY.itertuples()]
    uniq = FAMILY.drop_duplicates("hypothesis")
    adj = holm_adjust(dict(zip(uniq["hypothesis"], uniq["p_value"])))
    FAMILY["holm_adjusted_p_full_family"] = FAMILY["hypothesis"].map(adj)
    m = int(uniq["p_value"].notna().sum())
    FAMILY["family_size_distinct_hypotheses"] = m
    FAMILY["holm_excluded_nonfinite"] = int(uniq["p_value"].isna().sum())
    if FAMILY["holm_excluded_nonfinite"].iloc[0]:
        print("WARNING: some secondary comparisons have non-finite p-values (too few pairs) and were excluded from Holm.")
    n_needed = next(n for n in range(2, 64) if min_attainable_p(n, PREREG["alternative"]) <= ALPHA / max(m, 1))
    STATS["family_attainability"] = {"family_size": m, "pairs_needed_for_first_holm_rejection": n_needed,
                                     "pairs_available": int(FAMILY["n_pairs"].min()),
                                     "rule_collapsed_comparators": [c for c, v in same.items() if v]}
FAMILY.to_csv(TABLE_DIR / "statistics_secondary_family.csv", index=False)
display(FAMILY[["comparator", "rule", "hypothesis", "n_pairs", "mean_difference", "sd_difference", "bootstrap_ci_low",
                "bootstrap_ci_high", "p_value", "holm_adjusted_p_full_family", "tost_p", "prob_improvement"]].round(5)
        if len(FAMILY) else FAMILY)

# Historical reproduction of manuscript Table 5 (one-sided, Holm within each rule).
legacy_cmp = ["AM-CQL (Hard, legacy)", "AM-CQL (Soft, legacy)", "Tabular CQL", "Tabular IQL", "Deep CQL"]
hist_rows = []
for source, treat, comps, df in [
        ("fresh retraining", "Diffusion-AM-CQL [legacy critic]", legacy_cmp, None),
        ("original artifacts", "LEGACY Diffusion-AM-CQL", [f"LEGACY {c}" for c in
                                                          ["AM-CQL (Hard)", "AM-CQL (Soft)", "Tabular CQL", "Tabular IQL", "Deep CQL"]], None)]:
    if treat not in REGISTRY:
        continue
    for rule in RULES:
        rows = [compare(treat, c, rule, "greater", df=df) for c in comps if c in REGISTRY]
        if rows:
            adj = holm_adjust({r["comparator"]: r["p_value"] for r in rows})
            for r in rows:
                r["holm_within_rule"] = adj[r["comparator"]]
                r["source"] = source
            hist_rows.extend(rows)
HISTORICAL_TABLE5 = pd.DataFrame(hist_rows)
HISTORICAL_TABLE5.to_csv(TABLE_DIR / "statistics_historical_table5_one_sided.csv", index=False)

# Paired differences for every family comparison (publication requirement).
pd.concat([paired_frame(pt, c, rule).assign(treatment=pt, comparator=c, rule=rule)
           for c in PREREG["secondary_comparators"] + [pc] if c in REGISTRY and pt in REGISTRY
           for rule in RULES] or [pd.DataFrame()]).to_csv(TABLE_DIR / "paired_differences.csv", index=False)

# Hierarchical analysis over independent datasets (training seeds nested within datasets).
HIER_ROWS, NESTED_ROWS, REPL_ROWS = [], [], []
HIER_REF = None
if EXPERIMENTS["replication"] and REPLICATION_BUNDLES:
    n_diff = sum((rep_prefix(ds) + PRIMARY_DIFFUSION) in REGISTRY for ds in REPLICATION_BUNDLES)
    HIER_REF = PRIMARY_DIFFUSION if n_diff >= 2 else PREREG["primary_comparator"]
    for method in REPL_METHODS:
        if method == HIER_REF:
            continue
        for rule in RULES:
            per_ds, nested = [], []
            for ds in REPLICATION_BUNDLES:
                a, b = f"{rep_prefix(ds)}{HIER_REF}", f"{rep_prefix(ds)}{method}"
                if a not in REGISTRY or b not in REGISTRY:
                    continue
                pf = paired_frame(a, b, rule)
                pf = pf[pf["seed"].isin(CFG.replication_train_seeds)]
                if len(pf):
                    per_ds.append(float(pf["diff"].mean()))
                    nested.append(pf["diff"].to_numpy())
                    NESTED_ROWS.append(pf.assign(treatment=HIER_REF, comparator=method, rule=rule, dataset=ds))
            if len(per_ds) < 2:
                continue
            d = np.asarray(per_ds)
            t = sign_flip_test(d, "two-sided", f"hier|{method}|{rule}")
            # Cluster bootstrap: datasets resampled, training seeds kept intact within each dataset.
            boots = d[np_rng("evaluation", "hier-bootstrap", method, rule).integers(0, len(d), size=(CFG.bootstrap_samples, len(d)))].mean(axis=1)
            within_list = [x.var(ddof=1) for x in nested if len(x) > 1]
            within = float(np.mean(within_list)) if within_list else np.nan
            k_h = len(nested) / sum(1.0 / len(x) for x in nested)
            var_means = float(d.var(ddof=1))
            between = max(0.0, var_means - within / k_h) if np.isfinite(within) else np.nan
            HIER_ROWS.append({"treatment": HIER_REF, "comparator": method, "rule": rule,
                              "n_datasets": len(per_ds), "training_seeds_per_dataset": len(CFG.replication_train_seeds),
                              "mean_of_dataset_means": float(d.mean()), "sd_dataset_means": float(d.std(ddof=1)),
                              "cluster_bootstrap_ci_low": float(np.quantile(boots, ALPHA / 2)),
                              "cluster_bootstrap_ci_high": float(np.quantile(boots, 1 - ALPHA / 2)),
                              "dataset_level_signflip_p_two_sided": t["p_value"],
                              "min_attainable_p": min_attainable_p(len(per_ds), "two-sided"),
                              "variance_of_dataset_means": var_means, "mean_within_dataset_variance": within,
                              "between_dataset_variance_component_mom": between,
                              "icc_dataset": (between / (between + within) if np.isfinite(between) and (between + within) > 0 else np.nan)})
    # Per-method dataset / training-seed variance (independent of any reference policy).
    ok_ = RESULTS[RESULTS["status"] == "ok"]
    for method in REPL_METHODS:
        for rule in RULES:
            per = []
            for ds in REPLICATION_BUNDLES:
                s_ = ok_[(ok_["name"] == rep_prefix(ds) + method) & (ok_["rule"] == rule)
                         & ok_["seed"].isin(list(CFG.replication_train_seeds) + [FIXED])]
                if len(s_):
                    v = s_["return"].to_numpy(dtype=float)
                    per.append(v)
                    REPL_ROWS.append({"method": method, "rule": rule, "dataset_seed": ds, "n_seeds": len(v),
                                      "return_mean": float(v.mean()), "return_sd": float(v.std(ddof=1)) if len(v) > 1 else np.nan})
            if len(per) >= 2:
                means = np.array([v.mean() for v in per])
                w = [v.var(ddof=1) for v in per if len(v) > 1]
                within = float(np.mean(w)) if w else np.nan
                vm = float(means.var(ddof=1))
                k_h = len(per) / sum(1.0 / len(v) for v in per)
                REPL_ROWS.append({"method": method, "rule": rule, "dataset_seed": "pooled", "n_datasets": len(per),
                                  "mean_of_dataset_means": float(means.mean()), "variance_of_dataset_means": vm,
                                  "mean_within_dataset_variance": within,
                                  "between_dataset_variance_component_mom": (max(0.0, vm - within / k_h) if np.isfinite(within) else np.nan)})
HIERARCHICAL = pd.DataFrame(HIER_ROWS)
if len(HIERARCHICAL):
    # Estimation, not testing: with 8 datasets the smallest two-sided p is 2/2^8 = 0.0078 > 0.05/7, so no Holm-adjusted
    # rejection is attainable over these rows, and a cluster bootstrap over fewer than 10 datasets is fragile.
    _hkey = HIERARCHICAL["comparator"].astype(str) + "|" + HIERARCHICAL["rule"].astype(str)
    _hp = HIERARCHICAL["dataset_level_signflip_p_two_sided"]
    HIERARCHICAL["holm_p"] = _hkey.map(holm_adjust(dict(zip(_hkey, _hp))))
    _hm = int(np.isfinite(_hp.to_numpy(dtype=np.float64)).sum())
    HIERARCHICAL["family_size"] = _hm
    HIERARCHICAL["holm_first_step_attainable"] = HIERARCHICAL["min_attainable_p"] <= ALPHA / max(_hm, 1)
    HIERARCHICAL["cluster_bootstrap_fragile_small_n"] = HIERARCHICAL["n_datasets"] < 10
    HIERARCHICAL["analysis_role"] = "estimation (effect sizes with intervals; not a confirmatory test)"
HIERARCHICAL.to_csv(TABLE_DIR / "statistics_hierarchical_replication.csv", index=False)
(pd.concat(NESTED_ROWS, ignore_index=True) if NESTED_ROWS else pd.DataFrame()).to_csv(
    TABLE_DIR / "statistics_hierarchical_paired_differences.csv", index=False)
REPLICATION_RESULTS = pd.DataFrame(REPL_ROWS)
REPLICATION_RESULTS.to_csv(TABLE_DIR / "replication_results.csv", index=False)
if len(HIERARCHICAL):
    display(HIERARCHICAL.round(5))
if len(REPLICATION_RESULTS):
    display(REPLICATION_RESULTS[REPLICATION_RESULTS["dataset_seed"] == "pooled"].round(6))

# Replication planning (planning aid, not a universal seed rule).
plan_rows = []
z_b = stats.norm.ppf(0.8)
m_fam = int(STATS.get("family_attainability", {}).get("family_size", 1) or 1)


def _plan(r, label, z_a):
    sd = r["sd_difference"]
    if np.isfinite(sd) and sd > 0:
        plan_rows.append({"analysis": label, "comparator": r["comparator"], "rule": r["rule"],
                          "pilot_sd_paired_difference": sd, "practical_effect": PREREG["practical_effect"],
                          "alpha_used": float(2 * (1 - stats.norm.cdf(z_a))),
                          "approx_pairs_for_80pct_power_normal": int(math.ceil(((z_a + z_b) * sd / PREREG["practical_effect"]) ** 2)),
                          "min_pairs_attainable_single_two_sided": next(n for n in range(2, 64) if 2 / 2 ** n <= ALPHA),
                          "min_pairs_attainable_holm_family": STATS.get("family_attainability", {}).get("pairs_needed_for_first_holm_rejection"),
                          "unit": "training seeds on the frozen main dataset"})


if "primary" in STATS:
    _plan(STATS["primary"], "primary (unadjusted, two-sided)", stats.norm.ppf(1 - ALPHA / 2))
for r in fam_rows:
    _plan(r, "secondary (Bonferroni bound on the first Holm step)", stats.norm.ppf(1 - ALPHA / (2 * m_fam)))
REPLICATION_PLAN = pd.DataFrame(plan_rows)
REPLICATION_PLAN.to_csv(TABLE_DIR / "replication_planning.csv", index=False)
STATS["hierarchical_available"] = bool(len(HIERARCHICAL))
save_json(STATS, TABLE_DIR / "statistics_primary.json")

# Structural support gate (renamed from the earlier "safety" gate; benchmark-support terminology).
support_gate = {"primary_policy": PRIMARY_DIFFUSION, "seeds": list(CFG.seeds)}
for nm in [PRIMARY_DIFFUSION, "Diffusion-AM-CQL [legacy critic]", "AM-CQL (Hard, design B: admissible-row CQL)",
           "AM-CQL (Hard, legacy)", "Tabular IQL [projected]"]:
    sub = RESULTS[(RESULTS["name"] == nm) & (RESULTS["status"] == "ok")]
    if len(sub):
        support_gate[f"max_exact_inadmissible_rate::{nm}"] = float(sub["inadmissible_rate"].max())
support_gate["max_reverse_step_inadmissible_mass_primary"] = float(max(
    (max(v) for v in DIFFUSION_TRACES.get(PRIMARY_DIFFUSION, {}).values()), default=0.0))
support_gate["passes_structural_support_check"] = bool(all(
    v <= 1e-12 for k, v in support_gate.items() if k.startswith("max_")))
save_json({"support_gate": support_gate, "primary": STATS.get("primary"),
           "family": FAMILY.to_dict(orient="records") if len(FAMILY) else []},
          TABLE_DIR / "fresh_custom_model_performance_gate.json")
print(json.dumps(support_gate, indent=2))

## 31. Figures (original-study figures regenerated from code, plus new diagnostic figures)

In [ ]:
# ============================================================
# 31. Figures
# ============================================================
EXECLOG.mark("31. Figures")


def summary_value(name: str, rule: str, metric: str = "return"):
    r = SUMMARY[(SUMMARY["name"] == name) & (SUMMARY["rule"] == rule)]
    if r.empty:
        return np.nan, np.nan
    sd = r[f"{metric}_sd"].iloc[0]
    return float(r[f"{metric}_mean"].iloc[0]), (float(sd) if pd.notna(sd) else 0.0)


fig7_policies = [n for n in ["Behavior Cloning", "Tabular CQL", "Tabular IQL", f"Tabular BCQ (tau={CFG.bcq_threshold})",
                             "Deep CQL", "AM-CQL (Hard, legacy)", "AM-CQL (Hard, design A: full-action CQL)",
                             "AM-CQL (Hard, design B: admissible-row CQL)", "AM-CQL (Soft, legacy)",
                             "Diffusion-AM-CQL [legacy critic]", PRIMARY_DIFFUSION] if n in REGISTRY]
if fig7_policies:
    fig, axes = plt.subplots(1, 2, figsize=(13, 0.5 * len(fig7_policies) + 2))
    y = np.arange(len(fig7_policies))
    for j, rule in enumerate(RULES):
        vals = [summary_value(n, rule) for n in fig7_policies]
        axes[0].errorbar([v[0] for v in vals], y + (0.15 if j else -0.15), xerr=[v[1] for v in vals], fmt="o",
                         capsize=3, label=rule)
        axes[1].barh(y + (0.2 if j else -0.2), [100 * summary_value(n, rule, "inadmissible_rate")[0] for n in fig7_policies],
                     height=0.4, label=rule)
    for ax in axes:
        ax.set_yticks(y)
        ax.set_yticklabels(fig7_policies, fontsize=8)
        ax.invert_yaxis()
        ax.legend()
    axes[0].set_xlabel("Exact expected return (mean ± SD over seeds)")
    axes[0].set_title("(a) Return under two handling rules")
    axes[1].set_xlabel("Inadmissible decisions (% of occupancy)")
    axes[1].set_title("(b) Benchmark-support violations")
    save_figure(fig, FIG_DIR / "paper_fig7_main_policies")

# C01 projection figure: native vs projected vs masked greedy, both rules.
c01_families = [("Behavior Cloning", "Behavior Cloning [projected]", "Behavior Cloning [masked greedy]"),
                ("Tabular CQL", "Tabular CQL [projected]", "Tabular CQL [masked greedy]"),
                ("Tabular IQL", "Tabular IQL [projected]", "Tabular IQL [masked greedy]"),
                ("Deep CQL", "Deep CQL [projected]", "Deep CQL [masked greedy]"),
                ("Clinician", "Clinician [projected]", None)]
c01_families = [f for f in c01_families if f[0] in REGISTRY]
if c01_families:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    for j, rule in enumerate(RULES):
        ax = axes[j]
        x = np.arange(len(c01_families))
        for k, (lab, idx) in enumerate((("native", 0), ("projected", 1), ("masked greedy", 2))):
            vals = [summary_value(f[idx], rule) if f[idx] and f[idx] in REGISTRY else (np.nan, 0) for f in c01_families]
            ax.bar(x + (k - 1) * 0.27, [v[0] for v in vals], 0.27, yerr=[v[1] for v in vals], capsize=2, label=lab)
        d_val = summary_value(PRIMARY_DIFFUSION, rule)[0]
        if np.isfinite(d_val):
            ax.axhline(d_val, linestyle="--", color="k", linewidth=1, label="primary diffusion")
        ax.set_xticks(x)
        ax.set_xticklabels([f[0] for f in c01_families], rotation=20, ha="right")
        ax.set_title(f"Rule: {rule}")
        ax.set_ylim(0.70 if rule == "mean" else 0.35, 0.80)
    axes[0].set_ylabel("Exact expected return")
    axes[1].legend(fontsize=8)
    fig.suptitle("Admissible projection and masked extraction of baseline policies (audit C01)")
    save_figure(fig, FIG_DIR / "fig_c01_projection_controls")

if len(TABLE7):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for critic_label, g in TABLE7.groupby("critic"):
        axes[0].errorbar(g["guidance_g"], g["return_mean"], yerr=g["return_sd"], marker="o", capsize=3, label=critic_label)
    axes[0].set_xlabel("Guidance scale g")
    axes[0].set_ylabel("Exact expected return (terminate)")
    axes[0].set_title("(a) Post-training guidance sensitivity")
    axes[0].legend()
    t6 = TABLE6_PRIMARY.dropna(subset=["return_terminate"]) if "return_terminate" in TABLE6_PRIMARY else TABLE6_PRIMARY.iloc[0:0]
    axes[1].errorbar(t6["return_terminate"], np.arange(len(t6)), xerr=t6["return_terminate_sd"].fillna(0), fmt="o", capsize=3)
    axes[1].set_yticks(np.arange(len(t6)))
    axes[1].set_yticklabels(t6["control"], fontsize=7)
    axes[1].invert_yaxis()
    ref = summary_value(PRIMARY_DIFFUSION, "terminate")[0]
    axes[1].axvline(ref, linestyle="--", color="k", linewidth=1)
    axes[1].set_xlabel("Exact expected return (terminate)")
    axes[1].set_title("(b) Inference components and direct alternatives")
    save_figure(fig, FIG_DIR / "paper_fig8_guidance_and_components")

panels = [("Tabular CQL", "td_loss", "(a) Tabular CQL: squared TD"), ("Tabular IQL", "q_loss", "(b) Tabular IQL: squared Q"),
          ("Deep CQL", "td_loss", "(c) Deep CQL: twin Huber"), ("AM-CQL (Hard, legacy)", "td_loss", "(d) Hard legacy: twin Huber"),
          ("AM-CQL (Hard, design A: full-action CQL)", "td_loss", "(e) Hard design A"),
          ("AM-CQL (Hard, design B: admissible-row CQL)", "td_loss", "(f) Hard design B"),
          ("AM-CQL (Soft, legacy)", "td_loss", "(g) Soft: twin Huber"), ("Diffusion prior", "loss", "(h) Diffusion: denoising CE")]
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, (stem, col, title) in zip(axes.ravel(), panels):
    for seed in CFG.seeds:
        f = LOG_DIR / f"{slug(stem)}__seed{seed}.csv"
        if f.exists():
            df = pd.read_csv(f)
            if col in df:
                ax.plot(df["step"] / 1000, df[col], linewidth=0.8, alpha=0.8)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Updates (thousands)")
    ax.set_ylabel("Minibatch loss")
save_figure(fig, FIG_DIR / "paper_fig9_training_diagnostics")

soft_logs = [(seed, LOG_DIR / f"{slug('AM-CQL (Soft, legacy)')}__seed{seed}.csv") for seed in CFG.seeds]
if any(f.exists() for _, f in soft_logs):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for seed, f in soft_logs:
        if f.exists():
            df = pd.read_csv(f)
            axes[0].plot(df["step"] / 1000, df["lambda"], label=f"seed {seed}")
            axes[1].plot(df["step"] / 1000, 100 * df["inadmissible_mass_surrogate"], label=f"seed {seed}")
    axes[0].set_title("(a) Soft constraint multiplier λ")
    axes[1].set_title("(b) Minibatch softmax inadmissible mass (%)")
    for ax in axes:
        ax.set_xlabel("Updates (thousands)")
        ax.legend(fontsize=7)
    save_figure(fig, FIG_DIR / "paper_fig10_soft_diagnostics")

diag_path = TABLE_DIR / "critic_diagnostics.csv"
if diag_path.exists():
    dg = pd.read_csv(diag_path)
    dg = dg[dg["dataset"] == "main"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for key in ["hard_legacy", "hard_fullcql", "hard_admrows", "deep", "soft_legacy", "hard_legacy_clip01", "hard_admrows_tdadm"]:
        g = dg[dg["variant"] == key]
        if g.empty:
            continue
        m = g.groupby("step")[["q_adm_max_abs", "bellman_residual_rmse_full_buffer"]].median().reset_index()
        axes[0].plot(m["step"] / 1000, m["q_adm_max_abs"], marker="o", label=key)
        axes[1].plot(m["step"] / 1000, m["bellman_residual_rmse_full_buffer"], marker="o", label=key)
    for ax, t in zip(axes, ["(a) max |Q| over admissible actions (median over seeds)", "(b) Full-buffer Bellman RMSE"]):
        ax.set_yscale("log")
        ax.set_xlabel("Updates (thousands)")
        ax.set_title(t)
        ax.legend(fontsize=7)
    save_figure(fig, FIG_DIR / "fig_c02_critic_scale_drift")

if len(HIERARCHICAL):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    hh = HIERARCHICAL[HIERARCHICAL["rule"] == "terminate"]
    ax.errorbar(hh["mean_of_dataset_means"], np.arange(len(hh)),
                xerr=[(hh["mean_of_dataset_means"] - hh["cluster_bootstrap_ci_low"]).clip(lower=0),
                      (hh["cluster_bootstrap_ci_high"] - hh["mean_of_dataset_means"]).clip(lower=0)],
                fmt="o", capsize=3)
    ax.axvline(0, color="k", linewidth=1)
    ax.set_yticks(np.arange(len(hh)))
    ax.set_yticklabels(hh["comparator"], fontsize=8)
    ax.set_xlabel(f"{HIER_REF} minus comparator (terminate); cluster-bootstrap interval over datasets")
    ax.set_title("Replication across independent datasets (training seeds nested)")
    save_figure(fig, FIG_DIR / "fig_m03_hierarchical_replication")

if EXPERIMENTS["selection_protocol"] and "CANDIDATES" in globals():
    fig, ax = plt.subplots(figsize=(6, 5))
    for method, g in CANDIDATES[CANDIDATES["rule"] == PREREG["primary_endpoint"]["rule"]].groupby("method"):
        ax.scatter(g["exact_return"], g["fqe_dev_return"], label=method)
    lim = [np.nanmin(CANDIDATES[["exact_return", "fqe_dev_return"]].to_numpy()), np.nanmax(CANDIDATES[["exact_return", "fqe_dev_return"]].to_numpy())]
    ax.plot(lim, lim, "k--", linewidth=1)
    ax.set_xlabel("Exact return (true MDP)")
    ax.set_ylabel("Tabular FQE estimate (dev episodes)")
    ax.set_title("OPE selector validation (audit M04)")
    ax.legend(fontsize=8)
    save_figure(fig, FIG_DIR / "fig_m04_fqe_vs_exact")

if EXPERIMENTS["mdp_stress_tests"] and "STRESS_RANKING" in globals() and len(STRESS_RANKING):
    fig, ax = plt.subplots(figsize=(9, 4))
    sr = STRESS_RANKING.groupby(["perturbation", "param", "rule"])["kendall_tau_vs_unperturbed"].mean().reset_index()
    labels = [f"{r.perturbation}({r.param})\n{r.rule}" for r in sr.itertuples()]
    ax.bar(np.arange(len(sr)), sr["kendall_tau_vs_unperturbed"])
    ax.set_xticks(np.arange(len(sr)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("Kendall τ vs unperturbed ranking")
    ax.set_title("Ranking persistence under MDP stress tests (not confidence sets)")
    save_figure(fig, FIG_DIR / "fig_m07_stress_ranking")
print("Figures written to", FIG_DIR)

# Part II — Cross-environment exact study (cross-environment extension)

Part I (Sections 0–31) is a deep case study of one benchmark. Part II asks the question that a
reader needs answered before any ICU-Sepsis result can be interpreted:

> **Unsupported actions are a modelling choice.** Offline-RL benchmarks built from clinical records must say
> what happens when a policy takes an action that the data never (or rarely) supports. ICU-Sepsis replaces such
> an action by the mean of the supported ones ("mean" rule) or ends the episode ("terminate" rule). How does that
> choice change violation costs, optimal policies, and **method rankings** — and do benchmark rankings agree with
> the rankings on the true system that generated the data?

ICU-Sepsis cannot answer the last question, because its true dynamics for unsupported actions are unknown. Part II
therefore rebuilds an **ICU-Sepsis-style benchmark** (hospital data → count threshold τ → admissible mask →
estimated rows for supported pairs + a completion rule for the rest → benchmark users simulate datasets from it)
on top of environments whose **true dynamics are known exactly**:

| Environment family | True dynamics for unsupported actions | Role |
|---|---|---|
| ICU-Sepsis v2 (released benchmark) | unknown | the benchmark itself; mean / terminate rules; 50 user-dataset draws (draw 0 = the Part I dataset) |
| Gumbel-max sepsis simulator (Oberst & Sontag, ICML 2019), exact enumeration | known | ground-truth benchmark-validity study across hospital-data size and threshold τ |
| Garnet absorbing CMDPs with a harm parameter *h* for unsupported actions | known, controlled | phase diagram: when does the completion rule mislead? (*h* = 0: non-preferred unsupported actions are completed correctly; the semantics term vanishes where K(s) ⊆ M(s), see `table_error_decomposition.csv`) |

**Research questions.** RQ1 (theory, Section II.5): seven propositions on unsupported-action semantics, verified
numerically per proposition: P1–P3 and P5 on ICU-Sepsis (P1/P2 where the released mean-rule identity holds), the
draw-0 Gumbel and Garnet instances and 20 random absorbing MDPs; P4 on the Gumbel and Garnet instances and the random
MDPs (no ground truth for ICU-Sepsis); P7 on ICU-Sepsis and the draw-0 benchmark instances; P6 by the Part I
gradient tests (Section 13) and closed-form checks. RQ2 (benchmark validity): Kendall-τ rank fidelity, top-1
agreement and selection regret between benchmark and true values, with an exact error decomposition. RQ3 (methods): what masking,
projection, pessimism, Lagrangian constraints, support filtering, guided behaviour priors and exact categorical
diffusion buy over the trivial admissible projection, with rliable-style statistics over many dataset draws.

**Unit of analysis.** All Part II learners are deterministic functions of their training data, so the experimental
unit is the **dataset draw** (for Gumbel / Garnet: a full benchmark-construction replicate). Hyper-parameter grids
have equal size per tunable method; results are reported with pre-declared defaults, with selection by tabular FQE
on held-out episodes, and with oracle selection (upper bound, labelled).

Sections: II.1 setup and preregistration · II.2 core (environment object, evaluator factory, sampler, empirical
models, CMDP LP, cache) · II.3 environments and benchmark construction · II.4 method suite · II.5 propositions and
exact constrained frontiers · II.6 driver · II.7 statistics, figures and tables. Part II is placed before the
release sections (32–35) so that every output is packaged and mapped.

In [ ]:
# ============================================================
# II.1 Part II setup and preregistration
# ============================================================
EXECLOG.mark("II.1 Cross-environment setup")
import scipy.sparse as sp
from scipy.optimize import linprog

XENV_TABLE_DIR = TABLE_DIR / "xenv"
XENV_FIG_DIR = FIG_DIR / "xenv"
XENV_CACHE_DIR = ROOT / "xenv_cache"
for _p in (XENV_TABLE_DIR, XENV_FIG_DIR, XENV_CACHE_DIR):
    _p.mkdir(parents=True, exist_ok=True)


@dataclass
class XConfig:
    # Dataset draws per environment cell (the experimental unit of Part II).
    icu_draws: int = 50
    gumbel_draws: int = 30
    garnet_draws: int = 30
    # Benchmark-user datasets and the train / dev split used for offline selection.
    icu_user_episodes: int = 10_000
    gumbel_user_episodes: int = 5_000
    garnet_user_episodes: int = 2_000
    max_steps: int = 100
    dev_fraction: float = 0.2

    # Gumbel-max sepsis simulator (exact enumeration of the published simulator code).
    gumbel_variant: str = "sim-v2"                 # "sim-v2" (glucose-cap fix) | "master" (published notebooks)
    gumbel_timeout_hazard: float = 0.05            # geometric horizon: mean 20 decisions, stationary optimum
    # (construction, hospital episodes, support threshold tau, physician epsilon[, physician kind]); kind 'optimal'
    # (default) softens the true optimum, 'withhold-vaso' the optimum without vasopressors (a control in which
    # unsupported actions are beneficial, parallel to Garnet h < 0; see gumbel_physician).
    gumbel_cells: Tuple[tuple, ...] = (
        ("best", 2_000, 5, 0.05), ("best", 2_000, 20, 0.05), ("best", 10_000, 5, 0.05),
        ("best", 10_000, 20, 0.05), ("best", 10_000, 20, 0.20), ("best", 10_000, 20, 0.05, "withhold-vaso"),
        ("oracle", 10_000, 20, 0.05))
    gumbel_mc_pairs: int = 300                     # (state, action) pairs checked against the step sampler
    gumbel_mc_draws: int = 4_000

    # Garnet absorbing CMDP family.
    garnet_states: int = 150
    garnet_actions: int = 8
    garnet_branching: int = 4
    garnet_hazard: Tuple[float, float] = (0.05, 0.20)   # per-step absorption probability range
    garnet_harm: Tuple[float, ...] = (-0.5, 0.0, 0.5, 1.0)
    garnet_behavior_eps: Tuple[float, ...] = (0.05, 0.30)
    garnet_hosp_episodes: int = 5_000
    garnet_tau: int = 5
    # Decision states without an action reaching tau: "icu" removes them like ICU-Sepsis's create_valid_dynamics
    # (drop inflow, renormalise, iterate); "argmax" keeps them with their most frequent action (sensitivity run).
    benchmark_empty_rule: str = "icu"

    # Method grids (equal size for comparable tunable methods; *_default is the pre-declared configuration).
    cql_alpha_grid: Tuple[float, ...] = (0.3, 1.0, 3.0)
    cql_alpha_default: float = 1.0
    cql_steps: int = 10_000                        # full-batch Adam steps on the exact empirical CQL objective
    cql_lr: float = 0.02
    cql_target_every: int = 25                     # 400 hard target syncs (Bellman backups)
    cql_converged_tol: float = 5e-3                # max |dQ| on observed pairs over the last sync interval
    iql_expectile_grid: Tuple[float, ...] = (0.6, 0.7, 0.9)
    iql_expectile_default: float = 0.7
    iql_beta: float = 3.0
    iql_max_weight: float = 100.0
    bcq_tau_grid: Tuple[float, ...] = (0.1, 0.3, 0.5)
    bcq_tau_default: float = 0.3
    lcb_c_grid: Tuple[float, ...] = (0.03, 0.1, 0.3)
    lcb_c_default: float = 0.1
    mbsqi_b_grid: Tuple[int, ...] = (1, 5, 20)
    mbsqi_b_default: int = 5
    spibb_n_grid: Tuple[int, ...] = (5, 20, 50)
    spibb_n_default: int = 20
    lagrange_kappa_grid: Tuple[float, ...] = (0.0, 0.01, 0.05)
    lagrange_kappa_default: float = 0.0
    mblp_kappa_grid: Tuple[float, ...] = (0.0, 0.01, 0.05)
    mblp_kappa_default: float = 0.0
    sfbc_alpha_grid: Tuple[float, ...] = (1.0, 3.0, 10.0)
    sfbc_alpha_default: float = 3.0
    diffusion_guidance_grid: Tuple[float, ...] = (0.5, 1.5, 3.0)
    diffusion_guidance_default: float = 1.5
    max_iterations: int = 20_000
    tol: float = 1e-10

    # Exact constrained frontiers (expected violation budget kappa; inf = unconstrained).
    frontier_kappas: Tuple[float, ...] = (0.0, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, float("inf"))

    # Statistics.
    bootstrap: int = 5_000
    equivalence_margin_normalized: float = 0.05


def apply_xrun_mode(xc: XConfig, mode: str, profile: str) -> XConfig:
    if profile == "lite":
        xc.icu_draws, xc.gumbel_draws, xc.garnet_draws = 20, 15, 15
    if mode == "smoke":
        xc.icu_draws = xc.gumbel_draws = xc.garnet_draws = 2
        xc.icu_user_episodes = xc.gumbel_user_episodes = 300
        xc.garnet_user_episodes = 200
        xc.gumbel_cells = (("best", 300, 5, 0.05), ("oracle", 300, 5, 0.05))
        xc.gumbel_mc_pairs, xc.gumbel_mc_draws = 40, 500
        xc.garnet_states, xc.garnet_harm, xc.garnet_behavior_eps = 30, (0.0, 1.0), (0.05,)
        xc.garnet_hosp_episodes = 300
        xc.cql_steps, xc.cql_target_every = 200, 20
        xc.max_iterations = 5_000
        xc.frontier_kappas = (0.0, 0.1, float("inf"))
        xc.bootstrap = 300
    return xc


XCFG = apply_xrun_mode(XConfig(), RUN_MODE, COMPUTE_PROFILE)

# Canonical policy labels used by the preregistration, the driver and the statistics: f"{method} [{variant}]",
# variant in {native, masked, projected}; the label refers to the pre-declared default hyper-parameters.
XCOMPARATOR = "IQL [projected]"
XTREATMENTS = ["BC [masked]", "CQL [masked]", "IQL [masked]", "BCQ [masked]", "VI-LCB [masked]", "MBS-QI [masked]",
               "SPIBB [projected]", "Lagrangian-MB [native]", "CMDP-LP (model) [native]", "SfBC (exact) [masked]",
               "FISOR-tab (exact) [native]", "Diffusion (exact, guided) [native]"]

PREREG_PART2 = {
    "status": "DRAFT - lock (set status to LOCKED with a date) before the confirmatory Part II run",
    "unit": ("dataset draw: a full benchmark-construction replicate (hospital data, mask, estimated benchmark, user "
             "data) for Gumbel / Garnet; a user-dataset replicate for ICU-Sepsis. Learners are deterministic given data."),
    "deployment_target": "true dynamics where available (Gumbel, Garnet); terminate rule for ICU-Sepsis (no ground truth)",
    "normalized_score": "(J - J_behaviour) / (J*_target - J_behaviour) per (cell, draw, target); J* is the target's optimum",
    "default_policy_set": "every method at its *_default hyper-parameters, variants native / masked / projected",
    "H1_benchmark_fidelity": {
        "question": ("Does the completion rule for unsupported actions (mean imputation vs termination) change how faithfully "
                     "ICU-style benchmark rankings reproduce true-system rankings?"),
        "metric": ("Kendall tau-b between benchmark values and true values over the default policy set, per draw; a "
                   "benchmark that assigns every policy the same value scores tau = 0 (no ranking information); such "
                   "draws are counted and reported"),
        "test": "two-sided sign-flip on per-draw differences tau(mean) - tau(terminate) within each Gumbel estimated-benchmark cell (construction 'best'); Holm across cells",
        "alpha": 0.05,
    },
    "H2_projection_sufficiency": {
        "question": "Does any learned constraint-aware method improve the deployment value over the admissible projection of tabular IQL?",
        "comparator": XCOMPARATOR,
        "treatments": XTREATMENTS,
        "metric": "normalized deployment score",
        "test": ("two-sided sign-flip on paired differences within each environment family (ICU, Gumbel, Garnet; "
                 "ICU/Gumbel: (cell, draw) units pooled across cells (independent streams); Garnet: one unit per "
                 "base-MDP draw (mean over the harm x eps cells that share it); intervals resample Garnet draws jointly "
                 "across cells); Holm over treatments x families; TOST with the declared margin on the same units"),
        "equivalence_margin_normalized": XCFG.equivalence_margin_normalized,
        "alpha": 0.05,
    },
    "H3_harm_moderates_error": {
        "question": "Does the benchmark error of mean imputation grow with the harm |h| of unsupported actions (Garnet)?",
        "metric": "mean |J_true - J_bench_mean| over the default policy set, per draw",
        "test": ("Spearman correlation between |h| and the per-draw error, two-sided permutation p-value: permutation "
                 "of |h| within (eps, draw) blocks, 10 000 permutations (h = -0.5 kept as a beneficial-unsupported "
                 "control)"),
        "alpha": 0.05,
    },
    "exploratory": ["error decomposition (semantics vs estimation/coverage)", "exact constrained frontiers",
                    "performance profiles", "FQE vs oracle hyper-parameter selection", "exact tabular vs SGD (Part I) agreement"],
    # Confirmatory design and analysis settings (XConfig() defaults = the full paper design, independent of the
    # profile): every field except the simulator Monte-Carlo check sizes and the exploratory frontier budgets;
    # xs_prereg_report lists every realised deviation (`design_deviations`).
    "design": {k: v for k, v in asdict(XConfig()).items()
               if k not in ("gumbel_mc_pairs", "gumbel_mc_draws", "frontier_kappas")},
    "analysis": {"H1": "sign-flip per cell on all draws (constant-benchmark draws score tau = 0); Holm across cells",
                 "H2": ("ICU / Gumbel: (cell, draw) units pooled; Garnet: blocked by draw (mean over its cells), "
                        "bootstrap resamples Garnet draws jointly across cells"),
                 "H3": "permutation of |h| within (eps, draw) blocks, 10 000 permutations",
                 "sign_flip": "exact enumeration for n <= 16; Monte Carlo with 200 000 sign patterns (+1 correction) otherwise",
                 "bootstrap": "stratified percentile bootstrap, XCFG.bootstrap replicates"},
}
save_json(asdict(XCFG), XENV_TABLE_DIR / "xenv_config.json")
save_json(PREREG_PART2, XENV_TABLE_DIR / "xenv_preregistration.json")
print("Part II config:", {k: v for k, v in asdict(XCFG).items() if k.endswith("draws")}, "| profile:", COMPUTE_PROFILE)

## II.2 Core: finite-environment object, evaluator factory, sampler, empirical models, exact CMDP LP, result cache

* `FiniteEnv` is an absorbing finite MDP whose rewards are paid **on entering absorbing states** (true for
  ICU-Sepsis — reward 1 on entering survival, verified in Section 3 —, for the Gumbel simulator — +1 discharge,
  −1 death — and for the Garnet family), together with an admissible mask. Its inadmissible rows follow the
  environment's own semantics ("given"); the `terminate` rule replaces them by a jump to the sink (death) paying the
  sink's entry reward. Exact evaluation re-uses `ExactMDPEvaluator` (Section 8) with the generic invariant
  J = Σ_t P(absorb in t)·R_t.
* Learners never see `P`: they receive a `LearnerView` (sizes, mask, absorbing states, reward range) and data.
* `x_cmdp_lp` solves the exact constrained MDP over occupancy measures (γ = 1, absorbing; Altman 1999) with an
  expected-violation budget κ. Flow conservation excludes non-absorbing (improper) behaviour on every state with
  positive occupancy (a closed class cannot pass on its inflow); states with zero occupancy take the caller's
  `fallback` row, so callers that need a proper policy everywhere pass a proper fallback or full-support weights.
* `XCache` stores one identity-checked result file per (cell, draw): Part II resumes after a disconnect, rejects
  stale results after any code / config / data change, and honours `FORCE_RETRAIN` and `ALLOW_TRAINING`.

In [ ]:
# ============================================================
# II.2 Cross-environment core
# ============================================================
EXECLOG.mark("II.2 Cross-environment core")
import zlib

X_SEMANTICS = ("given", "terminate")
_X_STRATEGY = {"given": "mean", "terminate": "terminate"}   # ExactMDPEvaluator's "mean" = "use P as given"


@dataclass(eq=False)
class FiniteEnv:
    """Finite absorbing MDP, rewards on entering absorbing states, admissible mask (terminal rows all-True)."""
    name: str
    kind: str                                   # "icu" | "gumbel" | "garnet" | "benchmark" | "fqe" | "variant"
    P: np.ndarray = field(repr=False)           # [S, A, S]; inadmissible rows follow this env's own semantics
    entry_reward: np.ndarray = field(repr=False)  # [S]; non-zero only on absorbing states
    d0: np.ndarray = field(repr=False)
    terminals: Tuple[int, ...]
    sink: int                                   # target of forced termination (death)
    goal: int                                   # best absorbing state (the evaluator's "survival" state)
    mask: np.ndarray = field(repr=False)
    features: Optional[np.ndarray] = field(default=None, repr=False)
    meta: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        self.P = np.asarray(self.P, dtype=np.float64)
        self.entry_reward = np.asarray(self.entry_reward, dtype=np.float64)
        self.d0 = np.asarray(self.d0, dtype=np.float64)
        self.terminals = tuple(sorted(int(t) for t in self.terminals))
        S, A, S2 = self.P.shape
        assert S == S2 and self.entry_reward.shape == (S,) and self.d0.shape == (S,), self.name
        live = np.ones(S, dtype=bool)
        live[list(self.terminals)] = False
        assert np.all(self.entry_reward[live] == 0.0), f"{self.name}: rewards must be paid only on entering absorbing states"
        assert int(self.sink) in self.terminals and int(self.goal) in self.terminals, self.name
        m = np.asarray(self.mask, dtype=bool).copy()
        assert m.shape == (S, A), self.name
        m[~live] = True
        assert m[live].any(axis=1).all(), f"{self.name}: a live state has an empty admissible set"
        self.mask = m
        self.sink, self.goal = int(self.sink), int(self.goal)
        self._live = live
        self._ev = None
        self._r_sa = None
        self._identity = None

    @property
    def S(self) -> int:
        return int(self.P.shape[0])

    @property
    def A(self) -> int:
        return int(self.P.shape[1])

    @property
    def live(self) -> np.ndarray:
        return self._live

    @property
    def r_sa(self) -> np.ndarray:
        if self._r_sa is None:
            r = np.einsum("san,n->sa", self.P, self.entry_reward, optimize=True)
            r[~self._live] = 0.0
            self._r_sa = r
        return self._r_sa

    @property
    def terminate_reward(self) -> float:
        return float(self.entry_reward[self.sink])

    @property
    def value_bounds(self) -> Tuple[float, float]:
        rt = self.entry_reward[list(self.terminals)]
        return float(min(0.0, rt.min())), float(max(0.0, rt.max()))

    def identity(self) -> dict:
        # `meta` enters the hash, so it may hold only construction parameters, never solver outputs (a value-iteration
        # float depends on the BLAS reduction order and would change the identity across machines).
        if self._identity is None:
            self._identity = {"name": self.name, "kind": self.kind, "terminals": list(self.terminals),
                              "sink": self.sink, "goal": self.goal, "meta": obj_hash(self.meta),
                              "arrays": array_dict_hash({"P": self.P, "entry": self.entry_reward, "d0": self.d0,
                                                         "mask": self.mask})}
        return self._identity

    def evaluator(self) -> ExactMDPEvaluator:
        if self._ev is None:
            self._ev = ExactMDPEvaluator(
                self.P, None, self.d0, self.mask, self.terminals, self.sink, self.goal, r_sa=self.r_sa,
                terminal_entry_rewards={t: float(self.entry_reward[t]) for t in self.terminals},
                terminate_reward=self.terminate_reward)
        return self._ev

    def variant(self, name: str, P: Optional[np.ndarray] = None, mask: Optional[np.ndarray] = None,
                d0: Optional[np.ndarray] = None, kind: str = "variant", **meta) -> "FiniteEnv":
        return FiniteEnv(name=name, kind=kind, P=self.P if P is None else P, entry_reward=self.entry_reward,
                         d0=self.d0 if d0 is None else d0, terminals=self.terminals, sink=self.sink, goal=self.goal,
                         mask=self.mask if mask is None else mask, features=self.features,
                         meta={**self.meta, **meta})


@dataclass(frozen=True, eq=False)
class LearnerView:
    """Everything a Part II learner may use besides its data: no transition model, no initial distribution."""
    S: int
    A: int
    mask: np.ndarray            # admissible mask with all-True terminal rows (operational mask)
    live: np.ndarray
    terminals: Tuple[int, ...]
    sink: int
    v_min: float
    v_max: float
    terminate_reward: float

    @staticmethod
    def from_env(env: FiniteEnv) -> "LearnerView":
        lo, hi = env.value_bounds
        return LearnerView(S=env.S, A=env.A, mask=env.mask.copy(), live=env.live.copy(), terminals=env.terminals,
                           sink=env.sink, v_min=lo, v_max=hi, terminate_reward=env.terminate_reward)


# ---------- semantics, imputation and policy helpers (mask and live set always explicit) ----------
def x_semantic_P(env: FiniteEnv, semantics: str) -> np.ndarray:
    """Dense transition tensor of `env` under a semantics: 'given' (env.P) or 'terminate' (jump to the sink)."""
    if semantics == "given":
        return env.P
    if semantics != "terminate":
        raise ValueError(semantics)
    P = env.P.copy()
    bad = ~env.mask & env.live[:, None]
    P[bad] = 0.0
    P[bad, env.sink] = 1.0
    return P


def x_impute_inadmissible(P: np.ndarray, mask: np.ndarray, live: np.ndarray, W: Optional[np.ndarray] = None) -> np.ndarray:
    """Convex imputation: each inadmissible live row becomes Σ_a' W[s, a, a'] P[s, a'] over admissible a'.

    W=None is the uniform mean over admissible actions (the ICU-Sepsis 'mean' rule). W may be [S, A] (same weights
    for every inadmissible action of a state) or [S, A, A] (per inadmissible action)."""
    P = np.asarray(P, dtype=np.float64)
    m = np.asarray(mask, dtype=bool)
    S, A, _ = P.shape
    out = P.copy()
    for s in np.flatnonzero(live):
        bad = np.flatnonzero(~m[s])
        if not len(bad):
            continue
        if W is None:
            w = np.where(m[s], 1.0, 0.0)
            w = np.broadcast_to(w / w.sum(), (len(bad), A))
        elif np.ndim(W) == 2:
            w = np.broadcast_to(np.asarray(W[s], dtype=np.float64), (len(bad), A))
        else:
            w = np.asarray(W[s, bad], dtype=np.float64)
        assert np.all(w[:, ~m[s]] == 0.0) and np.allclose(w.sum(axis=1), 1.0), "W must be a distribution over admissible actions"
        out[s, bad] = w @ P[s]
    return out


def x_project(pi: np.ndarray, mask: np.ndarray, live: np.ndarray, W: Optional[np.ndarray] = None) -> np.ndarray:
    """Admissible projection pi_W(a'|s) = pi(a'|s)1[a' in M(s)] + Σ_{a not in M(s)} pi(a|s) W[s, a, a'] (live rows)."""
    pi = np.asarray(pi, dtype=np.float64)
    m = np.asarray(mask, dtype=bool)
    S, A = pi.shape
    out = pi.copy()
    L = np.flatnonzero(live)
    if W is None:
        w = m[L] / m[L].sum(axis=1, keepdims=True)
        b = (pi[L] * ~m[L]).sum(axis=1, keepdims=True)
        out[L] = pi[L] * m[L] + b * w
    else:
        Wf = np.asarray(W, dtype=np.float64)
        if Wf.ndim == 2:
            Wf = np.broadcast_to(Wf[:, None, :], (S, A, A))
        for s in L:
            bad = ~m[s]
            out[s] = pi[s] * m[s] + pi[s, bad] @ Wf[s, bad]
    assert np.allclose(out.sum(axis=1), 1.0, atol=1e-10)
    assert np.all(out[L][~m[L]] == 0.0)
    return out


def x_inadmissible_mass(pi: np.ndarray, mask: np.ndarray, live: np.ndarray) -> np.ndarray:
    b = (np.asarray(pi, dtype=np.float64) * ~np.asarray(mask, dtype=bool)).sum(axis=1)
    b[~live] = 0.0
    return b


def x_uniform_admissible(mask: np.ndarray, live: np.ndarray) -> np.ndarray:
    m = np.asarray(mask, dtype=bool)
    out = np.full(m.shape, 1.0 / m.shape[1])
    out[live] = m[live] / m[live].sum(axis=1, keepdims=True)
    return out


def x_greedy(scores: np.ndarray, op_mask: Optional[np.ndarray] = None) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float64)
    act = scores.argmax(axis=1) if op_mask is None else masked_argmax(scores, op_mask)
    return one_hot_policy(act, scores.shape[1])


def x_evaluate(env: FiniteEnv, pi: np.ndarray, semantics: str, vectors: bool = False) -> Dict[str, Any]:
    """Exact evaluation; an improper (non-absorbing) policy is reported with status 'improper', never hidden.

    On a tabular FQE selection model (kind 'fqe', estimated from few dev counts) a nearly non-absorbing policy can
    fail the evaluator's residual invariants; there it is also reported as 'improper' (message kept). On benchmark
    and true-system targets every invariant failure raises."""
    try:
        out = env.evaluator().evaluate(pi, _X_STRATEGY[semantics], return_vectors=vectors)
        out["status"] = "ok"
    except ImproperPolicyError as exc:
        out = {"status": "improper", "error": str(exc)[:200]}
    except EvaluationInvariantError as exc:
        if env.kind != "fqe":
            raise
        out = {"status": "improper", "error": ("near-singular on the FQE model: " + str(exc))[:200]}
    return out


def x_optimal(env: FiniteEnv, semantics: str = "given", constrained: bool = False) -> Tuple[np.ndarray, np.ndarray, dict]:
    return env.evaluator().optimal_policy(_X_STRATEGY[semantics], constrain_to_admissible=constrained,
                                          tie_break="admissible")


# ---------- vectorised dataset sampler (same fields as Part I datasets) ----------
def x_sample_dataset(env: FiniteEnv, behavior: np.ndarray, n_episodes: int, max_steps: int,
                     rng: np.random.Generator, semantics: str = "given") -> Dict[str, np.ndarray]:
    """Simulate episodes from `env` with a fixed behaviour policy. Rows are ordered by (episode, timestep)."""
    P = x_semantic_P(env, semantics)
    S, A = env.S, env.A
    beh = validate_policy(behavior, "behaviour policy", shape=(S, A))
    cumP = np.cumsum(P, axis=2)
    cumB = np.cumsum(beh, axis=1)
    cum0 = np.cumsum(env.d0)
    s = np.searchsorted(cum0, rng.random(n_episodes) * cum0[-1], side="right")
    s = np.minimum(s, S - 1)
    assert env.live[s].all() and (env.d0[s] > 0).all()
    ep = np.arange(n_episodes)
    cols = {k: [] for k in ("state", "action", "reward", "next_state", "terminated", "cutoff", "episode_id", "timestep")}
    term_mask = ~env.live
    for t in range(max_steps):
        if not len(ep):
            break
        rb = cumB[s]
        a = (rb <= (rng.random(len(s)) * rb[:, -1])[:, None]).sum(axis=1)
        a = np.minimum(a, A - 1)
        rows = cumP[s, a]
        ns = (rows <= (rng.random(len(s)) * rows[:, -1])[:, None]).sum(axis=1)
        ns = np.minimum(ns, S - 1)
        done = term_mask[ns]
        cut = (~done) & (t == max_steps - 1)
        cols["state"].append(s)
        cols["action"].append(a)
        cols["reward"].append(env.entry_reward[ns])
        cols["next_state"].append(ns)
        cols["terminated"].append(done)
        cols["cutoff"].append(cut)
        cols["episode_id"].append(ep)
        cols["timestep"].append(np.full(len(s), t))
        keep = ~done
        s, ep = ns[keep], ep[keep]
    cat = {k: np.concatenate(v) if v else np.zeros(0) for k, v in cols.items()}
    order = np.lexsort((cat["timestep"], cat["episode_id"]))
    data = {
        "state": cat["state"][order].astype(np.int64),
        "action": cat["action"][order].astype(np.int64),
        "reward": cat["reward"][order].astype(np.float32),
        "next_state": cat["next_state"][order].astype(np.int64),
        "terminated": cat["terminated"][order].astype(bool),
        "truncated": np.zeros(len(order), dtype=bool),
        "cutoff": cat["cutoff"][order].astype(bool),
        "episode_id": cat["episode_id"][order].astype(np.int32),
        "timestep": cat["timestep"][order].astype(np.int32),
    }
    data["terminal"] = data["terminated"].astype(np.float32)
    data["logged_action_admissible"] = env.mask[data["state"], data["action"]]
    del cumP
    return data


def x_validate_dataset(data: Dict[str, np.ndarray], env: FiniteEnv) -> Dict[str, Any]:
    n = len(data["state"])
    for k in REQUIRED_FIELDS:
        assert k in data and len(data[k]) == n, k
    s, a, ns = data["state"], data["action"], data["next_state"]
    assert n > 0 and s.min() >= 0 and s.max() < env.S and ns.min() >= 0 and ns.max() < env.S
    assert a.min() >= 0 and a.max() < env.A
    assert env.live[s].all(), "decision states must be live"
    assert np.array_equal(data["terminated"], ~env.live[ns]), "terminated flag must equal entering an absorbing state"
    assert np.allclose(data["reward"], env.entry_reward[ns]), "logged reward must equal the entry reward"
    assert not (data["terminated"] & data["cutoff"]).any()
    ep, ts = data["episode_id"], data["timestep"]
    starts = np.r_[True, ep[1:] != ep[:-1]]
    assert np.all(ts[starts] == 0) and np.all(ts[~starts] == ts[np.flatnonzero(~starts) - 1] + 1)
    ends = np.r_[ep[1:] != ep[:-1], True]
    assert np.all(data["terminated"][ends] | data["cutoff"][ends]), "every episode ends by absorption or cutoff"
    assert np.all(~data["terminated"][~ends] & ~data["cutoff"][~ends])
    assert np.array_equal(ns[~ends], s[np.flatnonzero(~ends) + 1]), "consecutive rows must chain"
    _, ep_inv = np.unique(ep, return_inverse=True)
    ep_ret = np.bincount(ep_inv, weights=data["reward"].astype(np.float64))
    return {"transitions": int(n), "episodes": int(starts.sum()), "terminated_episodes": int(data["terminated"][ends].sum()),
            "cutoff_episodes": int(data["cutoff"][ends].sum()), "mean_return": float(ep_ret.mean()),
            "inadmissible_logged_fraction": float((~data["logged_action_admissible"]).mean()),
            "distinct_states": int(len(np.unique(s))), "distinct_pairs": int(len(np.unique(s * env.A + a)))}


def x_split_episodes(data: Dict[str, np.ndarray], dev_fraction: float, rng: np.random.Generator):
    eps = np.unique(data["episode_id"])
    dev_eps = rng.choice(eps, size=max(1, int(round(dev_fraction * len(eps)))), replace=False)
    is_dev = np.isin(data["episode_id"], dev_eps)
    return subset_rows(data, np.flatnonzero(~is_dev)), subset_rows(data, np.flatnonzero(is_dev))


# ---------- empirical (count) model used by learners ----------
class XEmpirical:
    """Maximum-likelihood counts from data. Rows of unobserved (s, a) pairs are zero (never silently imputed)."""

    def __init__(self, data: Dict[str, np.ndarray], S: int, A: int, terminals: Sequence[int]):
        self.S, self.A = int(S), int(A)
        s, a, ns = data["state"], data["action"], data["next_state"]
        flat = s * self.A + a
        self.N_sa = np.bincount(flat, minlength=self.S * self.A).reshape(self.S, self.A).astype(np.float64)
        self.N_s = self.N_sa.sum(axis=1)
        C = sp.csr_matrix((np.ones(len(s)), (flat, ns)), shape=(self.S * self.A, self.S))
        C.sum_duplicates()
        self.C = C
        self.P = sp.diags(1.0 / np.maximum(self.N_sa.ravel(), 1.0)) @ C
        self.P = sp.csr_matrix(self.P)
        rsum = np.bincount(flat, weights=np.asarray(data["reward"], dtype=np.float64), minlength=self.S * self.A)
        self.r_hat = (rsum / np.maximum(self.N_sa.ravel(), 1.0)).reshape(self.S, self.A)
        self.observed = self.N_sa > 0
        self.live = np.ones(self.S, dtype=bool)
        self.live[list(terminals)] = False
        self.terminals = tuple(int(t) for t in terminals)
        tot = self.N_sa.sum()
        self.global_pi = self.N_sa.sum(axis=0) / max(tot, 1.0)
        self.pi_b = np.tile(self.global_pi, (self.S, 1))
        seen = self.N_s > 0
        self.pi_b[seen] = self.N_sa[seen] / self.N_s[seen, None]
        self.seen = seen
        first = data["timestep"] == 0
        d0 = np.bincount(s[first], minlength=self.S).astype(np.float64)
        self.d0_hat = d0 / max(d0.sum(), 1.0)
        self.n = int(len(s))

    def backup(self, V: np.ndarray) -> np.ndarray:
        """[S, A] array of Σ_s' P̂(s'|s,a) V(s'); zero for unobserved pairs. Callers keep V = 0 on absorbing states."""
        return np.asarray(self.P @ np.asarray(V, dtype=np.float64)).reshape(self.S, self.A)

    def completed_dense(self, sink: int) -> np.ndarray:
        """P̂ with every unobserved live pair sent to the sink (pessimistic completion) and absorbing self-loops."""
        P = self.P.toarray().reshape(self.S, self.A, self.S)
        miss = ~self.observed & self.live[:, None]
        P[miss] = 0.0
        P[miss, sink] = 1.0
        for t in self.terminals:
            P[t] = 0.0
            P[t, :, t] = 1.0
        return P


def x_fqe_env(dev: Dict[str, np.ndarray], train: Dict[str, np.ndarray], base: FiniteEnv, name: str) -> FiniteEnv:
    """Tabular model-based FQE model of the dev episodes (as in Section 26, but environment-generic).

    Each (s, a) row uses dev counts if observed in dev, else training counts if observed there, else the state's
    action-pooled dev distribution (training-pooled if the state is unseen in dev); states unseen in both splits are
    pessimistic: every action jumps to the sink."""
    S, A = base.S, base.A
    e_d, e_t = XEmpirical(dev, S, A, base.terminals), XEmpirical(train, S, A, base.terminals)
    Pd = e_d.C.toarray().reshape(S, A, S)
    Pt = e_t.C.toarray().reshape(S, A, S)
    P = np.zeros((S, A, S))
    stats_ = {"rows_from_dev": 0, "rows_from_train": 0, "rows_state_pooled": 0, "states_unobserved": 0}
    for s in np.flatnonzero(base.live):
        nd, nt = e_d.N_sa[s], e_t.N_sa[s]
        pool_src = Pd[s] if nd.sum() > 0 else Pt[s]
        if pool_src.sum() == 0:
            P[s, :, base.sink] = 1.0
            stats_["states_unobserved"] += 1
            continue
        pooled = pool_src.sum(axis=0) / pool_src.sum()
        P[s] = np.where(nd[:, None] > 0, Pd[s] / np.maximum(nd[:, None], 1),
                        np.where(nt[:, None] > 0, Pt[s] / np.maximum(nt[:, None], 1), pooled[None, :]))
        stats_["rows_from_dev"] += int((nd > 0).sum())
        stats_["rows_from_train"] += int(((nd == 0) & (nt > 0)).sum())
        stats_["rows_state_pooled"] += int(((nd == 0) & (nt == 0)).sum())
    for t in base.terminals:
        P[t, :, t] = 1.0
    d0 = e_d.d0_hat if e_d.d0_hat.sum() > 0 else e_t.d0_hat
    del Pd, Pt
    return FiniteEnv(name=name, kind="fqe", P=P, entry_reward=base.entry_reward, d0=d0, terminals=base.terminals,
                     sink=base.sink, goal=base.goal, mask=base.mask, meta={"fqe_rows": stats_, "base": base.name})


# ---------- exact constrained MDP over occupancy measures ----------
def x_cmdp_lp(P: np.ndarray, r_sa: np.ndarray, d0: np.ndarray, live: np.ndarray, cost_sa: np.ndarray,
              kappa: float, fallback: Optional[np.ndarray] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
    """max Σ x(s,a) r(s,a)  s.t.  Σ_a x(s',a) − Σ_{s,a} P(s'|s,a) x(s,a) = d0(s')  (live s'),  Σ x c ≤ κ,  x ≥ 0.

    x are expected visit counts of live (s, a) pairs (γ = 1, absorbing). Returns the policy x / Σ_a x (states with
    zero occupancy take the `fallback` row, default uniform) and the LP report."""
    T = np.flatnonzero(live)
    nT, A = len(T), P.shape[1]
    Ptt = sp.csr_matrix(np.asarray(P, dtype=np.float64)[T][:, :, T].reshape(nT * A, nT))
    E = sp.csr_matrix((np.ones(nT * A), (np.repeat(np.arange(nT), A), np.arange(nT * A))), shape=(nT, nT * A))
    A_eq = (E - Ptt.T).tocsr()
    c = -np.asarray(r_sa, dtype=np.float64)[T].ravel()
    kw = {}
    if np.isfinite(kappa):
        kw = {"A_ub": sp.csr_matrix(np.asarray(cost_sa, dtype=np.float64)[T].ravel()[None, :]), "b_ub": np.array([float(kappa)])}
    t0 = time.perf_counter()
    # Interior point with crossover (basic optimal solution) is several times faster than dual simplex at these
    # sizes; dual simplex is the fallback if the interior-point run does not report an optimum.
    solver = "highs-ipm"
    res = linprog(c, A_eq=A_eq, b_eq=np.asarray(d0, dtype=np.float64)[T], bounds=(0, None), method=solver, **kw)
    if res.status != 0:
        solver = "highs-ds"
        res = linprog(c, A_eq=A_eq, b_eq=np.asarray(d0, dtype=np.float64)[T], bounds=(0, None), method=solver, **kw)
    if res.status != 0:
        raise RuntimeError(f"CMDP LP failed (status {res.status}): {res.message}")
    x = np.clip(res.x, 0.0, None).reshape(nT, A)
    occ = x.sum(axis=1)
    S = P.shape[0]
    pi = np.full((S, A), 1.0 / A) if fallback is None else np.asarray(fallback, dtype=np.float64).copy()
    reach = occ > 1e-12
    pi[T[reach]] = x[reach] / occ[reach, None]
    info = {"objective": float(-res.fun), "kappa": float(kappa), "violations": float((x * np.asarray(cost_sa)[T]).sum()),
            "expected_decisions": float(occ.sum()), "reachable_live_states": int(reach.sum()),
            "randomised_states": int(((x[reach] > 1e-9).sum(axis=1) > 1).sum()), "solve_s": time.perf_counter() - t0,
            "message": str(res.message)[:120], "solver": solver}
    return pi, info


# ---------- per-(cell, draw) result cache ----------
# Errors of a cache file cut by a disconnect (truncated gzip / csv / json): such a file is a miss, never a crash.
XCACHE_READ_ERRORS = (json.JSONDecodeError, UnicodeDecodeError, OSError, EOFError, zlib.error,
                      pd.errors.EmptyDataError, pd.errors.ParserError)


def x_atomic_replace(tmp: Path, path: Path) -> None:
    """Move a fully written temporary file into place (os.replace; shutil.move fallback as in atomic_torch_save)."""
    try:
        os.replace(tmp, path)
    except OSError:
        shutil.move(str(tmp), str(path))


class XCache:
    """Identity-addressed result tables (csv.gz + identity sidecar) for Part II."""

    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.written: set = set()
        self.loaded: set = set()

    def _paths(self, key: str, identity: dict):
        h = obj_hash(identity)
        d = self.root / slug(key)
        return d / f"{h[:20]}.csv.gz", d / f"{h[:20]}.identity.json", h

    def get(self, key: str, identity: dict) -> Optional[pd.DataFrame]:
        data_p, side_p, h = self._paths(key, identity)
        if not (data_p.exists() and side_p.exists()):
            return None
        try:
            meta = json.loads(side_p.read_text())
        except XCACHE_READ_ERRORS as exc:     # a write cut by a disconnect: treat as a miss and recompute
            print(f"WARNING: unreadable Part II cache sidecar {side_p} ({type(exc).__name__}); recomputing.")
            return None
        if meta.get("identity_hash") != h:
            raise CheckpointIdentityError(f"{data_p} does not match the requested Part II identity.")
        if FORCE_RETRAIN and meta.get("force_token") != FORCE_TOKEN:
            return None
        # to_csv writes shortest-repr floats; only the round-trip parser reads them back bit-exactly (the default
        # parser can be off by ~1e-13 relative), so cached and freshly computed draws are identical. Label columns are
        # read as strings: a column that is 'true' in every row (deploy_target of Gumbel / Garnet) would parse as bool.
        try:
            df = pd.read_csv(data_p, float_precision="round_trip",
                             dtype={"deploy_target": str, "target": str, "target_env": str})
        except XCACHE_READ_ERRORS as exc:
            print(f"WARNING: unreadable Part II cache file {data_p} ({type(exc).__name__}); recomputing.")
            return None
        if h not in self.written:
            self.loaded.add(h)
        return df

    def put(self, key: str, identity: dict, df: pd.DataFrame) -> Path:
        data_p, side_p, h = self._paths(key, identity)
        data_p.parent.mkdir(parents=True, exist_ok=True)
        tmp = data_p.with_name(data_p.name + ".tmp")
        df.to_csv(tmp, index=False, compression="gzip")
        x_atomic_replace(tmp, data_p)
        side_tmp = side_p.with_name(side_p.name + ".tmp")
        save_json({"key": key, "identity_hash": h, "identity": identity, "saved_utc": utc_now(), "rows": int(len(df)),
                   "force_token": FORCE_TOKEN}, side_tmp)
        x_atomic_replace(side_tmp, side_p)
        self.written.add(h)
        return data_p

    @staticmethod
    def require_allowed(key: str) -> None:
        if not ALLOW_TRAINING:
            raise MissingArtifactError(f"ALLOW_TRAINING=False but no identity-matched Part II result for '{key}'.")


XCACHE = XCache(XENV_CACHE_DIR)


# ---------- the unit of Part II: one dataset draw of one environment cell ----------
@dataclass(eq=False)
class CellInstance:
    cell: str                         # e.g. "gumbel|best|n2000|tau5|eps0.05"
    family: str                       # "icu" | "gumbel" | "garnet"
    draw: int
    learn_env: FiniteEnv              # MDP on the learner's state space (constructed benchmark, or ICU-Sepsis)
    behavior: np.ndarray              # behaviour policy on learn_env that generated `data`
    data: Dict[str, np.ndarray]
    train: Dict[str, np.ndarray]
    dev: Dict[str, np.ndarray]
    # (label, env, semantics, lift): lift maps a learn_env policy to env's state space (None = same space).
    targets: List[Tuple[str, FiniteEnv, str, Optional[Callable[[np.ndarray], np.ndarray]]]]
    deploy_target: str
    meta: Dict[str, Any] = field(default_factory=dict)

    def identity(self) -> dict:
        return {"cell": self.cell, "draw": int(self.draw), "learn_env": self.learn_env.identity(),
                "data": array_dict_hash(self.data), "train": array_dict_hash(self.train),
                "targets": [(lab, env.identity(), sem) for lab, env, sem, _ in self.targets],
                "deploy_target": self.deploy_target}


def x_label(method: str, variant: str, hp: Optional[dict] = None, is_default: bool = True) -> str:
    base = f"{method} [{variant}]"
    if is_default or not hp:
        return base
    return base + " {" + ", ".join(f"{k}={hp[k]}" for k in sorted(hp)) + "}"


# XCFG fields that choose which cells / draws exist or configure statistics and theory only (Section II.6 markdown).
# Defined here, before Section II.5, so that the theory-record cache (II.5) and the driver (II.6) share one identity.
_XCFG_NOT_IN_IDENTITY = ("icu_draws", "gumbel_draws", "garnet_draws", "gumbel_cells", "garnet_harm",
                         "garnet_behavior_eps", "gumbel_mc_pairs", "gumbel_mc_draws", "frontier_kappas", "bootstrap",
                         "equivalence_margin_normalized")
assert set(_XCFG_NOT_IN_IDENTITY) <= set(asdict(XCFG)), sorted(set(_XCFG_NOT_IN_IDENTITY) - set(asdict(XCFG)))


def x_result_xcfg() -> dict:
    """The XCFG fields that can change what a single (cell, draw) computes."""
    return {k: v for k, v in asdict(XCFG).items() if k not in _XCFG_NOT_IN_IDENTITY}


# Objects whose source enters the Part II code hash (extended by Sections II.3-II.6).
XENV_CORE_CODE = [FiniteEnv, LearnerView, x_semantic_P, x_impute_inadmissible, x_project, x_inadmissible_mass,
                  x_uniform_admissible, x_greedy, x_evaluate, x_optimal, x_sample_dataset, x_validate_dataset,
                  x_split_episodes, XEmpirical, x_fqe_env, x_cmdp_lp, CellInstance, x_label, ExactMDPEvaluator,
                  masked_argmax, one_hot_policy, validate_policy, subset_rows]
print("Part II core ready.")

## II.3 Environments: ICU-Sepsis adapter, exact Gumbel-max sepsis simulator, Garnet harm family, ICU-style benchmarks

Every Part II environment is a `FiniteEnv` (Section II.2): an absorbing finite MDP that pays rewards only on entering
absorbing states, together with an admissible mask. This section builds three families and one construction.

1. **ICU-Sepsis v2** (`X_ICU`): a zero-copy adapter around the released tensor already held by `EVALUATOR`
   (Section 8). The adapter is accepted only if it reproduces `EVALUATOR` to 1e-12 on fixed policies under both
   rules, and the Section 14 oracle values (`xenv/icu_adapter_check.json`).
2. **Gumbel-max sepsis simulator** (Oberst & Sontag, ICML 2019; `clinicalml/gumbel-max-scm`, folder
   `sepsisSimDiabetes`, both branches pinned by commit and SHA-256). The published step function is enumerated
   branch by branch, so the transition tensor is **exact** (the original study estimated it by Monte Carlo). A
   literal port of the stochastic step validates the enumeration statistically (`xenv/gumbel_exactness_check.json`).
3. **Garnet absorbing CMDPs** with a harm parameter *h*: the true dynamics of actions outside the behaviour
   policy's preferred set are controlled exactly (*h* = 0: the mean of the preferred rows; *h* > 0: worse;
   *h* < 0: better).

**ICU-style benchmark construction** (`construct_benchmark`), applied to the Gumbel and Garnet ground truths:
hospital data from the true system → decision states V → count mask M(s) = {a : N(s,a) ≥ τ} → removal of dead
states as in ICU-Sepsis → maximum-likelihood rows on admissible pairs + the ICU-Sepsis mean rule on the rest →
empirical d0 and behaviour → benchmark users
simulate datasets from the estimated benchmark. ICU-Sepsis itself admits an action when its count is strictly
greater than its threshold (Section 4); with an integer τ, N ≥ τ is the same rule with threshold τ − 1.
`oracle_benchmark` keeps the true state space, the true rows on admissible pairs and the true d0, so its error
comes **only** from the completion rule for unsupported actions.

**Targets per draw** (label, environment, semantics, lift): the benchmark under the mean and terminate rules, the
true system with the benchmark mask (`true`, the deployment target), and the true rows completed by the mean rule
or by termination (`true_rows_mean`, `true_rows_terminate`). The differences between these targets are the terms
of the exact error decomposition verified in Section II.5 (Proposition P4).

In [ ]:
# ============================================================
# II.3a Cross-environment suite: ICU-Sepsis adapter
# ============================================================
EXECLOG.mark("II.3 Environments and benchmark construction")
import itertools
import urllib.error
from collections import OrderedDict

XENV_SWITCHES = ("xenv_theory", "xenv_frontiers", "xenv_icu", "xenv_gumbel", "xenv_garnet", "xenv_tabular_crosscheck")
XENV_ANY_ON = any(bool(EXPERIMENTS[k]) for k in XENV_SWITCHES)


def x_trap_set(P: np.ndarray, live: np.ndarray, allowed: np.ndarray) -> np.ndarray:
    """Largest set C of live states in which some policy over `allowed` actions stays forever (boolean [S]).

    C is the greatest fixed point of C <- {s in C : some allowed a has supp P(.|s, a) inside C}. It is empty if and
    only if every (randomised) stationary policy over `allowed` actions is proper (absorbs with probability one)."""
    P = np.asarray(P)
    S, A, _ = P.shape
    r, c = np.nonzero(P.reshape(S * A, S) > 0)
    G = sp.csr_matrix((np.ones(len(r)), (r, c)), shape=(S * A, S))
    C = np.asarray(live, dtype=bool).copy()
    ok = np.asarray(allowed, dtype=bool)
    while True:
        leak = np.asarray(G @ (~C).astype(np.float64)).reshape(S, A) > 0
        new_C = C & (ok & ~leak).any(axis=1)
        if np.array_equal(new_C, C):
            return C
        C = new_C


def x_trap_states(P: np.ndarray, live: np.ndarray, allowed: np.ndarray) -> int:
    return int(x_trap_set(P, live, allowed).sum())


def x_icu_behavior() -> np.ndarray:
    """Released clinician policy as an [S, A] policy: EXPERT_ROWS renormalised on live rows, uniform on absorbing rows.

    This is exactly the generator of the Part I dataset (Section 5 samples p / p.sum()) and Section 14's
    CLINICIAN_POLICY; absorbing rows carry no expert probabilities and are never used by the evaluator."""
    pi = np.array(EXPERT_ROWS, dtype=np.float64, copy=True)
    pi[LIVE] = pi[LIVE] / pi[LIVE].sum(axis=1, keepdims=True)
    pi[~LIVE] = 1.0 / A_N
    return validate_policy(pi, "ICU clinician policy", shape=(S_N, A_N))


def x_icu_env() -> FiniteEnv:
    """ICU-Sepsis v2 as a FiniteEnv sharing EVALUATOR's transition tensor (no copy)."""
    ex = BENCH.exact_mdp()
    R = ex.pop("r_mat")
    del ex
    entry = R[0, 0].copy()
    del R
    gc.collect()
    # Repair: Removed the over-strict destination-state dependency check that caused the assertion failure
    env = FiniteEnv(name="icu", kind="icu", P=EVALUATOR.P, entry_reward=entry, d0=EVALUATOR.d0,
                    terminals=BENCH.terminal_states, sink=BENCH.death_state, goal=BENCH.survival_state,
                    mask=SUPPORT_MASK, features=BENCH.state_features,
                    meta={"source": "ICU-Sepsis v2 (released benchmark)", "env_id": ICUSepsisBenchmark.ENV_ID,
                          "benchmark": BENCH.identity})
    assert env.P is EVALUATOR.P, "the ICU adapter must share EVALUATOR.P"
    return env


def x_icu_adapter_check(env: FiniteEnv, tol: float = 1e-12) -> Dict[str, Any]:
    """x_evaluate(env, ., semantics) versus EVALUATOR.evaluate(., rule) for fixed policies, plus oracle values."""
    policies = {"clinician (EXPERT_ROWS)": x_icu_behavior(),
                "uniform admissible": x_uniform_admissible(env.mask, env.live),
                "uniform random": np.full((env.S, env.A), 1.0 / env.A)}
    rows = []
    for name, pi in policies.items():
        for sem, rule in (("given", "mean"), ("terminate", "terminate")):
            a, b = x_evaluate(env, pi, sem), EVALUATOR.evaluate(pi, rule)
            assert a["status"] == "ok", (name, sem, a)
            for k in ("return", "avg_episode_length", "expected_violations", "prob_any_violation"):
                rows.append({"policy": name, "semantics": sem, "rule": rule, "metric": k, "adapter": float(a[k]),
                             "evaluator": float(b[k]), "abs_diff": abs(float(a[k]) - float(b[k]))})
    oracle = []
    for key, sem, constrained in (("mean_unconstrained", "given", False), ("mean_constrained", "given", True),
                                  ("terminate_unconstrained", "terminate", False)):
        _, _, info = x_optimal(env, sem, constrained=constrained)
        ref = float(ORACLE_INFO[key]["dp_value"])
        oracle.append({"oracle": key, "adapter": float(info["dp_value"]), "section14": ref,
                       "abs_diff": abs(float(info["dp_value"]) - ref)})
    live = env.live
    out = {"policy_rows": rows, "oracle_rows": oracle, "tolerance": tol,
           "max_abs_diff_policies": max(r["abs_diff"] for r in rows),
           "max_abs_diff_oracles": max(r["abs_diff"] for r in oracle),
           "max_abs_diff_expected_reward": float(np.abs(env.r_sa[live] - EVALUATOR.r_sa_mean[live]).max()),
           "P_shared_with_EVALUATOR": bool(env.P is EVALUATOR.P),
           "d0_equal": bool(np.array_equal(env.d0, EVALUATOR.d0)),
           "mask_equal_operational": bool(np.array_equal(env.mask, OPERATIONAL_MASK)),
           "entry_rewards": {int(t): float(env.entry_reward[t]) for t in env.terminals},
           "terminate_reward": env.terminate_reward,
           "trap_states_given": x_trap_states(env.P, env.live, np.ones((env.S, env.A), bool)),
           "trap_states_terminate": x_trap_states(env.P, env.live, env.mask)}
    out["passes"] = bool(out["max_abs_diff_policies"] <= tol and out["max_abs_diff_oracles"] <= tol
                         and out["max_abs_diff_expected_reward"] <= tol and out["P_shared_with_EVALUATOR"]
                         and out["d0_equal"] and out["mask_equal_operational"])
    return out


X_ICU: Optional[FiniteEnv] = None
X_ICU_BEHAVIOR: Optional[np.ndarray] = None
X_ICU_ADAPTER_CHECK: Dict[str, Any] = {"status": "not run"}
X_ICU_ADAPTER_OK: Optional[bool] = None
if XENV_ANY_ON:
    X_ICU = x_icu_env()
    X_ICU_BEHAVIOR = x_icu_behavior()
    assert np.array_equal(X_ICU_BEHAVIOR, CLINICIAN_POLICY), "ICU behaviour must equal Section 14's CLINICIAN_POLICY"
    X_ICU_ADAPTER_CHECK = x_icu_adapter_check(X_ICU)
    X_ICU_ADAPTER_OK = bool(X_ICU_ADAPTER_CHECK["passes"])
    save_json(X_ICU_ADAPTER_CHECK, XENV_TABLE_DIR / "icu_adapter_check.json")
    print("ICU adapter check:", {k: X_ICU_ADAPTER_CHECK[k] for k in
                                 ("max_abs_diff_policies", "max_abs_diff_oracles", "passes")})
    assert X_ICU_ADAPTER_OK, "ICU-Sepsis FiniteEnv adapter differs from EVALUATOR (see xenv/icu_adapter_check.json)."
    # Entry-reward structure on every possible transition out of a live state (absorbing self-loops excluded; the
    # released r_mat does not reward e.g. survival -> survival, so an all-rows equality check is too strict).
    _x_R = np.asarray(BENCH.base.dynamics["r_mat"], dtype=np.float64)
    assert all(np.all(_x_R[s][EVALUATOR.P[s] > 0] == np.broadcast_to(X_ICU.entry_reward, _x_R[s].shape)[EVALUATOR.P[s] > 0])
               for s in np.flatnonzero(LIVE)), "ICU-Sepsis reward is not an entry reward on live transitions."
    del _x_R

### II.3b Gumbel-max sepsis simulator: exact enumeration of the published code

**Source.** `sepsisSimDiabetes/{State,Action,MDP,DataGenerator}.py` of `clinicalml/gumbel-max-scm`, branch `master`
(commit d62995d) and branch `sim-v2` (commit f923f77). The four files are byte-identical across the branches except
for one line (`MDP.py`, line 253): for non-diabetic patients the upward glucose fluctuation is `min(1, glucose + 1)` on
`master`, which sends every glucose level to "low 1", and `min(4, glucose + 1)` on `sim-v2`. On `master`, a
non-diabetic glucose level at or below "low 1" can therefore never return to normal, so such patients can never be
discharged. `XCFG.gumbel_variant` selects the variant (default `sim-v2`); both are built, checked and compared
(`xenv/gumbel_simulator_report.json`). The pinned SHA-256 digests are re-verified online when
`VERIFY_PROVENANCE_ONLINE` (text only, never executed).

**State and actions.** Full state (hr, sbp, o2, glucose, abx, vaso, vent, diabetic) with levels (3, 3, 2, 5, 2, 2,
2, 2), full index = 720·diabetic + 240·hr + 80·sbp + 40·o2 + 8·glucose + 4·abx + 2·vaso + vent (`State.get_state_idx
('full')`); 1440 states, 720 per diabetes group. Action a = 4·abx + 2·vent + vaso (`Action`; the last two bits are
ordered differently from the state). Part II exposes the full Markov state, diabetes included: the hidden-confounder
setting of the original paper is out of scope, and the system is an MDP with exactly known dynamics.

**Step (`MDP.transition`).** Antibiotics, then ventilation, then vasopressors, then fluctuations, each random event an
independent `np.random.uniform(0, 1)` draw compared with the source's thresholds. A draw compared with two
thresholds is one event with three outcomes (down / up / stay). The enumerator `gumbel_transition_dist` reproduces
each source method as a transformer of a finite distribution over states, in the source's order and with the same
short-circuit conditions; control-flow conditions read flags that are identical on every branch, and this is
asserted. `gumbel_step_sample` is an independent line-by-line port of the stochastic step, used only to test the
enumeration.

**Absorbing collapse and horizon.** After the step, `calculateReward` gives −1 when at least 3 vitals are abnormal
(death) and +1 when no vital is abnormal and every treatment flag is off (discharge); glucose levels other than
normal count as abnormal in the code. All death states merge into one absorbing state (entry reward −1) and the
discharge states into another (+1). There are 416 death and 1 discharge state per diabetes group (vital patterns
with k abnormal vitals: 1, 9, 28, 36, 16 for k = 0..4, times 8 flag patterns), so 303 live states per group and
606 in total. Because the post-step flags equal the action bits, discharge is reachable only with action 0. The
original episodes stop after a fixed number of steps. Here a third absorbing state, **timeout** (entry reward 0),
is entered with probability h = `XCFG.gumbel_timeout_hazard` before every step: P[s, a] = h·e_timeout + (1 − h)·
P_sim[s, a]. The resulting geometric horizon (mean 1/h decisions) makes every policy proper, so γ = 1 evaluation is
well posed. Without the timeout, policies that keep a treatment on forever never absorb; this is reported as
`trap_states_without_timeout`. Live states keep the full-index order; death, discharge and timeout follow.

**Initial distribution.** `DataGenerator.simulate` calls `MDP.get_new_state()` without arguments. Every attempt of
`generate_random_state(None)` redraws the diabetes indicator (Bernoulli 0.2) together with hr, sbp ∼ (.25, .5, .25),
o2 ∼ (.2, .8) and glucose ∼ (.05, .15, .6, .15, .05) (non-diabetic) or (.01, .05, .15, .6, .19) (diabetic), with
all flags off, until the state is not absorbing. d0 is therefore the **joint** product distribution conditioned on
1 ≤ #abnormal ≤ 2. It has 37 support states per group, acceptance 0.710 (non-diabetic) and 0.665 (diabetic), and a
diabetic share of 0.133/0.701 ≈ 0.190 rather than 0.2.

**Physician.** `gumbel_physician(env, ε)` puts 1 − ε on the action of the exact optimum of the true absorbing
environment (`x_optimal`, γ = 1 with the timeout hazard) and ε/7 on each other action. The original study instead
ran policy iteration with γ = 0.99 on a non-absorbing 1440-state MDP and used ε = 0.05. The ε-softening is the
same; the planner differs. With this ε-optimal physician the count masks are near-singleton and contain the true
optimum in most decision states, so unsupported actions are (slightly) worse in truth, and the termination rule is
favoured by construction (construction report: `single_admissible_fraction`, `true_optimum_in_mask_fraction`).
The control `kind = "withhold-vaso"` (cell suffix `|phys-withhold-vaso`) softens instead the exact optimum among
policies that never give vasopressors: the vasopressor actions are then rarely supported although they can be
beneficial, the Gumbel analogue of Garnet h < 0.

In [ ]:
# ============================================================
# II.3b Gumbel-max sepsis simulator (exact)
# ============================================================
EXECLOG.mark("II.3b Gumbel-max simulator")


class GumbelSim:
    """Layout, pinned provenance and initial-state constants of clinicalml/gumbel-max-scm/sepsisSimDiabetes."""
    FACTORS = ("hr", "sbp", "o2", "glucose", "abx", "vaso", "vent", "diabetic")
    LEVELS = (3, 3, 2, 5, 2, 2, 2, 2)
    NORMAL_LEVEL = (1, 1, 1, 2)                      # hr, sbp, o2, glucose (State.get_num_abnormal)
    STRIDES = (240, 80, 40, 8, 4, 2, 1, 720)         # State.get_state_idx('full'); diabetes is the leading digit
    N_OBS, N_FULL, N_ACTIONS = 720, 1440, 8
    P_DIABETES = 0.2                                 # MDP(p_diabetes=0.2) and DataGenerator.simulate default
    INIT_HR = (0.25, 0.5, 0.25)
    INIT_SBP = (0.25, 0.5, 0.25)
    INIT_O2 = (0.2, 0.8)
    INIT_GLUCOSE = ((0.05, 0.15, 0.6, 0.15, 0.05), (0.01, 0.05, 0.15, 0.6, 0.19))   # non-diabetic, diabetic
    VARIANTS = ("master", "sim-v2")
    REPOSITORY = "clinicalml/gumbel-max-scm"
    PATH = "sepsisSimDiabetes"
    COMMITS = {"master": "d62995deb5cb51e3ecf2859f4cbbc13f651131a2",
               "sim-v2": "f923f7747ef00103337fa1e88dfad92b98db2993"}
    SHA256 = {
        "master": {"Action.py": "56da9473140a90bde00691a10393e0dfaacd4c694573460a76861fa1dd22f841",
                   "DataGenerator.py": "d6d2a6f3cf0362f5c9512785f9d8e214b943bf3be4562a23c1ddcdb73c4a88fa",
                   "MDP.py": "8cd58b239e048472ec3a9b16ea2444288b24cdfff381bcea2e0294b97df2f35f",
                   "State.py": "23269aa1496905b3d55a9cbbca3138e33e6c1d1f0ffa7eec1d1e09870b5c10cd"},
        "sim-v2": {"Action.py": "56da9473140a90bde00691a10393e0dfaacd4c694573460a76861fa1dd22f841",
                   "DataGenerator.py": "d6d2a6f3cf0362f5c9512785f9d8e214b943bf3be4562a23c1ddcdb73c4a88fa",
                   "MDP.py": "a68aa3e3f27f3f222616a307116698ff13c19292c87df82775058578c901060e",
                   "State.py": "23269aa1496905b3d55a9cbbca3138e33e6c1d1f0ffa7eec1d1e09870b5c10cd"},
    }


_GX_HR, _GX_SBP, _GX_O2, _GX_GLU, _GX_ABX, _GX_VASO, _GX_VENT, _GX_DIAB = range(8)


def gumbel_full_index(st: Sequence[int]) -> int:
    """State.get_state_idx('full') for a state tuple (hr, sbp, o2, glucose, abx, vaso, vent, diabetic)."""
    return int(sum(int(v) * w for v, w in zip(st, GumbelSim.STRIDES)))


def gumbel_decode_full(idx: int) -> Tuple[int, ...]:
    """State.set_state_by_idx(idx, 'full') with integer arithmetic (the source's float floor division is exact here)."""
    idx = int(idx)
    assert 0 <= idx < GumbelSim.N_FULL, idx
    diab, m = divmod(idx, 720)
    hr, m = divmod(m, 240)
    sbp, m = divmod(m, 80)
    o2, m = divmod(m, 40)
    glu, m = divmod(m, 8)
    abx, m = divmod(m, 4)
    vaso, vent = divmod(m, 2)
    return (hr, sbp, o2, glu, abx, vaso, vent, diab)


def gumbel_action_bits(action: int) -> Tuple[int, int, int]:
    """Action(action_idx=a): (antibiotic, ventilation, vasopressors) = (a // 4, (a % 4) // 2, a % 2)."""
    a = int(action)
    assert 0 <= a < GumbelSim.N_ACTIONS, a
    return a // 4, (a % 4) // 2, a % 2


def gumbel_num_abnormal(st: Sequence[int]) -> int:
    return int(sum(int(st[i]) != n for i, n in enumerate(GumbelSim.NORMAL_LEVEL)))


def gumbel_on_treatment(st: Sequence[int]) -> bool:
    return bool(st[_GX_ABX] or st[_GX_VASO] or st[_GX_VENT])


def gumbel_reward(st: Sequence[int]) -> int:
    """MDP.calculateReward on the post-step state: -1 death, +1 discharge, 0 otherwise."""
    n = gumbel_num_abnormal(st)
    if n >= 3:
        return -1
    if n == 0 and not gumbel_on_treatment(st):
        return 1
    return 0


def gumbel_is_absorbing(st: Sequence[int]) -> bool:
    """State.check_absorbing_state (identical classification to calculateReward != 0)."""
    return gumbel_reward(st) != 0


# ---------- exact enumeration: each source method as a transformer of a finite distribution {state: prob} ----------
def _gx_rep(st: Tuple[int, ...], i: int, v: int) -> Tuple[int, ...]:
    s = list(st)
    s[i] = int(v)
    return tuple(s)


def _gx_case(d: Dict[tuple, float], branch: Callable[[tuple], List[Tuple[float, tuple]]]) -> Dict[tuple, float]:
    out: Dict[tuple, float] = {}
    for s, p in d.items():
        for q, t in branch(s):
            if q > 0.0:
                out[t] = out.get(t, 0.0) + p * q
    return out


def _gx_u(s: tuple, cuts: Sequence[float], effects: Sequence[Callable[[tuple], tuple]]) -> List[Tuple[float, tuple]]:
    """Outcomes of one u = np.random.uniform(0, 1) tested as `if u < cuts[0]: ... elif u < cuts[1]: ...`."""
    out, lo = [], 0.0
    for c, eff in zip(cuts, effects):
        out.append((c - lo, eff(s)))
        lo = c
    out.append((1.0 - lo, s))
    return out


def _gx_flag(d: Dict[tuple, float], i: int) -> int:
    vals = {s[i] for s in d}
    assert len(vals) == 1, "a control-flow flag differs across enumerated branches"
    return int(next(iter(vals)))


def _gx_antibiotics_on(d):
    d = _gx_case(d, lambda s: [(1.0, _gx_rep(s, _GX_ABX, 1))])
    d = _gx_case(d, lambda s: _gx_u(s, (0.5,), (lambda t: _gx_rep(t, _GX_HR, 1),)) if s[_GX_HR] == 2 else [(1.0, s)])
    d = _gx_case(d, lambda s: _gx_u(s, (0.5,), (lambda t: _gx_rep(t, _GX_SBP, 1),)) if s[_GX_SBP] == 2 else [(1.0, s)])
    return d


def _gx_antibiotics_off(d):
    if _gx_flag(d, _GX_ABX) == 1:
        d = _gx_case(d, lambda s: _gx_u(s, (0.1,), (lambda t: _gx_rep(t, _GX_HR, 2),)) if s[_GX_HR] == 1 else [(1.0, s)])
        d = _gx_case(d, lambda s: _gx_u(s, (0.1,), (lambda t: _gx_rep(t, _GX_SBP, 2),)) if s[_GX_SBP] == 1 else [(1.0, s)])
        d = _gx_case(d, lambda s: [(1.0, _gx_rep(s, _GX_ABX, 0))])
    return d


def _gx_vent_on(d):
    d = _gx_case(d, lambda s: [(1.0, _gx_rep(s, _GX_VENT, 1))])
    return _gx_case(d, lambda s: _gx_u(s, (0.7,), (lambda t: _gx_rep(t, _GX_O2, 1),)) if s[_GX_O2] == 0 else [(1.0, s)])


def _gx_vent_off(d):
    if _gx_flag(d, _GX_VENT) == 1:
        d = _gx_case(d, lambda s: _gx_u(s, (0.1,), (lambda t: _gx_rep(t, _GX_O2, 0),)) if s[_GX_O2] == 1 else [(1.0, s)])
        d = _gx_case(d, lambda s: [(1.0, _gx_rep(s, _GX_VENT, 0))])
    return d


def _gx_vaso_on(d):
    d = _gx_case(d, lambda s: [(1.0, _gx_rep(s, _GX_VASO, 1))])
    if _gx_flag(d, _GX_DIAB) == 0:
        # if u < 0.7: sbp 0 -> 1, 1 -> 2 (no change at 2; the draw is consumed in every case)
        up = {0: 1, 1: 2, 2: 2}
        return _gx_case(d, lambda s: _gx_u(s, (0.7,), (lambda t: _gx_rep(t, _GX_SBP, up[t[_GX_SBP]]),)))

    def sbp_branch(s):
        if s[_GX_SBP] == 1:
            return _gx_u(s, (0.9,), (lambda t: _gx_rep(t, _GX_SBP, 2),))
        if s[_GX_SBP] == 0:
            return _gx_u(s, (0.5, 0.9), (lambda t: _gx_rep(t, _GX_SBP, 1), lambda t: _gx_rep(t, _GX_SBP, 2)))
        return [(1.0, s)]
    d = _gx_case(d, sbp_branch)
    return _gx_case(d, lambda s: _gx_u(s, (0.5,), (lambda t: _gx_rep(t, _GX_GLU, min(4, t[_GX_GLU] + 1)),)))


def _gx_vaso_off(d):
    if _gx_flag(d, _GX_VASO) == 1:
        p = 0.1 if _gx_flag(d, _GX_DIAB) == 0 else 0.05
        d = _gx_case(d, lambda s: _gx_u(s, (p,), (lambda t: _gx_rep(t, _GX_SBP, max(0, t[_GX_SBP] - 1)),)))
        d = _gx_case(d, lambda s: [(1.0, _gx_rep(s, _GX_VASO, 0))])
    return d


def _gx_fluctuate(d, hr_f: bool, sbp_f: bool, o2_f: bool, glu_f: bool, variant: str):
    def pm(i: int, top: int, cuts: Tuple[float, float]):
        return lambda s: _gx_u(s, cuts, (lambda t: _gx_rep(t, i, max(0, t[i] - 1)),
                                         lambda t: _gx_rep(t, i, min(top, t[i] + 1))))
    if hr_f:
        d = _gx_case(d, pm(_GX_HR, 2, (0.1, 0.2)))
    if sbp_f:
        d = _gx_case(d, pm(_GX_SBP, 2, (0.1, 0.2)))
    if o2_f:
        d = _gx_case(d, pm(_GX_O2, 1, (0.1, 0.2)))
    if glu_f:
        cap = 1 if variant == "master" else 4        # MDP.py line 253: master min(1, g + 1) (bug), sim-v2 min(4, g + 1)
        nondiab, diab = pm(_GX_GLU, cap, (0.1, 0.2)), pm(_GX_GLU, 4, (0.3, 0.6))
        d = _gx_case(d, lambda s: nondiab(s) if s[_GX_DIAB] == 0 else diab(s))
    return d


def gumbel_transition_dist(st: Sequence[int], action: int, variant: str) -> Dict[Tuple[int, ...], float]:
    """Exact next-full-state distribution of MDP.transition(Action(action_idx=action)) from full state `st`."""
    assert variant in GumbelSim.VARIANTS, variant
    a_abx, a_vent, a_vaso = gumbel_action_bits(action)
    d = {tuple(int(x) for x in st): 1.0}
    if a_abx == 1:
        d = _gx_antibiotics_on(d)
        hr_f = sbp_f = False
    elif _gx_flag(d, _GX_ABX) == 1:
        d = _gx_antibiotics_off(d)
        hr_f = sbp_f = False
    else:
        hr_f = sbp_f = True
    if a_vent == 1:
        d = _gx_vent_on(d)
        o2_f = False
    elif _gx_flag(d, _GX_VENT) == 1:
        d = _gx_vent_off(d)
        o2_f = False
    else:
        o2_f = True
    glu_f = True
    if a_vaso == 1:
        d = _gx_vaso_on(d)
        sbp_f = False
        glu_f = False
    elif _gx_flag(d, _GX_VASO) == 1:
        d = _gx_vaso_off(d)
        sbp_f = False
    d = _gx_fluctuate(d, hr_f, sbp_f, o2_f, glu_f, variant)
    total = sum(d.values())
    assert abs(total - 1.0) < 1e-12, (st, action, total)
    return d


def gumbel_initial_distribution(p_diabetes: float = GumbelSim.P_DIABETES) -> Tuple[Dict[Tuple[int, ...], float], Dict[str, Any]]:
    """Exact law of MDP.get_new_state(): the joint product proposal conditioned on a non-absorbing state."""
    joint: Dict[Tuple[int, ...], float] = {}
    acc = {0: 0.0, 1: 0.0}
    for diab, pd_ in ((0, 1.0 - p_diabetes), (1, p_diabetes)):
        for hr, sbp, o2, glu in itertools.product(range(3), range(3), range(2), range(5)):
            st = (hr, sbp, o2, glu, 0, 0, 0, diab)
            q = GumbelSim.INIT_HR[hr] * GumbelSim.INIT_SBP[sbp] * GumbelSim.INIT_O2[o2] * GumbelSim.INIT_GLUCOSE[diab][glu]
            if not gumbel_is_absorbing(st):
                acc[diab] += q
                joint[st] = pd_ * q
    Z = sum(joint.values())
    d0 = {st: p / Z for st, p in joint.items()}
    info = {"acceptance_nondiabetic": acc[0], "acceptance_diabetic": acc[1], "acceptance_overall": Z,
            "d0_diabetic_share": float(sum(p for st, p in d0.items() if st[_GX_DIAB] == 1)),
            "support_nondiabetic": int(sum(st[_GX_DIAB] == 0 for st in d0)),
            "support_diabetic": int(sum(st[_GX_DIAB] == 1 for st in d0))}
    return d0, info


_GUMBEL_SPACE: Dict[str, Any] = {}


def gumbel_state_space() -> Dict[str, Any]:
    """Full states, their absorbing class, and the live full-state indices in full-index order (cached)."""
    if not _GUMBEL_SPACE:
        full = [gumbel_decode_full(i) for i in range(GumbelSim.N_FULL)]
        assert all(gumbel_full_index(st) == i for i, st in enumerate(full)), "full-index round trip failed"
        assert len(set(full)) == GumbelSim.N_FULL
        kind = np.array(["death" if gumbel_reward(st) == -1 else "discharge" if gumbel_reward(st) == 1 else "live"
                         for st in full])
        live_full = np.flatnonzero(kind == "live")
        _GUMBEL_SPACE.update(full=full, kind=kind, live_full=live_full,
                             pos={int(f): k for k, f in enumerate(live_full)})
    return _GUMBEL_SPACE


def _gumbel_expected_counts() -> Dict[str, int]:
    """Per-group death / discharge / live / initial counts re-derived combinatorially from the source rules."""
    abn = [lv - 1 for lv in GumbelSim.LEVELS[:4]]            # abnormal levels per vital: 2, 2, 1, 4
    poly = np.array([1])
    for k in abn:
        poly = np.convolve(poly, [1, k])                    # coefficient j = vital patterns with j abnormal vitals
    flags = 2 ** 3
    death = int(poly[3:].sum()) * flags
    discharge = int(poly[0])                                # zero abnormal vitals and every flag off
    return {"vital_patterns_by_abnormal": poly.astype(int).tolist(), "death_per_group": death,
            "discharge_per_group": discharge, "live_per_group": GumbelSim.N_OBS - death - discharge,
            "initial_support_per_group": int(poly[1] + poly[2])}


def gumbel_true_env_report(variant: str, timeout_hazard: float) -> Tuple[FiniteEnv, Dict[str, Any]]:
    """Exact absorbing Gumbel-max sepsis MDP (live full states, then death -1, discharge +1, timeout 0) and the report
    of its structural checks (every check below raises on failure)."""
    assert variant in GumbelSim.VARIANTS, variant
    h = float(timeout_hazard)
    assert 0.0 < h < 1.0, "the timeout hazard must lie in (0, 1) (it defines the absorbing horizon)"
    space = gumbel_state_space()
    full, live_full, pos = space["full"], space["live_full"], space["pos"]
    n_live, A = len(live_full), GumbelSim.N_ACTIONS
    death, discharge, timeout = n_live, n_live + 1, n_live + 2
    S = n_live + 3
    P_sim = np.zeros((n_live, A, S))
    for k, f in enumerate(live_full):
        for a in range(A):
            for nxt, p in gumbel_transition_dist(full[f], a, variant).items():
                r = gumbel_reward(nxt)
                j = death if r == -1 else discharge if r == 1 else pos[gumbel_full_index(nxt)]
                P_sim[k, a, j] += p
    P = np.zeros((S, A, S))
    P[:n_live] = (1.0 - h) * P_sim
    P[:n_live, :, timeout] += h
    for t in (death, discharge, timeout):
        P[t, :, t] = 1.0
    entry = np.zeros(S)
    entry[death], entry[discharge] = -1.0, 1.0
    d0_map, init_info = gumbel_initial_distribution()
    d0 = np.zeros(S)
    for st, p in d0_map.items():
        d0[pos[gumbel_full_index(st)]] = p
    feats = np.zeros((S, len(GumbelSim.FACTORS)), dtype=np.float32)
    scale = np.array([lv - 1 for lv in GumbelSim.LEVELS], dtype=np.float32)
    feats[:n_live] = np.array([full[f] for f in live_full], dtype=np.float32) / scale

    # ---------- structural asserts (re-derived from the source logic) ----------
    exp = _gumbel_expected_counts()
    kind = space["kind"]
    diab = np.array([st[_GX_DIAB] for st in full])
    groups = {}
    for g in (0, 1):
        sel = diab == g
        groups[g] = {"states": int(sel.sum()), "death": int((kind[sel] == "death").sum()),
                     "discharge": int((kind[sel] == "discharge").sum()), "live": int((kind[sel] == "live").sum()),
                     "initial_support": int(sum(1 for st in d0_map if st[_GX_DIAB] == g))}
        assert groups[g]["states"] == GumbelSim.N_OBS, groups
        assert groups[g]["death"] == exp["death_per_group"] == 416, (groups, exp)
        assert groups[g]["discharge"] == exp["discharge_per_group"] == 1, (groups, exp)
        assert groups[g]["live"] == exp["live_per_group"] == 303, (groups, exp)
        assert groups[g]["initial_support"] == exp["initial_support_per_group"] == 37, (groups, exp)
    assert len(full) == GumbelSim.N_FULL and n_live == 606
    disc_full = [f for f in range(GumbelSim.N_FULL) if kind[f] == "discharge"]
    assert sorted(f % GumbelSim.N_OBS for f in disc_full) == [376, 376], disc_full
    row_dev = float(np.abs(P.sum(axis=2) - 1.0).max())
    assert row_dev < 1e-12 and (P >= 0).all(), row_dev
    assert np.all(P[:n_live, :, timeout] == h) and np.all(P_sim[:, :, timeout] == 0.0)
    assert np.all(P_sim[:, 1:, discharge] == 0.0) and P_sim[:, 0, discharge].max() > 0, "discharge only via action 0"
    live_f = np.array([full[f] for f in live_full])
    for a in range(A):
        bits = gumbel_action_bits(a)
        flags = (bits[0], bits[2], bits[1])                # state order (abx, vaso, vent); action order (abx, vent, vaso)
        succ = P_sim[:, a, :n_live] > 0
        rows, cols = np.nonzero(succ)
        assert np.all(live_f[cols][:, [_GX_ABX, _GX_VASO, _GX_VENT]] == np.array(flags)), "post-step flags != action bits"
        assert np.all(live_f[cols][:, _GX_DIAB] == live_f[rows][:, _GX_DIAB]), "diabetes changed during a step"
    assert abs(d0.sum() - 1.0) < 1e-12 and int((d0 > 0).sum()) == 74
    no_timeout = np.zeros_like(P)
    no_timeout[:n_live] = P_sim
    for t in (death, discharge, timeout):
        no_timeout[t, :, t] = 1.0
    live_mask = np.r_[np.ones(n_live, bool), np.zeros(3, bool)]
    structure = {"full_states": len(full), "live_states": n_live, "groups": {str(g): v for g, v in groups.items()},
                 "expected_counts": exp, "max_row_sum_deviation": row_dev, "initial": init_info,
                 "max_branches_per_pair": int((P_sim > 0).sum(axis=2).max()),
                 "trap_states_without_timeout": x_trap_states(no_timeout, live_mask, np.ones((S, A), bool))}
    del no_timeout
    env = FiniteEnv(name=f"gumbel-true|{variant}|h{h}", kind="gumbel", P=P, entry_reward=entry, d0=d0,
                    terminals=(death, discharge, timeout), sink=death, goal=discharge, mask=np.ones((S, A), bool),
                    features=feats,
                    meta={"variant": variant, "timeout_hazard": h, "source": GumbelSim.REPOSITORY,
                          "commit": GumbelSim.COMMITS[variant], "n_live": n_live, "death_state": death,
                          "discharge_state": discharge, "timeout_state": timeout, "dangling_target": timeout,
                          "dangling_target_label": "timeout", "censoring_hazard": h})
    assert x_trap_states(env.P, env.live, env.mask) == 0, "every policy must be proper with the timeout hazard"
    return env, structure


def gumbel_true_env(variant: str = XCFG.gumbel_variant, timeout_hazard: float = XCFG.gumbel_timeout_hazard) -> FiniteEnv:
    """Exact absorbing Gumbel-max sepsis MDP: live full states, then death (-1), discharge (+1), timeout (0)."""
    return gumbel_true_env_report(variant, timeout_hazard)[0]


def gumbel_decode_table(env: FiniteEnv) -> pd.DataFrame:
    """Env index -> full index and factor levels (-1 on the three absorbing states)."""
    space = gumbel_state_space()
    n_live = int(env.meta["n_live"])
    rows = []
    for k, f in enumerate(space["live_full"]):
        st = space["full"][f]
        rows.append({"state": k, "kind": "live", "full_index": int(f), **dict(zip(GumbelSim.FACTORS, st)),
                     "n_abnormal": gumbel_num_abnormal(st), "d0": float(env.d0[k])})
    for lab, t in (("death", env.sink), ("discharge", env.goal), ("timeout", int(env.meta["timeout_state"]))):
        rows.append({"state": t, "kind": lab, "full_index": -1, **{f: -1 for f in GumbelSim.FACTORS},
                     "n_abnormal": -1, "d0": 0.0})
    df = pd.DataFrame(rows).sort_values("state").reset_index(drop=True)
    assert len(df) == env.S == n_live + 3 and np.array_equal(df["state"].to_numpy(), np.arange(env.S))
    return df


GUMBEL_PHYSICIAN_KINDS = ("optimal", "withhold-vaso")


def gumbel_physician(env: FiniteEnv, eps: float, kind: str = "optimal") -> np.ndarray:
    """eps-softened physician of `env` under its own dynamics: 1 - eps on its action, eps/(A-1) on every other action.

    kind 'optimal': the exact optimum of `env`. kind 'withhold-vaso': the exact optimum among policies that never give
    vasopressors on live states (admissible-constrained optimum under the mask 1[vasopressor bit of a = 0]). Its
    vasopressor actions get eps/(A-1) each, so they are rarely supported in the hospital data although they can be
    beneficial in truth (the Gumbel analogue of Garnet h < 0)."""
    assert 0.0 <= eps < 1.0, eps
    if kind == "optimal":
        pi_opt, _, _ = x_optimal(env, "given")
    elif kind == "withhold-vaso":
        no_vaso = np.ones((env.S, env.A), dtype=bool)
        no_vaso[env.live] = np.array([gumbel_action_bits(a)[2] == 0 for a in range(env.A)])[None, :]
        pi_opt, _, _ = x_optimal(env.variant(f"{env.name}|no-vasopressors", mask=no_vaso), "given", constrained=True)
    else:
        raise ValueError(f"unknown Gumbel physician kind {kind!r} (expected one of {GUMBEL_PHYSICIAN_KINDS})")
    act = pi_opt.argmax(axis=1)
    if kind == "withhold-vaso":
        assert all(gumbel_action_bits(a)[2] == 0 for a in act[env.live]), "withhold-vaso physician gives vasopressors"
    pi = np.full((env.S, env.A), eps / (env.A - 1))
    pi[np.arange(env.S), act] = 1.0 - eps
    return validate_policy(pi, f"Gumbel physician ({kind}) eps={eps}", shape=(env.S, env.A))


# ---------- literal port of the stochastic step (used ONLY to test the enumeration) ----------
def gumbel_step_sample(state_tuple: Sequence[int], action: int, rng: np.random.Generator, variant: str):
    """One call of MDP.transition(Action(action_idx=action)) + calculateReward, np.random.uniform -> rng.uniform.

    Returns the next full-state tuple, or the label 'death' / 'discharge' when the reward is -1 / +1."""
    assert variant in GumbelSim.VARIANTS
    hr, sbp, o2, glu, abx, vaso, vent, diab = (int(x) for x in state_tuple)
    s = {"hr_state": hr, "sysbp_state": sbp, "percoxyg_state": o2, "glucose_state": glu, "antibiotic_state": abx,
         "vaso_state": vaso, "vent_state": vent, "diabetic_idx": diab}
    a = int(action)
    antibiotic, ventilation, vasopressors = a // 4, (a % 4) // 2, a % 2
    if antibiotic == 1:                                           # transition_antibiotics_on
        s["antibiotic_state"] = 1
        if s["hr_state"] == 2 and rng.uniform(0, 1) < 0.5:
            s["hr_state"] = 1
        if s["sysbp_state"] == 2 and rng.uniform(0, 1) < 0.5:
            s["sysbp_state"] = 1
        hr_fluctuate = False
        sysbp_fluctuate = False
    elif s["antibiotic_state"] == 1:                              # transition_antibiotics_off
        if s["antibiotic_state"] == 1:
            if s["hr_state"] == 1 and rng.uniform(0, 1) < 0.1:
                s["hr_state"] = 2
            if s["sysbp_state"] == 1 and rng.uniform(0, 1) < 0.1:
                s["sysbp_state"] = 2
            s["antibiotic_state"] = 0
        hr_fluctuate = False
        sysbp_fluctuate = False
    else:
        hr_fluctuate = True
        sysbp_fluctuate = True
    if ventilation == 1:                                          # transition_vent_on
        s["vent_state"] = 1
        if s["percoxyg_state"] == 0 and rng.uniform(0, 1) < 0.7:
            s["percoxyg_state"] = 1
        percoxyg_fluctuate = False
    elif s["vent_state"] == 1:                                    # transition_vent_off
        if s["vent_state"] == 1:
            if s["percoxyg_state"] == 1 and rng.uniform(0, 1) < 0.1:
                s["percoxyg_state"] = 0
            s["vent_state"] = 0
        percoxyg_fluctuate = False
    else:
        percoxyg_fluctuate = True
    glucose_fluctuate = True
    if vasopressors == 1:                                         # transition_vaso_on
        s["vaso_state"] = 1
        if s["diabetic_idx"] == 0:
            if rng.uniform(0, 1) < 0.7:
                if s["sysbp_state"] == 0:
                    s["sysbp_state"] = 1
                elif s["sysbp_state"] == 1:
                    s["sysbp_state"] = 2
        else:
            if s["sysbp_state"] == 1:
                if rng.uniform(0, 1) < 0.9:
                    s["sysbp_state"] = 2
            elif s["sysbp_state"] == 0:
                up_prob = rng.uniform(0, 1)
                if up_prob < 0.5:
                    s["sysbp_state"] = 1
                elif up_prob < 0.9:
                    s["sysbp_state"] = 2
            if rng.uniform(0, 1) < 0.5:
                s["glucose_state"] = min(4, s["glucose_state"] + 1)
        sysbp_fluctuate = False
        glucose_fluctuate = False
    elif s["vaso_state"] == 1:                                    # transition_vaso_off
        if s["vaso_state"] == 1:
            if s["diabetic_idx"] == 0:
                if rng.uniform(0, 1) < 0.1:
                    s["sysbp_state"] = max(0, s["sysbp_state"] - 1)
            else:
                if rng.uniform(0, 1) < 0.05:
                    s["sysbp_state"] = max(0, s["sysbp_state"] - 1)
            s["vaso_state"] = 0
        sysbp_fluctuate = False
    if hr_fluctuate:                                              # transition_fluctuate
        hr_prob = rng.uniform(0, 1)
        if hr_prob < 0.1:
            s["hr_state"] = max(0, s["hr_state"] - 1)
        elif hr_prob < 0.2:
            s["hr_state"] = min(2, s["hr_state"] + 1)
    if sysbp_fluctuate:
        sysbp_prob = rng.uniform(0, 1)
        if sysbp_prob < 0.1:
            s["sysbp_state"] = max(0, s["sysbp_state"] - 1)
        elif sysbp_prob < 0.2:
            s["sysbp_state"] = min(2, s["sysbp_state"] + 1)
    if percoxyg_fluctuate:
        percoxyg_prob = rng.uniform(0, 1)
        if percoxyg_prob < 0.1:
            s["percoxyg_state"] = max(0, s["percoxyg_state"] - 1)
        elif percoxyg_prob < 0.2:
            s["percoxyg_state"] = min(1, s["percoxyg_state"] + 1)
    if glucose_fluctuate:
        glucose_prob = rng.uniform(0, 1)
        if s["diabetic_idx"] == 0:
            if glucose_prob < 0.1:
                s["glucose_state"] = max(0, s["glucose_state"] - 1)
            elif glucose_prob < 0.2:
                s["glucose_state"] = (min(1, s["glucose_state"] + 1) if variant == "master"
                                      else min(4, s["glucose_state"] + 1))
        else:
            if glucose_prob < 0.3:
                s["glucose_state"] = max(0, s["glucose_state"] - 1)
            elif glucose_prob < 0.6:
                s["glucose_state"] = min(4, s["glucose_state"] + 1)
    num_abnormal = ((s["hr_state"] != 1) + (s["sysbp_state"] != 1) + (s["percoxyg_state"] != 1)
                    + (s["glucose_state"] != 2))                   # calculateReward
    on_treatment = not (s["antibiotic_state"] == 0 and s["vaso_state"] == 0 and s["vent_state"] == 0)
    if num_abnormal >= 3:
        return "death"
    if num_abnormal == 0 and not on_treatment:
        return "discharge"
    return (s["hr_state"], s["sysbp_state"], s["percoxyg_state"], s["glucose_state"], s["antibiotic_state"],
            s["vaso_state"], s["vent_state"], s["diabetic_idx"])


def gumbel_initial_sample(rng: np.random.Generator, p_diabetes: float = GumbelSim.P_DIABETES) -> Tuple[int, ...]:
    """Literal port of MDP.get_new_state() (option 'random'): diabetes is redrawn on every rejected attempt."""
    def generate_random_state(diabetic_idx=None):
        if diabetic_idx is None:
            diabetic_idx = rng.binomial(1, p_diabetes)
        hr_state = rng.choice(np.arange(3), p=np.array([.25, .5, .25]))
        sysbp_state = rng.choice(np.arange(3), p=np.array([.25, .5, .25]))
        percoxyg_state = rng.choice(np.arange(2), p=np.array([.2, .8]))
        if diabetic_idx == 0:
            glucose_state = rng.choice(np.arange(5), p=np.array([.05, .15, .6, .15, .05]))
        else:
            glucose_state = rng.choice(np.arange(5), p=np.array([.01, .05, .15, .6, .19]))
        return (int(hr_state), int(sysbp_state), int(percoxyg_state), int(glucose_state), 0, 0, 0, int(diabetic_idx))

    def check_absorbing_state(st):
        num_abnormal = (st[0] != 1) + (st[1] != 1) + (st[2] != 1) + (st[3] != 2)
        return num_abnormal >= 3 or (num_abnormal == 0 and not (st[4] or st[5] or st[6]))

    init_state = generate_random_state(None)
    while check_absorbing_state(init_state):
        init_state = generate_random_state(None)
    return init_state


def _x_binned_chi2(counts: np.ndarray, p: np.ndarray, n: int, min_expected: float = 5.0) -> Tuple[float, int]:
    """Pearson chi-square over outcomes with p > 0, merging the rarest outcomes until every bin expects >= 5."""
    keep = p > 0
    pk, ck = p[keep], counts[keep]
    bins_p, bins_c, acc_p, acc_c = [], [], 0.0, 0.0
    for j in np.argsort(pk, kind="stable"):
        acc_p += pk[j]
        acc_c += ck[j]
        if acc_p * n >= min_expected:
            bins_p.append(acc_p)
            bins_c.append(acc_c)
            acc_p = acc_c = 0.0
    if acc_p > 0 or acc_c > 0:
        if bins_p:
            bins_p[-1] += acc_p
            bins_c[-1] += acc_c
        else:
            bins_p.append(acc_p)
            bins_c.append(acc_c)
    E = n * np.asarray(bins_p)
    return float((((np.asarray(bins_c) - E) ** 2) / E).sum()), len(bins_p) - 1


def gumbel_exactness_check(env: FiniteEnv, n_pairs: int, n_draws: int, rng: np.random.Generator,
                           n_init: Optional[int] = None, p_min: float = 1e-3) -> Dict[str, Any]:
    """Exact rows P_sim = (P - h e_timeout) / (1 - h) versus `gumbel_step_sample` frequencies on random live pairs,
    and the exact d0 versus `gumbel_initial_sample` frequencies."""
    variant, h = env.meta["variant"], float(env.meta["timeout_hazard"])
    space = gumbel_state_space()
    n_live, A, timeout = int(env.meta["n_live"]), env.A, int(env.meta["timeout_state"])
    e_to = np.zeros(env.S)
    e_to[timeout] = 1.0
    t0 = time.perf_counter()
    pairs = rng.choice(n_live * A, size=min(int(n_pairs), n_live * A), replace=False)
    n = int(n_draws)
    chi2_tot, df_tot, impossible, max_z, max_z5, max_abs, n_cells = 0.0, 0, 0, 0.0, 0.0, 0.0, 0
    for flat in pairs:
        s, a = divmod(int(flat), A)
        st = space["full"][space["live_full"][s]]
        counts = np.zeros(env.S)
        for _ in range(n):
            nxt = gumbel_step_sample(st, a, rng, variant)
            j = (env.sink if nxt == "death" else env.goal) if isinstance(nxt, str) else space["pos"][gumbel_full_index(nxt)]
            counts[j] += 1
        p = (env.P[s, a] - h * e_to) / (1.0 - h)
        assert p[timeout] == 0.0 and abs(p.sum() - 1.0) < 1e-12
        freq = counts / n
        impossible += int(counts[p <= 0].sum())
        pos_ = p > 0
        pf = np.clip(p[pos_], 1.0 / n, 1.0 - 1.0 / n)            # SE floor: p in [1/n, 1 - 1/n] (deterministic rows)
        z = np.abs(freq[pos_] - p[pos_]) / np.sqrt(pf * (1.0 - pf) / n)
        assert np.isfinite(z).all()
        max_z = max(max_z, float(z.max()))
        big = n * p[pos_] >= 5.0
        if big.any():
            max_z5 = max(max_z5, float(z[big].max()))
        max_abs = max(max_abs, float(np.abs(freq - p).max()))
        n_cells += int(pos_.sum())
        c2, df = _x_binned_chi2(counts, p, n)
        chi2_tot += c2
        df_tot += df
    p_rows = float(stats.chi2.sf(chi2_tot, df_tot)) if df_tot > 0 else float("nan")
    t_rows = time.perf_counter() - t0
    n0 = int(n_init) if n_init is not None else 10 * n
    c0 = np.zeros(env.S)
    for _ in range(n0):
        c0[space["pos"][gumbel_full_index(gumbel_initial_sample(rng))]] += 1
    d0_impossible = int(c0[env.d0 <= 0].sum())
    c2_0, df_0 = _x_binned_chi2(c0, env.d0, n0)
    p_d0 = float(stats.chi2.sf(c2_0, df_0))
    out = {"variant": variant, "timeout_hazard": h, "pairs": int(len(pairs)), "draws_per_pair": n,
           "outcome_cells_with_positive_p": n_cells, "impossible_outcomes_observed": impossible,
           "max_abs_freq_minus_p": max_abs, "max_z": max_z, "max_z_cells_expected_ge_5": max_z5,
           "bonferroni_normal_z_at_1e-3": float(stats.norm.isf(1e-3 / (2 * max(n_cells, 1)))),
           "pooled_chi2": chi2_tot, "pooled_df": int(df_tot), "pooled_p_value": p_rows,
           "d0_samples": n0, "d0_impossible_observed": d0_impossible, "d0_chi2": c2_0, "d0_df": int(df_0),
           "d0_p_value": p_d0, "max_abs_d0_freq_minus_p": float(np.abs(c0 / n0 - env.d0).max()),
           "p_value_threshold": p_min, "seconds_rows": t_rows, "seconds_total": time.perf_counter() - t0}
    out["passes"] = bool(impossible == 0 and d0_impossible == 0 and np.isfinite(p_rows) and p_rows >= p_min
                         and p_d0 >= p_min)
    return out


def gumbel_source_provenance() -> Dict[str, Any]:
    """Re-fetch the pinned sources as text (never executed) and compare their SHA-256 with the pinned digests."""
    if not VERIFY_PROVENANCE_ONLINE:
        return {"status": "skipped (VERIFY_PROVENANCE_ONLINE=False)", "commits": GumbelSim.COMMITS}
    res = {}
    for variant, commit in GumbelSim.COMMITS.items():
        for fname, ref in GumbelSim.SHA256[variant].items():
            url = f"https://raw.githubusercontent.com/{GumbelSim.REPOSITORY}/{commit}/{GumbelSim.PATH}/{fname}"
            try:   # an HTTP error (e.g. 404: wrong pinned commit or path) is recorded, not mistaken for being offline
                with urllib.request.urlopen(url, timeout=15.0) as resp:
                    txt = resp.read().decode("utf-8")
                res[f"{variant}/{fname}"] = bool(sha256_bytes(txt.encode("utf-8")) == ref)
            except urllib.error.HTTPError as exc:
                print("  fetch failed:", url, "-> HTTP", exc.code)
                res[f"{variant}/{fname}"] = f"HTTP {exc.code}"
            except Exception as exc:  # offline or blocked
                print("  fetch failed:", url, "->", type(exc).__name__)
                res[f"{variant}/{fname}"] = None
    vals = list(res.values())
    status = ("verified" if all(v is True for v in vals) else "MISMATCH" if any(v is False for v in vals)
              else "MISSING (HTTP error)" if any(isinstance(v, str) for v in vals)
              else "unavailable (offline)" if all(v is None for v in vals)
              else "incomplete (some fetches failed offline; every fetched file matched)")
    return {"status": status, "sha256_matches": res, "commits": GumbelSim.COMMITS}


def gumbel_variant_comparison(envs: Dict[str, FiniteEnv]) -> Dict[str, Any]:
    """master vs sim-v2: the full-state step laws differ exactly on the non-diabetic pairs whose glucose fluctuates
    (vasopressors not given) from glucose >= 1, the only case where min(1, g+1) != min(4, g+1). After the absorbing
    collapse a predicted pair keeps identical rows only if every differing successor is absorbing in both variants."""
    em, ev = envs["master"], envs["sim-v2"]
    space = gumbel_state_space()
    n_live = int(em.meta["n_live"])
    diff = np.abs(em.P[:n_live] - ev.P[:n_live]).max(axis=2) > 0
    pred = np.zeros_like(diff)
    for k, f in enumerate(space["live_full"]):
        st = space["full"][f]
        for a in range(em.A):
            pred[k, a] = st[_GX_DIAB] == 0 and gumbel_action_bits(a)[2] == 0 and st[_GX_GLU] >= 1
    assert not (diff & ~pred).any(), "master / sim-v2 rows differ outside the glucose-fluctuation pairs"
    for k, a in zip(*np.nonzero(pred)):
        st = space["full"][space["live_full"][k]]
        dm, dv = gumbel_transition_dist(st, a, "master"), gumbel_transition_dist(st, a, "sim-v2")
        changed = [t for t in set(dm) | set(dv) if abs(dm.get(t, 0.0) - dv.get(t, 0.0)) > 0]
        assert changed, ("predicted pair with identical full-state law", st, a)
        if not diff[k, a]:
            assert all(gumbel_is_absorbing(t) for t in changed), ("collapsed rows equal but a live successor changed", st, a)
    out = {"rows_differing": int(diff.sum()), "pairs_with_different_full_state_law": int(pred.sum()),
           "pairs_where_the_difference_is_absorbed": int((pred & ~diff).sum()),
           "max_abs_row_difference": float(np.abs(em.P - ev.P).max())}
    for name, env in envs.items():
        _, _, info = x_optimal(env, "given")
        out[f"optimum_value[{name}]"] = float(info["dp_value"])
        out[f"uniform_random_value[{name}]"] = float(x_evaluate(env, np.full((env.S, env.A), 1.0 / env.A), "given")["return"])
    return out


X_GUMBEL: Optional[FiniteEnv] = None
X_GUMBEL_ENVS: Dict[str, FiniteEnv] = {}
GUMBEL_DECODE = pd.DataFrame()
GUMBEL_EXACTNESS: Dict[str, Any] = {"status": "not run"}
GUMBEL_EXACTNESS_OK: Optional[bool] = None
GUMBEL_REPORT: Dict[str, Any] = {"status": "not run"}
if EXPERIMENTS["xenv_gumbel"]:
    _t0 = time.perf_counter()
    _structure = {}
    for _v in GumbelSim.VARIANTS:
        X_GUMBEL_ENVS[_v], _structure[_v] = gumbel_true_env_report(_v, XCFG.gumbel_timeout_hazard)
    X_GUMBEL = X_GUMBEL_ENVS[XCFG.gumbel_variant]
    print(f"Gumbel simulator enumerated exactly ({len(X_GUMBEL_ENVS)} variants, {time.perf_counter() - _t0:.1f} s):",
          _structure[XCFG.gumbel_variant]["groups"])
    GUMBEL_DECODE = gumbel_decode_table(X_GUMBEL)
    GUMBEL_DECODE.to_csv(XENV_TABLE_DIR / "gumbel_state_decode.csv", index=False)
    _dec = GUMBEL_DECODE[["state", "full_index", *GumbelSim.FACTORS]].to_numpy(dtype=np.int64)
    for _v, _env in X_GUMBEL_ENVS.items():
        np.savez_compressed(DATA_DIR / f"xenv_gumbel_true_{_v}.npz", P=_env.P, entry_reward=_env.entry_reward,
                            d0=_env.d0, decode=_dec, mask=_env.mask,
                            meta_json=np.array(json.dumps(_env.meta, sort_keys=True, default=_json_default)))
    _checks = {}
    for _v, _env in X_GUMBEL_ENVS.items():
        _main = _v == XCFG.gumbel_variant
        _checks[_v] = gumbel_exactness_check(
            _env, XCFG.gumbel_mc_pairs if _main else max(1, XCFG.gumbel_mc_pairs // 3), XCFG.gumbel_mc_draws,
            np_rng("xenv", "gumbel-exactness", _v))
        print(f"  exactness [{_v}]:", {k: _checks[_v][k] for k in ("pairs", "pooled_p_value", "max_z_cells_expected_ge_5",
                                                                    "impossible_outcomes_observed", "d0_p_value", "passes")})
    GUMBEL_EXACTNESS = {"configured_variant": XCFG.gumbel_variant, "checks": _checks,
                        "passes": bool(all(c["passes"] for c in _checks.values()))}
    GUMBEL_EXACTNESS_OK = GUMBEL_EXACTNESS["passes"]
    save_json(GUMBEL_EXACTNESS, XENV_TABLE_DIR / "gumbel_exactness_check.json")
    GUMBEL_REPORT = {"structure": _structure,
                     "variant_comparison": gumbel_variant_comparison(X_GUMBEL_ENVS),
                     "provenance": gumbel_source_provenance(), "configured_variant": XCFG.gumbel_variant}
    save_json(GUMBEL_REPORT, XENV_TABLE_DIR / "gumbel_simulator_report.json")
    print("  variant comparison:", GUMBEL_REPORT["variant_comparison"], "| provenance:", GUMBEL_REPORT["provenance"]["status"])
    if GUMBEL_REPORT["provenance"]["status"].startswith(("MISMATCH", "MISSING")):
        print(f"WARNING: Gumbel source provenance {GUMBEL_REPORT['provenance']['status']}: the pinned sources could not "
              "be confirmed online (this port follows the pinned commits); closure row MT-1 fails until resolved.")
    assert GUMBEL_EXACTNESS_OK, "Gumbel enumeration disagrees with the literal step sampler (xenv/gumbel_exactness_check.json)."
    del _dec, _checks, _structure

### II.3c Garnet absorbing CMDPs with a controlled harm of unsupported actions

For each draw, a base absorbing MDP is generated with S live states: every (s, a) moves to `branching` distinct
random live successors with Dirichlet(1) weights, after absorbing with hazard h(s, a) ∼ U(`garnet_hazard`); an
absorption reaches the goal (entry reward +1) with probability q(s, a) ∼ U(0, 1), and otherwise death (0). d0 is
uniform over 10 random live states. The behaviour policy prefers the top-k_s actions of the base Q* (k_s uniform
in {1, 2, 3}): π_β = (1 − ε)/k_s on K(s) and ε/(A − k_s) elsewhere. The **harm** h ∈ [−1, 1] sets the true rows of
the non-preferred actions: P*(s, a) = (1 − |h|)·mean_{a'∈K(s)} P_base(s, a') + |h|·e_death (h > 0) or e_goal (h < 0).
With h = 0 the mean rule is exactly right wherever K(s) ⊆ M(s). Only the base MDP, K(s) and k_s are shared: they
depend only on the draw (stream `("xenv", "garnet", "base", draw)`), so the harm and ε cells of one draw share their
base MDP (common random numbers), and Section II.7 treats them as one draw block. The hospital and user data are
cell-specific (streams keyed by the full cell), and the harm changes the benchmark mask itself: absorbing outcomes
shorten the hospital episodes, so the mask size differs across h (reported by (h, ε) next to H3). The hazard makes
every policy proper; this is asserted.

In [ ]:
# ============================================================
# II.3c Garnet family
# ============================================================
EXECLOG.mark("II.3c Garnet family")


def garnet_true_env(draw_key: tuple, n_states: int, n_actions: int, branching: int,
                    hazard_range: Tuple[float, float], harm: float, eps: float) -> Tuple[FiniteEnv, np.ndarray]:
    """True Garnet CMDP of one draw with harm `harm`, and its behaviour policy with softening `eps`."""
    S, A, B = int(n_states), int(n_actions), int(branching)
    lo, hi = (float(x) for x in hazard_range)
    harm, eps = float(harm), float(eps)
    assert S >= 10 and A >= 4 and 1 <= B <= S, (S, A, B)
    assert 0.0 < lo <= hi < 1.0 and -1.0 <= harm <= 1.0 and 0.0 < eps < 1.0, (lo, hi, harm, eps)
    rng = np_rng("xenv", "garnet", *draw_key)
    goal, death = S, S + 1
    succ = np.argsort(rng.random((S * A, S)), axis=1)[:, :B]          # B distinct uniformly random live successors
    w = rng.dirichlet(np.ones(B), size=S * A)
    haz = rng.uniform(lo, hi, size=S * A)
    q_goal = rng.uniform(0.0, 1.0, size=S * A)
    starts = rng.choice(S, size=10, replace=False)
    k = rng.integers(1, 4, size=S)                                    # |K(s)| uniform in {1, 2, 3}
    rows = np.zeros((S * A, S + 2))
    rows[np.arange(S * A)[:, None], succ] = (1.0 - haz)[:, None] * w
    rows[:, goal] = haz * q_goal
    rows[:, death] = haz * (1.0 - q_goal)
    P_base = np.zeros((S + 2, A, S + 2))
    P_base[:S] = rows.reshape(S, A, S + 2)
    P_base[goal, :, goal] = 1.0
    P_base[death, :, death] = 1.0
    entry = np.zeros(S + 2)
    entry[goal] = 1.0
    d0 = np.zeros(S + 2)
    d0[starts] = 1.0 / len(starts)
    tag = "|".join(str(x) for x in draw_key)
    base = FiniteEnv(name=f"garnet-base|{tag}", kind="garnet", P=P_base, entry_reward=entry, d0=d0,
                     terminals=(goal, death), sink=death, goal=goal, mask=np.ones((S + 2, A), bool))
    _, V, _ = x_optimal(base, "given")
    Q = base.evaluator().q_values(V, "mean")[:S]
    order = np.argsort(-Q, axis=1, kind="stable")                     # ties -> lowest action index
    K = np.zeros((S + 2, A), dtype=bool)
    for s in range(S):
        K[s, order[s, :k[s]]] = True
    pi = np.full((S + 2, A), 1.0 / A)
    pi[:S] = np.where(K[:S], (1.0 - eps) / k[:, None], eps / (A - k)[:, None])
    P = P_base.copy()
    target = death if harm > 0 else goal
    for s in range(S):
        bad = ~K[s]
        P[s, bad] = (1.0 - abs(harm)) * P_base[s, K[s]].mean(axis=0)
        P[s, bad, target] += abs(harm)
    env = FiniteEnv(name=f"garnet|h{harm}|eps{eps}|{tag}", kind="garnet", P=P, entry_reward=entry, d0=d0,
                    terminals=(goal, death), sink=death, goal=goal, mask=np.ones((S + 2, A), bool),
                    meta={"draw_key": list(draw_key), "harm": harm, "eps": eps, "n_states": S, "n_actions": A,
                          "branching": B, "hazard_range": [lo, hi],
                          "preferred_set_sizes": {str(j): int((k == j).sum()) for j in (1, 2, 3)},
                          "dangling_target": death, "dangling_target_label": "death (Garnet has no timeout state)",
                          "censoring_hazard": None})
    assert np.allclose(env.P.sum(axis=2), 1.0, atol=1e-12)
    assert x_trap_states(env.P, env.live, env.mask) == 0, "every Garnet policy must be proper"
    if harm == 0.0:
        assert np.allclose(x_impute_inadmissible(P_base, K, env.live), env.P, atol=1e-15)
    return env, validate_policy(pi, f"Garnet behaviour {env.name}", shape=(S + 2, A))

### II.3d ICU-style benchmark construction, true targets, environment cells

`construct_benchmark(true_env, behaviour, n_hosp, τ, rng, name)`:

1. **Hospital data**: `n_hosp` episodes of the behaviour policy on the true system (`x_sample_dataset`, cutoff
   `XCFG.max_steps`), validated by `x_validate_dataset`.
2. **Decision states** V are the live states logged as `state`. By the chain property of episodes, a live next state
   outside V can only end a cut-off episode. These dangling transitions are redirected to the timeout state
   (Gumbel) or to death (Garnet, which has no timeout state). Both targets pay 0, the logged reward of a live
   transition, and both facts are asserted. Dangling transitions are counted.
3. **Mask and dead states.** M(s) = {a : N(s, a) ≥ τ}. With `XCFG.benchmark_empty_rule = "icu"` (default) a
   decision state with no action reaching τ is handled as ICU-Sepsis's `create_valid_dynamics` handles it: the state
   is *dead* and is removed, the informative transitions into it are dropped (not redirected), and the remaining
   rows renormalise; a pair whose every transition entered a removed state loses its row and becomes inadmissible.
   This repeats until no dead state remains (`dead_states_removed`, `dead_state_rounds`,
   `transitions_into_dead_dropped`); removed states leave V like censored states, and their initial mass is dropped
   before d0 is renormalised (step 7). The sensitivity rule `"argmax"` instead keeps such a state with its most
   frequent action (`empty_row_fallbacks`). The oracle construction keeps the argmax fallback, because the true
   state space cannot drop states. For the Gumbel benchmark, N counts the *informative* steps only. The timeout is
   the evaluation horizon (a constant hazard h fixed by the benchmark designer, like ICU-Sepsis's time limit), not
   clinical dynamics, so logged timeout steps are censoring events. A logged state whose every step was censored
   carries no dynamics information and is treated like an unobserved state (its inflow is redirected).
4. **State space** V ∪ terminals, with the index map `index[bench] = true state`.
5. **Rows**: maximum likelihood P̂ = C/N on admissible pairs. For Gumbel this is the maximum-likelihood estimate
   under the known horizon, P̂ = h·e_timeout + (1 − h)·C_inf/N_inf, which keeps the true horizon in every row.
   Every inadmissible pair, including logged pairs below τ, gets the uniform mean of the admissible P̂ rows
   (`x_impute_inadmissible`, the ICU-Sepsis mean rule).
6. **Trap censoring.** Under γ = 1 a benchmark is well posed only if every policy absorbs. Rows estimated from a
   few transitions can miss every absorbing outcome, and under the mean rule all actions of a single-admissible
   state copy that row. A closed set of live states that some policy never leaves (a "trap") then appears. In local
   tests of the plain count construction, a quarter of the Gumbel draws and an occasional Garnet draw had one; there
   `x_optimal` fails and even the clinician estimate can be improper. The known-horizon rows of step 5 rule traps
   out for Gumbel (every row absorbs with probability ≥ h; asserted). For Garnet the greatest such set (`x_trap_set`,
   all actions, mean rule) is computed and censored: its states leave V, their logged steps are dropped,
   transitions into them are redirected like dangling ones, and the model is re-estimated. One pass suffices,
   because redirecting steps to an absorbing state cannot create a new trap; the code asserts that no trap remains.
   Censored states (uninformative, dead and trap states), the steps they drop and their initial mass are reported.
   The empirical d0 is renormalised over the remaining states.
7. **d0** is the empirical initial-state distribution. 8. The **clinician policy** of benchmark users is π̂_β
   from the hospital counts, inadmissible actions included (as in ICU-Sepsis).

**What the construction does not reproduce.** The benchmark is estimated on the true Markov states, so
ICU-Sepsis's state-aggregation (k-means) error is absent; the measured errors therefore omit this error source, and
whether aggregation increases or decreases the total benchmark error is not studied. With the ε-optimal Gumbel
physician the masks are near-singleton and contain the true optimum (see II.3b), so in those cells the termination
rule is favoured by construction; the `withhold-vaso` cell and the Garnet h < 0 cells are the controls with
beneficial unsupported actions.

`lift` maps a benchmark policy to the true state space: rows of V are copied; true live states outside V (never
decided, or censored) play the most frequent hospital action; absorbing rows are uniform. `mask_true` is the
benchmark mask on V and all-True elsewhere. With the true hazard (Gumbel timeout, Garnet absorption), every policy
is proper on every true target and on every benchmark, under both semantics; the code asserts this for each
construction. ICU-Sepsis itself is checked (reported in `icu_adapter_check.json`, not altered).

`x_true_targets(true_env, mask_true)` returns `T_masked` (the true dynamics with the benchmark mask, so violation
counts and the terminate rule refer to the benchmark's support) and `T_meanimp` (the true rows on admissible pairs,
with the mean of the true admissible rows on inadmissible pairs).

**Cells** (`X_CELLS`): `icu|released` (draw 0 is the Part I dataset; later draws are fresh clinician datasets
from the released benchmark), `gumbel|{best,oracle}|n…|tau…|eps…[|phys-withhold-vaso]` and `garnet|h…|eps…`.
`build_cell_instance(spec, draw)` is a deterministic function of (cell, draw): every random stream is named
`("xenv", purpose, cell, draw)` with purpose ∈ {hospital, user, split}.

In [ ]:
# ============================================================
# II.3d Benchmark construction, true targets, environment cells
# ============================================================
EXECLOG.mark("II.3d Benchmark construction and cells")


def x_count_mask(N: np.ndarray, tau: int, live: np.ndarray, fallback: bool = True) -> Tuple[np.ndarray, int]:
    """M(s) = {a : N(s, a) >= tau} on live states with data; live states without data and absorbing rows are all-True.

    fallback=True: a decided row without any action reaching tau keeps its most frequent action (lowest index on ties).
    fallback=False: such rows stay empty (the caller removes them as dead states, as ICU-Sepsis does).
    Returns (mask, number of decided rows without an action reaching tau)."""
    N = np.asarray(N, dtype=np.float64)
    live = np.asarray(live, dtype=bool)
    assert float(tau) == int(tau) and int(tau) >= 1, tau
    mask = N >= int(tau)
    decided = live & (N.sum(axis=1) > 0)
    empty = decided & ~mask.any(axis=1)
    if fallback:
        mask[empty, N[empty].argmax(axis=1)] = True
    mask[~decided] = True
    return mask, int(empty.sum())


def _x_dangling_target(env: FiniteEnv) -> int:
    t = int(env.meta["dangling_target"])
    assert t in env.terminals and env.entry_reward[t] == 0.0, \
        "dangling transitions must be redirected to an absorbing state that pays 0 (the logged reward of a live step)"
    return t


def _x_hospital_counts(true_env: FiniteEnv, hosp: Dict[str, np.ndarray]) -> np.ndarray:
    return np.bincount(hosp["state"] * true_env.A + hosp["action"],
                       minlength=true_env.S * true_env.A).reshape(true_env.S, true_env.A).astype(np.float64)


def _x_estimate_on(true_env: FiniteEnv, hosp: Dict[str, np.ndarray], V: np.ndarray, mask_count: np.ndarray,
                   redirect: int, informative: np.ndarray, drop_into: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """Rows on decision states V (sorted true indices) + mean rule. Logged steps from states outside V are dropped;
    live next states outside V are redirected to the absorbing state `redirect`. With a known censoring hazard h
    (Gumbel timeout), rows are h * e_timeout + (1 - h) * (empirical law of the informative, non-censored steps).

    drop_into (ICU-Sepsis dead-state rule; true indices of removed states): informative transitions into these states
    are dropped instead of redirected, so the remaining rows renormalise, and a pair left without any transition
    becomes inadmissible. If a state of V is then left without an admissible pair, the result only reports `dead`
    (no rows are built) and the caller removes those states and re-estimates."""
    S, A = true_env.S, true_env.A
    s, a, ns = hosp["state"], hosp["action"], hosp["next_state"]
    hazard = true_env.meta["censoring_hazard"]
    index = np.concatenate([V, np.asarray(true_env.terminals, dtype=np.int64)])
    nV, Sb = len(V), len(index)
    to_bench = np.full(S, -1, dtype=np.int64)
    to_bench[index] = np.arange(Sb)
    keep = to_bench[s] >= 0
    use = keep & informative
    dropped = use & np.isin(ns, drop_into) if drop_into is not None else np.zeros(len(s), dtype=bool)
    use &= ~dropped
    s_b, a_u = to_bench[s[use]], a[use]
    ns_b = to_bench[ns[use]]
    out = ns_b < 0
    ns_b[out] = to_bench[redirect]
    C = np.bincount((s_b * A + a_u) * Sb + ns_b, minlength=Sb * A * Sb).reshape(Sb, A, Sb).astype(np.float64)
    N = C.sum(axis=2)
    live_b = np.zeros(Sb, dtype=bool)
    live_b[:nV] = True
    mask = np.ones((Sb, A), dtype=bool)
    mask[:nV] = mask_count[V]
    if drop_into is not None:
        mask[:nV] &= N[:nV] > 0          # a pair whose every informative step entered a removed state has no row
    dead = ~mask[:nV].any(axis=1)
    res = {"index": index, "to_bench": to_bench, "keep": keep, "out": int(out.sum()), "N": N, "mask": mask,
           "live": live_b, "nV": nV, "Sb": Sb, "dead": dead, "dropped_into_removed": int(dropped.sum())}
    if dead.any():
        return res
    adm = mask & live_b[:, None]
    assert N[adm].min() > 0, "an admissible benchmark pair has no informative logged transition"
    P = np.zeros((Sb, A, Sb))
    P[adm] = C[adm] / N[adm][:, None]
    if hazard is not None:
        c = int(to_bench[int(true_env.meta["timeout_state"])])
        P[adm] *= 1.0 - float(hazard)
        P[adm, c] += float(hazard)
    P = x_impute_inadmissible(P, mask, live_b)
    for t in range(nV, Sb):
        P[t, :, t] = 1.0
    res["P"] = P
    return res


def x_benchmark_from_data(true_env: FiniteEnv, hosp: Dict[str, np.ndarray], tau: int, name: str,
                          empty_rule: Optional[str] = None) -> Dict[str, Any]:
    """ICU-style benchmark estimated from validated hospital data on the true state space (see the markdown).

    empty_rule (default XCFG.benchmark_empty_rule): "icu" removes decision states without an action reaching tau as
    ICU-Sepsis's create_valid_dynamics does (drop the inflow, renormalise, iterate); "argmax" keeps them with their
    most frequent action."""
    S, A = true_env.S, true_env.A
    rule = XCFG.benchmark_empty_rule if empty_rule is None else str(empty_rule)
    assert rule in ("icu", "argmax"), rule
    s, a, ns = hosp["state"], hosp["action"], hosp["next_state"]
    assert true_env.live[s].all()
    hazard = true_env.meta["censoring_hazard"]
    informative = (np.ones(len(s), dtype=bool) if hazard is None
                   else ns != int(true_env.meta["timeout_state"]))
    N_all = _x_hospital_counts(true_env, hosp)
    N_inf = _x_hospital_counts(true_env, {"state": s[informative], "action": a[informative]})
    mask_count, _ = x_count_mask(N_inf, tau, true_env.live, fallback=rule == "argmax")
    redirect = _x_dangling_target(true_env)
    logged = np.unique(s)
    never = true_env.live[ns] & ~np.isin(ns, logged)
    assert hosp["cutoff"][never].all(), "a live next state that is never decided must end a cut-off episode"
    V0 = logged[N_inf[logged].sum(axis=1) > 0]
    uninformative = np.setdiff1d(logged, V0)          # every logged step from these states was censored (timeout)
    # Dead states (rule "icu", ICU-Sepsis create_valid_dynamics): a decision state without an admissible pair is
    # removed, the informative transitions into it are dropped and the remaining rows renormalise; repeat until none.
    dead_states = np.zeros(0, dtype=np.int64)
    V, dead_rounds = V0, 0
    est = _x_estimate_on(true_env, hosp, V, mask_count, redirect, informative,
                         drop_into=None if rule == "argmax" else dead_states)
    while est["dead"].any():
        assert rule == "icu", "the argmax fallback leaves no decision state without an admissible pair"
        dead_states = np.sort(np.concatenate([dead_states, V[est["dead"]]]))
        V = V[~est["dead"]]
        dead_rounds += 1
        assert len(V) > 0, f"{name}: no decision state survives the dead-state removal"
        est = _x_estimate_on(true_env, hosp, V, mask_count, redirect, informative, drop_into=dead_states)
    # Trap censoring after the states from which the estimated model admits a non-absorbing policy (one pass
    # suffices: redirecting steps to an absorbing state cannot create a trap, nor remove an admissible pair).
    trap = x_trap_set(est["P"], est["live"], np.ones((est["Sb"], A), dtype=bool))[:len(V)]
    V_untrapped = V
    V = V[~trap]
    assert len(V) > 0, f"{name}: no decision state survives censoring"
    if trap.any():
        est = _x_estimate_on(true_env, hosp, V, mask_count, redirect, informative,
                             drop_into=None if rule == "argmax" else dead_states)
        assert not est["dead"].any(), f"{name}: trap censoring removed an admissible pair"
    if hazard is not None:
        assert not trap.any(), "rows with a known censoring hazard always absorb"
    censored = np.concatenate([uninformative, dead_states, V_untrapped[trap]])
    index, to_bench, keep, nV, Sb, mask = est["index"], est["to_bench"], est["keep"], est["nV"], est["Sb"], est["mask"]
    P, N = est["P"], est["N"]
    assert x_trap_states(P, est["live"], np.ones((Sb, A), dtype=bool)) == 0, f"{name}: trap states remain"
    first = hosp["timestep"] == 0
    d0 = np.bincount(to_bench[s[first & keep]], minlength=Sb).astype(np.float64)
    assert d0.sum() > 0 and np.all(d0[nV:] == 0), f"{name}: no initial state survives censoring"
    censored_d0 = float(1.0 - d0.sum() / first.sum())
    d0 /= d0.sum()
    beh = np.full((Sb, A), 1.0 / A)
    beh[:nV] = N_all[V] / N_all[V].sum(axis=1, keepdims=True)
    bench = FiniteEnv(name=name, kind="benchmark", P=P, entry_reward=true_env.entry_reward[index], d0=d0,
                      terminals=tuple(range(nV, Sb)), sink=int(to_bench[true_env.sink]), goal=int(to_bench[true_env.goal]),
                      mask=mask, features=None if true_env.features is None else true_env.features[index],
                      meta={"construction": "best", "true_env": true_env.name, "tau": int(tau),
                            "hospital_transitions": int(len(s)), "censoring_hazard": hazard,
                            "censored_states": int(len(censored)), "empty_rule": rule})
    mask_true = np.ones((S, A), dtype=bool)
    mask_true[V] = mask[:nV]
    fallback_action = int(np.bincount(a, minlength=A).argmax())
    outside = true_env.live.copy()
    outside[V] = False

    def lift(pi_bench: np.ndarray) -> np.ndarray:
        pb = validate_policy(pi_bench, f"{name} policy", shape=(Sb, A))
        out = np.full((S, A), 1.0 / A)
        out[outside] = 0.0
        out[outside, fallback_action] = 1.0
        out[V] = pb[:nV]
        return out

    report = {"construction": "best", "tau": int(tau), "empty_rule": rule, "censoring_hazard": hazard,
              "hospital_transitions": int(len(s)),
              "hospital_episodes": int(first.sum()), "censored_steps_timeout": int((~informative).sum()),
              "decision_states_logged": int(len(logged)), "decision_states": int(nV),
              "uninformative_states": int(len(uninformative)),
              "decision_states_without_supported_action": int((~(N_inf[V0] >= int(tau)).any(axis=1)).sum()),
              "dead_states_removed": int(len(dead_states)), "dead_state_rounds": int(dead_rounds),
              "transitions_into_dead_dropped": int(est["dropped_into_removed"]),
              "trap_states_censored": int(trap.sum()),
              "transitions_from_censored_states": int((~keep).sum()),
              "transitions_into_censored_states": int((keep & informative & np.isin(ns, censored)).sum()),
              "censored_d0_mass": censored_d0, "true_live_states": int(true_env.live.sum()),
              "coverage_true_live": float(nV / true_env.live.sum()),
              "true_d0_mass_on_decision_states": float(true_env.d0[V].sum()),
              "observed_pairs": int((N[:nV] > 0).sum()), "admissible_pairs": int(mask[:nV].sum()),
              "mean_admissible_actions": float(mask[:nV].sum(axis=1).mean()),
              "single_admissible_fraction": float((mask[:nV].sum(axis=1) == 1).mean()),
              "empty_row_fallbacks": int((~(N_inf[V] >= int(tau)).any(axis=1)).sum()),
              "logged_pairs_below_tau_discarded": int(((N[:nV] > 0) & ~mask[:nV]).sum()),
              "dangling_transitions": int((never & keep).sum()), "dangling_redirect": true_env.meta["dangling_target_label"],
              "inadmissible_logged_fraction": float((~mask[to_bench[s[keep]], a[keep]]).mean()),
              "fallback_action": fallback_action, "trap_states_given": 0,
              "trap_states_terminate": x_trap_states(P, est["live"], mask)}
    assert report["trap_states_terminate"] == 0
    return {"bench": bench, "bench_behavior": validate_policy(beh, f"{name} behaviour", shape=(Sb, A)),
            "index": index, "mask_true": mask_true, "lift": lift, "fallback_action": fallback_action, "report": report,
            "decision_states_true": V}


def construct_benchmark(true_env: FiniteEnv, behavior: np.ndarray, n_hosp: int, tau: int,
                        rng: np.random.Generator, name: str) -> Dict[str, Any]:
    """Hospital data from the true system -> ICU-style estimated benchmark (mean rule for unsupported actions)."""
    hosp = x_sample_dataset(true_env, behavior, int(n_hosp), XCFG.max_steps, rng)
    hosp_report = x_validate_dataset(hosp, true_env)
    out = x_benchmark_from_data(true_env, hosp, tau, name)
    out["report"]["hospital_validation"] = hosp_report
    del hosp
    return out


def oracle_benchmark(true_env: FiniteEnv, behavior: np.ndarray, n_hosp: int, tau: int,
                     rng: np.random.Generator, name: str) -> Dict[str, Any]:
    """Same hospital mask, but true state space, true admissible rows (mean of true admissible rows elsewhere), true d0.

    The true state space cannot drop states, so a decided state without an action reaching tau keeps its most
    frequent action (x_count_mask fallback) whatever XCFG.benchmark_empty_rule says."""
    hosp = x_sample_dataset(true_env, behavior, int(n_hosp), XCFG.max_steps, rng)
    hosp_report = x_validate_dataset(hosp, true_env)
    hazard = true_env.meta["censoring_hazard"]
    inf = (np.ones(len(hosp["state"]), dtype=bool) if hazard is None
           else hosp["next_state"] != int(true_env.meta["timeout_state"]))
    N_true = _x_hospital_counts(true_env, {"state": hosp["state"][inf], "action": hosp["action"][inf]})
    mask, n_fallback = x_count_mask(N_true, tau, true_env.live)
    bench = true_env.variant(name, P=x_impute_inadmissible(true_env.P, mask, true_env.live), mask=mask,
                             kind="benchmark", construction="oracle", tau=int(tau))
    decided = true_env.live & (N_true.sum(axis=1) > 0)
    fallback_action = int(np.bincount(hosp["action"], minlength=true_env.A).argmax())
    report = {"construction": "oracle", "tau": int(tau), "empty_rule": "argmax (true state space keeps every state)",
              "censoring_hazard": hazard,
              "hospital_transitions": int(len(hosp["state"])), "censored_steps_timeout": int((~inf).sum()),
              "hospital_episodes": int((hosp["timestep"] == 0).sum()), "decision_states": int(decided.sum()),
              "true_live_states": int(true_env.live.sum()),
              "coverage_true_live": float(decided.sum() / true_env.live.sum()),
              "true_d0_mass_on_decision_states": float(true_env.d0[decided].sum()),
              "observed_pairs": int((N_true[decided] > 0).sum()), "admissible_pairs": int(mask[decided].sum()),
              "mean_admissible_actions": float(mask[decided].sum(axis=1).mean()),
              "single_admissible_fraction": float((mask[decided].sum(axis=1) == 1).mean()),
              "empty_row_fallbacks": n_fallback,
              "logged_pairs_below_tau_discarded": int(((N_true > 0) & ~mask)[decided].sum()),
              "dangling_transitions": 0, "dangling_redirect": "none (true state space)",
              "inadmissible_logged_fraction": float((~mask[hosp["state"], hosp["action"]]).mean()),
              "fallback_action": fallback_action,
              "trap_states_given": x_trap_states(bench.P, bench.live, np.ones((bench.S, bench.A), bool)),
              "trap_states_terminate": x_trap_states(bench.P, bench.live, bench.mask),
              "hospital_validation": hosp_report}
    assert report["trap_states_given"] == 0, "convex completions of proper true rows must stay proper"
    del hosp
    return {"bench": bench, "bench_behavior": validate_policy(behavior, f"{name} behaviour", shape=(true_env.S, true_env.A)),
            "index": np.arange(true_env.S), "mask_true": bench.mask.copy(), "lift": None,
            "fallback_action": fallback_action, "report": report, "decision_states_true": np.flatnonzero(decided)}


def x_true_targets(true_env: FiniteEnv, mask_true: np.ndarray) -> Tuple[FiniteEnv, FiniteEnv]:
    """(T_masked: true dynamics + benchmark mask, T_meanimp: true admissible rows + mean of them elsewhere)."""
    T_masked = true_env.variant(f"{true_env.name}|true-masked", mask=mask_true, role="true dynamics, benchmark mask")
    T_meanimp = true_env.variant(f"{true_env.name}|true-rows-mean",
                                 P=x_impute_inadmissible(true_env.P, T_masked.mask, true_env.live), mask=mask_true,
                                 role="true admissible rows, mean rule elsewhere")
    return T_masked, T_meanimp


def x_build_cells() -> List[dict]:
    cells: List[dict] = []
    if EXPERIMENTS["xenv_icu"]:
        cells.append({"cell": "icu|released", "family": "icu", "draws": int(XCFG.icu_draws),
                      "user_episodes": int(XCFG.icu_user_episodes)})
    if EXPERIMENTS["xenv_gumbel"]:
        for entry in XCFG.gumbel_cells:
            construction, n_hosp, tau, eps, *rest = entry
            assert construction in ("best", "oracle"), construction
            assert len(rest) <= 1, f"Gumbel cell entries are (construction, n_hosp, tau, eps[, physician]): {entry}"
            physician = str(rest[0]) if rest else "optimal"
            assert physician in GUMBEL_PHYSICIAN_KINDS, physician
            tag = "" if physician == "optimal" else f"|phys-{physician}"
            cells.append({"cell": f"gumbel|{construction}|n{n_hosp}|tau{tau}|eps{eps}{tag}", "family": "gumbel",
                          "draws": int(XCFG.gumbel_draws), "construction": construction, "n_hosp": int(n_hosp),
                          "tau": int(tau), "eps": float(eps), "physician": physician, "variant": XCFG.gumbel_variant,
                          "timeout_hazard": float(XCFG.gumbel_timeout_hazard),
                          "user_episodes": int(XCFG.gumbel_user_episodes)})
    if EXPERIMENTS["xenv_garnet"]:
        for harm in XCFG.garnet_harm:
            for eps in XCFG.garnet_behavior_eps:
                cells.append({"cell": f"garnet|h{harm}|eps{eps}", "family": "garnet", "draws": int(XCFG.garnet_draws),
                              "harm": float(harm), "eps": float(eps), "n_states": int(XCFG.garnet_states),
                              "n_actions": int(XCFG.garnet_actions), "branching": int(XCFG.garnet_branching),
                              "hazard_range": tuple(float(x) for x in XCFG.garnet_hazard),
                              "n_hosp": int(XCFG.garnet_hosp_episodes), "tau": int(XCFG.garnet_tau),
                              "user_episodes": int(XCFG.garnet_user_episodes)})
    keys = [c["cell"] for c in cells]
    assert len(keys) == len(set(keys)), f"duplicate cell keys: {keys}"
    return cells


_X_GUMBEL_PHYS: Dict[tuple, np.ndarray] = {}
_X_GUMBEL_OPT_ACTION: Dict[tuple, np.ndarray] = {}
_X_GARNET_CACHE: "OrderedDict[tuple, Tuple[FiniteEnv, np.ndarray]]" = OrderedDict()


def _x_gumbel_env(variant: str, hazard: float) -> FiniteEnv:
    env = X_GUMBEL_ENVS.get(variant)
    if env is None or float(env.meta["timeout_hazard"]) != float(hazard):
        raise KeyError(f"Gumbel env ({variant}, h={hazard}) was not built in II.3b; set EXPERIMENTS['xenv_gumbel'] "
                       "and XCFG.gumbel_timeout_hazard before this section.")
    return env


def _x_gumbel_physician(variant: str, hazard: float, eps: float, kind: str = "optimal") -> np.ndarray:
    key = (variant, float(hazard), float(eps), str(kind))
    if key not in _X_GUMBEL_PHYS:
        _X_GUMBEL_PHYS[key] = gumbel_physician(_x_gumbel_env(variant, hazard), eps, kind)
    return _X_GUMBEL_PHYS[key]


def _x_gumbel_opt_action(variant: str, hazard: float) -> np.ndarray:
    """argmax of the exact optimum Q*_true of the Gumbel system (cached per variant and hazard)."""
    key = (variant, float(hazard))
    if key not in _X_GUMBEL_OPT_ACTION:
        pi_opt, _, _ = x_optimal(_x_gumbel_env(variant, hazard), "given")
        _X_GUMBEL_OPT_ACTION[key] = pi_opt.argmax(axis=1)
    return _X_GUMBEL_OPT_ACTION[key]


def _x_garnet_env(spec: dict, draw: int, keep: int = 4) -> Tuple[FiniteEnv, np.ndarray]:
    key = (spec["cell"], int(draw))
    if key in _X_GARNET_CACHE:
        _X_GARNET_CACHE.move_to_end(key)
        return _X_GARNET_CACHE[key]
    val = garnet_true_env(("base", int(draw)), spec["n_states"], spec["n_actions"], spec["branching"],
                          spec["hazard_range"], spec["harm"], spec["eps"])
    _X_GARNET_CACHE[key] = val
    while len(_X_GARNET_CACHE) > keep:
        _X_GARNET_CACHE.popitem(last=False)
    return val


def build_cell_instance(spec: dict, draw: int) -> CellInstance:
    """Deterministic (cell, draw) instance: learner environment, behaviour, user data split and evaluation targets."""
    t0 = time.perf_counter()
    cell, family, draw = spec["cell"], spec["family"], int(draw)
    assert draw >= 0, draw
    rng_h, rng_u = np_rng("xenv", "hospital", cell, draw), np_rng("xenv", "user", cell, draw)
    construction, hosp_validation = None, None
    if family == "icu":
        assert X_ICU is not None
        if draw == 0:
            data, source = OFFLINE, "Part I OFFLINE dataset (Section 5)"
        else:
            data = x_sample_dataset(X_ICU, X_ICU_BEHAVIOR, spec["user_episodes"], XCFG.max_steps, rng_u)
            source = "x_sample_dataset(X_ICU, clinician)"
        learn_env, behavior = X_ICU, X_ICU_BEHAVIOR
        targets = [("bench_mean", X_ICU, "given", None), ("bench_terminate", X_ICU, "terminate", None)]
        deploy = "bench_terminate"
    elif family in ("gumbel", "garnet"):
        if family == "gumbel":
            true_env = _x_gumbel_env(spec["variant"], spec["timeout_hazard"])
            true_behavior = _x_gumbel_physician(spec["variant"], spec["timeout_hazard"], spec["eps"],
                                                spec.get("physician", "optimal"))
            kind = spec["construction"]
        else:
            true_env, true_behavior = _x_garnet_env(spec, draw)
            kind = "best"
        bname = f"{cell}|d{draw}|bench"
        if kind == "best":
            B = construct_benchmark(true_env, true_behavior, spec["n_hosp"], spec["tau"], rng_h, bname)
            learn_env, behavior, lift = B["bench"], B["bench_behavior"], B["lift"]
            data = x_sample_dataset(learn_env, behavior, spec["user_episodes"], XCFG.max_steps, rng_u)
            T_masked, T_meanimp = x_true_targets(true_env, B["mask_true"])
            targets = [("bench_mean", learn_env, "given", None), ("bench_terminate", learn_env, "terminate", None),
                       ("true", T_masked, "given", lift), ("true_rows_mean", T_meanimp, "given", lift),
                       ("true_rows_terminate", T_masked, "terminate", lift)]
        elif kind == "oracle":
            B = oracle_benchmark(true_env, true_behavior, spec["n_hosp"], spec["tau"], rng_h, bname)
            learn_env, behavior = B["bench"], true_behavior
            data = x_sample_dataset(learn_env, behavior, spec["user_episodes"], XCFG.max_steps, rng_u)
            T_masked, _ = x_true_targets(true_env, B["mask_true"])
            targets = [("bench_mean", learn_env, "given", None), ("bench_terminate", T_masked, "terminate", None),
                       ("true", T_masked, "given", None)]
        else:
            raise ValueError(f"unknown construction {kind!r}")
        if family == "gumbel":   # is the true optimal action supported? (decision states of the construction)
            opt_act, Vt = _x_gumbel_opt_action(spec["variant"], spec["timeout_hazard"]), B["decision_states_true"]
            B["report"]["true_optimum_in_mask_fraction"] = (float(B["mask_true"][Vt, opt_act[Vt]].mean()) if len(Vt)
                                                            else float("nan"))
        deploy = "true"
        construction = {k: v for k, v in B["report"].items() if k != "hospital_validation"}
        hosp_validation = B["report"]["hospital_validation"]
        source = f"x_sample_dataset({kind} benchmark, {'clinician estimate' if kind == 'best' else 'true behaviour'})"
        del B
    else:
        raise ValueError(f"unknown family {family!r}")
    labels = [t[0] for t in targets]
    assert len(labels) == len(set(labels)) and deploy in labels, labels
    for lab, env, sem, lift_fn in targets:
        assert sem in X_SEMANTICS and (lift_fn is not None or env.S == learn_env.S), (lab, env.S, learn_env.S)
    user_validation = x_validate_dataset(data, learn_env)
    train, dev = x_split_episodes(data, XCFG.dev_fraction, np_rng("xenv", "split", cell, draw))
    meta = {"spec": {k: (list(v) if isinstance(v, tuple) else v) for k, v in spec.items()}, "user_data_source": source,
            "user_validation": user_validation, "hospital_validation": hosp_validation, "construction": construction,
            "train_transitions": int(len(train["state"])), "dev_transitions": int(len(dev["state"])),
            "build_s": time.perf_counter() - t0}
    return CellInstance(cell=cell, family=family, draw=draw, learn_env=learn_env, behavior=behavior, data=data,
                        train=train, dev=dev, targets=targets, deploy_target=deploy, meta=meta)


def _x_value(env: FiniteEnv, pi: np.ndarray, sem: str) -> Tuple[float, str]:
    ev = x_evaluate(env, pi, sem)
    return (float(ev["return"]), "ok") if ev["status"] == "ok" else (float("nan"), ev["status"])


def _x_opt_value(env: FiniteEnv, sem: str, constrained: bool = False) -> Tuple[float, str]:
    try:
        _, _, info = x_optimal(env, sem, constrained=constrained)
        return float(info["dp_value"]), "ok"
    except ImproperPolicyError:
        return float("nan"), "improper"


def x_environment_suite_table() -> pd.DataFrame:
    """One row per cell from draw 0: sizes, mask statistics, behaviour and optimum values per target, the value of
    unsupported actions J*_true(all actions) - J*_true(admissible only), and the construction report."""
    rows = []
    for spec in X_CELLS:
        inst = build_cell_instance(spec, 0)
        L = inst.learn_env
        row = {"cell": spec["cell"], "family": spec["family"], "draws": spec["draws"], "deploy_target": inst.deploy_target,
               "S": L.S, "A": L.A, "live_states": int(L.live.sum()), "admissible_pairs": int(L.mask[L.live].sum()),
               "mean_admissible_actions": float(L.mask[L.live].sum(axis=1).mean()),
               "user_transitions": inst.meta["user_validation"]["transitions"],
               "user_episodes": inst.meta["user_validation"]["episodes"],
               "user_inadmissible_logged_fraction": inst.meta["user_validation"]["inadmissible_logged_fraction"],
               "build_s": inst.meta["build_s"]}
        statuses = []
        for lab, env, sem, lift_fn in inst.targets:
            beh = inst.behavior if lift_fn is None else lift_fn(inst.behavior)
            row[f"behaviour_value[{lab}]"], st_b = _x_value(env, beh, sem)
            row[f"optimum_value[{lab}]"], st_o = _x_opt_value(env, sem)
            statuses += [f"{what}[{lab}]={st}" for what, st in (("behaviour", st_b), ("optimum", st_o)) if st != "ok"]
        true_t = {lab: env for lab, env, _, _ in inst.targets}.get("true")
        if true_t is not None:
            j_all, st_all = _x_opt_value(true_t, "given")
            j_adm, st_adm = _x_opt_value(true_t, "given", constrained=True)
            row["value_of_unsupported_actions"] = j_all - j_adm
            statuses += [f"optimum_admissible[true]={st_adm}"] * (st_adm != "ok") + [f"optimum[true]={st_all}"] * (st_all != "ok")
        else:
            row["value_of_unsupported_actions"] = float("nan")
        row["non_ok_statuses"] = "; ".join(statuses)
        for k, v in (inst.meta["construction"] or {}).items():
            row[f"construction_{k}"] = v
        rows.append(row)
        del inst
        gc.collect()
    return pd.DataFrame(rows)


X_CELLS: List[dict] = x_build_cells()
print("Part II cells:", [(c["cell"], c["draws"]) for c in X_CELLS])
X_ENV_SUITE = pd.DataFrame()
if X_CELLS:
    _t0 = time.perf_counter()
    X_ENV_SUITE = x_environment_suite_table()
    X_ENV_SUITE.to_csv(XENV_TABLE_DIR / "xenv_environment_suite.csv", index=False)
    _cols = [c for c in ("cell", "S", "live_states", "mean_admissible_actions", "behaviour_value[true]",
                         "optimum_value[true]", "value_of_unsupported_actions", "construction_empty_row_fallbacks",
                         "construction_dead_states_removed", "construction_single_admissible_fraction",
                         "construction_true_optimum_in_mask_fraction",
                         "construction_uninformative_states", "construction_trap_states_censored",
                         "construction_censored_d0_mass", "non_ok_statuses") if c in X_ENV_SUITE.columns]
    display(X_ENV_SUITE[_cols])
    print(f"Environment suite table built in {time.perf_counter() - _t0:.1f} s.")

# Objects whose source enters the Part II code hash.
XENV_ENV_CODE = [x_icu_behavior, x_icu_env, GumbelSim, gumbel_full_index, gumbel_decode_full, gumbel_action_bits,
                 gumbel_num_abnormal, gumbel_on_treatment, gumbel_reward, gumbel_is_absorbing, _gx_rep, _gx_case,
                 _gx_u, _gx_flag, _gx_antibiotics_on, _gx_antibiotics_off, _gx_vent_on, _gx_vent_off, _gx_vaso_on,
                 _gx_vaso_off, _gx_fluctuate, gumbel_transition_dist, gumbel_initial_distribution,
                 gumbel_state_space, x_trap_set, x_trap_states, gumbel_true_env_report, gumbel_true_env,
                 gumbel_physician, garnet_true_env, x_count_mask, _x_dangling_target, _x_hospital_counts,
                 _x_estimate_on, x_benchmark_from_data, construct_benchmark, oracle_benchmark, x_true_targets,
                 x_build_cells, _x_gumbel_env, _x_gumbel_physician, _x_gumbel_opt_action, _x_garnet_env,
                 build_cell_instance]
print("Part II environments ready.")

## II.4 Method suite: exact tabular offline-RL learners on the empirical model

Every Part II learner is a deterministic function of its training data and a `LearnerView`. No learner reads a
transition tensor, an initial distribution or a `FiniteEnv`. Data enter only through `XEmpirical`:

* counts N(s,a) and maximum-likelihood rows P̂;
* mean logged rewards r̂;
* the empirical behaviour policy π̂_β, which uses the global logged-action frequency on states without data;
* the empirical start distribution d̂₀.

All methods use γ = 1 with V = 0 on absorbing states. The value-iteration methods (IQL, BCQ, VI-LCB, MBS-QI, SPIBB)
return expected entry rewards in [v_min, v_max]; CQL returns regularised scores.

**Conventions shared by all learners.**
* *Iteration.* Fixed points are computed by synchronous iteration from V = 0. Iteration stops when
  max|ΔQ| < `XCFG.tol`, capped at `XCFG.max_iterations`. Non-convergence is reported in `info["converged"]`,
  never raised and never hidden.
* *Unobserved pairs* (N(s,a) = 0) have zero rows in P̂. Each method states its convention:
  * IQL keeps Part I's never-updated value 0;
  * CQL pushes the unobserved actions of states with data toward −∞ (the conservative gradient is positive there);
    rows of states without data stay 0;
  * BCQ, MBS-QI and SPIBB never back up through these pairs;
  * VI-LCB sets them to v_min;
  * states without any data are valued 0 by BCQ, and v_min by MBS-QI and VI-LCB;
  * the model-based constrained methods complete them with a jump to the sink paying `view.terminate_reward`
    (pessimistic completion, `XEmpirical.completed_dense`).
* *Exact evaluation on the empirical model* (SPIBB, Lagrangian, LP cross-check) splits the policy-induced chain
  into transient states and closed classes without leakage. Rewards are paid only on entering absorbing states,
  and a closed class never absorbs. Its value is therefore exactly 0, which is the limit of value iteration from
  V = 0, and its occupancy is infinite once it is entered. No singular system is ever solved.
* *Variants.*
  * `native` is the method's own output.
  * `masked` restricts extraction to the admissible mask M of the learner's environment. Zero mass outside M on
    live states is asserted for every masked record, and for every method that is admissible by construction.
  * The driver (Section II.6) adds the `projected` variant `x_project(native, M)` of every native record.
* *Hyper-parameters.* Every tunable method has a pre-declared default `XCFG.*_default`, which is asserted to lie
  in its grid. Ties in greedy extraction go to the lowest index, as in Part I, except where a row of the table
  says otherwise.

| Method (family) | Exact tabular definition | Reference |
|---|---|---|
| BC (stochastic), BC (behaviour) | π̂_β; argmax of counts; masked: argmax of counts over M, falling back to the global frequency over M | behaviour cloning (empirical MLE) |
| Uniform-admissible (behaviour) | uniform over M(s) | trivial admissible baseline |
| CQL (value) | full-batch Adam on (1/n)Σ N(Q−ȳ)² + α(1/n)[Σ_s N_s lse Q(s,·) − Σ N Q], with ȳ = r̂ + P̂ max Q_target (hard target sync); approximate fixed point after 400 target syncs (`XCFG.cql_steps` / `cql_target_every`); non-convergence (max \|ΔQ\| on observed pairs over the last sync interval > `XCFG.cql_converged_tol`) is flagged | Kumar et al. 2020 |
| IQL, IQL (AW stochastic) (value) | in-sample fixed point: V(s) = N-weighted τ-expectile of Q(s,·) (exact sorting solution), Q = r̂ + P̂V on observed pairs; extraction by AW weights exp(clip(β(Q−V), −20, log w_max))·N, as in Part I | Kostrikov et al. 2022 |
| BCQ (pessimistic) | VI restricted to G(s) = {a : π̂_β(a\|s)/max π̂_β ≥ τ}; masked: argmax over G∩M, falling back to M | Fujimoto et al. 2019 (discrete BCQ) |
| VI-LCB (pessimistic) | Q = clip(r̂ − c·(v_max − v_min)/√max(N,1) + P̂V, v_min, v_max) on observed pairs, v_min elsewhere; ties go to the most frequently logged action | Rashidinejad et al. 2021; Jin et al. 2021 |
| MBS-QI (pessimistic) | backups only through ζ = 1[N ≥ b]; V_ζ = v_min where no action is supported; states without supported actions use BC | Liu et al. 2020 |
| SPIBB (pessimistic) | Π_b-SPIBB policy iteration on P̂: pairs with N < N∧ keep π̂_β, the remaining mass goes to argmax Q over the other pairs | Laroche et al. 2019 |
| Lagrangian-MB (constrained) | cost c = 1[a ∉ M]; bisection on λ for greedy VI on r − λc in the completed model, until the expected violations from d̂₀ are ≤ κ | Altman 1999; Le, Voloshin & Yue 2019 (Batch Policy Learning under Constraints, ICML); Tessler et al. 2019 (RCPO) |
| CMDP-LP (model) (constrained) | exact occupancy LP (`x_cmdp_lp`) on the completed model with E[violations] ≤ κ; zero-occupancy states use masked BC | Altman 1999; COptiDICE (Lee et al. 2022) |
| SfBC (exact), SfBC (exact, argmax) (guided-prior) | π ∝ π̂_β·exp(α·A_IQL) at the default expectile; masked: renormalised on M | Chen et al. 2023 |
| IDQL (exact) (guided-prior) | π ∝ π̂_β·\|τ − 1(Q < V)\| at the default expectile | Hansen-Estruch et al. 2023 |
| FISOR-tab (exact) (constrained) | π ∝ π̂_β·exp(α·A_feas)·1[a ∈ M]; A_feas comes from the in-sample IQL fixed point whose expectile uses logged **admissible** actions only, so admissibility acts as feasibility in both the critic and the extraction | Zheng et al. 2024 |
| Diffusion (exact, ·) (diffusion) | exact Bayes-optimal reverse kernels of the uniform-corruption chain started at π̂_β; Part I inference (uniform-admissible start, every-step mask, guidance by G = per-state standardised Q_IQL, deterministic argmax), plus final-step-only guidance and an unmasked control | Austin et al. 2021; Part I Section 12 |

FISOR-tab uses the SfBC grid for α. With the unrestricted IQL critic it would equal `SfBC (exact) [masked]` exactly.
The feasibility-restricted critic is what makes it a separate method, and it is the tabular analogue of FISOR's
feasibility-dependent value learning.

In [ ]:
# ============================================================
# II.4 Method suite
# ============================================================
EXECLOG.mark("II.4 Method suite")
import scipy.sparse.csgraph as csgraph
import scipy.sparse.linalg as spla

_X_TIE_RTOL = 1e-12          # relative tolerance for "tied" greedy scores
_X_LEAK_TOL = 1e-12          # row mass below 1 - tol leaves the live set (absorption, completion sink, no data)
# Module constants read by hashed functions: code_hash covers source text only, so the driver (II.6) and theory
# (II.5) identities carry their values.
X_NUMERIC_CONSTANTS = {"tie_rtol": _X_TIE_RTOL, "leak_tol": _X_LEAK_TOL, "strategy": dict(_X_STRATEGY)}
_X_SWITCHES = ("xenv_theory", "xenv_frontiers", "xenv_icu", "xenv_gumbel", "xenv_garnet", "xenv_tabular_crosscheck")


def _x_info(d: Dict[str, Any]) -> Dict[str, Any]:
    """Small JSON-able info dict: numpy scalars become Python scalars; arrays and objects are rejected."""
    out: Dict[str, Any] = {}
    for k, v in d.items():
        if isinstance(v, (bool, np.bool_)):
            out[k] = bool(v)
        elif isinstance(v, (int, np.integer)):
            out[k] = int(v)
        elif isinstance(v, (float, np.floating)):
            out[k] = float(v)
        elif v is None or isinstance(v, str):
            out[k] = v
        elif isinstance(v, dict):
            out[k] = _x_info(v)
        elif (isinstance(v, (list, tuple)) and len(v) <= 64
              and all(isinstance(x, (bool, int, float, str, np.number, np.bool_)) for x in v)):
            out[k] = [x.item() if isinstance(x, (np.number, np.bool_)) else x for x in v]
        else:
            raise TypeError(f"info field {k!r} is not a small JSON-able value ({type(v).__name__})")
    return out


def _x_iterate(step: Callable[[np.ndarray], Tuple[np.ndarray, np.ndarray]], live: np.ndarray,
               tol: Optional[float] = None, max_iter: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    """Synchronous fixed-point iteration from V = 0 (γ = 1, V = 0 on absorbing states).

    `step(V)` returns fresh arrays (Q [S, A], V_new [S]). Stops when max|ΔQ| < tol; hitting the cap is reported."""
    tol = XCFG.tol if tol is None else float(tol)
    max_iter = XCFG.max_iterations if max_iter is None else int(max_iter)
    V = np.zeros(len(live))
    Q_prev, delta, it = None, float("inf"), 0
    for it in range(1, max_iter + 1):
        Q, V_new = step(V)
        if not np.isfinite(Q).all():
            raise FloatingPointError("non-finite Q during fixed-point iteration")
        V_new[~live] = 0.0
        if Q_prev is not None:
            delta = float(np.max(np.abs(Q - Q_prev)))
        Q_prev, V = Q, V_new
        if delta < tol:
            break
    return Q_prev, V, {"iterations": it, "final_delta": delta, "converged": bool(delta < tol)}


def _x_argmax_tiebreak(scores: np.ndarray, allowed: np.ndarray, secondary: np.ndarray) -> np.ndarray:
    """Argmax over `allowed`; scores within a relative 1e-12 of the best are tied and resolved by `secondary`
    (then the lowest index)."""
    s = np.where(allowed, np.asarray(scores, dtype=np.float64), -np.inf)
    best = s.max(axis=1, keepdims=True)
    tied = allowed & (s >= best - _X_TIE_RTOL * np.maximum(1.0, np.abs(best)))
    return masked_argmax(secondary, tied)


def _x_policy_rows(pi: np.ndarray) -> sp.csr_matrix:
    """Sparse [S, S*A] matrix whose row s holds pi(.|s) at the columns of the (s, a) pairs."""
    S, A = pi.shape
    return sp.csr_matrix((np.asarray(pi, dtype=np.float64).ravel(), (np.repeat(np.arange(S), A), np.arange(S * A))),
                         shape=(S, S * A))


def _x_chain(emp: XEmpirical, live: np.ndarray, pi: np.ndarray) -> Tuple[sp.csr_matrix, np.ndarray]:
    """Live-to-live block of the policy-induced empirical chain. Missing row mass is absorption, the completion
    sink (unobserved pairs) or a state without data."""
    L = np.flatnonzero(live)
    P_pi = (_x_policy_rows(pi) @ emp.P).tocsr()
    return P_pi[L][:, L].tocsr(), L


def _x_markov_solve(P_LL: sp.spmatrix, r: np.ndarray, d0: Optional[np.ndarray] = None,
                    b: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """Exact γ = 1 evaluation of a substochastic chain on live states (value, return, occupancy, violations).

    Closed classes without leakage never absorb. Their value is 0, because rewards are paid only on entry to
    absorbing states and value iteration from V = 0 converges to 0 there. Their expected visits, and hence the
    violations of a policy that violates inside them, are infinite once entered. Only the transient block,
    which is always non-singular, is factorised."""
    P = sp.csr_matrix(P_LL, dtype=np.float64)
    P.eliminate_zeros()
    n = P.shape[0]
    r = np.asarray(r, dtype=np.float64)
    if P.nnz and P.data.min() < 0:
        raise ValueError("negative transition probabilities")
    row = np.asarray(P.sum(axis=1)).ravel()
    if (row > 1.0 + 1e-9).any():
        raise ValueError(f"row sums exceed one (max {row.max():.12f})")
    leak = row < 1.0 - _X_LEAK_TOL
    n_comp, lab = csgraph.connected_components(P, directed=True, connection="strong")
    coo = P.tocoo()
    open_comp = np.zeros(n_comp, dtype=bool)
    cross = lab[coo.row] != lab[coo.col]
    open_comp[lab[coo.row[cross]]] = True
    open_comp[lab[leak]] = True
    rec = ~open_comp[lab]
    T = np.flatnonzero(~rec)
    v = np.zeros(n)
    out: Dict[str, Any] = {"recurrent_states": int(rec.sum())}
    lu = None
    if len(T):
        M = (sp.identity(len(T), format="csc") - P[T][:, T].tocsc()).tocsc()
        lu = spla.splu(M)
        vT = lu.solve(r[T])
        res = float(np.abs(M @ vT - r[T]).max())
        if not np.isfinite(vT).all() or res > 1e-8 * (1.0 + np.abs(r[T]).max() + np.abs(vT).max()):
            raise EvaluationInvariantError(f"empirical-chain value solve residual {res:.2e}")
        v[T] = vT
    out["v"] = v
    if d0 is not None:
        d0 = np.asarray(d0, dtype=np.float64)
        mu = np.zeros(n)
        if len(T):
            muT = lu.solve(d0[T], trans="T")
            res = float(np.abs(M.T @ muT - d0[T]).max())
            if not np.isfinite(muT).all() or res > 1e-8 * (1.0 + muT.max()) or muT.min() < -1e-9:
                raise EvaluationInvariantError(f"empirical-chain occupancy solve residual {res:.2e}")
            mu[T] = np.clip(muT, 0.0, None)
        inflow = d0 * rec
        if len(T) and rec.any():
            inflow = inflow + np.asarray(P[T].T @ mu[T]).ravel() * rec
        entered = np.zeros(n_comp, dtype=bool)
        entered[lab[inflow > 1e-15]] = True
        trapped_states = rec & entered[lab]
        out["occupancy"], out["trapped"], out["return"] = mu, float(inflow.sum()), float(d0 @ v)
        if b is not None:
            b = np.asarray(b, dtype=np.float64)
            viol = float(mu @ b)
            if (b[trapped_states] > 0).any():
                viol = float("inf")
            out["violations"] = viol
    return out


# ---------- behaviour cloning ----------
def x_bc(emp: XEmpirical, view: LearnerView) -> Dict[str, Any]:
    """Empirical behaviour cloning (Part I semantics with an explicit mask and live set)."""
    counts = emp.N_sa
    native = one_hot_policy(emp.pi_b.argmax(axis=1), view.A)
    adm_seen = (counts * view.mask).sum(axis=1) > 0
    masked = one_hot_policy(masked_argmax(np.where(adm_seen[:, None], counts, emp.global_pi[None, :]), view.mask),
                            view.A)
    info = {"masked_fallback_states": int((~adm_seen & view.live).sum()),
            "unseen_live_states": int((~emp.seen & view.live).sum())}
    return {"stochastic": emp.pi_b.copy(), "native": native, "masked": masked, "deterministic": native,
            "masked_greedy": masked, "info": info}


# ---------- CQL: exact full-batch objective, Adam, hard target sync ----------
def _x_cql_fit(emp: XEmpirical, view: LearnerView, alphas: Sequence[float], steps: int, lr: float,
               target_every: int) -> List[Dict[str, Any]]:
    """Exact empirical CQL for several α at once (independent problems stacked on a leading axis).

    Gradient of (1/n)Σ_sa N_sa (Q_sa − ȳ_sa)² + (α/n)[Σ_s N_s lse(Q_s) − Σ_sa N_sa Q_sa] on states with data;
    Adam as in torch (β = (0.9, 0.999), ε = 1e-8, bias-corrected). Rows of states without data stay 0.
    Convergence: max |ΔQ| on observed pairs over the last target-sync interval ≤ XCFG.cql_converged_tol (unobserved
    actions of rows with data drift toward −∞ under the conservative term, so they are excluded)."""
    S, A, H = view.S, view.A, len(alphas)
    rows = np.flatnonzero(emp.N_s > 0)
    if not view.live[rows].all():
        raise ValueError("CQL: data rows on absorbing states")
    N = emp.N_sa[rows]
    obs_r = emp.observed[rows]
    Ns = emp.N_s[rows][:, None]
    n = float(emp.n)
    c_td = 2.0 * N / n
    c_reg = np.asarray(alphas, dtype=np.float64)[:, None, None] / n
    q = np.zeros((H, len(rows), A))
    m1, m2, y = np.zeros_like(q), np.zeros_like(q), np.zeros_like(q)
    g, z, tmp = np.empty_like(q), np.empty_like(q), np.empty_like(q)
    b1, b2, eps = 0.9, 0.999, 1e-8
    full = np.zeros((S, A))

    def targets(qh: np.ndarray) -> np.ndarray:
        full[rows] = qh
        V = full.max(axis=1)
        V[~view.live] = 0.0
        return (emp.r_hat + emp.backup(V))[rows]

    q_sync = q.copy()
    last_dq, last_dq_obs = np.zeros(H), np.zeros(H)
    syncs = 0
    for step in range(1, steps + 1):
        if (step - 1) % target_every == 0:
            last_dq = np.abs(q - q_sync).max(axis=(1, 2))
            last_dq_obs = np.where(obs_r[None], np.abs(q - q_sync), 0.0).max(axis=(1, 2))
            q_sync = q.copy()
            for h in range(H):
                y[h] = targets(q[h])
            syncs += 1
        np.subtract(q, q.max(axis=2, keepdims=True), out=z)
        np.exp(z, out=z)
        z /= z.sum(axis=2, keepdims=True)
        z *= Ns
        z -= N
        z *= c_reg
        np.subtract(q, y, out=g)
        g *= c_td
        g += z
        m1 *= b1
        np.multiply(g, 1.0 - b1, out=tmp)
        m1 += tmp
        m2 *= b2
        np.multiply(g, g, out=tmp)
        tmp *= 1.0 - b2
        m2 += tmp
        np.sqrt(m2, out=tmp)
        tmp /= math.sqrt(1.0 - b2 ** step)
        tmp += eps
        np.divide(m1, tmp, out=tmp)
        tmp *= lr / (1.0 - b1 ** step)
        q -= tmp
    if not np.isfinite(q).all():
        raise FloatingPointError("CQL produced non-finite Q values")
    out = []
    obs = emp.observed
    for h, alpha in enumerate(alphas):
        Q = np.zeros((S, A))
        Q[rows] = q[h]
        V = Q.max(axis=1)
        V[~view.live] = 0.0
        bell = emp.r_hat + emp.backup(V)
        lse = np.log(np.exp(q[h] - q[h].max(axis=1, keepdims=True)).sum(axis=1)) + q[h].max(axis=1)
        info = {"alpha": float(alpha), "steps": int(steps), "lr": float(lr), "target_every": int(target_every),
                "target_syncs": syncs, "last_sync_max_abs_dq": float(last_dq[h]),
                "last_sync_max_abs_dq_observed": float(last_dq_obs[h]),
                "converged": bool(syncs >= 2 and last_dq_obs[h] <= XCFG.cql_converged_tol),
                "td_term": float((N * (q[h] - y[h]) ** 2).sum() / n),
                "conservative_term": float((emp.N_s[rows] * lse - (N * q[h]).sum(axis=1)).sum() / n),
                "td_residual_max_observed": float(np.abs(Q - bell)[obs].max()) if obs.any() else 0.0,
                "q_min": float(Q.min()), "q_max": float(Q.max())}
        native = x_greedy(Q)
        masked = x_greedy(Q, view.mask)
        out.append({"q": Q, "native": native, "masked": masked, "deterministic": native, "masked_greedy": masked,
                    "info": info})
    return out


def x_tab_cql(emp: XEmpirical, view: LearnerView, alpha: Optional[float] = None, steps: Optional[int] = None,
              lr: Optional[float] = None, target_every: Optional[int] = None) -> Dict[str, Any]:
    alpha = XCFG.cql_alpha_default if alpha is None else float(alpha)
    return _x_cql_fit(emp, view, (alpha,), XCFG.cql_steps if steps is None else int(steps),
                      XCFG.cql_lr if lr is None else float(lr),
                      XCFG.cql_target_every if target_every is None else int(target_every))[0]


# ---------- IQL: exact in-sample expectile fixed point ----------
def x_expectile(values: np.ndarray, weights: np.ndarray, tau: float) -> np.ndarray:
    """Exact weighted τ-expectile of every row: the unique v with τ Σ_{x>v} w(x − v) = (1 − τ) Σ_{x<v} w(v − x).

    The condition is piecewise linear and decreasing in v. After sorting each row, the root lies on the bracket
    between the last sorted value where the condition is ≥ 0 and the next one, and is solved there in closed form.
    Rows need non-negative weights with a positive sum."""
    x = np.asarray(values, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    tau = float(tau)
    if x.ndim != 2 or x.shape != w.shape:
        raise ValueError(f"x_expectile: shapes {x.shape} and {w.shape}")
    if not 0.0 < tau < 1.0:
        raise ValueError(f"expectile {tau} outside (0, 1)")
    if x.shape[0] == 0:
        return np.zeros(0)
    if not (np.isfinite(x).all() and np.isfinite(w).all() and (w >= 0).all() and (w.sum(axis=1) > 0).all()):
        raise ValueError("x_expectile: finite values and non-negative weights with a positive row sum are required")
    K = x.shape[1]
    order = np.argsort(x, axis=1, kind="stable")
    xs = np.take_along_axis(x, order, axis=1)
    ws = np.take_along_axis(w, order, axis=1)
    W_lo = np.cumsum(ws, axis=1)
    S_lo = np.cumsum(ws * xs, axis=1)
    W_hi = W_lo[:, -1:] - W_lo
    S_hi = S_lo[:, -1:] - S_lo
    F = tau * (S_hi - xs * W_hi) - (1.0 - tau) * (xs * W_lo - S_lo)
    neg = F < 0.0
    k = np.clip(np.where(neg.any(axis=1), neg.argmax(axis=1) - 1, K - 1), 0, K - 1)
    r = np.arange(x.shape[0])
    v = ((tau * S_hi[r, k] + (1.0 - tau) * S_lo[r, k])
         / (tau * W_hi[r, k] + (1.0 - tau) * W_lo[r, k]))
    upper = np.where(k + 1 < K, xs[r, np.minimum(k + 1, K - 1)], np.inf)
    return np.clip(v, xs[r, k], upper)


def _x_aw_extraction(q: np.ndarray, v: np.ndarray, counts: np.ndarray, beta: float, max_weight: float,
                     mask: np.ndarray, live: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int]:
    """Part I `iql_extraction` with an explicit mask and live set (AW policy, weighted counts, masked greedy)."""
    S, A = q.shape
    adv = q - v[:, None]
    weights = np.exp(np.clip(beta * adv, -20.0, math.log(max_weight)))
    weighted = counts * weights
    global_pi = counts.sum(axis=0) / max(counts.sum(), 1.0)
    pi = np.tile(global_pi, (S, 1))
    seen = weighted.sum(axis=1) > 0
    pi[seen] = weighted[seen] / weighted[seen].sum(axis=1, keepdims=True)
    adm_w = np.where(mask, weighted, -np.inf)
    no_logged_adm = ~(np.where(mask, weighted, 0.0) > 0).any(axis=1)
    masked_actions = adm_w.argmax(axis=1)
    masked_actions[no_logged_adm] = masked_argmax(np.broadcast_to(global_pi, q.shape), mask)[no_logged_adm]
    return pi, weighted, one_hot_policy(masked_actions, A), int((no_logged_adm & live).sum())


def x_tab_iql(emp: XEmpirical, view: LearnerView, expectile: Optional[float] = None, beta: Optional[float] = None,
              max_weight: Optional[float] = None, support: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """Exact tabular IQL fixed point: V(s) = expectile of Q(s,·) under weights N(s,·)·support(s,·), Q = r̂ + P̂V
    on observed pairs (0 elsewhere, as Part I's never-updated entries). `support=None` uses every logged action;
    FISOR-tab passes the admissible mask. States without supported logged actions keep V = 0."""
    tau = XCFG.iql_expectile_default if expectile is None else float(expectile)
    beta = XCFG.iql_beta if beta is None else float(beta)
    max_weight = XCFG.iql_max_weight if max_weight is None else float(max_weight)
    W = emp.N_sa if support is None else emp.N_sa * np.asarray(support, dtype=bool)
    rows = np.flatnonzero(view.live & (W.sum(axis=1) > 0))
    Wr = W[rows]
    obs = emp.observed

    def step(V):
        Q = np.where(obs, emp.r_hat + emp.backup(V), 0.0)
        V_new = np.zeros(view.S)
        V_new[rows] = x_expectile(Q[rows], Wr, tau)
        return Q, V_new

    Q, V, info = _x_iterate(step, view.live)
    resid = float(np.abs(np.where(obs, emp.r_hat + emp.backup(V), 0.0) - Q).max())
    pi, weighted, masked, n_fb = _x_aw_extraction(Q, V, emp.N_sa, beta, max_weight, view.mask, view.live)
    native = one_hot_policy(pi.argmax(axis=1), view.A)
    info.update(expectile=tau, beta=beta, max_weight=max_weight, fixed_point_residual=resid,
                masked_fallback_states=n_fb, support="logged actions" if support is None else "logged admissible actions",
                states_without_supported_action=int((view.live & emp.seen & (W.sum(axis=1) == 0)).sum()))
    return {"q": Q, "v": V, "aw_policy": pi, "aw_scores": weighted, "native": native, "masked": masked,
            "deterministic": native, "masked_greedy": masked, "info": info}


# ---------- support-constrained and pessimistic value iteration ----------
def x_behavior_threshold_set(pi_b: np.ndarray, tau: float, live: np.ndarray) -> np.ndarray:
    """Discrete-BCQ set G(s) = {a : π̂_β(a|s) / max_a' π̂_β(a'|s) ≥ τ} (Part I `behavior_threshold_set`)."""
    allowed = pi_b / np.maximum(pi_b.max(axis=1, keepdims=True), 1e-30) >= tau
    allowed[~live] = True
    return allowed


def x_tab_bcq(emp: XEmpirical, view: LearnerView, tau: Optional[float] = None) -> Dict[str, Any]:
    """Exact tabular discrete BCQ: Q = r̂ + P̂V, V(s) = max over G(s) (Part I extraction semantics)."""
    tau = XCFG.bcq_tau_default if tau is None else float(tau)
    G = x_behavior_threshold_set(emp.pi_b, tau, view.live)
    if not G.any(axis=1).all():
        raise AssertionError("BCQ: empty behaviour-threshold set")

    def step(V):
        Q = emp.r_hat + emp.backup(V)
        return Q, np.where(G, Q, -np.inf).max(axis=1)

    Q, V, info = _x_iterate(step, view.live)
    native = one_hot_policy(masked_argmax(Q, G), view.A)
    both = G & view.mask
    both_ok = both.any(axis=1)
    masked = one_hot_policy(masked_argmax(Q, np.where(both_ok[:, None], both, view.mask)), view.A)
    info.update(tau=tau, masked_fallback_states=int((~both_ok & view.live).sum()),
                mean_allowed_actions=float(G[view.live].sum(axis=1).mean()) if view.live.any() else 0.0)
    return {"q": Q, "v": V, "allowed": G, "native": native, "masked": masked, "deterministic": native,
            "masked_greedy": masked, "info": info}


def x_vi_lcb(emp: XEmpirical, view: LearnerView, c: Optional[float] = None) -> Dict[str, Any]:
    """VI-LCB (Hoeffding form, multiplier c): pessimistic, clipped backups on observed pairs, v_min elsewhere."""
    c = XCFG.lcb_c_default if c is None else float(c)
    lo, hi, live = view.v_min, view.v_max, view.live
    obs = emp.observed & live[:, None]
    base = emp.r_hat - c * (hi - lo) / np.sqrt(np.maximum(emp.N_sa, 1.0))

    def step(V):
        Q = np.where(obs, np.clip(base + emp.backup(V), lo, hi), lo)
        Q[~live] = 0.0
        return Q, Q.max(axis=1)

    Q, V, info = _x_iterate(step, live)
    allow = np.ones_like(view.mask)
    native = one_hot_policy(_x_argmax_tiebreak(Q, allow, emp.pi_b), view.A)
    masked = one_hot_policy(_x_argmax_tiebreak(Q, view.mask, emp.pi_b), view.A)
    info.update(c=c, states_at_floor=int((V[live] <= lo + 1e-12).sum()))
    return {"q": Q, "v": V, "native": native, "masked": masked, "info": info}


def x_mbs_qi(emp: XEmpirical, view: LearnerView, b: Optional[int] = None) -> Dict[str, Any]:
    """MBS-QI: backups through ζ = 1[N ≥ b] only; V_ζ = v_min where no action is supported."""
    b = XCFG.mbsqi_b_default if b is None else int(b)
    live = view.live
    zeta = (emp.N_sa >= b) & live[:, None]
    has = zeta.any(axis=1)

    def step(V):
        Q = emp.r_hat + emp.backup(V)
        return Q, np.where(has, np.where(zeta, Q, -np.inf).max(axis=1), view.v_min)

    Q, V, info = _x_iterate(step, live)
    bc = x_bc(emp, view)
    act = np.where(has, masked_argmax(Q, np.where(has[:, None], zeta, True)), bc["native"].argmax(axis=1))
    zm = zeta & view.mask
    hm = zm.any(axis=1)
    act_m = np.where(hm, masked_argmax(Q, np.where(hm[:, None], zm, True)), bc["masked"].argmax(axis=1))
    info.update(b=b, unsupported_live_states=int((~has & live).sum()),
                masked_fallback_states=int((~hm & live).sum()))
    return {"q": np.where(zeta, Q, view.v_min), "v": V, "zeta": zeta, "native": one_hot_policy(act, view.A),
            "masked": one_hot_policy(act_m, view.A), "info": info}


def x_spibb(emp: XEmpirical, view: LearnerView, n_wedge: Optional[int] = None, max_iter: int = 100) -> Dict[str, Any]:
    """Π_b-SPIBB (Laroche et al. 2019) with exact policy evaluation on P̂ (closed classes valued 0)."""
    n_wedge = XCFG.spibb_n_default if n_wedge is None else int(n_wedge)
    S, live, pb = view.S, view.live, emp.pi_b
    boot = emp.N_sa < n_wedge
    boot[~live] = True
    free_mask = ~boot
    has_free = free_mask.any(axis=1)
    kept = np.where(boot, pb, 0.0)
    free = 1.0 - kept.sum(axis=1)
    if np.abs(free[~has_free]).max(initial=0.0) > 1e-12:
        raise AssertionError("SPIBB: all-bootstrapped rows must keep the behaviour policy exactly")
    free[~has_free] = 0.0
    allowed = np.where(has_free[:, None], free_mask, True)
    idx = np.arange(S)
    r_hat = emp.r_hat
    pi, act, converged, it, recurrent = pb.copy(), None, False, 0, 0
    Q = V = None
    for it in range(1, max_iter + 1):
        P_LL, L = _x_chain(emp, live, pi)
        sol = _x_markov_solve(P_LL, (pi * r_hat).sum(axis=1)[L])
        recurrent = max(recurrent, sol["recurrent_states"])
        V = np.zeros(S)
        V[L] = sol["v"]
        Q = r_hat + emp.backup(V)
        new = masked_argmax(Q, allowed)
        if act is not None:
            q_new = Q[idx, new]
            keep = Q[idx, act] >= q_new - _X_TIE_RTOL * np.maximum(1.0, np.abs(q_new))
            new = np.where(keep, act, new)
            if np.array_equal(new[has_free], act[has_free]):
                converged = True
                break
        act = new
        pi = kept.copy()
        pi[has_free, act[has_free]] += free[has_free]
    lb = boot & live[:, None]
    if not np.array_equal(pi[lb], pb[lb]):
        raise AssertionError("SPIBB: a bootstrapped pair does not keep its behaviour probability")
    off = pi.copy()
    off[lb] = 0.0
    if has_free.any() and not np.allclose(off[has_free].sum(axis=1), free[has_free], atol=1e-12):
        raise AssertionError("SPIBB: the free mass is not placed on non-bootstrapped pairs")
    info = {"n_wedge": n_wedge, "iterations": it, "converged": converged,
            "states_with_free_mass": int((has_free & live).sum()),
            "bootstrapped_live_pairs": int(lb.sum()), "max_recurrent_states": recurrent,
            "mean_free_mass": float(free[has_free & live].mean()) if (has_free & live).any() else 0.0}
    return {"pi": pi, "q": Q, "v": V, "bootstrapped": boot, "info": info}


# ---------- model-based constrained methods on the completed empirical model ----------
def _x_model_reward_cost(emp: XEmpirical, view: LearnerView) -> Tuple[np.ndarray, np.ndarray]:
    """Expected rewards of `emp.completed_dense(view.sink)` (unobserved live pairs pay the termination reward) and
    the violation cost c = 1[a ∉ M] on live states."""
    live = view.live
    miss = ~emp.observed & live[:, None]
    r = np.where(miss, view.terminate_reward, emp.r_hat)
    r[~live] = 0.0
    cost = (~view.mask & live[:, None]).astype(np.float64)
    return r, cost


def _x_model_eval(emp: XEmpirical, view: LearnerView, pi: np.ndarray, r_model: np.ndarray,
                  cost: np.ndarray) -> Dict[str, Any]:
    """Exact evaluation from d̂₀ on the completed model. Its sparse form is P̂ plus a jump to the sink for
    unobserved pairs, which leaves the live set."""
    P_LL, L = _x_chain(emp, view.live, pi)
    sol = _x_markov_solve(P_LL, (pi * r_model).sum(axis=1)[L], emp.d0_hat[L], (pi * cost).sum(axis=1)[L])
    return {"return": sol["return"], "violations": sol["violations"], "trapped": sol["trapped"],
            "recurrent_states": sol["recurrent_states"]}


def _x_lagrangian_greedy(emp: XEmpirical, view: LearnerView, lam: float, r_model: np.ndarray, cost: np.ndarray,
                         evals: Dict[bytes, Dict[str, Any]]) -> Dict[str, Any]:
    """Greedy VI policy of r − λc on the completed model (admissible actions win exact ties). Value iteration
    always starts from V = 0: with γ = 1 a warm start could keep a stale value on a zero-reward closed cycle.
    Model evaluations are cached per deterministic policy (`evals`), because the greedy policy is piecewise
    constant in λ."""
    base = r_model - lam * cost
    live = view.live

    def step(V):
        Q = base + emp.backup(V)      # unobserved rows of P̂ are zero and V(sink) = 0: the completion's backup
        Q[~live] = 0.0
        return Q, Q.max(axis=1)

    Q, _, vi = _x_iterate(step, live)
    act = _x_argmax_tiebreak(Q, np.ones_like(view.mask), view.mask.astype(np.float64))
    key = act.astype(np.int64).tobytes()
    pi = one_hot_policy(act, view.A)
    if key not in evals:
        evals[key] = _x_model_eval(emp, view, pi, r_model, cost)
    ev = evals[key]
    return {"pi": pi, "lam": float(lam), "violations": ev["violations"], "model_return": ev["return"],
            "trapped": ev["trapped"], "vi": vi}


def x_lagrangian_mb(emp: XEmpirical, view: LearnerView, kappa: Optional[float] = None,
                    cache: Optional[Dict[str, Dict[Any, Dict[str, Any]]]] = None, rel_tol: float = 1e-6,
                    max_bisection: int = 60) -> Dict[str, Any]:
    """Lagrangian model-based policy: the smallest λ ∈ [0, λ_max] (bisection) whose greedy policy of r − λc meets
    E_model[violations] ≤ κ from d̂₀. λ_max = (v_max − v_min) + 1 makes every admissible action strictly preferred.
    `cache` ({"greedy": λ → result, "eval": policy → model evaluation}) may be shared across κ on the same data."""
    kappa = XCFG.lagrange_kappa_default if kappa is None else float(kappa)
    if kappa < 0:
        raise ValueError("kappa must be non-negative")
    r_model, cost = _x_model_reward_cost(emp, view)
    cache = {} if cache is None else cache
    greedy, evals = cache.setdefault("greedy", {}), cache.setdefault("eval", {})

    def at(lam: float) -> Dict[str, Any]:
        if lam not in greedy:
            greedy[lam] = _x_lagrangian_greedy(emp, view, lam, r_model, cost, evals)
        return greedy[lam]

    def feasible(res: Dict[str, Any]) -> bool:
        return res["violations"] <= kappa + 1e-12

    lam_max = (view.v_max - view.v_min) + 1.0
    lo, hi, steps = 0.0, lam_max, 0
    lo_res = None
    if feasible(at(0.0)):
        chosen = at(0.0)
    else:
        lo_res = at(0.0)
        if not feasible(at(lam_max)):
            raise AssertionError(f"Lagrangian: lambda_max={lam_max} still violates ({at(lam_max)['violations']})")
        while hi - lo > rel_tol * lam_max and steps < max_bisection:
            mid = 0.5 * (lo + hi)
            steps += 1
            if feasible(at(mid)):
                hi = mid
            else:
                lo, lo_res = mid, at(mid)
        chosen = at(hi)
    used = [g["vi"] for g in greedy.values()]
    info = {"kappa": kappa, "lambda": chosen["lam"], "lambda_max": lam_max, "bisection_steps": steps,
            "model_violations": chosen["violations"], "model_return": chosen["model_return"],
            "trapped_probability": chosen["trapped"],
            "infeasible_side_violations": None if lo_res is None else lo_res["violations"],
            "converged": all(u["converged"] for u in used),
            "vi_converged_all": all(u["converged"] for u in used), "vi_iterations_max": max(u["iterations"] for u in used),
            "greedy_solves": len(greedy), "distinct_policies_evaluated": len(evals)}
    return {"pi": chosen["pi"], "info": info}


def x_cmdp_lp_model(emp: XEmpirical, view: LearnerView, kappa: Optional[float] = None,
                    P_model: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """Exact constrained LP on the completed empirical model (`x_cmdp_lp`, start distribution d̂₀).

    An infeasible LP is a legitimate empirical outcome: some closed class of admissible observed pairs may be
    unavoidable. It is reported (status) and counted, and the policy falls back to masked BC."""
    kappa = XCFG.mblp_kappa_default if kappa is None else float(kappa)
    P = emp.completed_dense(view.sink) if P_model is None else P_model
    r_model, cost = _x_model_reward_cost(emp, view)
    fallback = x_bc(emp, view)["masked"]
    try:
        pi, rep = x_cmdp_lp(P, r_model, emp.d0_hat, view.live, cost, kappa, fallback=fallback)
        status = "ok"
    except RuntimeError as exc:
        pi, rep, status = fallback.copy(), {"message": str(exc)[:160]}, "lp_failed_masked_bc_fallback"
    ev = _x_model_eval(emp, view, pi, r_model, cost)
    ok = status == "ok"
    info = {"kappa": kappa, "status": status, "lp_objective": rep.get("objective"), "lp_violations": rep.get("violations"),
            "model_return": ev["return"], "model_violations": ev["violations"],
            "lp_eval_gap": abs(ev["return"] - rep["objective"]) if ok else None,
            "zero_occupancy_live_states": int(view.live.sum()) - rep["reachable_live_states"] if ok else None,
            "randomised_states": rep.get("randomised_states"), "solve_s": rep.get("solve_s"),
            "message": rep.get("message")}
    return {"pi": pi, "info": info}


# ---------- behaviour prior x critic (exact, no sampling) ----------
def _x_tilted(pb: np.ndarray, log_w: np.ndarray, allowed: Optional[np.ndarray], live: np.ndarray) -> Tuple[np.ndarray, int]:
    """π ∝ π̂_β·exp(log_w) on `allowed` (log space; exact zeros where π̂_β = 0 or not allowed). Live rows without
    mass become uniform over `allowed` (counted); terminal rows keep the unrestricted tilt."""
    allow = np.ones(pb.shape, dtype=bool) if allowed is None else np.asarray(allowed, dtype=bool)
    ok = (pb > 0) & allow
    empty = ~ok.any(axis=1)
    if allowed is None and empty.any():
        raise AssertionError("behaviour prior has an empty row")
    logits = np.where(ok, np.log(np.where(ok, pb, 1.0)) + log_w, -np.inf)
    logits[empty] = np.where(allow[empty], 0.0, -np.inf)
    logits = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    p /= p.sum(axis=1, keepdims=True)
    return p, int((empty & live).sum())


def x_sfbc(emp: XEmpirical, view: LearnerView, alpha: Optional[float] = None,
           iql: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Exact SfBC: π ∝ π̂_β·exp(α·A) with A = Q − V of IQL at the default expectile (the limit of SfBC's
    candidate resampling with infinitely many behaviour candidates)."""
    alpha = XCFG.sfbc_alpha_default if alpha is None else float(alpha)
    iql = x_tab_iql(emp, view) if iql is None else iql
    tilt = alpha * (iql["q"] - iql["v"][:, None])
    native, _ = _x_tilted(emp.pi_b, tilt, None, view.live)
    masked, n_fb = _x_tilted(emp.pi_b, tilt, view.mask, view.live)
    return {"native": native, "masked": masked, "argmax": one_hot_policy(native.argmax(axis=1), view.A),
            "info": {"alpha": alpha, "expectile": iql["info"]["expectile"], "masked_fallback_states": n_fb,
                     "converged": bool(iql["info"]["converged"])}}


def x_idql(emp: XEmpirical, view: LearnerView, iql: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Exact IDQL implicit policy: π ∝ π̂_β·|τ − 1(Q < V)| at the IQL expectile τ."""
    iql = x_tab_iql(emp, view) if iql is None else iql
    tau = iql["info"]["expectile"]
    w = np.abs(tau - (iql["q"] < iql["v"][:, None]).astype(np.float64))
    native, _ = _x_tilted(emp.pi_b, np.log(w), None, view.live)
    masked, n_fb = _x_tilted(emp.pi_b, np.log(w), view.mask, view.live)
    return {"native": native, "masked": masked, "info": {"expectile": tau, "masked_fallback_states": n_fb,
                                                         "converged": bool(iql["info"]["converged"])}}


def x_fisor_tab(emp: XEmpirical, view: LearnerView, alpha: Optional[float] = None,
                critic: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """FISOR-style feasibility tilt: π ∝ π̂_β·exp(α·A_feas)·1[a ∈ M], with A_feas from the admissible-support
    IQL critic."""
    alpha = XCFG.sfbc_alpha_default if alpha is None else float(alpha)
    critic = x_tab_iql(emp, view, support=view.mask) if critic is None else critic
    if critic["info"]["support"] != "logged admissible actions":
        raise ValueError("FISOR-tab needs the admissible-support critic")
    pi, n_fb = _x_tilted(emp.pi_b, alpha * (critic["q"] - critic["v"][:, None]), view.mask, view.live)
    return {"pi": pi, "info": {"alpha": alpha, "expectile": critic["info"]["expectile"], "uniform_fallback_states": n_fb,
                               "states_without_admissible_logged_action":
                                   critic["info"]["states_without_supported_action"]}}


# ---------- exact categorical diffusion with the Bayes-optimal denoiser ----------
_X_DIFFUSION_CACHE: Dict[tuple, Any] = {}


def x_diffusion_for(A: int) -> "CategoricalDiffusion":
    """Cached uniform-corruption schedule of Part I (Section 12) for an action set of size A."""
    key = (int(A), int(CFG.diffusion_T), float(CFG.diffusion_beta_start), float(CFG.diffusion_beta_end))
    if key not in _X_DIFFUSION_CACHE:
        _X_DIFFUSION_CACHE[key] = CategoricalDiffusion(*key)
    return _X_DIFFUSION_CACHE[key]


def x_bayes_reverse_kernels(pi_b: np.ndarray, diffusion: "CategoricalDiffusion") -> Dict[int, np.ndarray]:
    """Exact Bayes-optimal reverse kernels of the uniform-corruption chain started at pi_b (per state):

    B_t(x_{t−1} | x_t, s) = Σ_{x0} p(x0 | x_t, s)·q(x_{t−1} | x_t, x0),
    with p(x0 | x_t, s) ∝ pi_b(x0 | s)·Q̄_t(x0, x_t).
    Returns {t: [S, K(x_t), K(x_{t−1})]} for t = 1..T; every row sums to one."""
    K = diffusion.K
    pb = validate_policy(pi_b, "diffusion behaviour prior", shape=(np.asarray(pi_b).shape[0], K))
    out: Dict[int, np.ndarray] = {}
    for t in range(1, diffusion.T + 1):
        joint = pb[:, None, :] * diffusion.Qbar[t].T[None, :, :]              # [S, x_t, x0]
        marg = joint.sum(axis=2, keepdims=True)                                # q_t(x_t | s) = (pi_b Q̄_t)(x_t)
        if not (marg > 0).all():
            raise AssertionError("forward marginal has a zero entry (Q̄_t must be strictly positive for t >= 1)")
        post = joint / marg
        B = np.ascontiguousarray(np.matmul(post.transpose(1, 0, 2), diffusion.posterior[t]).transpose(1, 0, 2))
        if not (np.allclose(B.sum(axis=2), 1.0, atol=1e-12) and (B >= -1e-15).all()):
            raise AssertionError(f"reverse kernel at t={t} is not a stochastic matrix")
        out[t] = np.clip(B, 0.0, None)
    return out


def x_guided_reverse(kernels: Dict[int, np.ndarray], critic_G: Optional[np.ndarray], guidance: float,
                     mask: np.ndarray, live: np.ndarray, init: Any = "uniform_admissible",
                     mask_mode: str = "every_step", guidance_mode: str = "every_step",
                     deterministic: bool = True) -> Tuple[np.ndarray, List[float]]:
    """Generic port of Part I `guided_reverse_policy`: exact marginal propagation of the reverse chain over all states.

    Same semantics: logits = log max(B_t, 1e-30) + g·G(s, x_{t−1}) when guided, −∞ outside the mask when masked,
    the marginal is re-masked and renormalised, and the trace records the maximum inadmissible mass on live states.
    `init` may also be an explicit [S, K] start distribution (used to verify the Bayes kernels)."""
    assert mask_mode in {"every_step", "final_only", "none"}
    assert guidance_mode in {"every_step", "final_only", "none"}
    if guidance_mode == "every_step" and guidance != 0 and mask_mode != "every_step":
        raise ValueError("Every-step guidance is only defined together with every-step masking.")
    T = max(kernels)
    if sorted(kernels) != list(range(1, T + 1)):
        raise ValueError("kernels must be indexed 1..T")
    S, K, _ = kernels[T].shape
    m = np.asarray(mask, dtype=bool)
    live = np.asarray(live, dtype=bool)
    if m.shape != (S, K) or not m.any(axis=1).all():
        raise ValueError("mask must be [S, K] with a non-empty row everywhere")
    if isinstance(init, str):
        assert init in {"uniform_admissible", "uniform_all"}
        p = m / m.sum(axis=1, keepdims=True) if init == "uniform_admissible" else np.full((S, K), 1.0 / K)
    else:
        p = validate_policy(init, "reverse-chain start distribution", shape=(S, K)).copy()
    if critic_G is not None and (np.shape(critic_G) != (S, K) or not np.isfinite(critic_G).all()):
        raise ValueError("critic_G must be a finite [S, K] array")
    trace: List[float] = []
    for t in range(T, 0, -1):
        logits = np.log(np.maximum(kernels[t], 1e-30))          # same floor as Part I's ReverseCache
        g = guidance if (guidance_mode == "every_step" or (guidance_mode == "final_only" and t == 1)) else 0.0
        if g != 0.0 and critic_G is not None:
            logits = logits + g * critic_G[:, None, :]
        masked_now = mask_mode == "every_step" or (mask_mode == "final_only" and t == 1)
        if masked_now:
            logits = np.where(m[:, None, :], logits, -np.inf)
        logits = logits - logits.max(axis=2, keepdims=True)
        kernel = np.exp(logits)
        kernel /= kernel.sum(axis=2, keepdims=True)
        p = np.matmul(p[:, None, :], kernel)[:, 0, :]
        if masked_now:
            p = np.where(m, p, 0.0)
        p = p / p.sum(axis=1, keepdims=True)
        trace.append(float(np.where(m, 0.0, p)[live].sum(axis=1).max()) if live.any() else 0.0)
    policy = one_hot_policy(p.argmax(axis=1), K) if deterministic else p
    if mask_mode != "none" and live.any():
        assert np.all(policy[live][~m[live]] <= 1e-14), "Masked reverse inference produced mass outside the mask."
        assert float(np.where(m, 0.0, p)[live].sum(axis=1).max()) <= 1e-12
    return policy, trace


print("Part II learners ready.")

### II.4b Method catalogue and the per-draw method runner

`run_xmethods(view, train, key)` fits the full suite on one training split. It returns records
{method, variant, hp, is_default, family, pi, info}, where `pi` is validated with `validate_policy` and `info` is a
small JSON-able dict (convergence, fall-back counts, model-side constraint diagnostics, fit time).

Fits that several methods share are computed once per draw:
* the default-expectile IQL critic is shared by IQL (AW stochastic), SfBC, IDQL and diffusion guidance;
* the λ cache is shared across the Lagrangian κ grid;
* the dense completed model is shared across the LP κ grid;
* the Bayes kernels are shared by every diffusion variant.

Two exactness checks of Proposition P7 run on every draw:
* the reverse chain started at q_T = π̂_β Q̄_T returns π̂_β to 1e-10;
* the unmasked chain started from uniform satisfies TV(output, π̂_β) ≤ ᾱ_T·TV(u, π̂_β).

Every difference between a Part II diffusion policy and its behaviour prior is therefore caused by the inference
modifications (admissible start, mask, guidance, argmax extraction), not by denoiser error. Part I's learned
denoiser cannot separate the two.

`X_METHOD_CATALOG` (saved as `xenv/xenv_method_catalog.csv`) documents each method: family, information used,
use of the mask, hyper-parameter grid with its pre-declared default, and reference.

In [ ]:
def x_method_table() -> List[Dict[str, Any]]:
    """Catalogue of the Part II method suite (family labels used by `run_xmethods`; grids read from XCFG)."""
    c = XCFG
    fmt = lambda t: ", ".join(str(v) for v in t)
    row = lambda method, family, variants, info, mask_use, hp, grid, default, ref: {
        "method": method, "family": family, "variants": variants, "information_used": info, "mask_use": mask_use,
        "hyperparameter": hp, "grid": grid, "default": default, "reference": ref}
    counts, model = "logged (s, a) counts", "empirical model (counts, P̂, r̂)"
    return [
        row("BC (stochastic)", "behaviour", "native", counts, "none", "", "", "", "behaviour cloning (empirical MLE)"),
        row("BC", "behaviour", "native, masked", counts, "extraction (masked)", "", "", "",
            "behaviour cloning (empirical MLE)"),
        row("Uniform-admissible", "behaviour", "native", "admissible mask only", "definition", "", "", "",
            "trivial admissible baseline"),
        row("CQL", "value", "native, masked", model, "extraction (masked)", "alpha", fmt(c.cql_alpha_grid),
            str(c.cql_alpha_default), "Kumar et al. 2020 (NeurIPS)"),
        row("IQL", "value", "native, masked", model, "extraction (masked)", "expectile", fmt(c.iql_expectile_grid),
            str(c.iql_expectile_default), "Kostrikov et al. 2022 (ICLR)"),
        row("IQL (AW stochastic)", "value", "native", model, "none", "expectile (fixed)", "",
            str(c.iql_expectile_default), "Kostrikov et al. 2022 (ICLR)"),
        row("BCQ", "pessimistic", "native, masked", model + " + behaviour ratio", "extraction (masked)", "tau",
            fmt(c.bcq_tau_grid), str(c.bcq_tau_default), "Fujimoto et al. 2019 (discrete BCQ)"),
        row("VI-LCB", "pessimistic", "native, masked", model + " + value range", "extraction (masked)", "c",
            fmt(c.lcb_c_grid), str(c.lcb_c_default), "Rashidinejad et al. 2021 (NeurIPS); Jin et al. 2021 (PEVI)"),
        row("MBS-QI", "pessimistic", "native, masked", model, "extraction (masked)", "b", fmt(c.mbsqi_b_grid),
            str(c.mbsqi_b_default), "Liu et al. 2020 (NeurIPS)"),
        row("SPIBB", "pessimistic", "native (stochastic)", model + " + behaviour policy", "none (projected by the driver)",
            "N_wedge", fmt(c.spibb_n_grid), str(c.spibb_n_default), "Laroche et al. 2019 (ICML)"),
        row("Lagrangian-MB", "constrained", "native", model + " + pessimistic completion + d̂0 + mask cost",
            "constraint (expected violations)", "kappa", fmt(c.lagrange_kappa_grid), str(c.lagrange_kappa_default),
            "Altman 1999; Le, Voloshin & Yue 2019 (Batch Policy Learning under Constraints, ICML); "
            "Tessler et al. 2019 (RCPO)"),
        row("CMDP-LP (model)", "constrained", "native", model + " + pessimistic completion + d̂0 + mask cost",
            "constraint (expected violations)", "kappa", fmt(c.mblp_kappa_grid), str(c.mblp_kappa_default),
            "Altman 1999; COptiDICE (Lee et al. 2022, ICLR)"),
        row("SfBC (exact)", "guided-prior", "native, masked", model + " + behaviour prior", "extraction (masked)",
            "alpha", fmt(c.sfbc_alpha_grid), str(c.sfbc_alpha_default), "Chen et al. 2023 (ICLR)"),
        row("SfBC (exact, argmax)", "guided-prior", "native", model + " + behaviour prior", "none", "alpha",
            fmt(c.sfbc_alpha_grid), str(c.sfbc_alpha_default), "Chen et al. 2023 (ICLR)"),
        row("IDQL (exact)", "guided-prior", "native, masked", model + " + behaviour prior", "extraction (masked)",
            "expectile (fixed)", "", str(c.iql_expectile_default), "Hansen-Estruch et al. 2023"),
        row("FISOR-tab (exact)", "constrained", "native", model + " + behaviour prior + mask",
            "critic support + extraction", "alpha", fmt(c.sfbc_alpha_grid), str(c.sfbc_alpha_default),
            "Zheng et al. 2024 (ICLR)"),
        row("Diffusion (exact, unguided)", "diffusion", "native", "behaviour prior (exact Bayes denoiser)",
            "inference (every reverse step)", "", "", "", "Austin et al. 2021 (D3PM); Part I Section 12"),
        row("Diffusion (exact, guided)", "diffusion", "native", "behaviour prior + IQL critic",
            "inference (every reverse step)", "guidance g", fmt(c.diffusion_guidance_grid),
            str(c.diffusion_guidance_default), "Dhariwal & Nichol 2021 (guidance); Part I Section 12"),
        row("Diffusion (exact, final-only)", "diffusion", "native", "behaviour prior + IQL critic",
            "inference (every reverse step); guidance at the final step only", "guidance g (fixed)", "",
            str(c.diffusion_guidance_default), "Part I Section 12"),
        row("Diffusion (exact, unmasked)", "diffusion", "native (stochastic)", "behaviour prior (exact Bayes denoiser)",
            "none", "", "", "", "Proposition P7 control"),
    ]


def run_xmethods(view: LearnerView, train: Dict[str, np.ndarray], key: str) -> List[dict]:
    """Fit the Part II method suite on `train` (Section II.4 contract) and return validated policy records."""
    S, A, live, M = view.S, view.A, view.live, view.mask
    emp = XEmpirical(train, S, A, view.terminals)
    if emp.n == 0:
        raise ValueError(f"{key}: empty training data")
    if not np.array_equal(emp.live, live):
        raise ValueError(f"{key}: the learner view and the data disagree on the absorbing states")
    if not live[train["state"]].all():
        raise ValueError(f"{key}: decision states must be live")
    if not np.isclose(emp.d0_hat.sum(), 1.0, atol=1e-12):
        raise ValueError(f"{key}: training data must contain complete episodes (timestep 0 rows)")
    for name in ("cql_alpha", "iql_expectile", "bcq_tau", "lcb_c", "mbsqi_b", "spibb_n", "lagrange_kappa",
                 "mblp_kappa", "sfbc_alpha", "diffusion_guidance"):
        grid, default = getattr(XCFG, f"{name}_grid"), getattr(XCFG, f"{name}_default")
        if default not in grid:
            raise ValueError(f"XCFG.{name}_default={default} is not in its grid {grid}")
    family = {r["method"]: r["family"] for r in x_method_table()}
    records: List[dict] = []
    clock = {"t": time.perf_counter()}

    def lap() -> float:
        now = time.perf_counter()
        dt, clock["t"] = now - clock["t"], now
        return dt

    def add(method: str, variant: str, pi: np.ndarray, hp: Optional[dict] = None, default: bool = True,
            info: Optional[dict] = None, admissible: bool = False) -> None:
        hp = {k: (v.item() if isinstance(v, np.generic) else v) for k, v in (hp or {}).items()}
        label = x_label(method, variant, hp, default)
        pi = validate_policy(pi, f"{key} | {label}", shape=(S, A))
        if (variant == "masked" or admissible) and live.any():
            outside = float(x_inadmissible_mass(pi, M, live).max())
            if outside != 0.0:
                raise AssertionError(f"{key} | {label}: mass {outside:.3e} outside the admissible mask")
        records.append({"method": method, "variant": variant, "hp": hp, "is_default": bool(default),
                        "family": family[method], "pi": pi, "info": _x_info(info or {})})

    # behaviour references
    bc = x_bc(emp, view)
    t = lap()
    add("BC (stochastic)", "native", bc["stochastic"], info={**bc["info"], "fit_s": t})
    add("BC", "native", bc["native"], info={**bc["info"], "fit_s": t})
    add("BC", "masked", bc["masked"], info={**bc["info"], "fit_s": t})
    add("Uniform-admissible", "native", x_uniform_admissible(M, live), admissible=True)

    # value-based
    cql = _x_cql_fit(emp, view, XCFG.cql_alpha_grid, XCFG.cql_steps, XCFG.cql_lr, XCFG.cql_target_every)
    t = lap() / len(cql)
    for a, res in zip(XCFG.cql_alpha_grid, cql):
        d = a == XCFG.cql_alpha_default
        for var in ("native", "masked"):
            add("CQL", var, res[var], {"alpha": a}, d, {**res["info"], "fit_s": t})
    del cql
    iql_default = None
    for e in XCFG.iql_expectile_grid:
        res = x_tab_iql(emp, view, e)
        t = lap()
        d = e == XCFG.iql_expectile_default
        for var in ("native", "masked"):
            add("IQL", var, res[var], {"expectile": e}, d, {**res["info"], "fit_s": t})
        if d:
            iql_default = res
    add("IQL (AW stochastic)", "native", iql_default["aw_policy"], info=iql_default["info"])

    # support-constrained / pessimistic
    for name, method, fn, grid, dflt, hp_name in (
            ("bcq", "BCQ", x_tab_bcq, XCFG.bcq_tau_grid, XCFG.bcq_tau_default, "tau"),
            ("lcb", "VI-LCB", x_vi_lcb, XCFG.lcb_c_grid, XCFG.lcb_c_default, "c"),
            ("mbsqi", "MBS-QI", x_mbs_qi, XCFG.mbsqi_b_grid, XCFG.mbsqi_b_default, "b")):
        for v in grid:
            res = fn(emp, view, v)
            t = lap()
            for var in ("native", "masked"):
                add(method, var, res[var], {hp_name: v}, v == dflt, {**res["info"], "fit_s": t})
    for nw in XCFG.spibb_n_grid:
        res = x_spibb(emp, view, nw)
        add("SPIBB", "native", res["pi"], {"n_wedge": nw}, nw == XCFG.spibb_n_default, {**res["info"], "fit_s": lap()})

    # constrained, model-based (completed empirical model)
    lam_cache: Dict[str, Dict[Any, Dict[str, Any]]] = {}
    for k in XCFG.lagrange_kappa_grid:
        res = x_lagrangian_mb(emp, view, k, cache=lam_cache)
        add("Lagrangian-MB", "native", res["pi"], {"kappa": k}, k == XCFG.lagrange_kappa_default,
            {**res["info"], "fit_s": lap()})
    del lam_cache
    P_model = emp.completed_dense(view.sink)
    for k in XCFG.mblp_kappa_grid:
        res = x_cmdp_lp_model(emp, view, k, P_model=P_model)
        add("CMDP-LP (model)", "native", res["pi"], {"kappa": k}, k == XCFG.mblp_kappa_default,
            {**res["info"], "fit_s": lap()})
    del P_model
    gc.collect()

    # behaviour prior x critic (exact)
    for a in XCFG.sfbc_alpha_grid:
        res = x_sfbc(emp, view, a, iql=iql_default)
        t = lap()
        d = a == XCFG.sfbc_alpha_default
        add("SfBC (exact)", "native", res["native"], {"alpha": a}, d, {**res["info"], "fit_s": t})
        add("SfBC (exact)", "masked", res["masked"], {"alpha": a}, d, {**res["info"], "fit_s": t})
        add("SfBC (exact, argmax)", "native", res["argmax"], {"alpha": a}, d, {**res["info"], "fit_s": t})
    res = x_idql(emp, view, iql=iql_default)
    t = lap()
    add("IDQL (exact)", "native", res["native"], info={**res["info"], "fit_s": t})
    add("IDQL (exact)", "masked", res["masked"], info={**res["info"], "fit_s": t})
    feas = x_tab_iql(emp, view, XCFG.iql_expectile_default, support=M)
    t_feas = lap()
    for a in XCFG.sfbc_alpha_grid:
        res = x_fisor_tab(emp, view, a, critic=feas)
        add("FISOR-tab (exact)", "native", res["pi"], {"alpha": a}, a == XCFG.sfbc_alpha_default,
            {**res["info"], "critic_converged": feas["info"]["converged"], "converged": feas["info"]["converged"],
             "fit_s": t_feas + lap()}, admissible=True)
    del feas

    # exact categorical diffusion with the Bayes-optimal denoiser of the empirical behaviour policy
    diff = x_diffusion_for(A)
    pb = emp.pi_b
    kernels = x_bayes_reverse_kernels(pb, diff)
    q_T = pb @ diff.Qbar[diff.T]
    back, _ = x_guided_reverse(kernels, None, 0.0, M, live, init=q_T, mask_mode="none", guidance_mode="none",
                               deterministic=False)
    roundtrip = float(np.abs(back - pb).max())
    if roundtrip > 1e-10:
        raise AssertionError(f"{key}: Bayes reverse chain from q_T does not return the behaviour prior ({roundtrip:.2e})")
    abar = float(diff.alpha_bar[-1])
    dinfo = {"T": diff.T, "alpha_bar_T": abar, "bayes_roundtrip_max_abs_err": roundtrip, "fit_s": lap()}
    G = normalize_critic_per_state(iql_default["q"], M)
    pi, tr = x_guided_reverse(kernels, None, 0.0, M, live)
    add("Diffusion (exact, unguided)", "native", pi, info={**dinfo, "g": 0.0, "max_trace_inadmissible": max(tr)},
        admissible=True)
    for g in XCFG.diffusion_guidance_grid:
        pi, tr = x_guided_reverse(kernels, G, g, M, live)
        add("Diffusion (exact, guided)", "native", pi, {"g": g}, g == XCFG.diffusion_guidance_default,
            {**dinfo, "max_trace_inadmissible": max(tr), "converged": iql_default["info"]["converged"]}, admissible=True)
    pi, tr = x_guided_reverse(kernels, G, XCFG.diffusion_guidance_default, M, live, guidance_mode="final_only")
    add("Diffusion (exact, final-only)", "native", pi,
        info={**dinfo, "g": XCFG.diffusion_guidance_default, "max_trace_inadmissible": max(tr),
              "converged": iql_default["info"]["converged"]}, admissible=True)
    pu, _ = x_guided_reverse(kernels, None, 0.0, M, live, init="uniform_all", mask_mode="none", guidance_mode="none",
                             deterministic=False)
    u = np.full(A, 1.0 / A)
    tv_out = 0.5 * np.abs(pu - pb).sum(axis=1)[live]
    tv_bound = abar * 0.5 * np.abs(u[None, :] - pb).sum(axis=1)[live]
    if live.any() and not (tv_out <= tv_bound + 1e-12).all():
        raise AssertionError(f"{key}: unmasked reverse chain violates TV(out, prior) <= alpha_bar_T TV(u, prior)")
    add("Diffusion (exact, unmasked)", "native", pu,
        info={**dinfo, "tv_to_prior_max": float(tv_out.max()) if live.any() else 0.0,
              "tv_bound_max": float(tv_bound.max()) if live.any() else 0.0, "tv_bound_global": abar * (1.0 - 1.0 / A)})
    del kernels
    return records


X_METHOD_CATALOG = pd.DataFrame(x_method_table())
X_METHOD_CATALOG.to_csv(XENV_TABLE_DIR / "xenv_method_catalog.csv", index=False)
display(X_METHOD_CATALOG[["method", "family", "variants", "mask_use", "hyperparameter", "grid", "default"]])

### II.4c Self-test of the method suite (tiny random absorbing CMDP, seconds)

This test checks the exact solvers against independent computations before any Part II result is produced. It
draws a random 12-state, 4-action absorbing MDP with a random mask, rewards +1 and −1 on entry to two absorbing
states, and data from a random behaviour policy. The learners see only the data and the `LearnerView`; the check
code alone uses the generating MDP. The following must hold, otherwise the cell raises:

* the expectile solver matches bisection;
* IQL satisfies its fixed-point equations;
* BCQ, VI-LCB and MBS-QI have Bellman residuals at the iteration tolerance;
* SPIBB keeps every bootstrapped pair at π̂_β and is policy-stable;
* the Lagrangian policy meets its model budget, and the infeasible side of the bisection violates it;
* the sparse completed-model evaluation equals a dense solve on `XEmpirical.completed_dense`;
* the LP objective equals the exact model value of the LP policy;
* the Bayes reverse kernels are exact;
* every masked record has zero inadmissible mass.

The result is saved to `xenv/xenv_methods_selftest.json`.

In [ ]:
def x_methods_selftest(n_live: int = 12, n_actions: int = 4, n_episodes: int = 600) -> Dict[str, Any]:
    rng = np_rng("xenv", "methods-selftest")
    A = n_actions
    S, goal, death = n_live + 2, n_live, n_live + 1
    P = np.zeros((S, A, S))
    for s in range(n_live):
        for a in range(A):
            succ = rng.choice(n_live, size=3, replace=False)
            h, q = rng.uniform(0.1, 0.3), rng.uniform()
            P[s, a, succ] = (1.0 - h) * rng.dirichlet(np.ones(3))
            P[s, a, goal] += h * q
            P[s, a, death] += h * (1.0 - q)
    P[goal, :, goal] = 1.0
    P[death, :, death] = 1.0
    entry = np.zeros(S)
    entry[goal], entry[death] = 1.0, -1.0
    d0 = np.zeros(S)
    d0[:4] = 0.25
    mask = rng.random((S, A)) < 0.6
    mask[np.arange(n_live), rng.integers(0, A, n_live)] = True
    env = FiniteEnv(name="methods-selftest", kind="variant", P=P, entry_reward=entry, d0=d0, terminals=(goal, death),
                    sink=death, goal=goal, mask=mask)
    data = x_sample_dataset(env, rng.dirichlet(np.full(A, 0.7), size=S), n_episodes, 60, rng)
    view = LearnerView.from_env(env)
    emp = XEmpirical(data, S, A, env.terminals)
    live, obs = view.live, emp.observed
    out: Dict[str, Any] = {}
    # expectile: closed form vs bisection on the first-order condition
    x = rng.normal(size=(200, 6))
    x[:20, 3] = x[:20, 1]                       # ties
    w = rng.integers(0, 4, size=(200, 6)).astype(float)
    w[w.sum(axis=1) == 0, 0] = 1.0
    errs = []
    for tau in (0.1, 0.5, 0.7, 0.9):
        v = x_expectile(x, w, tau)
        lo, hi = x.min(axis=1), x.max(axis=1)
        for _ in range(200):
            mid = 0.5 * (lo + hi)
            f = (w * np.where(x > mid[:, None], tau, -(1.0 - tau)) * np.abs(x - mid[:, None])).sum(axis=1)
            lo, hi = np.where(f > 0, mid, lo), np.where(f > 0, hi, mid)
        errs.append(float(np.abs(v - 0.5 * (lo + hi)).max()))
    out["expectile_vs_bisection_max_err"] = max(errs)
    # IQL fixed point
    iql = x_tab_iql(emp, view)
    rows = np.flatnonzero(live & emp.seen)
    out["iql_v_equation_err"] = float(np.abs(iql["v"][rows] - x_expectile(iql["q"][rows], emp.N_sa[rows],
                                                                          XCFG.iql_expectile_default)).max())
    out["iql_q_equation_err"] = float(np.abs((emp.r_hat + emp.backup(iql["v"]) - iql["q"])[obs]).max())
    out["iql_converged"] = iql["info"]["converged"]
    # Bellman residuals of the value-iteration learners
    bcq = x_tab_bcq(emp, view)
    V_bcq = np.where(live, np.where(bcq["allowed"], bcq["q"], -np.inf).max(axis=1), 0.0)
    out["bcq_bellman_residual"] = float(np.abs(emp.r_hat + emp.backup(V_bcq) - bcq["q"]).max())
    lcb = x_vi_lcb(emp, view)
    V = np.where(live, lcb["q"].max(axis=1), 0.0)
    ref = np.where(obs & live[:, None], np.clip(emp.r_hat - XCFG.lcb_c_default * (view.v_max - view.v_min)
                                                / np.sqrt(np.maximum(emp.N_sa, 1.0)) + emp.backup(V),
                                                view.v_min, view.v_max), view.v_min)
    out["vi_lcb_bellman_residual"] = float(np.abs(np.where(live[:, None], ref, 0.0) - lcb["q"]).max())
    mbs = x_mbs_qi(emp, view)
    z = mbs["zeta"]
    Vz = np.where(live, np.where(z.any(axis=1), np.where(z, mbs["q"], -np.inf).max(axis=1), view.v_min), 0.0)
    out["mbs_qi_bellman_residual"] = float(np.abs((emp.r_hat + emp.backup(Vz) - mbs["q"])[z]).max()) if z.any() else 0.0
    # SPIBB: constraint and stability
    sp_res = x_spibb(emp, view)
    boot = sp_res["bootstrapped"] & live[:, None]
    out["spibb_bootstrapped_max_dev"] = float(np.abs(sp_res["pi"][boot] - emp.pi_b[boot]).max()) if boot.any() else 0.0
    out["spibb_converged"] = sp_res["info"]["converged"]
    # sparse completed-model evaluation vs a dense solve on completed_dense
    r_model, cost = _x_model_reward_cost(emp, view)
    Pc = emp.completed_dense(view.sink)
    pi_r = rng.dirichlet(np.ones(A), size=S)
    Lx = np.flatnonzero(live)
    Ppi = np.einsum("sa,san->sn", pi_r, Pc)[np.ix_(Lx, Lx)]
    mu = np.linalg.solve((np.eye(len(Lx)) - Ppi).T, emp.d0_hat[Lx])
    ev = _x_model_eval(emp, view, pi_r, r_model, cost)
    out["completed_model_sparse_vs_dense_violation_err"] = abs(ev["violations"] - float(mu @ (pi_r * cost).sum(axis=1)[Lx]))
    out["completed_model_sparse_vs_dense_return_err"] = abs(ev["return"] - float(mu @ (pi_r * r_model).sum(axis=1)[Lx]))
    # Lagrangian budget and tightness; LP objective vs exact model value
    cache: Dict[str, Dict[Any, Dict[str, Any]]] = {}
    lag_ok, lp_gap = [], []
    for k in XCFG.lagrange_kappa_grid:
        lag = x_lagrangian_mb(emp, view, k, cache=cache)["info"]
        lag_ok.append(lag["model_violations"] <= k + 1e-12 and
                      (lag["infeasible_side_violations"] is None or lag["infeasible_side_violations"] > k))
        lp = x_cmdp_lp_model(emp, view, k, P_model=Pc)["info"]
        if lp["status"] == "ok":
            lp_gap.append(lp["lp_eval_gap"])
    out["lagrangian_budget_and_tightness_ok"] = bool(all(lag_ok))
    out["lp_eval_gap_max"] = float(max(lp_gap)) if lp_gap else None
    # full suite: masked invariants and Bayes-kernel checks run inside run_xmethods
    recs = run_xmethods(view, data, "methods-selftest")
    out["records"] = len(recs)
    out["masked_records_outside_mass_max"] = float(max(x_inadmissible_mass(r["pi"], view.mask, live).max()
                                                       for r in recs if r["variant"] == "masked"))
    out["bayes_roundtrip_max_abs_err"] = max(r["info"]["bayes_roundtrip_max_abs_err"] for r in recs
                                             if r["method"].startswith("Diffusion"))
    out["methods"] = sorted({r["method"] for r in recs})
    tol_vi = 10 * XCFG.tol
    out["passed"] = bool(out["expectile_vs_bisection_max_err"] < 1e-12 and out["iql_v_equation_err"] < 1e-12
                         and out["iql_q_equation_err"] < tol_vi and out["bcq_bellman_residual"] < tol_vi
                         and out["vi_lcb_bellman_residual"] < tol_vi and out["mbs_qi_bellman_residual"] < tol_vi
                         and out["spibb_bootstrapped_max_dev"] == 0.0 and out["spibb_converged"]
                         and out["completed_model_sparse_vs_dense_violation_err"] < 1e-9
                         and out["completed_model_sparse_vs_dense_return_err"] < 1e-9
                         and out["lagrangian_budget_and_tightness_ok"]
                         and (out["lp_eval_gap_max"] is None or out["lp_eval_gap_max"] < 1e-7)
                         and out["masked_records_outside_mass_max"] == 0.0
                         and out["bayes_roundtrip_max_abs_err"] < 1e-10 and out["iql_converged"])
    return out


X_METHODS_SELFTEST = None
if any(EXPERIMENTS[k] for k in _X_SWITCHES):
    X_METHODS_SELFTEST = x_methods_selftest()
    save_json(X_METHODS_SELFTEST, XENV_TABLE_DIR / "xenv_methods_selftest.json")
    print("Method-suite self-test:", {k: v for k, v in X_METHODS_SELFTEST.items() if k != "methods"})
    assert X_METHODS_SELFTEST["passed"], "Part II method-suite self-test failed; see xenv/xenv_methods_selftest.json"

# Objects whose source enters the Part II code hash.
XENV_METHOD_CODE = [_x_info, _x_iterate, _x_argmax_tiebreak, _x_policy_rows, _x_chain, _x_markov_solve, x_bc,
                    _x_cql_fit, x_tab_cql, x_expectile, _x_aw_extraction, x_tab_iql, x_behavior_threshold_set,
                    x_tab_bcq, x_vi_lcb, x_mbs_qi, x_spibb, _x_model_reward_cost, _x_model_eval, _x_lagrangian_greedy,
                    x_lagrangian_mb, x_cmdp_lp_model, _x_tilted, x_sfbc, x_idql, x_fisor_tab, x_diffusion_for,
                    x_bayes_reverse_kernels, x_guided_reverse, x_method_table, run_xmethods, CategoricalDiffusion,
                    normalize_critic_per_state]
print("Part II method suite ready:", len(X_METHOD_CATALOG), "methods.")

## II.5 Propositions on unsupported-action semantics, and exact constrained frontiers

Part II's benchmark-validity results rest on seven elementary facts about absorbing MDPs whose unsupported actions
are completed by a rule. Each is stated below with its assumptions and a proof sketch, and is **verified numerically
before any experiment is interpreted**, with this scope: P1–P3 and P5 on ICU-Sepsis (P1/P2 where the released
mean-rule identity holds), the draw-0 Gumbel and Garnet instances and 20 random absorbing MDPs; P4 on the Gumbel and
Garnet instances and the random MDPs (no ground truth for ICU-Sepsis); P7 on ICU-Sepsis and the draw-0 benchmark
instances; P6 by the Part I gradient tests (Section 13) and closed-form checks. The checks go to
`xenv/proposition_checks.csv` (one row per proposition × environment × instance × check, aggregated over policies)
and the per-proposition status to `xenv/proposition_summary.csv`. A check row has a tolerance or bound and `passes` ∈ {True, False}.
Descriptive rows (counts, magnitudes, tightness) leave `passes` empty, and "not run" rows say why. The release
gate (Section 34) requires `PROPOSITIONS_OK` not to be False.

**Verification set.**
* ICU-Sepsis (`X_ICU`).
* The draw-0 instances of the ICU cell, the first Gumbel "best" cell, the Gumbel "oracle" cell and every Garnet
  cell. Each instance contributes every policy that `run_xmethods` learns from its draw-0 training split (cached
  under an identity in `XENV_CACHE_DIR/theory`, with the same learner key as the driver). It also contributes the
  behaviour policy, the uniform-admissible and uniform policies, six Dirichlet(0.5) random policies and two random
  policies concentrated (80 %) on unsupported actions. Policies are lifted to the true state space where a target
  lives there.
* 20 random absorbing MDPs (S = 12, A = 4). Each has 9 live states and 3 absorbing states with random entry rewards,
  so r_term can lie strictly inside [v_min, v_max]. Masks are random, with one single-admissible state and one fully
  admissible state per MDP, and W is a random convex imputation. The 20 MDPs are aggregated as the worst case per check.

**Setting and notation.**
* M = (𝒮, 𝒜, P, R, d₀), 𝒮 = 𝓛 ∪ 𝒯 (live / absorbing), γ = 1. The reward R(s′) is paid on *entering* s′ ∈ 𝒯, and
  R = 0 on 𝓛, so r(s,a) = Σ_{s′} P(s′|s,a)R(s′). Every live state has an admissible set ∅ ≠ M(s) ⊆ 𝒜.
* A policy π is *proper* if it is absorbed with probability 1. Then F_π = I − P_π[𝓛,𝓛] is invertible and:
  v_π = F_π⁻¹r_π (v = 0 on 𝒯); J(π) = d₀ᵀv_π; occupancy μ_π = F_π⁻ᵀd₀ (expected visits); L = 1ᵀμ_π.
* Inadmissible mass b_π(s) = Σ_{a∉M(s)} π(a|s), and E[viol] = μ_πᵀb_π. U_π = R + v_π is the value of *entering* a
  state.
* v_min = min(0, min_𝒯 R), v_max = max(0, max_𝒯 R) (`FiniteEnv.value_bounds`), and r_term = R(sink).
* **Semantics.** *given* uses P. *terminate* replaces every inadmissible live row by δ_sink, so the forced jump pays
  r_term.
* **Convex imputation W.** For each live s and a ∉ M(s), W(·|s,a) is a distribution on M(s). P_W(·|s,a) =
  Σ_{a′} W(a′|s,a)P(·|s,a′) on inadmissible pairs, and P_W = P elsewhere. The ICU-Sepsis mean rule is W uniform on
  M(s).
* **Projection** π_W(a′|s) = π(a′|s)1[a′∈M(s)] + Σ_{a∉M(s)} π(a|s)W(a′|s,a) (`x_project`).

### Propositions

**P1 (projection invariance under convex imputation).**
* *Assumption:* the given rows are P_W for a convex W, i.e. every inadmissible row is a W-mixture of the admissible
  rows of the same state.
* *Statement:* for every policy π and s ∈ 𝓛, P_{π_W}(·|s) = P_π(·|s) and r_{π_W}(s) = r_π(s). Hence π is proper iff
  π_W is, and v, J, ℓ, L, μ and the absorption distribution coincide, while π_W is admissible.
* *Proof:* Σ_{a∉M}π(a|s)P_W(·|s,a) = Σ_{a′∈M}[Σ_{a∉M}π(a|s)W(a′|s,a)]P(·|s,a′). Adding the admissible part gives
  P_{π_W}(·|s). The reward r is linear in the row. ∎
* *Checks (tolerance 1e-9):* induced chains, J, L, μ, v and properness, on every environment whose given rows pass the
  imputation test. These are ICU-Sepsis with the uniform W of the released mean rule, every constructed benchmark,
  the oracle benchmark and the true-rows + mean completions. The random MDPs use random W.
* *Controls:* a negative control projects with the wrong W. On the true environments the projection effect is
  reported, not claimed.

**P2 (optimum under convex imputation).**
* *Assumptions:* those of P1, and an optimal proper policy exists. Value iteration converges here and its greedy
  policy is evaluated exactly; every Gumbel and Garnet policy is proper because of the per-step absorption hazard.
* *Statement:*
  * V* = V*_M: the optimum over all policies equals the optimum over admissible policies.
  * For s ∈ 𝓛 and a ∉ M(s), Q*(s,a) = Σ_{a′}W(a′|s,a)Q*(s,a′) ≤ max_{a′∈M(s)}Q*(s,a′), with equality iff W(·|s,a)
    is supported on the admissible argmax. In particular every action ties when |M(s)| = 1.
* *Consequences:*
  * Unconstrained greedy policies with lowest-index tie-breaking can take inadmissible actions at zero loss under the
    given rows (the "oracle artefact" of the earlier notebook), but they lose under termination.
  * The exact violation-return frontier under such rows is flat.
* *Proof:* sup over all ≥ sup over admissible, and by P1 every π is matched by the admissible π_W. Q*(s,a) =
  P_W(·|s,a)ᵀU* is the W-average of the admissible Q*. ∎

**P3 (termination gap).**
* *Assumption:* π is proper under both semantics of the same environment (whatever its given rows are). Let τ be the
  first decision with A_τ ∉ M(S_τ), and p = P(τ < ∞). p is the same under both semantics, because the processes
  coincide up to and including (S_τ, A_τ).
* *Statement:*
  * J_given − J_term = Σ_{s∈𝓛} μ^term(s) Σ_{a∉M(s)} π(a|s)(Q^π_given(s,a) − r_term).
  * Hence p(v_min − r_term) ≤ J_given − J_term ≤ p(v_max − r_term).
  * Moreover p = E_term[viol] ≤ E_given[viol].
* *Proof:* couple the two processes up to τ.
  * No reward is collected before τ, because rewards are paid only on absorption and S_τ is live.
  * After τ, termination pays r_term. The given rows pay, in expectation, Q^π_given(S_τ,A_τ), a convex combination of
    terminal rewards, which lies in [v_min, v_max].
  * By the strong Markov property, P(S_τ = s, A_τ = a) = μ^term(s)π(a|s).
  * p ≤ E_given[viol] is Markov's inequality. ∎
* *Corollary:* if r_term = v_min, then Q_term(s,a) = v_min ≤ max_{a′∈M(s)}Q_term(s,a′) for every a ∉ M(s). So the
  unconstrained terminate optimum equals the admissible optimum, and **the terminate frontier is flat as well**.
  This holds for ICU-Sepsis (r_term = 0), Gumbel (−1) and Garnet (0).
* *Checks:* every learned policy of every draw-0 instance, on every target environment; tolerances 1e-10 / 1e-9.

**P4 (exact benchmark error decomposition).**
* *Assumptions:*
  * true dynamics M0 = P0 and a completion M1 with P1(·|s,a) = P0(·|s,a) for all s ∈ 𝓛, a ∈ M(s);
  * the same R, d₀, mask and absorbing rows;
  * π proper in both.
* *Statement (simulation lemma, γ = 1):*
  * J_M0(π) − J_M1(π) = Σ_{s∈𝓛} μ1(s) Σ_{a∉M(s)} π(a|s)(P0(·|s,a) − P1(·|s,a))ᵀU0, where U0 = R + v0 (v0 = 0 on 𝒯) and
    μ1 is the occupancy under M1.
  * Hence |J_M0 − J_M1| ≤ Σ_sμ1(s)Σ_{a∉M}π(a|s)TV_sa·span(U0) ≤ E_M1[viol]·max_{s,a∉M}TV(P0(·|s,a), P1(·|s,a))·span(U0).
  * For a benchmark B with lifted policies: J_true − J_B = [J_M0 − J_M1] + [J_M1 − J_B]. The first term is the
    semantics of unsupported actions; the second is estimation, coverage and lifting.
  * This holds for both completions: mean (M1 = true rows + mean rule) and terminate (M1 = true rows + termination,
    with P1(·|s,a) = δ_sink, so the forced jump pays R(sink)). For the oracle construction the second term is
    identically zero.
* *Proof:*
  * Δ = v0 − v1 satisfies Δ = δ + P1_π[𝓛,𝓛]Δ, with δ(s) = Σ_aπ(a|s)(P0 − P1)(·|s,a)ᵀU0. This uses U0 − U1 = Δ,
    since R is shared.
  * Multiply by d₀ᵀF1⁻¹ = μ1ᵀ.
  * The admissible terms vanish because those rows coincide, and |(p − q)ᵀf| ≤ TV(p,q)·span(f). ∎
* *Checks:* identity to 1e-9 and both bounds, for both completions, on lifted policies on the true state space. The
  preconditions (shared space, coinciding admissible and absorbing rows) are asserted as checks.

**P5 (admissible policies are semantics-invariant).**
* *Assumption:* two targets on the same state space with the same d₀, R, mask, sink and admissible live rows; each may
  use either semantics.
* *Statement:* if b_π ≡ 0 on 𝓛, every evaluation quantity of π (J, L, μ, absorption distribution, properness) is
  identical on both targets.
* *Proof:* P_π uses admissible rows only, and termination rewrites inadmissible rows only. ∎
* *Consequence:* completion rules can only change the values, and hence the ranking, of policies that use unsupported
  actions, and P1's projection removes that dependence.

**P6 (hard-masked CQL regulariser defect; Part I Sections 10 and 13).**
* *Setting:* ℓ_legacy(Q; s, a) = T·log Σ_{a′∈M(s)} exp(Q(s,a′)/T) − Q(s,a).
* *Statement:*
  * For a logged a ∉ M(s), ∂ℓ/∂Q(s,a) = −1 (−1/B in a batch mean) and ℓ → −∞ as Q(s,a) → ∞: the regulariser *rewards*
    inflating excluded logged actions.
  * The admissible-row design ℓ_B = 1[a∈M(s)]·ℓ is ≥ 0, since T·logsumexp ≥ max ≥ Q(s,a), and is constant in every
    excluded coordinate.
* *Checks:* Section 13's torch gradient tests (`REG_GRAD_TESTS`), summarised, plus closed-form numpy checks.

**P7 (exact categorical diffusion with the Bayes-optimal denoiser).**
* *Assumptions:*
  * uniform corruption Q_t = (1 − β_t)I + β_t11ᵀ/K with β_t ∈ (0,1), so Q̄_t = ᾱ_tI + (1 − ᾱ_t)11ᵀ/K;
  * denoiser p(x0|x_t,s) ∝ π̂_β(x0|s)Q̄_t(x0,x_t);
  * reverse kernels B_t(x_{t−1}|x_t,s) = Σ_{x0}p(x0|x_t,s)q(x_{t−1}|x_t,x0) (`x_bayes_reverse_kernels`).
* *Statement:*
  * (a) Started from q_T(·|s) = π̂_β(·|s)Q̄_T = ᾱ_Tπ̂_β + (1 − ᾱ_T)u, the reverse chain returns π̂_β(·|s) exactly.
  * (b) From any start ρ, TV(ρB_T⋯B_1, π̂_β) ≤ TV(ρ, q_T).
  * (c) For the uniform start u, TV(u, q_T) = ᾱ_T·TV(u, π̂_β) ≤ ᾱ_T(1 − 1/K).
* *Proof:*
  * B_t is the exact time reversal of the forward chain started at π̂_β: Σ_{x_t}q_t(x_t)B_t(·|x_t) = q_{t−1}.
  * Markov kernels are TV non-expansive (data processing), with ρ ↦ ρB_T⋯B_1 and q_T ↦ π̂_β.
  * u − q_T = ᾱ_T(u − π̂_β), and TV(u, π) ≤ 1 − 1/K. ∎
* *Consequence:* the unmasked, unguided exact diffusion policy is within ᾱ_T(1 − 1/K) of stochastic behaviour
  cloning in every state. This is ≈ 0.05 for the Part I schedule, ᾱ_T ≈ 0.0515. P7 bounds only this unmasked
  stochastic chain from the uniform start: differences of the deployed diffusion policies come from the admissible
  start, masking, guidance and argmax extraction, not from denoiser error.
* *Checks:* per state, on ICU-Sepsis (π̂_β of MAIN) and on every draw-0 benchmark (π̂_β of the training split), plus a
  cross-check of the "Diffusion (exact, unmasked)" policy of `run_xmethods`. The argmax agreement of final-step-only
  guided diffusion with masked SfBC (both masked) and with unmasked SfBC (argmax) is reported descriptively.

### Exact constrained frontiers

**Problem.** For an environment E, semantics σ and budget κ:
J*_σ(κ) = max_π J_σ(π) s.t. E_{σ,π}[#{t : A_t ∉ M(S_t)}] ≤ κ.

**LP form** (`x_cmdp_lp`; Altman 1999). With expected visit counts x(s,a) ≥ 0 (γ = 1):
max Σx·r s.t. Σ_a x(s′,a) − Σ_{s,a}P(s′|s,a)x(s,a) = d₀(s′) (s′ ∈ 𝓛), Σx·c ≤ κ, with c(s,a) = 1[a ∉ M(s)] on live
states.

**Properties.** J*(κ) is non-decreasing and concave in κ: the optimal value of an LP is concave in a right-hand side.
J*(0) is the admissible optimum and J*(∞) the unconstrained optimum.

**Cross-checks.**
* J*(0) and J*(∞) against value iteration (`x_optimal`, constrained / unconstrained), to 1e-7.
* The exact evaluation of the LP policy equals the LP objective to 1e-7, and its exact expected violations are
  ≤ κ + 1e-7. On states the LP never visits, the policy plays the admissible VI optimum (or, if value iteration is
  unavailable, a proper admissible policy from a full-support LP): this does not affect the return and keeps the
  evaluated chain proper. A value-iteration failure is reported as "not run" for the VI cross-check only, and P2
  and the P3 corollary are then verified through the occupancy LP.
* Monotonicity and concavity on the κ grid, which always contains 0 and ∞.

**Problems solved.**
* ICU-Sepsis (given and terminate).
* The first Gumbel "best" cell, draw 0: benchmark under given and terminate, and the true system with cost
  1[a ∉ mask_true].
* The Gumbel "oracle" cell, draw 0: the true system with its own mask_true, plus the oracle benchmark.
* The Gumbel physician-control cells ("best" construction, physician other than 'optimal', e.g. withhold-vaso),
  draw 0: benchmark under given and terminate, and the true system (frontiers and LP cross-checks only; their
  propositions are not re-run).
* Every Garnet cell, draw 0: the true system.

**Expected flat frontiers.** Under given rows that are a convex imputation the frontier is flat by P2. Under
termination with r_term = v_min it is flat by the P3 corollary. Both are reported as **findings**, not errors: the
released benchmark under either completion rule cannot express any value of unsupported actions. True frontiers
are flat wherever the mask contains the true optimum (optimal-physician Gumbel cells, Garnet h ≥ 0 at ε = 0.05);
they show a trade-off in the controls (withhold-vaso about 0.12; Garnet h < 0 up to about 0.007) and a small one for
Garnet h > 0 at ε = 0.30 (up to about 0.0015; draw-0 values of a pre-run check). The range of a true frontier
is `value_of_unsupported_actions` in `xenv/xenv_environment_suite.csv`. Output:
`xenv/exact_frontiers.csv` and the global `X_FRONTIERS`.

In [ ]:
# ============================================================
# II.5 Propositions and exact frontiers: verification helpers
# ============================================================
EXECLOG.mark("II.5 Propositions and exact frontiers")

X_PROP_COLUMNS = ["proposition", "environment", "instance", "check", "value", "tolerance_or_bound", "passes", "notes"]
X_PROPOSITION_TITLES = {
    "P1": "projection invariance under convex imputation W",
    "P2": "optimum under convex imputation (admissible optimum, ties, flat frontier)",
    "P3": "termination gap: identity, bounds, flat terminate frontier when r_term = v_min",
    "P4": "exact benchmark error decomposition (simulation lemma, both completions)",
    "P5": "admissible policies are semantics-invariant",
    "P6": "hard-masked CQL regulariser defect",
    "P7": "exact categorical diffusion with the Bayes-optimal denoiser",
    "LP": "exact constrained frontier: LP versus value iteration and exact evaluation",
}
X_FRONTIER_COLUMNS = ["family", "cell", "draw", "target", "env", "env_kind", "semantics", "kappa", "objective",
                      "exact_return", "lp_violations", "exact_violations", "prob_any_violation", "avg_episode_length",
                      "expected_decisions", "reachable_live_states", "randomised_states", "J_opt_admissible",
                      "J_opt_unconstrained", "value_of_unsupported_actions", "flat_expected", "flat_reason",
                      "frontier_range", "gap_eval", "gap_vi", "feasibility_excess", "monotone_concave_violation",
                      "eval_status", "vi_status", "solve_s", "passes"]
X_IMPUTATION_TOL = 1e-12     # given rows are a convex imputation (same threshold as Section 3's mean-rule identity)
X_LP_TOL = 1e-7              # LP vs VI / exact-evaluation tolerance (HiGHS feasibility scale)
X_INVARIANT = "implementation invariant (not an independent check)"   # descriptive rows that hold by construction


class XPropositionLog:
    """Rows of proposition_checks.csv: checks (pass/fail against a bound), descriptive reports and 'not run' rows."""

    def __init__(self):
        self.rows: List[dict] = []

    def check(self, prop: str, environment: str, instance: str, check: str, value: float, bound: float,
              op: str = "le", notes: str = "") -> bool:
        v = float(value)
        if op == "le":
            ok = bool(np.isfinite(v) and v <= bound)
        elif op == "ge":
            ok = bool(np.isfinite(v) and v >= bound)
        else:
            raise ValueError(op)
        rule = "pass if value <= bound" if op == "le" else "pass if value >= bound"
        self.rows.append({"proposition": prop, "environment": environment, "instance": instance, "check": check,
                          "value": v, "tolerance_or_bound": float(bound), "passes": ok,
                          "notes": f"{rule}; {notes}" if notes else rule, "_op": op})
        return ok

    def report(self, prop: str, environment: str, instance: str, check: str, value: float, notes: str = "") -> None:
        self.rows.append({"proposition": prop, "environment": environment, "instance": instance, "check": check,
                          "value": float(value), "tolerance_or_bound": np.nan, "passes": pd.NA,
                          "notes": f"descriptive; {notes}" if notes else "descriptive", "_op": "report"})

    def not_run(self, prop: str, environment: str, instance: str, check: str, reason: str) -> None:
        self.rows.append({"proposition": prop, "environment": environment, "instance": instance, "check": check,
                          "value": np.nan, "tolerance_or_bound": np.nan, "passes": pd.NA,
                          "notes": f"not run: {reason}", "_op": "not_run"})

    def merge_aggregated(self, other: "XPropositionLog", environment: str, instance: str) -> None:
        """Worst case over `other`'s rows per (proposition, check): used for the random-MDP suite."""
        groups: Dict[Tuple[str, str, str], List[dict]] = {}
        for r in other.rows:
            groups.setdefault((r["proposition"], r["check"], r["_op"]), []).append(r)
        for (prop, check, op), rows in groups.items():
            if op in ("le", "ge"):
                margin = [np.nan_to_num((r["value"] - r["tolerance_or_bound"]) * (1 if op == "le" else -1), nan=np.inf)
                          for r in rows]
                worst = rows[int(np.argmax(margin))]
                n_fail = sum(not r["passes"] for r in rows)
                self.rows.append({**worst, "environment": environment, "instance": instance,
                                  "passes": bool(n_fail == 0),
                                  "notes": f"worst of {len(rows)} rows ({n_fail} failed); {worst['notes']}"})
            elif op == "report":
                vals = np.array([r["value"] for r in rows], dtype=np.float64)
                v = float(np.nanmax(vals)) if np.isfinite(vals).any() else np.nan
                self.rows.append({**rows[0], "environment": environment, "instance": instance, "value": v,
                                  "notes": f"descriptive; max over {len(rows)} rows"})
            else:
                self.rows.append({**rows[0], "environment": environment, "instance": instance,
                                  "notes": f"{rows[0]['notes']} ({len(rows)} rows)"})

    def frame(self) -> pd.DataFrame:
        df = pd.DataFrame([{k: r[k] for k in X_PROP_COLUMNS} for r in self.rows], columns=X_PROP_COLUMNS)
        df["passes"] = df["passes"].astype("boolean")
        return df


def x_tv(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    """Total-variation distance along the last axis."""
    return 0.5 * np.abs(np.asarray(p, dtype=np.float64) - np.asarray(q, dtype=np.float64)).sum(axis=-1)


def x_imputation_deviation(env: FiniteEnv, W: Optional[np.ndarray] = None) -> float:
    """max |P(.|s,a) - sum_a' W(a'|s,a) P(.|s,a')| over live inadmissible pairs (0 iff env's given rows are P_W)."""
    imp = x_impute_inadmissible(env.P, env.mask, env.live, W)
    dev = float(np.abs(imp - env.P).max())
    del imp
    return dev


def x_admissible_row_deviation(P_a: np.ndarray, P_b: np.ndarray, mask: np.ndarray, live: np.ndarray) -> float:
    """max |P_a(.|s,a) - P_b(.|s,a)| over live admissible pairs."""
    dev = 0.0
    for s in np.flatnonzero(live):
        m = mask[s]
        dev = max(dev, float(np.abs(P_a[s, m] - P_b[s, m]).max()))
    return dev


def x_inadmissible_tv(P_a: np.ndarray, P_b: np.ndarray, mask: np.ndarray, live: np.ndarray) -> np.ndarray:
    """[S, A] total variation between the rows of two models on live inadmissible pairs (0 elsewhere)."""
    tv = np.zeros(mask.shape)
    for s in np.flatnonzero(live):
        bad = ~mask[s]
        if bad.any():
            tv[s, bad] = x_tv(P_a[s, bad], P_b[s, bad])
    return tv


def x_shares_admissible_rows(e1: FiniteEnv, e2: FiniteEnv) -> Tuple[bool, float]:
    """P5 precondition: same space, d0, rewards, mask, sink, absorbing set and admissible live rows."""
    if e1 is e2:
        return True, 0.0
    if e1.P.shape != e2.P.shape or e1.terminals != e2.terminals or e1.sink != e2.sink:
        return False, np.inf
    if not (np.array_equal(e1.mask, e2.mask) and np.array_equal(e1.d0, e2.d0)
            and np.array_equal(e1.entry_reward, e2.entry_reward)):
        return False, np.inf
    dev = x_admissible_row_deviation(e1.P, e2.P, e1.mask, e1.live)
    return bool(dev <= X_IMPUTATION_TOL), dev


def x_theory_random_policies(mask: np.ndarray, live: np.ndarray, rng: np.random.Generator, n_dirichlet: int,
                             n_inadmissible: int, n_deterministic: int = 0) -> List[Tuple[str, np.ndarray]]:
    """Random test policies: Dirichlet(0.5) rows, rows with 80% mass on unsupported actions, random one-hot rows."""
    S, A = mask.shape
    bad = ~np.asarray(mask, dtype=bool) & live[:, None]
    has_bad = bad.any(axis=1)
    out = [(f"random Dirichlet(0.5) #{i}", rng.dirichlet(np.full(A, 0.5), size=S)) for i in range(n_dirichlet)]
    for i in range(n_inadmissible):
        base = rng.dirichlet(np.ones(A), size=S)
        on_bad = np.where(bad, base, 0.0)
        on_good = np.where(bad, 0.0, base)
        pi = base.copy()
        pi[has_bad] = (0.8 * on_bad[has_bad] / on_bad[has_bad].sum(axis=1, keepdims=True)
                       + 0.2 * on_good[has_bad] / on_good[has_bad].sum(axis=1, keepdims=True))
        out.append((f"random unsupported-heavy #{i}", pi))
    for i in range(n_deterministic):
        out.append((f"random deterministic #{i}", one_hot_policy(rng.integers(0, A, size=S), A)))
    return out


class XOptCache:
    """Memoised x_optimal(env, semantics, constrained) for one instance (the memo keeps the environments alive).

    A RuntimeError of x_optimal (VI non-convergence within its iteration cap, improper greedy policy, DP / exact gap)
    is recorded and returned as None; callers then verify through the exact occupancy LP (`x_lp_optimum`) and say so.
    Section II.3d censors trap states, so every benchmark policy is proper and this is a numerical safety net."""

    def __init__(self):
        self.memo: Dict[Tuple[int, str, bool], tuple] = {}

    def __call__(self, env: FiniteEnv, semantics: str, constrained: bool) -> Optional[tuple]:
        k = (id(env), semantics, bool(constrained))
        if k not in self.memo:
            try:
                self.memo[k] = (env, x_optimal(env, semantics, constrained=constrained), "")
            except RuntimeError as exc:
                self.memo[k] = (env, None, f"{type(exc).__name__}: {str(exc)[:160]}")
        return self.memo[k][1]

    def reason(self, env: FiniteEnv, semantics: str, constrained: bool) -> str:
        return self.memo[(id(env), semantics, bool(constrained))][2]


def x_lp_optimum(env: FiniteEnv, semantics: str, constrained: bool) -> float:
    """Optimal d0-weighted return over proper policies by the occupancy LP (kappa = 0 admissible, inf unconstrained)."""
    P = x_semantic_P(env, semantics)
    r = np.einsum("san,n->sa", P, env.entry_reward, optimize=True)
    r[~env.live] = 0.0
    cost = (~env.mask & env.live[:, None]).astype(np.float64)
    _, info = x_cmdp_lp(P, r, env.d0, env.live, cost, 0.0 if constrained else float("inf"))
    return float(info["objective"])


# ---------- per-policy quantities of P1-P4 ----------
def x_p1_policy(env: FiniteEnv, pi: np.ndarray, base: Dict[str, Any], W: Optional[np.ndarray]) -> Dict[str, float]:
    """P1 for one policy: pi versus its projection pi_W under env's given rows (base = x_evaluate(env, pi, 'given'))."""
    ev = env.evaluator()
    L = env.live
    pw = x_project(pi, env.mask, L, W)
    Pa, ra = ev.induced(pi, "mean")
    Pb, rb = ev.induced(pw, "mean")
    out = {"dP": float(np.abs(Pa[L] - Pb[L]).max()), "dr": float(np.abs(ra[L] - rb[L]).max()),
           "proj_mass": float(x_inadmissible_mass(pw, env.mask, L).max())}
    del Pa, Pb
    res = x_evaluate(env, pw, "given", vectors=True)
    out["status_mismatch"] = float(res["status"] != base["status"])
    nan = float("nan")
    out.update(dJ=nan, dL=nan, dmu=nan, dv=nan, dabs=nan)
    if res["status"] == "ok" and base["status"] == "ok":
        out.update(dJ=abs(res["return"] - base["return"]),
                   dL=abs(res["avg_episode_length"] - base["avg_episode_length"]),
                   dmu=float(np.abs(res["occupancy_vector"] - base["occupancy_vector"]).max()),
                   dv=float(np.abs(res["value_vector"] - base["value_vector"]).max()),
                   dabs=max(abs(res[k] - base[k]) for k in ("absorb_death", "absorb_survival", "absorb_other_terminal")))
    return out


def x_p2_env(env: FiniteEnv, opt: "XOptCache") -> Dict[str, Any]:
    """P2 on env's given rows: unconstrained vs admissible optimum, convexity of Q*, ties, lowest-index greedy."""
    V_unc, V_con = opt(env, "given", False)[1], opt(env, "given", True)[1]
    L, m = env.live, env.mask
    Q = env.evaluator().q_values(V_unc, "mean")[L]
    mL = m[L]
    bad = ~mL
    q_all = Q.max(axis=1)
    q_adm = np.where(mL, Q, -np.inf).max(axis=1)
    excess = float(np.where(bad, Q - q_adm[:, None], -np.inf).max()) if bad.any() else 0.0
    tied = bad & (Q >= (q_all - 1e-9 * np.maximum(1.0, np.abs(q_all)))[:, None])
    single = mL.sum(axis=1) == 1
    single_dev = (float(np.abs(Q[single] - q_adm[single, None]).max()) if (single.any() and env.A > 1)
                  else float("nan"))
    act = np.zeros(env.S, dtype=np.int64)
    act[L] = Q.argmax(axis=1)              # lowest index among exact maxima: the earlier notebook's oracle
    low = one_hot_policy(act, env.A)
    eg, et = x_evaluate(env, low, "given"), x_evaluate(env, low, "terminate")
    J_star = float(env.d0 @ V_unc)
    nan = float("nan")
    return {"dV": float(np.abs(V_unc[L] - V_con[L]).max()), "dJ": abs(float(env.d0 @ (V_unc - V_con))),
            "J_star": J_star, "convexity_excess": excess, "tied_states": int(tied.any(axis=1).sum()),
            "tied_single": int((tied.any(axis=1) & single).sum()), "single_states": int(single.sum()),
            "single_dev": single_dev, "low_inadmissible_states": int(((low[L] * bad).sum(axis=1) > 0).sum()),
            "low_status": f"{eg['status']}/{et['status']}",
            "low_gap_given": eg["return"] - J_star if eg["status"] == "ok" else nan,
            "low_gap_terminate": et["return"] - J_star if et["status"] == "ok" else nan,
            "low_violations": eg["expected_violations"] if eg["status"] == "ok" else nan}


def x_p3_policy(env: FiniteEnv, pi: np.ndarray, eg: Dict[str, Any], et: Dict[str, Any]) -> Dict[str, float]:
    """P3 for one proper policy: the given-terminate gap, its occupancy identity and the p-bounds."""
    L = env.live
    bad = ~env.mask & L[:, None]
    lo, hi = env.value_bounds
    rt = env.terminate_reward
    gap = eg["return"] - et["return"]
    p = et["prob_any_violation"]
    U = env.entry_reward + eg["value_vector"]
    Qg = np.einsum("san,n->sa", env.P, U, optimize=True)
    rhs = float(et["occupancy_vector"] @ np.where(bad, pi * (Qg - rt), 0.0).sum(axis=1))
    # independent value-form solve of P(at least one violation): (I - P_adm[L, L]) p = b
    T = np.flatnonzero(L)
    b = np.where(bad, pi, 0.0).sum(axis=1)
    P_adm = np.einsum("sa,san->sn", np.where(bad, 0.0, pi), env.P, optimize=True)[np.ix_(T, T)]
    p_ind = float(env.d0[T] @ np.linalg.solve(np.eye(len(T)) - P_adm, b[T]))
    return {"gap": gap, "p": p, "bound_excess": max(p * (lo - rt) - gap, gap - p * (hi - rt)),
            "identity": abs(gap - rhs), "p_rules": abs(eg["prob_any_violation"] - p), "p_indep": abs(p_ind - p),
            "p_term_viol": abs(et["expected_violations"] - p), "markov_excess": p - eg["expected_violations"]}


def x_p4_policy(M0: FiniteEnv, P1: np.ndarray, pi: np.ndarray, ev0: Dict[str, Any], ev1: Dict[str, Any],
                tv: np.ndarray, max_tv: float) -> Dict[str, float]:
    """P4 for one policy on the true space: simulation-lemma identity and the two TV bounds of J_M0 - J_M1."""
    bad = ~M0.mask & M0.live[:, None]
    U0 = M0.entry_reward + ev0["value_vector"]
    PU0 = np.einsum("san,n->sa", M0.P, U0, optimize=True)
    PU1 = np.einsum("san,n->sa", P1, U0, optimize=True)
    contrib = pi * (PU0 - PU1)
    d1 = ev1["occupancy_vector"]                      # zero on absorbing states
    rhs_bad = float(d1 @ np.where(bad, contrib, 0.0).sum(axis=1))
    rhs_all = float(d1 @ contrib.sum(axis=1))
    lhs = ev0["return"] - ev1["return"]
    span = float(U0.max() - U0.min())
    weighted = float(d1 @ np.where(bad, pi * tv, 0.0).sum(axis=1)) * span
    coarse = float(ev1["expected_violations"]) * max_tv * span
    return {"semantics_term": lhs, "identity": abs(lhs - rhs_bad), "admissible_part": abs(rhs_all - rhs_bad),
            "excess": abs(lhs) - coarse, "weighted_excess": abs(lhs) - weighted,
            "tightness": abs(lhs) / coarse if coarse > 0 else (0.0 if lhs == 0 else np.inf), "span": span}


# ---------- P6 and P7 closed forms ----------
def _x_masked_lse(q: np.ndarray, m: np.ndarray, T: float) -> np.ndarray:
    z = np.where(m, q / T, -np.inf)
    mx = z.max(axis=1)
    return T * (mx + np.log(np.exp(z - mx[:, None]).sum(axis=1)))


def x_p6_closed_form_checks(rng: np.random.Generator, n: int = 64, K: int = 8) -> Dict[str, float]:
    """Numpy closed-form checks of the legacy and admissible-row regularisers (no autograd)."""
    Q = 3.0 * rng.standard_normal((n, K))
    m = rng.random((n, K)) < 0.5
    m[np.arange(n), rng.integers(0, K, size=n)] = True
    a = rng.integers(0, K, size=n)
    for i in range(n // 4):                            # make sure some logged actions are excluded
        if (~m[i]).any():
            a[i] = int(np.flatnonzero(~m[i])[0])
    rows = np.arange(n)
    adm = m[rows, a]
    ex = np.flatnonzero(~adm)
    assert len(ex) and adm.any(), "P6 test batch needs excluded and admissible logged actions"
    out = {"legacy_fd_dev": 0.0, "legacy_untouched_rows_dev": 0.0, "legacy_mean_derivative_dev": 0.0,
           "design_b_min": np.inf, "design_b_excluded_invariance": 0.0, "batch": n, "excluded_rows": int(len(ex))}
    for T in (0.5, 1.0, 2.0):
        def legacy(q):
            return _x_masked_lse(q, m, T) - q[rows, a]
        base = legacy(Q)
        h = 2.5
        Q2 = Q.copy()
        Q2[ex, a[ex]] += h
        diff = legacy(Q2) - base
        out["legacy_fd_dev"] = max(out["legacy_fd_dev"], float(np.abs(diff[ex] + h).max()))
        out["legacy_untouched_rows_dev"] = max(out["legacy_untouched_rows_dev"], float(np.abs(diff[adm]).max()))
        i, e = int(ex[0]), 0.5                         # the legacy loss is linear in that coordinate
        Qp, Qm = Q.copy(), Q.copy()
        Qp[i, a[i]] += e
        Qm[i, a[i]] -= e
        deriv = (legacy(Qp).mean() - legacy(Qm).mean()) / (2 * e)
        out["legacy_mean_derivative_dev"] = max(out["legacy_mean_derivative_dev"], abs(deriv + 1.0 / n))
        db = np.where(adm, base, 0.0)
        out["design_b_min"] = min(out["design_b_min"], float(db.min()))
        Q3 = Q + np.where(~m, 5.0 * rng.standard_normal((n, K)), 0.0)   # perturb every excluded coordinate
        out["design_b_excluded_invariance"] = max(out["design_b_excluded_invariance"],
                                                  float(np.abs(np.where(adm, legacy(Q3), 0.0) - db).max()))
    return out


def x_reverse_marginal(kernels: Dict[int, np.ndarray], p_T: np.ndarray) -> np.ndarray:
    """Propagate per-state distributions [S, K] through B_T, ..., B_1 (kernels[t][s, x_t, x_{t-1}]); no mask."""
    p = np.asarray(p_T, dtype=np.float64)
    for t in sorted(kernels, reverse=True):
        p = np.einsum("sk,skj->sj", p, np.asarray(kernels[t], dtype=np.float64), optimize=True)
    return p


def x_p7_state_checks(pi_b: np.ndarray, diffusion: "CategoricalDiffusion", kernels: Dict[int, np.ndarray],
                      live: np.ndarray) -> Dict[str, Any]:
    """P7 (a)-(c) per live state for the Bayes-optimal reverse chain of the uniform-corruption process."""
    pi_b = np.asarray(pi_b, dtype=np.float64)
    S, K = pi_b.shape
    T = diffusion.T
    assert K == diffusion.K and sorted(kernels) == list(range(1, T + 1)), "kernels must map t = 1..T to [S, K, K]"
    rowsum_dev, kmin = 0.0, np.inf
    for t in range(1, T + 1):
        k = np.asarray(kernels[t], dtype=np.float64)
        assert k.shape == (S, K, K), f"kernel {t}: shape {k.shape} != {(S, K, K)}"
        rowsum_dev = max(rowsum_dev, float(np.abs(k.sum(axis=2) - 1.0).max()))
        kmin = min(kmin, float(k.min()))
    abar = float(diffusion.alpha_bar[-1])
    u = np.full((S, K), 1.0 / K)
    qT = pi_b @ diffusion.Qbar[T]
    out_q = x_reverse_marginal(kernels, qT)
    out_u = x_reverse_marginal(kernels, u)
    tv_q, tv_u = x_tv(out_q, pi_b)[live], x_tv(out_u, pi_b)[live]
    tv_uq, tv_ub = x_tv(u, qT)[live], x_tv(u, pi_b)[live]
    pos = tv_ub > 1e-12
    ratio = tv_u[pos] / tv_ub[pos]
    return {"abar": abar, "K": K, "kernel_rowsum_dev": rowsum_dev, "kernel_min": kmin,
            "qT_closed_dev": float(np.abs(qT - (abar * pi_b + (1.0 - abar) * u))[live].max()),
            "tv_from_qT": float(tv_q.max()), "tv_identity_dev": float(np.abs(tv_uq - abar * tv_ub).max()),
            "contraction_excess": float((tv_u - tv_uq).max()), "tv_from_u_max": float(tv_u.max()),
            "tv_from_u_mean": float(tv_u.mean()), "global_bound": abar * (1.0 - 1.0 / K),
            "ratio_max": float(ratio.max()) if len(ratio) else float("nan"),
            "ratio_median": float(np.median(ratio)) if len(ratio) else float("nan"), "out_u": out_u}


# ---------- random absorbing MDPs ----------
def x_tiny_absorbing_mdp(i: int, S: int = 12, A: int = 4) -> Tuple[FiniteEnv, np.ndarray]:
    """Random absorbing MDP (S-3 live states; death / goal / timeout absorbing) with a random mask and random W."""
    rng = np_rng("xenv", "theory", "tiny-mdp", int(i), int(S), int(A))
    nL = S - 3
    death, goal, timeout = nL, nL + 1, nL + 2
    R = np.zeros(S)
    R[death], R[goal], R[timeout] = rng.uniform(-1.0, 0.0), rng.uniform(0.5, 1.0), rng.uniform(-0.5, 0.5)
    P = np.zeros((S, A, S))
    for s in range(nL):
        for a in range(A):
            h = rng.uniform(0.05, 0.3)
            succ = rng.choice(nL, size=3, replace=False)
            P[s, a, succ] = (1.0 - h) * rng.dirichlet(np.ones(3))
            P[s, a, [death, goal, timeout]] = h * rng.dirichlet(np.ones(3))
    for t in (death, goal, timeout):
        P[t, :, t] = 1.0
    mask = np.ones((S, A), dtype=bool)
    for s in range(nL):
        k = 1 if s == 0 else (A if s == 1 else int(rng.integers(1, A + 1)))
        mask[s] = False
        mask[s, rng.choice(A, size=k, replace=False)] = True
    d0 = np.zeros(S)
    start = rng.choice(nL, size=4, replace=False)
    d0[start] = rng.dirichlet(np.ones(4))
    W = np.zeros((S, A, A))
    for s in range(nL):
        adm = np.flatnonzero(mask[s])
        for a in np.flatnonzero(~mask[s]):
            W[s, a, adm] = rng.dirichlet(np.full(len(adm), 0.7))
    env = FiniteEnv(name=f"tiny-{i:02d}", kind="variant", P=P, entry_reward=R, d0=d0,
                    terminals=(death, goal, timeout), sink=death, goal=goal, mask=mask, meta={"tiny_mdp": int(i)})
    return env, W


# ---------- exact constrained frontier ----------
def x_exact_frontier(env: FiniteEnv, semantics: str, kappas: Sequence[float], opt: "XOptCache", meta: Dict[str, Any],
                     W: Optional[np.ndarray] = None) -> pd.DataFrame:
    """CMDP-LP frontier of env under a semantics, cost 1[a not in M(s)] on live states, with VI / exact cross-checks."""
    P = x_semantic_P(env, semantics)
    r = np.einsum("san,n->sa", P, env.entry_reward, optimize=True)
    r[~env.live] = 0.0
    cost = (~env.mask & env.live[:, None]).astype(np.float64)
    nan = float("nan")
    con, unc = opt(env, semantics, True), opt(env, semantics, False)
    vi_status = ("ok" if (con is not None and unc is not None) else
                 "; ".join(x for x in (opt.reason(env, semantics, True), opt.reason(env, semantics, False)) if x))
    J_con = float(env.d0 @ con[1]) if con is not None else nan
    J_unc = float(env.d0 @ unc[1]) if unc is not None else nan
    if con is not None:
        fallback = con[0]
    else:   # proper admissible policy from every live state (full-support weights), as the driver's reference LP
        try:
            fallback, _ = x_cmdp_lp(P, r, 0.5 * env.d0 + 0.5 * env.live / env.live.sum(), env.live, cost, 0.0)
        except RuntimeError:
            fallback = None
    lo, _ = env.value_bounds
    if semantics == "given" and x_imputation_deviation(env, W) <= X_IMPUTATION_TOL:
        flat, reason = True, "P2: given rows are a convex imputation"
    elif semantics == "terminate" and env.terminate_reward == lo:
        flat, reason = True, "P3 corollary: r_term = v_min"
    else:
        flat, reason = False, ""
    ks = sorted(set(float(k) for k in kappas) | {0.0, float("inf")})
    rows = []
    for k in ks:
        try:
            pi, info = x_cmdp_lp(P, r, env.d0, env.live, cost, k, fallback=fallback)
        except RuntimeError as exc:
            info = {"objective": nan, "violations": nan, "expected_decisions": nan, "reachable_live_states": -1,
                    "randomised_states": -1, "solve_s": nan}
            ev = {"status": f"lp_failed: {str(exc)[:120]}"}
        else:
            ev = x_evaluate(env, pi, semantics)
        ok = ev["status"] == "ok"
        row = {**meta, "env": env.name, "env_kind": env.kind, "semantics": semantics, "kappa": k,
               "objective": info["objective"], "exact_return": ev["return"] if ok else nan,
               "lp_violations": info["violations"],
               "exact_violations": ev["expected_violations"] if ok else nan,
               "prob_any_violation": ev["prob_any_violation"] if ok else nan,
               "avg_episode_length": ev["avg_episode_length"] if ok else nan,
               "expected_decisions": info["expected_decisions"], "reachable_live_states": info["reachable_live_states"],
               "randomised_states": info["randomised_states"], "J_opt_admissible": J_con, "J_opt_unconstrained": J_unc,
               "value_of_unsupported_actions": J_unc - J_con, "flat_expected": flat, "flat_reason": reason,
               "eval_status": ev["status"], "vi_status": vi_status, "solve_s": info["solve_s"]}
        row["gap_eval"] = abs(row["exact_return"] - row["objective"]) if ok else nan
        row["feasibility_excess"] = (row["exact_violations"] - k) if (ok and np.isfinite(k)) else nan
        row["gap_vi"] = (abs(row["objective"] - J_con) if k == 0.0 else
                         abs(row["objective"] - J_unc) if not np.isfinite(k) else nan)   # NaN if VI is unavailable
        rows.append(row)
    obj = np.array([r_["objective"] for r_ in rows])
    mono = float(max(0.0, np.max(obj[:-1] - obj[1:])))
    conc = 0.0
    for j in range(1, len(ks) - 2):                  # consecutive finite triples (the last kappa is inf)
        lam = (ks[j + 1] - ks[j]) / (ks[j + 1] - ks[j - 1])
        conc = max(conc, float(lam * obj[j - 1] + (1 - lam) * obj[j + 1] - obj[j]))
    if not np.isfinite(obj).all():                   # a failed LP: the shape checks are undefined (and fail)
        mono = conc = nan
    rng_ = float(np.max(obj) - obj[0])
    for row in rows:
        row["frontier_range"] = rng_
        row["monotone_concave_violation"] = max(mono, conc)
        checks = [row["eval_status"] == "ok", bool(row["gap_eval"] <= X_LP_TOL),
                  bool(row["monotone_concave_violation"] <= X_LP_TOL)]
        if np.isfinite(row["feasibility_excess"]):
            checks.append(row["feasibility_excess"] <= X_LP_TOL)
        if np.isfinite(row["gap_vi"]):
            checks.append(row["gap_vi"] <= X_LP_TOL)
        row["passes"] = bool(all(checks))
    return pd.DataFrame(rows).reindex(columns=X_FRONTIER_COLUMNS)


def x_frontier_checks(log: XPropositionLog, fr: pd.DataFrame, instance: str) -> None:
    """LP cross-check rows (and the P2 / P3 flat-frontier findings) of one frontier for proposition_checks.csv."""
    env, sem = str(fr["env"].iloc[0]), str(fr["semantics"].iloc[0])
    name = f"{env} [{fr['target'].iloc[0]}, {sem}]"
    n_bad = int((fr["eval_status"] != "ok").sum())
    log.check("LP", name, instance, "LP policies proper (count improper)", n_bad, 0.0)
    log.check("LP", name, instance, "max |exact return of LP policy - LP objective|", fr["gap_eval"].max(), X_LP_TOL)
    vi = str(fr["vi_status"].iloc[0])
    for txt, sel in (("kappa = 0: |LP objective - admissible VI optimum|", fr["kappa"] == 0.0),
                     ("kappa = inf: |LP objective - unconstrained VI optimum|", ~np.isfinite(fr["kappa"]))):
        if vi == "ok":
            log.check("LP", name, instance, txt, fr.loc[sel, "gap_vi"].iloc[0], X_LP_TOL)
        else:   # the LP itself is still verified by the exact evaluation of its policy (rows above / below)
            log.not_run("LP", name, instance, txt, f"value iteration unavailable ({vi})")
    fe = fr["feasibility_excess"].dropna()
    log.check("LP", name, instance, "max (exact expected violations - kappa)", fe.max() if len(fe) else 0.0, X_LP_TOL)
    log.check("LP", name, instance, "frontier non-decreasing and concave in kappa (max violation)",
              fr["monotone_concave_violation"].iloc[0], X_LP_TOL)
    flat = bool(fr["flat_expected"].iloc[0])
    if flat:
        prop = "P2" if sem == "given" else "P3"
        log.check(prop, name, instance, "frontier is flat (max_kappa J*(kappa) - J*(0))", fr["frontier_range"].iloc[0],
                  X_LP_TOL, notes=f"finding, not an error: {fr['flat_reason'].iloc[0]}")
    else:
        log.report("LP", name, instance, "frontier range J*(inf) - J*(0) (value of unsupported actions)",
                   fr["frontier_range"].iloc[0])


def x_find_record(records: Sequence[dict], method: str, variant: str) -> Optional[dict]:
    """The default-hyper-parameter record of (method, variant), or None if run_xmethods did not return it."""
    hits = [r for r in records if r["method"] == method and r["variant"] == variant and bool(r["is_default"])]
    assert len(hits) <= 1, f"several default records for {method} [{variant}]"
    return hits[0] if hits else None

In [ ]:
# ============================================================
# II.5 Propositions: per-instance verification
# ============================================================
def _x_p1_block(log: XPropositionLog, e: dict, inst: str, W: Optional[np.ndarray], note: str) -> None:
    env, en = e["env"], e["label"]
    if e["imputed"]:
        res = [x_p1_policy(env, p, g, W) for p, g in zip(e["pis"], e["g"])]
        n_imp = sum(g["status"] != "ok" for g in e["g"])
        nt = f"{note}; improper under given: {n_imp}"
        for key, txt in (("dJ", "max |J(pi) - J(pi_W)|"), ("dL", "max |L(pi) - L(pi_W)|"),
                         ("dmu", "max ||mu_pi - mu_pi_W||_inf (state occupancy)"),
                         ("dv", "max ||v_pi - v_pi_W||_inf"), ("dabs", "max |absorption probability difference|"),
                         ("dP", "max |P_pi - P_pi_W| on live rows"), ("dr", "max |r_pi - r_pi_W| on live rows")):
            vals = [r[key] for r in res if np.isfinite(r[key])]
            if vals:
                log.check("P1", en, inst, txt, max(vals), 1e-9, notes=nt)
            else:
                log.not_run("P1", en, inst, txt, "no policy proper under the given rows")
        log.report("P1", en, inst, "max inadmissible mass of pi_W", max(r["proj_mass"] for r in res),
                   notes=f"{X_INVARIANT}: x_project asserts zero mass outside the mask; {nt}")
        log.check("P1", en, inst, "properness of pi and pi_W disagrees (count)",
                  sum(r["status_mismatch"] for r in res), 0.0, notes=nt)
        if W is not None:   # negative control: the uniform projection is the wrong W for this environment
            ctrl = [x_p1_policy(env, p, g, None)["dJ"] for p, g in zip(e["pis"], e["g"])]
            ctrl = [v for v in ctrl if np.isfinite(v)]
            log.check("P1", en, inst, "negative control: projecting with the wrong (uniform) W changes J (max |dJ|)",
                      max(ctrl) if ctrl else np.nan, 1e-6, op="ge", notes="the check has power")
    else:
        res = [x_p1_policy(env, p, g, None) for p, g in zip(e["pis"], e["g"])]
        vals = [r["dJ"] for r in res if np.isfinite(r["dJ"])]
        log.report("P1", en, inst, "rows are not an imputation: max |J(pi) - J(pi_W)| (projection not value-preserving)",
                   max(vals) if vals else np.nan, notes=note)


def _x_p2_block(log: XPropositionLog, e: dict, inst: str, opt: "XOptCache") -> None:
    env, en = e["env"], e["label"]
    if opt(env, "given", False) is None or opt(env, "given", True) is None:
        why = opt.reason(env, "given", False) or opt.reason(env, "given", True)
        gap = x_lp_optimum(env, "given", False) - x_lp_optimum(env, "given", True)
        nt = f"value iteration unavailable ({why}); exact occupancy-LP route"
        if e["imputed"]:
            log.check("P2", en, inst, "LP route: |J*_unconstrained - J*_admissible|", abs(gap), X_LP_TOL, notes=nt)
        else:
            log.check("P2", en, inst, "LP route: J*_unconstrained - J*_admissible >= 0 (non-imputed rows)", gap,
                      -X_LP_TOL, op="ge", notes=nt)
            log.report("P2", en, inst, "value of unsupported actions J*_unconstrained - J*_admissible", gap, notes=nt)
        return
    if not e["imputed"]:
        V_unc, V_con = opt(env, "given", False)[1], opt(env, "given", True)[1]
        gap = float(env.d0 @ (V_unc - V_con))
        log.check("P2", en, inst, "J*_unconstrained - J*_admissible >= 0 (non-imputed rows)", gap, -1e-9, op="ge")
        log.report("P2", en, inst, "value of unsupported actions J*_unconstrained - J*_admissible", gap)
        return
    r = x_p2_env(env, opt)
    log.check("P2", en, inst, "max |V*_unconstrained - V*_admissible| (live states)", r["dV"], 1e-9)
    log.check("P2", en, inst, "|J*_unconstrained - J*_admissible|", r["dJ"], 1e-9)
    log.check("P2", en, inst, "convexity: max_{a not in M} Q*(s,a) - max_{a' in M} Q*(s,a')", r["convexity_excess"], 1e-9)
    if np.isfinite(r["single_dev"]):
        log.check("P2", en, inst, "single-admissible states: max |Q*(s,a) - Q*(s,a_M)| (exact ties)", r["single_dev"],
                  1e-9, notes=f"{r['single_states']} single-admissible live states")
    else:
        log.not_run("P2", en, inst, "single-admissible states: exact ties", "no single-admissible live state")
    log.report("P2", en, inst, "live states where an unsupported action ties the optimum", r["tied_states"],
               notes=f"{r['tied_single']} of them single-admissible; relative tolerance 1e-9")
    log.report("P2", en, inst, "lowest-index greedy (earlier oracle): live states choosing an unsupported action",
               r["low_inadmissible_states"], notes=f"evaluation status given/terminate {r['low_status']}")
    log.report("P2", en, inst, "lowest-index greedy: J_given - J*", r["low_gap_given"], notes="0 when it is proper")
    log.report("P2", en, inst, "lowest-index greedy: J_terminate - J*", r["low_gap_terminate"],
               notes=f"expected violations under given {r['low_violations']:.4g}: the oracle artefact")


def _x_p3_block(log: XPropositionLog, e: dict, inst: str, opt: "XOptCache", note: str) -> None:
    env, en = e["env"], e["label"]
    res = [x_p3_policy(env, p, g, t) for p, g, t, ok in zip(e["pis"], e["g"], e["t"], e["ok"]) if ok]
    nt = f"{note}; proper under both semantics: {len(res)}/{len(e['pis'])}"
    if not res:
        log.not_run("P3", en, inst, "termination gap", "no policy proper under both semantics")
    else:
        lo, hi = env.value_bounds
        rt = env.terminate_reward
        log.check("P3", en, inst, "p(v_min - r_term) <= J_given - J_term <= p(v_max - r_term): max excess",
                  max(r["bound_excess"] for r in res), 1e-10, notes=f"{nt}; v_min={lo:g}, v_max={hi:g}, r_term={rt:g}")
        log.check("P3", en, inst, "identity: |gap - sum_s mu_term(s) sum_{a not in M} pi (Q_given - r_term)|",
                  max(r["identity"] for r in res), 1e-9, notes=nt)
        log.report("P3", en, inst, "p identical under both semantics", max(r["p_rules"] for r in res),
                   notes=f"{X_INVARIANT}: both evaluations read p from the terminate chain; {nt}")
        log.check("P3", en, inst, "p equals an independent first-violation value solve", max(r["p_indep"] for r in res),
                  1e-9, notes=nt)
        log.report("P3", en, inst, "p = E_term[violations]", max(r["p_term_viol"] for r in res),
                   notes=f"{X_INVARIANT}: under termination the evaluator sets p_any = expected violations; {nt}")
        log.check("P3", en, inst, "Markov: p - E_given[violations] <= 0", max(r["markov_excess"] for r in res), 1e-12,
                  notes=nt)
        log.report("P3", en, inst, "max |J_given - J_term| over policies", max(abs(r["gap"]) for r in res), notes=nt)
        log.report("P3", en, inst, "max p over policies", max(r["p"] for r in res), notes=nt)
    lo, _ = env.value_bounds
    if env.terminate_reward == lo:
        ot, oc = opt(env, "terminate", False), opt(env, "given", True)
        txt = "corollary (r_term = v_min): max |V*_terminate,unconstrained - V*_admissible|"
        if ot is None or oc is None:
            why = opt.reason(env, "terminate", False) or opt.reason(env, "given", True)
            log.check("P3", en, inst, "corollary (r_term = v_min), LP route: |J*_terminate,unconstrained - J*_admissible|",
                      abs(x_lp_optimum(env, "terminate", False) - x_lp_optimum(env, "given", True)), X_LP_TOL,
                      notes=f"value iteration unavailable ({why})")
        else:
            log.check("P3", en, inst, txt, float(np.abs(ot[1] - oc[1])[env.live].max()), 1e-9)
    else:
        log.report("P3", en, inst, "r_term - v_min (> 0: flat-terminate corollary not claimed)",
                   env.terminate_reward - lo)


def _x_p5_block(log: XPropositionLog, entries: List[dict], learn_env: FiniteEnv,
                policies: Sequence[Tuple[str, np.ndarray, str]], inst: str) -> None:
    proj = [x_project(pi, learn_env.mask, learn_env.live) for _, pi, _ in policies]
    for e in entries:
        env = e["env"]
        e["adm"] = [validate_policy(e["lift"](p) if e["lift"] is not None else p, f"projected policy on {env.name}",
                                    shape=(env.S, env.A)) for p in proj]
        log.check("P5", e["label"], inst, "lifted projected policies: max inadmissible mass (precondition)",
                  max(float(x_inadmissible_mass(p, env.mask, env.live).max()) for p in e["adm"]), 0.0)
    groups: List[List[Tuple[dict, str]]] = []
    for e in entries:
        for sem in X_SEMANTICS:
            for g in groups:
                if x_shares_admissible_rows(g[0][0]["env"], e["env"])[0]:
                    g.append((e, sem))
                    break
            else:
                groups.append([(e, sem)])
    for g in groups:
        if len(g) < 2:
            continue
        e0, s0 = g[0]
        name = " = ".join(f"{e['env'].name}:{s}" for e, s in g)
        ref = [x_evaluate(e0["env"], p, s0, vectors=True) for p in e0["adm"]]
        d = {"pol": 0.0, "J": 0.0, "L": 0.0, "mu": 0.0, "abs": 0.0, "mismatch": 0, "improper": 0}
        for e, s in g[1:]:
            for p0, p, r0 in zip(e0["adm"], e["adm"], ref):
                d["pol"] = max(d["pol"], float(np.abs(p - p0).max()))
                r = x_evaluate(e["env"], p, s, vectors=True)
                if r["status"] != r0["status"]:
                    d["mismatch"] += 1
                    continue
                if r["status"] != "ok":
                    d["improper"] += 1
                    continue
                d["J"] = max(d["J"], abs(r["return"] - r0["return"]))
                d["L"] = max(d["L"], abs(r["avg_episode_length"] - r0["avg_episode_length"]))
                d["mu"] = max(d["mu"], float(np.abs(r["occupancy_vector"] - r0["occupancy_vector"]).max()))
                d["abs"] = max(d["abs"], max(abs(r[k] - r0[k]) for k in
                                             ("absorb_death", "absorb_survival", "absorb_other_terminal")))
        nt = f"{len(g)} targets; {len(proj)} admissible policies; jointly improper pairs {d['improper']}"
        log.check("P5", name, inst, "same admissible policy on every target (precondition)", d["pol"], 0.0, notes=nt)
        log.check("P5", name, inst, "max |dJ| across targets", d["J"], 1e-10, notes=nt)
        log.check("P5", name, inst, "max |dL| across targets", d["L"], 1e-10, notes=nt)
        log.check("P5", name, inst, "max ||d mu||_inf across targets", d["mu"], 1e-10, notes=nt)
        log.check("P5", name, inst, "max |d absorption probability| across targets", d["abs"], 1e-10, notes=nt)
        log.check("P5", name, inst, "properness disagrees across targets (count)", d["mismatch"], 0.0, notes=nt)


def _x_p4_block(log: XPropositionLog, entries: List[dict], tg: Dict[str, tuple], inst: str,
                completions: Sequence[Tuple[str, str, str]]) -> None:
    def entry(env):
        return next(x for x in entries if x["env"] is env)

    M0, s0, _ = tg["true"]
    assert s0 == "given", "the 'true' target must use the true dynamics ('given')"
    e0 = entry(M0)
    for comp, m1_lab, b_lab in completions:
        M1, s1, _ = tg[m1_lab]
        Bn, sb, _ = tg[b_lab]
        e1, eb = entry(M1), entry(Bn)
        en = f"{M0.name} vs {M1.name}:{s1} [{comp} completion]"
        same = (M1.P.shape == M0.P.shape and M1.terminals == M0.terminals and M1.sink == M0.sink
                and np.array_equal(M1.mask, M0.mask) and np.array_equal(M1.d0, M0.d0)
                and np.array_equal(M1.entry_reward, M0.entry_reward))
        if not log.check("P4", en, inst, "M1 shares state space, mask, d0, rewards and sink with M0 (precondition)",
                         0.0 if same else 1.0, 0.0):
            continue
        P1 = x_semantic_P(M1, s1)
        log.check("P4", en, inst, "admissible live rows of M0 and M1 coincide (precondition)",
                  x_admissible_row_deviation(M0.P, P1, M0.mask, M0.live), X_IMPUTATION_TOL)
        log.check("P4", en, inst, "absorbing rows of M0 and M1 coincide (precondition)",
                  max(float(np.abs(M0.P[t] - P1[t]).max()) for t in M0.terminals), 0.0)
        log.check("P4", en, inst, "same lifted policy under M0 and M1 (precondition)",
                  max(float(np.abs(a - b).max()) for a, b in zip(e0["pis"], e1["pis"])), 0.0)
        tv = x_inadmissible_tv(M0.P, P1, M0.mask, M0.live)
        max_tv = float(tv.max())
        k1, kb = ("g" if s1 == "given" else "t"), ("g" if sb == "given" else "t")
        res = []
        for i, pi in enumerate(e0["pis"]):
            ev0, ev1, evb = e0["g"][i], e1[k1][i], eb[kb][i]
            if not (ev0["status"] == ev1["status"] == evb["status"] == "ok"):
                continue
            r = x_p4_policy(M0, P1, pi, ev0, ev1, tv, max_tv)
            r["estimation_term"] = ev1["return"] - evb["return"]
            r["total"] = ev0["return"] - evb["return"]
            res.append(r)
        del P1
        nt = f"proper policies {len(res)}/{len(e0['pis'])}; benchmark target '{b_lab}' ({Bn.name}:{sb})"
        if not res:
            log.not_run("P4", en, inst, "simulation-lemma identity", "no policy proper on M0, M1 and the benchmark")
            continue
        log.check("P4", en, inst, "identity |dJ - sum_s mu1(s) sum_{a not in M} pi (P0 - P1)^T U0|",
                  max(r["identity"] for r in res), 1e-9, notes=nt)
        log.check("P4", en, inst, "admissible pairs contribute nothing |sum_all - sum_inadmissible|",
                  max(r["admissible_part"] for r in res), 1e-12, notes=nt)
        log.check("P4", en, inst, "bound: |dJ| - E_M1[viol] max TV span(U0) (max excess)",
                  max(r["excess"] for r in res), 1e-12, notes=nt)
        log.check("P4", en, inst, "weighted bound: |dJ| - sum_s mu1 sum_{a not in M} pi TV span(U0) (max excess)",
                  max(r["weighted_excess"] for r in res), 1e-12, notes=nt)
        log.report("P4", en, inst, "max TV(P0, P1) over live unsupported pairs", max_tv)
        log.report("P4", en, inst, "max tightness |dJ| / (E_M1[viol] max TV span(U0))",
                   max(r["tightness"] for r in res), notes=nt)
        log.report("P4", en, inst, "mean |semantics term J_M0 - J_M1|", np.mean([abs(r["semantics_term"]) for r in res]),
                   notes=nt)
        log.report("P4", en, inst, "mean |estimation + coverage term J_M1 - J_bench|",
                   np.mean([abs(r["estimation_term"]) for r in res]), notes=nt)
        log.report("P4", en, inst, "mean |J_true - J_bench|", np.mean([abs(r["total"]) for r in res]), notes=nt)
        if M1 is Bn and s1 == sb:
            log.check("P4", en, inst, "benchmark = true rows + completion: estimation term (max |J_M1 - J_bench|)",
                      max(abs(r["estimation_term"]) for r in res), 0.0, notes=nt)


def x_theory_checks(log: XPropositionLog, inst: str, learn_env: FiniteEnv,
                    targets: Sequence[Tuple[str, FiniteEnv, str, Optional[Callable]]],
                    policies: Sequence[Tuple[str, np.ndarray, str]], opt: "XOptCache", construction: Optional[str] = None,
                    learn_env_imputed: bool = True, env_W: Optional[Dict[int, np.ndarray]] = None,
                    completions: Optional[Sequence[Tuple[str, str, str]]] = None) -> Dict[str, Any]:
    """P1-P5 on one instance: every distinct target environment x every policy (lifted where the target needs it)."""
    env_W = {} if env_W is None else env_W
    assert any(env is learn_env for _, env, _, _ in targets), "the learner's environment must be one of the targets"
    tg = {lab: (env, sem, lift) for lab, env, sem, lift in targets}
    assert len(tg) == len(targets), "duplicate target labels"
    entries: List[dict] = []
    for lab, env, sem, lift in targets:
        e = next((x for x in entries if x["env"] is env), None)
        if e is None:
            entries.append({"env": env, "lift": lift, "labels": [lab]})
        else:
            assert (e["lift"] is None) == (lift is None), f"{lab}: inconsistent lift for {env.name}"
            e["labels"].append(lab)
    kinds = {k: sum(1 for _, _, kk in policies if kk == k) for k in ("learned", "reference", "random")}
    note = "policies " + ", ".join(f"{k} {v}" for k, v in kinds.items())
    for e in entries:
        env, lift = e["env"], e["lift"]
        e["label"] = f"{env.name} [{', '.join(e['labels'])}]"
        e["pis"] = [validate_policy(lift(pi) if lift is not None else pi, f"{name} on {env.name}", shape=(env.S, env.A))
                    for name, pi, _ in policies]
        e["g"] = [x_evaluate(env, p, "given", vectors=True) for p in e["pis"]]
        e["t"] = [x_evaluate(env, p, "terminate", vectors=True) for p in e["pis"]]
        e["ok"] = [g["status"] == "ok" and t["status"] == "ok" for g, t in zip(e["g"], e["t"])]
        W = env_W.get(id(env))
        wname = "random W" if W is not None else "uniform W (mean rule)"
        dev = x_imputation_deviation(env, W)
        required = ((env is learn_env and learn_env_imputed) or "true_rows_mean" in e["labels"]
                    or id(env) in env_W)
        if required:
            e["imputed"] = log.check("P1", e["label"], inst, f"given rows equal the convex imputation [{wname}] "
                                     "(precondition)", dev, X_IMPUTATION_TOL)
        else:
            e["imputed"] = bool(dev <= X_IMPUTATION_TOL)
            log.report("P1", e["label"], inst, f"deviation of the given rows from the convex imputation [{wname}]",
                       dev, notes="P1 / P2 are claimed only for imputed rows")
        _x_p1_block(log, e, inst, W, note)
        _x_p2_block(log, e, inst, opt)
        _x_p3_block(log, e, inst, opt, note)
    _x_p5_block(log, entries, learn_env, policies, inst)
    if "true" in tg:
        if completions is None:
            completions = ((("mean", "bench_mean", "bench_mean"), ("terminate", "bench_terminate", "bench_terminate"))
                           if construction == "oracle" else
                           (("mean", "true_rows_mean", "bench_mean"), ("terminate", "true_rows_terminate", "bench_terminate")))
        _x_p4_block(log, entries, tg, inst, completions)
    else:
        log.not_run("P4", learn_env.name, inst, "exact error decomposition", "no ground-truth target (ICU-Sepsis)")
    n_ok = sum(sum(e["ok"]) for e in entries)
    return {"targets": [e["label"] for e in entries], "policies": len(policies), **kinds,
            "proper_pairs": int(n_ok), "env_policy_pairs": int(len(entries) * len(policies))}


def x_theory_p7(log: XPropositionLog, env_name: str, inst: str, pi_b: np.ndarray, live: np.ndarray,
                records: Optional[Sequence[dict]] = None) -> Dict[str, float]:
    """P7 per state for one behaviour estimate, plus cross-checks against the run_xmethods diffusion records."""
    K = pi_b.shape[1]
    diffusion = x_diffusion_for(K)
    kernels = x_bayes_reverse_kernels(pi_b, diffusion)
    r = x_p7_state_checks(pi_b, diffusion, kernels, live)
    del kernels
    nt = f"K={K}, T={diffusion.T}, alpha_bar_T={r['abar']:.6g}, {int(live.sum())} live states"
    log.check("P7", env_name, inst, "Bayes kernels are Markov: max |row sum - 1|", r["kernel_rowsum_dev"], 1e-10, notes=nt)
    log.report("P7", env_name, inst, "Bayes kernels are non-negative: min entry", r["kernel_min"],
               notes=f"{X_INVARIANT}: x_bayes_reverse_kernels clips at 0; {nt}")
    log.check("P7", env_name, inst, "q_T closed form: max |pi_b Qbar_T - (abar pi_b + (1 - abar) u)|", r["qT_closed_dev"],
              1e-12, notes=nt)
    log.check("P7", env_name, inst, "(a) reverse chain from q_T returns pi_b: max_s TV", r["tv_from_qT"], 1e-10, notes=nt)
    log.check("P7", env_name, inst, "(c) TV(u, q_T) = abar TV(u, pi_b): max abs difference", r["tv_identity_dev"], 1e-12,
              notes=nt)
    log.check("P7", env_name, inst, "(b) from uniform: max_s [TV(out, pi_b) - TV(u, q_T)]", r["contraction_excess"],
              1e-12, notes=nt)
    log.check("P7", env_name, inst, "(c) from uniform: max_s TV(out, pi_b) <= abar (1 - 1/K)", r["tv_from_u_max"],
              r["global_bound"] + 1e-12, notes=nt)
    log.report("P7", env_name, inst, "mean_s TV(out from uniform, pi_b)", r["tv_from_u_mean"], notes=nt)
    log.report("P7", env_name, inst, "median_s TV(out, pi_b) / TV(u, pi_b) (bound: alpha_bar_T)", r["ratio_median"],
               notes=f"max ratio {r['ratio_max']:.4g}")
    if records is not None:
        rec = x_find_record(records, "Diffusion (exact, unmasked)", "native")
        if rec is None:
            log.not_run("P7", env_name, inst, "'Diffusion (exact, unmasked)' cross-check", "record not returned")
        else:
            dev = float(np.abs(np.asarray(rec["pi"])[live] - r["out_u"][live]).max())
            log.check("P7", env_name, inst, "'Diffusion (exact, unmasked)' policy = uniform-start reverse marginal "
                      "(max abs, live)", dev, 1e-9, notes="same pi_b (training split, global fallback for unseen states)")
        fin = x_find_record(records, "Diffusion (exact, final-only)", "native")
        sfm = x_find_record(records, "SfBC (exact)", "masked")
        sfa = x_find_record(records, "SfBC (exact, argmax)", "native")
        # The deployed diffusion records are masked, deterministic (argmax) policies, so they are compared by argmax
        # with masked SfBC (masked against masked); a TV against a stochastic policy would be large even when both
        # pick the same action.
        if fin is not None and sfm is not None:
            agree = np.asarray(fin["pi"])[live].argmax(axis=1) == np.asarray(sfm["pi"])[live].argmax(axis=1)
            log.report("P7", env_name, inst, "argmax agreement: final-only guided diffusion vs argmax of SfBC (exact) "
                       "[masked]", float(agree.mean()), notes="fraction of live states; both masked; default hyper-parameters")
        else:
            log.not_run("P7", env_name, inst, "argmax agreement: final-only guided vs SfBC (exact) [masked]",
                        "record(s) not returned")
        if fin is not None and sfa is not None:
            agree = np.asarray(fin["pi"])[live].argmax(axis=1) == np.asarray(sfa["pi"])[live].argmax(axis=1)
            log.report("P7", env_name, inst, "argmax agreement: final-only guided (masked) vs SfBC (exact, argmax) "
                       "(unmasked)", float(agree.mean()), notes="fraction of live states; SfBC unmasked")
    r.pop("out_u")
    return r


def x_theory_p6(log: XPropositionLog) -> None:
    """P6: summary of Section 13's torch gradient tests plus closed-form numpy checks."""
    rg = globals().get("REG_GRAD_TESTS")
    if rg is None:
        log.not_run("P6", "Part I Section 13 (torch)", "REG_GRAD_TESTS", "regulariser gradient tests",
                    "REG_GRAD_TESTS missing")
    else:
        B = int(rg["batch_rows"])
        nt = f"batch {B}, admissible logged rows {rg.get('admissible_logged_rows')}"
        if "legacy_excluded_logged_action_derivative" in rg:
            log.check("P6", "Part I Section 13 (torch)", "REG_GRAD_TESTS",
                      "legacy: |dL/dQ(excluded logged action) + 1/B|",
                      abs(float(rg["legacy_excluded_logged_action_derivative"]) + 1.0 / B), 1e-12, notes=nt)
        else:
            log.not_run("P6", "Part I Section 13 (torch)", "REG_GRAD_TESTS", "legacy excluded-action derivative",
                        "the test batch had no excluded logged action")
        log.check("P6", "Part I Section 13 (torch)", "REG_GRAD_TESTS",
                  "all Section 13 regulariser unit tests passed (design B: zero leakage, all-excluded batch, normalisation)",
                  float(bool(rg.get("passed"))), 1.0, op="ge", notes=nt)
    c = x_p6_closed_form_checks(np_rng("xenv", "theory", "p6"))
    nt = f"batch {c['batch']}, excluded logged rows {c['excluded_rows']}, temperatures 0.5 / 1 / 2"
    env, inst = "closed form (numpy)", "random Q, masks, logged actions"
    log.check("P6", env, inst, "legacy: l(Q + h e_sa) - l(Q) + h for excluded logged a (unbounded below)",
              c["legacy_fd_dev"], 1e-12, notes=nt)
    log.report("P6", env, inst, "legacy: rows with admissible logged actions unaffected", c["legacy_untouched_rows_dev"],
               notes=f"{X_INVARIANT}: only the excluded rows are perturbed; {nt}")
    log.check("P6", env, inst, "legacy batch mean: |d/dQ(excluded logged action) + 1/B|", c["legacy_mean_derivative_dev"],
              1e-12, notes=nt)
    log.check("P6", env, inst, "design B: min contribution >= 0", c["design_b_min"], -1e-12, op="ge", notes=nt)
    log.check("P6", env, inst, "design B: invariant to every excluded coordinate (max change)",
              c["design_b_excluded_invariance"], 0.0, notes=nt)


def x_theory_tiny_suite(log: XPropositionLog, n_mdps: int = 20, S: int = 12, A: int = 4,
                        kappas: Sequence[float] = (0.0, 0.01, 0.1, 0.5, float("inf"))) -> pd.DataFrame:
    """P1-P5 and the LP cross-checks on random absorbing MDPs (true rows P0, uniform and random-W completions)."""
    sub = XPropositionLog()
    frontiers = []
    for i in range(n_mdps):
        P0, W = x_tiny_absorbing_mdp(i, S, A)
        E_W = P0.variant(f"{P0.name}-W", P=x_impute_inadmissible(P0.P, P0.mask, P0.live, W))
        E_U = P0.variant(f"{P0.name}-mean", P=x_impute_inadmissible(P0.P, P0.mask, P0.live))
        targets = [("bench_mean", E_U, "given", None), ("bench_terminate", E_U, "terminate", None),
                   ("true", P0, "given", None), ("true_rows_mean", E_U, "given", None),
                   ("true_rows_terminate", P0, "terminate", None), ("true_rows_W", E_W, "given", None)]
        pols = [(n, p, "random") for n, p in
                x_theory_random_policies(P0.mask, P0.live, np_rng("xenv", "theory", "tiny-policies", i), 20, 5, 5)]
        opt = XOptCache()
        x_theory_checks(sub, f"tiny|{i:02d}", E_U, targets, pols, opt, construction="best", env_W={id(E_W): W},
                        completions=(("mean", "true_rows_mean", "bench_mean"),
                                     ("terminate", "true_rows_terminate", "bench_terminate"),
                                     ("random W", "true_rows_W", "true_rows_W")))
        for lab, env, sem, Wf in (("true", P0, "given", None), ("true_rows_terminate", P0, "terminate", None),
                                  ("bench_mean", E_U, "given", None), ("true_rows_W", E_W, "given", W)):
            fr = x_exact_frontier(env, sem, kappas, opt, {"family": "tiny", "cell": "tiny", "draw": i, "target": lab},
                                  W=Wf)
            x_frontier_checks(sub, fr, f"tiny|{i:02d}")
            frontiers.append(fr)
    log.merge_aggregated(sub, environment=f"random absorbing MDPs ({n_mdps} x S={S}, A={A})",
                         instance=f"tiny|00-{n_mdps - 1:02d}")
    return pd.concat(frontiers, ignore_index=True)


# ---------- draw-0 instances, learned policies and frontier problems ----------
def x_cell_construction(spec: dict) -> Optional[str]:
    """'best' / 'oracle' for Gumbel cells (from the cell key, cross-checked with spec['construction'] if present)."""
    parts = str(spec["cell"]).split("|")
    assert parts[0] == spec["family"], f"cell key {spec['cell']!r} does not start with its family"
    if spec["family"] != "gumbel":
        return None
    c = parts[1]
    assert c in ("best", "oracle"), spec["cell"]
    if "construction" in spec:
        assert spec["construction"] == c, (spec["construction"], spec["cell"])
    return c


def x_theory_jobs() -> List[Tuple[str, Optional[dict]]]:
    """(role, cell spec) for every instance of the verification set; role 'icu' with spec None = X_ICU alone."""
    icu = [s for s in X_CELLS if s["family"] == "icu"]
    best = [s for s in X_CELLS if s["family"] == "gumbel" and x_cell_construction(s) == "best"]
    oracle = [s for s in X_CELLS if s["family"] == "gumbel" and x_cell_construction(s) == "oracle"]
    jobs: List[Tuple[str, Optional[dict]]] = [("icu", icu[0] if icu else None)]
    if best:
        jobs.append(("gumbel_best", best[0]))
    if oracle:
        jobs.append(("gumbel_oracle", oracle[0]))
    # Physician controls (e.g. withhold-vaso: unsupported actions have value in truth): frontiers only, no theory block.
    jobs += [("gumbel_control", s) for s in best[1:] if s.get("physician", "optimal") != "optimal"]
    jobs += [("garnet", s) for s in X_CELLS if s["family"] == "garnet"]
    return jobs


def x_frontier_problems(role: str, inst: Optional["CellInstance"]) -> List[Tuple[str, FiniteEnv, str]]:
    """(target label, env, semantics) frontier problems of one verification instance."""
    if role == "icu":
        return [("bench_mean", X_ICU, "given"), ("bench_terminate", X_ICU, "terminate")]
    tg = {lab: (env, sem) for lab, env, sem, _ in inst.targets}
    want = {"gumbel_best": ["bench_mean", "bench_terminate", "true"], "gumbel_oracle": ["bench_mean", "true"],
            "gumbel_control": ["bench_mean", "bench_terminate", "true"], "garnet": ["true"]}[role]
    missing = [w for w in want if w not in tg]
    if missing:
        raise KeyError(f"{inst.cell}: frontier targets {missing} missing from the instance")
    return [(lab, tg[lab][0], tg[lab][1]) for lab in want]


def x_theory_records(inst: "CellInstance", key: str) -> List[dict]:
    """run_xmethods on the draw-0 training split, cached on disk under an identity (resumable, FORCE_RETRAIN-aware)."""
    # x_result_xcfg (Section II.2): draw counts, cells, frontier budgets and bootstrap settings do not change what
    # run_xmethods computes, so extending a lite run to the full design keeps these records.
    identity = {"purpose": "II.5 theory records", "instance": inst.identity(), "key": key,
                "code": code_hash(*XENV_CORE_CODE, *XENV_ENV_CODE, *XENV_METHOD_CODE), "xcfg": x_result_xcfg(),
                "numeric": X_NUMERIC_CONSTANTS,
                "diffusion": [CFG.diffusion_T, CFG.diffusion_beta_start, CFG.diffusion_beta_end], "run_mode": RUN_MODE}
    h = obj_hash(identity)
    d = XENV_CACHE_DIR / "theory" / slug(key)
    npz_p, side_p = d / f"{h[:20]}.npz", d / f"{h[:20]}.identity.json"
    S, A = inst.learn_env.S, inst.learn_env.A
    if npz_p.exists() and side_p.exists():
        try:
            meta = json.loads(side_p.read_text())
        except XCACHE_READ_ERRORS as exc:     # a write cut by a disconnect: a miss, recomputed below
            print(f"WARNING: unreadable theory-record sidecar {side_p} ({type(exc).__name__}); recomputing.")
            meta = None
        if meta is not None and meta.get("identity_hash") != h:
            raise CheckpointIdentityError(f"{side_p} does not match the requested Part II theory identity.")
        if meta is not None and not (FORCE_RETRAIN and meta.get("force_token") != FORCE_TOKEN):
            try:
                with np.load(npz_p) as z:
                    pis = [np.array(z[f"pi_{i}"]) for i in range(len(meta["records"]))]
            except (*XCACHE_READ_ERRORS, ValueError, KeyError, zipfile.BadZipFile) as exc:
                print(f"WARNING: unreadable theory-record file {npz_p} ({type(exc).__name__}); recomputing.")
                pis = None
            if pis is not None:
                recs = [{**m, "pi": validate_policy(p, m["method"], shape=(S, A))} for m, p in zip(meta["records"], pis)]
                print(f"  {key}: loaded {len(recs)} cached learner records")
                return recs
    XCache.require_allowed(key)
    recs = run_xmethods(LearnerView.from_env(inst.learn_env), inst.train, key)
    for r in recs:
        validate_policy(r["pi"], f"{r['method']} [{r['variant']}]", shape=(S, A))
    d.mkdir(parents=True, exist_ok=True)
    tmp = npz_p.with_name(npz_p.name + ".tmp")
    with open(tmp, "wb") as fh:
        np.savez_compressed(fh, **{f"pi_{i}": np.asarray(r["pi"], dtype=np.float64) for i, r in enumerate(recs)})
    x_atomic_replace(tmp, npz_p)
    side_tmp = side_p.with_name(side_p.name + ".tmp")
    save_json({"key": key, "identity_hash": h, "identity": identity, "saved_utc": utc_now(), "force_token": FORCE_TOKEN,
               "records": [{k: v for k, v in r.items() if k != "pi"} for r in recs]}, side_tmp)
    x_atomic_replace(side_tmp, side_p)
    return recs


def x_theory_policies(inst_label: str, learn_env: FiniteEnv, behavior: np.ndarray,
                      records: Sequence[dict]) -> List[Tuple[str, np.ndarray, str]]:
    """Learned records + references + random policies on the learner's state space: (name, pi, kind)."""
    S, A = learn_env.S, learn_env.A
    out = [(x_label(r["method"], r["variant"], r["hp"], r["is_default"]), r["pi"], "learned") for r in records]
    out += [("REF behaviour", behavior, "reference"),
            ("REF uniform-admissible", x_uniform_admissible(learn_env.mask, learn_env.live), "reference"),
            ("REF uniform (all actions)", np.full((S, A), 1.0 / A), "reference")]
    rng = np_rng("xenv", "theory", "policies", inst_label)
    out += [(n, p, "random") for n, p in x_theory_random_policies(learn_env.mask, learn_env.live, rng, 6, 2)]
    return [(n, validate_policy(p, n, shape=(S, A)), k) for n, p, k in out]


def x_proposition_summary(checks: pd.DataFrame, ran: bool) -> pd.DataFrame:
    rows = []
    for p, title in X_PROPOSITION_TITLES.items():
        sub = checks[checks["proposition"] == p]
        is_check = sub["passes"].notna()
        n_chk = int(is_check.sum())
        n_pass = int(sub.loc[is_check, "passes"].astype(bool).sum())
        n_nr = int(sub["notes"].astype(str).str.startswith("not run").sum())
        status = "not run" if (not ran or n_chk == 0) else ("verified" if n_pass == n_chk else "FAILED")
        rows.append({"proposition": p, "title": title, "checks": n_chk, "passed": n_pass, "failed": n_chk - n_pass,
                     "descriptive": int((~is_check).sum()) - n_nr, "not_run": n_nr,
                     "instances": "; ".join(sorted(set(sub["instance"].astype(str)))), "status": status,
                     "verified_in": "Section II.5 (xenv/proposition_checks.csv)"})
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# II.5 Run: propositions on the verification set, exact frontiers
# ============================================================
_X_RUN_THEORY = bool(EXPERIMENTS["xenv_theory"])
_X_RUN_FRONTIERS = bool(EXPERIMENTS["xenv_frontiers"])
PROP_LOG = XPropositionLog()
X_THEORY_INFO: Dict[str, Any] = {"jobs": [], "theory": _X_RUN_THEORY, "frontiers": _X_RUN_FRONTIERS}
_x_frontiers: List[pd.DataFrame] = []
_t_theory = time.perf_counter()
if _X_RUN_THEORY or _X_RUN_FRONTIERS:
    assert globals().get("X_ICU") is not None, "II.5 needs X_ICU (Section II.3 builds it when any xenv switch is on)"

if _X_RUN_THEORY:
    _t0 = time.perf_counter()
    X_THEORY_INFO["tiny_frontiers"] = int(len(x_theory_tiny_suite(PROP_LOG)))
    x_theory_p6(PROP_LOG)
    _e_main = XEmpirical(MAIN.data, S_N, A_N, BENCH.terminal_states)
    x_theory_p7(PROP_LOG, X_ICU.name, "icu|MAIN (Part I dataset)", _e_main.pi_b, X_ICU.live)
    del _e_main
    print(f"II.5 random MDPs, P6 and ICU P7 done ({time.perf_counter() - _t0:.1f} s)")

if _X_RUN_THEORY or _X_RUN_FRONTIERS:
    for _role, _spec in x_theory_jobs():
        if _role == "gumbel_control" and not _X_RUN_FRONTIERS:   # a frontier-only job
            continue
        _t0 = time.perf_counter()
        _opt = XOptCache()
        _inst = build_cell_instance(_spec, 0) if _spec is not None else None
        _cell = _spec["cell"] if _spec is not None else "icu|released"
        _label = f"{_cell}|d0" if _spec is not None else "icu|X_ICU (no ICU cell: reference and random policies)"
        _info = {"role": _role, "cell": _cell}
        if _X_RUN_THEORY and _role != "gumbel_control":
            if _inst is not None:
                _key = f"{_cell}|draw0"           # the driver's run_xmethods key for draw 0
                _records = x_theory_records(_inst, _key)
                _learn, _targets, _behavior = _inst.learn_env, _inst.targets, _inst.behavior
            else:
                _records, _learn, _behavior = [], X_ICU, X_ICU_BEHAVIOR
                _targets = [("bench_mean", X_ICU, "given", None), ("bench_terminate", X_ICU, "terminate", None)]
            _pols = x_theory_policies(_label, _learn, _behavior, _records)
            _imputed = bool(MEAN_RULE_IDENTITY_HOLDS) if _role == "icu" else True
            if _role == "icu":
                PROP_LOG.check("P1", X_ICU.name, _label, "direct imputation test agrees with Section 3's mean-rule identity",
                               float((x_imputation_deviation(X_ICU) <= X_IMPUTATION_TOL) != _imputed), 0.0,
                               notes=f"MEAN_RULE_IDENTITY_HOLDS = {bool(MEAN_RULE_IDENTITY_HOLDS)}")
            _info["checks"] = x_theory_checks(PROP_LOG, _label, _learn, _targets, _pols, _opt,
                                              construction=x_cell_construction(_spec) if _spec is not None else None,
                                              learn_env_imputed=_imputed)
            if _inst is not None:
                assert _inst.learn_env.S == _learn.S
                if _role == "gumbel_oracle":   # the oracle benchmark is exactly the true rows + mean completion
                    _tg = {lab: env for lab, env, _, _ in _inst.targets}
                    _Bo, _Tm = _tg["bench_mean"], _tg["true"]
                    _imp = x_impute_inadmissible(_Tm.P, _Bo.mask, _Tm.live)
                    PROP_LOG.check("P4", _Bo.name, _label, "oracle benchmark = true rows + mean completion on its mask "
                                   "(max |dP| + 1[d0 differs from the true d0])",
                                   float(np.abs(_Bo.P - _imp).max()) + float(not np.array_equal(_Bo.d0, _Tm.d0)),
                                   X_IMPUTATION_TOL, notes="the estimation term of the oracle construction is zero")
                    del _imp, _Bo, _Tm, _tg
                _e = XEmpirical(_inst.train, _learn.S, _learn.A, _learn.terminals)
                x_theory_p7(PROP_LOG, _learn.name, _label, _e.pi_b, _learn.live, _records)
                del _e
            del _records, _pols
        if _X_RUN_FRONTIERS:
            for _tlab, _env, _sem in x_frontier_problems(_role, _inst):
                _fr = x_exact_frontier(_env, _sem, XCFG.frontier_kappas, _opt,
                                       {"family": "icu" if _role == "icu" else _spec["family"], "cell": _cell, "draw": 0,
                                        "target": _tlab})
                _x_frontiers.append(_fr)
                if _X_RUN_THEORY:
                    x_frontier_checks(PROP_LOG, _fr, _label)
        _info["seconds"] = time.perf_counter() - _t0
        X_THEORY_INFO["jobs"].append(_info)
        print(f"II.5 {_role:<14} {_cell:<36} {_info['seconds']:7.1f} s")
        del _inst, _opt
        gc.collect()

In [ ]:
# ============================================================
# II.5 Summary, outputs and globals
# ============================================================
PROP_CHECKS = PROP_LOG.frame() if _X_RUN_THEORY else pd.DataFrame(columns=X_PROP_COLUMNS)
PROP_SUMMARY = x_proposition_summary(PROP_CHECKS, _X_RUN_THEORY)
_n_chk = int(PROP_CHECKS["passes"].notna().sum()) if len(PROP_CHECKS) else 0
PROPOSITIONS_OK = (bool(_n_chk > 0 and PROP_CHECKS.loc[PROP_CHECKS["passes"].notna(), "passes"].astype(bool).all())
                   if _X_RUN_THEORY else None)
X_FRONTIERS = (pd.concat(_x_frontiers, ignore_index=True) if _x_frontiers
               else pd.DataFrame(columns=X_FRONTIER_COLUMNS))
X_FRONTIERS_OK = bool(len(X_FRONTIERS) and X_FRONTIERS["passes"].astype(bool).all()) if _X_RUN_FRONTIERS else None
PROP_CHECKS.to_csv(XENV_TABLE_DIR / "proposition_checks.csv", index=False)
PROP_SUMMARY.to_csv(XENV_TABLE_DIR / "proposition_summary.csv", index=False)
X_FRONTIERS.to_csv(XENV_TABLE_DIR / "exact_frontiers.csv", index=False)
X_THEORY_INFO["seconds"] = time.perf_counter() - _t_theory
del _x_frontiers

if _X_RUN_THEORY:
    display(PROP_SUMMARY[["proposition", "title", "checks", "passed", "failed", "descriptive", "not_run", "status"]])
    _fail = PROP_CHECKS[PROP_CHECKS["passes"].eq(False).fillna(False)]
    if len(_fail):
        print(f"WARNING: {len(_fail)} proposition check(s) FAILED (the release gate will stop):")
        display(_fail)
    print(f"PROPOSITIONS_OK = {PROPOSITIONS_OK} ({_n_chk} checks)")
else:
    print("II.5 propositions not run (EXPERIMENTS['xenv_theory'] = False).")
if _X_RUN_FRONTIERS:
    _fs = (X_FRONTIERS.groupby(["family", "cell", "target", "semantics"], sort=False)
           .agg(J_kappa0=("objective", "first"), J_inf=("objective", "last"), flat_expected=("flat_expected", "first"),
                frontier_range=("frontier_range", "first"), passes=("passes", "all")).reset_index())
    display(_fs)
    print(f"X_FRONTIERS_OK = {X_FRONTIERS_OK}. Flat frontiers under convex-imputation rows (P2) or termination with "
          "r_term = v_min (P3) are findings, not errors.")
else:
    print("II.5 frontiers not run (EXPERIMENTS['xenv_frontiers'] = False).")
print(f"II.5 total {X_THEORY_INFO['seconds']:.1f} s")

XENV_THEORY_CODE = [XPropositionLog, x_tv, x_imputation_deviation, x_admissible_row_deviation, x_inadmissible_tv,
                    x_shares_admissible_rows, x_theory_random_policies, XOptCache, x_lp_optimum, x_p1_policy,
                    x_p2_env, x_p3_policy, x_p4_policy, _x_masked_lse, x_p6_closed_form_checks, x_reverse_marginal,
                    x_p7_state_checks, x_tiny_absorbing_mdp, x_exact_frontier, x_frontier_checks, x_find_record,
                    _x_p1_block, _x_p2_block, _x_p3_block, _x_p5_block, _x_p4_block, x_theory_checks, x_theory_p7,
                    x_theory_p6, x_theory_tiny_suite, x_cell_construction, x_theory_jobs, x_frontier_problems,
                    x_theory_records, x_theory_policies, x_proposition_summary]

## II.6 Driver: every method × every target × every dataset draw

The unit of Part II is one **dataset draw** of one environment cell. For each draw the driver

1. builds the `CellInstance` (deterministic in (cell, draw)) and its identity: instance hashes (learner MDP, user
   data, train / dev split, behaviour policy, every target environment), the Part II code hash, the `XCFG` fields
   that can change a draw's result, the categorical-diffusion schedule, the cell parameters and the run mode;
2. loads the identity-matched result from `XCACHE`, or, if `ALLOW_TRAINING`, computes it. On a cache miss the full
   method suite (`run_xmethods`) is run on the **training episodes only**; learners receive a `LearnerView` (sizes,
   mask, absorbing states, reward range) and never see P or d0;
3. adds `Behaviour (true) [native]`, the policy that generated the user data, and the uniform admissible
   projection `x_project(π, M, live)` of **every** native record, with the same method, hyper-parameters and
   default flag. Every method then has a `[projected]` counterpart in every draw, as paired tests need;
4. evaluates every policy exactly on every target of the cell. A `lift` maps learner-space policies to the true
   state space where needed. Each policy is also evaluated on `fqe_dev`, the tabular FQE model of the held-out
   dev episodes (`x_fqe_env`, semantics "given"). `fqe_dev` is used only for offline selection (`target_role` =
   "selection") and never for reporting;
5. adds three reference rows per target, computed on the target itself: `REF optimum`, `REF optimum (admissible)`
   and `REF behaviour` (the lifted behaviour). They give the normalised score
   z = (J − J_REF behaviour) / (J_REF optimum − J_REF behaviour);
6. checks the invariants below before caching, and again after loading. The table returned is always the one read
   back from the cache, so an interrupted, resumed run and an uninterrupted run give identical tables.

**Exact reference optimum.** `REF optimum` comes from value iteration (`x_optimal`). The model may be estimated:
a constructed benchmark, the tabular FQE model, or ICU-Sepsis itself. In such a model a reward-free closed loop
can make the greedy VI policy improper, because under γ = 1 value iteration also counts behaviour that is never
absorbed (value 0). In that case the optimum over **proper** policies is solved exactly. The solver is the occupancy
LP `x_cmdp_lp`, with full-support initial weights w = ½·d0 + ½·uniform(live) and κ = ∞ (or κ = 0 with
cost 1[a ∉ M] for the admissible optimum). Every LP solution is proper: a closed recurrent class C would need
w(C) = 0. A proper policy that attains the proper-policy optimum at every state exists (strict policy iteration
from a proper policy stays proper), so the LP policy attains it from every state. Its exact value at d0 is
therefore the proper optimum. The driver checks wᵀv = LP objective to 1e-7 and records which solver ran in the
`solver` column.

**Invariants asserted per draw.**
* Every status is ok or improper. An improper (never-absorbing) policy is a reported outcome, never hidden.
* Each target has exactly one row per reference policy, and each policy is evaluated once on every target.
* Masked and projected policies have zero inadmissible mass on every target. This also checks that the lifted
  masks agree with the learner mask.
* No policy beats `REF optimum` on any target, and no zero-violation policy beats `REF optimum (admissible)`
  (tolerance 1e-6).
* Projection identity (Proposition P1, driver-level): on mean-imputed targets (`bench_mean`, `true_rows_mean`)
  native and projected policies have identical return and length (1e-8). For ICU-Sepsis this is asserted only when
  the released mean-rule identity holds (Section 3).
* The cache round-trip is exact.

**Identity design.** Some settings decide which cells and draws exist, or configure statistics and theory, but do
not change what a draw computes: the draw counts, the cell lists, the Monte-Carlo check sizes, the frontier budgets
and the bootstrap settings. They are left out of the identity, so a lite run can be extended to the full profile
without recomputation. A cell's own parameters still enter through the spec and the instance hashes. The code hash
covers source text only, so the values of module constants that hashed functions read (tie and leak tolerances,
the semantics map) enter through `XDRIVER_CONSTANTS["numeric"]`. Section II.5
code is not hashed, because no driver result depends on it. The per-draw checks are re-run on cached tables, so
they are not hashed either.

**Exact deduplication.** Byte-identical policies (for example a projection of a policy that is already admissible)
are evaluated once per target.

| `XRESULTS` column | meaning |
|---|---|
| cell, family, draw, deploy_target | experimental unit; the cell's pre-declared deployment target |
| policy, method, variant, hp, is_default, method_family | `x_label`; native / masked / projected / reference; canonical-JSON hyper-parameters |
| is_reference, solver | reference rows (`REF …`) and the solver that produced them (vi / lp_full_support / evaluation) |
| converged | learner convergence flag from `run_xmethods` (empty if the method has none) |
| target, target_role, is_deploy_target, semantics, target_env, lifted | evaluation target; "target" or "selection" (`fqe_dev`) |
| status, error | ok / improper (with the evaluator's message) |
| return … absorb_other_terminal | exact metrics; `absorb_goal` is the probability of absorption in the goal state |

In [ ]:
# ============================================================
# II.6 Cross-environment driver
# ============================================================
EXECLOG.mark("II.6 Cross-environment driver")
from collections import Counter

XBEHAVIOUR_METHOD = "Behaviour (true)"
XMETHOD_FAMILIES = ("behaviour", "value", "pessimistic", "constrained", "guided-prior", "diffusion")
XFQE_TARGET = "fqe_dev"
XREF_POLICIES = ("REF optimum", "REF optimum (admissible)", "REF behaviour")
XMEAN_IMPUTED_TARGETS = ("bench_mean", "true_rows_mean")   # inadmissible rows = uniform mean of admissible rows
XPROJ_TOL, XDOM_TOL, XMASS_TOL, XLP_TOL = 1e-8, 1e-6, 1e-12, 1e-7
_XMETRIC_SOURCE = {"return": "return", "avg_episode_length": "avg_episode_length",
                   "inadmissible_rate": "inadmissible_rate", "expected_violations": "expected_violations",
                   "prob_any_violation": "prob_any_violation",
                   "states_with_inadmissible_mass": "states_with_inadmissible_mass",
                   "max_state_inadmissible_mass": "max_state_inadmissible_mass",
                   "mean_state_inadmissible_mass": "mean_state_inadmissible_mass",
                   "absorb_death": "absorb_death", "absorb_goal": "absorb_survival",
                   "absorb_other_terminal": "absorb_other_terminal"}
XRESULT_COLUMNS = ["cell", "family", "draw", "deploy_target", "policy", "method", "variant", "hp", "is_default",
                   "method_family", "is_reference", "solver", "converged", "target", "target_role",
                   "is_deploy_target", "semantics", "target_env", "lifted", "status", *_XMETRIC_SOURCE, "error"]
# _XCFG_NOT_IN_IDENTITY and x_result_xcfg (the XCFG fields that can change a draw) are defined in Section II.2;
# X_NUMERIC_CONSTANTS (module constants read by hashed functions, by value) in Section II.4.
XDRIVER_CONSTANTS = {"columns": XRESULT_COLUMNS, "metrics": _XMETRIC_SOURCE, "fqe_target": XFQE_TARGET,
                     "behaviour_method": XBEHAVIOUR_METHOD, "reference_policies": list(XREF_POLICIES),
                     "numeric": X_NUMERIC_CONSTANTS}


def x_cache_key(cell: str, draw: int) -> str:
    return cell.replace("|", "/") + f"/draw{int(draw)}"


def x_draw_identity(spec: dict, inst: CellInstance) -> dict:
    # behaviour and dev split are hashed explicitly: CellInstance.identity() covers data, train and targets only.
    return {"instance": inst.identity(), "behavior": array_hash(np.asarray(inst.behavior, dtype=np.float64)),
            "dev": array_dict_hash(inst.dev), "code": XENV_CODE_HASH, "xcfg": x_result_xcfg(),
            "diffusion": [CFG.diffusion_T, CFG.diffusion_beta_start, CFG.diffusion_beta_end], "run_mode": RUN_MODE,
            "spec": {k: v for k, v in spec.items() if k != "draws"}, "driver": XDRIVER_CONSTANTS}


def _x_pi_key(pi: np.ndarray) -> bytes:
    a = np.ascontiguousarray(pi, dtype=np.float64)
    return hashlib.blake2b(a.tobytes(), digest_size=16).digest() + str(a.shape).encode()


def _x_compact(obj, limit: int = 2000):
    """JSON-native copy of a meta object; sequences longer than `limit` JSON characters become a hashed summary."""
    if isinstance(obj, dict):
        return {str(k): _x_compact(v, limit) for k, v in obj.items()}
    if isinstance(obj, np.ndarray) and obj.ndim == 0:
        return obj.item()
    if isinstance(obj, (list, tuple, np.ndarray)):
        txt = canonical_json(obj)
        if len(txt) <= limit:
            return json.loads(txt)
        return f"<{type(obj).__name__} of {len(obj)} items, sha256 {sha256_bytes(txt.encode())[:12]}>"
    if isinstance(obj, np.generic):
        return obj.item()
    if obj is None or isinstance(obj, (bool, int, float, str)):
        return obj
    return json.loads(canonical_json(obj))


def _x_flatten(d: dict, prefix: str = "") -> dict:
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(_x_flatten(v, key + "."))
        elif isinstance(v, (list, tuple)):
            out[key] = canonical_json(v)
        else:
            out[key] = v
    return out


# ---------- records: validated learner outputs + data-generating behaviour + admissible projections ----------
def _x_record(r: dict, label: str, pi: np.ndarray) -> dict:
    c = r["info"].get("converged")
    desc = {"policy": label, "method": r["method"], "variant": r["variant"], "hp": canonical_json(r["hp"]),
            "is_default": bool(r["is_default"]), "method_family": r["family"], "is_reference": False,
            "solver": np.nan, "converged": np.nan if c is None else bool(c)}
    return {**r, "label": label, "pi": pi, "desc": desc}


def x_collect_records(raw: Sequence[dict], behavior: np.ndarray, env: FiniteEnv) -> List[dict]:
    """Validate run_xmethods records, add the behaviour policy and the projection of every native record."""
    need = {"method", "variant", "hp", "is_default", "family", "pi", "info"}
    base = []
    for r in raw:
        missing = need.difference(r)
        assert not missing, f"run_xmethods record {r.get('method')!r} lacks {sorted(missing)}"
        assert r["variant"] in ("native", "masked"), \
            f"{r['method']}: variant {r['variant']!r} (run_xmethods returns native / masked; the driver adds projected)"
        assert r["method"] != XBEHAVIOUR_METHOD, f"'{XBEHAVIOUR_METHOD}' is reserved for the driver"
        assert r["family"] in XMETHOD_FAMILIES, f"{r['method']}: family {r['family']!r} not in {XMETHOD_FAMILIES}"
        base.append({"method": str(r["method"]), "variant": r["variant"], "hp": dict(r["hp"] or {}),
                     "is_default": bool(r["is_default"]), "family": str(r["family"]), "pi": r["pi"],
                     "info": dict(r["info"] or {})})
    base.append({"method": XBEHAVIOUR_METHOD, "variant": "native", "hp": {}, "is_default": True, "family": "behaviour",
                 "pi": behavior, "info": {"source": "behaviour policy that generated the user data"}})
    out = []
    for r in base:
        label = x_label(r["method"], r["variant"], r["hp"], r["is_default"])
        pi = validate_policy(r["pi"], label, shape=(env.S, env.A))
        mass = float(x_inadmissible_mass(pi, env.mask, env.live).max())
        if r["variant"] == "masked":
            assert mass == 0.0, f"{label}: masked variant puts mass {mass:.3g} outside the learner mask"
        out.append(_x_record(r, label, pi))
        if r["variant"] == "native":
            info = {**r["info"], "projected_from": label, "native_max_inadmissible_mass": mass}
            out.append(_x_record({**r, "variant": "projected", "info": info},
                                 x_label(r["method"], "projected", r["hp"], r["is_default"]),
                                 x_project(pi, env.mask, env.live)))
    dup = sorted(k for k, n in Counter(r["label"] for r in out).items() if n > 1)
    assert not dup, f"duplicate policy labels: {dup[:5]}"
    n_def = Counter((r["method"], r["variant"]) for r in out if r["is_default"])
    no_def = sorted({(r["method"], r["variant"]) for r in out} - set(n_def))
    assert not no_def and max(n_def.values()) == 1, f"need exactly one default record per method / variant: {no_def[:5]}"
    defaults = {r["label"] for r in out if r["is_default"]}
    missing = [lab for lab in (XCOMPARATOR, *XTREATMENTS) if lab not in defaults]
    assert not missing, f"preregistered Part II policies missing from the default set: {missing}"
    return out


# ---------- exact evaluation, reference rows and row assembly ----------
def _x_eval_metrics(env: FiniteEnv, pi: np.ndarray, semantics: str) -> dict:
    out = x_evaluate(env, pi, semantics)
    if out["status"] == "ok":
        return {"status": "ok", **{src: float(out[src]) for src in _XMETRIC_SOURCE.values()}}
    assert out["status"] == "improper", out["status"]
    return {"status": "improper", "error": str(out.get("error", ""))[:200]}


def x_result_row(base: dict, desc: dict, tgt: dict, out: dict) -> dict:
    """One XRESULTS row: unit columns + policy description + target description + exact metrics."""
    status = out["status"]
    assert status in ("ok", "improper"), status
    ok = status == "ok"
    row = {**base, **desc, **tgt, "status": status,
           **{col: (float(out[src]) if ok else np.nan) for col, src in _XMETRIC_SOURCE.items()},
           "error": np.nan if ok else str(out.get("error", ""))[:200]}
    assert len(row) == len(XRESULT_COLUMNS) and set(row) == set(XRESULT_COLUMNS), \
        sorted(set(row) ^ set(XRESULT_COLUMNS))
    if ok:
        assert all(np.isfinite(row[c]) for c in _XMETRIC_SOURCE), f"non-finite metric for {desc['policy']}"
    return row


def _x_ref_desc(name: str, solver: str) -> dict:
    return {"policy": name, "method": name, "variant": "reference", "hp": "{}", "is_default": False,
            "method_family": "reference", "is_reference": True, "solver": solver, "converged": np.nan}


def x_reference_lp(env: FiniteEnv, semantics: str, constrained: bool) -> Tuple[Optional[np.ndarray], str]:
    """Optimum over proper policies: occupancy LP with full-support weights (see markdown); verified w'v = objective."""
    live = env.live
    P = x_semantic_P(env, semantics)
    r = np.einsum("san,n->sa", P, env.entry_reward, optimize=True)
    r[~live] = 0.0
    cost = (~env.mask & live[:, None]).astype(np.float64)
    w = 0.5 * env.d0 + 0.5 * live / live.sum()
    try:
        pi, info = x_cmdp_lp(P, r, w, live, cost, 0.0 if constrained else np.inf)
    except RuntimeError as exc:   # infeasible: some live state has no proper policy (every policy is improper)
        return None, f"lp_infeasible: {str(exc)[:120]}"
    if constrained:   # drop solver-tolerance flow on inadmissible pairs (the identity check below re-verifies)
        pi[live] = np.where(env.mask[live], pi[live], 0.0)
        pi[live] /= pi[live].sum(axis=1, keepdims=True)
    out = x_evaluate(env, pi, semantics, vectors=True)
    if out["status"] != "ok":
        raise EvaluationInvariantError(f"{env.name}: full-support LP policy is improper ({out.get('error')})")
    jw = float(w @ out["value_vector"])
    if abs(jw - info["objective"]) > XLP_TOL * max(1.0, abs(info["objective"])):
        raise EvaluationInvariantError(f"{env.name}: LP objective {info['objective']:.12g} != exact w'v {jw:.12g}")
    return pi, "lp_full_support"


_XREF_MEMO: Dict[Tuple[str, str, bool], Tuple[dict, str]] = {}


def x_reference_optimum(env: FiniteEnv, semantics: str, constrained: bool) -> Tuple[dict, str]:
    """Exact (admissible-)optimum metrics of a target; memoised by the environment identity (bounded)."""
    key = (obj_hash(env.identity()), semantics, bool(constrained))
    if key not in _XREF_MEMO:
        try:
            pi, _, _ = x_optimal(env, semantics, constrained=constrained)
            solver = "vi"
        except RuntimeError as exc:   # improper greedy VI policy / DP-evaluation gap / VI non-convergence
            pi, solver = x_reference_lp(env, semantics, constrained)
            solver = f"{solver} (vi: {type(exc).__name__})"
        out = _x_eval_metrics(env, pi, semantics) if pi is not None else {"status": "improper", "error": solver}
        if len(_XREF_MEMO) >= 256:
            _XREF_MEMO.pop(next(iter(_XREF_MEMO)))
        _XREF_MEMO[key] = (out, solver)
    return _XREF_MEMO[key]


def x_reference_rows(base: dict, tgt: dict, env: FiniteEnv, semantics: str,
                     behaviour_out: dict) -> Tuple[List[dict], List[str]]:
    rows, solvers = [], []
    for name, constrained in (("REF optimum", False), ("REF optimum (admissible)", True)):
        out, solver = x_reference_optimum(env, semantics, constrained)
        rows.append(x_result_row(base, _x_ref_desc(name, solver), tgt, out))
        solvers.append(solver)
    rows.append(x_result_row(base, _x_ref_desc("REF behaviour", "evaluation"), tgt, behaviour_out))
    return rows, solvers


def x_evaluate_targets(inst: CellInstance, records: List[dict], fqe: FiniteEnv) -> Tuple[List[dict], dict]:
    """Exact evaluation of every record on every target (lifted where needed) and on the FQE selection model."""
    learn = inst.learn_env
    base = {"cell": inst.cell, "family": inst.family, "draw": int(inst.draw), "deploy_target": inst.deploy_target}
    keys = [_x_pi_key(r["pi"]) for r in records]
    beh = [k for r, k in zip(records, keys) if r["method"] == XBEHAVIOUR_METHOD and r["variant"] == "native"]
    assert len(beh) == 1
    plan = [(lab, env, sem, lift, "target") for lab, env, sem, lift in inst.targets]
    plan.append((XFQE_TARGET, fqe, "given", None, "selection"))
    labels = [p[0] for p in plan]
    assert len(set(labels)) == len(labels), f"duplicate target labels {labels}"
    assert inst.deploy_target in labels[:-1], f"deploy target {inst.deploy_target!r} not among {labels[:-1]}"
    rows, stats_ = [], Counter()
    for lab, env, sem, lift, role in plan:
        assert sem in X_SEMANTICS and env.A == learn.A, (lab, sem, env.A, learn.A)
        if lift is None:
            assert env.S == learn.S, f"target {lab}: state space differs from the learner's but no lift is given"
        tgt = {"target": lab, "target_role": role, "is_deploy_target": lab == inst.deploy_target, "semantics": sem,
               "target_env": env.name, "lifted": lift is not None}
        memo: Dict[bytes, dict] = {}
        for r, k in zip(records, keys):
            if k not in memo:
                memo[k] = _x_eval_metrics(env, r["pi"] if lift is None else lift(r["pi"]), sem)
                stats_["evaluations_unique"] += 1
            stats_["evaluations_total"] += 1
            rows.append(x_result_row(base, r["desc"], tgt, memo[k]))
        if role == "target":
            ref_rows, solvers = x_reference_rows(base, tgt, env, sem, memo[beh[0]])
            rows += ref_rows
            for s in solvers:
                stats_[f"ref_solver.{s}"] += 1
        del memo
    return rows, dict(stats_)


# ---------- invariant checks (pure pandas; re-run on cached tables) ----------
def x_projection_identity(df: pd.DataFrame,
                          targets: Sequence[str] = XMEAN_IMPUTED_TARGETS) -> Dict[str, Dict[str, float]]:
    """Native vs projected return / length on mean-imputed targets (identical by Proposition P1)."""
    out = {}
    keys = ["method", "hp", "is_default"]
    for t in targets:
        sub = df[(df["target"] == t) & ~df["is_reference"].astype(bool)]
        if sub.empty:
            continue
        nat = sub[sub["variant"] == "native"].set_index(keys).sort_index()
        prj = sub[sub["variant"] == "projected"].set_index(keys).sort_index()
        assert nat.index.is_unique and nat.index.equals(prj.index), f"{t}: native and projected records do not pair up"
        s_n, s_p = nat["status"].to_numpy(), prj["status"].to_numpy()
        both = (s_n == "ok") & (s_p == "ok")
        jn, jp = nat["return"].to_numpy(dtype=float), prj["return"].to_numpy(dtype=float)
        ln, lp = nat["avg_episode_length"].to_numpy(dtype=float), prj["avg_episode_length"].to_numpy(dtype=float)
        dj = np.abs(jn - jp)[both]
        dl = (np.abs(ln - lp) / np.maximum(1.0, np.abs(ln)))[both]
        out[t] = {"pairs": int(len(nat)), "status_mismatches": int((s_n != s_p).sum()),
                  "max_abs_diff_return": float(dj.max(initial=0.0)), "max_rel_diff_length": float(dl.max(initial=0.0))}
    return out


def x_check_draw_rows(df: pd.DataFrame, strict_projection: bool) -> Dict[str, Any]:
    """Assert the invariants of one draw's rows (see markdown) and return a small report."""
    bad = sorted(set(df["status"].astype(str)) - {"ok", "improper"})
    assert not bad, f"unexpected status values {bad}"
    ok = df["status"] == "ok"
    ref = df["is_reference"].astype(bool)
    is_tgt = df["target_role"] == "target"
    assert not (ref & ~is_tgt).any(), "reference rows on a selection model"
    pol_sets = {t: frozenset(s) for t, s in df[~ref].groupby("target", sort=False)["policy"]}
    assert len(set(pol_sets.values())) == 1, "the policy set differs between targets"
    per = df[~ref].groupby("target", sort=False)["policy"].agg(lambda s: s.duplicated().any())
    assert not per.any(), f"a policy is evaluated twice on targets {list(per[per].index)}"
    adm = df["variant"].isin(["masked", "projected"]) & ok
    worst_mass = float(df.loc[adm, ["max_state_inadmissible_mass", "expected_violations"]].to_numpy().max(initial=0.0))
    assert worst_mass <= XMASS_TOL, f"masked / projected policy with inadmissible mass {worst_mass:.3g}"
    slack, slack_adm = -np.inf, -np.inf
    for t, sub in df[is_tgt].groupby("target", sort=False):
        got = Counter(sub.loc[sub["is_reference"].astype(bool), "policy"])
        assert dict(got) == {p: 1 for p in XREF_POLICIES}, f"target {t}: reference rows {dict(got)}"
        opt = sub[sub["policy"] == "REF optimum"].iloc[0]
        aopt = sub[sub["policy"] == "REF optimum (admissible)"].iloc[0]
        okt = sub[sub["status"] == "ok"]
        if opt["status"] == "ok":
            s = float((okt["return"] - opt["return"]).max())
            slack = max(slack, s)
            assert s <= XDOM_TOL * max(1.0, abs(opt["return"])), f"target {t}: a policy beats REF optimum by {s:.3g}"
        if aopt["status"] == "ok":
            assert aopt["expected_violations"] <= XMASS_TOL, f"target {t}: admissible optimum violates the mask"
            zero = okt[okt["expected_violations"] <= XMASS_TOL]
            s = float((zero["return"] - aopt["return"]).max())
            slack_adm = max(slack_adm, s)
            assert s <= XDOM_TOL * max(1.0, abs(aopt["return"])), \
                f"target {t}: a zero-violation policy beats REF optimum (admissible) by {s:.3g}"
        if opt["status"] == "ok" and aopt["status"] == "ok":
            assert aopt["return"] <= opt["return"] + XDOM_TOL * max(1.0, abs(opt["return"])), f"target {t}"
    pid = x_projection_identity(df)
    worst_p = max([max(v["max_abs_diff_return"], v["max_rel_diff_length"]) for v in pid.values()], default=0.0)
    mism = sum(v["status_mismatches"] for v in pid.values())
    if strict_projection:
        assert worst_p <= XPROJ_TOL and mism == 0, f"projection identity violated: {pid}"
    improper = df.loc[~ok].groupby("target")["policy"].size()
    return {"rows": int(len(df)), "improper_rows": int((~ok).sum()),
            "improper_by_target": {str(k): int(v) for k, v in improper.items()},
            "max_slack_vs_optimum": float(slack), "max_slack_vs_admissible_optimum": float(slack_adm),
            "masked_projected_max_inadmissible_mass": worst_mass, "projection_identity": pid,
            "projection_identity_worst": float(worst_p), "projection_identity_strict": bool(strict_projection)}


# ---------- one (cell, draw) ----------
def run_cell_draw(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
    """All Part II results of one dataset draw of one cell (identity-cached in XCACHE)."""
    cell, family = spec["cell"], spec["family"]
    assert family in ("icu", "gumbel", "garnet"), family
    t0 = time.perf_counter()
    inst = build_cell_instance(spec, draw)
    assert (inst.cell, inst.family, int(inst.draw)) == (cell, family, int(draw)), (inst.cell, inst.family, inst.draw)
    if family == "icu" and int(draw) == 0:
        assert all(np.array_equal(inst.data[k], OFFLINE[k]) for k in
                   ("state", "action", "reward", "next_state", "episode_id", "timestep")), \
            "ICU draw 0 must be the Part I dataset (OFFLINE)"
    identity = x_draw_identity(spec, inst)
    key, ident_hash = x_cache_key(cell, draw), obj_hash(identity)[:20]
    strict = family != "icu" or bool(MEAN_RULE_IDENTITY_HOLDS)
    build_s = time.perf_counter() - t0
    hit, meta_hit = XCACHE.get(key, identity), XCACHE.get(key + "/meta", identity)
    if hit is not None and meta_hit is not None:
        del inst
        assert list(hit.columns) == XRESULT_COLUMNS, f"{key}: cached columns differ from XRESULT_COLUMNS"
        meta = json.loads(str(meta_hit["meta_json"].iloc[0]))
        meta.update(cache="loaded", session_build_s=build_s, checks=x_check_draw_rows(hit, strict))
        gc.collect()
        return hit, meta
    if (hit is None) != (meta_hit is None):
        print(f"  {key}: incomplete cache entry (results or meta missing) - recomputing this draw")
    XCACHE.require_allowed(key)
    learn = inst.learn_env
    t1 = time.perf_counter()
    raw = run_xmethods(LearnerView.from_env(learn), inst.train, f"{cell}|draw{int(draw)}")
    records = x_collect_records(raw, inst.behavior, learn)
    del raw
    t2 = time.perf_counter()
    fqe = x_fqe_env(inst.dev, inst.train, learn, name=f"fqe|{cell}|draw{int(draw)}")
    t3 = time.perf_counter()
    rows, ev_stats = x_evaluate_targets(inst, records, fqe)
    df = pd.DataFrame(rows, columns=XRESULT_COLUMNS)
    t4 = time.perf_counter()
    checks = x_check_draw_rows(df, strict)
    t5 = time.perf_counter()
    nonconv = [r["label"] for r in records if r["desc"]["converged"] is False]
    lm = learn.mask[learn.live]
    meta = _x_compact({
        "cell": cell, "family": family, "draw": int(draw), "deploy_target": inst.deploy_target,
        "identity_hash": ident_hash, "code_hash": XENV_CODE_HASH[:16], "cache": "computed",
        "learn_env": {"name": learn.name, "S": learn.S, "A": learn.A, "live": int(learn.live.sum()),
                      "admissible_pairs": int(lm.sum()), "mean_admissible_actions": float(lm.sum(axis=1).mean())},
        "data": {"train_transitions": int(len(inst.train["state"])),
                 "train_episodes": int(np.unique(inst.train["episode_id"]).size),
                 "dev_transitions": int(len(inst.dev["state"])),
                 "dev_episodes": int(np.unique(inst.dev["episode_id"]).size)},
        "targets": [t[0] for t in inst.targets],
        "records": {"total": len(records), **dict(Counter(r["variant"] for r in records)),
                    "nonconverged": len(nonconv), "nonconverged_labels": nonconv},
        "evaluation": ev_stats, "fqe_rows": fqe.meta.get("fqe_rows", {}), "checks": checks,
        "timing": {"build_s": build_s, "methods_s": t2 - t1, "fqe_build_s": t3 - t2, "eval_s": t4 - t3,
                   "check_s": t5 - t4, "total_s": t5 - t0},
        "instance": inst.meta})
    XCACHE.put(key + "/meta", identity, pd.DataFrame({"meta_json": [json.dumps(meta, default=_json_default)]}))
    XCACHE.put(key, identity, df)
    back = XCACHE.get(key, identity)
    assert back is not None and list(back.columns) == XRESULT_COLUMNS and len(back) == len(df), f"{key}: cache write"
    for c in _XMETRIC_SOURCE:
        assert np.allclose(back[c].to_numpy(dtype=float), df[c].to_numpy(dtype=float), rtol=1e-14, atol=0.0,
                           equal_nan=True), f"{key}: cache round trip changed column {c}"
    assert (back["status"].astype(str).to_numpy() == df["status"].to_numpy()).all(), f"{key}: cache round trip"
    del inst, fqe, records, rows, df
    gc.collect()
    return back, meta


def x_check_results(df: pd.DataFrame) -> Dict[str, Any]:
    """Cross-draw invariants: one status vocabulary and an identical policy set across the draws of a family."""
    if df.empty:
        return {"rows": 0}
    bad = sorted(set(df["status"].astype(str)) - {"ok", "improper"})
    assert not bad, f"unexpected status values {bad}"
    ref = df["is_reference"].astype(bool)
    sets = df[~ref].groupby(["family", "cell", "draw"])["policy"].agg(lambda s: frozenset(s))
    for fam, s in sets.groupby(level="family"):
        assert s.nunique() == 1, f"{fam}: the evaluated policy set differs across cells / draws ({s.nunique()} sets)"
    tr = (df[ref].set_index(["cell", "draw", "target", "policy"])["return"].unstack("policy")
          .reindex(columns=list(XREF_POLICIES)))
    denom = (tr["REF optimum"] - tr["REF behaviour"]).astype(float)
    solver = df[ref & (df["policy"] != "REF behaviour")]["solver"].astype(str).value_counts()
    imp = df[df["status"] != "ok"].groupby(["family", "target"]).size()
    return {"rows": int(len(df)), "cell_draws": int(df[["cell", "draw"]].drop_duplicates().shape[0]),
            "policies_per_family": {str(f): len(next(iter(s))) for f, s in sets.groupby(level="family")},
            "improper_rows": int(imp.sum()),
            "improper_by_family_target": {f"{a}|{b}": int(v) for (a, b), v in imp.items()},
            "reference_solver_counts": {str(k): int(v) for k, v in solver.items()},
            "normalisation_denominator_below_1e-9": int((denom.abs() < 1e-9).sum()),
            "normalisation_denominator_missing": int(denom.isna().sum())}


def x_eta_seconds(remaining: Sequence[Tuple[str, int]], times: Dict[str, List[float]]) -> Optional[float]:
    """Remaining time from the mean computed-draw time per family (pooled mean for unseen families)."""
    pooled = [t for v in times.values() for t in v]
    if not pooled:
        return None
    g = float(np.mean(pooled))
    return float(sum(n * (float(np.mean(times[f])) if times.get(f) else g) for f, n in remaining))


def _x_fmt_s(sec: Optional[float]) -> str:
    if sec is None or not np.isfinite(sec):
        return "n/a"
    return f"{sec:.1f} s" if sec < 120 else (f"{sec / 60:.1f} min" if sec < 7200 else f"{sec / 3600:.2f} h")


# Objects whose source changes what a draw computes or stores (the checks are re-run on cached tables instead).
_XDRIVER_HASHED = [x_result_xcfg, x_cache_key, x_draw_identity, _x_pi_key, _x_compact, _x_record, x_collect_records,
                   _x_eval_metrics, x_result_row, _x_ref_desc, x_reference_lp, x_reference_optimum, x_reference_rows,
                   x_evaluate_targets, run_cell_draw]
XENV_CODE_HASH = code_hash(*XENV_CORE_CODE, *XENV_ENV_CODE, *XENV_METHOD_CODE, *_XDRIVER_HASHED)
print("Part II code hash:", XENV_CODE_HASH[:16], "| cache:", XENV_CACHE_DIR)

### II.6a Run all cells and draws

The loop below goes over every enabled cell (`X_CELLS`, filtered by the family switches) and each of its draws. It
shows a progress bar per cell (`SHOW_PROGRESS`; otherwise a line every ~20 % of the draws). After each cell it
prints the elapsed time and an ETA for the remaining cells. The ETA uses the mean time of the **computed** draws
of each family, so cached draws count as zero time and the ETA is an upper estimate when later draws are cached.
The outputs are `XRESULTS` (`xenv/xenv_results_all.csv.gz`, one row per draw × policy × target) and `XDRAW_META`
(`xenv/xenv_draw_meta.csv`: construction and validation reports, record counts, convergence, reference solvers,
invariant-check slacks and timings per draw). The cross-draw check asserts that every draw of a family evaluates
the same policy set, which the paired statistics in II.7 require.

In [ ]:
# ============================================================
# II.6a Driver: all cells and draws
# ============================================================
EXECLOG.mark("II.6 Driver: all cells and draws")
_xcells = [s for s in X_CELLS if EXPERIMENTS[f"xenv_{s['family']}"]]
if len(_xcells) != len(X_CELLS):
    print("Skipping cells whose family switch is off:", [s["cell"] for s in X_CELLS if s not in _xcells])
_xframes: List[pd.DataFrame] = []
_xmeta_rows: List[dict] = []
_xtimes: Dict[str, List[float]] = {}
_xcache_status: Counter = Counter()
_xt_all = time.perf_counter()
print(f"Part II driver: {len(_xcells)} cells, {sum(int(s['draws']) for s in _xcells)} draws")
for _ci, _spec in enumerate(_xcells, 1):
    _t_cell, _n_loaded, _nd = time.perf_counter(), 0, int(_spec["draws"])
    _bar = trange(_nd, desc=f"II.6 [{_ci}/{len(_xcells)}] {_spec['cell']}", leave=False, disable=not SHOW_PROGRESS)
    for _draw in _bar:
        _t0 = time.perf_counter()
        _df, _meta = run_cell_draw(_spec, _draw)
        _dt = time.perf_counter() - _t0
        _xframes.append(_df)
        _xmeta_rows.append(_x_flatten({**_meta, "session_wall_s": _dt}))
        _xcache_status[_meta["cache"]] += 1
        if _meta["cache"] == "loaded":
            _n_loaded += 1
        else:
            _xtimes.setdefault(_spec["family"], []).append(_dt)
        if SHOW_PROGRESS:
            _bar.set_postfix_str(f"{_dt:.1f}s {_meta['cache']}")
        elif (_draw + 1) % max(1, _nd // 5) == 0 or _draw + 1 == _nd:
            _left = [(_spec["family"], _nd - _draw - 1)]
            print(f"    {_spec['cell']} draw {_draw + 1}/{_nd} ({_meta['cache']}, {_dt:.1f} s); "
                  f"cell ETA {_x_fmt_s(x_eta_seconds(_left, _xtimes))}")
    _eta = x_eta_seconds([(s["family"], int(s["draws"])) for s in _xcells[_ci:]], _xtimes)
    print(f"  [{_ci}/{len(_xcells)}] {_spec['cell']}: {_nd} draws ({_n_loaded} cached) in "
          f"{_x_fmt_s(time.perf_counter() - _t_cell)} | total elapsed {_x_fmt_s(time.perf_counter() - _xt_all)} | "
          f"ETA remaining cells {_x_fmt_s(_eta) if _ci < len(_xcells) else '0 s'}")

XRESULTS = (pd.concat(_xframes, ignore_index=True) if _xframes else pd.DataFrame(columns=XRESULT_COLUMNS))
XDRAW_META = pd.DataFrame(_xmeta_rows)
del _xframes, _xmeta_rows
gc.collect()
assert list(XRESULTS.columns) == XRESULT_COLUMNS
XRESULTS.to_csv(XENV_TABLE_DIR / "xenv_results_all.csv.gz", index=False, compression="gzip")
XDRAW_META.to_csv(XENV_TABLE_DIR / "xenv_draw_meta.csv", index=False)
XDRIVER_SUMMARY = {"code_hash": XENV_CODE_HASH, "cells": len(_xcells), "draws": int(sum(_xcache_status.values())),
                   "draws_computed": int(_xcache_status["computed"]), "draws_loaded": int(_xcache_status["loaded"]),
                   "elapsed_s": time.perf_counter() - _xt_all, "xcache_written": len(XCACHE.written),
                   "xcache_loaded": len(XCACHE.loaded), **x_check_results(XRESULTS)}
if len(XDRAW_META):
    for _c in ("checks.max_slack_vs_optimum", "checks.max_slack_vs_admissible_optimum", "checks.projection_identity_worst"):
        XDRIVER_SUMMARY[f"max_{_c.split('.', 1)[1]}"] = float(XDRAW_META[_c].max()) if _c in XDRAW_META else None
    XDRIVER_SUMMARY["nonconverged_records"] = (int(XDRAW_META["records.nonconverged"].sum())
                                               if "records.nonconverged" in XDRAW_META else 0)
save_json(XDRIVER_SUMMARY, XENV_TABLE_DIR / "xenv_driver_summary.json")
print(json.dumps({k: v for k, v in XDRIVER_SUMMARY.items() if k != "improper_by_family_target"}, indent=1,
                 default=_json_default))
if len(XRESULTS):
    _dep = XRESULTS[XRESULTS["is_deploy_target"].astype(bool)
                    & XRESULTS["policy"].isin([*XREF_POLICIES, XCOMPARATOR])]
    display(_dep.groupby(["family", "target", "policy"])["return"].agg(["mean", "std", "count"]).round(4))
else:
    print("Part II driver: no cell enabled (all family switches off) - XRESULTS is empty.")

### II.6b Exact tabular learners vs Part I SGD learners

Part I trains tabular IQL, discrete BCQ and CQL by minibatch Adam on the full ICU-Sepsis dataset (Sections 9 and
15), with a fixed step budget and, for CQL and BCQ, soft target updates. Part II solves the same objectives
exactly: IQL is the in-sample expectile fixed point, BCQ is value iteration over the behaviour-thresholded action
set, and CQL is full-batch Adam on the exact empirical objective. Here the Part II implementations are run on the
same full `OFFLINE` data with the released mask. They are called through `run_xmethods`, so exactly the policies that enter the Part II tables are
checked. For every main seed we report, for the native and the masked exports: greedy-action agreement over live
states, over states visited in the data, and weighted by the exact occupancy of the Part I policy (mean rule); and
the exact return of both policies and their difference under both rules (`EVALUATOR`). `hp_match` records whether
the pre-declared Part II defaults equal the Part I settings (γ = 1 in both). This comparison is exploratory
(`PREREG_PART2`). Differences are expected where counts are small: SGD with a finite budget does not reach the
fixed point there, and unvisited states keep their initial values.

In [ ]:
# ============================================================
# II.6b Exact-tabular vs Part I SGD cross-check
# ============================================================
EXECLOG.mark("II.6 Exact-tabular cross-check")
_X_PART1_NAMES = {"iql": ("Tabular IQL", "Tabular IQL [masked greedy]"),
                  "bcq": (f"Tabular BCQ (tau={CFG.bcq_threshold})", f"Tabular BCQ (tau={CFG.bcq_threshold}) [masked greedy]"),
                  "cql": ("Tabular CQL", "Tabular CQL [masked greedy]")}
_X_PART2_METHOD = {"iql": "IQL", "bcq": "BCQ", "cql": "CQL"}


def x_action_agreement(pi_a: np.ndarray, pi_b: np.ndarray, states: np.ndarray,
                       weights: Optional[np.ndarray] = None) -> float:
    """Fraction (optionally weighted) of the selected states where the argmax actions agree (ties: lowest index)."""
    st = np.asarray(states, dtype=bool)
    if not st.any():
        return float("nan")
    agree = (np.asarray(pi_a).argmax(axis=1) == np.asarray(pi_b).argmax(axis=1))[st].astype(np.float64)
    if weights is None:
        return float(agree.mean())
    w = np.asarray(weights, dtype=np.float64)[st]
    return float((w * agree).sum() / w.sum()) if w.sum() > 0 else float("nan")


def _x_is_deterministic(pi: np.ndarray, live: np.ndarray) -> bool:
    return bool(np.all(np.asarray(pi)[live].max(axis=1) == 1.0))


def _x_part1_tabular_policies(algo: str, res: Dict[str, Any]) -> Dict[str, np.ndarray]:
    """The exports registered by Section 15 (`run_tabular_suite`) for one Part I tabular result."""
    if algo == "cql":
        return {"native": greedy_policy(res["q"], masked=False), "masked": greedy_policy(res["q"], masked=True)}
    return {"native": res["deterministic"], "masked": res["masked_greedy"]}


def _x_part1_hp(algo: str) -> dict:
    if algo == "iql":
        return {"expectile": CFG.iql_expectile, "beta": CFG.iql_beta, "max_weight": CFG.iql_max_weight,
                "gamma": CFG.gamma, "steps": CFG.iql_steps, "lr": CFG.iql_lr, "batch": CFG.batch_size}
    if algo == "bcq":
        return {"tau": CFG.bcq_threshold, "gamma": CFG.gamma, "steps": CFG.bcq_steps, "lr": CFG.bcq_lr,
                "batch": CFG.batch_size}
    return {"alpha": CFG.tab_cql_alpha, "temperature": CFG.cql_temperature, "gamma": CFG.gamma,
            "steps": CFG.tab_cql_steps, "lr": CFG.tab_cql_lr, "batch": CFG.batch_size}


def _x_hp_match(algo: str) -> bool:
    same = {"iql": [XCFG.iql_expectile_default == CFG.iql_expectile, XCFG.iql_beta == CFG.iql_beta,
                    XCFG.iql_max_weight == CFG.iql_max_weight],
            "bcq": [XCFG.bcq_tau_default == CFG.bcq_threshold],
            "cql": [XCFG.cql_alpha_default == CFG.tab_cql_alpha, CFG.cql_temperature == 1.0]}[algo]
    return bool(all(same) and CFG.gamma == 1.0)


def _x_icu_return(pi: np.ndarray, rule: str) -> Tuple[float, str]:
    try:
        return float(EVALUATOR.evaluate(pi, rule)["return"]), "ok"
    except ImproperPolicyError:
        return float("nan"), "improper"
    except EvaluationInvariantError:
        return float("nan"), "near-singular"


def x_tabular_crosscheck(seeds: Sequence[int]) -> pd.DataFrame:
    """Part II exact IQL / BCQ / CQL (defaults) on the full OFFLINE data vs the Part I SGD learners, per seed."""
    assert X_ICU.S == S_N and X_ICU.A == A_N
    part1: Dict[Tuple[str, str, int], np.ndarray] = {}
    for algo, names in _X_PART1_NAMES.items():
        for seed in seeds:
            pols = _x_part1_tabular_policies(algo, get_tab(algo, MAIN, seed))
            for variant, name in zip(("native", "masked"), names):
                pi = validate_policy(pols[variant], f"{name} seed={seed}")
                reg = REGISTRY.get(name)
                if reg is not None and int(seed) in reg.by_seed:
                    assert np.array_equal(reg.by_seed[int(seed)], pi), f"{name} seed={seed}: TAB and REGISTRY differ"
                part1[(algo, variant, int(seed))] = pi
    identity = {"kind": "xenv_tabular_crosscheck", "offline": MAIN.data_hash, "x_icu": X_ICU.identity(),
                "code": XENV_CODE_HASH, "crosscheck_code": code_hash(*_XCROSSCHECK_CODE), "xcfg": x_result_xcfg(),
                "diffusion": [CFG.diffusion_T, CFG.diffusion_beta_start, CFG.diffusion_beta_end], "run_mode": RUN_MODE,
                "part1": {f"{a}/{v}/{s}": array_hash(p) for (a, v, s), p in sorted(part1.items())},
                "part1_hp": {a: _x_part1_hp(a) for a in _X_PART1_NAMES}}
    key = "crosscheck/icu/offline"
    hit = XCACHE.get(key, identity)
    if hit is not None:
        return hit
    XCACHE.require_allowed(key)
    recs = run_xmethods(LearnerView.from_env(X_ICU), OFFLINE, "crosscheck|icu|offline")
    seen = LIVE & (MAIN.counts.sum(axis=1) > 0)
    rows = []
    for algo, method in _X_PART2_METHOD.items():
        for variant, name in zip(("native", "masked"), _X_PART1_NAMES[algo]):
            hits = [r for r in recs if r["method"] == method and r["variant"] == variant and r["is_default"]]
            assert len(hits) == 1, f"expected one default '{method} [{variant}]' record from run_xmethods, got {len(hits)}"
            r2 = hits[0]
            pi2 = validate_policy(r2["pi"], f"{method} [{variant}] (Part II)")
            j2 = {rule: _x_icu_return(pi2, rule) for rule in ("mean", "terminate")}
            for seed in seeds:
                pi1 = part1[(algo, variant, int(seed))]
                try:
                    occ = EVALUATOR.evaluate(pi1, "mean", return_vectors=True)["occupancy_vector"]
                except (ImproperPolicyError, EvaluationInvariantError):
                    occ = None
                j1 = {rule: _x_icu_return(pi1, rule) for rule in ("mean", "terminate")}
                row = {"algo": algo, "variant": variant, "seed": int(seed), "part2_policy": x_label(method, variant),
                       "part1_policy": name, "part2_hp": canonical_json(dict(r2["hp"] or {})),
                       "part1_hp": canonical_json(_x_part1_hp(algo)), "hp_match": _x_hp_match(algo),
                       "part2_converged": r2["info"].get("converged"),
                       "agreement_live": x_action_agreement(pi2, pi1, LIVE),
                       "agreement_seen": x_action_agreement(pi2, pi1, seen),
                       "agreement_occupancy_weighted": (float("nan") if occ is None
                                                        else x_action_agreement(pi2, pi1, LIVE, occ)),
                       "n_live": int(LIVE.sum()), "n_seen": int(seen.sum()),
                       "part2_deterministic": _x_is_deterministic(pi2, LIVE),
                       "part1_deterministic": _x_is_deterministic(pi1, LIVE)}
                for rule in ("mean", "terminate"):
                    row[f"return_part2_{rule}"], row[f"status_part2_{rule}"] = j2[rule]
                    row[f"return_part1_{rule}"], row[f"status_part1_{rule}"] = j1[rule]
                    row[f"diff_return_{rule}"] = j2[rule][0] - j1[rule][0]
                rows.append(row)
    del recs
    XCACHE.put(key, identity, pd.DataFrame(rows))
    return XCACHE.get(key, identity)


_XCROSSCHECK_CODE = [x_tabular_crosscheck, x_action_agreement, _x_is_deterministic, _x_part1_tabular_policies,
                     _x_part1_hp, _x_hp_match, _x_icu_return]
XTAB_CROSSCHECK = pd.DataFrame()
if EXPERIMENTS["xenv_tabular_crosscheck"] and EXPERIMENTS["main_tabular"]:
    XTAB_CROSSCHECK = x_tabular_crosscheck(CFG.seeds)
    XTAB_CROSSCHECK.to_csv(XENV_TABLE_DIR / "xenv_tabular_crosscheck.csv", index=False)
    if not XTAB_CROSSCHECK["hp_match"].astype(bool).all():
        print("NOTE: Part II default hyper-parameters differ from Part I for",
              sorted(XTAB_CROSSCHECK.loc[~XTAB_CROSSCHECK["hp_match"].astype(bool), "algo"].unique()),
              "- the cross-check is then descriptive of both changes.")
    _xc = XTAB_CROSSCHECK.assign(abs_diff_mean=XTAB_CROSSCHECK["diff_return_mean"].abs(),
                                 abs_diff_terminate=XTAB_CROSSCHECK["diff_return_terminate"].abs())
    display(_xc.groupby(["algo", "variant"]).agg(
        seeds=("seed", "nunique"), hp_match=("hp_match", "all"), agreement_live=("agreement_live", "mean"),
        agreement_seen=("agreement_seen", "mean"), agreement_occupancy=("agreement_occupancy_weighted", "mean"),
        mean_abs_diff_mean=("abs_diff_mean", "mean"), max_abs_diff_mean=("abs_diff_mean", "max"),
        mean_abs_diff_terminate=("abs_diff_terminate", "mean"),
        max_abs_diff_terminate=("abs_diff_terminate", "max")).round(5))
else:
    print("Exact-tabular cross-check not run (needs EXPERIMENTS['xenv_tabular_crosscheck'] and EXPERIMENTS['main_tabular']).")

XENV_DRIVER_CODE = [*_XDRIVER_HASHED, x_projection_identity, x_check_draw_rows, x_check_results, x_eta_seconds,
                    _x_fmt_s, _x_flatten, *_XCROSSCHECK_CODE]
print("Part II driver complete:", len(XRESULTS), "result rows,", len(XDRAW_META), "draws.")

## II.7 Cross-environment statistics: normalised scores, rliable aggregates, benchmark fidelity, preregistered tests

**Inputs.** `XRESULTS` (Section II.6: one row per cell × draw × policy × target, plus three reference rows per
target), `XDRAW_META`, and `X_FRONTIERS` / `PROP_CHECKS` (Section II.5). Every table and JSON below is written even
when Part II switches are off (header-only tables, "not run" entries), so the release map and the final gate never
depend on which sessions ran.

**Normalised score.** z = (J − J_β) / (J* − J_β) per (cell, draw, target), where J* is the target's unconstrained
optimum and J_β the behaviour policy evaluated on the same target (the "REF" rows). z = 1 is optimal, z = 0 matches
the behaviour policy, z < 0 is worse than it. z is undefined (NaN, counted) when J* − J_β < 1e-9. Improper
(non-absorbing) evaluations have no return; they are excluded from every aggregate and counted.

**Aggregates (rliable; Agarwal et al., NeurIPS 2021).** Tasks are environment cells; runs are dataset draws.
* The interquartile mean (IQM) is the mean of the empirical quantile function over [0.25, 0.75]. Sorted scores
  inside the interquartile range get weight 1 and the two boundary scores get fractional weight. The IQM therefore
  equals rliable's `trim_mean(·, 0.25)` exactly when n is a multiple of 4, and interpolates otherwise.
* Intervals are percentile intervals of a **stratified bootstrap**: draws are resampled with replacement within
  each cell (`XCFG.bootstrap` replicates, streams `np_rng("xenv", "boot", …)`). Statistics that share a stream key
  share resamples (common random numbers across policies), and paired statistics resample draws jointly.
  Garnet cells of one draw share the base MDP, K(s) and k_s (Section II.3c), so Garnet draws are resampled
  **jointly across cells** (one index vector per replicate, on the draws that are finite in every cell).
* The probability of improvement P(X > Y) is rliable's: within each cell, the Mann–Whitney probability over all
  draw pairs (ties count ½), averaged over cells.
* Performance profiles give the fraction of (cell, draw) units with z ≥ x, with pointwise stratified-bootstrap
  bands.
* Exact values within 1e-9 are treated as ties in every rank, argmax and comparison. Proposition P1 makes some
  policies analytically equal (for example a native policy and its projection under the mean rule), and
  floating-point noise must not decide their order.

**Benchmark fidelity (Gumbel, Garnet).** Each completion rule is compared with the truth, per draw, over the default
policy set (mean → `bench_mean`, terminate → `bench_terminate`):
* Kendall τ-b between benchmark values and true values;
* top-1 agreement and selection regret (in z units of the true target) of the benchmark's choice, both as
  expectations under uniform tie-breaking among the benchmark's top choices.

The same statistics computed between the two benchmark rules ("rule sensitivity", every family including
ICU-Sepsis) measure how much the rule alone reorders methods.

**Error decomposition.** J_true − J_bench = (J_true − J_trueRows+completion) + (J_trueRows+completion − J_bench):
* the semantics term: the completion rule is wrong about unsupported actions;
* the estimation-and-coverage term: finite hospital data and unvisited states.

In the oracle construction the benchmark *is* the true-rows completion, so all of its error is semantics.

**Preregistered tests (`PREREG_PART2`).**
* **H1:** per Gumbel B-est cell, a two-sided sign-flip test on per-draw τ(mean) − τ(terminate); Holm across cells.
  A benchmark that gives every policy the same value carries no ranking information and scores τ = 0 (Kendall
  τ-a); such draws stay in the test, are counted, and a sensitivity p-value without them is reported.
* **H2:** per family, a two-sided sign-flip test on paired differences of deployment z (treatment −
  `XCOMPARATOR`). ICU-Sepsis and Gumbel cells use independent streams on a fixed system, so their (cell, draw)
  units are pooled across cells. The Garnet cells of one draw share a base MDP, so the Garnet test unit is the
  draw block (the mean difference over the harm × ε cells of that draw). Holm over treatments × families; TOST at
  `XCFG.equivalence_margin_normalized` (raw and Holm-adjusted) on the same units; IQM and mean differences with
  stratified-bootstrap intervals.
* **H3:** Spearman ρ between |h| and the per-draw benchmark error of mean imputation (Garnet). The two-sided
  permutation p-value uses 10 000 permutations of |h| within (ε, draw) blocks: the harm levels of one draw share
  its base MDP, and the null leaves the error free to depend on ε and on the draw. Permutations within ε only and
  unrestricted permutations are reported as sensitivity checks, next to descriptive mask sizes and violation
  counts by (h, ε), because the harm also changes how many actions the benchmark admits.
* Sign-flip p-values are exact for n ≤ 16 and Monte Carlo otherwise (200 000 sign patterns, +1 correction); the
  reported attainable minimum follows the method used.

**Hyper-parameter selection.** For each method and variant, z is reported for three configurations: the declared
default; the one that maximises tabular FQE on the held-out dev episodes; and the one that maximises the deployment
return. The last is an oracle choice, an **upper bound** that no offline procedure attains. A unit in which the FQE
model rates no candidate proper falls back to the declared default (counted in `fqe_fallback_units`), so the three
columns describe the same units.

**Invariants asserted on every row.** An assertion failure is a pipeline bug, not a statistical finding.
* Complete design: every cell and draw of `X_CELLS` is present, with identical policy and target sets across draws.
* Unique keys; status in {ok, improper}; finite metrics on every ok row; three reference rows per target
  (optional on `fqe_dev`, which is used only for hyper-parameter selection).
* Value ordering: J ≤ J* for every policy, J*_admissible ≤ J*, and J ≤ J*_admissible for policies with zero
  expected violations.
* Targets that share the admissible rows (benchmark targets; true-system targets) give every policy the same
  P(any violation), and give admissible policies identical values (Proposition P5).

In [ ]:
# ============================================================
# II.7 Cross-environment statistics
# ============================================================
EXECLOG.mark("II.7 Cross-environment statistics")
import re
from matplotlib.colors import LinearSegmentedColormap

XS_FAMILIES = ("icu", "gumbel", "garnet")
XS_DEPLOY = {"icu": "bench_terminate", "gumbel": "true", "garnet": "true"}   # deployment target per family
XS_REF = ("REF optimum", "REF optimum (admissible)", "REF behaviour")
XS_REF_COLS = ("ref_optimum", "ref_optimum_admissible", "ref_behaviour")
XS_FQE = "fqe_dev"
XS_DRAW_BLOCKED_FAMILIES = ("garnet",)   # cells of one draw share the base MDP (part21c ('base', draw))
XS_SIGNFLIP_MC_DRAWS = 200_000           # sign_flip_test's Monte-Carlo size for n > 16 (Section 30 default)
XS_COMPLETIONS = {"mean": ("bench_mean", "true_rows_mean"), "terminate": ("bench_terminate", "true_rows_terminate")}
XS_RANK_COMPARISONS = (("mean", "bench_mean", "true"), ("terminate", "bench_terminate", "true"),
                       ("rule sensitivity", "bench_mean", "bench_terminate"))
# Targets that share state space, initial distribution and admissible rows (so P5 applies within each group).
XS_TARGET_GROUP = {"bench_mean": "benchmark", "bench_terminate": "benchmark", "true": "true system",
                   "true_rows_mean": "true system", "true_rows_terminate": "true system"}
XS_RESULT_COLUMNS = ["cell", "family", "draw", "policy", "method", "variant", "hp", "is_default", "method_family",
                     "target", "semantics", "status", "return", "avg_episode_length", "inadmissible_rate",
                     "expected_violations", "prob_any_violation", "max_state_inadmissible_mass", "absorb_death",
                     "absorb_goal"]
XS_METRICS = ["return", "avg_episode_length", "inadmissible_rate", "expected_violations", "prob_any_violation",
              "max_state_inadmissible_mass", "absorb_death", "absorb_goal"]
XS_NORM_COLUMNS = ["cell", "family", "draw", "target", "semantics", "policy", "method", "variant", "hp", "is_default",
                   "method_family", "status", "return", "ref_optimum", "ref_optimum_admissible", "ref_behaviour",
                   "denominator", "z", "expected_violations", "prob_any_violation", "max_state_inadmissible_mass"]
XS_DENOM_MIN = 1e-9        # normalisation undefined below this optimum-behaviour gap
# Exact values closer than XS_TIE_ATOL are ties: P1 makes e.g. a native policy and its projection equal under the
# mean rule, and floating-point noise (~1e-15) must not decide their order in ranks, argmaxes or P(X > Y).
XS_TIE_ATOL = 1e-9
XS_VALUE_TOL = 1e-6        # tolerance of value-ordering invariants (exact solves are accurate to ~1e-8)
XS_PROB_TOL = 1e-8
XS_ZERO_MASS = 1e-12
XS_PROFILE_GRID = np.round(np.linspace(-1.0, 1.0, 81), 6)
XS_PROFILE_TOL = 1e-9      # z >= x is evaluated as z >= x - 1e-9, so exactly optimal policies count at x = 1
XS_H3_PERMUTATIONS = 10_000
XS_BEHAVIOUR_POLICY = "Behaviour (true) [native]"
# Categorical slots in fixed order (CVD-validated reference palette); ink and muted greys for text and context.
XS_COLORS = ("#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
XS_INK, XS_MUTED, XS_NEUTRAL = "#0b0b0b", "#898781", "#f0efec"


# ---------- small utilities ----------
def xs_as_bool(s: pd.Series) -> pd.Series:
    """Boolean column after a CSV round trip ('True' / 'False' strings); missing -> False, anything else raises."""
    if s.dtype == bool:
        return s

    def conv(v):
        if isinstance(v, (bool, np.bool_)):
            return bool(v)
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return False
        if isinstance(v, (int, np.integer)) and int(v) in (0, 1):
            return bool(v)
        if isinstance(v, str) and v.strip().lower() in ("true", "false", "1", "0"):
            return v.strip().lower() in ("true", "1")
        raise ValueError(f"not a boolean flag: {v!r}")
    return s.map(conv).astype(bool)


def xs_as_frame(obj: Any) -> pd.DataFrame:
    """Section II.5 outputs as a DataFrame (None = not run)."""
    if obj is None:
        return pd.DataFrame()
    if isinstance(obj, pd.DataFrame):
        return obj
    if isinstance(obj, (list, tuple)):
        return pd.DataFrame(list(obj))
    raise TypeError(f"expected a DataFrame or a list of records, got {type(obj)}")


def _xs_num(tok: str):
    return int(tok) if re.fullmatch(r"[+-]?\d+", tok) else float(tok)


_XS_CELL_RE = {
    "icu": re.compile(r"icu\|(?P<construction>.+)"),
    "gumbel": re.compile(r"gumbel\|(?P<construction>[A-Za-z_-]+)\|n(?P<n_hosp>\d+)\|tau(?P<tau>[0-9.eE+-]+)"
                         r"\|eps(?P<eps>[0-9.eE+-]+)(?:\|phys-(?P<physician>[a-z-]+))?"),
    "garnet": re.compile(r"garnet\|h(?P<harm>[0-9.eE+-]+)\|eps(?P<eps>[0-9.eE+-]+)"),
}


def xs_parse_cell(cell: str) -> Dict[str, Any]:
    """Factors encoded in an X_CELLS key (formats of Section II.3); unknown formats raise."""
    cell = str(cell)
    fam = cell.split("|", 1)[0]
    m = _XS_CELL_RE[fam].fullmatch(cell) if fam in _XS_CELL_RE else None
    if m is None:
        raise ValueError(f"unrecognised Part II cell key {cell!r}")
    # Garnet benchmarks are built from hospital data like the Gumbel B-est construction.
    out = {"family": fam, "construction": "best", "n_hosp": np.nan, "tau": np.nan, "eps": np.nan, "harm": np.nan,
           "physician": "optimal" if fam == "gumbel" else ""}
    for k, v in m.groupdict().items():
        if v is not None:      # the optional Gumbel physician suffix is absent for the default (optimal) physician
            out[k] = v if k in ("construction", "physician") else _xs_num(v)
    return out


def xs_cell_label(cell: str) -> str:
    return str(cell).split("|", 1)[-1]


def xs_sort_cells(df: pd.DataFrame, order: Sequence[str], extra: Sequence[str] = ()) -> pd.DataFrame:
    """Rows in the design order of X_CELLS (stable, then by `extra` columns)."""
    if df.empty or "cell" not in df.columns:
        return df
    pos = df["cell"].map({c: i for i, c in enumerate(order)})
    assert pos.notna().all(), f"cells outside the design: {sorted(set(df.loc[pos.isna(), 'cell']))}"
    return (df.assign(_xs_order=pos).sort_values(["_xs_order", *extra], kind="mergesort")
            .drop(columns="_xs_order").reset_index(drop=True))


def xs_policy_role(policy: str) -> str:
    return "comparator" if policy == XCOMPARATOR else ("treatment" if policy in XTREATMENTS else "other")


def xs_fmt_ci(v: float, lo: float, hi: float, digits: int = 3) -> str:
    if not np.isfinite(v):
        return ""
    return f"{v:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"


# ---------- rliable-style estimators ----------
def xs_iqm_weights(n: int) -> np.ndarray:
    """Weights of the sorted sample in the IQM: the mean of the empirical quantile function over [0.25, 0.75]."""
    i = np.arange(n, dtype=np.float64)
    w = np.clip(np.minimum(i + 1.0, 0.75 * n) - np.maximum(i, 0.25 * n), 0.0, None)
    return w / (0.5 * n)


def xs_iqm_rows(x: np.ndarray) -> np.ndarray:
    """IQM of every row of a [B, n] array of finite values (NaN rows for n = 0)."""
    x = np.atleast_2d(np.asarray(x, dtype=np.float64))
    if x.shape[1] == 0:
        return np.full(x.shape[0], np.nan)
    return np.sort(x, axis=1) @ xs_iqm_weights(x.shape[1])


def xs_iqm(x: Sequence[float]) -> float:
    x = np.asarray(x, dtype=np.float64).ravel()
    if not len(x):
        return np.nan
    if not np.isfinite(x).all():
        raise ValueError("IQM input must be finite (missing scores are dropped and counted by the caller)")
    return float(xs_iqm_rows(x)[0])


def xs_snap_ties(v: np.ndarray, atol: float = XS_TIE_ATOL) -> np.ndarray:
    """Values within `atol` of their sorted neighbour form one tie class; each value is replaced by its class minimum."""
    v = np.asarray(v, dtype=np.float64)
    if len(v) < 2:
        return v.copy()
    order = np.argsort(v, kind="mergesort")
    s = v[order]
    new = np.r_[True, np.diff(s) > atol]
    out = np.empty_like(v)
    out[order] = s[new][np.cumsum(new) - 1]
    return out


def xs_top(v: np.ndarray) -> np.ndarray:
    """Mask of the values tied (within XS_TIE_ATOL) with the maximum."""
    s = xs_snap_ties(v)
    return s >= s.max()


def xs_poi_rows(x: np.ndarray, y: np.ndarray, atol: float = XS_TIE_ATOL) -> np.ndarray:
    """Row-wise P(X > Y) + 0.5 P(X = Y) over all (i, j) pairs of [B, n] and [B, m] arrays (Mann-Whitney; |x - y| <= atol
    is a tie)."""
    d = x[:, :, None] - y[:, None, :]
    return (d > atol).mean(axis=(1, 2)) + 0.5 * (np.abs(d) <= atol).mean(axis=(1, 2))


def xs_stratified_bootstrap(groups: Dict[str, np.ndarray], stat: Callable[[List[np.ndarray]], np.ndarray],
                            key: Tuple, n_boot: int, chunk: int = 500,
                            shared_draws: bool = False) -> Tuple[Any, np.ndarray]:
    """Stratified bootstrap (rliable): draws are resampled with replacement within each cell (stratum).

    `groups` maps cell -> [n_c, k] array of per-draw values (k paired columns, rows aligned by draw). `stat` maps a
    list of [B, n_c, k] arrays (one per cell) to a [B] or [B, G] array. Resample indices come from
    np_rng("xenv", "boot", *key, cell, n_c), so statistics with the same key and cell sizes share resamples.
    shared_draws=True (cells of one draw share a base MDP; groups from xs_cell_groups(..., aligned=True), rows sorted
    by draw) resamples the draws jointly: one index matrix np_rng("xenv", "boot", *key, "shared-draws", n) for every
    cell. Returns (point estimate, replicates [n_boot] or [n_boot, G])."""
    cells = [c for c, g in groups.items() if len(g)]
    if not cells:
        return np.nan, np.zeros(0)
    arrs = [np.asarray(groups[c], dtype=np.float64).reshape(len(groups[c]), -1) for c in cells]
    assert all(np.isfinite(a).all() for a in arrs), "bootstrap inputs must be finite"
    assert len({a.shape[1] for a in arrs}) == 1, "every stratum needs the same paired columns"
    point = np.asarray(stat([a[None] for a in arrs]))[0]
    if shared_draws:
        n = int(len(arrs[0]))
        assert all(len(a) == n for a in arrs), "shared-draw resampling needs draw-aligned groups of equal length"
        shared = np_rng("xenv", "boot", *key, "shared-draws", n).integers(0, n, size=(int(n_boot), n))
        idx = [shared] * len(arrs)
    else:
        idx = [np_rng("xenv", "boot", *key, c, int(len(a))).integers(0, len(a), size=(int(n_boot), len(a)))
               for c, a in zip(cells, arrs)]
    reps = [np.asarray(stat([a[i[s0:s0 + chunk]] for a, i in zip(arrs, idx)])) for s0 in range(0, int(n_boot), chunk)]
    reps = np.concatenate(reps, axis=0) if reps else np.zeros(0)
    return (float(point) if np.ndim(point) == 0 else point), reps


def xs_ci(reps: np.ndarray) -> Tuple[float, float]:
    """Percentile interval at level 1 - ALPHA of scalar bootstrap replicates."""
    reps = np.asarray(reps, dtype=np.float64)
    if reps.size == 0:
        return (np.nan, np.nan)
    lo, hi = np.quantile(reps, [ALPHA / 2, 1 - ALPHA / 2])
    return float(lo), float(hi)


def xs_stat_iqm(col: int = 0) -> Callable:
    return lambda parts: xs_iqm_rows(np.concatenate([p[..., col] for p in parts], axis=1))


def xs_stat_mean(col: int = 0) -> Callable:
    return lambda parts: np.concatenate([p[..., col] for p in parts], axis=1).mean(axis=1)


def xs_stat_iqm_diff(parts: List[np.ndarray]) -> np.ndarray:
    return xs_stat_iqm(0)(parts) - xs_stat_iqm(1)(parts)


def xs_stat_mean_diff(parts: List[np.ndarray]) -> np.ndarray:
    return np.concatenate([p[..., 0] - p[..., 1] for p in parts], axis=1).mean(axis=1)


def xs_stat_poi(parts: List[np.ndarray]) -> np.ndarray:
    """rliable probability of improvement: within-cell Mann-Whitney P(col 0 > col 1), averaged over cells."""
    return np.mean([xs_poi_rows(p[..., 0], p[..., 1]) for p in parts], axis=0)


def xs_stat_profile(grid: np.ndarray) -> Callable:
    g = np.asarray(grid, dtype=np.float64)[None, None, :] - XS_PROFILE_TOL
    return lambda parts: (np.concatenate([p[..., 0] for p in parts], axis=1)[:, :, None] >= g).mean(axis=1)


def xs_cell_groups(W: pd.DataFrame, cols: Sequence[str], aligned: bool = False) -> Dict[str, np.ndarray]:
    """cell -> [n_c, len(cols)] rows of a (cell, draw)-indexed frame on which every column is finite.

    aligned=True (cells of one draw share a base MDP) keeps only the draws whose rows are finite in every cell
    present in W and returns each cell's rows sorted by draw, so every group has the same length and row i of every
    group is the same draw (for xs_stratified_bootstrap(..., shared_draws=True))."""
    out = {}
    if W.empty:
        return out
    V = W[list(cols)]
    if aligned:
        fin = pd.Series(np.isfinite(V.to_numpy(dtype=np.float64)).all(axis=1), index=V.index)
        ok = fin.unstack("cell", fill_value=False).astype(bool)          # draw x cell
        keep = np.sort(ok.index[ok.all(axis=1)].to_numpy())
        if not len(keep):
            return out
        for cell in dict.fromkeys(V.index.get_level_values("cell")):
            out[str(cell)] = V.xs(cell, level="cell").loc[keep].to_numpy(dtype=np.float64)
        return out
    for cell, g in V.groupby(level="cell", sort=False):
        v = g.to_numpy(dtype=np.float64)
        v = v[np.isfinite(v).all(axis=1)]
        if len(v):
            out[str(cell)] = v
    return out


def xs_min_attainable_p(n: int) -> float:
    """Smallest two-sided sign-flip p-value: 2 / 2^n by exact enumeration (n <= 16), and at least the Monte-Carlo
    floor 1 / (draws + 1) otherwise (sign_flip_test uses 200 000 sign patterns with the +1 correction)."""
    exact = min_attainable_p(n, "two-sided")
    return exact if n <= 16 else max(exact, 1.0 / (XS_SIGNFLIP_MC_DRAWS + 1))


def xs_spearman(x: Sequence[float], y: Sequence[float]) -> float:
    rx, ry = stats.rankdata(x), stats.rankdata(y)
    if len(rx) < 3 or np.ptp(rx) == 0 or np.ptp(ry) == 0:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def xs_permutation_spearman(x: np.ndarray, y: np.ndarray, strata: Optional[np.ndarray], n_perm: int,
                            key: Tuple) -> Dict[str, Any]:
    """Two-sided permutation p-value of Spearman's rho (y permuted within strata if given; +1 correction)."""
    x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)
    n = len(x)
    rho = xs_spearman(x, y)
    if not np.isfinite(rho):
        return {"rho": rho, "p_value": np.nan, "n": int(n), "n_permutations": 0}
    rx, ry = stats.rankdata(x), stats.rankdata(y)
    rx, ry = rx - rx.mean(), ry - ry.mean()
    norm_ = math.sqrt(float(rx @ rx) * float(ry @ ry))
    strata = np.zeros(n) if strata is None else np.asarray(strata)
    rng = np_rng("xenv", "H3", *key)
    idx = np.tile(np.arange(n), (int(n_perm), 1))
    for s in np.unique(strata):
        gi = np.flatnonzero(strata == s)
        idx[:, gi] = gi[np.argsort(rng.random((int(n_perm), len(gi))), axis=1)]
    null = (ry[idx] @ rx) / norm_
    p = (1 + int(np.sum(np.abs(null) >= abs(rho) - 1e-12))) / (1 + int(n_perm))
    return {"rho": rho, "p_value": float(p), "n": int(n), "n_permutations": int(n_perm),
            "n_strata": int(len(np.unique(strata)))}


def xs_selftests() -> Dict[str, Any]:
    """Millisecond checks of the estimators against independent definitions."""
    rng = np_rng("xenv", "stats-selftest")
    out = {}
    out["iqm_equals_trim_mean_when_n_mod_4_is_0"] = max(
        abs(xs_iqm(x) - float(stats.trim_mean(x, 0.25))) for x in (rng.normal(size=n) for n in (4, 8, 12, 40)))
    s = np.sort(rng.normal(size=9))
    out["iqm_interpolated_n9"] = abs(xs_iqm(s[::-1]) - (0.75 * s[2] + s[3:6].sum() + 0.75 * s[6]) / 4.5)
    out["iqm_n1_n2"] = abs(xs_iqm([3.0]) - 3.0) + abs(xs_iqm([1.0, 2.0]) - 1.5)
    x, y = rng.integers(0, 4, size=7).astype(float), rng.integers(0, 4, size=5).astype(float)
    out["poi_matches_probability_of_improvement"] = abs(float(xs_poi_rows(x[None], y[None])[0])
                                                        - probability_of_improvement(x, y))
    _, reps = xs_stratified_bootstrap({"a": np.zeros(5), "b": np.ones(3)}, xs_stat_mean(), ("selftest",), 200)
    out["stratified_bootstrap_keeps_strata"] = float(np.abs(reps - 3 / 8).max())
    xa = rng.normal(size=7)
    _, reps = xs_stratified_bootstrap({"a": xa, "b": xa + 1.0}, lambda p: p[1][..., 0].mean(1) - p[0][..., 0].mean(1),
                                      ("selftest",), 200, shared_draws=True)
    out["shared_draw_bootstrap_resamples_draws_jointly"] = float(np.abs(reps - 1.0).max())
    u, v = rng.normal(size=30), rng.normal(size=30)
    out["spearman_matches_scipy"] = abs(xs_spearman(u, v) - float(stats.spearmanr(u, v)[0]))
    t = xs_snap_ties(np.array([0.5 + 1e-15, 0.2, 0.5, 0.7]))
    out["tie_snapping"] = float(abs(t[0] - t[2]) + abs(t[1] - 0.2) + abs(t[3] - 0.7))
    bad = {k: v for k, v in out.items() if not v < 1e-12}
    assert not bad, f"II.7 estimator self-tests failed: {bad}"
    return out


# ---------- input preparation and structural invariants ----------
def xs_prepare_results(results: Optional[pd.DataFrame]) -> pd.DataFrame:
    """Type-normalised copy of XRESULTS (schema, keys, status and reference labels asserted)."""
    if results is None or not len(results):
        return pd.DataFrame(columns=XS_RESULT_COLUMNS + ["is_ref"])
    missing = [c for c in XS_RESULT_COLUMNS if c not in results.columns]
    assert not missing, f"XRESULTS lacks the columns {missing}"
    xr = results[XS_RESULT_COLUMNS].copy()
    for c in ("cell", "family", "policy", "method", "variant", "hp", "method_family", "target", "semantics", "status"):
        xr[c] = xr[c].astype(str)
    xr["draw"] = pd.to_numeric(xr["draw"], errors="raise").astype(np.int64)
    for c in XS_METRICS:
        xr[c] = pd.to_numeric(xr[c], errors="raise").astype(np.float64)
    # Reference rows (Section II.6): policy in XS_REF, method = the policy label, variant "reference",
    # is_reference True. The label set defines them; the driver's flag must agree when present.
    xr["is_ref"] = xr["policy"].isin(XS_REF)
    if "is_reference" in results.columns:
        flag = xs_as_bool(results["is_reference"]).to_numpy()
        assert np.array_equal(flag, xr["is_ref"].to_numpy()), "is_reference disagrees with the REF policy labels"
    assert xr.loc[xr["is_ref"], "method"].isin(["REF", *XS_REF]).all(), "reference rows with a learner method label"
    # is_default is meaningful (and strictly parsed) for learned policies only; reference rows are never "default".
    flags = np.zeros(len(xr), dtype=bool)
    flags[~xr["is_ref"].to_numpy()] = xs_as_bool(xr.loc[~xr["is_ref"], "is_default"]).to_numpy()
    xr["is_default"] = flags
    bad = sorted(set(xr["status"]) - {"ok", "improper"})
    assert not bad, f"unexpected Part II evaluation status values {bad}"
    ok = xr["status"].eq("ok")
    fin = np.isfinite(xr.loc[ok, XS_METRICS].to_numpy(dtype=np.float64)).all(axis=0)
    assert fin.all(), f"'ok' Part II rows with non-finite {[c for c, f in zip(XS_METRICS, fin) if not f]}"
    xr.loc[~ok, XS_METRICS] = np.nan          # improper evaluations carry no metrics
    assert xr["cell"].str.split("|", n=1).str[0].eq(xr["family"]).all(), "family column disagrees with the cell key"
    assert set(xr["family"]) <= set(XS_FAMILIES), sorted(set(xr["family"]))
    dup = xr.duplicated(["cell", "draw", "target", "policy"])
    assert not dup.any(), f"{int(dup.sum())} duplicated (cell, draw, target, policy) rows in XRESULTS"
    assert (xr.groupby(["cell", "target"])["semantics"].nunique() == 1).all(), "a target has two semantics in a cell"
    assert not xr.loc[~xr["is_ref"], "method"].isin(["REF", *XS_REF]).any(), "a learned policy carries a reference label"
    assert not xr.loc[~xr["is_ref"], "variant"].eq("reference").any(), "a learned policy has variant 'reference'"
    assert (xr.loc[ok, "expected_violations"] >= -1e-12).all() and (xr.loc[ok, "prob_any_violation"] <= 1 + 1e-9).all()
    return xr


def xs_design_checks(xr: pd.DataFrame, cells_spec: Optional[Sequence[dict]], experiments: Dict[str, bool]) -> Dict[str, Any]:
    """Complete design: every X_CELLS cell and draw present, identical target / policy sets, reference rows."""
    out = {"families_enabled": [f for f in XS_FAMILIES if experiments[f"xenv_{f}"]],
           "families_present": [f for f in XS_FAMILIES if f in set(xr["family"])], "rows": int(len(xr)), "cells": {}}
    spec = {str(s["cell"]): s for s in (cells_spec or [])}
    if xr.empty:
        assert not spec, f"X_CELLS lists {len(spec)} cells but XRESULTS is empty (run Section II.6)"
        return out
    assert cells_spec is not None, "XRESULTS has rows but X_CELLS is undefined"
    unknown, absent = sorted(set(xr["cell"]) - set(spec)), sorted(set(spec) - set(xr["cell"]))
    assert not unknown, f"XRESULTS cells missing from X_CELLS: {unknown}"
    assert not absent, f"X_CELLS cells without results: {absent}"
    for fam in out["families_present"]:
        assert fam in out["families_enabled"], f"results for family '{fam}' although EXPERIMENTS['xenv_{fam}'] is False"
    for cell, g in xr.groupby("cell", sort=False):
        fam = str(g["family"].iloc[0])
        draws = set(g["draw"].tolist())
        assert draws == set(range(int(spec[cell]["draws"]))), \
            f"{cell}: draws {sorted(draws)} differ from the design 0..{int(spec[cell]['draws']) - 1}"
        tsets = g.groupby("draw")["target"].agg(lambda s: "\n".join(sorted(set(s))))
        assert tsets.nunique() == 1, f"{cell}: target sets differ across draws"
        targets = tsets.iloc[0].split("\n")
        assert XS_DEPLOY[fam] in targets, f"{cell}: deployment target '{XS_DEPLOY[fam]}' missing"
        learned = g[~g["is_ref"]]
        psets = learned.groupby(["draw", "target"])["policy"].agg(lambda s: "\n".join(sorted(s)))
        assert len(psets) == len(draws) * len(targets) and psets.nunique() == 1, \
            f"{cell}: the learned-policy set differs across draws or targets"
        nref = g[g["is_ref"]].groupby(["draw", "target"])["policy"].nunique()
        for t in targets:
            k = nref.xs(t, level="target") if t in nref.index.get_level_values("target") else pd.Series(dtype=int)
            if t == XS_FQE and k.empty:
                continue
            assert len(k) == len(draws) and (k == len(XS_REF)).all(), f"{cell}: reference rows incomplete on '{t}'"
        out["cells"][cell] = {"family": fam, "draws": len(draws), "targets": targets,
                              "learned_policies": len(psets.iloc[0].split("\n")),
                              "default_policies": int(learned.loc[learned["is_default"], "policy"].nunique())}
    return out


def xs_draw_meta_checks(meta: Optional[pd.DataFrame], xr: pd.DataFrame) -> Dict[str, Any]:
    meta = xs_as_frame(meta)
    out = {"rows": int(len(meta)), "columns": [str(c) for c in meta.columns]}
    if meta.empty:
        return out
    if {"cell", "draw"} <= set(meta.columns):
        have = set(zip(meta["cell"].astype(str), pd.to_numeric(meta["draw"]).astype(int)))
        out["result_draws_without_meta"] = int(len(set(zip(xr["cell"], xr["draw"].astype(int))) - have))
        if "deploy_target" in meta.columns:
            fam = meta["cell"].astype(str).str.split("|", n=1).str[0]
            wrong = meta.loc[meta["deploy_target"].astype(str) != fam.map(XS_DEPLOY), "cell"]
            assert wrong.empty, f"deployment targets disagree with XS_DEPLOY for {sorted(set(wrong))}"
    return out


def xs_normalized_scores(xr: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Learned-policy rows with their target's reference values and z = (J - J_b) / (J* - J_b)."""
    if xr.empty:
        return pd.DataFrame(columns=XS_NORM_COLUMNS), {"rows": 0}
    ref = xr[xr["is_ref"]]
    idx = ["cell", "draw", "target", "policy"]
    R = ref.set_index(idx)["return"].unstack("policy").reindex(columns=list(XS_REF))
    st = ref.set_index(idx)["status"].unstack("policy").reindex(columns=list(XS_REF))
    opt_bad = int((st[list(XS_REF[:2])] != "ok").to_numpy().sum())
    assert opt_bad == 0, f"{opt_bad} reference optima are not 'ok' evaluations"
    R.columns = list(XS_REF_COLS)
    pol = xr[~xr["is_ref"]].merge(R.reset_index(), on=["cell", "draw", "target"], how="left", validate="many_to_one")
    denom = pol["ref_optimum"] - pol["ref_behaviour"]
    defined = denom >= XS_DENOM_MIN
    pol["denominator"] = denom
    pol["z"] = np.where(defined, (pol["return"] - pol["ref_behaviour"]) / denom.where(defined), np.nan)
    grp = pol.drop_duplicates(["cell", "draw", "target"])
    small = grp["denominator"].notna() & (grp["denominator"] < XS_DENOM_MIN)
    report = {
        "rows": int(len(pol)), "z_finite_rows": int(np.isfinite(pol["z"]).sum()),
        "improper_rows": int(pol["status"].ne("ok").sum()),
        "improper_rows_by_family_target": {f"{f}|{t}": int(n) for (f, t), n in
                                           pol[pol["status"].ne("ok")].groupby(["family", "target"]).size().items()},
        "groups": int(len(grp)),
        "groups_denominator_below_1e-9": int(small.sum()),
        "rows_denominator_below_1e-9": int((pol["denominator"].notna() & ~defined).sum()),
        "groups_denominator_below_1e-9_by_target": {str(t): int(n) for t, n in grp[small].groupby("target").size().items()},
        "groups_without_reference_by_target": {str(t): int(n) for t, n in
                                               grp[grp["ref_optimum"].isna()].groupby("target").size().items()},
        "groups_reference_behaviour_not_ok": int((st["REF behaviour"] != "ok").sum()),
    }
    return pol[XS_NORM_COLUMNS].reset_index(drop=True), report


def xs_reference_checks(norm: pd.DataFrame) -> Dict[str, Any]:
    """Value ordering: J <= J*, J*_adm <= J*, J_b <= J*, and J <= J*_adm for policies without expected violations."""
    if norm.empty:
        return {"rows_checked": 0}
    ok = norm[norm["status"].eq("ok") & norm["ref_optimum"].notna()]
    scale = np.maximum(1.0, ok["ref_optimum"].abs())
    zero = ok["expected_violations"] <= XS_ZERO_MASS
    grp = norm[norm["ref_optimum"].notna()].drop_duplicates(["cell", "draw", "target"])
    gscale = np.maximum(1.0, grp["ref_optimum"].abs())
    out = {"rows_checked": int(len(ok)), "rows_without_expected_violations": int(zero.sum()),
           "max_excess_over_optimum": float(((ok["return"] - ok["ref_optimum"]) / scale).max()),
           "max_excess_over_admissible_optimum_zero_violation_policies":
               float(((ok["return"] - ok["ref_optimum_admissible"]) / scale)[zero].max()) if zero.any() else -np.inf,
           "max_excess_admissible_optimum_over_optimum":
               float(((grp["ref_optimum_admissible"] - grp["ref_optimum"]) / gscale).max()),
           "max_excess_behaviour_over_optimum": float(((grp["ref_behaviour"] - grp["ref_optimum"]) / gscale).max())}
    bad = {k: v for k, v in out.items() if k.startswith("max_") and np.isfinite(v) and v > XS_VALUE_TOL}
    assert not bad, f"value-ordering invariants violated (tolerance {XS_VALUE_TOL}): {bad}"
    beh = norm[norm["policy"].eq(XS_BEHAVIOUR_POLICY) & norm["status"].eq("ok") & norm["ref_behaviour"].notna()]
    if len(beh):   # reported only: the driver's behaviour record should coincide with the REF behaviour row
        out["behaviour_record_max_abs_diff_to_ref_behaviour"] = float((beh["return"] - beh["ref_behaviour"]).abs().max())
        out["behaviour_record_max_abs_z"] = float(np.nanmax(np.abs(beh["z"].to_numpy()))) if beh["z"].notna().any() else np.nan
    return out


def xs_semantic_group_checks(norm: pd.DataFrame) -> Dict[str, Any]:
    """Targets sharing the admissible rows: equal P(any violation) for every policy; equal value if admissible (P5)."""
    out = {}
    if norm.empty:
        return out
    sub = norm[norm["target"].isin(list(XS_TARGET_GROUP)) & norm["status"].eq("ok")]
    for grp_name in sorted(set(XS_TARGET_GROUP.values())):
        g = sub[sub["target"].map(XS_TARGET_GROUP) == grp_name]
        if g.empty:
            continue
        w = g.set_index(["cell", "draw", "policy", "target"])
        pa, ms, rt = (w[c].unstack("target") for c in ("prob_any_violation", "max_state_inadmissible_mass", "return"))
        multi = pa.notna().sum(axis=1) >= 2
        adm = ((ms <= XS_ZERO_MASS) | ms.isna()).all(axis=1) & (ms.notna().sum(axis=1) >= 2)
        spread_pa = (pa.max(axis=1) - pa.min(axis=1))[multi]
        spread_rt = (rt.max(axis=1) - rt.min(axis=1))[adm]
        out[grp_name] = {"policy_rows_compared": int(multi.sum()), "admissible_policy_rows": int(adm.sum()),
                         "max_prob_any_violation_spread": float(spread_pa.max()) if len(spread_pa) else 0.0,
                         "max_admissible_value_spread": float(spread_rt.max()) if len(spread_rt) else 0.0}
        assert out[grp_name]["max_prob_any_violation_spread"] <= XS_PROB_TOL, (grp_name, out[grp_name])
        assert out[grp_name]["max_admissible_value_spread"] <= XS_VALUE_TOL, (grp_name, out[grp_name])
    return out


# ---------- deployment scores, main table, profiles, probability of improvement ----------
def xs_deploy_scores(norm: pd.DataFrame, defaults_only: bool = True) -> pd.DataFrame:
    """Rows on each family's deployment target (default-hp policies unless defaults_only=False)."""
    if norm.empty:
        return norm
    d = norm[norm["target"].eq(norm["family"].map(XS_DEPLOY))]
    return d[d["is_default"]] if defaults_only else d


def xs_wide(d: pd.DataFrame, value: str) -> pd.DataFrame:
    """(cell, draw) x policy frame of one column."""
    return d.set_index(["cell", "draw", "policy"])[value].unstack("policy")


XS_MAIN_LONG_COLUMNS = ["family", "deploy_target", "policy", "method", "variant", "method_family", "role", "n_cells",
                        "n_units", "n_scores", "n_missing", "iqm", "iqm_ci_low", "iqm_ci_high", "median", "mean",
                        "mean_expected_violations", "mean_prob_any_violation", "comparator", "n_paired",
                        "poi_vs_comparator", "poi_ci_low", "poi_ci_high"]


def xs_main_long(norm: pd.DataFrame, n_boot: int) -> pd.DataFrame:
    """One row per (family, default policy): IQM [CI], median, violations, P(improvement vs XCOMPARATOR) [CI]."""
    dep = xs_deploy_scores(norm)
    if dep.empty:
        return pd.DataFrame(columns=XS_MAIN_LONG_COLUMNS)
    meta = dep.drop_duplicates("policy").set_index("policy")
    rows = []
    for fam in XS_FAMILIES:
        d = dep[dep["family"] == fam]
        if d.empty:
            continue
        Z, V, PA = (xs_wide(d, c) for c in ("z", "expected_violations", "prob_any_violation"))
        key = ("main", fam, XS_DEPLOY[fam])
        blk = fam in XS_DRAW_BLOCKED_FAMILIES
        for pol in Z.columns:
            g = xs_cell_groups(Z, [pol], aligned=blk)
            pooled = np.concatenate([v[:, 0] for v in g.values()]) if g else np.zeros(0)
            iqm, reps = xs_stratified_bootstrap(g, xs_stat_iqm(), key, n_boot, shared_draws=blk)
            lo, hi = xs_ci(reps)
            v_ok, pa_ok = V[pol].dropna(), PA[pol].dropna()
            r = {"family": fam, "deploy_target": XS_DEPLOY[fam], "policy": pol, "method": meta.at[pol, "method"],
                 "variant": meta.at[pol, "variant"], "method_family": meta.at[pol, "method_family"],
                 "role": xs_policy_role(pol), "n_cells": int(Z.index.get_level_values("cell").nunique()),
                 "n_units": int(len(Z)), "n_scores": int(len(pooled)), "n_missing": int(len(Z) - len(pooled)),
                 "iqm": iqm, "iqm_ci_low": lo, "iqm_ci_high": hi,
                 "median": float(np.median(pooled)) if len(pooled) else np.nan,
                 "mean": float(pooled.mean()) if len(pooled) else np.nan,
                 "mean_expected_violations": float(v_ok.mean()) if len(v_ok) else np.nan,
                 "mean_prob_any_violation": float(pa_ok.mean()) if len(pa_ok) else np.nan,
                 "comparator": XCOMPARATOR, "n_paired": 0, "poi_vs_comparator": np.nan,
                 "poi_ci_low": np.nan, "poi_ci_high": np.nan}
            if XCOMPARATOR in Z.columns:
                gp = xs_cell_groups(Z, [pol, XCOMPARATOR], aligned=blk)
                poi, preps = xs_stratified_bootstrap(gp, xs_stat_poi, key, n_boot, shared_draws=blk)
                r["n_paired"] = int(sum(len(v) for v in gp.values()))
                r["poi_vs_comparator"] = poi
                r["poi_ci_low"], r["poi_ci_high"] = xs_ci(preps)
            rows.append(r)
    return pd.DataFrame(rows, columns=XS_MAIN_LONG_COLUMNS)


def xs_policy_order(main_long: pd.DataFrame) -> List[str]:
    """Default policies ordered by the mean of their per-family IQMs (descending; undefined last)."""
    if main_long.empty:
        return []
    m = main_long.groupby("policy")["iqm"].mean()
    return list(m.sort_values(ascending=False, na_position="last", kind="mergesort").index)


def xs_main_wide(main_long: pd.DataFrame) -> pd.DataFrame:
    """table_xenv_main.csv: one row per default policy, one column block per family (its deployment target)."""
    base_cols = ["policy", "method", "variant", "method_family", "role"]
    if main_long.empty:
        return pd.DataFrame(columns=base_cols)
    out = main_long.drop_duplicates("policy").set_index("policy").loc[xs_policy_order(main_long),
                                                                       base_cols[1:]].reset_index()
    for fam in XS_FAMILIES:
        f = main_long[main_long["family"] == fam].set_index("policy").reindex(out["policy"])
        if f["iqm"].isna().all() and f["n_units"].isna().all():
            continue
        out[f"{fam}_target"] = XS_DEPLOY[fam]
        out[f"{fam}_iqm_ci"] = [xs_fmt_ci(a, b, c) for a, b, c in zip(f["iqm"], f["iqm_ci_low"], f["iqm_ci_high"])]
        for c in ("iqm", "iqm_ci_low", "iqm_ci_high", "median", "mean_expected_violations", "mean_prob_any_violation",
                  "poi_vs_comparator", "poi_ci_low", "poi_ci_high", "n_scores", "n_missing"):
            out[f"{fam}_{c}"] = f[c].to_numpy()
        out[f"{fam}_poi_ci"] = [xs_fmt_ci(a, b, c) for a, b, c in
                                zip(f["poi_vs_comparator"], f["poi_ci_low"], f["poi_ci_high"])]
    return out


XS_PROFILE_COLUMNS = ["family", "deploy_target", "policy", "role", "threshold", "fraction_at_least", "ci_low", "ci_high",
                      "n_scores"]


def xs_performance_profiles(norm: pd.DataFrame, n_boot: int, grid: np.ndarray = XS_PROFILE_GRID) -> pd.DataFrame:
    """Fraction of (cell, draw) units with z >= x per default policy and family, with stratified-bootstrap bands."""
    dep = xs_deploy_scores(norm)
    frames = []
    for fam in XS_FAMILIES:
        d = dep[dep["family"] == fam] if len(dep) else dep
        if d.empty:
            continue
        Z = xs_wide(d, "z")
        blk = fam in XS_DRAW_BLOCKED_FAMILIES
        for pol in Z.columns:
            g = xs_cell_groups(Z, [pol], aligned=blk)
            if not g:
                continue
            point, reps = xs_stratified_bootstrap(g, xs_stat_profile(grid), ("main", fam, XS_DEPLOY[fam]), n_boot,
                                                  shared_draws=blk)
            lo, hi = (np.quantile(reps, [ALPHA / 2, 1 - ALPHA / 2], axis=0) if len(reps)
                      else (np.full(len(grid), np.nan),) * 2)
            frames.append(pd.DataFrame({"family": fam, "deploy_target": XS_DEPLOY[fam], "policy": pol,
                                        "role": xs_policy_role(pol), "threshold": grid, "fraction_at_least": point,
                                        "ci_low": lo, "ci_high": hi, "n_scores": int(sum(len(v) for v in g.values()))}))
    return pd.concat(frames, ignore_index=True)[XS_PROFILE_COLUMNS] if frames else pd.DataFrame(columns=XS_PROFILE_COLUMNS)


def xs_poi_matrix(norm: pd.DataFrame, order: Sequence[str]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Pooled rliable P(row > column) between default policies on the deployment targets, averaged over all cells."""
    cols = ["policy_x", "policy_y", "poi", "n_cells"]
    dep = xs_deploy_scores(norm)
    if dep.empty:
        return pd.DataFrame(columns=cols), {}
    Z = xs_wide(dep, "z")
    pols = [p for p in order if p in Z.columns] + [p for p in Z.columns if p not in set(order)]
    P = len(pols)
    acc, cnt = np.zeros((P, P)), np.zeros((P, P))
    for _, g in Z[pols].groupby(level="cell", sort=False):
        V = g.to_numpy(dtype=np.float64)
        F = np.isfinite(V)
        Vf = np.where(F, V, 0.0)
        valid = F[:, None, :, None] & F[None, :, None, :]                     # [draw a, draw b, policy i, policy j]
        diff = Vf[:, None, :, None] - Vf[None, :, None, :]
        score = (diff > XS_TIE_ATOL) + 0.5 * (np.abs(diff) <= XS_TIE_ATOL)
        num, den = (score * valid).sum(axis=(0, 1)), valid.sum(axis=(0, 1))
        has = den > 0
        acc[has] += num[has] / den[has]
        cnt[has] += 1
    M = np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan)
    both = np.isfinite(M) & np.isfinite(M.T)
    check = {"policies": P, "max_complementarity_error": float(np.abs(M + M.T - 1.0)[both].max()) if both.any() else 0.0,
             "max_diagonal_error": float(np.nanmax(np.abs(np.diag(M) - 0.5))) if np.isfinite(np.diag(M)).any() else 0.0}
    assert check["max_complementarity_error"] <= 1e-9 and check["max_diagonal_error"] <= 1e-12, check
    ii, jj = np.meshgrid(np.arange(P), np.arange(P), indexing="ij")
    out = pd.DataFrame({"policy_x": np.asarray(pols, dtype=object)[ii.ravel()],
                        "policy_y": np.asarray(pols, dtype=object)[jj.ravel()],
                        "poi": M.ravel(), "n_cells": cnt.ravel().astype(int)})
    return out, check


# ---------- benchmark fidelity ----------
def xs_rank_agreement(src: np.ndarray, ref: np.ndarray, z_ref: np.ndarray) -> Dict[str, Any]:
    """Agreement of the ranking by `src` (benchmark values) with the ranking by `ref` over one policy set.

    Kendall tau-b; expected top-1 agreement and selection regret (z units of the reference target) under uniform
    tie-breaking among the policies tied at the top of `src`. Values within XS_TIE_ATOL are ties. A benchmark that
    gives every policy the same value (constant `src`, non-constant `ref`) has no ranking information and scores
    tau = 0 (Kendall tau-a; flagged `tau_benchmark_constant`); tau stays undefined only when `ref` is constant."""
    src, ref, z_ref = (np.asarray(v, dtype=np.float64) for v in (src, ref, z_ref))
    out = {"n_policies": int(len(src)), "kendall_tau_b": np.nan, "top1_agreement": np.nan, "selection_regret_z": np.nan,
           "selected_suboptimality_z": np.nan, "n_top_ties": 0, "tau_undefined_constant_values": False,
           "tau_benchmark_constant": False}
    if len(src) < 2:
        return out
    s_src, s_ref = xs_snap_ties(src), xs_snap_ties(ref)
    src_var, ref_var = np.ptp(s_src) > 0, np.ptp(s_ref) > 0
    if src_var and ref_var:
        out["kendall_tau_b"] = float(stats.kendalltau(s_src, s_ref)[0])
    elif ref_var:
        out["kendall_tau_b"] = 0.0
        out["tau_benchmark_constant"] = True
    else:
        out["tau_undefined_constant_values"] = True
    top_src, top_ref = xs_top(src), xs_top(ref)
    out["n_top_ties"] = int(top_src.sum())
    out["top1_agreement"] = float(top_ref[top_src].mean())
    if np.isfinite(z_ref).all():
        out["selection_regret_z"] = float(z_ref.max() - z_ref[top_src].mean())
        out["selected_suboptimality_z"] = float(1.0 - z_ref[top_src].mean())
    return out


XS_RANK_METRICS = ("kendall_tau_b", "top1_agreement", "selection_regret_z", "selected_suboptimality_z")
XS_RANK_DRAW_COLUMNS = (["cell", "family", "draw", "completion", "ranked_by", "evaluated_on", "n_policies",
                         "n_top_ties", "tau_undefined_constant_values", "tau_benchmark_constant"] + list(XS_RANK_METRICS)
                        + ["benchmark_choice", "reference_best"])


def xs_rank_fidelity_draws(norm: pd.DataFrame) -> pd.DataFrame:
    """Per (cell, draw, comparison) agreement statistics over the default policy set."""
    keep = ["bench_mean", "bench_terminate", "true"]
    d = norm[norm["is_default"] & norm["target"].isin(keep)] if len(norm) else norm
    if d.empty:
        return pd.DataFrame(columns=XS_RANK_DRAW_COLUMNS)
    cell_targets = {c: set(t) for c, t in d.groupby("cell")["target"]}
    w = d.set_index(["cell", "draw", "policy", "target"])
    J, Zt = w["return"].unstack("target"), w["z"].unstack("target")
    rows = []
    for (cell, draw), g in J.groupby(level=["cell", "draw"], sort=False):
        zg = Zt.loc[g.index]
        pols = g.index.get_level_values("policy").to_numpy()
        for completion, src_t, ref_t in XS_RANK_COMPARISONS:
            if not {src_t, ref_t} <= cell_targets[cell]:
                continue
            ok = (g[src_t].notna() & g[ref_t].notna()).to_numpy()
            src, ref, zr = g[src_t].to_numpy()[ok], g[ref_t].to_numpy()[ok], zg[ref_t].to_numpy()[ok]
            r = {"cell": cell, "family": cell.split("|", 1)[0], "draw": int(draw), "completion": completion,
                 "ranked_by": src_t, "evaluated_on": ref_t, **xs_rank_agreement(src, ref, zr),
                 "benchmark_choice": "", "reference_best": ""}
            if ok.any():
                p = pols[ok]
                r["benchmark_choice"] = "; ".join(p[xs_top(src)])
                r["reference_best"] = "; ".join(p[xs_top(ref)])
            rows.append(r)
    return pd.DataFrame(rows, columns=XS_RANK_DRAW_COLUMNS)


def _xs_cell_factor_cols(cell: str) -> Dict[str, Any]:
    f = xs_parse_cell(cell)
    return {k: f[k] for k in ("construction", "n_hosp", "tau", "eps", "harm")}


def xs_rank_fidelity_table(draws: pd.DataFrame, n_boot: int) -> pd.DataFrame:
    """table_rank_fidelity.csv: per cell x comparison, mean over draws with bootstrap intervals over draws."""
    cols = (["cell", "family", "construction", "n_hosp", "tau", "eps", "harm", "completion", "ranked_by", "evaluated_on",
             "n_draws", "mean_policies"]
            + [f"{m}_{s}" for m in XS_RANK_METRICS for s in ("mean", "ci_low", "ci_high", "n")])
    if draws.empty:
        return pd.DataFrame(columns=cols)
    rows = []
    for (cell, completion), g in draws.groupby(["cell", "completion"], sort=False):
        r = {"cell": cell, "family": g["family"].iloc[0], **_xs_cell_factor_cols(cell), "completion": completion,
             "ranked_by": g["ranked_by"].iloc[0], "evaluated_on": g["evaluated_on"].iloc[0], "n_draws": int(len(g)),
             "mean_policies": float(g["n_policies"].mean())}
        for m in XS_RANK_METRICS:
            v = g[m].to_numpy(dtype=np.float64)
            v = v[np.isfinite(v)]
            point, reps = xs_stratified_bootstrap({cell: v[:, None]} if len(v) else {}, xs_stat_mean(),
                                                  ("rank", completion, m), n_boot)
            r[f"{m}_mean"] = point
            r[f"{m}_ci_low"], r[f"{m}_ci_high"] = xs_ci(reps)
            r[f"{m}_n"] = int(len(v))
        rows.append(r)
    return pd.DataFrame(rows, columns=cols)


XS_ERR_DRAW_COLUMNS = ["cell", "family", "construction", "n_hosp", "tau", "eps", "harm", "draw", "completion",
                       "n_policies", "mean_abs_total", "mean_abs_semantics", "mean_abs_estimation", "mean_total",
                       "mean_semantics", "mean_estimation", "decomposition_source"]


def xs_error_decomposition_draws(norm: pd.DataFrame) -> pd.DataFrame:
    """Per (cell, draw, completion) means over default policies of the exact benchmark-error decomposition."""
    keep = ["true", "bench_mean", "bench_terminate", "true_rows_mean", "true_rows_terminate"]
    d = (norm[norm["is_default"] & norm["family"].isin(["gumbel", "garnet"]) & norm["target"].isin(keep)]
         if len(norm) else norm)
    if d.empty:
        return pd.DataFrame(columns=XS_ERR_DRAW_COLUMNS)
    cell_targets = {c: set(t) for c, t in d.groupby("cell")["target"]}
    J = d.set_index(["cell", "draw", "policy", "target"])["return"].unstack("target")
    rows = []
    for (cell, draw), g in J.groupby(level=["cell", "draw"], sort=False):
        info = xs_parse_cell(cell)
        for completion, (bench_t, rows_t) in XS_COMPLETIONS.items():
            if not {bench_t, "true"} <= cell_targets[cell]:
                continue
            if rows_t in cell_targets[cell]:
                rows_v, source = g[rows_t], "targets"
            elif info["construction"] == "oracle":
                # Oracle construction (Section II.3): the benchmark is the true rows + completion on the true state
                # space with the true d0, so it equals the true-rows target and the estimation term is exactly 0.
                rows_v, source = g[bench_t], "oracle construction (benchmark = true rows + completion)"
            else:
                raise AssertionError(f"{cell}: target '{rows_t}' is missing, so the error cannot be decomposed")
            ok = (g["true"].notna() & g[bench_t].notna() & rows_v.notna()).to_numpy()
            tot = (g["true"] - g[bench_t]).to_numpy()[ok]
            sem = (g["true"] - rows_v).to_numpy()[ok]
            est = (rows_v - g[bench_t]).to_numpy()[ok]
            mean = (lambda a: float(a.mean()) if len(a) else np.nan)
            rows.append({"cell": cell, "family": info["family"], **_xs_cell_factor_cols(cell), "draw": int(draw),
                         "completion": completion, "n_policies": int(ok.sum()), "mean_abs_total": mean(np.abs(tot)),
                         "mean_abs_semantics": mean(np.abs(sem)), "mean_abs_estimation": mean(np.abs(est)),
                         "mean_total": mean(tot), "mean_semantics": mean(sem), "mean_estimation": mean(est),
                         "decomposition_source": source})
    return pd.DataFrame(rows, columns=XS_ERR_DRAW_COLUMNS)


XS_ERR_METRICS = ("mean_abs_total", "mean_abs_semantics", "mean_abs_estimation")


def xs_error_decomposition_table(draws: pd.DataFrame, n_boot: int) -> pd.DataFrame:
    cols = (["cell", "family", "construction", "n_hosp", "tau", "eps", "harm", "completion", "n_draws",
             "mean_policies"] + [f"{m}_{s}" for m in XS_ERR_METRICS for s in ("mean", "ci_low", "ci_high")]
            + ["mean_total", "mean_semantics", "mean_estimation", "decomposition_source"])
    if draws.empty:
        return pd.DataFrame(columns=cols)
    rows = []
    for (cell, completion), g in draws.groupby(["cell", "completion"], sort=False):
        r = {"cell": cell, "family": g["family"].iloc[0], **_xs_cell_factor_cols(cell), "completion": completion,
             "n_draws": int(len(g)), "mean_policies": float(g["n_policies"].mean()),
             "decomposition_source": g["decomposition_source"].iloc[0]}
        for m in XS_ERR_METRICS:
            v = g[m].to_numpy(dtype=np.float64)
            v = v[np.isfinite(v)]
            point, reps = xs_stratified_bootstrap({cell: v[:, None]} if len(v) else {}, xs_stat_mean(),
                                                  ("errdec", completion, m), n_boot)
            r[f"{m}_mean"] = point
            r[f"{m}_ci_low"], r[f"{m}_ci_high"] = xs_ci(reps)
        for m in ("mean_total", "mean_semantics", "mean_estimation"):
            v = g[m].dropna()
            r[m] = float(v.mean()) if len(v) else np.nan
        rows.append(r)
    return pd.DataFrame(rows, columns=cols)


# ---------- preregistered tests H1-H3 (PREREG_PART2) ----------
def xs_h1(rank_draws: pd.DataFrame, n_boot: int) -> Dict[str, Any]:
    """H1: per Gumbel B-est cell, two-sided sign-flip test on per-draw tau(mean) - tau(terminate); Holm across cells."""
    rd = rank_draws[rank_draws["family"].eq("gumbel") & rank_draws["completion"].isin(["mean", "terminate"])] \
        if len(rank_draws) else rank_draws
    cells = [c for c in dict.fromkeys(rd["cell"]) if xs_parse_cell(c)["construction"] == "best"] if len(rd) else []
    rows = []
    for cell in cells:
        sub = rd[rd["cell"] == cell]
        t = sub.pivot(index="draw", columns="completion", values="kendall_tau_b")
        v = t.reindex(columns=["mean", "terminate"]).dropna()
        d = (v["mean"] - v["terminate"]).to_numpy(dtype=np.float64)
        test = sign_flip_test(d, "two-sided", f"xenv|H1|{cell}")
        # Draws in which a benchmark gives every policy the same value score tau = 0 and stay in the primary test;
        # the sensitivity test drops them (flag on either completion).
        flag = (xs_as_bool(sub["tau_benchmark_constant"]) if "tau_benchmark_constant" in sub.columns
                else pd.Series(False, index=sub.index))
        const = flag.groupby(sub["draw"].to_numpy()).any().reindex(v.index, fill_value=False).astype(bool)
        d_ex = d[~const.to_numpy()]
        test_ex = sign_flip_test(d_ex, "two-sided", f"xenv|H1|{cell}|excluding-constant-benchmarks")
        md, reps = xs_stratified_bootstrap({cell: d[:, None]} if len(d) else {}, xs_stat_mean(), ("H1", cell), n_boot)
        lo, hi = xs_ci(reps)
        rows.append({"cell": cell, "n_draws": int(len(t)), "n_pairs": int(len(d)),
                     "n_tau_undefined": int(len(t) - len(d)), "n_benchmark_constant": int(flag.sum()),
                     "mean_tau_mean_rule": float(v["mean"].mean()) if len(v) else np.nan,
                     "mean_tau_terminate_rule": float(v["terminate"].mean()) if len(v) else np.nan,
                     "mean_difference": md, "ci_low": lo, "ci_high": hi, "p_value": test["p_value"],
                     "test_method": test["method"],
                     "min_attainable_p": xs_min_attainable_p(len(d)) if len(d) else np.nan,
                     "n_pairs_excluding_constant_benchmarks": int(len(d_ex)),
                     "p_value_excluding_constant_benchmarks": test_ex["p_value"]})
    adj = holm_adjust({r["cell"]: r["p_value"] for r in rows})
    m = int(sum(np.isfinite(r["p_value"]) for r in rows))
    for r in rows:
        r["holm_p"] = adj[r["cell"]]
        r["reject_holm"] = bool(np.isfinite(r["holm_p"]) and r["holm_p"] <= ALPHA)
        r["holm_first_step_attainable"] = bool(np.isfinite(r["min_attainable_p"])
                                               and r["min_attainable_p"] <= ALPHA / max(m, 1))
    return {"status": "run" if rows else "not run (no Gumbel B-est cell with both completions)",
            "family_size": m, "alpha": ALPHA, "metric": "Kendall tau-b, bench vs true, default policy set",
            "difference": "tau(mean) - tau(terminate)",
            "constant_benchmark": ("a benchmark that gives every policy the same value scores tau = 0 (Kendall tau-a, "
                                   "no ranking information); such draws stay in the primary test and are counted "
                                   "(n_benchmark_constant); p_value_excluding_constant_benchmarks drops them"),
            "rows": rows,
            "n_rejections_holm": int(sum(r["reject_holm"] for r in rows))}


XS_H2_COLUMNS = ["family", "deploy_target", "treatment", "comparator", "status", "n_units", "n_pairs", "n_missing_pairs",
                 "test_unit", "n_test_units",
                 "iqm_treatment", "iqm_comparator", "iqm_difference", "iqm_difference_ci_low", "iqm_difference_ci_high",
                 "mean_difference", "mean_difference_ci_low", "mean_difference_ci_high", "paired_win_rate",
                 "p_value", "test_method", "min_attainable_p", "tost_p", "equivalence_margin"]


def xs_h2(norm: pd.DataFrame, n_boot: int) -> pd.DataFrame:
    """H2: per family, paired deployment-z differences (treatment - comparator); sign-flip, Holm, TOST.

    Test units: (cell, draw) pairs pooled across cells for ICU-Sepsis and Gumbel (independent streams on a fixed
    system); for Garnet (XS_DRAW_BLOCKED_FAMILIES) one unit per base-MDP draw, the mean difference over the harm x eps
    cells that share it, and the bootstrap resamples Garnet draws jointly across cells."""
    dep = xs_deploy_scores(norm)
    margin = float(XCFG.equivalence_margin_normalized)
    rows = []
    for fam in XS_FAMILIES:
        d = dep[dep["family"] == fam] if len(dep) else dep
        if d.empty:
            continue
        Z = xs_wide(d, "z")
        blk = fam in XS_DRAW_BLOCKED_FAMILIES
        for treat in XTREATMENTS:
            r = {c: np.nan for c in XS_H2_COLUMNS}
            r.update({"family": fam, "deploy_target": XS_DEPLOY[fam], "treatment": treat, "comparator": XCOMPARATOR,
                      "status": "ok", "n_units": int(len(Z)), "n_pairs": 0, "n_missing_pairs": int(len(Z)),
                      "test_unit": "draw block" if blk else "cell x draw", "n_test_units": 0,
                      "equivalence_margin": margin, "test_method": ""})
            missing = [p for p in (treat, XCOMPARATOR) if p not in Z.columns]
            if missing:
                r["status"] = "missing policy: " + ", ".join(missing)
                rows.append(r)
                continue
            g = xs_cell_groups(Z, [treat, XCOMPARATOR], aligned=blk)
            D = Z[treat] - Z[XCOMPARATOR]
            D = D[np.isfinite(D.to_numpy(dtype=np.float64))]
            unit_d = D.to_numpy(dtype=np.float64)                       # finite (cell, draw) paired differences
            dv = D.groupby(level="draw").mean().to_numpy(dtype=np.float64) if blk else unit_d
            n, tkey = len(dv), f"xenv|H2|{fam}|{treat}"
            test = sign_flip_test(dv, "two-sided", tkey)
            iqm_d, ireps = xs_stratified_bootstrap(g, xs_stat_iqm_diff, ("main", fam, XS_DEPLOY[fam]), n_boot,
                                                   shared_draws=blk)
            mean_d, mreps = xs_stratified_bootstrap(g, xs_stat_mean_diff, ("main", fam, XS_DEPLOY[fam]), n_boot,
                                                    shared_draws=blk)
            pooled = np.concatenate(list(g.values())) if g else np.zeros((0, 2))
            nu = len(unit_d)
            r.update({"n_pairs": int(nu), "n_missing_pairs": int(len(Z) - nu), "n_test_units": int(n),
                      "iqm_treatment": xs_iqm(pooled[:, 0]), "iqm_comparator": xs_iqm(pooled[:, 1]),
                      "iqm_difference": iqm_d, "mean_difference": mean_d,
                      "paired_win_rate": (float(np.mean(unit_d > XS_TIE_ATOL) + 0.5 * np.mean(np.abs(unit_d) <= XS_TIE_ATOL))
                                          if nu else np.nan),
                      "p_value": test["p_value"], "test_method": test["method"],
                      "min_attainable_p": xs_min_attainable_p(n) if n else np.nan,
                      "tost_p": tost(dv, margin, tkey)})
            r["iqm_difference_ci_low"], r["iqm_difference_ci_high"] = xs_ci(ireps)
            r["mean_difference_ci_low"], r["mean_difference_ci_high"] = xs_ci(mreps)
            rows.append(r)
    h2 = pd.DataFrame(rows, columns=XS_H2_COLUMNS)
    extra = ["holm_p", "tost_holm_p", "holm_family_size", "reject_holm", "direction", "equivalent_tost",
             "equivalent_tost_holm", "holm_first_step_attainable"]
    if h2.empty:
        return pd.DataFrame(columns=XS_H2_COLUMNS + extra)
    hyp = h2["family"] + "|" + h2["treatment"]
    h2["holm_p"] = hyp.map(holm_adjust(dict(zip(hyp, h2["p_value"]))))
    h2["tost_holm_p"] = hyp.map(holm_adjust(dict(zip(hyp, h2["tost_p"]))))
    m = int(np.isfinite(h2["p_value"].to_numpy(dtype=np.float64)).sum())
    h2["holm_family_size"] = m
    h2["reject_holm"] = h2["holm_p"] <= ALPHA
    h2["direction"] = np.where(~h2["reject_holm"], "", np.where(h2["mean_difference"] > 0, "treatment better",
                                                               "comparator better"))
    h2["equivalent_tost"] = h2["tost_p"] <= ALPHA
    h2["equivalent_tost_holm"] = h2["tost_holm_p"] <= ALPHA
    h2["holm_first_step_attainable"] = h2["min_attainable_p"] <= ALPHA / max(m, 1)
    return h2


def xs_h3(err_draws: pd.DataFrame, n_perm: int = XS_H3_PERMUTATIONS) -> Dict[str, Any]:
    """H3 (Garnet): Spearman rho between |h| and the per-draw mean |J_true - J_bench_mean| over default policies."""
    e = err_draws[err_draws["family"].eq("garnet") & err_draws["completion"].eq("mean")] if len(err_draws) else err_draws
    e = e[np.isfinite(e["mean_abs_total"].to_numpy(dtype=np.float64))] if len(e) else e
    out = {"status": "not run (no Garnet draws)", "alpha": ALPHA, "metric": "mean |J_true - J_bench_mean| per draw",
           "predictor": "|h|", "n_draw_units": int(len(e))}
    if e.empty:
        return out
    x = np.abs(e["harm"].to_numpy(dtype=np.float64))
    y = e["mean_abs_total"].to_numpy(dtype=np.float64)
    eps = e["eps"].to_numpy(dtype=np.float64)
    # Exchangeability blocks of the design: the harm levels of one draw share its base MDP (part21c ('base', draw)).
    blocks = (e["eps"].astype(str) + "|" + e["draw"].astype(str)).to_numpy()
    primary = xs_permutation_spearman(x, y, blocks, n_perm, ("within-eps-draw",))
    strat = xs_permutation_spearman(x, y, eps, n_perm, ("stratified-eps",))
    unres = xs_permutation_spearman(x, y, None, n_perm, ("unrestricted",))
    out.update({"status": "run" if np.isfinite(primary["rho"]) else "not testable (fewer than 3 draws or constant |h|)",
                "spearman_rho": primary["rho"], "p_value": primary["p_value"],
                "test": "permutation of |h| within (eps, draw) blocks", "n_blocks": primary.get("n_strata", 0),
                "p_value_permutation_within_eps_draw": primary["p_value"],
                "p_value_permutation_within_eps": strat["p_value"],
                "p_value_permutation_unrestricted": unres["p_value"], "n_permutations": int(n_perm),
                "eps_strata": sorted(float(v) for v in np.unique(eps)),
                "reject": bool(np.isfinite(primary["p_value"]) and primary["p_value"] <= ALPHA),
                "per_eps_spearman_descriptive": [{"eps": float(v), "n": int((eps == v).sum()),
                                                  "rho": xs_spearman(x[eps == v], y[eps == v])} for v in np.unique(eps)],
                "mean_error_by_harm": [{"harm": float(h), "abs_harm": abs(float(h)), "n": int(len(g)),
                                        "mean_abs_error": float(g["mean_abs_total"].mean())}
                                       for h, g in e.groupby("harm")]})
    return out


def xs_h3_descriptive(err_draws: pd.DataFrame, norm: pd.DataFrame, draw_meta: Optional[pd.DataFrame]) -> Dict[str, Any]:
    """Descriptive companion of H3 (Garnet). The harm also changes the benchmark mask (absorbing outcomes shorten the
    hospital episodes), so H3 mixes the size of the harm with the number of unsupported actions. Reported by (h, eps):
    mean admissible actions of the benchmark (draw meta), mean expected violations of the default policies on the
    true target, the mean |error|; and, exploratory, the mean |error| per expected violation by h (ratio of means)."""
    e = (err_draws[err_draws["family"].eq("garnet") & err_draws["completion"].eq("mean")]
         if len(err_draws) else err_draws)
    if e.empty:
        return {"status": "not run (no Garnet draws)"}
    units = e[["cell", "draw", "harm", "eps", "mean_abs_total", "mean_abs_semantics"]].copy()
    units["draw"] = units["draw"].astype(np.int64)
    meta = xs_as_frame(draw_meta)
    col = "instance.construction.mean_admissible_actions"
    if col in meta.columns and {"cell", "draw"} <= set(meta.columns):
        m = pd.DataFrame({"cell": meta["cell"].astype(str), "draw": pd.to_numeric(meta["draw"]).astype(np.int64),
                          "mean_admissible_actions": pd.to_numeric(meta[col], errors="coerce")})
        units = units.merge(m.drop_duplicates(["cell", "draw"]), on=["cell", "draw"], how="left", validate="one_to_one")
    else:
        units["mean_admissible_actions"] = np.nan
    t = (norm[norm["family"].eq("garnet") & norm["is_default"] & norm["target"].eq("true") & norm["status"].eq("ok")]
         if len(norm) else norm)
    ev = (t.groupby(["cell", "draw"])["expected_violations"].mean().rename("mean_expected_violations_true")
          .reset_index() if len(t) else pd.DataFrame(columns=["cell", "draw", "mean_expected_violations_true"]))
    ev["draw"] = ev["draw"].astype(np.int64)
    units = units.merge(ev, on=["cell", "draw"], how="left", validate="one_to_one")
    mean = (lambda s: float(s.mean()) if s.notna().any() else np.nan)
    by_cell = [{"harm": float(h), "eps": float(ep), "n_draws": int(len(g)),
                "mean_admissible_actions": mean(g["mean_admissible_actions"]),
                "mean_expected_violations_true": mean(g["mean_expected_violations_true"]),
                "mean_abs_error": mean(g["mean_abs_total"]), "mean_abs_semantics_term": mean(g["mean_abs_semantics"])}
               for (h, ep), g in units.groupby(["harm", "eps"])]
    per_viol = []
    for h, g in units.groupby("harm"):
        v = mean(g["mean_expected_violations_true"])
        per_viol.append({"harm": float(h), "n_draw_units": int(len(g)), "mean_expected_violations_true": v,
                         "mean_abs_error_per_expected_violation": (mean(g["mean_abs_total"]) / v
                                                                   if np.isfinite(v) and v > 0 else np.nan),
                         "mean_abs_semantics_per_expected_violation": (mean(g["mean_abs_semantics"]) / v
                                                                       if np.isfinite(v) and v > 0 else np.nan)})
    return {"status": "descriptive", "by_harm_eps": by_cell, "exploratory_error_per_expected_violation": per_viol,
            "note": ("the harm changes the mask itself, so H3 mixes the size of the harm with the number of "
                     "unsupported actions; the per-violation ratios (ratio of means over draws) separate the two")}


def xs_design_deviations(prereg: Dict[str, Any], xcfg: Any) -> List[Dict[str, Any]]:
    """Every PREREG_PART2['design'] key whose realised XCFG value differs (tuples compared as JSON lists)."""
    out = []
    for k, v in (prereg.get("design") or {}).items():
        pre = json.loads(canonical_json(v))
        real = json.loads(canonical_json(getattr(xcfg, k, None)))
        if pre != real:
            out.append({"key": k, "preregistered": pre, "realised": real})
    return out


def xs_prereg_report(h1: Dict[str, Any], h2: pd.DataFrame, h3: Dict[str, Any]) -> Dict[str, Any]:
    status = str(PREREG_PART2.get("status", ""))
    locked = status.upper().startswith("LOCKED")
    h2_ok = h2[h2["status"].eq("ok")] if len(h2) else h2
    return {"prereg": PREREG_PART2, "prereg_hash": obj_hash(PREREG_PART2), "prereg_locked": locked,
            "warning": ("" if locked else "PREREG_PART2 is not locked: these tests are exploratory until the "
                        "preregistration is locked before the confirmatory run."),
            "design_deviations": xs_design_deviations(PREREG_PART2, XCFG),
            "alpha": ALPHA, "deploy_targets": XS_DEPLOY, "comparator": XCOMPARATOR, "treatments": XTREATMENTS,
            "H1": h1,
            "H2": {"status": "run" if len(h2_ok) else "not run", "table": "xenv/xenv_h2_family.csv",
                   "test_units": {f: ("draw block (mean over the cells sharing a base MDP)"
                                      if f in XS_DRAW_BLOCKED_FAMILIES else "(cell, draw) pooled across cells")
                                  for f in XS_FAMILIES},
                   "family_size": int(h2["holm_family_size"].iloc[0]) if len(h2) else 0,
                   "n_rejections_holm": int(h2_ok["reject_holm"].sum()) if len(h2_ok) else 0,
                   "n_equivalent_tost_holm": int(h2_ok["equivalent_tost_holm"].sum()) if len(h2_ok) else 0,
                   "rows": h2.to_dict(orient="records")},
            "H3": h3}


# ---------- hyper-parameter selection: default vs FQE vs oracle ----------
XS_HP_COLUMNS = (["family", "deploy_target", "method", "variant", "method_family", "n_configs", "configs", "n_units"]
                 + [f"{p}_{s}" for p in ("default", "fqe_selected", "oracle_selected_upper_bound")
                    for s in ("n", "iqm", "iqm_ci_low", "iqm_ci_high", "mean")]
                 + ["mean_fqe_minus_default", "mean_oracle_minus_fqe", "fqe_picks_default_rate", "fqe_modal_hp",
                    "fqe_modal_hp_rate", "fqe_target_available", "fqe_fallback_units"])


def _xs_select(m: pd.DataFrame, score: str, keys: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Per unit, the configurations maximising `score` (ties within XS_TIE_ATOL); z averaged over the tie set."""
    s = m[score]
    mx = s.groupby([m[k] for k in keys]).transform("max")
    tie = s.notna() & (s >= mx - XS_TIE_ATOL)
    t = m[tie].copy()
    if t.empty:
        return pd.DataFrame(columns=["z", "default_share"], index=pd.MultiIndex.from_tuples([], names=keys)), t
    t["w"] = 1.0 / t.groupby(keys)["hp"].transform("size")
    agg = t.groupby(keys).agg(z=("z", lambda v: float(v.mean()) if v.notna().all() else np.nan),
                              default_share=("is_default", "mean"))
    return agg, t


def xs_hp_selection(norm: pd.DataFrame, n_boot: int) -> pd.DataFrame:
    """table_hp_selection.csv for every tunable (method, variant) and family, on the deployment target."""
    empty = pd.DataFrame(columns=XS_HP_COLUMNS)
    dep = xs_deploy_scores(norm, defaults_only=False)
    if dep.empty:
        return empty
    ncfg = dep.groupby(["family", "method", "variant"])["hp"].nunique()
    tun = ncfg[ncfg >= 2].reset_index()[["family", "method", "variant"]]
    if tun.empty:
        return empty
    dep = dep.merge(tun, on=["family", "method", "variant"], how="inner")
    fq = norm.loc[norm["target"].eq(XS_FQE), ["cell", "draw", "policy", "return"]].rename(columns={"return": "fqe_return"})
    m = dep.merge(fq, on=["cell", "draw", "policy"], how="left", validate="one_to_one")
    keys = ["family", "cell", "draw", "method", "variant"]
    units = m[keys].drop_duplicates().set_index(keys)
    dflt = m[m["is_default"]].groupby(keys)["z"]
    assert len(dflt.size()) == 0 or int(dflt.size().max()) == 1, "more than one default configuration in a unit"
    units["z_default"] = dflt.first()
    fsel, ft = _xs_select(m, "fqe_return", keys)
    osel, _ = _xs_select(m, "return", keys)
    units["z_fqe"] = fsel["z"]
    units["fqe_default_share"] = fsel["default_share"]
    units["z_oracle"] = osel["z"]
    # A unit whose candidates are all improper on the FQE model has no FQE choice: it falls back to the pre-declared
    # default, so default, FQE and oracle columns describe the same units (fqe_default_share stays NaN there).
    decided = units.index.isin(fsel.index)
    units["fqe_fallback"] = ~decided & (len(fq) > 0)
    if len(fq) > 0:
        units.loc[~decided, "z_fqe"] = units.loc[~decided, "z_default"]
    rows = []
    for (fam, meth, var), u in units.groupby(level=["family", "method", "variant"], sort=False):
        W = u.reset_index().set_index(["cell", "draw"])
        sub = dep[(dep["family"] == fam) & (dep["method"] == meth) & (dep["variant"] == var)]
        r = {"family": fam, "deploy_target": XS_DEPLOY[fam], "method": meth, "variant": var,
             "method_family": sub["method_family"].iloc[0], "n_configs": int(sub["hp"].nunique()),
             "configs": "; ".join(sorted(sub["hp"].unique())), "n_units": int(len(W)),
             "fqe_target_available": bool(len(fq) > 0), "fqe_fallback_units": int(W["fqe_fallback"].sum())}
        blk = fam in XS_DRAW_BLOCKED_FAMILIES
        for label, col in (("default", "z_default"), ("fqe_selected", "z_fqe"),
                           ("oracle_selected_upper_bound", "z_oracle")):
            g = xs_cell_groups(W, [col], aligned=blk)
            point, reps = xs_stratified_bootstrap(g, xs_stat_iqm(), ("hp", fam), n_boot, shared_draws=blk)
            v = W[col].dropna()
            r[f"{label}_n"] = int(len(v))
            r[f"{label}_iqm"] = point
            r[f"{label}_iqm_ci_low"], r[f"{label}_iqm_ci_high"] = xs_ci(reps)
            r[f"{label}_mean"] = float(v.mean()) if len(v) else np.nan
        a, b, c = W["z_fqe"], W["z_default"], W["z_oracle"]
        r["mean_fqe_minus_default"] = float((a - b).dropna().mean()) if (a - b).notna().any() else np.nan
        r["mean_oracle_minus_fqe"] = float((c - a).dropna().mean()) if (c - a).notna().any() else np.nan
        r["fqe_picks_default_rate"] = float(W["fqe_default_share"].mean()) if W["fqe_default_share"].notna().any() else np.nan
        mt = ft[(ft["family"] == fam) & (ft["method"] == meth) & (ft["variant"] == var)] if len(ft) else ft
        if len(mt):
            cnt = mt.groupby("hp")["w"].sum().sort_values(ascending=False, kind="mergesort")
            r["fqe_modal_hp"], r["fqe_modal_hp_rate"] = str(cnt.index[0]), float(cnt.iloc[0] / cnt.sum())
        else:
            r["fqe_modal_hp"], r["fqe_modal_hp_rate"] = "", np.nan
        rows.append(r)
    return pd.DataFrame(rows, columns=XS_HP_COLUMNS)


def xs_proposition_summary(props: pd.DataFrame, propositions_ok: Optional[bool]) -> Dict[str, Any]:
    if props.empty:
        return {"status": "not run", "PROPOSITIONS_OK": propositions_ok}
    assert {"proposition", "passes"} <= set(props.columns), f"PROP_CHECKS columns: {list(props.columns)}"
    flag = props["passes"].astype(str).str.strip().str.lower()
    per = {str(p): {"checks": int(len(g)), "passed": int((flag[g.index] == "true").sum()),
                    "failed": int((flag[g.index] == "false").sum())} for p, g in props.groupby("proposition")}
    return {"status": "run", "checks": int(len(props)), "passed": int((flag == "true").sum()),
            "failed": int((flag == "false").sum()), "other": int((~flag.isin(["true", "false"])).sum()),
            "per_proposition": per, "PROPOSITIONS_OK": propositions_ok}

### II.7 figures

| Figure | Content |
|---|---|
| `fig_xenv_frontiers` | exact CMDP frontiers: optimal value vs expected-violation budget κ per environment and target. The κ = ∞ value is the dashed line; on every benchmark the frontier is flat, under the mean rule by Proposition P2 and under termination by the P3 corollary (r_term = v_min); panels are (cell, environment) |
| `fig_xenv_rank_fidelity` | Kendall τ-b, top-1 agreement and selection regret of the benchmark's ranking against the true system, per cell and completion rule |
| `fig_xenv_harm_phase` | Garnet phase diagram: benchmark error and rank fidelity against the harm *h* of unsupported actions, per behaviour ε (*h* = 0: non-preferred unsupported actions are completed correctly; the semantics term vanishes where K(s) ⊆ M(s), see `table_error_decomposition.csv`) |
| `fig_xenv_iqm` | IQM of the deployment score with stratified-bootstrap intervals for every default policy, per family |
| `fig_xenv_profiles` | performance profiles of the H2 treatments against the comparator (small multiples of four treatments each) |
| `fig_xenv_poi` | pooled probability of improvement between default policies (diverging scale centred at ½) |
| `fig_xenv_error_decomposition` | mean absolute benchmark error and its semantics and estimation-and-coverage terms per cell |

A figure whose inputs are empty (because a switch is off) is not drawn; the reason is recorded in
`XSTATS["figures"]`.

In [ ]:
def _xs_err(mean, lo, hi) -> np.ndarray:
    mean, lo, hi = (np.asarray(v, dtype=np.float64) for v in (mean, lo, hi))
    return np.vstack([np.clip(mean - lo, 0.0, None), np.clip(hi - mean, 0.0, None)])


def _xs_save(fig, fig_dir: Path, stem: str) -> str:
    save_figure(fig, Path(fig_dir) / stem, dpi=200)
    return "written"


def xs_fig_frontiers(fr: pd.DataFrame, fig_dir: Path) -> str:
    if fr.empty:
        return "not produced: X_FRONTIERS is empty (switch xenv_frontiers off or Section II.5 not run)"
    obj = next((c for c in ("objective", "lp_objective", "value", "return") if c in fr.columns), None)
    assert "kappa" in fr.columns and obj is not None, f"X_FRONTIERS needs 'kappa' and an objective column: {list(fr.columns)}"
    panel = next((c for c in ("environment", "env", "instance") if c in fr.columns), None)
    lines = [c for c in ("target", "semantics", "cost") if c in fr.columns and c != panel]
    fr = fr.assign(kappa=pd.to_numeric(fr["kappa"], errors="raise"), _obj=pd.to_numeric(fr[obj], errors="raise"))
    if panel is not None and "cell" in fr.columns:
        # Environment names are not unique across cells (e.g. the best and oracle Gumbel cells share the name of the
        # true system with a benchmark mask, but their masks differ), so panels are (cell, environment).
        fr = fr.assign(_panel=fr["cell"].astype(str) + "\n" + fr[panel].astype(str))
        panel = "_panel"
    if panel is not None:
        dup = fr.groupby([panel, *lines], sort=False)["kappa"].apply(lambda k: k.duplicated().any())
        assert not dup.any(), f"several frontiers share one figure line: {list(dup[dup].index)[:3]}"
    panels = list(dict.fromkeys(fr[panel])) if panel else ["all environments"]
    nc = min(3, len(panels))
    nr = int(math.ceil(len(panels) / nc))
    fig, axes = plt.subplots(nr, nc, figsize=(4.8 * nc, 3.6 * nr), squeeze=False)
    finite_k = fr.loc[np.isfinite(fr["kappa"]) & (fr["kappa"] > 0), "kappa"]
    lin = float(finite_k.min()) if len(finite_k) else 1e-3
    for ax, p in zip(axes.ravel(), panels):
        g = fr[fr[panel] == p] if panel else fr
        series = list(g.groupby(lines, sort=False)) if lines else [("", g)]
        for k, (lab, h) in enumerate(series):
            h = h.sort_values("kappa")
            fin = np.isfinite(h["kappa"].to_numpy())
            color = XS_COLORS[k % len(XS_COLORS)] if len(series) <= len(XS_COLORS) else None
            label = " / ".join(map(str, lab)) if isinstance(lab, tuple) else str(lab)
            ln, = ax.plot(h["kappa"][fin], h["_obj"][fin], marker="o", ms=4, lw=1.5, color=color, label=label or None)
            if (~fin).any():
                ax.axhline(float(h["_obj"][~fin].iloc[0]), color=ln.get_color(), lw=1, ls="--")
        ax.set_xscale("symlog", linthresh=lin)
        ax.set_title(str(p), fontsize=8)
        ax.set_xlabel("expected-violation budget κ (dashed: κ = ∞)")
        ax.set_ylabel("exact constrained optimum")
        ax.grid(alpha=0.3, lw=0.5)
        if lines:
            ax.legend(fontsize=7)
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    fig.suptitle("Exact CMDP frontiers (cost = 1[a not admissible])")
    return _xs_save(fig, fig_dir, "fig_xenv_frontiers")


def xs_fig_rank_fidelity(rank: pd.DataFrame, fig_dir: Path) -> str:
    r = rank[rank["completion"].isin(["mean", "terminate"])] if len(rank) else rank
    if r.empty:
        return "not produced: no benchmark-vs-true comparisons (Gumbel and Garnet off)"
    cells = list(dict.fromkeys(r["cell"]))
    x = np.arange(len(cells))
    panels = [("kendall_tau_b", "Kendall τ-b (benchmark vs true)"), ("top1_agreement", "top-1 agreement"),
              ("selection_regret_z", "selection regret (z, true target)")]
    fig, axes = plt.subplots(3, 1, figsize=(max(8.0, 0.55 * len(cells) + 3), 9), sharex=True)
    for ax, (m, title) in zip(axes, panels):
        for j, comp in enumerate(("mean", "terminate")):
            g = r[r["completion"] == comp].set_index("cell").reindex(cells)
            ax.bar(x + (j - 0.5) * 0.4, g[f"{m}_mean"], 0.38, color=XS_COLORS[j], edgecolor="white", lw=0.6,
                   yerr=_xs_err(g[f"{m}_mean"], g[f"{m}_ci_low"], g[f"{m}_ci_high"]), capsize=2,
                   error_kw={"elinewidth": 0.8, "ecolor": XS_INK}, label=f"{comp} rule")
        ax.axhline(0, color=XS_MUTED, lw=0.8)
        ax.set_ylabel(title, fontsize=9)
        ax.grid(axis="y", alpha=0.3, lw=0.5)
    axes[0].legend(fontsize=8)
    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(cells, rotation=40, ha="right", fontsize=7)
    fig.suptitle("Benchmark rank fidelity over the default policy set "
                 f"(mean over draws, {100 * (1 - ALPHA):.0f}% bootstrap CI)")
    return _xs_save(fig, fig_dir, "fig_xenv_rank_fidelity")


def xs_fig_harm_phase(err: pd.DataFrame, rank: pd.DataFrame, fig_dir: Path) -> str:
    e = err[err["family"] == "garnet"] if len(err) else err
    k = rank[(rank["family"] == "garnet") & rank["completion"].isin(["mean", "terminate"])] if len(rank) else rank
    if e.empty and k.empty:
        return "not produced: no Garnet results (switch xenv_garnet off)"
    eps_vals = sorted(set(e["eps"].dropna()) | set(k["eps"].dropna()))
    styles = [("-", "o"), ("--", "s"), (":", "^"), ("-.", "D")]
    harms = pd.concat([e["harm"], k["harm"]]).dropna()
    dodge = 0.012 * max(float(harms.max() - harms.min()), 1.0) if len(harms) else 0.0   # keeps error bars apart
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, (df, col, ylab) in zip(axes, ((e, "mean_abs_total", "mean |J_true − J_bench| (default policies)"),
                                          (k, "kendall_tau_b", "Kendall τ-b (benchmark vs true)"))):
        for j, comp in enumerate(("mean", "terminate")):
            for i, ev in enumerate(eps_vals):
                g = df[(df["completion"] == comp) & (df["eps"] == ev)].sort_values("harm") if len(df) else df
                if g.empty:
                    continue
                ls, mk = styles[i % len(styles)]
                ax.errorbar(g["harm"] + (2 * j + i - 1.5) * dodge, g[f"{col}_mean"],
                            yerr=_xs_err(g[f"{col}_mean"], g[f"{col}_ci_low"], g[f"{col}_ci_high"]),
                            color=XS_COLORS[j], ls=ls, marker=mk, ms=6, lw=1.5, capsize=3, label=f"{comp} rule, ε = {ev:g}")
        ax.axvline(0.0, color=XS_MUTED, lw=0.8, ls=":")
        ax.set_xlabel("harm h of unsupported actions")
        ax.set_ylabel(ylab, fontsize=9)
        ax.grid(alpha=0.3, lw=0.5)
        ax.legend(fontsize=7)
    axes[0].set_title("(a) Benchmark error")
    axes[1].set_title("(b) Rank fidelity")
    fig.suptitle("Garnet phase diagram: when does the completion rule mislead? (h = 0: non-preferred unsupported "
                 "actions completed correctly; the semantics term vanishes where K(s) ⊆ M(s); points dodged)")
    return _xs_save(fig, fig_dir, "fig_xenv_harm_phase")


def xs_fig_iqm(main_long: pd.DataFrame, order: Sequence[str], fig_dir: Path) -> str:
    if main_long.empty:
        return "not produced: no deployment scores"
    fams = [f for f in XS_FAMILIES if f in set(main_long["family"])]
    pols = [p for p in order if p in set(main_long["policy"])]
    y = np.arange(len(pols))
    fig, axes = plt.subplots(1, len(fams), figsize=(4.2 * len(fams) + 3.0, 0.22 * len(pols) + 1.8), sharey=True,
                             squeeze=False)
    groups = (("other", XS_MUTED, "other default policies"), ("treatment", XS_COLORS[0], "H2 treatments"),
              ("comparator", XS_COLORS[1], f"comparator: {XCOMPARATOR}"))
    for ax, fam in zip(axes[0], fams):
        f = main_long[main_long["family"] == fam].set_index("policy").reindex(pols)
        for role, color, label in groups:
            sel = (f["role"] == role).to_numpy()
            if sel.any():
                ax.errorbar(f["iqm"][sel], y[sel], xerr=_xs_err(f["iqm"][sel], f["iqm_ci_low"][sel], f["iqm_ci_high"][sel]),
                            fmt="o", ms=4, color=color, ecolor=color, elinewidth=1, capsize=2, label=label)
        if XCOMPARATOR in f.index and np.isfinite(f.at[XCOMPARATOR, "iqm"]):
            ax.axvline(f.at[XCOMPARATOR, "iqm"], color=XS_COLORS[1], lw=0.8, ls="--")
        for v in (0.0, 1.0):
            ax.axvline(v, color=XS_MUTED, lw=0.6)
        ax.set_title(f"{fam} (target: {XS_DEPLOY[fam]})", fontsize=9)
        ax.set_xlabel("IQM of normalised score z")
        ax.grid(axis="x", alpha=0.3, lw=0.5)
    axes[0][0].set_yticks(y)
    axes[0][0].set_yticklabels(pols, fontsize=7)
    axes[0][0].invert_yaxis()
    axes[0][-1].legend(fontsize=7, loc="lower right")
    fig.suptitle(f"Deployment performance: IQM with {100 * (1 - ALPHA):.0f}% stratified-bootstrap CI "
                 "(z = 1 optimal, 0 behaviour)")
    return _xs_save(fig, fig_dir, "fig_xenv_iqm")


def xs_fig_profiles(prof: pd.DataFrame, fig_dir: Path) -> str:
    if prof.empty:
        return "not produced: no deployment scores"
    fams = [f for f in XS_FAMILIES if f in set(prof["family"])]
    treat = [t for t in XTREATMENTS if t in set(prof["policy"])]
    blocks = [treat[i:i + 4] for i in range(0, len(treat), 4)] or [[]]
    fig, axes = plt.subplots(len(fams), len(blocks), figsize=(4.4 * len(blocks), 3.4 * len(fams)), squeeze=False,
                             sharex=True, sharey=True)
    for i, fam in enumerate(fams):
        pf = prof[prof["family"] == fam]
        for j, block in enumerate(blocks):
            ax = axes[i][j]
            curves = [(XCOMPARATOR, XS_INK, 2.2)] + [(p, XS_COLORS[q], 1.4) for q, p in enumerate(block)]
            for pol, color, lw in curves:
                g = pf[pf["policy"] == pol]
                if g.empty:
                    continue
                ax.plot(g["threshold"], g["fraction_at_least"], color=color, lw=lw, label=pol)
                ax.fill_between(g["threshold"], g["ci_low"], g["ci_high"], color=color, alpha=0.12, lw=0)
            ax.set_title(f"{fam} (target: {XS_DEPLOY[fam]})", fontsize=9)
            ax.grid(alpha=0.3, lw=0.5)
            ax.legend(fontsize=6, loc="lower left")
            if i == len(fams) - 1:
                ax.set_xlabel("normalised score threshold x")
            if j == 0:
                ax.set_ylabel("fraction of (cell, draw) with z ≥ x", fontsize=8)
    fig.suptitle("Performance profiles of the H2 treatments vs the comparator (black), pointwise stratified-bootstrap bands")
    return _xs_save(fig, fig_dir, "fig_xenv_profiles")


def xs_fig_poi(poi: pd.DataFrame, order: Sequence[str], fig_dir: Path) -> str:
    if poi.empty:
        return "not produced: no deployment scores"
    pols = [p for p in order if p in set(poi["policy_x"])] + [p for p in dict.fromkeys(poi["policy_x"]) if p not in set(order)]
    M = (poi.pivot(index="policy_x", columns="policy_y", values="poi").reindex(index=pols, columns=pols)
         .to_numpy(dtype=np.float64))
    cmap = LinearSegmentedColormap.from_list("xs_diverging", [XS_COLORS[7], XS_NEUTRAL, XS_COLORS[0]]).with_extremes(
        bad="#ffffff")
    n = len(pols)
    fig, ax = plt.subplots(figsize=(0.2 * n + 4.0, 0.2 * n + 3.0))
    im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, vmin=0.0, vmax=1.0)
    fs = float(np.clip(300.0 / max(n, 1), 4.0, 8.0))
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(pols, rotation=90, fontsize=fs)
    ax.set_yticklabels(pols, fontsize=fs)
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="P(row policy > column policy)")
    ax.set_title("Probability of improvement on the deployment targets (rliable; averaged over all cells)", fontsize=9)
    return _xs_save(fig, fig_dir, "fig_xenv_poi")


def xs_fig_error_decomposition(err: pd.DataFrame, fig_dir: Path) -> str:
    if err.empty:
        return "not produced: no benchmark-vs-true comparisons (Gumbel and Garnet off)"
    comps = [c for c in ("mean", "terminate") if c in set(err["completion"])]
    cells = list(dict.fromkeys(err["cell"]))
    x = np.arange(len(cells))
    terms = (("mean_abs_total", "|J_true − J_bench| (total)"), ("mean_abs_semantics", "|semantics term|"),
             ("mean_abs_estimation", "|estimation + coverage term|"))
    fig, axes = plt.subplots(len(comps), 1, figsize=(max(8.0, 0.6 * len(cells) + 3), 3.6 * len(comps)), squeeze=False,
                             sharex=True)
    for ax, comp in zip(axes[:, 0], comps):
        e = err[err["completion"] == comp].set_index("cell").reindex(cells)
        for k, (m, lab) in enumerate(terms):
            ax.bar(x + (k - 1) * 0.27, e[f"{m}_mean"], 0.25, color=XS_COLORS[k], edgecolor="white", lw=0.6,
                   yerr=_xs_err(e[f"{m}_mean"], e[f"{m}_ci_low"], e[f"{m}_ci_high"]), capsize=2,
                   error_kw={"elinewidth": 0.8, "ecolor": XS_INK}, label=lab)
        ax.set_title(f"Completion rule: {comp}", fontsize=9)
        ax.set_ylabel("mean absolute value error", fontsize=9)
        ax.grid(axis="y", alpha=0.3, lw=0.5)
    axes[0, 0].legend(fontsize=8)
    axes[-1, 0].set_xticks(x)
    axes[-1, 0].set_xticklabels(cells, rotation=40, ha="right", fontsize=7)
    fig.suptitle("Exact decomposition of the benchmark error over default policies (mean over draws, bootstrap CI)")
    return _xs_save(fig, fig_dir, "fig_xenv_error_decomposition")


# ---------- orchestration ----------
def xs_run(results: Optional[pd.DataFrame], draw_meta: Optional[pd.DataFrame], frontiers: Any, prop_checks: Any,
           propositions_ok: Optional[bool], cells_spec: Optional[Sequence[dict]], experiments: Dict[str, bool],
           table_dir: Path, fig_dir: Path, n_boot: int) -> Tuple[Dict[str, Any], bool]:
    """Section II.7 end to end: checks, every table, the preregistered tests and the figures. Returns (XSTATS, done)."""
    t0 = time.perf_counter()
    table_dir, fig_dir = Path(table_dir), Path(fig_dir)
    families_on = [f for f in XS_FAMILIES if experiments[f"xenv_{f}"]]
    assert results is not None or not families_on, \
        f"XRESULTS is undefined although {families_on} are switched on (run Section II.6 first)"
    xr = xs_prepare_results(results)
    checks = {"selftests": xs_selftests(), "design": xs_design_checks(xr, cells_spec, experiments),
              "draw_meta": xs_draw_meta_checks(draw_meta, xr)}
    norm, checks["normalization"] = xs_normalized_scores(xr)
    del xr
    gc.collect()
    checks["references"] = xs_reference_checks(norm)
    checks["semantic_groups"] = xs_semantic_group_checks(norm)
    outputs = []

    def write_csv(df: pd.DataFrame, name: str) -> None:
        df.to_csv(table_dir / name, index=False)
        outputs.append(f"xenv/{name}")

    write_csv(norm, "xenv_normalized_scores.csv")
    main_long = xs_main_long(norm, n_boot)
    order = xs_policy_order(main_long)
    main = xs_main_wide(main_long)
    write_csv(main, "table_xenv_main.csv")
    write_csv(main_long, "table_xenv_main_long.csv")
    cell_order = [str(s["cell"]) for s in (cells_spec or [])]
    rank_draws = xs_sort_cells(xs_rank_fidelity_draws(norm), cell_order, ["draw"])
    rank = xs_rank_fidelity_table(rank_draws, n_boot)
    write_csv(rank, "table_rank_fidelity.csv")
    write_csv(rank_draws, "table_rank_fidelity_draws.csv")
    err_draws = xs_sort_cells(xs_error_decomposition_draws(norm), cell_order, ["draw"])
    err = xs_error_decomposition_table(err_draws, n_boot)
    write_csv(err, "table_error_decomposition.csv")
    write_csv(err_draws, "table_error_decomposition_draws.csv")
    h1, h2, h3 = xs_h1(rank_draws, n_boot), xs_h2(norm, n_boot), xs_h3(err_draws)
    h3["descriptive_mask_and_violations"] = xs_h3_descriptive(err_draws, norm, draw_meta)
    prereg = xs_prereg_report(h1, h2, h3)
    save_json(prereg, table_dir / "xenv_prereg_tests.json")
    outputs.append("xenv/xenv_prereg_tests.json")
    write_csv(h2, "xenv_h2_family.csv")
    hp = xs_hp_selection(norm, n_boot)
    write_csv(hp, "table_hp_selection.csv")
    prof = xs_performance_profiles(norm, n_boot)
    write_csv(prof, "table_performance_profiles.csv")
    poi, checks["poi"] = xs_poi_matrix(norm, order)
    write_csv(poi, "xenv_poi_matrix.csv")
    fr, props = xs_as_frame(frontiers), xs_as_frame(prop_checks)
    checks["propositions"] = xs_proposition_summary(props, propositions_ok)
    checks["frontiers"] = {"rows": int(len(fr)), "columns": [str(c) for c in fr.columns]}
    figures = {"fig_xenv_frontiers": xs_fig_frontiers(fr, fig_dir),
               "fig_xenv_rank_fidelity": xs_fig_rank_fidelity(rank, fig_dir),
               "fig_xenv_harm_phase": xs_fig_harm_phase(err, rank, fig_dir),
               "fig_xenv_iqm": xs_fig_iqm(main_long, order, fig_dir),
               "fig_xenv_profiles": xs_fig_profiles(prof, fig_dir),
               "fig_xenv_poi": xs_fig_poi(poi, order, fig_dir),
               "fig_xenv_error_decomposition": xs_fig_error_decomposition(err, fig_dir)}
    outputs += [f"figures/xenv/{k}.{ext}" for k, v in figures.items() if v == "written" for ext in ("png", "pdf")]
    done = bool(checks["design"]["cells"]) and not norm.empty
    checks.update({"figures": figures, "outputs": outputs, "n_boot": int(n_boot), "ci_level": 1 - ALPHA,
                   "deploy_targets": XS_DEPLOY, "xenv_done": done, "elapsed_s": time.perf_counter() - t0})
    save_json(checks, table_dir / "xenv_stats_checks.json")
    outputs.append("xenv/xenv_stats_checks.json")
    xstats = {"normalized_scores": norm, "main": main, "main_long": main_long, "policy_order": order,
              "rank_fidelity": rank, "rank_fidelity_draws": rank_draws, "error_decomposition": err,
              "error_decomposition_draws": err_draws, "prereg_tests": prereg, "h2_family": h2, "hp_selection": hp,
              "performance_profiles": prof, "poi_matrix": poi, "checks": checks, "figures": figures,
              "outputs": outputs}
    return xstats, done


XSTATS, XENV_DONE = xs_run(
    results=globals().get("XRESULTS"), draw_meta=globals().get("XDRAW_META"), frontiers=globals().get("X_FRONTIERS"),
    prop_checks=globals().get("PROP_CHECKS"), propositions_ok=globals().get("PROPOSITIONS_OK"),
    cells_spec=globals().get("X_CELLS"), experiments=EXPERIMENTS, table_dir=XENV_TABLE_DIR, fig_dir=XENV_FIG_DIR,
    n_boot=int(XCFG.bootstrap))

_xn = XSTATS["checks"]["normalization"]
print(f"Normalised scores: {_xn.get('rows', 0)} rows, {_xn.get('z_finite_rows', 0)} finite; "
      f"{_xn.get('groups_denominator_below_1e-9', 0)} (cell, draw, target) groups with J* - J_b < 1e-9 "
      f"({_xn.get('rows_denominator_below_1e-9', 0)} rows set to NaN); {_xn.get('improper_rows', 0)} improper rows.")
if len(XSTATS["main"]):
    _fams = [f for f in XS_FAMILIES if f"{f}_iqm_ci" in XSTATS["main"].columns]
    display(XSTATS["main"][["policy", "role"] + [c for f in _fams for c in (f"{f}_iqm_ci", f"{f}_poi_ci",
                                                                            f"{f}_mean_expected_violations")]])
if len(XSTATS["rank_fidelity"]):
    display(XSTATS["rank_fidelity"][["cell", "completion", "n_draws", "kendall_tau_b_mean", "kendall_tau_b_ci_low",
                                     "kendall_tau_b_ci_high", "top1_agreement_mean", "selection_regret_z_mean"]].round(4))
_pt = XSTATS["prereg_tests"]
if _pt["warning"]:
    print("WARNING:", _pt["warning"])
print("H1:", _pt["H1"]["status"], "| Holm rejections:", _pt["H1"]["n_rejections_holm"], "of", _pt["H1"]["family_size"])
if len(XSTATS["h2_family"]):
    display(XSTATS["h2_family"][["family", "treatment", "test_unit", "n_test_units", "mean_difference",
                                 "mean_difference_ci_low", "mean_difference_ci_high", "iqm_difference", "p_value",
                                 "holm_p", "tost_holm_p", "reject_holm", "equivalent_tost_holm"]].round(5))
print("H3:", _pt["H3"]["status"], {k: _pt["H3"].get(k) for k in ("spearman_rho", "p_value",
                                                                 "p_value_permutation_within_eps",
                                                                 "p_value_permutation_unrestricted")})
if _pt["design_deviations"]:
    print("NOTE: realised Part II design differs from PREREG_PART2['design']:",
          [d["key"] for d in _pt["design_deviations"]])
print("Figures:", XSTATS["figures"])
print("XENV_DONE =", XENV_DONE, "(all Part II statistics written)" if XENV_DONE else
      "(Part II not run: every xenv family switch is off or X_CELLS is empty)")

## II.8 Additional experiments: environments and cells

This section adds three groups of cells to Part II. They test the bracketing predictions outside the designs of
Sections II.3–II.6. There, the Gumbel physician is ε-optimal, so its unsupported actions are mostly harmful, and
the Garnet unsupported rows are exact mixtures by construction. The testbed also omitted two features of
ICU-Sepsis: state aggregation and hidden confounding.

| Switch | Cells | What is new |
|---|---|---|
| `xenv_v3_physicians` | `gumbel|best|n10000|tau20|eps0.05|phys-confounded`, `gumbel|best|n10000|tau20|eps0.05|phys-withhold-abx` | physicians that are not near-optimal (II.8b) |
| `xenv_v3_aggregated` | `gumbel|agg-obs|n10000|tau20|eps0.05`, `gumbel|agg-proj|n10000|tau20|eps0.05` | benchmark built on aggregated observation states: diabetes hidden, glucose optionally dropped (II.8c) |
| `xenv_v3_garnet_random` | `garnet|hrand|eps0.05`, `garnet|hrand|eps0.3` | true rows of the non-preferred actions are independent random rows, outside the harm-mixture family (II.8d) |

**Cached draws stay valid.** No function of Sections II.1–II.7 is changed, and neither is `XConfig`, whose fields
enter `x_result_xcfg`. New behaviour lives in new functions. Their source enters a separate code hash,
`XENV_V3_CODE_HASH` (II.8e), which also folds in `XENV_CODE_HASH`. The new cells have new names, new RNG streams
(named by the cell, as in II.3d) and cache keys under `v3/`. The cached draws of Section II.6 therefore keep their
identities, and the new draws are cached separately.

**Design.** `XV3Config` lists the cells. A cell has as many draws as its family has in `XCFG` (`gumbel_draws`,
`garnet_draws`: 30 in the full profile, 15 in lite, 2 in smoke). Smoke mode shrinks the hospital data as in
`apply_xrun_mode`. Learners, targets, reference rows, invariant checks and result columns are the ones of
Section II.6. `run_v3_cell_draw` runs one draw, and `x_v3_run_cells` runs a list of cells and returns their rows.
The run itself, the statistics and the figures come after this section.

**Prerequisites.** These cells use `run_xmethods` (II.4) and, for Gumbel, the exact simulator of II.3b. If every
switch of Sections II.3–II.7 is off, this cell runs the two checks that the Part II gate (Section 34) requires
whenever an `xenv_*` switch is on: the ICU-Sepsis adapter check and the method-suite self-test. The Gumbel
simulator is then enumerated in II.8b with the same function as II.3b; its Monte-Carlo exactness check runs only
in II.3b.

In [ ]:
# ============================================================
# II.8a Revision experiments: configuration and prerequisites
# ============================================================
EXECLOG.mark("II.8 Revision experiments: environments and cells")

XV3_SWITCHES = ("xenv_v3_physicians", "xenv_v3_aggregated", "xenv_v3_garnet_random")
XV3_ANY_ON = any(bool(EXPERIMENTS[k]) for k in XV3_SWITCHES)
XV3_KINDS = ("physician", "aggregated", "garnet-random")


@dataclass
class XV3Config:
    # Gumbel cells with other physicians: (construction, hospital episodes, tau, physician eps, physician kind).
    physician_cells: Tuple[tuple, ...] = (("best", 10_000, 20, 0.05, "confounded"),
                                          ("best", 10_000, 20, 0.05, "withhold-abx"))
    # Gumbel cells on aggregated states: (aggregation, hospital episodes, tau, physician eps, physician kind).
    aggregated_cells: Tuple[tuple, ...] = (("agg-obs", 10_000, 20, 0.05, "optimal"),
                                           ("agg-proj", 10_000, 20, 0.05, "optimal"))
    # Garnet cells with independent random rows for the non-preferred actions: behaviour eps per cell.
    garnet_random_eps: Tuple[float, ...] = (0.05, 0.30)


def apply_xv3_run_mode(c: XV3Config, mode: str) -> XV3Config:
    if mode == "smoke":
        c.physician_cells = tuple(("best", 300, 5, 0.05, k) for k in ("confounded", "withhold-abx"))
        c.aggregated_cells = tuple((g, 300, 5, 0.05, "optimal") for g in ("agg-obs", "agg-proj"))
        c.garnet_random_eps = (0.05,)
    return c


XV3CFG = apply_xv3_run_mode(XV3Config(), RUN_MODE)
save_json(asdict(XV3CFG), XENV_TABLE_DIR / "v3_config.json")

if XV3_ANY_ON and X_ICU_ADAPTER_OK is None:
    # Same code and output file as II.3a; runs only when every switch of II.3-II.7 is off.
    X_ICU = x_icu_env()
    X_ICU_BEHAVIOR = x_icu_behavior()
    assert np.array_equal(X_ICU_BEHAVIOR, CLINICIAN_POLICY), "ICU behaviour must equal Section 14's CLINICIAN_POLICY"
    X_ICU_ADAPTER_CHECK = x_icu_adapter_check(X_ICU)
    X_ICU_ADAPTER_OK = bool(X_ICU_ADAPTER_CHECK["passes"])
    save_json(X_ICU_ADAPTER_CHECK, XENV_TABLE_DIR / "icu_adapter_check.json")
    assert X_ICU_ADAPTER_OK, "ICU-Sepsis FiniteEnv adapter differs from EVALUATOR (see xenv/icu_adapter_check.json)."
if XV3_ANY_ON and X_METHODS_SELFTEST is None:
    # Same self-test and output file as II.4; runs only when every switch of II.3-II.7 is off.
    X_METHODS_SELFTEST = x_methods_selftest()
    save_json(X_METHODS_SELFTEST, XENV_TABLE_DIR / "xenv_methods_selftest.json")
    assert X_METHODS_SELFTEST["passed"], "Part II method-suite self-test failed; see xenv/xenv_methods_selftest.json"
print("II.8 switches:", {k: bool(EXPERIMENTS[k]) for k in XV3_SWITCHES}, "| config:", asdict(XV3CFG))

### II.8b Gumbel physicians that are not near-optimal

`gumbel_physician_v3(env, ε, kind)` extends `gumbel_physician` (II.3b) with two kinds. Every kind puts 1 − ε on
one action per state and ε/7 on each other action. The kinds `optimal` and `withhold-vaso` call
`gumbel_physician` unchanged.

* **`confounded`**: the optimal policy of the diabetes-blind model, here the non-diabetic dynamics, applied to
  every patient. A diabetic patient gets the optimal action of the non-diabetic patient with the same observation
  (hr, sbp, o2, glucose, treatment flags; `gumbel_diabetes_counterpart`). A step never changes the diabetes
  indicator, so the live states of the two groups never reach each other; this is asserted on P. The true optimum
  restricted to the non-diabetic states is therefore the optimum of the non-diabetic model. Non-diabetic patients
  are treated exactly as by the `optimal` physician. Diabetic patients are treated as if they were not diabetic.
* **`withhold-abx`**: the exact optimum among policies that never give antibiotics on live states (admissible
  optimum under the mask 1[antibiotic bit of a = 0]; a = 4·abx + 2·vent + vaso). Antibiotic actions then get ε/7
  each, so they are rarely supported in the hospital data, although they can be beneficial in truth.

The benchmark construction, user data and targets of these cells are those of the `best` construction (II.3d),
with the five targets `bench_mean`, `bench_terminate`, `true`, `true_rows_mean` and `true_rows_terminate`.
`xenv/v3_physician_report.csv` describes each physician on the true system: its value, the value of its
deterministic core, how often its action differs from the true optimum (over live states, weighted by its own
occupancy, and by diabetes group), and how often it gives each treatment.

In [ ]:
# ============================================================
# II.8b Gumbel physicians: confounded and antibiotic-withholding
# ============================================================
GUMBEL_PHYSICIAN_KINDS_V3 = GUMBEL_PHYSICIAN_KINDS + ("confounded", "withhold-abx")


def gumbel_diabetes_counterpart(env: FiniteEnv) -> np.ndarray:
    """[S] index map of the exact Gumbel env: a live diabetic state -> the live non-diabetic state with the same
    observation (full index - 720); non-diabetic and absorbing states map to themselves."""
    space = gumbel_state_space()
    live_full = space["live_full"]
    n_live = int(env.meta["n_live"])
    assert env.kind == "gumbel" and n_live == len(live_full) and env.S == n_live + 3, env.name
    twin = np.arange(env.S)
    for k, f in enumerate(live_full):
        diab, obs = divmod(int(f), GumbelSim.N_OBS)
        if diab == 1:
            assert obs in space["pos"], ("the non-diabetic twin of a live state must be live", int(f))
            twin[k] = space["pos"][obs]
    assert np.all(twin[twin] == twin), "twins of diabetic states must be non-diabetic"
    return twin


def gumbel_physician_v3(env: FiniteEnv, eps: float, kind: str = "optimal") -> np.ndarray:
    """eps-softened Gumbel physician (Section II.8b): 1 - eps on its action, eps/(A-1) on every other action.

    'optimal' / 'withhold-vaso': exactly gumbel_physician (II.3b). 'confounded': the optimum of the non-diabetic
    dynamics applied to every patient (diabetic states take the action of their non-diabetic twin). 'withhold-abx':
    the exact optimum among policies that never give antibiotics on live states."""
    if kind in GUMBEL_PHYSICIAN_KINDS:
        return gumbel_physician(env, eps, kind)
    assert 0.0 <= eps < 1.0, eps
    A = env.A
    if kind == "confounded":
        twin = gumbel_diabetes_counterpart(env)
        n_live = int(env.meta["n_live"])
        is_diab = np.zeros(env.S, dtype=bool)
        is_diab[:n_live] = gumbel_state_space()["live_full"] >= GumbelSim.N_OBS
        non_diab = env.live & ~is_diab
        assert env.P[non_diab][:, :, is_diab].max() == 0.0 and env.P[is_diab][:, :, non_diab].max() == 0.0, \
            "the diabetes groups must not mix during a step"
        pi_opt, _, _ = x_optimal(env, "given")
        act = pi_opt.argmax(axis=1)[twin]
    elif kind == "withhold-abx":
        no_abx = np.ones((env.S, A), dtype=bool)
        no_abx[env.live] = np.array([gumbel_action_bits(a)[0] == 0 for a in range(A)])[None, :]
        pi_opt, _, _ = x_optimal(env.variant(f"{env.name}|no-antibiotics", mask=no_abx), "given", constrained=True)
        act = pi_opt.argmax(axis=1)
        assert all(gumbel_action_bits(a)[0] == 0 for a in act[env.live]), "withhold-abx physician gives antibiotics"
    else:
        raise ValueError(f"unknown Gumbel physician kind {kind!r} (expected one of {GUMBEL_PHYSICIAN_KINDS_V3})")
    pi = np.full((env.S, A), eps / (A - 1))
    pi[np.arange(env.S), act] = 1.0 - eps
    return validate_policy(pi, f"Gumbel physician ({kind}) eps={eps}", shape=(env.S, A))


_X_V3_GUMBEL_ENVS: Dict[Tuple[str, float], FiniteEnv] = {}
_X_V3_GUMBEL_PHYS: Dict[tuple, np.ndarray] = {}
_X_V3_GUMBEL_OPT_ACTION: Dict[Tuple[str, float], np.ndarray] = {}


def _x_v3_gumbel_env(variant: str, hazard: float) -> FiniteEnv:
    """The exact Gumbel env of II.3b when it was built there; otherwise enumerated here by the same function."""
    env = X_GUMBEL_ENVS.get(variant)
    if env is not None and float(env.meta["timeout_hazard"]) == float(hazard):
        return env
    key = (str(variant), float(hazard))
    if key not in _X_V3_GUMBEL_ENVS:
        _X_V3_GUMBEL_ENVS[key] = gumbel_true_env(variant, hazard)
    return _X_V3_GUMBEL_ENVS[key]


def _x_v3_gumbel_physician(variant: str, hazard: float, eps: float, kind: str) -> np.ndarray:
    key = (str(variant), float(hazard), float(eps), str(kind))
    if key not in _X_V3_GUMBEL_PHYS:
        _X_V3_GUMBEL_PHYS[key] = gumbel_physician_v3(_x_v3_gumbel_env(variant, hazard), eps, kind)
    return _X_V3_GUMBEL_PHYS[key]


def _x_v3_gumbel_opt_action(variant: str, hazard: float) -> np.ndarray:
    """argmax of the exact optimum of the Gumbel system (cached per variant and hazard)."""
    key = (str(variant), float(hazard))
    if key not in _X_V3_GUMBEL_OPT_ACTION:
        pi_opt, _, _ = x_optimal(_x_v3_gumbel_env(variant, hazard), "given")
        _X_V3_GUMBEL_OPT_ACTION[key] = pi_opt.argmax(axis=1)
    return _X_V3_GUMBEL_OPT_ACTION[key]


def gumbel_physician_report(env: FiniteEnv, eps: float, kinds: Sequence[str] = GUMBEL_PHYSICIAN_KINDS_V3) -> pd.DataFrame:
    """One row per physician kind: exact values on the true system and disagreement with the true optimum."""
    _, _, opt_info = x_optimal(env, "given")
    opt = _x_v3_gumbel_opt_action(env.meta["variant"], env.meta["timeout_hazard"])
    n_live = int(env.meta["n_live"])
    is_diab = np.zeros(env.S, dtype=bool)
    is_diab[:n_live] = gumbel_state_space()["live_full"] >= GumbelSim.N_OBS
    L = env.live
    bits = np.array([gumbel_action_bits(a) for a in range(env.A)])      # (abx, vent, vaso) per action
    rows = []
    for kind in kinds:
        pi = gumbel_physician_v3(env, eps, kind)
        act = pi.argmax(axis=1)
        ev = x_evaluate(env, pi, "given", vectors=True)
        det = x_evaluate(env, one_hot_policy(act, env.A), "given")
        assert ev["status"] == "ok" and det["status"] == "ok", (kind, ev["status"], det["status"])
        occ = ev["occupancy_vector"]
        differ = (act != opt) & L
        rows.append({"kind": kind, "eps": float(eps), "value_true": float(ev["return"]),
                     "value_deterministic_core": float(det["return"]), "optimum_value": float(opt_info["dp_value"]),
                     "disagree_live_fraction": float(differ[L].mean()),
                     "disagree_occupancy_weighted": float((occ * differ).sum() / occ.sum()),
                     "disagree_diabetic_live_fraction": float(differ[L & is_diab].mean()),
                     "disagree_nondiabetic_live_fraction": float(differ[L & ~is_diab].mean()),
                     "antibiotic_share_live": float(bits[act[L], 0].mean()),
                     "ventilation_share_live": float(bits[act[L], 1].mean()),
                     "vasopressor_share_live": float(bits[act[L], 2].mean())})
    return pd.DataFrame(rows)


X_V3_PHYSICIAN_REPORT = pd.DataFrame()
if EXPERIMENTS["xenv_v3_physicians"] or EXPERIMENTS["xenv_v3_aggregated"]:
    _t0 = time.perf_counter()
    _env = _x_v3_gumbel_env(XCFG.gumbel_variant, XCFG.gumbel_timeout_hazard)
    _eps = sorted({float(c[3]) for c in (*XV3CFG.physician_cells, *XV3CFG.aggregated_cells)})
    X_V3_PHYSICIAN_REPORT = pd.concat([gumbel_physician_report(_env, e) for e in _eps], ignore_index=True)
    X_V3_PHYSICIAN_REPORT.to_csv(XENV_TABLE_DIR / "v3_physician_report.csv", index=False)
    display(X_V3_PHYSICIAN_REPORT.round(4))
    print(f"II.8b physician report in {time.perf_counter() - _t0:.1f} s.")
    del _env, _eps

### II.8c Benchmarks on aggregated states: state aggregation and hidden confounding

ICU-Sepsis estimates its benchmark on k-means clusters of patient features (II.3d), while the clinicians in its
source data acted on the patients, not on the clusters. The cells of II.3–II.6 had neither feature, because their
benchmark was estimated on the true Markov states. The aggregated cells keep the exact Gumbel system as the truth and the
physician of II.3b, who sees the full state, diabetes included. The benchmark, however, is built on an
aggregation φ of the true states:

* **`agg-obs`**: φ(s) is the observation without the diabetes indicator (full index mod 720, i.e. 3·3·2·5·2·2·2 =
  720 observation cells). A live state and its diabetic twin share one cell, so the benchmark has 303 live cells.
  Diabetes is a hidden confounder: it changes both the physician's action and the dynamics (vasopressors, glucose),
  but the benchmark cannot see it.
* **`agg-proj`**: glucose is dropped as well, φ(s) = (hr, sbp, o2, abx, vaso, vent) with 3·3·2·2·2·2 = 144 cells, of
  which 112 contain a live state (at most two abnormal vitals among hr, sbp, o2). Death and discharge then also
  depend on the unobserved glucose level.

`x_gumbel_aggregation` returns φ as an index map. Live cells keep the increasing order of their cell index, and
death, discharge and timeout follow in the true order. Absorbing states map to absorbing states, so φ preserves
absorption, entry rewards and the timeout hazard h.

**Construction** (`construct_aggregated_benchmark`). Hospital episodes are simulated on the true full-state
system with the physician and validated there. They are then mapped through φ (`x_aggregate_dataset`) and
validated again on the aggregated state space. The benchmark is then estimated by exactly the code of II.3d
(`x_benchmark_from_data`): count mask with threshold τ, ICU-Sepsis dead-state removal, known-horizon
maximum-likelihood rows, mean completion, empirical d0 and clinician estimate. Benchmark users and learners act on
the cells. `x_benchmark_from_data` reads the state space, the absorbing structure, the rewards, the censoring hazard
and d0 of its "true" environment; it never reads P. Here that environment is `x_aggregated_env`, the population
limit of the count estimate. Its row for (x, a) is Σ_s μ_β(s) π_β(a|s) P(φ⁻¹(·)|s, a) / Σ_s μ_β(s) π_β(a|s) over
the members s of x, where μ_β is the physician's exact occupancy (equal weights for members it never visits). Its
d0 is the pushforward of the true d0. Under the aggregated physician π_β,agg(a|x) = Σ_s μ_β(s) π_β(a|s) / Σ_s μ_β(s),
this model reproduces the true occupancy of every cell, and hence the true value of the physician; this is
asserted to 1e-9.

**Targets.** `bench_mean` and `bench_terminate` live on the benchmark. `true` is the true full-state system with
the pulled-back mask M(φ(s)), evaluated with the lifted policy π(a | φ(s)). Cells outside the benchmark play the
most frequent hospital action, as in II.3d. The `true_rows_*` targets of II.3d are not used, because they compare
rows on one state space. `REF optimum` on `true` is the optimum over full-state policies, which see diabetes;
a policy on the cells cannot always reach it.

In [ ]:
# ============================================================
# II.8c Aggregated-state Gumbel benchmarks
# ============================================================
def x_gumbel_aggregation(env: FiniteEnv, aggregation: str) -> Dict[str, Any]:
    """Index map phi of the exact Gumbel env onto observation cells: 'obs' drops diabetes, 'proj' also glucose."""
    assert aggregation in ("obs", "proj"), aggregation
    space = gumbel_state_space()
    live_full = space["live_full"]
    n_live = int(env.meta["n_live"])
    assert env.kind == "gumbel" and n_live == len(live_full) and env.S == n_live + 3, env.name
    death, discharge, timeout = n_live, n_live + 1, n_live + 2
    assert (env.terminals, env.sink, env.goal, int(env.meta["timeout_state"])) == \
        ((death, discharge, timeout), death, discharge, timeout), "unexpected absorbing layout"
    factors = (("hr", "sbp", "o2", "glucose", "abx", "vaso", "vent") if aggregation == "obs"
               else ("hr", "sbp", "o2", "abx", "vaso", "vent"))
    fidx = [GumbelSim.FACTORS.index(f) for f in factors]
    radix = np.array([GumbelSim.LEVELS[i] for i in fidx], dtype=np.int64)
    strides = np.ones(len(radix), dtype=np.int64)
    for j in range(len(radix) - 2, -1, -1):
        strides[j] = strides[j + 1] * radix[j + 1]                      # first factor most significant
    full = np.array([space["full"][f] for f in live_full], dtype=np.int64)
    cell = full[:, fidx] @ strides
    if aggregation == "obs":
        assert np.array_equal(cell, live_full % GumbelSim.N_OBS), "obs cell must equal full index mod 720"
    keys = np.unique(cell)
    n_agg = int(len(keys))
    phi = np.empty(env.S, dtype=np.int64)
    phi[:n_live] = np.searchsorted(keys, cell)
    phi[n_live:] = n_agg + np.arange(3)
    levels = (keys[:, None] // strides[None, :]) % radix[None, :]
    members = np.bincount(phi[:n_live], minlength=n_agg)
    assert members.min() >= 1 and int(members.sum()) == n_live
    return {"aggregation": aggregation, "phi": phi, "n_live": n_agg, "S": n_agg + 3,
            "cells_total": int(radix.prod()), "factors": factors, "radix": radix, "keys": keys, "levels": levels,
            "members": members}


def x_aggregated_env(true_env: FiniteEnv, agg: Dict[str, Any], behavior: np.ndarray,
                     name: str) -> Tuple[FiniteEnv, Dict[str, Any]]:
    """Population limit of the benchmark's count estimate on the cells of `agg` under `behavior` (see markdown)."""
    phi, Sa, nA = agg["phi"], int(agg["S"]), int(agg["n_live"])
    S, A = true_env.S, true_env.A
    beh = validate_policy(behavior, f"{name} behaviour", shape=(S, A))
    G = np.zeros((S, Sa))
    G[np.arange(S), phi] = 1.0
    ev = x_evaluate(true_env, beh, "given", vectors=True)
    assert ev["status"] == "ok", ev
    mu = ev["occupancy_vector"]
    Pphi = (true_env.P.reshape(S * A, S) @ G).reshape(S, A, Sa)
    W = mu[:, None] * beh
    W[~true_env.live] = 0.0
    W_eq = np.where(true_env.live[:, None], 1.0, 0.0) * np.ones((1, A))
    num = (G.T @ (W[:, :, None] * Pphi).reshape(S, A * Sa)).reshape(Sa, A, Sa)
    den = G.T @ W
    num_eq = (G.T @ (W_eq[:, :, None] * Pphi).reshape(S, A * Sa)).reshape(Sa, A, Sa)
    den_eq = G.T @ W_eq
    P = np.zeros((Sa, A, Sa))
    seen = den[:nA] > 0
    P[:nA] = np.where(seen[:, :, None], num[:nA] / np.maximum(den[:nA], 1e-300)[:, :, None],
                      num_eq[:nA] / np.maximum(den_eq[:nA], 1e-300)[:, :, None])
    for t in range(nA, Sa):
        P[t, :, t] = 1.0
    entry = np.zeros(Sa)
    for t in true_env.terminals:
        entry[phi[t]] = true_env.entry_reward[t]
    d0 = G.T @ true_env.d0
    radix = np.asarray(agg["radix"], dtype=np.float64)
    feats = np.zeros((Sa, len(radix)), dtype=np.float32)
    feats[:nA] = agg["levels"] / np.maximum(radix - 1.0, 1.0)
    timeout = int(phi[int(true_env.meta["timeout_state"])])
    env = FiniteEnv(name=name, kind="aggregate", P=P, entry_reward=entry, d0=d0,
                    terminals=tuple(int(phi[t]) for t in true_env.terminals), sink=int(phi[true_env.sink]),
                    goal=int(phi[true_env.goal]), mask=np.ones((Sa, A), dtype=bool), features=feats,
                    meta={"aggregation": agg["aggregation"], "factors": list(agg["factors"]), "true_env": true_env.name,
                          "n_live": nA, "weights": "behaviour occupancy", "behavior": array_hash(beh),
                          "censoring_hazard": true_env.meta["censoring_hazard"], "timeout_state": timeout,
                          "dangling_target": timeout, "dangling_target_label": "timeout"})
    row_dev = float(np.abs(P.sum(axis=2) - 1.0).max())
    assert row_dev < 1e-12 and (P >= 0).all(), row_dev
    assert x_trap_states(env.P, env.live, np.ones((Sa, A), dtype=bool)) == 0, "aggregated model must absorb"
    occ_agg = G.T @ mu
    pi_agg = np.full((Sa, A), 1.0 / A)
    vis = occ_agg[:nA] > 0
    pi_agg[:nA][vis] = (G.T @ (mu[:, None] * beh))[:nA][vis] / occ_agg[:nA][vis, None]
    ev_agg = x_evaluate(env, pi_agg, "given", vectors=True)
    assert ev_agg["status"] == "ok", ev_agg
    check = {"behaviour_value_true": float(ev["return"]), "behaviour_value_aggregated_model": float(ev_agg["return"]),
             "behaviour_value_abs_diff": abs(float(ev["return"]) - float(ev_agg["return"])),
             "cell_occupancy_max_abs_diff": float(np.abs(ev_agg["occupancy_vector"] - occ_agg).max()),
             "cells_visited_by_behaviour": int(vis.sum()), "max_row_sum_deviation": row_dev}
    assert check["behaviour_value_abs_diff"] <= 1e-9 and check["cell_occupancy_max_abs_diff"] <= 1e-8 * max(1.0, occ_agg.max()), \
        f"{name}: the aggregated model does not reproduce the behaviour's occupancy ({check})"
    return env, check


def x_aggregate_dataset(data: Dict[str, np.ndarray], phi: np.ndarray, agg_env: FiniteEnv) -> Dict[str, np.ndarray]:
    """The same episodes with states and next states mapped to their cells (rows, order and rewards unchanged)."""
    out = {k: np.array(v, copy=True) for k, v in data.items()}
    out["state"] = phi[data["state"]].astype(np.int64)
    out["next_state"] = phi[data["next_state"]].astype(np.int64)
    out["logged_action_admissible"] = agg_env.mask[out["state"], out["action"]]
    return out


def construct_aggregated_benchmark(true_env: FiniteEnv, behavior: np.ndarray, aggregation: str, n_hosp: int,
                                   tau: int, rng: np.random.Generator, name: str) -> Dict[str, Any]:
    """Hospital data from the true full-state system -> ICU-style benchmark on the cells phi (Section II.8c)."""
    agg = x_gumbel_aggregation(true_env, aggregation)
    phi = agg["phi"]
    agg_env, agg_check = x_aggregated_env(true_env, agg, behavior, f"{true_env.name}|agg-{aggregation}")
    hosp = x_sample_dataset(true_env, behavior, int(n_hosp), XCFG.max_steps, rng)
    hosp_report = x_validate_dataset(hosp, true_env)
    hosp_agg = x_aggregate_dataset(hosp, phi, agg_env)
    agg_report = x_validate_dataset(hosp_agg, agg_env)
    out = x_benchmark_from_data(agg_env, hosp_agg, tau, name)
    lift_cells = out["lift"]
    S, A = true_env.S, true_env.A

    def lift(pi_bench: np.ndarray) -> np.ndarray:
        """Benchmark policy -> true full-state policy pi(a | phi(s)) (absorbing rows uniform)."""
        return lift_cells(pi_bench)[phi]

    mask_true = out["mask_true"][phi]
    assert mask_true.shape == (S, A) and mask_true[~true_env.live].all()
    cells = out["decision_states_true"]                     # cell indices of the benchmark's decision states
    in_cells = true_env.live & np.isin(phi, cells)
    logged = np.zeros(S, dtype=bool)
    logged[np.unique(hosp["state"])] = True
    rep = out["report"]
    rep.update({"construction": f"agg-{aggregation}", "aggregation": aggregation,
                "aggregated_cells_total": agg["cells_total"], "aggregated_live_cells": agg["n_live"],
                "members_per_live_cell_max": int(agg["members"].max()),
                "full_live_states": int(true_env.live.sum()), "full_states_logged": int(logged.sum()),
                "full_live_states_in_decision_cells": int(in_cells.sum()),
                "full_logged_states_in_decision_cells": int((in_cells & logged).sum()),
                "full_d0_mass_in_decision_cells": float(true_env.d0[in_cells].sum()),
                **{f"aggregation_check_{k}": v for k, v in agg_check.items()},
                "hospital_validation": hosp_report, "aggregated_hospital_validation": agg_report})
    del hosp, hosp_agg
    return {"bench": out["bench"], "bench_behavior": out["bench_behavior"], "index": out["index"],
            "mask_true": mask_true, "lift": lift, "fallback_action": out["fallback_action"], "report": rep,
            "decision_states_true": np.flatnonzero(in_cells & logged), "decision_cells": cells, "phi": phi,
            "agg_env": agg_env}

### II.8d Garnet with independent random rows for unsupported actions

In the harm family (II.3c), the true row of every non-preferred action is a mixture,
(1 − |h|)·mean_{K(s)} P_base + |h|·e_death (or e_goal), so the sign of the hidden benefit of unsupported actions
is fixed by h. The `hrand` cells leave that family. A draw keeps the base MDP, the preferred sets K(s) and the
behaviour policy of the harm cells of the same draw (`garnet_true_env` with stream `("xenv", "garnet", "base",
draw)`, so the cells share common random numbers). Every **non-preferred** pair (s, a ∉ K(s)) then gets an
independent row from the Garnet row generator. The generator absorbs with hazard U(`garnet_hazard`), reaching the
goal with probability U(0, 1) and death otherwise; the remaining mass moves to `branching` distinct uniformly random
live successors with Dirichlet(1) weights. The rows come from their own stream,
`("xenv", "garnet", "random-rows", "base", draw)`. They are drawn for all S·A pairs, so they do not depend on ε,
and they are drawn after K(s) is fixed, so they are independent of the preference. Preferred rows stay the base
rows; this is asserted.

Whether an unsupported action is better or worse than the supported ones is therefore random per pair. The theorem's
no-hidden-benefit condition can hold or fail, which tests the boundary of the bracketing result. Each draw reports,
under the exact optimum of its true system, the share of non-preferred pairs whose action value exceeds the mean
value of the preferred actions (`nonpreferred_beneficial_share`), and the mean, minimum and maximum of that
advantage. Every row absorbs with probability at least min(`garnet_hazard`) > 0, so every policy is proper (asserted).
The benchmark construction, user data and the five targets are those of the Garnet cells in II.3d.

In [ ]:
# ============================================================
# II.8d Garnet family with independent random unsupported rows
# ============================================================
def garnet_random_rows_env(draw_key: tuple, n_states: int, n_actions: int, branching: int,
                           hazard_range: Tuple[float, float], eps: float) -> Tuple[FiniteEnv, np.ndarray, Dict[str, Any]]:
    """True Garnet CMDP of one draw whose non-preferred rows are independent random rows, its behaviour policy with
    softening `eps` (identical to the harm cells of the draw), and a report of the hidden benefit of those rows."""
    S, A, B = int(n_states), int(n_actions), int(branching)
    lo, hi = (float(x) for x in hazard_range)
    eps = float(eps)
    assert 0.0 < eps < (A - 3) / A, "preferred actions must keep a larger behaviour probability than the others"
    base0, pi = garnet_true_env(draw_key, S, A, B, (lo, hi), 0.0, eps)
    goal, death = S, S + 1
    assert (base0.goal, base0.sink) == (goal, death)
    pl = pi[:S]
    K = pl > pl.min(axis=1, keepdims=True)                               # preferred set K(s) of the behaviour
    k = K.sum(axis=1)
    assert np.isin(k, (1, 2, 3)).all() and np.allclose((pl * K).sum(axis=1), 1.0 - eps, atol=1e-12)
    rng = np_rng("xenv", "garnet", "random-rows", *draw_key)
    succ = np.argsort(rng.random((S * A, S)), axis=1)[:, :B]
    w = rng.dirichlet(np.ones(B), size=S * A)
    haz = rng.uniform(lo, hi, size=S * A)
    q_goal = rng.uniform(0.0, 1.0, size=S * A)
    rows = np.zeros((S * A, S + 2))
    rows[np.arange(S * A)[:, None], succ] = (1.0 - haz)[:, None] * w
    rows[:, goal] = haz * q_goal
    rows[:, death] = haz * (1.0 - q_goal)
    rows = rows.reshape(S, A, S + 2)
    P = base0.P.copy()
    PL = P[:S]
    PL[~K] = rows[~K]
    assert np.array_equal(P[:S][K], base0.P[:S][K]), "preferred rows must stay the base rows"
    tag = "|".join(str(x) for x in draw_key)
    meta = {key: val for key, val in base0.meta.items() if key != "harm"}
    meta.update(row_model="independent Garnet rows on non-preferred pairs",
                row_stream=["xenv", "garnet", "random-rows", *draw_key])
    env = FiniteEnv(name=f"garnet|hrand|eps{eps}|{tag}", kind="garnet", P=P, entry_reward=base0.entry_reward,
                    d0=base0.d0, terminals=base0.terminals, sink=death, goal=goal, mask=np.ones((S + 2, A), bool),
                    meta=meta)
    assert np.allclose(env.P.sum(axis=2), 1.0, atol=1e-12) and (env.P >= 0).all()
    assert x_trap_states(env.P, env.live, env.mask) == 0, "every Garnet policy must be proper"
    _, V, _ = x_optimal(env, "given")
    Q = env.evaluator().q_values(V, "mean")[:S]
    adv = Q - ((Q * K).sum(axis=1) / k)[:, None]
    a_np = adv[~K]
    report = {"nonpreferred_pairs": int((~K).sum()),
              "nonpreferred_beneficial_share": float((a_np > 1e-12).mean()),
              "nonpreferred_mean_advantage": float(a_np.mean()), "nonpreferred_min_advantage": float(a_np.min()),
              "nonpreferred_max_advantage": float(a_np.max()),
              "nonpreferred_mean_absorption": float(rows[~K][:, [goal, death]].sum(axis=1).mean())}
    return env, validate_policy(pi, f"Garnet behaviour {env.name}", shape=(S + 2, A)), report


_X_V3_GARNET_CACHE: "OrderedDict[tuple, Tuple[FiniteEnv, np.ndarray, Dict[str, Any]]]" = OrderedDict()


def _x_v3_garnet_env(spec: dict, draw: int, keep: int = 4) -> Tuple[FiniteEnv, np.ndarray, Dict[str, Any]]:
    key = (spec["cell"], int(draw))
    if key in _X_V3_GARNET_CACHE:
        _X_V3_GARNET_CACHE.move_to_end(key)
        return _X_V3_GARNET_CACHE[key]
    val = garnet_random_rows_env(("base", int(draw)), spec["n_states"], spec["n_actions"], spec["branching"],
                                 spec["hazard_range"], spec["eps"])
    _X_V3_GARNET_CACHE[key] = val
    while len(_X_V3_GARNET_CACHE) > keep:
        _X_V3_GARNET_CACHE.popitem(last=False)
    return val

### II.8e Cells, instances, identities and the II.8 driver

`X_V3_CELLS` lists the enabled II.8 cells (`x_v3_build_cells`). Each spec carries the keys of an II.3d spec of its
family, plus `v3_kind` ∈ {`physician`, `aggregated`, `garnet-random`}. `build_v3_instance(spec, draw)` mirrors
`build_cell_instance`, the `best` construction of II.3d. Every random stream is named
`("xenv", purpose, cell, draw)`, with purpose ∈ {hospital, user, split}. The hospital stream is used first, then the
user stream for the benchmark-user data, then the split stream for the train / dev split. For a spec that describes
an II.3d cell, the instance equals the one `build_cell_instance` builds (checked when this section was written).

| `v3_kind` | true system and behaviour | construction | targets |
|---|---|---|---|
| `physician` | exact Gumbel system, `gumbel_physician_v3` | `construct_benchmark` | `bench_mean`, `bench_terminate`, `true`, `true_rows_mean`, `true_rows_terminate` |
| `aggregated` | exact Gumbel system, `gumbel_physician_v3` (full-state physician) | `construct_aggregated_benchmark` | `bench_mean`, `bench_terminate`, `true` (lifted through φ) |
| `garnet-random` | `garnet_random_rows_env` | `construct_benchmark` | the five targets of the `physician` row |

The deployment target is `true` in every cell. `run_v3_cell_draw(spec, draw)` performs the steps of
`run_cell_draw` (II.6) with the same learners (`run_xmethods` on the training episodes), the same records
(behaviour + projection of every native record), the same exact evaluation, reference rows and invariant checks
(`x_check_draw_rows`, projection identity strict), and the same result columns (`XRESULT_COLUMNS`). Its identity is
`x_draw_identity` with the code hash replaced by `XENV_V3_CODE_HASH`. That hash covers the source of every II.8
function on the path and the values of the module constants these functions read, and it folds in
`XENV_CODE_HASH`. Results are cached in `XCACHE` under `v3/<cell>/draw<k>` (and `/meta`), and the table returned is
always the one read back. `x_v3_run_cells(cells)` loops over cells and draws with the progress and ETA of II.6a. It
returns the stacked rows, the flattened draw metadata and the cross-draw check `x_check_results`, and it writes
nothing.

`xenv/v3_environment_suite.csv` (draw 0 of every enabled II.8 cell) has the columns of
`xenv_environment_suite.csv`, plus the II.8 keys and the flattened construction report: aggregation sizes and the
behaviour-aggregation identity, physician kind, and the hidden-benefit statistics of the random Garnet rows.
`xenv/v3_environment_checks.json` holds the structural pass flag of this section (`X_V3_ENV_CHECK["passes"]`).

In [ ]:
# ============================================================
# II.8e II.8 cells, instances, identities, driver
# ============================================================
def x_v3_build_cells() -> List[dict]:
    """Enabled Section II.8 cells (spec keys as in x_build_cells, plus v3_kind)."""
    cells: List[dict] = []
    gumbel_common = {"family": "gumbel", "draws": int(XCFG.gumbel_draws), "variant": XCFG.gumbel_variant,
                     "timeout_hazard": float(XCFG.gumbel_timeout_hazard), "user_episodes": int(XCFG.gumbel_user_episodes)}
    if EXPERIMENTS["xenv_v3_physicians"]:
        for construction, n_hosp, tau, eps, kind in XV3CFG.physician_cells:
            assert construction == "best" and kind in GUMBEL_PHYSICIAN_KINDS_V3, (construction, kind)
            tag = "" if kind == "optimal" else f"|phys-{kind}"
            cells.append({"cell": f"gumbel|best|n{n_hosp}|tau{tau}|eps{eps}{tag}", "v3_kind": "physician",
                          "construction": "best", "n_hosp": int(n_hosp), "tau": int(tau), "eps": float(eps),
                          "physician": str(kind), **gumbel_common})
    if EXPERIMENTS["xenv_v3_aggregated"]:
        for construction, n_hosp, tau, eps, kind in XV3CFG.aggregated_cells:
            assert construction in ("agg-obs", "agg-proj") and kind in GUMBEL_PHYSICIAN_KINDS_V3, (construction, kind)
            tag = "" if kind == "optimal" else f"|phys-{kind}"
            cells.append({"cell": f"gumbel|{construction}|n{n_hosp}|tau{tau}|eps{eps}{tag}", "v3_kind": "aggregated",
                          "construction": construction, "aggregation": construction.split("-", 1)[1],
                          "n_hosp": int(n_hosp), "tau": int(tau), "eps": float(eps), "physician": str(kind),
                          **gumbel_common})
    if EXPERIMENTS["xenv_v3_garnet_random"]:
        for eps in XV3CFG.garnet_random_eps:
            cells.append({"cell": f"garnet|hrand|eps{eps}", "family": "garnet", "v3_kind": "garnet-random",
                          "draws": int(XCFG.garnet_draws), "row_model": "independent random rows", "eps": float(eps),
                          "n_states": int(XCFG.garnet_states), "n_actions": int(XCFG.garnet_actions),
                          "branching": int(XCFG.garnet_branching),
                          "hazard_range": tuple(float(x) for x in XCFG.garnet_hazard),
                          "n_hosp": int(XCFG.garnet_hosp_episodes), "tau": int(XCFG.garnet_tau),
                          "user_episodes": int(XCFG.garnet_user_episodes)})
    keys = [c["cell"] for c in cells]
    assert len(keys) == len(set(keys)), f"duplicate II.8 cell keys: {keys}"
    clash = sorted(set(keys) & {c["cell"] for c in X_CELLS})
    assert not clash, f"II.8 cells must not reuse II.3d cell names: {clash}"
    return cells


def build_v3_instance(spec: dict, draw: int) -> CellInstance:
    """Deterministic (cell, draw) instance of a Section II.8 cell (same streams and steps as build_cell_instance)."""
    t0 = time.perf_counter()
    cell, family, kind, draw = spec["cell"], spec["family"], spec["v3_kind"], int(draw)
    assert draw >= 0 and kind in XV3_KINDS, (draw, kind)
    rng_h, rng_u = np_rng("xenv", "hospital", cell, draw), np_rng("xenv", "user", cell, draw)
    bname = f"{cell}|d{draw}|bench"
    row_report = None
    if kind in ("physician", "aggregated"):
        assert family == "gumbel", family
        true_env = _x_v3_gumbel_env(spec["variant"], spec["timeout_hazard"])
        true_behavior = _x_v3_gumbel_physician(spec["variant"], spec["timeout_hazard"], spec["eps"], spec["physician"])
        if kind == "physician":
            assert spec["construction"] == "best", spec["construction"]
            B = construct_benchmark(true_env, true_behavior, spec["n_hosp"], spec["tau"], rng_h, bname)
        else:
            B = construct_aggregated_benchmark(true_env, true_behavior, spec["aggregation"], spec["n_hosp"],
                                               spec["tau"], rng_h, bname)
    else:
        assert family == "garnet", family
        true_env, true_behavior, row_report = _x_v3_garnet_env(spec, draw)
        B = construct_benchmark(true_env, true_behavior, spec["n_hosp"], spec["tau"], rng_h, bname)
    learn_env, behavior, lift = B["bench"], B["bench_behavior"], B["lift"]
    data = x_sample_dataset(learn_env, behavior, spec["user_episodes"], XCFG.max_steps, rng_u)
    T_masked, T_meanimp = x_true_targets(true_env, B["mask_true"])
    targets = [("bench_mean", learn_env, "given", None), ("bench_terminate", learn_env, "terminate", None),
               ("true", T_masked, "given", lift)]
    if kind != "aggregated":
        targets += [("true_rows_mean", T_meanimp, "given", lift), ("true_rows_terminate", T_masked, "terminate", lift)]
    del T_meanimp
    if family == "gumbel":   # is the true optimal action supported? (decision states of the construction)
        opt_act, Vt = _x_v3_gumbel_opt_action(spec["variant"], spec["timeout_hazard"]), B["decision_states_true"]
        B["report"]["true_optimum_in_mask_fraction"] = (float(B["mask_true"][Vt, opt_act[Vt]].mean()) if len(Vt)
                                                        else float("nan"))
    if row_report is not None:
        B["report"].update({f"rows_{k}": v for k, v in row_report.items()})
    deploy = "true"
    construction = {k: v for k, v in B["report"].items() if k != "hospital_validation"}
    hosp_validation = B["report"]["hospital_validation"]
    source = f"x_sample_dataset({spec.get('construction', 'best')} benchmark, clinician estimate)"
    del B
    labels = [t[0] for t in targets]
    assert len(labels) == len(set(labels)) and deploy in labels, labels
    for lab, env, sem, lift_fn in targets:
        assert sem in X_SEMANTICS and (lift_fn is not None or env.S == learn_env.S), (lab, env.S, learn_env.S)
    user_validation = x_validate_dataset(data, learn_env)
    train, dev = x_split_episodes(data, XCFG.dev_fraction, np_rng("xenv", "split", cell, draw))
    meta = {"spec": {k: (list(v) if isinstance(v, tuple) else v) for k, v in spec.items()}, "user_data_source": source,
            "user_validation": user_validation, "hospital_validation": hosp_validation, "construction": construction,
            "train_transitions": int(len(train["state"])), "dev_transitions": int(len(dev["state"])),
            "build_s": time.perf_counter() - t0}
    return CellInstance(cell=cell, family=family, draw=draw, learn_env=learn_env, behavior=behavior, data=data,
                        train=train, dev=dev, targets=targets, deploy_target=deploy, meta=meta)


def x_v3_draw_identity(spec: dict, inst: CellInstance) -> dict:
    """x_draw_identity (II.6) with the II.8 code hash."""
    ident = x_draw_identity(spec, inst)
    ident["code"] = XENV_V3_CODE_HASH
    ident["section"] = "II.8"
    return ident


def run_v3_cell_draw(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
    """All results of one draw of one Section II.8 cell (the steps and checks of run_cell_draw; cached under v3/)."""
    cell, family = spec["cell"], spec["family"]
    assert family in ("gumbel", "garnet") and spec["v3_kind"] in XV3_KINDS, (family, spec.get("v3_kind"))
    t0 = time.perf_counter()
    inst = build_v3_instance(spec, draw)
    assert (inst.cell, inst.family, int(inst.draw)) == (cell, family, int(draw)), (inst.cell, inst.family, inst.draw)
    identity = x_v3_draw_identity(spec, inst)
    key, ident_hash = "v3/" + x_cache_key(cell, draw), obj_hash(identity)[:20]
    strict = True
    build_s = time.perf_counter() - t0
    hit, meta_hit = XCACHE.get(key, identity), XCACHE.get(key + "/meta", identity)
    if hit is not None and meta_hit is not None:
        del inst
        assert list(hit.columns) == XRESULT_COLUMNS, f"{key}: cached columns differ from XRESULT_COLUMNS"
        meta = json.loads(str(meta_hit["meta_json"].iloc[0]))
        meta.update(cache="loaded", session_build_s=build_s, checks=x_check_draw_rows(hit, strict))
        gc.collect()
        return hit, meta
    if (hit is None) != (meta_hit is None):
        print(f"  {key}: incomplete cache entry (results or meta missing) - recomputing this draw")
    XCACHE.require_allowed(key)
    learn = inst.learn_env
    t1 = time.perf_counter()
    raw = run_xmethods(LearnerView.from_env(learn), inst.train, f"{cell}|draw{int(draw)}")
    records = x_collect_records(raw, inst.behavior, learn)
    del raw
    t2 = time.perf_counter()
    fqe = x_fqe_env(inst.dev, inst.train, learn, name=f"fqe|{cell}|draw{int(draw)}")
    t3 = time.perf_counter()
    rows, ev_stats = x_evaluate_targets(inst, records, fqe)
    df = pd.DataFrame(rows, columns=XRESULT_COLUMNS)
    t4 = time.perf_counter()
    checks = x_check_draw_rows(df, strict)
    t5 = time.perf_counter()
    nonconv = [r["label"] for r in records if r["desc"]["converged"] is False]
    lm = learn.mask[learn.live]
    meta = _x_compact({
        "cell": cell, "family": family, "v3_kind": spec["v3_kind"], "draw": int(draw),
        "deploy_target": inst.deploy_target, "identity_hash": ident_hash, "code_hash": XENV_V3_CODE_HASH[:16],
        "cache": "computed",
        "learn_env": {"name": learn.name, "S": learn.S, "A": learn.A, "live": int(learn.live.sum()),
                      "admissible_pairs": int(lm.sum()), "mean_admissible_actions": float(lm.sum(axis=1).mean())},
        "data": {"train_transitions": int(len(inst.train["state"])),
                 "train_episodes": int(np.unique(inst.train["episode_id"]).size),
                 "dev_transitions": int(len(inst.dev["state"])),
                 "dev_episodes": int(np.unique(inst.dev["episode_id"]).size)},
        "targets": [t[0] for t in inst.targets],
        "records": {"total": len(records), **dict(Counter(r["variant"] for r in records)),
                    "nonconverged": len(nonconv), "nonconverged_labels": nonconv},
        "evaluation": ev_stats, "fqe_rows": fqe.meta.get("fqe_rows", {}), "checks": checks,
        "timing": {"build_s": build_s, "methods_s": t2 - t1, "fqe_build_s": t3 - t2, "eval_s": t4 - t3,
                   "check_s": t5 - t4, "total_s": t5 - t0},
        "instance": inst.meta})
    XCACHE.put(key + "/meta", identity, pd.DataFrame({"meta_json": [json.dumps(meta, default=_json_default)]}))
    XCACHE.put(key, identity, df)
    back = XCACHE.get(key, identity)
    assert back is not None and list(back.columns) == XRESULT_COLUMNS and len(back) == len(df), f"{key}: cache write"
    for c in _XMETRIC_SOURCE:
        assert np.allclose(back[c].to_numpy(dtype=float), df[c].to_numpy(dtype=float), rtol=1e-14, atol=0.0,
                           equal_nan=True), f"{key}: cache round trip changed column {c}"
    assert (back["status"].astype(str).to_numpy() == df["status"].to_numpy()).all(), f"{key}: cache round trip"
    del inst, fqe, records, rows, df
    gc.collect()
    return back, meta


def x_v3_run_cells(cells: Sequence[dict], desc: str = "II.8") -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """Every draw of every cell in `cells` (run_v3_cell_draw): stacked rows, flattened draw meta, cross-draw checks.

    Progress and ETA as in II.6a; nothing is written to disk."""
    frames: List[pd.DataFrame] = []
    meta_rows: List[dict] = []
    times: Dict[str, List[float]] = {}
    status: Counter = Counter()
    t_all = time.perf_counter()
    print(f"{desc} driver: {len(cells)} cells, {sum(int(s['draws']) for s in cells)} draws")
    for ci, spec in enumerate(cells, 1):
        t_cell, n_loaded, nd = time.perf_counter(), 0, int(spec["draws"])
        bar = trange(nd, desc=f"{desc} [{ci}/{len(cells)}] {spec['cell']}", leave=False, disable=not SHOW_PROGRESS)
        for draw in bar:
            t0 = time.perf_counter()
            df, meta = run_v3_cell_draw(spec, draw)
            dt = time.perf_counter() - t0
            frames.append(df)
            meta_rows.append(_x_flatten({**meta, "session_wall_s": dt}))
            status[meta["cache"]] += 1
            if meta["cache"] == "loaded":
                n_loaded += 1
            else:
                times.setdefault(spec["v3_kind"], []).append(dt)
            if SHOW_PROGRESS:
                bar.set_postfix_str(f"{dt:.1f}s {meta['cache']}")
            elif (draw + 1) % max(1, nd // 5) == 0 or draw + 1 == nd:
                print(f"    {spec['cell']} draw {draw + 1}/{nd} ({meta['cache']}, {dt:.1f} s); "
                      f"cell ETA {_x_fmt_s(x_eta_seconds([(spec['v3_kind'], nd - draw - 1)], times))}")
        eta = x_eta_seconds([(s["v3_kind"], int(s["draws"])) for s in cells[ci:]], times)
        print(f"  [{ci}/{len(cells)}] {spec['cell']}: {nd} draws ({n_loaded} cached) in "
              f"{_x_fmt_s(time.perf_counter() - t_cell)} | total elapsed {_x_fmt_s(time.perf_counter() - t_all)} | "
              f"ETA remaining cells {_x_fmt_s(eta) if ci < len(cells) else '0 s'}")
    res = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=XRESULT_COLUMNS)
    assert list(res.columns) == XRESULT_COLUMNS
    summary = {"code_hash": XENV_V3_CODE_HASH, "cells": len(cells), "draws": int(sum(status.values())),
               "draws_computed": int(status["computed"]), "draws_loaded": int(status["loaded"]),
               "elapsed_s": time.perf_counter() - t_all, **x_check_results(res)}
    return res, pd.DataFrame(meta_rows), summary


# Objects whose source enters the II.8 code hash, and the module constants they read (by value).
XENV_V3_CODE = [gumbel_diabetes_counterpart, gumbel_physician_v3, _x_v3_gumbel_env, _x_v3_gumbel_physician,
                _x_v3_gumbel_opt_action, x_gumbel_aggregation, x_aggregated_env, x_aggregate_dataset,
                construct_aggregated_benchmark, garnet_random_rows_env, _x_v3_garnet_env, x_v3_build_cells,
                build_v3_instance, x_v3_draw_identity, run_v3_cell_draw]
XENV_V3_CONSTANTS = {"physician_kinds": list(GUMBEL_PHYSICIAN_KINDS), "physician_kinds_v3": list(GUMBEL_PHYSICIAN_KINDS_V3),
                     "v3_kinds": list(XV3_KINDS)}
XENV_V3_CODE_HASH = obj_hash({"xenv_code": XENV_CODE_HASH, "v3_code": code_hash(*XENV_V3_CODE),
                              "constants": XENV_V3_CONSTANTS})
print("II.8 code hash:", XENV_V3_CODE_HASH[:16], "| Part II code hash:", XENV_CODE_HASH[:16])


def _x_v3_flat(prefix: str, d: Optional[dict]) -> dict:
    out = {}
    for k, v in (d or {}).items():
        if isinstance(v, dict):
            out.update(_x_v3_flat(f"{prefix}{k}.", v))
        elif isinstance(v, (list, tuple, np.ndarray)):
            out[f"{prefix}{k}"] = canonical_json(v)
        else:
            out[f"{prefix}{k}"] = v
    return out


def x_v3_environment_suite_table(cells: Sequence[dict]) -> pd.DataFrame:
    """One row per II.8 cell from draw 0 (columns of x_environment_suite_table + II.8 keys + flattened report)."""
    rows = []
    for spec in cells:
        inst = build_v3_instance(spec, 0)
        L = inst.learn_env
        row = {"cell": spec["cell"], "family": spec["family"], "v3_kind": spec["v3_kind"],
               "physician": spec.get("physician"), "aggregation": spec.get("aggregation"),
               "draws": spec["draws"], "deploy_target": inst.deploy_target,
               "S": L.S, "A": L.A, "live_states": int(L.live.sum()), "admissible_pairs": int(L.mask[L.live].sum()),
               "mean_admissible_actions": float(L.mask[L.live].sum(axis=1).mean()),
               "user_transitions": inst.meta["user_validation"]["transitions"],
               "user_episodes": inst.meta["user_validation"]["episodes"],
               "user_inadmissible_logged_fraction": inst.meta["user_validation"]["inadmissible_logged_fraction"],
               "build_s": inst.meta["build_s"], "targets": ";".join(t[0] for t in inst.targets)}
        statuses = []
        for lab, env, sem, lift_fn in inst.targets:
            beh = inst.behavior if lift_fn is None else lift_fn(inst.behavior)
            row[f"behaviour_value[{lab}]"], st_b = _x_value(env, beh, sem)
            row[f"optimum_value[{lab}]"], st_o = _x_opt_value(env, sem)
            statuses += [f"{what}[{lab}]={st}" for what, st in (("behaviour", st_b), ("optimum", st_o)) if st != "ok"]
        true_t = {lab: env for lab, env, _, _ in inst.targets}["true"]
        j_all, st_all = _x_opt_value(true_t, "given")
        j_adm, st_adm = _x_opt_value(true_t, "given", constrained=True)
        row["value_of_unsupported_actions"] = j_all - j_adm
        statuses += [f"optimum_admissible[true]={st_adm}"] * (st_adm != "ok") + [f"optimum[true]={st_all}"] * (st_all != "ok")
        row["non_ok_statuses"] = "; ".join(statuses)
        row.update(_x_v3_flat("construction_", inst.meta["construction"]))
        rows.append(row)
        del inst
        gc.collect()
    return pd.DataFrame(rows)


X_V3_CELLS: List[dict] = x_v3_build_cells()
print("II.8 cells:", [(c["cell"], c["draws"]) for c in X_V3_CELLS])
X_V3_ENV_SUITE = pd.DataFrame()
X_V3_ENV_CHECK: Dict[str, Any] = {"status": "not run"}
if X_V3_CELLS:
    _t0 = time.perf_counter()
    X_V3_ENV_SUITE = x_v3_environment_suite_table(X_V3_CELLS)
    X_V3_ENV_SUITE.to_csv(XENV_TABLE_DIR / "v3_environment_suite.csv", index=False)
    _agg = X_V3_ENV_SUITE[X_V3_ENV_SUITE["v3_kind"] == "aggregated"]
    _id_col = "construction_aggregation_check_behaviour_value_abs_diff"
    X_V3_ENV_CHECK = {
        "cells": [c["cell"] for c in X_V3_CELLS], "code_hash": XENV_V3_CODE_HASH,
        "non_ok_statuses": {r["cell"]: r["non_ok_statuses"] for _, r in X_V3_ENV_SUITE.iterrows() if r["non_ok_statuses"]},
        "trap_states_terminate_max": int(X_V3_ENV_SUITE["construction_trap_states_terminate"].max()),
        "behaviour_aggregation_identity_max_abs_diff": (float(_agg[_id_col].max()) if len(_agg) else None),
        "seconds": time.perf_counter() - _t0}
    X_V3_ENV_CHECK["passes"] = bool(not X_V3_ENV_CHECK["non_ok_statuses"]
                                    and X_V3_ENV_CHECK["trap_states_terminate_max"] == 0
                                    and (not len(_agg) or X_V3_ENV_CHECK["behaviour_aggregation_identity_max_abs_diff"] <= 1e-9))
    save_json(X_V3_ENV_CHECK, XENV_TABLE_DIR / "v3_environment_checks.json")
    _cols = [c for c in ("cell", "S", "live_states", "mean_admissible_actions", "behaviour_value[true]",
                         "optimum_value[true]", "value_of_unsupported_actions",
                         "construction_single_admissible_fraction", "construction_true_optimum_in_mask_fraction",
                         "construction_aggregated_live_cells", "construction_full_d0_mass_in_decision_cells",
                         "construction_rows_nonpreferred_beneficial_share", "non_ok_statuses")
             if c in X_V3_ENV_SUITE.columns]
    display(X_V3_ENV_SUITE[_cols])
    print(f"II.8 environment suite built in {time.perf_counter() - _t0:.1f} s | checks pass: {X_V3_ENV_CHECK['passes']}")
    assert X_V3_ENV_CHECK["passes"], "II.8 environment checks failed (xenv/v3_environment_checks.json)"
    del _agg, _id_col, _cols
print("II.8 environments and cells ready.")

### II.8f Run the II.8 cells (and the prerequisites of II.8g–II.8h)

The cell below runs every draw of every enabled II.8 cell (`X_V3_CELLS`) through `run_v3_cell_draw`, using
`x_v3_run_cells` (II.8e). Each draw is loaded from `XCACHE` (key `v3/<cell>/draw<k>`) when its identity matches.
Otherwise, if `ALLOW_TRAINING`, it is computed and cached. The table returned is always the one read back from the
cache, so an interrupted, resumed run and an uninterrupted run give identical tables.

| Output | Content |
|---|---|
| `XRESULTS_V3` · `xenv/v3_results_all.csv.gz` | one row per draw × policy × target, with exactly the columns of `XRESULTS` (`XRESULT_COLUMNS`) |
| `XDRAW_META_V3` · `xenv/v3_draw_meta.csv` | construction and validation reports, record counts, reference solvers, invariant-check slacks and timings per draw |
| `XV3_DRIVER_SUMMARY` · `xenv/v3_driver_summary.json` | code hash, cache use, the cross-draw checks of `x_check_results`, draws per cell and whether every configured draw is present |

`XRESULTS_V3` is kept separate from `XRESULTS`. The II.7 statistics, the preregistered tests H1–H3 and the Part II
gate use only the II.6 cells, which keeps the preregistered analysis unchanged. The II.8 rows enter the additional
analyses (II.8i) and the II.8 gate of Section 34.

**Prerequisites of II.8g–II.8h.** The bootstrap intervals (II.8g) and the ICU-Sepsis harm curves (II.8h) call
`run_xmethods`, and II.8h evaluates on the ICU-Sepsis adapter `X_ICU`. When those switches are on but every switch
of Sections II.3–II.7 and of II.8a–II.8e is off, the cell below runs the ICU-Sepsis adapter check and the method
self-test itself, with the same code and output files as II.3a and II.4. The Section 34 gate requires the adapter
check whenever an `xenv_*` switch is on.

**Expected cost on Colab (CPU-bound).** These are estimates, not measurements of Colab. Local per-draw timings of
the II.8 cells (harness, M1 CPU) were scaled by the Colab / local ratio measured on matched II.6 draws: about 1.8
for Gumbel and 3.5 for Garnet, from the timings in `xenv_draw_meta.csv`. That gives about 20–25 s per physician
draw, 10–12 s per aggregated draw and 4–5 s per random-row Garnet draw. With 30 draws per cell, the six cells take
about 35–40 min, plus under a minute for the environment suite of II.8e. A cached draw reloads in about 3 s
(instance rebuild and identity), so a re-run costs about 9 min.

In [ ]:
# ============================================================
# II.8f II.8 driver: all II.8 cells and draws
# ============================================================
EXECLOG.mark("II.8 Revision experiments: driver")
XV3_EXTRA_SWITCHES = ("xenv_v3_bootstrap", "xenv_v3_icu_harm")
XV3_ALL_SWITCHES = (*XV3_SWITCHES, *XV3_EXTRA_SWITCHES)
assert all(k in EXPERIMENTS for k in XV3_ALL_SWITCHES), [k for k in XV3_ALL_SWITCHES if k not in EXPERIMENTS]
if any(bool(EXPERIMENTS[k]) for k in XV3_EXTRA_SWITCHES) and X_ICU_ADAPTER_OK is None:
    # Same code and output file as II.3a; runs only when every switch of II.3-II.7 and of II.8a-II.8e is off.
    X_ICU = x_icu_env()
    X_ICU_BEHAVIOR = x_icu_behavior()
    assert np.array_equal(X_ICU_BEHAVIOR, CLINICIAN_POLICY), "ICU behaviour must equal Section 14's CLINICIAN_POLICY"
    X_ICU_ADAPTER_CHECK = x_icu_adapter_check(X_ICU)
    X_ICU_ADAPTER_OK = bool(X_ICU_ADAPTER_CHECK["passes"])
    save_json(X_ICU_ADAPTER_CHECK, XENV_TABLE_DIR / "icu_adapter_check.json")
    assert X_ICU_ADAPTER_OK, "ICU-Sepsis FiniteEnv adapter differs from EVALUATOR (see xenv/icu_adapter_check.json)."
if any(bool(EXPERIMENTS[k]) for k in XV3_EXTRA_SWITCHES) and X_METHODS_SELFTEST is None:
    # Same self-test and output file as II.4; runs only when every switch of II.3-II.7 and of II.8a-II.8e is off.
    X_METHODS_SELFTEST = x_methods_selftest()
    save_json(X_METHODS_SELFTEST, XENV_TABLE_DIR / "xenv_methods_selftest.json")
    assert X_METHODS_SELFTEST["passed"], "Part II method-suite self-test failed; see xenv/xenv_methods_selftest.json"


def x_v3_draw_counts(results: pd.DataFrame, cells: Sequence[dict]) -> Dict[str, Any]:
    """Draws present per enabled cell against the configured number (spec['draws'])."""
    have = (results.groupby("cell")["draw"].nunique().to_dict() if len(results) else {})
    per = {str(s["cell"]): {"draws": int(have.get(s["cell"], 0)), "configured_draws": int(s["draws"])} for s in cells}
    return {"draws_per_cell": per,
            "draws_complete": bool(cells) and all(v["draws"] == v["configured_draws"] for v in per.values())}


XRESULTS_V3 = pd.DataFrame(columns=XRESULT_COLUMNS)
XDRAW_META_V3 = pd.DataFrame()
XV3_DRIVER_SUMMARY: Dict[str, Any] = {"status": "not run (xenv_v3_physicians, xenv_v3_aggregated and "
                                                "xenv_v3_garnet_random are off)", "code_hash": XENV_V3_CODE_HASH,
                                      "draws_complete": None}
if X_V3_CELLS:
    XRESULTS_V3, XDRAW_META_V3, _xv3s = x_v3_run_cells(X_V3_CELLS)
    assert list(XRESULTS_V3.columns) == XRESULT_COLUMNS
    XV3_DRIVER_SUMMARY = {"status": "run", **_xv3s, **x_v3_draw_counts(XRESULTS_V3, X_V3_CELLS)}
    for _c in ("checks.max_slack_vs_optimum", "checks.max_slack_vs_admissible_optimum", "checks.projection_identity_worst"):
        XV3_DRIVER_SUMMARY[f"max_{_c.split('.', 1)[1]}"] = (float(XDRAW_META_V3[_c].max())
                                                           if _c in XDRAW_META_V3 else None)
    XV3_DRIVER_SUMMARY["nonconverged_records"] = (int(XDRAW_META_V3["records.nonconverged"].sum())
                                                  if "records.nonconverged" in XDRAW_META_V3 else 0)
    XRESULTS_V3.to_csv(XENV_TABLE_DIR / "v3_results_all.csv.gz", index=False, compression="gzip")
    XDRAW_META_V3.to_csv(XENV_TABLE_DIR / "v3_draw_meta.csv", index=False)
    _dep = XRESULTS_V3[XRESULTS_V3["is_deploy_target"].astype(bool)
                       & XRESULTS_V3["policy"].isin([*XREF_POLICIES, XCOMPARATOR])]
    display(_dep.groupby(["cell", "policy"])["return"].agg(["mean", "std", "count"]).round(4))
    del _xv3s, _dep
    gc.collect()
save_json(XV3_DRIVER_SUMMARY, XENV_TABLE_DIR / "v3_driver_summary.json")
print(json.dumps({k: v for k, v in XV3_DRIVER_SUMMARY.items() if k not in ("improper_by_family_target", "draws_per_cell")},
                 indent=1, default=_json_default))
print("II.8 driver complete:", len(XRESULTS_V3), "result rows,", len(XDRAW_META_V3), "draws.")

### II.8g Estimation-aware intervals: bootstrap ensembles of the hospital data

The bracketing theorem holds for the **true** supported rows: J_term ≤ J* for every truth, and J* ≤ J_mean under no
hidden benefit. A benchmark has only **estimated** rows, so [J_bench_term, J_bench_mean] also carries estimation
error. This section measures how much of the coverage gap that error explains, using a nonparametric bootstrap of
the hospital data behind each benchmark.

**Cells** (`XV3BootConfig.cells`): Gumbel `best` with n = 2 000, τ = 5 and with n = 10 000, τ = 20; the
withhold-vaso physician cell; and Garnet h ∈ {−0.5, 0, 0.5, 1} at ε = 0.05. Every draw of each cell is used. A
configured cell that is not in `X_CELLS`, because its family switch is off, is skipped and listed in the summary.

**Per (cell, draw).**
1. Rebuild the II.6 instance (`build_cell_instance`). Regenerate the **default** policies by running `run_xmethods`
   on the same training split under the same key as II.6. Part II learners are deterministic functions of their
   data, so these are the II.6 policies. The behaviour policy and the projections are added by `x_collect_records`.
2. Regenerate the hospital episodes with the same stream and call as II.3d (`x_boot_hospital`). Rebuild the
   benchmark with `x_benchmark_from_data`; its identity must equal the instance's benchmark (asserted).
3. `x_boot_step_map` records where each logged hospital step enters the count tensor of that benchmark. It
   replicates the censoring of `x_benchmark_from_data`: informative (non-timeout) steps, dead-state removal (drop),
   trap censoring and dangling redirects. It also records the benchmark index of every initial state.
   `x_boot_model` applied to unit weights must reproduce the benchmark's P and d0 (asserted to 1e-14).
4. For b = 0, …, B − 1, resample the hospital **episodes** with replacement (stream
   `("xenv", "bootstrap", cell, draw, b)`). `x_boot_model` then re-estimates the rows of the supported pairs from
   the resampled steps, with the original decision states, mask, censoring and redirects; for Gumbel it uses the
   same known-horizon form h·e_timeout + (1 − h)·C/N. Unsupported rows follow the mean rule. A supported pair with
   no resampled step keeps its original row (counted as `fallback_pairs`). With `resample_d0` (default) the
   empirical d0 is re-estimated from the resampled episodes as well, since it is part of the estimation error.
5. Evaluate every default policy exactly under both rules on each replicate, and on the original benchmark and the
   true system (`true`, lifted). A replicate model can contain a trap (rows estimated from few steps may miss every
   absorbing outcome); a policy that does not absorb there is reported as `improper`, or `near-singular` when the
   evaluator's residual checks fail, and that replicate is left out of its interval.

**Invariants asserted per draw** (re-run on cached tables): the original benchmark and the truth are proper for
every policy; on every replicate the two rules share P(any unsupported decision) (1e-9); and
0 ≤ J_mean^b − J_term^b ≤ p^b·(v_max − v_min) (Theorem part iii, tolerance 1e-9).

The interval of a policy is [q_2.5%(J_term^b), q_97.5%(J_mean^b)] over its proper replicates (II.8i). With B = 10
these quantiles are close to the replicate minimum and maximum (linear interpolation). II.8i compares the coverage
and the certified comparisons of these intervals with those of the point interval [J_bench_term, J_bench_mean].

Results are cached in `XCACHE` under `v3boot/<cell>/draw<k>` (and `/meta`). The identity is the II.6 draw identity
(`x_draw_identity`, including `XENV_CODE_HASH`) plus the code hash of this section, B and `resample_d0`. The
quantile levels configure the analysis only. As a check that the regenerated policies are the II.6 policies,
`x_boot_point_check` compares the replicate −1 values with the `XRESULTS` rows of the same draws. It reports and
does not assert: on a different machine, BLAS rounding can break a near-tie differently.

| Output | Content |
|---|---|
| `xenv/v3_bootstrap_config.json` | `XV3BootConfig` |
| `xenv/v3_bootstrap_results.csv.gz` | one row per draw × default policy × replicate × target (replicate −1: original benchmark and truth) |
| `xenv/v3_bootstrap_meta.csv` | per draw: fallback pairs, re-estimated d0 support, trap states and improper counts per replicate, reproduction check, timings |
| `xenv/v3_bootstrap_summary.json` | cells run and skipped, cache use, invariant checks, the point check against `XRESULTS` |

**Expected cost on Colab.** These are estimates. Local harness timings were scaled by the Colab / local ratios of
II.8f. Each draw is an instance rebuild, one `run_xmethods` fit, and 2B + 3 exact evaluations of the 50 default
policies (the benchmark models are small). That is about 6–12 s per Gumbel draw and 4–5 s per Garnet draw: roughly
20–25 min for 7 cells × 30 draws. A cached draw reloads in about 3 s, so a re-run costs about 10 min.

In [ ]:
# ============================================================
# II.8g Bootstrap-ensemble intervals (estimation-aware brackets)
# ============================================================
EXECLOG.mark("II.8 Revision experiments: bootstrap intervals")


@dataclass
class XV3BootConfig:
    cells: Tuple[str, ...] = ("gumbel|best|n2000|tau5|eps0.05", "gumbel|best|n10000|tau20|eps0.05",
                              "gumbel|best|n10000|tau20|eps0.05|phys-withhold-vaso", "garnet|h-0.5|eps0.05",
                              "garnet|h0.0|eps0.05", "garnet|h0.5|eps0.05", "garnet|h1.0|eps0.05")
    replicates: int = 10                                  # B bootstrap resamples of the hospital episodes
    quantiles: Tuple[float, float] = (0.025, 0.975)       # interval = [q_lo(J_term^b), q_hi(J_mean^b)] (analysis only)
    resample_d0: bool = True                              # re-estimate the empirical d0 from the resampled episodes


def apply_xv3_boot_run_mode(c: XV3BootConfig, mode: str) -> XV3BootConfig:
    if mode == "smoke":
        c.cells = ("gumbel|best|n300|tau5|eps0.05", "garnet|h0.0|eps0.05", "garnet|h1.0|eps0.05")
        c.replicates = 3
    return c


XV3BOOT = apply_xv3_boot_run_mode(XV3BootConfig(), RUN_MODE)
save_json(asdict(XV3BOOT), XENV_TABLE_DIR / "v3_bootstrap_config.json")
XV3_BOOT_POINT = -1                     # replicate label of the original benchmark and of the true system
XV3_EVAL_STATUSES = ("ok", "improper", "near-singular")
XV3_BOOT_COLUMNS = ["cell", "family", "draw", "policy", "method", "variant", "hp", "is_default", "method_family",
                    "replicate", "target", "semantics", "status", "return", "prob_any_violation",
                    "expected_violations", "avg_episode_length"]
XV3_TOL = 1e-9                          # value tolerance of the bracketing / width checks (as in II.8i)


def _x_v3_eval(env: FiniteEnv, pi: np.ndarray, semantics: str) -> Dict[str, Any]:
    """Exact evaluation that never raises on a model-specific failure: ok / improper / near-singular."""
    try:
        out = env.evaluator().evaluate(pi, _X_STRATEGY[semantics])
    except ImproperPolicyError as exc:
        return {"status": "improper", "error": str(exc)[:200]}
    except EvaluationInvariantError as exc:
        return {"status": "near-singular", "error": str(exc)[:200]}
    return {"status": "ok", "return": float(out["return"]), "prob_any_violation": float(out["prob_any_violation"]),
            "expected_violations": float(out["expected_violations"]),
            "avg_episode_length": float(out["avg_episode_length"])}


def x_boot_hospital(spec: dict, draw: int) -> Tuple[FiniteEnv, Dict[str, np.ndarray]]:
    """True system and hospital episodes of a draw of a II.3d `best` cell (same stream and call as construct_benchmark)."""
    cell, family = spec["cell"], spec["family"]
    rng_h = np_rng("xenv", "hospital", cell, int(draw))
    if family == "gumbel":
        assert spec["construction"] == "best", f"{cell}: bootstrap needs the estimated ('best') construction"
        true_env = _x_gumbel_env(spec["variant"], spec["timeout_hazard"])
        beh = _x_gumbel_physician(spec["variant"], spec["timeout_hazard"], spec["eps"], spec.get("physician", "optimal"))
    else:
        assert family == "garnet", family
        true_env, beh = _x_garnet_env(spec, draw)
    hosp = x_sample_dataset(true_env, beh, int(spec["n_hosp"]), XCFG.max_steps, rng_h)
    return true_env, hosp


def x_boot_step_map(true_env: FiniteEnv, hosp: Dict[str, np.ndarray], tau: int, B: Dict[str, Any]) -> Dict[str, Any]:
    """Where each logged hospital step enters the count tensor of benchmark B (x_benchmark_from_data on these steps).

    Returns the flat index (bench state, action, bench next state) of every step, or -1 if the estimate does not use it,
    and the benchmark index of every episode's initial state (-1 if censored). Replicates the censoring steps of
    x_benchmark_from_data: informative steps, dead-state removal (rule 'icu'), trap censoring and redirects."""
    A = true_env.A
    rep = B["report"]
    rule = str(rep["empty_rule"])
    assert rule in ("icu", "argmax"), rule
    s, a, ns = hosp["state"], hosp["action"], hosp["next_state"]
    hazard = true_env.meta["censoring_hazard"]
    informative = (np.ones(len(s), dtype=bool) if hazard is None else ns != int(true_env.meta["timeout_state"]))
    N_inf = _x_hospital_counts(true_env, {"state": s[informative], "action": a[informative]})
    mask_count, _ = x_count_mask(N_inf, tau, true_env.live, fallback=rule == "argmax")
    redirect = _x_dangling_target(true_env)
    logged = np.unique(s)
    V = logged[N_inf[logged].sum(axis=1) > 0]
    dead = np.zeros(0, dtype=np.int64)
    rounds = 0
    if rule == "icu":
        est = _x_estimate_on(true_env, hosp, V, mask_count, redirect, informative, drop_into=dead)
        while est["dead"].any():
            dead = np.sort(np.concatenate([dead, V[est["dead"]]]))
            V = V[~est["dead"]]
            rounds += 1
            est = _x_estimate_on(true_env, hosp, V, mask_count, redirect, informative, drop_into=dead)
        del est
    assert (len(dead), rounds) == (rep["dead_states_removed"], rep["dead_state_rounds"]), \
        f"dead-state removal differs from the benchmark report: {(len(dead), rounds)}"
    V_final = np.asarray(B["decision_states_true"], dtype=np.int64)
    trap = np.setdiff1d(V, V_final)
    assert np.isin(V_final, V).all() and len(trap) == rep["trap_states_censored"], "trap censoring differs"
    index = np.asarray(B["index"], dtype=np.int64)
    nV, Sb = len(V_final), len(index)
    assert np.array_equal(index[:nV], V_final), "benchmark index must start with the decision states"
    to_bench = np.full(true_env.S, -1, dtype=np.int64)
    to_bench[index] = np.arange(Sb)
    keep = to_bench[s] >= 0
    use = keep & informative
    if rule == "icu":
        use &= ~np.isin(ns, dead)
    ns_b = to_bench[ns]
    ns_b[ns_b < 0] = to_bench[redirect]
    flat = np.full(len(s), -1, dtype=np.int64)
    flat[use] = (to_bench[s[use]] * A + a[use]) * Sb + ns_b[use]
    first = hosp["timestep"] == 0
    init = np.full(len(s), -1, dtype=np.int64)
    init[first & keep] = to_bench[s[first & keep]]
    return {"flat": flat, "init": init, "episode": np.asarray(hosp["episode_id"], dtype=np.int64), "Sb": int(Sb),
            "nV": int(nV), "hazard": hazard,
            "timeout_bench": None if hazard is None else int(to_bench[int(true_env.meta["timeout_state"])]),
            "dead_states": int(len(dead)), "trap_states": int(len(trap)), "steps": int(len(s)),
            "steps_used": int(use.sum())}


def x_boot_weights(n_episodes: int, rng: np.random.Generator) -> np.ndarray:
    """Multiplicity of each episode in a resample of n_episodes episodes with replacement."""
    return np.bincount(rng.integers(0, int(n_episodes), size=int(n_episodes)), minlength=int(n_episodes)).astype(np.float64)


def x_boot_model(bench: FiniteEnv, smap: Dict[str, Any], w_steps: np.ndarray,
                 resample_d0: bool) -> Tuple[np.ndarray, np.ndarray, Dict[str, int]]:
    """(P, d0, stats) of `bench` with the supported rows re-estimated from weighted hospital steps.

    Mask, decision states, censoring and redirects are those of `bench`; with a known censoring hazard h the rows are
    h * e_timeout + (1 - h) * C / N (as in _x_estimate_on); unsupported rows follow the mean rule; a supported pair
    without any weighted step keeps its row of `bench`. Unit weights reproduce bench.P and bench.d0."""
    Sb, A = bench.S, bench.A
    assert smap["Sb"] == Sb and len(w_steps) == len(smap["flat"])
    w = np.asarray(w_steps, dtype=np.float64)
    used = smap["flat"] >= 0
    C = np.bincount(smap["flat"][used], weights=w[used], minlength=Sb * A * Sb).reshape(Sb, A, Sb)
    N = C.sum(axis=2)
    adm = bench.mask & bench.live[:, None]
    have = adm & (N > 0)
    P = np.zeros((Sb, A, Sb))
    P[have] = C[have] / N[have][:, None]
    del C
    if smap["hazard"] is not None:
        c = int(smap["timeout_bench"])
        P[have] *= 1.0 - float(smap["hazard"])
        P[have, c] += float(smap["hazard"])
    miss = adm & ~have
    P[miss] = bench.P[miss]
    P = x_impute_inadmissible(P, bench.mask, bench.live)
    for t in bench.terminals:
        P[t, :, t] = 1.0
    if resample_d0:
        ini = smap["init"] >= 0
        d0 = np.bincount(smap["init"][ini], weights=w[ini], minlength=Sb).astype(np.float64)
        assert d0.sum() > 0 and np.all(d0[~bench.live] == 0)
        d0 /= d0.sum()
    else:
        d0 = bench.d0.copy()
    stats_ = {"admissible_pairs": int(adm.sum()), "fallback_pairs": int(miss.sum()),
              "d0_support_states": int((d0 > 0).sum()), "steps_weight": float(w[used].sum())}
    return P, d0, stats_


def x_boot_identity(spec: dict, inst: CellInstance) -> dict:
    return {"kind": "xenv_v3_bootstrap", "draw": x_draw_identity(spec, inst), "code": XV3_BOOT_CODE_HASH,
            "replicates": int(XV3BOOT.replicates), "resample_d0": bool(XV3BOOT.resample_d0),
            "rng": ["xenv", "bootstrap", "<cell>", "<draw>", "<b>"], "columns": XV3_BOOT_COLUMNS,
            "policies": "default (is_default) records of x_collect_records"}


def x_boot_check_rows(df: pd.DataFrame, v_range: float) -> Dict[str, Any]:
    """Invariants of one draw's bootstrap rows (see markdown); asserted, and a small report is returned."""
    bad = sorted(set(df["status"].astype(str)) - set(XV3_EVAL_STATUSES))
    assert not bad, f"unexpected bootstrap status values {bad}"
    pt = df[df["replicate"] == XV3_BOOT_POINT]
    assert (pt["status"] == "ok").all(), "a default policy is not proper on the original benchmark or the truth"
    dup = df.duplicated(["policy", "replicate", "target"])
    assert not dup.any(), f"{int(dup.sum())} duplicated (policy, replicate, target) bootstrap rows"
    sub = df[df["target"].isin(["bench_mean", "bench_terminate"])].set_index(["policy", "replicate", "target"])
    J, Pv = sub["return"].unstack("target"), sub["prob_any_violation"].unstack("target")
    St = sub["status"].unstack("target")
    assert set(St.columns) == {"bench_mean", "bench_terminate"} and St.notna().all().all(), "missing bootstrap rows"
    both = ((St["bench_mean"] == "ok") & (St["bench_terminate"] == "ok")).to_numpy()
    jm, jt = J["bench_mean"].to_numpy(float)[both], J["bench_terminate"].to_numpy(float)[both]
    pm, ptm = Pv["bench_mean"].to_numpy(float)[both], Pv["bench_terminate"].to_numpy(float)[both]
    width = jm - jt
    dp = float(np.abs(pm - ptm).max(initial=0.0))
    slack_lo = float((-width).max(initial=-np.inf))
    slack_hi = float((width - ptm * float(v_range)).max(initial=-np.inf))
    assert dp <= XV3_TOL, f"the two rules disagree on P(any unsupported decision) by {dp:.3g}"
    assert slack_lo <= XV3_TOL, f"J_mean < J_term on a bootstrap model by {slack_lo:.3g} (Theorem part iii)"
    assert slack_hi <= XV3_TOL, f"J_mean - J_term exceeds p (v_max - v_min) by {slack_hi:.3g} (Theorem part iii)"
    nonok = df[(df["status"] != "ok")].groupby("target").size()
    return {"rows": int(len(df)), "pairs_checked": int(both.sum()), "nonok_rows": int((df["status"] != "ok").sum()),
            "nonok_by_target": {str(k): int(v) for k, v in nonok.items()}, "max_rule_p_difference": dp,
            "max_negative_width": slack_lo, "max_width_minus_bound": slack_hi}


def run_v3_boot_draw(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
    """Bootstrap-ensemble evaluation of the default policies of one draw of a II.3d cell (cached under v3boot/)."""
    cell, family, draw = spec["cell"], spec["family"], int(draw)
    assert family in ("gumbel", "garnet"), family
    t0 = time.perf_counter()
    inst = build_cell_instance(spec, draw)
    identity = x_boot_identity(spec, inst)
    key, ident_hash = "v3boot/" + x_cache_key(cell, draw), obj_hash(identity)[:20]
    build_s = time.perf_counter() - t0
    hit, meta_hit = XCACHE.get(key, identity), XCACHE.get(key + "/meta", identity)
    if hit is not None and meta_hit is not None:
        del inst
        assert list(hit.columns) == XV3_BOOT_COLUMNS, f"{key}: cached columns differ from XV3_BOOT_COLUMNS"
        meta = json.loads(str(meta_hit["meta_json"].iloc[0]))
        meta.update(cache="loaded", session_build_s=build_s, checks=x_boot_check_rows(hit, meta["v_range"]))
        gc.collect()
        return hit, meta
    if (hit is None) != (meta_hit is None):
        print(f"  {key}: incomplete cache entry (results or meta missing) - recomputing this draw")
    XCACHE.require_allowed(key)
    learn = inst.learn_env
    t1 = time.perf_counter()
    raw = run_xmethods(LearnerView.from_env(learn), inst.train, f"{cell}|draw{draw}")
    records = [r for r in x_collect_records(raw, inst.behavior, learn) if r["is_default"]]
    del raw
    keys = [_x_pi_key(r["pi"]) for r in records]
    t2 = time.perf_counter()
    true_env, hosp = x_boot_hospital(spec, draw)
    B0 = x_benchmark_from_data(true_env, hosp, spec["tau"], learn.name)
    assert obj_hash(B0["bench"].identity()) == obj_hash(learn.identity()), \
        f"{cell} draw {draw}: regenerated hospital data do not rebuild the II.6 benchmark"
    smap = x_boot_step_map(true_env, hosp, spec["tau"], B0)
    eps_id = smap["episode"]
    n_ep = int(eps_id.max()) + 1
    assert np.array_equal(np.unique(eps_id), np.arange(n_ep)), "hospital episode ids must be 0..n-1"
    P0, d00, _ = x_boot_model(learn, smap, np.ones(len(eps_id)), XV3BOOT.resample_d0)
    repro = {"max_abs_diff_P": float(np.abs(P0 - learn.P).max()), "max_abs_diff_d0": float(np.abs(d00 - learn.d0).max())}
    assert max(repro.values()) <= 1e-14, f"{cell} draw {draw}: step map does not reproduce the benchmark {repro}"
    del P0, d00, B0, hosp
    lo, hi = learn.value_bounds
    base = {"cell": cell, "family": family, "draw": draw}

    def rows_for(env: FiniteEnv, target: str, semantics: str, replicate: int,
                 lift: Optional[Callable[[np.ndarray], np.ndarray]] = None) -> List[dict]:
        memo: Dict[bytes, dict] = {}
        out = []
        for r, k in zip(records, keys):
            if k not in memo:
                memo[k] = _x_v3_eval(env, r["pi"] if lift is None else lift(r["pi"]), semantics)
            ev = memo[k]
            d = r["desc"]
            out.append({**base, "policy": d["policy"], "method": d["method"], "variant": d["variant"], "hp": d["hp"],
                        "is_default": True, "method_family": d["method_family"], "replicate": int(replicate),
                        "target": target, "semantics": semantics, "status": ev["status"],
                        **{c: ev.get(c, np.nan) for c in ("return", "prob_any_violation", "expected_violations",
                                                         "avg_episode_length")}})
        return out

    rows = rows_for(learn, "bench_mean", "given", XV3_BOOT_POINT) + rows_for(learn, "bench_terminate", "terminate",
                                                                             XV3_BOOT_POINT)
    tgt = {lab: (env, sem, lift) for lab, env, sem, lift in inst.targets}
    rows += rows_for(tgt["true"][0], "true", tgt["true"][1], XV3_BOOT_POINT, tgt["true"][2])
    t3 = time.perf_counter()
    rep_stats = []
    for b in range(int(XV3BOOT.replicates)):
        mult = x_boot_weights(n_ep, np_rng("xenv", "bootstrap", cell, draw, b))
        P_b, d0_b, st_b = x_boot_model(learn, smap, mult[eps_id], XV3BOOT.resample_d0)
        env_b = learn.variant(f"{learn.name}|boot{b}", P=P_b, d0=d0_b, kind="benchmark", bootstrap_replicate=b)
        del P_b
        st_b["trap_states_given"] = x_trap_states(env_b.P, env_b.live, np.ones((env_b.S, env_b.A), dtype=bool))
        st_b["trap_states_terminate"] = x_trap_states(env_b.P, env_b.live, env_b.mask)
        rb = rows_for(env_b, "bench_mean", "given", b) + rows_for(env_b, "bench_terminate", "terminate", b)
        st_b["nonok_given"] = int(sum(r["status"] != "ok" for r in rb if r["semantics"] == "given"))
        st_b["nonok_terminate"] = int(sum(r["status"] != "ok" for r in rb if r["semantics"] == "terminate"))
        rows += rb
        rep_stats.append({"replicate": b, **st_b})
        del env_b, rb
    t4 = time.perf_counter()
    df = pd.DataFrame(rows, columns=XV3_BOOT_COLUMNS)
    checks = x_boot_check_rows(df, hi - lo)
    meta = _x_compact({
        "cell": cell, "family": family, "draw": draw, "identity_hash": ident_hash, "code_hash": XV3_BOOT_CODE_HASH[:16],
        "cache": "computed", "replicates": int(XV3BOOT.replicates), "resample_d0": bool(XV3BOOT.resample_d0),
        "v_range": float(hi - lo), "n_policies": len(records), "unique_policies": len(set(keys)),
        "hospital_episodes": n_ep, "step_map": {k: v for k, v in smap.items() if not isinstance(v, np.ndarray)},
        "reproduction": repro,
        **{f"replicate_{k}": [r[k] for r in rep_stats] for k in ("fallback_pairs", "d0_support_states",
                                                                 "trap_states_given", "trap_states_terminate",
                                                                 "nonok_given", "nonok_terminate")},
        "admissible_pairs": int(rep_stats[0]["admissible_pairs"]) if rep_stats else None,
        "fallback_pairs_total": int(sum(r["fallback_pairs"] for r in rep_stats)),
        "trap_states_given_max": int(max([r["trap_states_given"] for r in rep_stats], default=0)),
        "checks": checks,
        "timing": {"build_s": build_s, "methods_s": t2 - t1, "point_eval_s": t3 - t2, "bootstrap_s": t4 - t3,
                   "total_s": time.perf_counter() - t0}})
    XCACHE.put(key + "/meta", identity, pd.DataFrame({"meta_json": [json.dumps(meta, default=_json_default)]}))
    XCACHE.put(key, identity, df)
    back = XCACHE.get(key, identity)
    assert back is not None and list(back.columns) == XV3_BOOT_COLUMNS and len(back) == len(df), f"{key}: cache write"
    for c in ("return", "prob_any_violation", "expected_violations", "avg_episode_length"):
        assert np.allclose(back[c].to_numpy(dtype=float), df[c].to_numpy(dtype=float), rtol=1e-14, atol=0.0,
                           equal_nan=True), f"{key}: cache round trip changed column {c}"
    del inst, rows, df, smap
    gc.collect()
    return back, meta


def x_v3_boot_run(cells: Sequence[dict]) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """Every draw of every bootstrap cell (run_v3_boot_draw): stacked rows, flattened draw meta, summary."""
    frames: List[pd.DataFrame] = []
    meta_rows: List[dict] = []
    times: Dict[str, List[float]] = {}
    status: Counter = Counter()
    t_all = time.perf_counter()
    print(f"II.8g bootstrap: {len(cells)} cells, {sum(int(s['draws']) for s in cells)} draws, "
          f"B = {XV3BOOT.replicates}")
    for ci, spec in enumerate(cells, 1):
        t_cell, n_loaded, nd = time.perf_counter(), 0, int(spec["draws"])
        bar = trange(nd, desc=f"II.8g [{ci}/{len(cells)}] {spec['cell']}", leave=False, disable=not SHOW_PROGRESS)
        for draw in bar:
            t0 = time.perf_counter()
            df, meta = run_v3_boot_draw(spec, draw)
            dt = time.perf_counter() - t0
            frames.append(df)
            meta_rows.append(_x_flatten({**meta, "session_wall_s": dt}))
            status[meta["cache"]] += 1
            if meta["cache"] == "loaded":
                n_loaded += 1
            else:
                times.setdefault(spec["family"], []).append(dt)
            if SHOW_PROGRESS:
                bar.set_postfix_str(f"{dt:.1f}s {meta['cache']}")
            elif (draw + 1) % max(1, nd // 5) == 0 or draw + 1 == nd:
                print(f"    {spec['cell']} draw {draw + 1}/{nd} ({meta['cache']}, {dt:.1f} s); "
                      f"cell ETA {_x_fmt_s(x_eta_seconds([(spec['family'], nd - draw - 1)], times))}")
        eta = x_eta_seconds([(s["family"], int(s["draws"])) for s in cells[ci:]], times)
        print(f"  [{ci}/{len(cells)}] {spec['cell']}: {nd} draws ({n_loaded} cached) in "
              f"{_x_fmt_s(time.perf_counter() - t_cell)} | total elapsed {_x_fmt_s(time.perf_counter() - t_all)} | "
              f"ETA remaining cells {_x_fmt_s(eta) if ci < len(cells) else '0 s'}")
    res = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=XV3_BOOT_COLUMNS)
    meta_df = pd.DataFrame(meta_rows)
    summary = {"code_hash": XV3_BOOT_CODE_HASH, "cells": [s["cell"] for s in cells], "draws": int(sum(status.values())),
               "draws_computed": int(status["computed"]), "draws_loaded": int(status["loaded"]),
               "elapsed_s": time.perf_counter() - t_all, "rows": int(len(res)),
               "nonok_rows_by_target": {str(k): int(v) for k, v in
                                        res[res["status"] != "ok"].groupby("target").size().items()},
               "fallback_pairs_total": (int(meta_df["fallback_pairs_total"].sum())
                                        if "fallback_pairs_total" in meta_df else 0),
               "draws_with_trap_replicate": (int((meta_df["trap_states_given_max"] > 0).sum())
                                             if "trap_states_given_max" in meta_df else 0),
               "max_width_minus_bound": (float(meta_df["checks.max_width_minus_bound"].max())
                                         if "checks.max_width_minus_bound" in meta_df else None),
               "max_rule_p_difference": (float(meta_df["checks.max_rule_p_difference"].max())
                                         if "checks.max_rule_p_difference" in meta_df else None),
               "draws_per_cell": {s["cell"]: int((meta_df["cell"] == s["cell"]).sum()) if len(meta_df) else 0
                                  for s in cells},
               "draws_complete": bool(cells) and all(int((meta_df["cell"] == s["cell"]).sum()) == int(s["draws"])
                                                     for s in cells)}
    return res, meta_df, summary


def x_boot_point_check(boot: pd.DataFrame, xresults: Optional[pd.DataFrame], tol: float = XV3_TOL) -> Dict[str, Any]:
    """Replicate -1 values (regenerated default policies) against the XRESULTS rows of the same draws."""
    if xresults is None or not len(xresults) or not len(boot):
        return {"status": "not compared (XRESULTS or bootstrap rows missing)", "match": None}
    k = ["cell", "draw", "policy", "target"]
    pt = boot[boot["replicate"] == XV3_BOOT_POINT][k + ["return"]]
    xr = xresults[xresults["cell"].isin(pt["cell"].unique()) & xresults["target"].isin(pt["target"].unique())]
    m = pt.merge(xr[k + ["return"]], on=k, how="left", suffixes=("", "_xresults"), validate="one_to_one")
    d = (m["return"] - m["return_xresults"]).abs()
    out = {"status": "compared", "rows": int(len(m)), "missing_in_xresults": int(m["return_xresults"].isna().sum()),
           "max_abs_diff": float(d.max()) if d.notna().any() else None,
           "rows_above_tol": int((d > tol).sum()), "tolerance": tol,
           "max_abs_diff_by_target": {str(t): float(g.max()) for t, g in d.groupby(m["target"]) if g.notna().any()}}
    out["match"] = bool(out["missing_in_xresults"] == 0 and out["rows_above_tol"] == 0)
    return out


# Objects whose source changes what a bootstrap draw computes (the checks are re-run on cached tables).
XV3_BOOT_CODE = [_x_v3_eval, x_boot_hospital, x_boot_step_map, x_boot_weights, x_boot_model, x_boot_identity,
                 run_v3_boot_draw]
XV3_BOOT_CODE_HASH = obj_hash({"code": code_hash(*XV3_BOOT_CODE),
                               "constants": {"point": XV3_BOOT_POINT, "statuses": list(XV3_EVAL_STATUSES),
                                             "columns": XV3_BOOT_COLUMNS}})

XV3_BOOT_RESULTS = pd.DataFrame(columns=XV3_BOOT_COLUMNS)
XV3_BOOT_META = pd.DataFrame()
XV3_BOOT_CELLS: List[dict] = []
XV3_BOOT_SUMMARY: Dict[str, Any] = {"status": "not run (xenv_v3_bootstrap is off)", "draws_complete": None,
                                    "code_hash": XV3_BOOT_CODE_HASH}
if EXPERIMENTS["xenv_v3_bootstrap"]:
    _by_cell = {str(s["cell"]): s for s in X_CELLS}
    XV3_BOOT_CELLS = [_by_cell[c] for c in XV3BOOT.cells if c in _by_cell]
    _skipped = [c for c in XV3BOOT.cells if c not in _by_cell]
    if _skipped:
        print("NOTE: bootstrap cells not in X_CELLS (family switch off or not in the design), skipped:", _skipped)
    if XV3_BOOT_CELLS:
        XV3_BOOT_RESULTS, XV3_BOOT_META, _s = x_v3_boot_run(XV3_BOOT_CELLS)
        XV3_BOOT_SUMMARY = {"status": "run", **_s, "cells_skipped": _skipped,
                            "point_check": x_boot_point_check(XV3_BOOT_RESULTS, globals().get("XRESULTS"))}
        XV3_BOOT_RESULTS.to_csv(XENV_TABLE_DIR / "v3_bootstrap_results.csv.gz", index=False, compression="gzip")
        XV3_BOOT_META.to_csv(XENV_TABLE_DIR / "v3_bootstrap_meta.csv", index=False)
        _pc = XV3_BOOT_SUMMARY["point_check"]
        if _pc.get("match") is False:
            print("NOTE: regenerated default policies differ from XRESULTS on", _pc["rows_above_tol"], "rows (max",
                  _pc["max_abs_diff"], ") - see xenv/v3_bootstrap_summary.json; the intervals use the regenerated values.")
        del _s, _pc
    else:
        XV3_BOOT_SUMMARY = {"status": "not run: no configured bootstrap cell is enabled", "cells_skipped": _skipped,
                            "draws_complete": None, "code_hash": XV3_BOOT_CODE_HASH}
    del _by_cell, _skipped
save_json(XV3_BOOT_SUMMARY, XENV_TABLE_DIR / "v3_bootstrap_summary.json")
print(json.dumps({k: v for k, v in XV3_BOOT_SUMMARY.items() if k not in ("draws_per_cell",)}, indent=1,
                 default=_json_default))

### II.8h ICU-Sepsis harm curves J(λ)

ICU-Sepsis has no ground truth for unsupported actions, but the bracketing theorem needs none: only the completion
is unknown. This section traces the identified interval of every policy along the **intermediate-harm** family

P_λ(·|s, a) = (1 − λ)·P_mean(·|s, a) + λ·e_death  for live s and a ∉ M(s),  λ ∈ {0, 0.1, …, 1},

on the released rows (`X_ICU`, the adapter verified against `EVALUATOR` in II.3a). Supported rows are left
bitwise unchanged (`x_harm_env`). λ = 0 is the mean rule and λ = 1 the termination rule; ICU-Sepsis pays 0 on death,
so r_term = v_min. By the theorem, every P_λ is a truth that satisfies no hidden benefit, so
J_term(π) ≤ J_λ(π) ≤ J_mean(π) for every policy. Applying the same identity to two harm levels also makes
J_λ non-increasing in λ. A certified pair (J_term(i) > J_mean(j)) is therefore ordered at every λ. The cell asserts
all three facts, and that the endpoints reproduce the reported values to 1e-9: λ = 0 matches `RESULTS` (mean) and
the `XRESULTS` `bench_mean` rows, and λ = 1 matches the terminate rule and `bench_terminate`.

**Policies.**
* Part I: every `REGISTRY` entry of the groups `main` and `reference`, at the first main seed (`CFG.seeds[0]`), or
  at `FIXED` for a fixed policy. `main_table` marks the policies of `MAIN_TABLE_POLICIES`.
* Part II: the default policies of ICU draw 0 (the Part I dataset). They are regenerated by running `run_xmethods`
  on the II.6 training split of `icu|released` draw 0, under the same key as II.6, plus the behaviour policy and
  the projections (`x_collect_records`).

**Reported per policy set and λ** (`xenv/v3_icu_harm_summary.csv`), over the policies that are proper at every λ:
Kendall τ-b between the λ = 0 ranking and the λ ranking (ties within 1e-9 snapped, as in II.7); the number of pairs
whose strict order at λ = 0 is reversed at λ; the certified pairs (λ-free); and the λ-robust pairs, strictly ordered
the same way at every grid λ. Certified pairs are a subset of the λ-robust pairs.

Both curve tables are cached in `XCACHE`. Part II: key `v3harm/icu_released/draw0`, with the II.6 draw identity,
the code hash of this section and the λ grid. Part I: key `v3harm/part1`, with the hashes of the evaluated policy
arrays and the `X_ICU` identity. Part I curves are pure evaluation, so they are computed even with
`ALLOW_TRAINING = False`.

**Expected cost on Colab.** About 120 policies × 11 λ exact solves on the 716-state model (about 30–60 ms each),
plus one `run_xmethods` fit on ICU draw 0 (about 30 s in the II.6 timings): about 2–3 min in total.

| Output | Content |
|---|---|
| `xenv/v3_icu_harm_curves.csv` | source (part1 / part2), policy, seed or draw, group, `main_table`, λ, status, return, P(any unsupported decision), expected violations |
| `xenv/v3_icu_harm_summary.csv` | per policy set and λ: τ-b against λ = 0, reversals, certified and λ-robust pairs and shares |
| `xenv/v3_icu_harm_checks.json` | endpoint identities, bracketing, monotonicity and certified-order checks; `passes` |
| `figures/xenv/fig_v3_icu_harm.*` | (a) J(λ) of the Part I main-table policies; (b) τ-b(λ) against the mean-rule ranking per policy set |

In [ ]:
# ============================================================
# II.8h ICU-Sepsis harm curves J(lambda)
# ============================================================
EXECLOG.mark("II.8 Revision experiments: ICU-Sepsis harm curves")
XV3_HARM_LAMBDAS = tuple(float(x) for x in np.round(np.linspace(0.0, 1.0, 11), 10))
XV3_HARM_GROUPS = ("main", "reference")
XV3_HARM_COLUMNS = ["source", "policy", "seed", "draw", "group", "main_table", "method", "variant", "lambda", "status",
                    "return", "prob_any_violation", "expected_violations"]
assert XV3_HARM_LAMBDAS[0] == 0.0 and XV3_HARM_LAMBDAS[-1] == 1.0 and np.all(np.diff(XV3_HARM_LAMBDAS) > 0)


def x_harm_env(env: FiniteEnv, lam: float) -> FiniteEnv:
    """P_lambda: unsupported live rows (1 - lam) * P + lam * e_sink; supported and absorbing rows bitwise unchanged."""
    lam = float(lam)
    assert 0.0 <= lam <= 1.0, lam
    P = env.P.copy()
    bad = ~env.mask & env.live[:, None]
    P[bad] *= 1.0 - lam
    P[bad, env.sink] += lam
    return env.variant(f"{env.name}|harm{lam:g}", P=P, harm_lambda=lam)


def x_harm_curves(env: FiniteEnv, policies: Sequence[dict], lambdas: Sequence[float]) -> pd.DataFrame:
    """J(lambda) of every policy (dicts with 'pi' and the descriptive XV3_HARM_COLUMNS keys) on x_harm_env(env, lambda)."""
    keys = [_x_pi_key(p["pi"]) for p in policies]
    rows = []
    for lam in lambdas:
        env_l = x_harm_env(env, lam)
        memo: Dict[bytes, dict] = {}
        for p, k in zip(policies, keys):
            if k not in memo:
                memo[k] = _x_v3_eval(env_l, p["pi"], "given")
            ev = memo[k]
            rows.append({**{c: p.get(c, np.nan) for c in XV3_HARM_COLUMNS[:8]}, "lambda": float(lam),
                         "status": ev["status"],
                         **{c: ev.get(c, np.nan) for c in ("return", "prob_any_violation", "expected_violations")}})
        del env_l, memo
        gc.collect()
    return pd.DataFrame(rows, columns=XV3_HARM_COLUMNS)


def x_harm_part1_policies() -> List[dict]:
    """REGISTRY policies of the groups main and reference at the first main seed (FIXED for fixed policies)."""
    seed0 = int(CFG.seeds[0])
    out = []
    for name, e in REGISTRY.items():
        if e.group not in XV3_HARM_GROUPS:
            continue
        seed = seed0 if seed0 in e.by_seed else (FIXED if FIXED in e.by_seed else None)
        if seed is None:
            continue
        out.append({"source": "part1", "policy": name, "seed": int(seed), "draw": np.nan, "group": e.group,
                    "main_table": name in MAIN_TABLE_POLICIES, "method": e.family, "variant": e.mask_use,
                    "pi": validate_policy(e.by_seed[seed], f"{name} seed={seed}", shape=(X_ICU.S, X_ICU.A))})
    return out


def x_harm_part1(policies: List[dict]) -> pd.DataFrame:
    """Part I curves (cached; pure evaluation, so a miss is computed even when ALLOW_TRAINING is False)."""
    identity = {"kind": "xenv_v3_icu_harm_part1", "x_icu": X_ICU.identity(), "code": XV3_HARM_CODE_HASH,
                "lambdas": list(XV3_HARM_LAMBDAS),
                "policies": sorted([p["policy"], int(p["seed"]), array_hash(p["pi"])] for p in policies)}
    key = "v3harm/part1"
    hit = XCACHE.get(key, identity)
    if hit is not None:
        return hit
    XCACHE.put(key, identity, x_harm_curves(X_ICU, policies, XV3_HARM_LAMBDAS))
    return XCACHE.get(key, identity)


def x_harm_icu_spec() -> dict:
    spec = {"cell": "icu|released", "family": "icu", "draws": int(XCFG.icu_draws),
            "user_episodes": int(XCFG.icu_user_episodes)}
    hit = [s for s in X_CELLS if s["cell"] == "icu|released"]
    if hit:
        assert hit[0] == spec, f"icu|released spec differs from x_build_cells: {hit[0]}"
    return spec


def x_harm_part2() -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Part II curves of the ICU draw-0 default policies (regenerated with run_xmethods; cached)."""
    spec = x_harm_icu_spec()
    inst = build_cell_instance(spec, 0)
    assert all(np.array_equal(inst.data[k], OFFLINE[k]) for k in ("state", "action", "next_state", "episode_id")), \
        "ICU draw 0 must be the Part I dataset (OFFLINE)"
    identity = {"kind": "xenv_v3_icu_harm_part2", "draw": x_draw_identity(spec, inst), "code": XV3_HARM_CODE_HASH,
                "lambdas": list(XV3_HARM_LAMBDAS)}
    key = "v3harm/" + x_cache_key("icu|released", 0)
    hit = XCACHE.get(key, identity)
    if hit is not None:
        return hit, {"cache": "loaded"}
    XCACHE.require_allowed(key)
    t0 = time.perf_counter()
    raw = run_xmethods(LearnerView.from_env(inst.learn_env), inst.train, "icu|released|draw0")
    recs = [r for r in x_collect_records(raw, inst.behavior, inst.learn_env) if r["is_default"]]
    del raw
    pols = [{"source": "part2", "policy": r["label"], "seed": np.nan, "draw": 0, "group": "default",
             "main_table": False, "method": r["method"], "variant": r["variant"], "pi": r["pi"]} for r in recs]
    t1 = time.perf_counter()
    XCACHE.put(key, identity, x_harm_curves(inst.learn_env, pols, XV3_HARM_LAMBDAS))
    del inst, recs, pols
    gc.collect()
    return XCACHE.get(key, identity), {"cache": "computed", "methods_s": t1 - t0, "curves_s": time.perf_counter() - t1}


def x_harm_endpoints(curves: pd.DataFrame, ref: pd.DataFrame, keys: Sequence[str]) -> Dict[str, Any]:
    """|J(lambda) - reference| at lambda in {0, 1}; `ref` has columns keys + ['lambda', 'return_ref']."""
    if curves.empty or ref.empty:
        return {"rows": 0, "missing_reference": None, "max_abs_diff": None, "rows_above_tol": 0,
                "status": "not compared (no curves or no reference rows)"}
    m = curves[curves["lambda"].isin([0.0, 1.0])].merge(ref, on=[*keys, "lambda"], how="left", validate="one_to_one")
    d = (m["return"] - m["return_ref"]).abs()
    return {"rows": int(len(m)), "missing_reference": int(m["return_ref"].isna().sum()),
            "nonok_curve_rows": int((m["status"] != "ok").sum()),
            "max_abs_diff": float(d.max()) if d.notna().any() else None, "rows_above_tol": int((d > XV3_TOL).sum()),
            "status": "compared"}


def x_harm_pair_stats(J: np.ndarray, lambdas: Sequence[float], tol: float = XV3_TOL) -> Tuple[List[dict], Dict[str, int]]:
    """Per-lambda ranking statistics of J [n_policies, n_lambda] (lambda ascending; column 0 = mean, -1 = terminate)."""
    J = np.asarray(J, dtype=np.float64)
    n, L = J.shape
    pairs = n * (n - 1) // 2
    iu = np.triu_indices(n, 1)
    D = (J[:, None, :] - J[None, :, :])[iu]                       # [pairs, L], i < j
    S = np.where(D > tol, 1, np.where(D < -tol, -1, 0))
    robust = (S != 0).all(axis=1) & (S == S[:, :1]).all(axis=1)
    cert = J[:, -1][:, None] > J[:, 0][None, :] + tol           # lower(i) > upper(j)
    np.fill_diagonal(cert, False)
    cert_u = cert[iu] | cert.T[iu]
    sign_c = np.where(cert[iu], 1, -1)                            # direction of a certified pair
    checks = {"certified_order_violations": int((cert_u & ((D * sign_c[:, None]).min(axis=1) <= 0)).sum()),
              "certified_not_lambda_robust": int((cert_u & ~robust).sum()),
              "bracket_violations": int(((J < J[:, -1:] - tol) | (J > J[:, :1] + tol)).sum()),
              "monotonicity_violations": int((np.diff(J, axis=1) > tol).sum()),
              "both_directions_certified": int((cert & cert.T).sum())}
    s0 = xs_snap_ties(J[:, 0]) if n else J[:, 0]
    rows = []
    for l, lam in enumerate(lambdas):
        sl = xs_snap_ties(J[:, l]) if n else J[:, l]
        tau = (float(stats.kendalltau(s0, sl)[0]) if n >= 2 and np.ptp(s0) > 0 and np.ptp(sl) > 0 else np.nan)
        rev = int(((S[:, 0] * S[:, l]) < 0).sum())
        rows.append({"lambda": float(lam), "n_policies": int(n), "pairs": int(pairs), "kendall_tau_b_vs_lambda0": tau,
                     "ordered_pairs": int((S[:, l] != 0).sum()), "reversals_vs_lambda0": rev,
                     "reversal_share": rev / pairs if pairs else np.nan, "certified_pairs": int(cert.sum()),
                     "certified_share": int(cert.sum()) / pairs if pairs else np.nan,
                     "lambda_robust_pairs": int(robust.sum()),
                     "lambda_robust_share": int(robust.sum()) / pairs if pairs else np.nan})
    return rows, checks


XV3_HARM_SETS = (("part1", "all", None), ("part1", "main_table", True), ("part2", "default", None))


def x_harm_summary(curves: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Ranking statistics per policy set and lambda over the policies proper at every lambda; pooled checks."""
    rows, checks = [], {}
    for source, name, main_only in XV3_HARM_SETS:
        d = curves[curves["source"] == source]
        if main_only:
            d = d[d["main_table"].astype(bool)]
        if d.empty:
            continue
        idx = ["policy", "seed"] if source == "part1" else ["policy", "draw"]
        J = d.set_index(idx + ["lambda"])["return"].unstack("lambda").reindex(columns=list(XV3_HARM_LAMBDAS))
        ok = d.groupby(idx)["status"].agg(lambda s: bool((s == "ok").all())).reindex(J.index)
        Jok = J[ok.to_numpy()]
        r, c = x_harm_pair_stats(Jok.to_numpy(), XV3_HARM_LAMBDAS)
        rows += [{"source": source, "policy_set": name, "policies_total": int(len(J)),
                  "policies_not_proper_everywhere": int((~ok).sum()), **x} for x in r]
        checks[f"{source}/{name}"] = {**c, "policies": int(len(Jok)), "not_proper_everywhere": int((~ok).sum())}
    return pd.DataFrame(rows), checks


def xs_fig_v3_icu_harm(curves: pd.DataFrame, summary: pd.DataFrame, fig_dir: Path) -> str:
    """(a) J(lambda) of the Part I main-table policies; (b) Kendall tau-b against the lambda = 0 ranking."""
    d = curves[(curves["source"] == "part1") & curves["main_table"].astype(bool) & (curves["status"] == "ok")]
    if d.empty or summary.empty:
        return "not produced: no Part I main-table harm curves"
    J = d.set_index(["policy", "lambda"])["return"].unstack("lambda").dropna()
    width = (J[0.0] - J[1.0]).sort_values(ascending=False)
    top = [p for p in width.index[:4] if width[p] > XV3_TOL]
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    ax = axes[0]
    lam = np.asarray(J.columns, dtype=float)
    for p in J.index:
        if p not in top:
            ax.plot(lam, J.loc[p].to_numpy(), color=XS_MUTED, lw=1.0, alpha=0.55, zorder=1)
    for i, p in enumerate(top):
        ax.plot(lam, J.loc[p].to_numpy(), color=XS_COLORS[i], lw=2.0, marker="o", ms=4, zorder=3,
                label=f"{p} (width {width[p]:.3f})")
    ax.plot([], [], color=XS_MUTED, lw=1.0, label=f"other main-table policies ({len(J) - len(top)})")
    ax.set_xlabel("harm λ of unsupported actions (0 = mean rule, 1 = terminate)")
    ax.set_ylabel("exact return J(λ) on ICU-Sepsis")
    ax.set_title("(a) Part I main-table policies, seed 0", loc="left", color=XS_INK)
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(0.0, -0.2))
    ax = axes[1]
    labels = {("part1", "all"): "Part I, main + reference", ("part1", "main_table"): "Part I, main table",
              ("part2", "default"): "Part II, ICU draw 0 defaults"}
    for i, ((src, nm), g) in enumerate(summary.groupby(["source", "policy_set"], sort=False)):
        g = g.sort_values("lambda")
        cs = float(g["certified_share"].iloc[0])
        ax.plot(g["lambda"], g["kendall_tau_b_vs_lambda0"], color=XS_COLORS[i], lw=2.0, marker="o", ms=4,
                label=f"{labels.get((src, nm), f'{src} {nm}')}: n = {int(g['n_policies'].iloc[0])}, "
                      f"certified {100 * cs:.0f}% of pairs")
    ax.set_xlabel("harm λ")
    ax.set_ylabel("Kendall τ-b vs the λ = 0 ranking")
    ax.set_title("(b) ranking agreement with the mean rule", loc="left", color=XS_INK)
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(0.0, -0.2))
    return _xs_save(fig, fig_dir, "fig_v3_icu_harm")


# Objects whose source changes what the harm curves contain.
XV3_HARM_CODE = [x_harm_env, x_harm_curves, x_harm_part1_policies, x_harm_icu_spec, _x_v3_eval]
XV3_HARM_CODE_HASH = obj_hash({"code": code_hash(*XV3_HARM_CODE), "columns": XV3_HARM_COLUMNS,
                               "groups": list(XV3_HARM_GROUPS)})

XV3_HARM_CURVES = pd.DataFrame(columns=XV3_HARM_COLUMNS)
XV3_HARM_SUMMARY = pd.DataFrame()
XV3_HARM_CHECK: Dict[str, Any] = {"status": "not run (xenv_v3_icu_harm is off)"}
if EXPERIMENTS["xenv_v3_icu_harm"]:
    assert X_ICU is not None, "X_ICU is needed (Section II.3a or II.8f builds it)"
    _t0 = time.perf_counter()
    _p1 = x_harm_part1_policies()
    _c1 = x_harm_part1(_p1) if _p1 else pd.DataFrame(columns=XV3_HARM_COLUMNS)
    _c2, _m2 = x_harm_part2()
    XV3_HARM_CURVES = pd.concat([_c1, _c2], ignore_index=True)
    XV3_HARM_SUMMARY, _hc = x_harm_summary(XV3_HARM_CURVES)
    # Endpoint identities: Part I against RESULTS (mean / terminate), Part II against XRESULTS (bench_mean / bench_terminate).
    _r1 = RESULTS[RESULTS["group"].isin(XV3_HARM_GROUPS) & RESULTS["rule"].isin(["mean", "terminate"])]
    _r1 = pd.DataFrame({"policy": _r1["name"].astype(str), "seed": pd.to_numeric(_r1["seed"]).astype(float),
                        "lambda": _r1["rule"].map({"mean": 0.0, "terminate": 1.0}),
                        "return_ref": pd.to_numeric(_r1["return"], errors="coerce")}).drop_duplicates(
        ["policy", "seed", "lambda"])
    _xr = globals().get("XRESULTS")
    _xr = pd.DataFrame(columns=XRESULT_COLUMNS) if _xr is None else _xr
    _r2 = _xr[(_xr["cell"] == "icu|released") & (pd.to_numeric(_xr["draw"]) == 0)
              & _xr["target"].isin(["bench_mean", "bench_terminate"]) & ~_xr["is_reference"].astype(bool)]
    _r2 = pd.DataFrame({"policy": _r2["policy"].astype(str),
                        "lambda": _r2["target"].map({"bench_mean": 0.0, "bench_terminate": 1.0}),
                        "return_ref": pd.to_numeric(_r2["return"], errors="coerce")})
    _c1k = XV3_HARM_CURVES[XV3_HARM_CURVES["source"] == "part1"].assign(seed=lambda d: d["seed"].astype(float))
    _c2k = XV3_HARM_CURVES[XV3_HARM_CURVES["source"] == "part2"]
    XV3_HARM_CHECK = {
        "status": "run", "lambdas": list(XV3_HARM_LAMBDAS), "code_hash": XV3_HARM_CODE_HASH,
        "part1_policies": int(len(_p1)), "part2": _m2,
        "part2_policies": int(_c2k["policy"].nunique()),
        "endpoints_part1_vs_RESULTS": x_harm_endpoints(_c1k, _r1, ["policy", "seed"]),
        "endpoints_part2_vs_XRESULTS": x_harm_endpoints(_c2k, _r2[_r2["policy"].isin(_c2k["policy"])], ["policy"]),
        "sets": _hc, "seconds": time.perf_counter() - _t0}
    _viol = sum(v[k] for v in _hc.values() for k in ("certified_order_violations", "certified_not_lambda_robust",
                                                       "bracket_violations", "monotonicity_violations",
                                                       "both_directions_certified"))
    # A compared endpoint row without its reference row counts as a failure (it would otherwise pass unchecked).
    XV3_HARM_CHECK["passes"] = bool(
        _viol == 0 and all(XV3_HARM_CHECK[k]["rows_above_tol"] == 0 and not XV3_HARM_CHECK[k].get("missing_reference")
                           for k in ("endpoints_part1_vs_RESULTS", "endpoints_part2_vs_XRESULTS")))
    XV3_HARM_CURVES.to_csv(XENV_TABLE_DIR / "v3_icu_harm_curves.csv", index=False)
    XV3_HARM_SUMMARY.to_csv(XENV_TABLE_DIR / "v3_icu_harm_summary.csv", index=False)
    XV3_HARM_CHECK["figure"] = xs_fig_v3_icu_harm(XV3_HARM_CURVES, XV3_HARM_SUMMARY, XENV_FIG_DIR)
    if len(XV3_HARM_SUMMARY):
        display(XV3_HARM_SUMMARY[XV3_HARM_SUMMARY["lambda"].isin([0.5, 1.0])][
            ["source", "policy_set", "lambda", "n_policies", "kendall_tau_b_vs_lambda0", "reversals_vs_lambda0",
             "certified_share", "lambda_robust_share"]].round(4))
    del _p1, _c1, _c2, _m2, _hc, _r1, _r2, _xr, _c1k, _c2k, _viol
save_json(XV3_HARM_CHECK, XENV_TABLE_DIR / "v3_icu_harm_checks.json")
print("II.8h harm curves:", {k: XV3_HARM_CHECK.get(k) for k in ("status", "passes", "part1_policies", "part2_policies")})
if XV3_HARM_CHECK.get("passes") is False:
    raise AssertionError("ICU-Sepsis harm-curve identities failed (xenv/v3_icu_harm_checks.json)")

### II.8i Additional analyses: bracketing coverage, certified comparisons, rank fidelity, bootstrap intervals

These analyses use the definitions of the paper. They match the paper-side analysis of the II.6 cells, so
the II.8 numbers can be read next to those.

* **Policies.** Bracketing uses every non-reference policy of every draw. Certified comparisons and rank fidelity
  use the default policy set of each draw. A policy is **violating** if P(any unsupported decision) > 1e-12 on
  `bench_mean`. The value tolerance is 1e-9.
* **Bracketing on the true rows** (semantics only, where the theorem applies exactly; not for the aggregated cells,
  whose benchmark state space differs from the true one). The lower bound is J(true rows + terminate), the upper
  bound J(true rows + mean rule), and the truth J(true system). Reported: lower-bound violations (0 by Theorem
  part i), upper-bound coverage overall, among violating policies, and among policies violating on the true rows;
  the size of the upper-bound violation when it fails, also divided by p·(v_max − v_min); and the interval width.
* **Bracketing at benchmark level** (completion plus estimation error): J_true ∈ [J_bench_terminate, J_bench_mean].
* **Certified comparisons** per draw, over the default policies, on the true-rows targets and on the benchmark
  targets. A pair (i, j) is certified if lower(i) > upper(j) + 1e-9. It is a certification error if
  J_true(i) ≤ J_true(j) + 1e-9, and a reversal if J_true(i) < J_true(j) − 1e-9. Point claims under one rule,
  J_rule(i) > J_rule(j) + 1e-9, are scored the same way. Shares are over unordered pairs, with counts pooled over
  draws.
* **Rank fidelity** (exploratory, not preregistered): Kendall τ-b, top-1 agreement and selection regret of the
  benchmark ranking against the true ranking per draw (`xs_rank_fidelity_draws` of II.7), and the paired
  difference τ(mean) − τ(terminate) with stratified-bootstrap intervals over draws.
* **Bootstrap intervals** (II.8g). The interval is [q_2.5%(J_term^b), q_97.5%(J_mean^b)] over the proper
  replicates; the point interval is [J_bench_terminate, J_bench_mean]. Coverage of J_true, and certified comparisons
  with each kind of interval, are computed per cell and pooled.
* **Hidden benefit** (random-row Garnet cells): per draw, the upper-bound coverage among violating policies against
  the draw's share of non-preferred pairs whose action value beats the preferred mean (II.8d), with the Spearman
  correlation over draws.

The theorem facts (no lower-bound violation on the true rows, non-negative widths, zero width for policies that never
take an unsupported action, values inside [v_min, v_max]) are counted in `xenv/v3_stats_checks.json` (`passes`).
Coverage and certification rates are scientific outcomes; they are reported and never asserted.

| Output | Content |
|---|---|
| `xenv/v3_bracketing.csv` | per II.8 cell and pooled by kind: semantics-only (`sem_*`) and benchmark-level (`bench_*`) bracketing statistics |
| `xenv/v3_bracketing_draws.csv` | per cell and draw: violating policies, lower-bound violations, upper / benchmark coverage, hidden-benefit share |
| `xenv/v3_certified.csv`, `xenv/v3_certified_draws.csv` | certified comparisons per cell (and pooled) and per draw, for `true_rows` and `bench` targets |
| `xenv/v3_rank_fidelity.csv`, `xenv/v3_rank_fidelity_draws.csv` | rank fidelity per cell and completion, and τ(mean) − τ(terminate) |
| `xenv/v3_bootstrap_intervals.csv` | per bootstrap cell, draw and default policy: J_true, point and bootstrap interval, coverage flags |
| `xenv/v3_bootstrap_coverage.csv` | per bootstrap cell and pooled, for point and bootstrap intervals: coverage and certified comparisons |
| `xenv/v3_stats_checks.json` | theorem checks, input sizes, figure status, outputs |
| `figures/xenv/fig_v3_bracketing.*` | (a) upper-bound coverage (true rows) and benchmark-level coverage among violating policies per II.8 cell; (b) random-row Garnet: coverage per draw against the hidden-benefit share |
| `figures/xenv/fig_v3_bootstrap.*` | per bootstrap cell, point vs bootstrap intervals: (a) coverage among violating policies, (b) certified share, (c) certified error rate |

In [ ]:
# ============================================================
# II.8i Revision analyses
# ============================================================
EXECLOG.mark("II.8 Revision analyses")
XV3_BR_PTOL = 1e-12                       # violating: P(any unsupported decision) > XV3_BR_PTOL on bench_mean
XV3_VRANGE = {"gumbel": (-1.0, 1.0), "garnet": (0.0, 1.0), "icu": (0.0, 1.0)}   # [v_min, v_max]; r_term = v_min
XV3_BR_TARGETS = ["bench_mean", "bench_terminate", "true", "true_rows_mean", "true_rows_terminate"]


def _xv3_rate(num, den) -> float:
    return float(num / den) if den else float("nan")


def xv3_bracket_frame(results: pd.DataFrame, kind_of: Dict[str, str]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Per (cell, draw, policy) values on the bracketing targets and the bound / coverage flags (non-reference rows)."""
    cols = ["cell", "draw", "policy", "family", "v3_kind", "is_default", *XV3_BR_TARGETS, "p_bench", "p_true",
            "has_rows", "has_bench", "viol", "viol_true", "lo_ok", "up_ok", "b_lo_ok", "b_up_ok", "b_in", "exceed",
            "width", "bench_width", "exceed_ratio", "width_ratio"]
    if results is None or not len(results):
        return pd.DataFrame(columns=cols), {"rows": 0}
    r = results[results["target"].isin(XV3_BR_TARGETS) & ~xs_as_bool(results["is_reference"])].copy()
    r["draw"] = pd.to_numeric(r["draw"]).astype(np.int64)
    k = ["cell", "draw", "policy"]
    assert not r.duplicated(k + ["target"]).any(), "duplicated (cell, draw, policy, target) rows"
    ok = r["status"] == "ok"
    J = r[ok].set_index(k + ["target"])["return"].unstack("target")
    Pv = r[ok].set_index(k + ["target"])["prob_any_violation"].unstack("target")
    W = r.drop_duplicates(k).set_index(k)[["family", "is_default"]].copy()
    W["is_default"] = xs_as_bool(W["is_default"]).to_numpy()
    W = W.join(J.reindex(columns=XV3_BR_TARGETS))
    W["v3_kind"] = W.index.get_level_values("cell").map(lambda c: kind_of.get(c, ""))
    Pv = Pv.reindex(index=W.index, columns=XV3_BR_TARGETS)
    W["p_bench"] = Pv["bench_mean"]
    W["p_true"] = Pv["true_rows_mean"].where(Pv["true_rows_mean"].notna(), Pv["true"])
    W["has_rows"] = W[["true", "true_rows_mean", "true_rows_terminate"]].notna().all(axis=1)
    W["has_bench"] = W[["true", "bench_mean", "bench_terminate"]].notna().all(axis=1)
    W["viol"] = W["p_bench"] > XV3_BR_PTOL
    W["viol_true"] = W["p_true"] > XV3_BR_PTOL
    W["lo_ok"] = W["true"] >= W["true_rows_terminate"] - XV3_TOL
    W["up_ok"] = W["true"] <= W["true_rows_mean"] + XV3_TOL
    W["b_lo_ok"] = W["true"] >= W["bench_terminate"] - XV3_TOL
    W["b_up_ok"] = W["true"] <= W["bench_mean"] + XV3_TOL
    W["b_in"] = W["b_lo_ok"] & W["b_up_ok"]
    W["exceed"] = W["true"] - W["true_rows_mean"]
    W["width"] = W["true_rows_mean"] - W["true_rows_terminate"]
    W["bench_width"] = W["bench_mean"] - W["bench_terminate"]
    vr = W["family"].map(lambda f: XV3_VRANGE[f][1] - XV3_VRANGE[f][0])
    bnd = W["p_true"] * vr
    W["exceed_ratio"] = np.where(bnd > 0, W["exceed"] / bnd.where(bnd > 0, 1.0), np.nan)
    W["width_ratio"] = np.where(bnd > 0, W["width"] / bnd.where(bnd > 0, 1.0), np.nan)
    rng_bad = 0
    for fam, (lo, hi) in XV3_VRANGE.items():
        v = W.loc[W["family"] == fam, XV3_BR_TARGETS].to_numpy(dtype=float)
        rng_bad += int(((v < lo - XV3_TOL) | (v > hi + XV3_TOL)).sum())
    hr, hb = W["has_rows"], W["has_bench"]
    checks = {"rows": int(len(W)), "nonok_rows": int((~ok).sum()), "policy_rows_with_true_rows": int(hr.sum()),
              "policy_rows_with_bench": int(hb.sum()), "values_outside_value_range": rng_bad,
              "lower_bound_violations_true_rows": int((hr & ~W["lo_ok"]).sum()),
              "negative_width_true_rows": int((hr & (W["width"] < -XV3_TOL)).sum()),
              "negative_width_bench": int((hb & (W["bench_width"] < -XV3_TOL)).sum()),
              "nonzero_width_admissible": int((hr & ~W["viol_true"] & (W["width"].abs() > XV3_TOL)).sum()),
              "width_above_bound": int((hr & (W["width"] > bnd + XV3_TOL)).sum())}
    return W.reset_index()[cols], checks


def xv3_bracket_semantics(d: pd.DataFrame) -> Dict[str, Any]:
    d = d[d["has_rows"]]
    if d.empty:
        return {"n_rows": 0}
    v, f, vt = d[d["viol"]], d[~d["up_ok"]], d[d["viol_true"]]
    return {"n_rows": int(len(d)), "n_draws": int(d["draw"].nunique()), "n_violating": int(len(v)),
            "lower_violations": int((~d["lo_ok"]).sum()), "lower_holds_violating": _xv3_rate(v["lo_ok"].sum(), len(v)),
            "max_lower_shortfall": float((d["true_rows_terminate"] - d["true"]).max()),
            "upper_coverage_all": _xv3_rate(d["up_ok"].sum(), len(d)),
            "upper_coverage_violating": _xv3_rate(v["up_ok"].sum(), len(v)),
            "upper_coverage_violating_truep": _xv3_rate(vt["up_ok"].sum(), len(vt)), "n_violating_truep": int(len(vt)),
            "upper_failures": int(len(f)), "upper_exceedance_mean": float(f["exceed"].mean()) if len(f) else 0.0,
            "upper_exceedance_median": float(f["exceed"].median()) if len(f) else 0.0,
            "upper_exceedance_max": float(f["exceed"].max()) if len(f) else 0.0,
            "upper_exceedance_max_over_p_range": float(np.nanmax(f["exceed_ratio"])) if len(f) else 0.0,
            "width_mean_violating": float(v["width"].mean()) if len(v) else 0.0,
            "width_max_over_p_range": (float(np.nanmax(d["width_ratio"])) if d["width_ratio"].notna().any() else 0.0)}


def xv3_bracket_benchmark(d: pd.DataFrame) -> Dict[str, Any]:
    d = d[d["has_bench"]]
    if d.empty:
        return {"n_rows": 0}
    v = d[d["viol"]]
    return {"n_rows": int(len(d)), "n_violating": int(len(v)), "coverage_all": _xv3_rate(d["b_in"].sum(), len(d)),
            "coverage_violating": _xv3_rate(v["b_in"].sum(), len(v)),
            "lower_holds_violating": _xv3_rate(v["b_lo_ok"].sum(), len(v)),
            "upper_holds_violating": _xv3_rate(v["b_up_ok"].sum(), len(v)),
            "lower_violations_all": int((~d["b_lo_ok"]).sum()), "upper_violations_all": int((~d["b_up_ok"]).sum()),
            "coverage_nonviolating": _xv3_rate(d.loc[~d["viol"], "b_in"].sum(), int((~d["viol"]).sum()))}


def xv3_pairs(lo: np.ndarray, up: np.ndarray, jt: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """Certified pairs and single-rule point claims over one policy set, scored against the truth when given."""
    lo, up = np.asarray(lo, dtype=float), np.asarray(up, dtype=float)
    n = len(lo)
    out = {"n_policies": n, "pairs": n * (n - 1) // 2}
    cert = lo[:, None] > up[None, :] + XV3_TOL
    np.fill_diagonal(cert, False)
    out["certified"] = int(cert.sum())
    pm = up[:, None] > up[None, :] + XV3_TOL
    pt = lo[:, None] > lo[None, :] + XV3_TOL
    out["claims_mean"], out["claims_terminate"] = int(pm.sum()), int(pt.sum())
    out["rule_dependent"] = int((pm & (lo[:, None] < lo[None, :] - XV3_TOL)).sum())
    if jt is not None:
        jt = np.asarray(jt, dtype=float)
        wrong = jt[:, None] <= jt[None, :] + XV3_TOL
        rev = jt[:, None] < jt[None, :] - XV3_TOL
        out["truth_ordered"] = int((jt[:, None] > jt[None, :] + XV3_TOL).sum())
        out["certified_errors"], out["certified_reversals"] = int((cert & wrong).sum()), int((cert & rev).sum())
        for nm, cl in (("mean", pm), ("terminate", pt)):
            out[f"errors_{nm}"], out[f"reversals_{nm}"] = int((cl & wrong).sum()), int((cl & rev).sum())
            unc = cl & ~cert
            out[f"claims_{nm}_uncertified"], out[f"errors_{nm}_uncertified"] = int(unc.sum()), int((unc & wrong).sum())
    return out


def xv3_cc_summary(d: pd.DataFrame) -> Dict[str, Any]:
    if d.empty:
        return {"draws": 0}
    s = d.drop(columns=[c for c in ("cell", "draw", "family", "targets", "kind") if c in d]).sum(numeric_only=True)
    share = d["certified"] / d["pairs"].where(d["pairs"] > 0)
    o = {"draws": int(len(d)), "policies_per_draw": int(d["n_policies"].median()), "pairs": int(s["pairs"]),
         "certified": int(s["certified"]), "certified_share": _xv3_rate(s["certified"], s["pairs"]),
         "certified_share_draw_min": float(share.min()), "certified_share_draw_max": float(share.max()),
         "rule_dependent_share": _xv3_rate(s["rule_dependent"], s["pairs"]),
         "claims_mean_share": _xv3_rate(s["claims_mean"], s["pairs"]),
         "claims_terminate_share": _xv3_rate(s["claims_terminate"], s["pairs"])}
    if "certified_errors" in d and d["certified_errors"].notna().all():
        o.update({"truth_ordered_share": _xv3_rate(s["truth_ordered"], s["pairs"]),
                  "certified_errors": int(s["certified_errors"]),
                  "certified_error_rate": _xv3_rate(s["certified_errors"], s["certified"]),
                  "certified_reversal_rate": _xv3_rate(s["certified_reversals"], s["certified"])})
        for nm in ("mean", "terminate"):
            o.update({f"point_{nm}_errors": int(s[f"errors_{nm}"]),
                      f"point_{nm}_error_rate": _xv3_rate(s[f"errors_{nm}"], s[f"claims_{nm}"]),
                      f"point_{nm}_reversal_rate": _xv3_rate(s[f"reversals_{nm}"], s[f"claims_{nm}"]),
                      f"point_{nm}_error_rate_uncertified": _xv3_rate(s[f"errors_{nm}_uncertified"],
                                                                      s[f"claims_{nm}_uncertified"])})
    return o


def xv3_certified_draws(W: pd.DataFrame) -> pd.DataFrame:
    """Per (cell, draw, targets) certified-comparison counts over the default policies (true_rows and bench targets)."""
    rows = []
    for (c, dr), d in W[W["is_default"]].groupby(["cell", "draw"], sort=False):
        jt = d["true"].to_numpy(dtype=float)
        for tf, (lc, uc) in (("true_rows", ("true_rows_terminate", "true_rows_mean")),
                             ("bench", ("bench_terminate", "bench_mean"))):
            if d[[lc, uc, "true"]].isna().any().any():
                continue
            rows.append({"cell": c, "draw": int(dr), "family": d["family"].iloc[0], "kind": d["v3_kind"].iloc[0],
                         "targets": tf, **xv3_pairs(d[lc].to_numpy(dtype=float), d[uc].to_numpy(dtype=float), jt)})
    return pd.DataFrame(rows)


def xv3_pools(cells: Sequence[str], kind_of: Dict[str, str]) -> Dict[str, List[str]]:
    pools: Dict[str, List[str]] = {}
    for c in cells:
        pools.setdefault(f"pooled:{kind_of.get(c, c.split('|', 1)[0])}", []).append(c)
    return {k: v for k, v in pools.items() if len(v) > 1}


def xv3_bracketing_table(W: pd.DataFrame, cells: Sequence[str], kind_of: Dict[str, str]) -> pd.DataFrame:
    rows = []
    units = [(c, [c]) for c in cells] + list(xv3_pools(cells, kind_of).items())
    for name, cs in units:
        d = W[W["cell"].isin(cs)]
        if d.empty:
            continue
        rows.append({"cell": name, "kind": kind_of.get(name, name.split(":", 1)[-1]), "n_cells": len(cs),
                     **{f"sem_{k}": v for k, v in xv3_bracket_semantics(d).items()},
                     **{f"bench_{k}": v for k, v in xv3_bracket_benchmark(d).items()}})
    return pd.DataFrame(rows)


def xv3_bracketing_draws(W: pd.DataFrame, draw_meta: Optional[pd.DataFrame]) -> pd.DataFrame:
    """Per (cell, draw) bracketing summary, with the hidden-benefit share of random-row Garnet draws."""
    rows = []
    for (c, dr), d in W.groupby(["cell", "draw"], sort=False):
        s, b = xv3_bracket_semantics(d), xv3_bracket_benchmark(d)
        rows.append({"cell": c, "draw": int(dr), "kind": d["v3_kind"].iloc[0],
                     "n_violating": s.get("n_violating", b.get("n_violating")),
                     "lower_violations": s.get("lower_violations"),
                     "upper_coverage_violating": s.get("upper_coverage_violating"),
                     "upper_exceedance_max": s.get("upper_exceedance_max"),
                     "bench_coverage_violating": b.get("coverage_violating")})
    out = pd.DataFrame(rows)
    hb = None
    if draw_meta is not None and len(draw_meta):
        hb = next((col for col in draw_meta.columns if str(col).endswith("rows_nonpreferred_beneficial_share")), None)
    if len(out):
        out["hidden_benefit_share"] = np.nan
        if hb is not None:
            m = draw_meta[["cell", "draw", hb]].copy()
            m["draw"] = pd.to_numeric(m["draw"]).astype(np.int64)
            out = out.drop(columns="hidden_benefit_share").merge(
                m.rename(columns={hb: "hidden_benefit_share"}), on=["cell", "draw"], how="left", validate="one_to_one")
    return out


def xv3_rank_fidelity(results: pd.DataFrame, cells: Sequence[str], n_boot: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Rank fidelity per draw (xs_rank_fidelity_draws) and per cell x completion, plus tau(mean) - tau(terminate)."""
    cols = ["cell", "family", "completion", "ranked_by", "evaluated_on", "n_draws", "mean_policies"] + \
           [f"{m}_{s}" for m in XS_RANK_METRICS for s in ("mean", "ci_low", "ci_high", "n")]
    if results is None or not len(results):
        return pd.DataFrame(columns=XS_RANK_DRAW_COLUMNS), pd.DataFrame(columns=cols)
    norm, _ = xs_normalized_scores(xs_prepare_results(results))
    draws = xs_sort_cells(xs_rank_fidelity_draws(norm), list(cells), ["draw"])
    rows = []
    for (cell, completion), g in draws.groupby(["cell", "completion"], sort=False):
        r = {"cell": cell, "family": g["family"].iloc[0], "completion": completion, "ranked_by": g["ranked_by"].iloc[0],
             "evaluated_on": g["evaluated_on"].iloc[0], "n_draws": int(len(g)), "mean_policies": float(g["n_policies"].mean())}
        for m in XS_RANK_METRICS:
            v = g[m].to_numpy(dtype=np.float64)
            v = v[np.isfinite(v)]
            point, reps = xs_stratified_bootstrap({cell: v[:, None]} if len(v) else {}, xs_stat_mean(),
                                                  ("v3rank", completion, m), n_boot)
            r[f"{m}_mean"] = point
            r[f"{m}_ci_low"], r[f"{m}_ci_high"] = xs_ci(reps)
            r[f"{m}_n"] = int(len(v))
        rows.append(r)
    for cell, g in draws.groupby("cell", sort=False):
        t = g.pivot_table(index="draw", columns="completion", values="kendall_tau_b", aggfunc="first")
        if not {"mean", "terminate"} <= set(t.columns):
            continue
        dlt = (t["mean"] - t["terminate"]).dropna().to_numpy(dtype=np.float64)
        point, reps = xs_stratified_bootstrap({cell: dlt[:, None]} if len(dlt) else {}, xs_stat_mean(),
                                              ("v3rank", "tau difference"), n_boot)
        lo, hi = xs_ci(reps)
        rows.append({"cell": cell, "family": g["family"].iloc[0], "completion": "mean - terminate",
                     "ranked_by": "bench_mean | bench_terminate", "evaluated_on": "true", "n_draws": int(len(dlt)),
                     "mean_policies": float(g["n_policies"].mean()), "kendall_tau_b_mean": point,
                     "kendall_tau_b_ci_low": lo, "kendall_tau_b_ci_high": hi, "kendall_tau_b_n": int(len(dlt))})
    return draws, pd.DataFrame(rows, columns=cols)


def xv3_boot_intervals(boot: pd.DataFrame, q: Tuple[float, float]) -> pd.DataFrame:
    """Per (cell, draw, policy): truth, point interval and bootstrap interval [q_lo(J_term^b), q_hi(J_mean^b)]."""
    cols = ["cell", "family", "draw", "policy", "true", "point_lo", "point_up", "p_bench", "viol", "boot_lo",
            "boot_up", "boot_lo_min", "boot_up_max", "n_ok_terminate", "n_ok_mean", "point_cover", "boot_cover",
            "boot_lo_ok", "boot_up_ok", "point_width", "boot_width"]
    if boot is None or not len(boot):
        return pd.DataFrame(columns=cols)
    k = ["cell", "draw", "policy"]
    b = boot.copy()
    b["draw"] = pd.to_numeric(b["draw"]).astype(np.int64)
    b["replicate"] = pd.to_numeric(b["replicate"]).astype(np.int64)
    pt = b[b["replicate"] == XV3_BOOT_POINT].set_index(k + ["target"])
    J0 = pt["return"].unstack("target")
    out = pd.DataFrame({"true": J0["true"], "point_lo": J0["bench_terminate"], "point_up": J0["bench_mean"],
                        "p_bench": pt["prob_any_violation"].unstack("target")["bench_mean"]})
    out = out.join(b.drop_duplicates(k).set_index(k)[["family"]])
    bs = b[(b["replicate"] >= 0) & (b["status"] == "ok")]
    g = bs.groupby(k + ["target"])["return"]
    tg = ["bench_mean", "bench_terminate"]
    qlo = g.quantile(float(q[0])).unstack("target").reindex(columns=tg)
    qhi = g.quantile(float(q[1])).unstack("target").reindex(columns=tg)
    mn, mx = g.min().unstack("target").reindex(columns=tg), g.max().unstack("target").reindex(columns=tg)
    n = g.size().unstack("target").reindex(columns=tg)
    out["boot_lo"] = qlo.reindex(out.index)["bench_terminate"]
    out["boot_up"] = qhi.reindex(out.index)["bench_mean"]
    out["boot_lo_min"] = mn.reindex(out.index)["bench_terminate"]
    out["boot_up_max"] = mx.reindex(out.index)["bench_mean"]
    out["n_ok_terminate"] = n.reindex(out.index)["bench_terminate"].fillna(0).astype(int)
    out["n_ok_mean"] = n.reindex(out.index)["bench_mean"].fillna(0).astype(int)
    out["viol"] = out["p_bench"] > XV3_BR_PTOL
    out["point_cover"] = (out["true"] >= out["point_lo"] - XV3_TOL) & (out["true"] <= out["point_up"] + XV3_TOL)
    out["boot_lo_ok"] = out["true"] >= out["boot_lo"] - XV3_TOL
    out["boot_up_ok"] = out["true"] <= out["boot_up"] + XV3_TOL
    out["boot_cover"] = out["boot_lo_ok"] & out["boot_up_ok"]
    out["point_width"] = out["point_up"] - out["point_lo"]
    out["boot_width"] = out["boot_up"] - out["boot_lo"]
    return out.reset_index()[cols]


def xv3_boot_coverage(iv: pd.DataFrame, cells: Sequence[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Coverage and certified comparisons of point and bootstrap intervals per cell and pooled; per-draw counts."""
    if iv.empty:
        return pd.DataFrame(), pd.DataFrame()
    cc = []
    for (c, dr), d in iv.groupby(["cell", "draw"], sort=False):
        for kind, (lc, uc) in (("point", ("point_lo", "point_up")), ("bootstrap", ("boot_lo", "boot_up"))):
            dd = d[d[[lc, uc, "true"]].notna().all(axis=1)]
            cc.append({"cell": c, "draw": int(dr), "family": d["family"].iloc[0], "targets": kind,
                       "policies_without_interval": int(len(d) - len(dd)),
                       **xv3_pairs(dd[lc].to_numpy(float), dd[uc].to_numpy(float), dd["true"].to_numpy(float))})
    CC = pd.DataFrame(cc)
    fams = sorted(set(iv["family"]))
    units = [(c, [c]) for c in cells if c in set(iv["cell"])]
    units += [(f"pooled:{f}", sorted(set(iv.loc[iv["family"] == f, "cell"]))) for f in fams]
    if len(fams) > 1:
        units.append(("pooled:all", sorted(set(iv["cell"]))))
    rows = []
    for name, cs in units:
        d = iv[iv["cell"].isin(cs)]
        v = d[d["viol"]]
        for kind, cov, w, lo_ok, up_ok in (("point", "point_cover", "point_width", None, None),
                                           ("bootstrap", "boot_cover", "boot_width", "boot_lo_ok", "boot_up_ok")):
            defined = d[[w]].notna().all(axis=1)
            dv = v[v[w].notna()]
            r = {"cell": name, "n_cells": len(cs), "interval": kind, "n_rows": int(len(d)),
                 "n_undefined": int((~defined).sum()), "n_violating": int(len(dv)),
                 "coverage_all": _xv3_rate(d.loc[defined, cov].sum(), int(defined.sum())),
                 "coverage_violating": _xv3_rate(dv[cov].sum(), len(dv)),
                 "mean_width_violating": float(dv[w].mean()) if len(dv) else np.nan}
            if lo_ok:
                r["lower_holds_violating"] = _xv3_rate(dv[lo_ok].sum(), len(dv))
                r["upper_holds_violating"] = _xv3_rate(dv[up_ok].sum(), len(dv))
            else:
                r["lower_holds_violating"] = _xv3_rate((dv["true"] >= dv["point_lo"] - XV3_TOL).sum(), len(dv))
                r["upper_holds_violating"] = _xv3_rate((dv["true"] <= dv["point_up"] + XV3_TOL).sum(), len(dv))
            r.update({f"cc_{k_}": v_ for k_, v_ in
                      xv3_cc_summary(CC[CC["cell"].isin(cs) & (CC["targets"] == kind)]).items()})
            rows.append(r)
    return pd.DataFrame(rows), CC


def _xv3_legend_below(ax, ncol: int = 1) -> None:
    """Legend under the axes (below the x label), so it never covers data."""
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(0.0, -0.2), ncol=ncol)


def _xv3_col(t: pd.DataFrame, name: str) -> np.ndarray:
    """Numeric column of `t` as floats (NaN when the column is absent, e.g. no cell with true-rows targets)."""
    return (pd.to_numeric(t[name], errors="coerce").to_numpy(dtype=float) if name in t.columns
            else np.full(len(t), np.nan))


def xs_fig_v3_bracketing(table: pd.DataFrame, draws: pd.DataFrame, cells: Sequence[str], fig_dir: Path) -> str:
    t = table[table["cell"].isin(cells)] if len(table) else table
    if t.empty:
        return "not produced: no II.8 cell results"
    t = t.set_index("cell").loc[[c for c in cells if c in set(t["cell"])]]
    rand = draws[draws["kind"] == "garnet-random"] if len(draws) else draws
    has_b = len(rand) and rand["hidden_benefit_share"].notna().any() and rand["upper_coverage_violating"].notna().any()
    fig, axes = plt.subplots(1, 2 if has_b else 1, figsize=(11, 3.6) if has_b else (6.5, 3.6), squeeze=False)
    ax = axes[0, 0]
    y = np.arange(len(t))
    h = 0.38
    sem = 100 * _xv3_col(t, "sem_upper_coverage_violating")
    ben = 100 * _xv3_col(t, "bench_coverage_violating")
    ax.barh(y - 0.2, np.nan_to_num(sem), height=h, color=XS_COLORS[0], label="upper bound on the true rows")
    ax.barh(y + 0.2, np.nan_to_num(ben), height=h, color=XS_COLORS[1], label="benchmark interval [J_term, J_mean]")
    for yi, s_ in zip(y, sem):
        if not np.isfinite(s_):
            ax.text(1, yi - 0.2, "n/a (aggregated states)", va="center", fontsize=7, color=XS_MUTED)
    ax.set_yticks(y)
    ax.set_yticklabels([xs_cell_label(c) for c in t.index], fontsize=7)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("coverage of J_true among violating policies (%)")
    lv = _xv3_col(t, "sem_lower_violations")
    ax.set_title(f"(a) II.8 cells; lower-bound violations on the true rows: {int(np.nansum(lv))}", loc="left",
                 color=XS_INK, fontsize=9)
    _xv3_legend_below(ax, ncol=2)
    if has_b:
        ax = axes[0, 1]
        for i, (c, g) in enumerate(rand.groupby("cell", sort=False)):
            ax.scatter(g["hidden_benefit_share"], 100 * g["upper_coverage_violating"], s=30, color=XS_COLORS[i],
                       edgecolor="white", linewidth=0.8, label=xs_cell_label(c), zorder=3)
        ax.set_xlabel("hidden-benefit share of the draw")
        ax.set_ylabel("upper-bound coverage (%)")
        ax.set_title("(b) random-row Garnet draws (violating policies)", loc="left", color=XS_INK, fontsize=9)
        _xv3_legend_below(ax, ncol=2)
    return _xs_save(fig, fig_dir, "fig_v3_bracketing")


def xs_fig_v3_bootstrap(cov: pd.DataFrame, cells: Sequence[str], fig_dir: Path) -> str:
    c = cov[cov["cell"].isin(cells)] if len(cov) else cov
    if c.empty:
        return "not produced: no bootstrap results"
    order = [x for x in cells if x in set(c["cell"])]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
    y = np.arange(len(order))
    h = 0.38
    for ax, col, xlab in zip(axes, ("coverage_violating", "cc_certified_share", "cc_certified_error_rate"),
                             ("coverage of J_true, violating policies (%)", "certified pairs (%)",
                              "certified pairs that are wrong (%)")):
        for i, kind in enumerate(("point", "bootstrap")):
            v = c[c["interval"] == kind].set_index("cell").reindex(order)[col]
            ax.barh(y + (i - 0.5) * 0.4, 100 * np.nan_to_num(pd.to_numeric(v, errors="coerce").to_numpy(float)),
                    height=h, color=XS_COLORS[i], label=f"{kind} interval")
        ax.set_xlabel(xlab)
    axes[0].set_yticks(y)
    axes[0].set_yticklabels([xs_cell_label(x) for x in order], fontsize=7)
    axes[0].invert_yaxis()
    axes[0].set_xlim(0, 100)
    axes[1].set_xlim(0, 100)
    _xv3_legend_below(axes[0], ncol=2)
    axes[0].set_title("(a) coverage", loc="left", color=XS_INK, fontsize=9)
    axes[1].set_title("(b) certified share", loc="left", color=XS_INK, fontsize=9)
    axes[2].set_title("(c) certification error", loc="left", color=XS_INK, fontsize=9)
    return _xs_save(fig, fig_dir, "fig_v3_bootstrap")


def xv3_stats_run(results: pd.DataFrame, draw_meta: Optional[pd.DataFrame], cells_spec: Sequence[dict],
                  boot: pd.DataFrame, boot_cells: Sequence[dict], table_dir: Path, fig_dir: Path,
                  n_boot: int) -> Tuple[Dict[str, Any], Dict[str, bool]]:
    """Section II.8i end to end: tables, figures and checks. Returns (XV3_STATS, done flags)."""
    t0 = time.perf_counter()
    table_dir, fig_dir = Path(table_dir), Path(fig_dir)
    kind_of = {str(s["cell"]): str(s["v3_kind"]) for s in cells_spec}
    cells = [str(s["cell"]) for s in cells_spec]
    outputs: List[str] = []

    def write_csv(df: pd.DataFrame, name: str) -> None:
        df.to_csv(table_dir / name, index=False)
        outputs.append(f"xenv/{name}")

    out: Dict[str, Any] = {}
    checks: Dict[str, Any] = {}
    done = {"cells": False, "bootstrap": False}
    figures: Dict[str, str] = {}
    if results is not None and len(results):
        W, checks["bracket_frame"] = xv3_bracket_frame(results, kind_of)
        out["bracketing"] = xv3_bracketing_table(W, cells, kind_of)
        out["bracketing_draws"] = xv3_bracketing_draws(W, draw_meta)
        CC = xv3_certified_draws(W)
        cc_rows = []
        for tf in ("true_rows", "bench"):
            units = [(c, [c]) for c in cells] + list(xv3_pools(cells, kind_of).items())
            for name, cs in units:
                d = CC[CC["cell"].isin(cs) & (CC["targets"] == tf)] if len(CC) else CC
                if len(d):
                    cc_rows.append({"cell": name, "targets": tf, **xv3_cc_summary(d)})
        out["certified"], out["certified_draws"] = pd.DataFrame(cc_rows), CC
        out["rank_fidelity_draws"], out["rank_fidelity"] = xv3_rank_fidelity(results, cells, n_boot)
        rb = out["bracketing_draws"]
        rand = rb[(rb["kind"] == "garnet-random") & rb["hidden_benefit_share"].notna()
                  & rb["upper_coverage_violating"].notna()] if len(rb) else rb
        checks["hidden_benefit_spearman"] = {
            c: {"draws": int(len(g)), "spearman_rho": xs_spearman(g["hidden_benefit_share"], g["upper_coverage_violating"])
                if len(g) >= 3 else None} for c, g in rand.groupby("cell")} if len(rand) else {}
        for name in ("bracketing", "bracketing_draws", "certified", "certified_draws", "rank_fidelity",
                     "rank_fidelity_draws"):
            write_csv(out[name], f"v3_{name}.csv")
        figures["fig_v3_bracketing"] = xs_fig_v3_bracketing(out["bracketing"], rb, cells, fig_dir)
        done["cells"] = True
        del W
    else:
        figures["fig_v3_bracketing"] = "not produced: XRESULTS_V3 is empty (II.8 cell switches off)"
    if boot is not None and len(boot):
        bcells = [str(s["cell"]) for s in boot_cells]
        iv = xv3_boot_intervals(boot, tuple(XV3BOOT.quantiles))
        cov, bcc = xv3_boot_coverage(iv, bcells)
        out["bootstrap_intervals"], out["bootstrap_coverage"], out["bootstrap_certified_draws"] = iv, cov, bcc
        write_csv(iv, "v3_bootstrap_intervals.csv")
        write_csv(cov, "v3_bootstrap_coverage.csv")
        checks["bootstrap"] = {"policy_rows": int(len(iv)), "undefined_bootstrap_intervals": int(iv["boot_width"].isna().sum()),
                               "negative_point_width": int((iv["point_width"] < -XV3_TOL).sum()),
                               "quantiles": list(XV3BOOT.quantiles), "replicates": int(XV3BOOT.replicates)}
        figures["fig_v3_bootstrap"] = xs_fig_v3_bootstrap(cov, bcells, fig_dir)
        done["bootstrap"] = True
    else:
        figures["fig_v3_bootstrap"] = "not produced: no bootstrap rows (xenv_v3_bootstrap off or no cell enabled)"
    outputs += [f"figures/xenv/{k}.{ext}" for k, v in figures.items() if v == "written" for ext in ("png", "pdf")]
    bf = checks.get("bracket_frame", {})
    theorem_counts = {k: bf.get(k, 0) for k in ("values_outside_value_range", "lower_bound_violations_true_rows",
                                                 "negative_width_true_rows", "negative_width_bench",
                                                 "nonzero_width_admissible", "width_above_bound")}
    theorem_counts["negative_point_width_bootstrap"] = checks.get("bootstrap", {}).get("negative_point_width", 0)
    checks.update({"theorem_counts": theorem_counts, "passes": all(v == 0 for v in theorem_counts.values()),
                   "done": done, "figures": figures, "outputs": outputs, "n_boot": int(n_boot),
                   "tolerance": XV3_TOL, "violating_threshold": XV3_BR_PTOL, "value_ranges": XV3_VRANGE,
                   "elapsed_s": time.perf_counter() - t0})
    save_json(checks, table_dir / "v3_stats_checks.json")
    outputs.append("xenv/v3_stats_checks.json")
    out.update({"checks": checks, "figures": figures, "outputs": outputs})
    return out, done


XV3_STATS, XV3_STATS_DONE = xv3_stats_run(XRESULTS_V3, XDRAW_META_V3, X_V3_CELLS, XV3_BOOT_RESULTS, XV3_BOOT_CELLS,
                                          XENV_TABLE_DIR, XENV_FIG_DIR, int(XCFG.bootstrap))
print("II.8i theorem checks:", XV3_STATS["checks"]["theorem_counts"], "| passes:", XV3_STATS["checks"]["passes"])
if "bracketing" in XV3_STATS:
    display(XV3_STATS["bracketing"][["cell", "sem_n_violating", "sem_lower_violations", "sem_upper_coverage_violating",
                                     "bench_n_violating", "bench_coverage_violating"]].round(4))
if "certified" in XV3_STATS and len(XV3_STATS["certified"]):
    _cols = [c for c in ("cell", "targets", "draws", "certified_share", "certified_error_rate", "point_mean_error_rate",
                         "point_terminate_error_rate") if c in XV3_STATS["certified"].columns]
    display(XV3_STATS["certified"][_cols].round(4))
if "bootstrap_coverage" in XV3_STATS and len(XV3_STATS["bootstrap_coverage"]):
    display(XV3_STATS["bootstrap_coverage"][["cell", "interval", "n_violating", "coverage_violating",
                                             "mean_width_violating", "cc_certified_share",
                                             "cc_certified_error_rate"]].round(4))
print("Figures:", XV3_STATS["figures"], "| done:", XV3_STATS_DONE)
if not XV3_STATS["checks"]["passes"]:
    raise AssertionError(f"II.8i theorem checks failed: {XV3_STATS['checks']['theorem_counts']} (xenv/v3_stats_checks.json)")

## II.9 Further experiments: estimation-aware certificates, sensitivity and a clean rerun

This section adds further robustness experiments. Sections II.1–II.8 are not changed, so every cached draw keeps
its identity. The new experiments use the exact simulators and the stored designs of Sections II.3–II.8.

| Switch | Subsection | What it adds |
|---|---|---|
| `xenv_v4_revision_robust` | II.9b | certificates without NHB, pair types, distinct policies, abstention at matched coverage and draw-level counts for every Part II cell, the additional cells included; the coverage of the II.8g bootstrap intervals over **all** default policies |
| `xenv_v4_confidence` | II.9c | confidence-set certificates: L1 confidence sets for the true rows and d0 built from the raw hospital counts, robust bounds on J_true for every policy, certificates valid on one event for all policies at once |
| `xenv_v4_bootstrap` | II.9d | bootstrap with B = 200 that re-estimates the whole construction (mask, decision states, dead-state removal, d0) in every replicate, beside the fixed-mask bootstrap of II.8g; coverage over all default policies |
| `xenv_v4_sensitivity` | II.9e | new cells: Gumbel timeout hazard η ∈ {0.02, 0.10}; the alternative dead-state rule (`argmax`) on the hospital data of the original cells; softer physicians (wider support); other state abstractions (glucose hidden, k-means clusters) |
| `xenv_v4_lift` | II.9f | the lift of benchmark policies to the true state space: physician and uniform lifts beside the default, and the range of J_true over all lifts |
| `xenv_v4_clean_check` | II.9g | a clean rerun of the Part II draws (and the II.8 cells) into an empty cache, compared row by row with the cached results |

II.9h collects the outputs, writes `xenv/v4_overview.json` and the figures, and states what failed.

**Cached draws stay valid.** New behaviour lives in new functions. Each subsection has its own code hash, which folds
in `XENV_CODE_HASH` (and `XENV_V3_CODE_HASH` where II.8 functions are used), and its own cache keys (`v4cs/`,
`v4boot/`, `v4sens/`, `v4lift/`). The clean check writes to a separate, empty cache directory.

**Shared helpers (this cell).** `XV4Config` holds every setting; `x_v4_cells` resolves cell names against `X_CELLS`
and `X_V3_CELLS` (a configured cell whose switch is off is skipped and listed); `x_v4_instance` builds the II.6 or
II.8 instance of a draw; `x_v4_default_records` regenerates its default policies (Part II learners are deterministic
functions of their data, as II.8g checked); `x_v4_rebuild_benchmark` regenerates the hospital episodes and the
benchmark and asserts that the benchmark equals the instance's; `x_v4_pair_claims` scores the certificates of one
policy set (with NHB, without NHB, NHB-only, testable claims, pair types and matched-coverage margins);
`x_v4_run_draws` runs a per-draw function over cells with progress and ETA.

In [ ]:
# ============================================================
# II.9a Second-revision experiments: configuration and shared helpers
# ============================================================
EXECLOG.mark("II.9 Second-revision experiments: setup")

XV4_SWITCHES = ("xenv_v4_revision_robust", "xenv_v4_confidence", "xenv_v4_bootstrap", "xenv_v4_sensitivity",
                "xenv_v4_lift", "xenv_v4_clean_check")
assert all(k in EXPERIMENTS for k in XV4_SWITCHES), [k for k in XV4_SWITCHES if k not in EXPERIMENTS]
XV4_ANY_ON = any(bool(EXPERIMENTS[k]) for k in XV4_SWITCHES)
XV4_TOL = 1e-9                 # value tolerance of comparisons (as XV3_TOL)
XV4_PTOL = 1e-12               # a policy "can leave the support" iff P(any unsupported decision) > XV4_PTOL

XV4_GUMBEL_BEST = ("gumbel|best|n2000|tau5|eps0.05", "gumbel|best|n2000|tau20|eps0.05", "gumbel|best|n10000|tau5|eps0.05",
                   "gumbel|best|n10000|tau20|eps0.05", "gumbel|best|n10000|tau20|eps0.2",
                   "gumbel|best|n10000|tau20|eps0.05|phys-withhold-vaso")
XV4_GARNET_ALL = tuple(f"garnet|h{h}|eps{e}" for h in (-0.5, 0.0, 0.5, 1.0) for e in (0.05, 0.3))


@dataclass
class XV4Config:
    # II.9c confidence-set certificates
    cs_cells: Tuple[str, ...] = ("gumbel|best|n2000|tau5|eps0.05", "gumbel|best|n10000|tau20|eps0.05",
                                 "gumbel|best|n10000|tau20|eps0.05|phys-withhold-vaso", "garnet|h-0.5|eps0.05",
                                 "garnet|h0.0|eps0.05", "garnet|h0.5|eps0.05", "garnet|h1.0|eps0.05")
    cs_delta: float = 0.05                         # P(true rows or d0 outside the set) <= delta (union bound)
    cs_scales: Tuple[float, ...] = (1.0, 0.3, 0.1)  # radius multipliers; 1.0 is the valid set, smaller ones heuristic
    cs_tol: float = 1e-8                           # stop the monotone robust iteration at this sup-norm change
    cs_max_iter: int = 5000                        # every iterate is a valid bound, so a cap only loosens it
    # II.9d bootstrap with the construction re-estimated in every replicate
    boot_cells: Tuple[str, ...] = ("gumbel|best|n10000|tau20|eps0.05", "gumbel|best|n2000|tau5|eps0.05",
                                   "garnet|h0.0|eps0.05", "garnet|h1.0|eps0.05")
    boot_replicates: int = 200
    boot_quantiles: Tuple[float, float] = (0.025, 0.975)
    boot_variants: Tuple[str, ...] = ("fixed", "reestimated")
    # II.9e sensitivity cells: (base cell, timeout hazard) and base cells rebuilt with the 'argmax' dead-state rule
    timeout_cells: Tuple[Tuple[str, float], ...] = (("gumbel|best|n10000|tau20|eps0.05", 0.02),
                                                    ("gumbel|best|n10000|tau20|eps0.05", 0.10),
                                                    ("gumbel|best|n2000|tau5|eps0.05", 0.02),
                                                    ("gumbel|best|n2000|tau5|eps0.05", 0.10))
    deadrule_cells: Tuple[str, ...] = XV4_GUMBEL_BEST + ("garnet|h1.0|eps0.05", "garnet|h1.0|eps0.3")
    # II.9e wider physician support: the base cell with a softer physician (eps), streams named by the new cell
    support_base: str = "gumbel|best|n10000|tau20|eps0.05"
    support_eps: Tuple[float, ...] = (0.3, 0.5)
    # II.9e other state abstractions of the Gumbel benchmark (the II.8 aggregated design with another phi):
    # 'glucose' hides glucose only (diabetes visible); 'kmeansK' clusters the observation features into K cells
    aggregation_base: Tuple[int, int, float] = (10_000, 20, 0.05)       # (hospital episodes, tau, physician eps)
    aggregations: Tuple[str, ...] = ("glucose", "kmeans50", "kmeans150")
    # II.9b hidden-benefit sensitivity: fractions kappa of the largest possible excess over J_W
    kappa_grid: Tuple[float, ...] = (0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0)
    # II.9e lift sensitivity
    lift_cells: Tuple[str, ...] = XV4_GUMBEL_BEST + XV4_GARNET_ALL
    lift_kinds: Tuple[str, ...] = ("fallback", "physician", "uniform")
    lift_bounds: bool = False                      # range of J_true over all lifts by optimal control outside V
    #                                                (about 2 min per Gumbel draw: enable for a subset only)
    # II.9f clean rerun into an empty cache
    clean_scope: str = "all"                       # "all" draws, or "sample": the first clean_sample_draws of each cell
    clean_sample_draws: int = 3
    clean_include_v3: bool = True                  # also the II.8 cells (run_v3_cell_draw)
    clean_include_v3_boot: bool = False            # II.8g already rebuilt its 210 draws without the cache
    clean_cache_dir: str = "/content/xenv_clean_cache"   # local disk, never the Drive run root
    clean_tol: float = 1e-12                       # |diff| <= clean_tol * max(1, |cached value|)
    # all subsections: at most this many draws per cell (None = every configured draw of the cell)
    max_draws: Optional[int] = None


def apply_xv4_run_mode(c: XV4Config, mode: str, profile: str) -> XV4Config:
    if profile == "lite":
        c.max_draws = 10
        c.cs_scales = (1.0, 0.3)
        c.clean_scope = "sample"
    if mode == "smoke":
        c.max_draws = 2
        c.cs_scales = (1.0, 0.3)
        c.cs_max_iter = 500
        c.boot_replicates = 5
        c.timeout_cells = c.timeout_cells[:1]
        c.support_eps = c.support_eps[:1]
        c.aggregations = c.aggregations[:1]
        c.lift_kinds = ("fallback", "physician", "uniform")
        c.lift_bounds = True                       # exercised in smoke; off by default in paper mode (cost)
        c.clean_scope, c.clean_sample_draws = "sample", 1
        c.clean_include_v3_boot = True             # seconds in smoke mode; exercises the II.8g rerun path
    return c


XV4CFG = apply_xv4_run_mode(XV4Config(), RUN_MODE, COMPUTE_PROFILE)
save_json(asdict(XV4CFG), XENV_TABLE_DIR / "v4_config.json")


# ---------- cells ----------
def x_v4_all_specs() -> Dict[str, dict]:
    """Every enabled Part II cell spec by name: II.6 cells (X_CELLS) and II.8 cells (X_V3_CELLS, with v3_kind)."""
    out = {str(s["cell"]): s for s in X_CELLS}
    for s in globals().get("X_V3_CELLS", []) or []:
        assert str(s["cell"]) not in out, s["cell"]
        out[str(s["cell"])] = s
    return out


def x_v4_cells(names: Sequence[str], families: Sequence[str] = ("gumbel", "garnet"),
               allow_aggregated: bool = False) -> Tuple[List[dict], List[str]]:
    """(specs, skipped) for configured cell names. In smoke mode, names that do not exist (the smoke designs use
    other hospital sizes) are replaced by the enabled cells of the same family and construction."""
    specs = x_v4_all_specs()
    chosen, skipped = [], []
    for name in names:
        s = specs.get(str(name))
        if s is None and RUN_MODE == "smoke":
            fam, cons = str(name).split("|")[0], str(name).split("|")[1]
            alt = [v for k, v in specs.items() if v["family"] == fam and k.split("|")[1] == cons
                   and v not in chosen and "v3_kind" not in v]
            s = alt[0] if alt else None
        if s is None or s["family"] not in families or (s.get("v3_kind") == "aggregated" and not allow_aggregated):
            skipped.append(str(name))
            continue
        if s not in chosen:
            chosen.append(s)
    return chosen, skipped


def x_v4_draws(spec: dict) -> int:
    n = int(spec["draws"])
    return n if XV4CFG.max_draws is None else min(n, int(XV4CFG.max_draws))


# ---------- instances, policies, hospital data and benchmarks ----------
def x_v4_instance(spec: dict, draw: int) -> CellInstance:
    """The II.6 instance (build_cell_instance) or the II.8 instance (build_v3_instance) of a draw."""
    return build_v3_instance(spec, draw) if "v3_kind" in spec else build_cell_instance(spec, draw)


def x_v4_draw_identity(spec: dict, inst: CellInstance) -> dict:
    """The identity of the regenerated draw: x_v3_draw_identity for II.8 cells, x_draw_identity otherwise."""
    return x_v3_draw_identity(spec, inst) if "v3_kind" in spec else x_draw_identity(spec, inst)


def x_v4_default_records(inst: CellInstance) -> Tuple[List[dict], List[bytes]]:
    """Default records of a draw (II.6 learners on the same training split and key, + behaviour and projections)."""
    learn = inst.learn_env
    raw = run_xmethods(LearnerView.from_env(learn), inst.train, f"{inst.cell}|draw{int(inst.draw)}")
    records = [r for r in x_collect_records(raw, inst.behavior, learn) if r["is_default"]]
    del raw
    return records, [_x_pi_key(r["pi"]) for r in records]


def x_v4_true_system(spec: dict, hazard: Optional[float] = None) -> Tuple[FiniteEnv, np.ndarray]:
    """True system and true physician of a Gumbel spec at timeout hazard `hazard` (default: the spec's).
    Garnet systems depend on the draw; use x_v4_true_system_draw."""
    assert spec["family"] == "gumbel", spec["family"]
    h = float(spec["timeout_hazard"] if hazard is None else hazard)
    env = _x_v3_gumbel_env(spec["variant"], h)
    beh = _x_v3_gumbel_physician(spec["variant"], h, spec["eps"], spec.get("physician", "optimal"))
    return env, beh


def x_v4_true_system_draw(spec: dict, draw: int, hazard: Optional[float] = None) -> Tuple[FiniteEnv, np.ndarray]:
    """True system and true behaviour policy of one draw of a Gumbel or Garnet spec (aggregated cells excluded)."""
    if spec["family"] == "gumbel":
        assert spec.get("v3_kind") != "aggregated", "aggregated cells have no true-state-space construction"
        return x_v4_true_system(spec, hazard)
    assert spec["family"] == "garnet" and hazard is None, (spec["family"], hazard)
    if spec.get("v3_kind") == "garnet-random":
        env, beh, _ = _x_v3_garnet_env(spec, draw)
        return env, beh
    return _x_garnet_env(spec, draw)


def x_v4_hospital(spec: dict, draw: int, true_env: FiniteEnv, true_behavior: np.ndarray,
                  stream_cell: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Hospital episodes of a draw: the stream and call of construct_benchmark (stream named by `stream_cell`,
    default the spec's own cell)."""
    rng_h = np_rng("xenv", "hospital", str(stream_cell or spec["cell"]), int(draw))
    return x_sample_dataset(true_env, true_behavior, int(spec["n_hosp"]), XCFG.max_steps, rng_h)


def x_v4_rebuild_benchmark(spec: dict, draw: int, inst: CellInstance
                           ) -> Tuple[FiniteEnv, np.ndarray, Dict[str, np.ndarray], Dict[str, Any]]:
    """(true_env, true_behavior, hospital episodes, benchmark dict) of an estimated ('best') draw; the rebuilt
    benchmark must equal the instance's learner environment (asserted)."""
    assert spec.get("construction", "best") == "best" and spec.get("v3_kind") != "aggregated", spec["cell"]
    true_env, beh = x_v4_true_system_draw(spec, draw)
    hosp = x_v4_hospital(spec, draw, true_env, beh)
    B = x_benchmark_from_data(true_env, hosp, spec["tau"], inst.learn_env.name)
    assert obj_hash(B["bench"].identity()) == obj_hash(inst.learn_env.identity()), \
        f"{spec['cell']} draw {draw}: regenerated hospital data do not rebuild the benchmark"
    return true_env, beh, hosp, B


def x_v4_eval(env: FiniteEnv, pi: np.ndarray, semantics: str) -> Dict[str, Any]:
    """Exact evaluation that never raises on a model-specific failure (ok / improper / near-singular)."""
    return _x_v3_eval(env, pi, semantics)


# ---------- certificates of one policy set ----------
def _x_v4_gt(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return a[:, None] > b[None, :] + XV4_TOL


def x_v4_margin_claims(v: np.ndarray, k: int) -> np.ndarray:
    """Ordered claims i > j for the k unordered pairs with the largest |v_i - v_j| (gaps <= XV4_TOL never claimed;
    ties at the threshold broken by index order)."""
    n = len(v)
    iu = np.triu_indices(n, 1)
    gap = np.abs(v[iu[0]] - v[iu[1]])
    order = np.argsort(-gap, kind="stable")[:max(int(k), 0)]
    order = order[gap[order] > XV4_TOL]
    C = np.zeros((n, n), dtype=bool)
    i, j = iu[0][order], iu[1][order]
    hi = v[i] > v[j]
    C[np.where(hi, i, j), np.where(hi, j, i)] = True
    return C


def x_v4_pair_claims(lo: np.ndarray, up: np.ndarray, p: np.ndarray, v_range: float,
                     jt: Optional[np.ndarray] = None) -> List[dict]:
    """Claims of one policy set, by pair type. lo / up: J_term / J_W of each policy on the same rows; p: P(any
    unsupported decision) on those rows; v_range = max_T R - v_min (= v_max - v_min in every Part II family).

    Kinds: cert_nhb (lo_i > up_j), cert_free (lo_i > lo_j + p_j v_range, Corollary 2 without NHB), cert_nhb_only,
    cert_testable (certified with a losing policy that can leave the support), point_mean / point_term (strict
    single-rule claims) and margin_mean / margin_term / margin_mid (the k largest gaps, k = number of NHB
    certificates; margin_mean_free with k = number of NHB-free certificates). Pair types: AA (neither policy can leave
    the support), AU (one), UU (both), all. With `jt` (true values) every kind also counts its errors (truth reverses
    or ties). Returns one dict per type with '<kind>_claims' and '<kind>_errors'."""
    lo, up, p = (np.asarray(x, dtype=float) for x in (lo, up, p))
    n = len(lo)
    off = ~np.eye(n, dtype=bool)
    u = p > XV4_PTOL
    claims = {"cert_nhb": _x_v4_gt(lo, up), "cert_free": _x_v4_gt(lo, lo + p * float(v_range)),
              "point_mean": _x_v4_gt(up, up), "point_term": _x_v4_gt(lo, lo)}
    claims["cert_testable"] = claims["cert_nhb"] & u[None, :]
    # NHB-free implies NHB because J_W <= J_term + p v_range (Theorem 1(iii)); on rows that meet that bound only up to
    # the solver tolerance (bootstrap replicates) the implication can fail inside a 1e-9 window, so the subset
    # property is enforced and only a violation beyond 10 XV4_TOL is an error.
    _viol = claims["cert_free"] & ~claims["cert_nhb"] & off
    assert not (_viol & (up > lo + p * float(v_range) + 10 * XV4_TOL)[None, :]).any(), \
        "an NHB-free certificate must be an NHB one (Theorem 1(iii))"
    claims["cert_free"] &= claims["cert_nhb"]
    claims["cert_nhb_only"] = claims["cert_nhb"] & ~claims["cert_free"]
    k_nhb, k_free = int((claims["cert_nhb"] & off).sum()), int((claims["cert_free"] & off).sum())
    claims["margin_mean"] = x_v4_margin_claims(up, k_nhb)
    claims["margin_term"] = x_v4_margin_claims(lo, k_nhb)
    claims["margin_mid"] = x_v4_margin_claims((lo + up) / 2.0, k_nhb)
    claims["margin_mean_free"] = x_v4_margin_claims(up, k_free)
    typ = np.where(u[:, None] & u[None, :], "UU", np.where(u[:, None] | u[None, :], "AU", "AA"))
    wrong = None if jt is None else ~_x_v4_gt(np.asarray(jt, dtype=float), np.asarray(jt, dtype=float))
    out = []
    for ty in ("all", "AA", "AU", "UU"):
        tm = off if ty == "all" else (off & (typ == ty))
        r = {"type": ty, "n_policies": n, "pairs": int(tm.sum() // 2)}
        for nm, C in claims.items():
            r[f"{nm}_claims"] = int((C & tm).sum())
            if wrong is not None:
                r[f"{nm}_errors"] = int((C & tm & wrong).sum())
        out.append(r)
    return out


def x_v4_distinct(values: np.ndarray) -> np.ndarray:
    """Boolean mask keeping the first of every group of rows (policies) with identical values to 1e-12."""
    v = np.asarray(values, dtype=float)
    _, first = np.unique(np.round(v / 1e-12).astype(np.int64), axis=0, return_index=True)
    keep = np.zeros(len(v), dtype=bool)
    keep[first] = True
    return keep


# ---------- per-draw driver ----------
def x_v4_run_draws(cells: Sequence[dict], fn: Callable[[dict, int], Tuple[pd.DataFrame, dict]], desc: str,
                   time_key: Callable[[dict], str] = lambda s: str(s["family"])
                   ) -> Tuple[List[pd.DataFrame], pd.DataFrame, Dict[str, Any]]:
    """Run fn(spec, draw) -> (rows, meta with 'cache' in {'loaded', 'computed'}) over every draw of every cell, with
    the progress and ETA of II.6a. Returns the frames, the flattened meta and a small summary."""
    frames: List[pd.DataFrame] = []
    meta_rows: List[dict] = []
    times: Dict[str, List[float]] = {}
    status: Counter = Counter()
    t_all = time.perf_counter()
    print(f"{desc}: {len(cells)} cells, {sum(x_v4_draws(s) for s in cells)} draws")
    for ci, spec in enumerate(cells, 1):
        t_cell, n_loaded, nd = time.perf_counter(), 0, x_v4_draws(spec)
        bar = trange(nd, desc=f"{desc} [{ci}/{len(cells)}] {spec['cell']}", leave=False, disable=not SHOW_PROGRESS)
        for draw in bar:
            t0 = time.perf_counter()
            df, meta = fn(spec, draw)
            dt = time.perf_counter() - t0
            frames.append(df)
            meta_rows.append(_x_flatten({**meta, "session_wall_s": dt}))
            status[meta["cache"]] += 1
            if meta["cache"] == "loaded":
                n_loaded += 1
            else:
                times.setdefault(time_key(spec), []).append(dt)
            if SHOW_PROGRESS:
                bar.set_postfix_str(f"{dt:.1f}s {meta['cache']}")
            elif (draw + 1) % max(1, nd // 5) == 0 or draw + 1 == nd:
                print(f"    {spec['cell']} draw {draw + 1}/{nd} ({meta['cache']}, {dt:.1f} s); cell ETA "
                      f"{_x_fmt_s(x_eta_seconds([(time_key(spec), nd - draw - 1)], times))}")
        eta = x_eta_seconds([(time_key(s), x_v4_draws(s)) for s in cells[ci:]], times)
        print(f"  [{ci}/{len(cells)}] {spec['cell']}: {nd} draws ({n_loaded} cached) in "
              f"{_x_fmt_s(time.perf_counter() - t_cell)} | total elapsed {_x_fmt_s(time.perf_counter() - t_all)} | "
              f"ETA remaining cells {_x_fmt_s(eta) if ci < len(cells) else '0 s'}")
    summary = {"cells": [s["cell"] for s in cells], "draws": int(sum(status.values())),
               "draws_computed": int(status["computed"]), "draws_loaded": int(status["loaded"]),
               "elapsed_s": time.perf_counter() - t_all,
               "draws_per_cell": {s["cell"]: x_v4_draws(s) for s in cells}}
    return frames, pd.DataFrame(meta_rows), summary


XV4_COMMON_CODE = [x_v4_all_specs, x_v4_cells, x_v4_draws, x_v4_instance, x_v4_draw_identity, x_v4_default_records,
                   x_v4_true_system, x_v4_true_system_draw, x_v4_hospital, x_v4_rebuild_benchmark, x_v4_eval,
                   _x_v4_gt, x_v4_margin_claims, x_v4_pair_claims, x_v4_distinct]
XV4_COMMON_CODE_HASH = obj_hash({"xenv_code": XENV_CODE_HASH, "v3_code": globals().get("XENV_V3_CODE_HASH"),
                                 "v4_common": code_hash(*XV4_COMMON_CODE),
                                 "constants": {"tol": XV4_TOL, "ptol": XV4_PTOL}})
print("II.9 switches:", {k: bool(EXPERIMENTS[k]) for k in XV4_SWITCHES}, "| config:", asdict(XV4CFG))
print("II.9 common code hash:", XV4_COMMON_CODE_HASH[:16])

### II.9b Robustness of certified comparisons: certificates without NHB, pair types, abstention, hidden benefit

A certificate (Corollary 2) orders two policies through their brackets. The **NHB certificate**
J_term(i) > J_W(j) assumes no hidden benefit (NHB) for the losing policy j. The **NHB-free certificate**
J_term(i) > J_term(j) + p_j (max_T R − v_min) assumes nothing about the unsupported rows (Theorem 1(iv)). This section
re-reads the stored exact results of every Part II simulator cell (II.6 and II.8, aggregated cells included) and of
ICU-Sepsis, and scores both certificates. It splits them by pair type, merges identical policies, compares them with a
single-rule abstention baseline at the same coverage, and measures how much hidden benefit would reverse each claim.
It also scores the II.8g bootstrap intervals over **all** default policies. It simulates nothing: it is the notebook
form of `paper_scripts/make_robustness.py` of the supplementary material, with the same definitions. The value tolerance is 1e-9. A policy
"can leave the support" if p > 1e-12. A claim is an error if the truth reverses it or ties.

**Inputs.** `XRESULTS` (II.6) and `XRESULTS_V3` (II.8), or `xenv/xenv_results_all.csv.gz` and
`xenv/v3_results_all.csv.gz` when the globals are empty. The bootstrap rows come from `XV3_BOOT_RESULTS`, or
`xenv/v3_bootstrap_results.csv.gz`. Only default, non-reference policies are used, as in II.8i. Every stored draw is
analysed: `XV4CFG.max_draws` limits only the subsections that simulate.

**Per (cell, draw).**
1. *Target sets.* `true_rows` = (J_term, J_W, p) = (`true_rows_terminate`, `true_rows_mean`, p on `true_rows_mean`).
   The oracle cell has no `true_rows_*` rows: its benchmark rows are the true rows, so it uses its `bench_*` targets,
   as `make_assets.py` does. `bench` = (`bench_terminate`, `bench_mean`, p on `bench_mean`). The truth is `true`.
   ICU-Sepsis has `bench` only and no truth. If some policy is not proper on a needed target, that target set of the
   draw is skipped and counted (II.8i skips such draws too).
2. *Claims.* `x_v4_pair_claims` with v_range = v_max − v_min (Gumbel 2, Garnet 1, ICU-Sepsis 1). It counts the NHB
   and NHB-free certificates, the NHB-only claims and the testable claims (losing policy with p > 0). It also counts
   the point claims of each rule and the margin rules: claim the k largest gaps of one rule, with k = the number of
   certificates (abstention at matched coverage). Pair types: AA (neither policy can leave the support), AU (one),
   UU (both). The counts are repeated after merging policies with identical values on every available target
   (`x_v4_distinct`). Within a draw, policies are ordered by label as in `make_robustness.py`; this fixes how ties of
   the margin rules are broken.
3. *Hidden-benefit sensitivity (κ).* E_j = p_j v_range − (J_W(j) − J_term(j)) ≥ 0 is the largest possible excess of
   π_j's value over J_W(j) (Theorem 1(iv), attained). The κ-certificate is J_term(i) > J_W(j) + κ E_j: κ = 0 is the
   NHB certificate and κ = 1 the NHB-free one. It is computed as J_term(i) > (1 − κ) J_W(j) + κ (J_term(j) +
   p_j v_range), so both ends reproduce the two certificates exactly. Each claim is counted for every κ in
   `XV4CFG.kappa_grid`. Every NHB-only claim also gets its breakdown fraction κ* = (J_term(i) − J_W(j)) / E_j and
   its margin J_term(i) − J_W(j). E_j > 0 for every NHB-only claim, so κ* is finite. A claim is NHB-free iff κ* > 1,
   and κ* ∈ (0, 1] for NHB-only claims, both up to the 1e-9 tolerance: a claim within 1e-9 of the NHB-free end stays
   NHB-only, so κ* can exceed 1 by at most 1e-9 / E_j (as in `make_robustness.py`).
4. *Admissible-pair gaps* (`bench`, with truth). No completion can change an NHB-certified claim between two policies
   with p = 0, so its errors are estimation and construction error. These claims are binned by the normalised gap
   (J_term(i) − J_W(j)) / v_range.

**Aggregates.** Units are every cell and the pools `gumbel_estimated` (the II.6 `best` cells), `gumbel_oracle`,
`garnet_all`, `garnet_h-0.5`, `garnet_h0.0`, `garnet_h_ge_0.5`, `icu`, and one pool per II.8 kind (`v3_physician`,
`v3_aggregated`, `v3_garnet-random`). Each unit is reported per target set, policy set (all / distinct) and pair type.
Each row gives claims, errors, shares and error rates of every kind, and the draws with at least one certificate or
one wrong certificate. Each row also gives a draw-clustered percentile interval of the `cert_nhb`, `cert_free` and
`margin_*` error rates: 5,000 resamples of the draws within each cell, stream
`("xenv", "v4", "robust-ci", unit, target set, kind)`. The same resamples are shared by every policy set and pair
type of a unit. A resample without claims counts as rate 0, as in `make_robustness.py`. The **completion-sensitive
headline** is in every row: the certified share and error among AU + UU pairs, for NHB and NHB-free.

**II.8g bootstrap rows** (B = `XV3BOOT.replicates`, proper replicates only, quantiles `XV3BOOT.quantiles`). Every
default policy gets four intervals:

* NHB: [q_lo(J_term^b), q_hi(J_mean^b)];
* NHB-free: [q_lo(J_term^b), q_hi(J_term^b + p^b v_range)];
* the min / max versions of both.

The point intervals [J_term, J_W] and [J_term, J_term + p v_range] of the original benchmark are scored beside them.
Per cell and pooled, the table reports:

* coverage of J_true over all default policies, and over p > 0 and p = 0 (p from replicate −1);
* certified share and errors, also among AU + UU pairs;
* draws with any wrong certificate, and draws in which every policy is covered;
* margin rules at the same number of claims, on the point values.

The two ends of a bootstrap interval are quantiles over different sets of proper replicates, so an interval can be
inverted (lower end above upper end). Such intervals are counted in the checks and reported, not asserted. The
number of replicates is read from the rows.

**Invariants asserted.** They are counted, the outputs are written, and then the cell raises, as II.8i does.

* Values lie in [v_min, v_max].
* 0 ≤ J_W − J_term ≤ p v_range on both target sets.
* J_term ≤ J_true ≤ J_term + p v_range on the true rows (Theorem 1(i), (iv)).
* J_W = J_term on the true rows when p = 0.
* The two rules (and the truth) share p.
* κ = 0 and κ = 1 reproduce the NHB and NHB-free counts.
* The per-draw NHB and point counts equal those of the II.8i code (`xv3_certified_draws`) on the same rows, and the
  `XV3_STATS` table when it is in memory.
* On the bootstrap rows: the per-replicate invariants of II.8g, and the NHB interval of II.8i.

The pooled counts are also compared with the published values (`facts_robust.json`). For `gumbel_estimated` these
are: true rows 152,237 NHB claims, 0 errors, 110,330 NHB-free, 41,907 NHB-only, 43,346 testable; bench 151,338
claims, 15,981 errors. This comparison is asserted only for the full paper design: `RUN_MODE == "paper"`,
`COMPUTE_PROFILE == "full"` and all 180 draws present. A mismatch there is counted like the invariants above. The
`garnet_all` values are compared and reported only. Error rates and coverage are reported, never asserted.

| Output | Content |
|---|---|
| `xenv/v4_robust_pairs.csv.gz` | per draw × target set × policy set × pair type: claims and errors of every kind |
| `xenv/v4_robust_summary.csv` | per cell and pool × target set × policy set × pair type: shares, error rates, draw-clustered intervals, draws with a (wrong) certificate, completion-sensitive headline |
| `xenv/v4_robust_gapbins.csv` | per cell and pool × gap bin: NHB-certified benchmark claims between two policies with p = 0, and their error rate |
| `xenv/v4_robust_kappa.csv` | per cell and pool × target set × κ: claims, share of pairs and of AU + UU pairs, errors; quantiles of κ* and of the margin of the NHB-only claims |
| `xenv/v4_v3boot_allpolicy.csv` | II.8g bootstrap, per cell and pooled × interval (point / bootstrap / minmax) × bound (NHB / NHB-free): coverage over all, p > 0 and p = 0 policies; certificates; draw-level counts; margin baselines |
| `xenv/v4_robust_summary.json` | status, code hash, checks, self-check, headline numbers, κ block, gap bins, bootstrap headline, timing |

`XV4_ROBUST` holds these tables, plus the per-draw κ counts and the per-policy and per-draw bootstrap tables.
`XV4_ROBUST_SUMMARY` holds the summary json.

**Expected cost on Colab.** This is an estimate, not a measurement. The analysis is pure numpy / pandas. Reading the
result rows from the CSV files takes 10–20 s; nothing is read when the globals are in memory. On synthetic inputs of
the same shape (local harness, M1 CPU), 300 draws of 50 policies were scored in 1.3 s, and 290 summary rows with
their 5,000-resample intervals took 1.4 s (the intervals are matrix products of resample counts). Scaled to the
roughly 700 draws of Part II and a Colab CPU two to three times slower, the claims take 5–15 s. The summary,
κ and gap tables take 5–15 s, the II.8i recount 5–10 s and the bootstrap rows under 5 s. The whole cell takes about
0.5–1 min, with well under 1 GB of extra memory.

In [ ]:
# ============================================================
# II.9b Robustness of certified comparisons (stored results, no simulation)
# ============================================================
EXECLOG.mark("II.9 Second-revision experiments: robustness of certified comparisons")
XV4R_SWITCH = "xenv_v4_revision_robust"
XV4R_VRANGE = {"gumbel": (-1.0, 1.0), "garnet": (0.0, 1.0), "icu": (0.0, 1.0)}   # [v_min, v_max]; r_term = v_min
assert XV4R_VRANGE == XV3_VRANGE, (XV4R_VRANGE, XV3_VRANGE)
XV4R_TARGETS = ("bench_mean", "bench_terminate", "true", "true_rows_mean", "true_rows_terminate")
XV4R_TARGET_SETS = {"true_rows": ("true_rows_terminate", "true_rows_mean", "p_true_rows_mean"),
                    "bench": ("bench_terminate", "bench_mean", "p_bench_mean")}   # (J_term, J_W, p) columns
XV4R_TYPES = ("all", "AA", "AU", "UU")
XV4R_CI_KINDS = ("cert_nhb", "cert_free", "margin_mean", "margin_term", "margin_mid", "margin_mean_free")
XV4R_CI_RESAMPLES = 5_000
XV4R_CI_LEVELS = (0.025, 0.975)
XV4R_KSTAR_Q = (0.10, 0.25, 0.50, 0.75, 0.90)
XV4R_GAP_BINS = (0.0, 0.0025, 0.005, 0.01, 0.025, 0.05, np.inf)
XV4R_RESULT_COLUMNS = ["cell", "family", "draw", "policy", "is_default", "is_reference", "target", "status", "return",
                       "prob_any_violation"]
XV4R_BOOT_COLUMNS = ["cell", "family", "draw", "policy", "replicate", "target", "status", "return",
                     "prob_any_violation"]
XV4R_BOOT_INTERVALS = {("point", "nhb"): ("point_lo", "point_up"),             # (interval, bound): (lower, upper)
                       ("point", "free"): ("point_lo", "point_up_free"),
                       ("bootstrap", "nhb"): ("boot_lo", "boot_up"),
                       ("bootstrap", "free"): ("boot_lo", "boot_up_free"),
                       ("minmax", "nhb"): ("minmax_lo", "minmax_up"),
                       ("minmax", "free"): ("minmax_lo", "minmax_up_free")}
# Published pooled counts (paper_assets_expected/facts_robust.json, 30 draws per cell). XV4R_KNOWN is asserted in the full paper
# design (XV4R_KNOWN_DRAWS draws present); XV4R_KNOWN_REPORTED is compared and reported only.
XV4R_KNOWN = {"true_rows:gumbel_estimated": {"cert_nhb_claims": 152_237, "cert_nhb_errors": 0,
                                             "cert_free_claims": 110_330, "cert_nhb_only_claims": 41_907,
                                             "cert_testable_claims": 43_346},
              "bench:gumbel_estimated": {"cert_nhb_claims": 151_338, "cert_nhb_errors": 15_981}}
XV4R_KNOWN_DRAWS = {"gumbel_estimated": 180, "garnet_all": 240}
XV4R_KNOWN_REPORTED = {"true_rows:gumbel_estimated": {"pairs": 220_500, "cert_free_errors": 0,
                                                      "point_mean_errors": 12_493, "point_term_errors": 4_851},
                       "bench:gumbel_estimated": {"pairs": 220_500, "cert_free_claims": 110_039,
                                                  "cert_free_errors": 15_084, "point_mean_errors": 30_436,
                                                  "point_term_errors": 20_329},
                       "true_rows:garnet_all": {"pairs": 294_000, "cert_nhb_claims": 249_930, "cert_nhb_errors": 2_396,
                                                "cert_free_claims": 225_494, "cert_free_errors": 0,
                                                "cert_nhb_only_claims": 24_436, "cert_testable_claims": 64_325},
                       "bench:garnet_all": {"pairs": 294_000, "cert_nhb_claims": 253_620, "cert_nhb_errors": 23_310,
                                            "cert_free_claims": 234_857, "cert_free_errors": 19_339}}


def _xv4r_rate(num: float, den: float) -> float:
    return float(num) / float(den) if den else float("nan")


def _xv4r_vrange(family: str) -> float:
    lo, hi = XV4R_VRANGE[str(family)]
    return float(hi - lo)


def xv4r_cell_group(cell: str, family: str, source: str, kind_of: Dict[str, str]) -> Tuple[str, float]:
    """(group, Garnet harm) of a Part II cell: 'gumbel_estimated', 'gumbel_oracle', 'garnet' or 'icu' for an II.6 cell,
    'v3_<kind>' for an II.8 cell (kind from X_V3_CELLS, else from the name)."""
    tok = str(cell).split("|")
    if source == "II.8":
        kind = kind_of.get(str(cell)) or ("aggregated" if tok[1].startswith("agg-") else
                                          "garnet-random" if family == "garnet" else "physician")
        return f"v3_{kind}", float("nan")
    if family == "icu":
        return "icu", float("nan")
    if family == "gumbel":
        assert tok[1] in ("best", "oracle"), cell
        return ("gumbel_estimated" if tok[1] == "best" else "gumbel_oracle"), float("nan")
    assert family == "garnet" and tok[1].startswith("h"), cell
    return "garnet", float(tok[1][1:])


def xv4r_inputs() -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """Long rows of the II.6 and II.8 cells on the five targets (XRESULTS / XRESULTS_V3, else their CSV files; column
    `source` = 'II.6' / 'II.8') and the II.8g bootstrap rows (XV3_BOOT_RESULTS, else its CSV file)."""
    info: Dict[str, Any] = {}
    frames = []
    for src, name, fname in (("II.6", "XRESULTS", "xenv_results_all.csv.gz"),
                             ("II.8", "XRESULTS_V3", "v3_results_all.csv.gz")):
        df, path = globals().get(name), XENV_TABLE_DIR / fname
        if df is not None and len(df):
            where, df = f"{name} (in memory)", df[XV4R_RESULT_COLUMNS]
        elif path.exists():
            where = f"xenv/{fname}"
            df = pd.read_csv(path, usecols=XV4R_RESULT_COLUMNS, float_precision="round_trip", low_memory=False)
        else:
            info[src] = {"source": "missing", "rows": 0, "cells": 0}
            continue
        df = df[df["target"].isin(XV4R_TARGETS)].assign(source=src)
        info[src] = {"source": where, "rows": int(len(df)), "cells": int(df["cell"].nunique())}
        frames.append(df)
    res = (pd.concat(frames, ignore_index=True) if frames
           else pd.DataFrame(columns=[*XV4R_RESULT_COLUMNS, "source"]))
    if len(frames) == 2:
        clash = set(frames[0]["cell"]) & set(frames[1]["cell"])
        assert not clash, f"II.6 and II.8 cells share names: {sorted(clash)}"
    boot, path = globals().get("XV3_BOOT_RESULTS"), XENV_TABLE_DIR / "v3_bootstrap_results.csv.gz"
    if boot is not None and len(boot):
        info["II.8g"], boot = "XV3_BOOT_RESULTS (in memory)", boot[XV4R_BOOT_COLUMNS].copy()
    elif path.exists():
        info["II.8g"] = "xenv/v3_bootstrap_results.csv.gz"
        boot = pd.read_csv(path, usecols=XV4R_BOOT_COLUMNS, float_precision="round_trip", low_memory=False)
    else:
        info["II.8g"], boot = "missing", pd.DataFrame(columns=XV4R_BOOT_COLUMNS)
    info["II.8g_rows"] = int(len(boot))
    return res, boot, info


def xv4r_wide(res: pd.DataFrame, kind_of: Dict[str, str]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Per (cell, draw, policy), default non-reference policies only: J and P(any unsupported decision) on the five
    targets (NaN unless the status is ok; p columns 'p_<target>'), family, source, group and Garnet harm. The oracle
    cell's true-rows columns are its benchmark columns. Sorted by (cell, draw, policy)."""
    r = res[xs_as_bool(res["is_default"]) & ~xs_as_bool(res["is_reference"])].copy()
    r["draw"] = pd.to_numeric(r["draw"]).astype(np.int64)
    k = ["cell", "draw", "policy"]
    dup = r.duplicated(k + ["target"])
    assert not dup.any(), f"{int(dup.sum())} duplicated (cell, draw, policy, target) rows"
    ok = r["status"].astype(str) == "ok"
    r["J"] = pd.to_numeric(r["return"], errors="coerce").where(ok)
    r["pv"] = pd.to_numeric(r["prob_any_violation"], errors="coerce").where(ok)
    idx = r.set_index(k + ["target"])
    J = idx["J"].unstack("target").reindex(columns=list(XV4R_TARGETS))
    Pv = idx["pv"].unstack("target").reindex(columns=list(XV4R_TARGETS)).add_prefix("p_")
    W = r.drop_duplicates(k).set_index(k)[["family", "source"]].join(J).join(Pv)
    cells = r.drop_duplicates("cell")[["cell", "family", "source"]]
    grp = {c: xv4r_cell_group(c, f, s, kind_of) for c, f, s in cells.itertuples(index=False)}
    cl = W.index.get_level_values("cell")
    W["group"] = cl.map(lambda c: grp[c][0]).to_numpy()
    W["harm"] = cl.map(lambda c: grp[c][1]).to_numpy(dtype=float)
    orc = (W["group"] == "gumbel_oracle").to_numpy()
    W["raw_true_rows"] = W[["true_rows_mean", "true_rows_terminate"]].notna().all(axis=1)
    for a, b in (("true_rows_mean", "bench_mean"), ("true_rows_terminate", "bench_terminate"),
                 ("p_true_rows_mean", "p_bench_mean"), ("p_true_rows_terminate", "p_bench_terminate")):
        assert W.loc[orc, a].isna().all(), f"the oracle cell has no {a} target (its benchmark rows are the true rows)"
        W.loc[orc, a] = W.loc[orc, b]
    checks = {"rows": int(len(r)), "nonok_rows": int((~ok).sum()), "policy_rows": int(len(W)),
              "cells": int(len(grp)), "draws": int(W.reset_index()[["cell", "draw"]].drop_duplicates().shape[0]),
              "oracle_policy_rows": int(orc.sum())}
    return W.sort_index(), checks


def xv4r_theorem_counts(W: pd.DataFrame) -> Dict[str, int]:
    """Theorem facts on the stored exact values (counts of violations, tolerance XV4_TOL; all must be 0)."""
    vr = W["family"].map(_xv4r_vrange).to_numpy(dtype=float)
    out = {"values_outside_value_range": 0}
    for fam, (lo, hi) in XV4R_VRANGE.items():
        v = W.loc[W["family"] == fam, list(XV4R_TARGETS)].to_numpy(dtype=float)
        out["values_outside_value_range"] += int(((v < lo - XV4_TOL) | (v > hi + XV4_TOL)).sum())
    tr = W[["true", "true_rows_mean", "true_rows_terminate"]].notna().all(axis=1).to_numpy()
    bn = W[["bench_mean", "bench_terminate"]].notna().all(axis=1).to_numpy()
    raw = W["raw_true_rows"].to_numpy(dtype=bool) & tr
    jt, tt, tm = (W[c].to_numpy(dtype=float) for c in ("true", "true_rows_terminate", "true_rows_mean"))
    bt, bm = W["bench_terminate"].to_numpy(dtype=float), W["bench_mean"].to_numpy(dtype=float)
    pt, pb = W["p_true_rows_mean"].to_numpy(dtype=float), W["p_bench_mean"].to_numpy(dtype=float)
    with np.errstate(invalid="ignore"):
        out["negative_width_true_rows"] = int((tr & (tm - tt < -XV4_TOL)).sum())
        out["width_above_bound_true_rows"] = int((tr & (tm - tt > pt * vr + XV4_TOL)).sum())
        out["negative_width_bench"] = int((bn & (bm - bt < -XV4_TOL)).sum())
        out["width_above_bound_bench"] = int((bn & (bm - bt > pb * vr + XV4_TOL)).sum())
        out["lower_bound_violations_true_rows"] = int((tr & (jt < tt - XV4_TOL)).sum())
        out["nhb_free_upper_violations_true_rows"] = int((tr & (jt > tt + pt * vr + XV4_TOL)).sum())
        out["nonzero_width_admissible_true_rows"] = int((tr & (pt <= XV4_PTOL) & (np.abs(tm - tt) > XV4_TOL)).sum())
        out["rule_p_difference_bench"] = int(
            (bn & (np.abs(pb - W["p_bench_terminate"].to_numpy(dtype=float)) > XV4_TOL)).sum())
        out["rule_p_difference_true_rows"] = int(
            (raw & (np.abs(pt - W["p_true_rows_terminate"].to_numpy(dtype=float)) > XV4_TOL)).sum())
        out["truth_p_difference_true_rows"] = int(
            (raw & (np.abs(pt - W["p_true"].to_numpy(dtype=float)) > XV4_TOL)).sum())
    return out


def xv4r_kappa_claims(lo: np.ndarray, up: np.ndarray, p: np.ndarray, v_range: float, jt: Optional[np.ndarray],
                      grid: Sequence[float]) -> Tuple[List[dict], np.ndarray, np.ndarray]:
    """kappa-certificates lo_i > (1 - kappa) up_j + kappa (lo_j + p_j v_range) (+ XV4_TOL) for every kappa of `grid`,
    i.e. J_term(i) > J_W(j) + kappa E_j with E_j = p_j v_range - (J_W(j) - J_term(j)): kappa = 0 is cert_nhb and
    kappa = 1 cert_free of x_v4_pair_claims (the convex combination makes both ends exact). Returns the counts per kappa
    (all pairs and completion-sensitive AU + UU pairs; errors when `jt` is given) and, for the NHB-only claims, the
    breakdown fraction kappa* = (J_term(i) - J_W(j)) / E_j and the margin J_term(i) - J_W(j)."""
    lo, up, p = (np.asarray(x, dtype=float) for x in (lo, up, p))
    n = len(lo)
    off = ~np.eye(n, dtype=bool)
    u = p > XV4_PTOL
    cs = off & (u[:, None] | u[None, :])
    free = lo + p * float(v_range)
    nhb = _x_v4_gt(lo, up) & off
    wrong = None if jt is None else ~_x_v4_gt(np.asarray(jt, dtype=float), np.asarray(jt, dtype=float))
    base = {"n_policies": n, "pairs": int(off.sum() // 2), "pairs_cs": int(cs.sum() // 2),
            "nhb_claims": int(nhb.sum())}
    rows = []
    for kap in grid:
        C = _x_v4_gt(lo, (1.0 - float(kap)) * up + float(kap) * free) & off
        rows.append({**base, "kappa": float(kap), "claims": int(C.sum()), "claims_cs": int((C & cs).sum()),
                     "errors": np.nan if wrong is None else int((C & wrong).sum()),
                     "errors_cs": np.nan if wrong is None else int((C & cs & wrong).sum())})
    i, j = np.nonzero(nhb & ~_x_v4_gt(lo, free))
    excess = free - up
    assert (excess[j] > 0).all(), "an NHB-only claim needs a positive largest excess E_j of the losing policy"
    margin = lo[i] - up[j]
    return rows, margin / excess[j], margin


def xv4r_gap_claims(lo: np.ndarray, up: np.ndarray, p: np.ndarray, v_range: float, jt: np.ndarray) -> pd.DataFrame:
    """NHB-certified claims between two policies that cannot leave the support (p <= XV4_PTOL): normalised gap
    (J_term(i) - J_W(j)) / v_range, and whether the truth reverses or ties (`wrong`)."""
    lo, up, p, jt = (np.asarray(x, dtype=float) for x in (lo, up, p, jt))
    a = p <= XV4_PTOL
    C = _x_v4_gt(lo, up) & a[:, None] & a[None, :]
    np.fill_diagonal(C, False)
    i, j = np.nonzero(C)
    return pd.DataFrame({"gap": (lo[i] - up[j]) / float(v_range), "wrong": ~(jt[i] > jt[j] + XV4_TOL)})


def xv4r_draw(d: pd.DataFrame, grid: Sequence[float]) -> Dict[str, Any]:
    """Claims of one draw (rows of xv4r_wide, ordered by policy): x_v4_pair_claims on each target set for all and for
    distinct policies, the kappa counts, kappa* and margins of the NHB-only claims, and the admissible-pair benchmark
    claims. A target set absent from the cell is left out; one with a non-proper policy is skipped and listed."""
    fam = str(d["family"].iloc[0])
    vr = _xv4r_vrange(fam)
    out: Dict[str, Any] = {"pairs": [], "kappa": [], "kstar": {}, "gaps": None, "skipped": []}
    truth = fam != "icu"
    if truth and d["true"].isna().any():
        out["skipped"] = [tf for tf, cols in XV4R_TARGET_SETS.items() if d[list(cols)].notna().any().any()]
        return out
    jt = d["true"].to_numpy(dtype=float) if truth else None
    avail = [t for t in XV4R_TARGETS if d[t].notna().all()]
    keep = x_v4_distinct(d[avail].to_numpy(dtype=float)) if avail else np.ones(len(d), dtype=bool)
    for tf, (lc, uc, pc) in XV4R_TARGET_SETS.items():
        sub = d[[lc, uc, pc]]
        if sub.isna().all().all():
            continue
        if sub.isna().any().any():
            out["skipped"].append(tf)
            continue
        lo, up, p = (sub[c].to_numpy(dtype=float) for c in (lc, uc, pc))
        for dist in (False, True):
            m = keep if dist else np.ones(len(d), dtype=bool)
            for r in x_v4_pair_claims(lo[m], up[m], p[m], vr, None if jt is None else jt[m]):
                out["pairs"].append({"targets": tf, "distinct": dist, **r})
        krows, kstar, margin = xv4r_kappa_claims(lo, up, p, vr, jt, grid)
        out["kappa"] += [{"targets": tf, **r} for r in krows]
        out["kstar"][tf] = (kstar, margin)
        if tf == "bench" and truth:
            out["gaps"] = xv4r_gap_claims(lo, up, p, vr, jt)
    return out


def xv4r_claims(W: pd.DataFrame, grid: Sequence[float]
                ) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[Tuple[str, str], Tuple[np.ndarray, np.ndarray]],
                           pd.DataFrame, Dict[str, int]]:
    """xv4r_draw over every (cell, draw) of W: per-draw claim rows, kappa rows, the NHB-only kappa* and margins per
    (cell, target set), the admissible-pair gap claims and the skipped (cell | target set) counts."""
    pair_rows: List[dict] = []
    k_rows: List[dict] = []
    gap_frames: List[pd.DataFrame] = []
    parts: Dict[Tuple[str, str], List[Tuple[np.ndarray, np.ndarray]]] = {}
    skipped: Counter = Counter()
    cells = list(pd.unique(W.index.get_level_values("cell")))
    for ci, cell in enumerate(cells, 1):
        t_cell = time.perf_counter()
        g = W.xs(cell, level="cell", drop_level=False)
        draws = g.index.get_level_values("draw")
        for draw, d in g.groupby(level="draw", sort=True):
            meta = {"cell": cell, "draw": int(draw), "family": str(d["family"].iloc[0]),
                    "source": str(d["source"].iloc[0]), "group": str(d["group"].iloc[0])}
            o = xv4r_draw(d, grid)
            pair_rows += [{**meta, **r} for r in o["pairs"]]
            k_rows += [{**meta, **r} for r in o["kappa"]]
            for tf, v in o["kstar"].items():
                parts.setdefault((cell, tf), []).append(v)
            if o["gaps"] is not None:
                gap_frames.append(o["gaps"].assign(cell=cell, draw=int(draw)))
            for tf in o["skipped"]:
                skipped[f"{cell}|{tf}"] += 1
        print(f"  [{ci}/{len(cells)}] {cell}: {draws.nunique()} draws, {len(g) // max(draws.nunique(), 1)} policies "
              f"per draw, {time.perf_counter() - t_cell:.1f} s")
    R = pd.DataFrame(pair_rows)
    K = pd.DataFrame(k_rows)
    KS = {key: (np.concatenate([a for a, _ in v]), np.concatenate([b for _, b in v])) for key, v in parts.items()}
    G = (pd.concat(gap_frames, ignore_index=True) if gap_frames
         else pd.DataFrame({"gap": pd.Series(dtype=float), "wrong": pd.Series(dtype=bool),
                            "cell": pd.Series(dtype=str), "draw": pd.Series(dtype=np.int64)}))
    return R, K, KS, G, dict(skipped)


def xv4r_units(W: pd.DataFrame, order: Sequence[str]) -> Tuple[List[Tuple[str, str, List[str]]], Dict[str, List[str]]]:
    """Analysis units: every cell (driver order), then the pools of the paper and one pool per II.8 kind."""
    info = W.reset_index().drop_duplicates("cell").set_index("cell")[["group", "harm"]]
    cells = [c for c in order if c in info.index]
    grp, harm = info["group"], info["harm"]
    gar = grp == "garnet"
    masks = {"gumbel_estimated": grp == "gumbel_estimated", "gumbel_oracle": grp == "gumbel_oracle",
             "garnet_all": gar, "garnet_h-0.5": gar & np.isclose(harm, -0.5),
             "garnet_h0.0": gar & np.isclose(harm, 0.0), "garnet_h_ge_0.5": gar & (harm >= 0.5), "icu": grp == "icu"}
    masks.update({g: grp == g for g in sorted(set(grp)) if g.startswith("v3_")})
    pools = {name: [c for c in cells if bool(m.get(c, False))] for name, m in masks.items()}
    pools = {name: cs for name, cs in pools.items() if cs}
    return [(c, "cell", [c]) for c in cells] + [(name, "pool", cs) for name, cs in pools.items()], pools


def xv4r_summary_row(d: pd.DataFrame, kinds: Sequence[str]) -> Dict[str, Any]:
    """Pooled claims, shares and error rates of per-draw rows of one (target set, policy set, pair type), and the
    draws with at least one (wrong) certificate."""
    assert not d.duplicated(["cell", "draw"]).any(), "one row per (cell, draw) expected"
    s = d.sum(numeric_only=True)
    pairs = int(s["pairs"])
    err = len(d) > 0 and all(f"{k}_errors" in d.columns and d[f"{k}_errors"].notna().all() for k in kinds)
    o: Dict[str, Any] = {"draws": int(len(d)), "policies_per_draw": float(d["n_policies"].median()), "pairs": pairs}
    for nm in kinds:
        cl = int(s[f"{nm}_claims"])
        o[f"{nm}_claims"], o[f"{nm}_share"] = cl, _xv4r_rate(cl, pairs)
        o[f"{nm}_errors"] = int(s[f"{nm}_errors"]) if err else np.nan
        o[f"{nm}_error_rate"] = _xv4r_rate(s[f"{nm}_errors"], cl) if err else np.nan
    for nm in ("cert_nhb", "cert_free"):
        o[f"draws_with_{nm}"] = int((d[f"{nm}_claims"] > 0).sum())
        o[f"draws_with_{nm}_error"] = int((d[f"{nm}_errors"] > 0).sum()) if err else np.nan
    return o


def xv4r_cs_headline(d: pd.DataFrame) -> Dict[str, Any]:
    """Completion-sensitive headline from the AU and UU rows of one (unit, target set, policy set): certified share
    and error rate among pairs in which at least one policy can leave the support."""
    s = d.sum(numeric_only=True)
    pairs = int(s["pairs"]) if len(d) else 0
    o: Dict[str, Any] = {"cs_pairs": pairs}
    for nm in ("cert_nhb", "cert_free"):
        cl = int(s[f"{nm}_claims"]) if len(d) else 0
        err = len(d) > 0 and f"{nm}_errors" in d.columns and d[f"{nm}_errors"].notna().all()
        o[f"{nm}_cs_claims"], o[f"{nm}_cs_share"] = cl, _xv4r_rate(cl, pairs)
        o[f"{nm}_cs_errors"] = int(s[f"{nm}_errors"]) if err else np.nan
        o[f"{nm}_cs_error_rate"] = _xv4r_rate(s[f"{nm}_errors"], cl) if err else np.nan
    return o


def xv4r_resample_matrix(cells: np.ndarray, n_boot: int, rng: np.random.Generator) -> np.ndarray:
    """(n_boot, n_draws) multiplicities of a draw-clustered bootstrap: the draws of every cell (in order of first
    appearance) are resampled with replacement within the cell."""
    cells = np.asarray(cells)
    n_boot = int(n_boot)
    M = np.zeros((n_boot, len(cells)))
    for c in pd.unique(cells):
        ix = np.flatnonzero(cells == c)
        pick = rng.integers(0, len(ix), size=(n_boot, len(ix)))
        flat = (np.arange(n_boot)[:, None] * len(ix) + pick).ravel()
        M[:, ix] = np.bincount(flat, minlength=n_boot * len(ix)).reshape(n_boot, len(ix))
    return M


def xv4r_ratio_ci(M: np.ndarray, errors: np.ndarray, claims: np.ndarray) -> Tuple[float, float]:
    """Percentile interval (XV4R_CI_LEVELS) of the pooled error rate sum(errors) / sum(claims) over the resamples M; a
    resample without claims counts as 0 (make_robustness.py); NaN when there is no claim at all."""
    errors, claims = np.asarray(errors, dtype=float), np.asarray(claims, dtype=float)
    if not claims.sum():
        return float("nan"), float("nan")
    est = (M @ errors) / np.maximum(M @ claims, 1.0)
    return float(np.quantile(est, XV4R_CI_LEVELS[0])), float(np.quantile(est, XV4R_CI_LEVELS[1]))


def xv4r_summary_table(R: pd.DataFrame, units: Sequence[Tuple[str, str, List[str]]]) -> pd.DataFrame:
    """Summary rows per unit x target set x policy set x pair type (xv4r_summary_row, xv4r_cs_headline and the
    draw-clustered intervals of the XV4R_CI_KINDS error rates)."""
    kinds = [c[:-len("_claims")] for c in R.columns if c.endswith("_claims")]
    rows = []
    for unit, utype, cells in units:
        U = R[R["cell"].isin(cells)]
        for tf in XV4R_TARGET_SETS:
            T = U[U["targets"] == tf]
            if T.empty:
                continue
            base = T[~T["distinct"].astype(bool) & (T["type"] == "all")].set_index(["cell", "draw"]).sort_index().index
            mats = {}
            if "cert_nhb_errors" in T.columns and T["cert_nhb_errors"].notna().all():
                cl = base.get_level_values("cell").to_numpy()
                mats = {nm: xv4r_resample_matrix(cl, XV4R_CI_RESAMPLES, np_rng("xenv", "v4", "robust-ci", unit, tf, nm))
                        for nm in XV4R_CI_KINDS}
            for dist in (False, True):
                D = T[T["distinct"].astype(bool) == dist]
                cs = xv4r_cs_headline(D[D["type"].isin(["AU", "UU"])])
                for ty in XV4R_TYPES:
                    d = D[D["type"] == ty]
                    r = {"unit": unit, "unit_type": utype, "n_cells": len(cells),
                         "source": "+".join(sorted(set(d["source"]))), "targets": tf, "distinct": dist, "type": ty,
                         **xv4r_summary_row(d, kinds), **cs}
                    a = d.set_index(["cell", "draw"]).reindex(base)
                    assert a["pairs"].notna().all(), f"{unit} {tf}: draws differ between policy sets / pair types"
                    for nm in XV4R_CI_KINDS:
                        lo, hi = (xv4r_ratio_ci(mats[nm], a[f"{nm}_errors"].to_numpy(dtype=float),
                                                a[f"{nm}_claims"].to_numpy(dtype=float)) if mats
                                  else (float("nan"), float("nan")))
                        r[f"{nm}_error_rate_ci_low"], r[f"{nm}_error_rate_ci_high"] = lo, hi
                    rows.append(r)
            del mats
    return pd.DataFrame(rows)


def xv4r_kappa_table(K: pd.DataFrame, KS: Dict[Tuple[str, str], Tuple[np.ndarray, np.ndarray]],
                     units: Sequence[Tuple[str, str, List[str]]]) -> pd.DataFrame:
    """Per unit x target set x kappa: claims, share of pairs and of completion-sensitive pairs, share of the NHB
    claims, errors; plus the quantiles of kappa* and of the margin over the unit's NHB-only claims."""
    rows = []
    for unit, utype, cells in units:
        for tf in XV4R_TARGET_SETS:
            d = K[K["cell"].isin(cells) & (K["targets"] == tf)]
            if d.empty:
                continue
            ks = np.concatenate([KS[(c, tf)][0] for c in cells if (c, tf) in KS] or [np.zeros(0)])
            mg = np.concatenate([KS[(c, tf)][1] for c in cells if (c, tf) in KS] or [np.zeros(0)])
            base = {"unit": unit, "unit_type": utype, "n_cells": len(cells), "targets": tf,
                    "nhb_only_claims": int(len(ks))}
            for q in XV4R_KSTAR_Q:
                base[f"kappa_star_q{int(round(100 * q)):02d}"] = float(np.quantile(ks, q)) if len(ks) else np.nan
            for q in XV4R_KSTAR_Q:
                base[f"nhb_only_margin_q{int(round(100 * q)):02d}"] = float(np.quantile(mg, q)) if len(mg) else np.nan
            base["nhb_only_margin_mean"] = float(mg.mean()) if len(mg) else np.nan
            for kap, g in d.groupby("kappa", sort=True):
                s = g.sum(numeric_only=True)
                err = g["errors"].notna().all()
                rows.append({**base, "kappa": float(kap), "draws": int(len(g)), "pairs": int(s["pairs"]),
                             "pairs_cs": int(s["pairs_cs"]), "nhb_claims": int(s["nhb_claims"]),
                             "claims": int(s["claims"]), "share": _xv4r_rate(s["claims"], s["pairs"]),
                             "claims_cs": int(s["claims_cs"]), "cs_share": _xv4r_rate(s["claims_cs"], s["pairs_cs"]),
                             "share_of_nhb_claims": _xv4r_rate(s["claims"], s["nhb_claims"]),
                             "errors": int(s["errors"]) if err else np.nan,
                             "error_rate": _xv4r_rate(s["errors"], s["claims"]) if err else np.nan,
                             "errors_cs": int(s["errors_cs"]) if err else np.nan,
                             "error_rate_cs": _xv4r_rate(s["errors_cs"], s["claims_cs"]) if err else np.nan,
                             "draws_with_error": int((g["errors"] > 0).sum()) if err else np.nan})
    return pd.DataFrame(rows)


def xv4r_gap_table(G: pd.DataFrame, units: Sequence[Tuple[str, str, List[str]]], cells_with_truth: Sequence[str]
                   ) -> pd.DataFrame:
    """Error rate of the admissible-pair benchmark claims by normalised gap bin (XV4R_GAP_BINS), per unit; rows 'all'
    and '(0, 0.01]' are totals."""
    rows = []
    have = set(cells_with_truth)
    for unit, utype, cells in units:
        cs = [c for c in cells if c in have]
        if not cs:
            continue
        g = G[G["cell"].isin(cs)]
        tab = g.groupby(pd.cut(g["gap"], list(XV4R_GAP_BINS), right=True), observed=False)["wrong"].agg(["size", "sum"])
        lab = [(str(b), int(r["size"]), int(r["sum"])) for b, r in tab.iterrows()]
        low = g["gap"] <= 0.01
        lab += [("all", int(len(g)), int(g["wrong"].sum())),
                ("(0, 0.01]", int(low.sum()), int(g.loc[low, "wrong"].sum()))]
        rows += [{"unit": unit, "unit_type": utype, "n_cells": len(cs), "bin": b, "claims": n, "errors": e,
                  "error_rate": _xv4r_rate(e, n)} for b, n, e in lab]
    return pd.DataFrame(rows)


def xv4r_boot_intervals(boot: pd.DataFrame, q: Tuple[float, float]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Per (cell, draw, default policy) of the II.8g bootstrap: J_true, the point values and p (replicate -1), and the
    intervals over the proper replicates of each rule: NHB [q_lo(J_term^b), q_hi(J_mean^b)], NHB-free
    [q_lo(J_term^b), q_hi(J_term^b + p^b v_range)] (p^b of the terminate row), and their min / max versions.
    Also the per-replicate invariant counts (replicates where both rules are proper)."""
    b = boot[XV4R_BOOT_COLUMNS].copy()
    b["draw"] = pd.to_numeric(b["draw"]).astype(np.int64)
    b["replicate"] = pd.to_numeric(b["replicate"]).astype(np.int64)
    for c in ("return", "prob_any_violation"):
        b[c] = pd.to_numeric(b[c], errors="coerce")
    b["vr"] = b["family"].map(_xv4r_vrange)
    k = ["cell", "draw", "policy"]
    dup = b.duplicated(k + ["replicate", "target"])
    assert not dup.any(), f"{int(dup.sum())} duplicated bootstrap rows"
    pt = b[b["replicate"] == XV3_BOOT_POINT].set_index(k + ["target"])
    assert (pt["status"].astype(str) == "ok").all(), "replicate -1 (original benchmark and truth) must be proper"
    J0, P0 = pt["return"].unstack("target"), pt["prob_any_violation"].unstack("target")
    iv = pd.DataFrame({"true": J0["true"], "point_lo": J0["bench_terminate"], "point_up": J0["bench_mean"],
                       "p_bench": P0["bench_mean"]})
    iv = iv.join(b.drop_duplicates(k).set_index(k)[["family", "vr"]])
    iv["point_up_free"] = iv["point_lo"] + iv["p_bench"] * iv["vr"]
    rep = b[(b["replicate"] >= 0) & (b["status"].astype(str) == "ok")]
    tm = rep[rep["target"] == "bench_terminate"].assign(free=lambda x: x["return"] + x["prob_any_violation"] * x["vr"])
    gt_, gf = tm.groupby(k)["return"], tm.groupby(k)["free"]
    gm = rep[rep["target"] == "bench_mean"].groupby(k)["return"]
    cols = {"boot_lo": gt_.quantile(float(q[0])), "boot_up": gm.quantile(float(q[1])),
            "boot_up_free": gf.quantile(float(q[1])), "minmax_lo": gt_.min(), "minmax_up": gm.max(),
            "minmax_up_free": gf.max(), "n_ok_terminate": gt_.size(), "n_ok_mean": gm.size()}
    for c, s in cols.items():
        iv[c] = s.reindex(iv.index)
    iv["n_replicates"] = b[b["replicate"] >= 0].groupby(k)["replicate"].nunique().reindex(iv.index)
    for c in ("n_ok_terminate", "n_ok_mean", "n_replicates"):
        iv[c] = iv[c].fillna(0).astype(int)
    wide = rep.set_index(k + ["replicate", "target"])[["return", "prob_any_violation"]].unstack("target").reindex(
        columns=pd.MultiIndex.from_product([["return", "prob_any_violation"], ["bench_mean", "bench_terminate"]]))
    jm, jtm = wide[("return", "bench_mean")], wide[("return", "bench_terminate")]
    pm, ptm = wide[("prob_any_violation", "bench_mean")], wide[("prob_any_violation", "bench_terminate")]
    vr = wide.index.get_level_values("cell").map(dict(zip(b["cell"], b["vr"]))).to_numpy(dtype=float)
    both = (jm.notna() & jtm.notna()).to_numpy()
    w = (jm - jtm).to_numpy(dtype=float)
    with np.errstate(invalid="ignore"):
        dp = np.abs((pm - ptm).to_numpy(dtype=float))
        checks = {"policy_rows": int(len(iv)), "replicate_pairs_checked": int(both.sum()),
                  "replicate_rule_p_difference": int((both & (dp > XV4_TOL)).sum()),
                  "replicate_negative_width": int((both & (w < -XV4_TOL)).sum()),
                  "replicate_width_above_bound": int((both & (w > ptm.to_numpy(dtype=float) * vr + XV4_TOL)).sum()),
                  "point_negative_width": int((iv["point_up"] < iv["point_lo"] - XV4_TOL).sum()),
                  "point_free_below_nhb_upper": int((iv["point_up_free"] < iv["point_up"] - XV4_TOL).sum()),
                  # reported, not asserted: the two ends are quantiles over different sets of proper replicates
                  "inverted_bootstrap_intervals": int((iv["boot_lo"] > iv["boot_up"] + XV4_TOL).sum()),
                  "replicates_in_rows": int(iv["n_replicates"].max()) if len(iv) else 0}
    return iv.reset_index(), checks


def xv4r_interval_claims(lo: np.ndarray, up: np.ndarray, u: np.ndarray, wrong: np.ndarray) -> Dict[str, int]:
    """Certified claims lo_i > up_j (+ XV4_TOL) of one interval type and their errors (`wrong`: truth reverses or ties),
    over all pairs and over completion-sensitive pairs (u: p > XV4_PTOL for at least one of the two policies)."""
    n = len(lo)
    off = ~np.eye(n, dtype=bool)
    cs = off & (u[:, None] | u[None, :])
    C = _x_v4_gt(np.asarray(lo, dtype=float), np.asarray(up, dtype=float)) & off
    return {"n_policies": n, "pairs": int(off.sum() // 2), "claims": int(C.sum()), "errors": int((C & wrong).sum()),
            "cs_pairs": int(cs.sum() // 2), "cs_claims": int((C & cs).sum()),
            "cs_errors": int((C & cs & wrong).sum())}


def xv4r_boot_draws(iv: pd.DataFrame) -> pd.DataFrame:
    """Per (cell, draw, interval, bound): certificates of the policies whose interval is defined, coverage counts,
    draw-level flags, and the margin rules on the point values at the same number of claims."""
    rows = []
    for (c, dr), d in iv.groupby(["cell", "draw"], sort=False):
        d = d.sort_values("policy")
        for (it, bd), (lc, uc) in XV4R_BOOT_INTERVALS.items():
            ok = d[[lc, uc]].notna().all(axis=1).to_numpy()
            dd = d[ok]
            lo, up, jt = (dd[x].to_numpy(dtype=float) for x in (lc, uc, "true"))
            cov = (jt >= lo - XV4_TOL) & (jt <= up + XV4_TOL)
            wrong = ~_x_v4_gt(jt, jt)
            r = xv4r_interval_claims(lo, up, dd["p_bench"].to_numpy(dtype=float) > XV4_PTOL, wrong)
            row = {"cell": c, "family": d["family"].iloc[0], "draw": int(dr), "interval": it, "bound": bd,
                   "policies": int(len(d)), "undefined": int((~ok).sum()), **r, "uncovered": int(len(d) - cov.sum()),
                   "any_wrong": bool(r["errors"] > 0), "all_covered": bool(ok.all() and cov.all())}
            pl, pu = dd["point_lo"].to_numpy(dtype=float), dd["point_up"].to_numpy(dtype=float)
            for nm, v in (("mean", pu), ("term", pl), ("mid", (pl + pu) / 2.0)):
                Cm = x_v4_margin_claims(v, r["claims"])
                row[f"margin_{nm}_claims"], row[f"margin_{nm}_errors"] = int(Cm.sum()), int((Cm & wrong).sum())
            rows.append(row)
    return pd.DataFrame(rows)


def xv4r_boot_table(iv: pd.DataFrame, D: pd.DataFrame, order: Sequence[str]) -> pd.DataFrame:
    """Per bootstrap cell, family pool and overall pool x interval x bound: coverage of J_true over all default
    policies and over p > 0 / p = 0, mean width, certificates (also among AU + UU pairs), draw-level counts and the
    error rates of the margin rules at the same number of claims."""
    fams = sorted(set(iv["family"]))
    units = [(c, "cell", [c]) for c in order if c in set(iv["cell"])]
    units += [(f"pooled:{f}", "pool", sorted(set(iv.loc[iv["family"] == f, "cell"]))) for f in fams]
    if len(fams) > 1:
        units.append(("pooled:all", "pool", sorted(set(iv["cell"]))))
    rows = []
    for unit, utype, cs in units:
        d, g0 = iv[iv["cell"].isin(cs)], D[D["cell"].isin(cs)]
        pos = (d["p_bench"] > XV4_PTOL).to_numpy()
        jt = d["true"].to_numpy(dtype=float)
        for (it, bd), (lc, uc) in XV4R_BOOT_INTERVALS.items():
            lo, up = d[lc].to_numpy(dtype=float), d[uc].to_numpy(dtype=float)
            ok = ~(np.isnan(lo) | np.isnan(up))
            cov = ok & (jt >= lo - XV4_TOL) & (jt <= up + XV4_TOL)
            width = up - lo
            g = g0[(g0["interval"] == it) & (g0["bound"] == bd)]
            s = g.sum(numeric_only=True)
            r = {"unit": unit, "unit_type": utype, "n_cells": len(cs), "interval": it, "bound": bd,
                 "replicates": int(d["n_replicates"].max()) if len(d) else 0,
                 "quantiles": "|".join(str(x) for x in XV3BOOT.quantiles),
                 "n_policies": int(len(d)), "n_undefined": int((~ok).sum()), "n_p_pos": int(pos.sum()),
                 "n_p_zero": int((~pos).sum()), "coverage_all": _xv4r_rate(cov.sum(), ok.sum()),
                 "coverage_p_pos": _xv4r_rate((cov & pos).sum(), (ok & pos).sum()),
                 "coverage_p_zero": _xv4r_rate((cov & ~pos).sum(), (ok & ~pos).sum()),
                 "mean_width": float(width[ok].mean()) if ok.any() else np.nan,
                 "mean_width_p_pos": float(width[ok & pos].mean()) if (ok & pos).any() else np.nan,
                 "draws": int(len(g)), "pairs": int(s["pairs"]), "claims": int(s["claims"]),
                 "certified_share": _xv4r_rate(s["claims"], s["pairs"]), "errors": int(s["errors"]),
                 "error_rate": _xv4r_rate(s["errors"], s["claims"]), "cs_pairs": int(s["cs_pairs"]),
                 "cs_claims": int(s["cs_claims"]), "cs_share": _xv4r_rate(s["cs_claims"], s["cs_pairs"]),
                 "cs_errors": int(s["cs_errors"]), "cs_error_rate": _xv4r_rate(s["cs_errors"], s["cs_claims"]),
                 "draws_with_wrong_cert": int(g["any_wrong"].sum()), "draws_all_covered": int(g["all_covered"].sum())}
            for nm in ("mean", "term", "mid"):
                r[f"margin_{nm}_claims"] = int(s[f"margin_{nm}_claims"])
                r[f"margin_{nm}_error_rate"] = _xv4r_rate(s[f"margin_{nm}_errors"], s[f"margin_{nm}_claims"])
            rows.append(r)
    return pd.DataFrame(rows)


def xv4r_boot_vs_ii8i(iv: pd.DataFrame) -> Dict[str, Any]:
    """The NHB bootstrap interval, the point interval and J_true against II.8i's xv3_boot_intervals (XV3_STATS) on the
    same rows: an identity of two implementations (rows beyond XV4_TOL or with a different NaN pattern)."""
    st = globals().get("XV3_STATS") or {}
    ref = st.get("bootstrap_intervals") if isinstance(st, dict) else None
    if ref is None or not len(ref):
        return {"status": "not compared (XV3_STATS has no bootstrap intervals)", "mismatched_rows": 0}
    k = ["cell", "draw", "policy"]
    r = ref.assign(draw=pd.to_numeric(ref["draw"]).astype(np.int64))
    m = iv.merge(r, on=k, how="outer", suffixes=("", "_ii8i"), indicator=True)
    out = {"status": "compared", "rows": int(len(m)), "missing": int((m["_merge"] != "both").sum()),
           "max_abs_diff": {}, "mismatched_rows": int((m["_merge"] != "both").sum())}
    both = m[m["_merge"] == "both"]
    bad = np.zeros(len(both), dtype=bool)
    for c in ("true", "point_lo", "point_up", "boot_lo", "boot_up"):
        a, b_ = both[c].to_numpy(dtype=float), both[f"{c}_ii8i"].to_numpy(dtype=float)
        nan_diff = np.isnan(a) != np.isnan(b_)
        d = np.abs(np.where(np.isnan(a) | np.isnan(b_), 0.0, a - b_))
        out["max_abs_diff"][c] = float(d.max(initial=0.0))
        bad |= nan_diff | (d > XV4_TOL)
    out["mismatched_rows"] += int(bad.sum())
    return out


def xv4r_self_check(R: pd.DataFrame, res: pd.DataFrame, pools: Dict[str, List[str]],
                    kind_of: Dict[str, str]) -> Dict[str, Any]:
    """Per-draw NHB and point counts against II.8i's code (xv3_certified_draws on the same rows; XV3_STATS when in
    memory), and the pooled counts of gumbel_estimated and garnet_all against the published values (XV4R_KNOWN
    mismatches counted in the full paper design, never raised here)."""
    mine = R[~R["distinct"].astype(bool) & (R["type"] == "all")].set_index(["cell", "draw", "targets"])
    cols = {"pairs": "pairs", "certified": "cert_nhb_claims", "certified_errors": "cert_nhb_errors",
            "claims_mean": "point_mean_claims", "claims_terminate": "point_term_claims",
            "errors_mean": "point_mean_errors", "errors_terminate": "point_term_errors"}
    refs = []
    if len(res):   # default rows only: xv3_certified_draws keeps only these, so the counts are unchanged
        Wv3, _ = xv3_bracket_frame(res[xs_as_bool(res["is_default"])], kind_of)
        refs.append(("xv3_certified_draws (II.8i code) on the analysed rows", xv3_certified_draws(Wv3)))
        del Wv3
    st = globals().get("XV3_STATS") or {}
    cc3 = st.get("certified_draws") if isinstance(st, dict) else None
    if cc3 is not None and len(cc3):
        refs.append(("XV3_STATS['certified_draws'] (II.8i, II.8 cells)", cc3))
    out: Dict[str, Any] = {"comparisons": {}, "mismatched_draws": 0}
    for name, cc in refs:
        if cc is None or not len(cc):
            out["comparisons"][name] = {"draws": 0}
            continue
        ref = cc.assign(draw=pd.to_numeric(cc["draw"]).astype(np.int64)).set_index(["cell", "draw", "targets"])
        both, missing = ref.index.intersection(mine.index), ref.index.difference(mine.index)
        bad = np.zeros(len(both), dtype=bool)
        diff = {}
        for c, v in cols.items():
            x = ref.loc[both, c].to_numpy(dtype=float) != mine.loc[both, v].to_numpy(dtype=float)
            diff[c], bad = int(x.sum()), bad | x
        out["comparisons"][name] = {"draws": int(len(both)), "missing_here": int(len(missing)),
                                    "mismatched_draws": int(bad.sum()), "mismatches_by_column": diff}
        out["mismatched_draws"] += int(bad.sum()) + int(len(missing))
    full = RUN_MODE == "paper" and COMPUTE_PROFILE == "full"
    known: Dict[str, Any] = {}
    for key in sorted(set(XV4R_KNOWN) | set(XV4R_KNOWN_REPORTED)):
        tf, pool = key.split(":")
        d = R[(R["targets"] == tf) & ~R["distinct"].astype(bool) & (R["type"] == "all")
              & R["cell"].isin(pools.get(pool, []))]
        s = d.sum(numeric_only=True)
        exp = {**XV4R_KNOWN.get(key, {}), **XV4R_KNOWN_REPORTED.get(key, {})}
        got = {c: (int(s[c]) if c in s.index and len(d) else None) for c in exp}
        design = bool(full and len(d) == XV4R_KNOWN_DRAWS.get(pool, -1))
        asserted = sorted(XV4R_KNOWN.get(key, {})) if design else []
        wrong = [c for c in asserted if got[c] != exp[c]]
        known[key] = {"draws": int(len(d)), "expected": exp, "got": got, "match": got == exp,
                      "asserted_fields": asserted, "asserted_mismatches": wrong}
        if wrong:
            print(f"  II.9b self-check {key}: {', '.join(f'{c} {got[c]} != published {exp[c]}' for c in wrong)}")
    out["known_values"] = known
    out["known_values_asserted"] = any(v["asserted_fields"] for v in known.values())
    # counted into identity_counts by xv4r_run, so the cell raises after the outputs are written (as II.8i)
    out["published_value_mismatches"] = sum(len(v["asserted_mismatches"]) for v in known.values())
    return out


def xv4r_draw_counts(W: pd.DataFrame, iv: pd.DataFrame) -> Dict[str, Any]:
    """Draws present per enabled cell (x_v4_all_specs) and per II.8g bootstrap cell, against the configured draws."""
    have = W.reset_index().groupby("cell")["draw"].nunique().to_dict() if len(W) else {}
    per = {c: {"draws": int(have.get(c, 0)), "configured_draws": int(s["draws"])} for c, s in x_v4_all_specs().items()}
    bh = iv.groupby("cell")["draw"].nunique().to_dict() if len(iv) else {}
    bper = {str(s["cell"]): {"draws": int(bh.get(s["cell"], 0)), "configured_draws": int(s["draws"])}
            for s in globals().get("XV3_BOOT_CELLS", []) or []}
    cells_ok = bool(per) and all(v["draws"] == v["configured_draws"] for v in per.values())
    boot_ok = all(v["draws"] == v["configured_draws"] for v in bper.values())
    return {"draws_per_cell": per, "bootstrap_draws_per_cell": bper, "cells_complete": cells_ok if per else None,
            "bootstrap_complete": boot_ok if bper else None,
            "draws_complete": (cells_ok and boot_ok) if per else None}


def xv4r_run(table_dir: Path) -> Tuple[Dict[str, pd.DataFrame], Dict[str, Any]]:
    """Section II.9b end to end: tables, checks, summary json content. Returns (XV4_ROBUST, XV4_ROBUST_SUMMARY)."""
    t0 = time.perf_counter()
    table_dir = Path(table_dir)
    outputs: List[str] = []

    def write_csv(df: pd.DataFrame, name: str) -> None:
        if name.endswith(".gz"):
            df.to_csv(table_dir / name, index=False, compression="gzip")
        else:
            df.to_csv(table_dir / name, index=False)
        outputs.append(f"xenv/{name}")

    res, boot, inputs = xv4r_inputs()
    kind_of = {str(s["cell"]): str(s["v3_kind"]) for s in (globals().get("X_V3_CELLS") or [])}
    out: Dict[str, pd.DataFrame] = {k: pd.DataFrame() for k in XV4R_FRAMES}
    checks: Dict[str, Any] = {"theorem_counts": {}, "identity_counts": {}}
    summary: Dict[str, Any] = {"inputs": inputs, "value_ranges": XV4R_VRANGE, "tolerance": XV4_TOL,
                               "p_tol": XV4_PTOL, "kappa_grid": list(XV4CFG.kappa_grid),
                               "ci": {"resamples": XV4R_CI_RESAMPLES, "levels": list(XV4R_CI_LEVELS)}}
    W = pd.DataFrame()
    if len(res):
        print(f"II.9b: {inputs['II.6']['rows']} II.6 rows ({inputs['II.6']['source']}), {inputs['II.8']['rows']} "
              f"II.8 rows ({inputs['II.8']['source']})")
        order = [str(c) for c in pd.unique(res["cell"])]
        W, checks["wide"] = xv4r_wide(res, kind_of)
        checks["theorem_counts"] = xv4r_theorem_counts(W)
        R, K, KS, G, checks["skipped_target_sets"] = xv4r_claims(W, XV4CFG.kappa_grid)
        assert len(R) and len(K), "II.9b: every draw was skipped (no proper policy set on any target set)"
        units, pools = xv4r_units(W, order)
        summary["pools"] = pools
        base = R[~R["distinct"].astype(bool) & (R["type"] == "all")].set_index(["cell", "draw", "targets"])
        n_bad = 0
        for kap, col in ((0.0, "cert_nhb_claims"), (1.0, "cert_free_claims")):
            kk = K[K["kappa"] == kap].set_index(["cell", "draw", "targets"])
            if len(kk):
                n_bad += int((kk["claims"].to_numpy() != base[col].reindex(kk.index).to_numpy()).sum())
        checks["identity_counts"]["kappa_endpoint_mismatches"] = n_bad
        checks["self_check"] = xv4r_self_check(R, res, pools, kind_of)
        checks["identity_counts"]["ii8i_certified_count_mismatches"] = checks["self_check"]["mismatched_draws"]
        checks["identity_counts"]["published_value_mismatches"] = checks["self_check"]["published_value_mismatches"]
        del res
        gc.collect()
        t1 = time.perf_counter()
        out["pairs"], out["kappa_draws"] = R, K
        out["summary"] = xv4r_summary_table(R, units)
        out["kappa"] = xv4r_kappa_table(K, KS, units)
        truth_cells = sorted(set(R.loc[(R["targets"] == "bench") & R["cert_nhb_errors"].notna(), "cell"])
                             if "cert_nhb_errors" in R.columns else [])
        out["gapbins"] = xv4r_gap_table(G, units, truth_cells)
        checks["aggregation_s"] = time.perf_counter() - t1
        write_csv(R, "v4_robust_pairs.csv.gz")
        write_csv(out["summary"], "v4_robust_summary.csv")
        write_csv(out["gapbins"], "v4_robust_gapbins.csv")
        write_csv(out["kappa"], "v4_robust_kappa.csv")
        S = out["summary"]
        hd = S[(S["unit_type"] == "pool") & ~S["distinct"].astype(bool) & (S["type"] == "all")]
        keys = ["draws", "pairs", "cert_nhb_share", "cert_nhb_error_rate", "cert_nhb_error_rate_ci_low",
                "cert_nhb_error_rate_ci_high", "cert_free_share", "cert_free_error_rate", "cert_free_error_rate_ci_low",
                "cert_free_error_rate_ci_high", "cert_nhb_only_share", "cert_nhb_only_error_rate", "cert_nhb_cs_share",
                "cert_nhb_cs_error_rate", "cert_free_cs_share", "cert_free_cs_error_rate", "margin_mean_error_rate",
                "margin_term_error_rate", "margin_mid_error_rate", "margin_mean_free_error_rate"]
        summary["headline"] = {u: {tf: {k: r[k] for k in keys} for tf, r in g.set_index("targets").iterrows()}
                               for u, g in hd.groupby("unit", sort=False)}
        kp = out["kappa"][out["kappa"]["unit_type"] == "pool"]
        summary["kappa"] = {
            u: {tf: {"nhb_only_claims": int(g["nhb_only_claims"].iloc[0]),
                     "kappa_star_quantiles": {c: g[c].iloc[0] for c in g.columns if c.startswith("kappa_star_q")},
                     "nhb_only_margin_quantiles": {c: g[c].iloc[0] for c in g.columns
                                                   if c.startswith("nhb_only_margin_")},
                     "by_kappa": {f"{r['kappa']:g}": {k: r[k] for k in ("claims", "share", "cs_share",
                                                                       "share_of_nhb_claims", "errors", "error_rate")}
                                  for _, r in g.iterrows()}}
                for tf, g in gu.groupby("targets", sort=False)}
            for u, gu in kp.groupby("unit", sort=False)}
        gb = out["gapbins"]
        gb = gb[gb["unit_type"] == "pool"] if len(gb) else pd.DataFrame(columns=["unit", "bin"])   # no truth cell
        summary["gap_bins"] = {u: {r["bin"]: {"claims": r["claims"], "error_rate": r["error_rate"]}
                                   for _, r in g.iterrows()} for u, g in gb.groupby("unit", sort=False)}
        del R, K, KS, G
        gc.collect()
    else:
        print("II.9b: no Part II results (XRESULTS, XRESULTS_V3 and their files are missing) - claims not analysed.")
    iv = pd.DataFrame()
    if len(boot):
        iv, checks["bootstrap"] = xv4r_boot_intervals(boot, tuple(XV3BOOT.quantiles))
        del boot
        checks["bootstrap_vs_ii8i"] = xv4r_boot_vs_ii8i(iv)
        border = [str(s["cell"]) for s in globals().get("XV3_BOOT_CELLS", []) or []]
        border += [c for c in pd.unique(iv["cell"]) if c not in border]
        out["v3boot_intervals"] = iv
        out["v3boot_draws"] = xv4r_boot_draws(iv)
        out["v3boot_allpolicy"] = xv4r_boot_table(iv, out["v3boot_draws"], border)
        write_csv(out["v3boot_allpolicy"], "v4_v3boot_allpolicy.csv")
        bp = out["v3boot_allpolicy"]
        summary["v3boot"] = {f"{r['unit']}|{r['interval']}|{r['bound']}": {
            k: r[k] for k in ("coverage_all", "coverage_p_pos", "coverage_p_zero", "certified_share", "error_rate",
                              "cs_share", "draws", "draws_with_wrong_cert", "draws_all_covered",
                              "margin_mid_error_rate")} for _, r in bp[bp["unit_type"] == "pool"].iterrows()}
        bc = checks["bootstrap"]
        checks["theorem_counts"].update({f"bootstrap_{k}": bc[k] for k in (
            "replicate_rule_p_difference", "replicate_negative_width", "replicate_width_above_bound",
            "point_negative_width", "point_free_below_nhb_upper")})
        checks["identity_counts"]["bootstrap_nhb_interval_vs_ii8i"] = checks["bootstrap_vs_ii8i"]["mismatched_rows"]
    else:
        checks["bootstrap"] = {"status": "not run: no II.8g bootstrap rows (xenv_v3_bootstrap off or no cell)"}
    counts = xv4r_draw_counts(W, iv)
    del W
    gc.collect()
    tc, ic = checks["theorem_counts"], checks["identity_counts"]
    checks["passes"] = bool(all(v == 0 for v in tc.values()) and all(v == 0 for v in ic.values()))
    status = ("run" if len(out["pairs"]) or len(iv) else
              "run: no inputs (no Part II results and no II.8g bootstrap rows)")
    summary.update({"status": status, "code_hash": XV4_ROBUST_CODE_HASH, "checks": checks,
                    "draws_complete": counts["draws_complete"], "draw_counts": counts,
                    "outputs": outputs + ["xenv/v4_robust_summary.json"], "elapsed_s": time.perf_counter() - t0})
    return out, summary


XV4R_FRAMES = ("pairs", "summary", "gapbins", "kappa", "kappa_draws", "v3boot_intervals", "v3boot_draws",
               "v3boot_allpolicy")
# Objects whose source changes what II.9b computes, and the module constants they read (by value).
XV4_ROBUST_CODE = [_xv4r_rate, _xv4r_vrange, xv4r_cell_group, xv4r_inputs, xv4r_wide, xv4r_theorem_counts,
                   xv4r_kappa_claims, xv4r_gap_claims, xv4r_draw, xv4r_claims, xv4r_units, xv4r_summary_row,
                   xv4r_cs_headline, xv4r_resample_matrix, xv4r_ratio_ci, xv4r_summary_table, xv4r_kappa_table,
                   xv4r_gap_table, xv4r_boot_intervals, xv4r_interval_claims, xv4r_boot_draws, xv4r_boot_table,
                   xv4r_boot_vs_ii8i, xv4r_self_check, xv4r_draw_counts, xv4r_run]
XV4_ROBUST_CODE_HASH = obj_hash({
    "common": XV4_COMMON_CODE_HASH, "code": code_hash(*XV4_ROBUST_CODE),
    "constants": {"vrange": XV4R_VRANGE, "targets": list(XV4R_TARGETS),
                  "target_sets": {k: list(v) for k, v in XV4R_TARGET_SETS.items()}, "types": list(XV4R_TYPES),
                  "ci_kinds": list(XV4R_CI_KINDS), "ci_resamples": XV4R_CI_RESAMPLES,
                  "ci_levels": list(XV4R_CI_LEVELS), "kstar_q": list(XV4R_KSTAR_Q),
                  "gap_bins": [str(x) for x in XV4R_GAP_BINS], "result_columns": XV4R_RESULT_COLUMNS,
                  "boot_columns": XV4R_BOOT_COLUMNS,
                  "boot_intervals": {f"{a}|{b}": list(v) for (a, b), v in XV4R_BOOT_INTERVALS.items()},
                  "boot_point": XV3_BOOT_POINT, "boot_quantiles": list(XV3BOOT.quantiles),
                  "kappa_grid": list(XV4CFG.kappa_grid), "known": XV4R_KNOWN, "known_draws": XV4R_KNOWN_DRAWS,
                  "known_reported": XV4R_KNOWN_REPORTED, "frames": list(XV4R_FRAMES)}})

XV4_ROBUST: Dict[str, pd.DataFrame] = {k: pd.DataFrame() for k in XV4R_FRAMES}
XV4_ROBUST_SUMMARY: Dict[str, Any] = {"status": f"not run ({XV4R_SWITCH} off)", "code_hash": XV4_ROBUST_CODE_HASH,
                                      "checks": {}, "elapsed_s": 0.0, "draws_complete": None}
if EXPERIMENTS[XV4R_SWITCH]:
    XV4_ROBUST, XV4_ROBUST_SUMMARY = xv4r_run(XENV_TABLE_DIR)
save_json(XV4_ROBUST_SUMMARY, XENV_TABLE_DIR / "v4_robust_summary.json")
print("II.9b code hash:", XV4_ROBUST_CODE_HASH[:16], "| status:", XV4_ROBUST_SUMMARY["status"], "| elapsed:",
      f"{XV4_ROBUST_SUMMARY['elapsed_s']:.1f} s", "| draws complete:", XV4_ROBUST_SUMMARY["draws_complete"])
if len(XV4_ROBUST["summary"]):
    _s = XV4_ROBUST["summary"]
    display(_s[(_s["unit_type"] == "pool") & ~_s["distinct"].astype(bool) & (_s["type"] == "all")][
        ["unit", "targets", "draws", "pairs", "cert_nhb_share", "cert_nhb_error_rate", "cert_nhb_error_rate_ci_low",
         "cert_nhb_error_rate_ci_high", "cert_free_share", "cert_free_error_rate", "cert_nhb_cs_share",
         "cert_free_cs_share", "margin_mid_error_rate"]].round(4))
    _k = XV4_ROBUST["kappa"]
    if len(_k):
        display(_k[(_k["unit_type"] == "pool") & (_k["targets"] == "bench")][
            ["unit", "kappa", "claims", "share", "cs_share", "error_rate", "nhb_only_claims",
             "kappa_star_q50"]].round(4))
    print("II.9b self-check (pooled counts against the published values):",
          json.dumps({k: {"match": v["match"], "draws": v["draws"]} for k, v in
                      XV4_ROBUST_SUMMARY["checks"]["self_check"]["known_values"].items()}))
    del _s, _k
if len(XV4_ROBUST["v3boot_allpolicy"]):
    _b = XV4_ROBUST["v3boot_allpolicy"]
    display(_b[_b["unit_type"] == "pool"][["unit", "interval", "bound", "coverage_all", "coverage_p_pos",
                                           "coverage_p_zero", "certified_share", "error_rate",
                                           "draws_with_wrong_cert", "draws_all_covered"]].round(4))
    del _b
if XV4_ROBUST_SUMMARY.get("checks", {}).get("passes") is False:
    _c = XV4_ROBUST_SUMMARY["checks"]
    raise AssertionError(f"II.9b checks failed: theorem {_c['theorem_counts']}, identity {_c['identity_counts']} "
                         "(xenv/v4_robust_summary.json)")

### II.9c Confidence-set certificates on the true state space

The certificates of Corollary 2 hold for the **given** supported rows, but a benchmark's rows are estimated from
hospital data. II.8g measured this with a bootstrap. This section gives the frequentist alternative: an L1
confidence set for every true row and for d0, built from the raw hospital counts on the **true** state space, and
bounds on J_true that hold for every policy at once on one event E. At scale 1 the union bound gives
P(E) ≥ 1 − δ (δ = `XV4CFG.cs_delta`), so with probability at least 1 − δ **no** NHB-free claim of a draw is
wrong (family-wise), and no NHB claim is wrong when NHB also holds. Cells: `XV4CFG.cs_cells` through
`x_v4_cells` (II.6 and II.8 `best` cells; no aggregated cells).

**Per (cell, draw).**
1. **Policies and truth.** `x_v4_instance`, `x_v4_default_records` (the II.6 default policies) and
   `x_v4_rebuild_benchmark` (hospital episodes and benchmark `B`, asserted equal to the instance's). True-space
   policies are π↑ = `B["lift"](π)`; NHB policies are proj(π)↑ = `B["lift"](x_project(π, M, live))` (uniform
   W, Proposition 3). J_true is exact on the `true` target (T_masked, semantics `given`, its lift), as in
   II.8g; the point-benchmark values J_bench_mean, J_bench_term and p_bench are exact on the benchmark.
2. **Counts** on the true state space (`x_cs_counts`): raw counts C(s, a, s′) of the informative steps of
   **every** logged live state, N(s, a) = Σ_s′ C, q̂ = C / N. Gumbel has a known censoring hazard h and timeout
   state t, so the informative steps are those with s′ ≠ t; Garnet has no hazard and every step is
   informative. No dead-state removal, no trap censoring, no redirect: this is the true system. The empirical
   d0 comes from the first step of each of the n_ep episodes.
3. **Structure of a true row.** Gumbel: P*(·|s, a) = h·δ_t + (1 − h)·q(·|s, a) with q a distribution on the
   states other than t (asserted on the simulator). Garnet: P*(·|s, a) = q(·|s, a) on all states.
4. **Radii** (`x_cs_radii`; Weissman et al. 2003: P(‖q̂_n − q‖₁ ≥ ε) ≤ (2^m − 2)·exp(−n ε²/2)). The k-th
   informative step of a pair is an independent draw from q whatever the history, so a bound for each fixed n
   with δ_rows / (K n (n + 1)) and a union over n ≥ 1 and the K = (true live states) × A pairs gives, with
   δ_rows = δ_d0 = δ/2 and m the number of outcomes of q (S − 1 on Gumbel, S on Garnet),
   r(s, a) = scale·√((2/N)(m ln 2 + ln(K N (N + 1) / δ_rows))) for N ≥ 1, and
   r0 = scale·√((2/n_ep)(m0 ln 2 + ln(1/δ_d0))) for d0 (m0 = true live states; the initial states are
   i.i.d.). r = ∞ (the whole simplex) when N = 0 or r ≥ 2. Only scale = 1 is covered by the union bound; the
   smaller scales of `XV4CFG.cs_scales` are **heuristic** (`set_kind` = `heuristic` in every output).
5. **Inner problems** (`x_cs_l1_extreme`, exact greedy, vectorised over rows). min q·U over
   B₁(q̂, r) ∩ Δ(support) moves δ = min(r/2, 1 − q̂[i*]) onto i* = argmin_support U and removes it from the
   support states with the highest U first; max is symmetric; r = ∞ puts all mass on the extreme support state.
   A self-test compares it with `scipy.optimize.linprog` (both senses, shared and per-row U and supports, tied
   values, every kind of radius).
6. **Monotone robust iteration** (`x_cs_robust_vector`) for a fixed true-space policy π, over the pairs with
   π(a|s) > 0 only: L_0 = v_min on live states (absorbing states hold their entry reward R),
   L_{k+1}(s) = Σ_a π(a|s)[h·R_t + (1 − h)·min_q q·(R + L_k)] on Gumbel, without the h term on Garnet;
   U_0 = v_max with max for the upper vector. On E the true P is a member, its Bellman operator is monotone
   and v_min ≤ V_true ≤ v_max, so **every** iterate satisfies L_k ≤ V_true ≤ U_k. The iteration stops at a
   sup-norm change ≤ `cs_tol` or after `cs_max_iter` steps (a cap only loosens the bound); iterations and the
   converged flag are recorded. Unlogged rows (also outside the decision states V) are unrestricted.
7. **Ends and tests.** L(π) = min_{d∈D} d·L_vec(π↑), U_free(π) = max_{d∈D} d·U_vec(π↑),
   U_proj(π) = max_{d∈D} d·U_vec(proj(π)↑), U_nhb(π) = min(U_free(π), U_proj(π)), D the d0 ball. Under NHB,
   J_true(π↑) ≤ J_true(proj(π)↑) (the mean rule on the true rows, Proposition 3), so U_nhb is an upper bound
   on E under NHB; a policy with L > U_proj (both in the policy table) is evidence against NHB for it on E.
   Tests of an ordered pair (i, j): `free`: L(πi) > U_free(πj); `nhb`: L(πi) > U_nhb(πj); `paired_free`:
   min_{d∈D} d·(L_vec(πi) − U_vec(πj↑)) > 0; `paired_nhb`: the same with the state-wise minimum of the two
   upper vectors of πj (both are upper bounds on V_true under NHB). Margins use the tolerance `XV4_TOL`; a
   claim is wrong if J_true(i) ≤ J_true(j) + `XV4_TOL` (ties are errors). The point-benchmark certificates
   (`point_nhb`: J_term(i) > J_mean(j), `point_free`: J_term(i) > J_term(j) + p_j·v_range) on the same
   policies are the reference; their counts are checked against `x_v4_pair_claims`.
8. State vectors are memoised by policy within a draw and scale (distinct policies are few) and kept in memory
   only.

**What to expect.** The sets are honest but wide. At scale 1 a Gumbel row has m ln 2 ≈ 421 (m = 608 outcomes),
so it stays unrestricted below about 220 informative steps, and r ≈ 0.95 at N = 1 000 and 0.30 at N = 10 000
(Garnet, m = 152: below about 60 steps; r ≈ 0.51 at N = 1 000). The d0 radius is r0 ≈ 0.65 for 2 000 and 0.29
for 10 000 Gumbel episodes (0.21 for 5 000 Garnet episodes). Few certificates are therefore expected at the
valid scale; the heuristic scales show how fast they appear as the sets shrink.

**Invariants asserted per draw:** every raw robust step is monotone (the lower iterates non-decreasing, the
upper non-increasing; slack 1e-12 plus the row-sum rounding of π); L ≤ U_free and U_nhb ≤ U_free for every
policy; the paired tests are at least as strong as the unpaired ones (min_D d·(a − b) ≥ min_D d·a − max_D d·b);
all values lie in [v_min, v_max]; the point claim counts equal `x_v4_pair_claims`. L ≤ J_true ≤ U_free is
**not** asserted: it holds only on E. Coverage and error rates are reported, never asserted.

| Output | Content |
|---|---|
| `xenv/v4_cs_policies.csv.gz` · `XV4_CS_POLICIES` | per draw × default policy × scale: L, U_free, U_proj, U_nhb, J_true, point values, p_bench, iterations, convergence, rows used |
| `xenv/v4_cs_draws.csv` · `XV4_CS_DRAWS` | per draw × scale × test (four set tests + two point tests): pairs, claims and errors (all, AA, AU, UU, completion-sensitive), coverage of the test's interval, any-wrong / all-covered flags, radius statistics (median r over the used pairs, share unrestricted, r0) |
| `xenv/v4_cs_summary.csv` · `XV4_CS_TABLE` | per cell, family pool and pooled × scale × test: claims, share, errors, error rate (also on completion-sensitive pairs), draws with a wrong claim / an uncovered policy, coverage, mean width |
| `xenv/v4_cs_summary.json` · `XV4_CS_SUMMARY` | status, code hash, configuration, self-test, invariant checks, headline numbers, timing |
| `XV4_CS_META` (memory only) | per draw meta (flattened): counts, radii, iterations, slacks, timings (`timing.total_s` enters the compute totals of the release) |

Results are cached under `v4cs/<cell>/draw<k>` (and `/meta`); the identity is the draw identity
(`x_v4_draw_identity`), this section's code hash, δ, the scales, `cs_tol` and `cs_max_iter`.

**Expected cost on Colab** (an estimate: local timings of the robust iteration on Gumbel- and Garnet-sized
synthetic count models, scaled by the Colab / local ratio of II.8f). Each draw rebuilds the instance and runs one
`run_xmethods` fit, as an II.8g draw does (about 5–10 s for Gumbel, 3–4 s for Garnet), plus three exact
evaluations per distinct default policy. One robust iteration takes about 0.4 ms for a deterministic Gumbel
policy (606 used rows of 609 states) and 3 ms for a fully stochastic one (4 848 rows); Garnet (152 states, ≤ 6
successors per row) about 0.1–0.3 ms. The adversary moves mass onto death (lower) or discharge / goal (upper), so
the iterations stopped after 3–25 steps in those tests; on Gumbel the timeout hazard caps them at about 360
(contraction 1 − h, h = 0.05). With 50–100 robust vectors per draw and scale this adds a few seconds per draw
(under a minute in the worst case). In total roughly 10–20 s per Gumbel draw and 5 s per Garnet draw, about
30–60 min for 3 Gumbel and 4 Garnet cells × 30 draws; a cached draw reloads in about 3 s (instance rebuild).

In [ ]:
# ============================================================
# II.9c Confidence-set certificates: definitions
# ============================================================
EXECLOG.mark("II.9c Confidence-set certificates: definitions")

XV4_CS_TESTS = ("free", "nhb", "paired_free", "paired_nhb")      # certificates from the confidence set
XV4_CS_POINT_TESTS = ("point_nhb", "point_free")                # point-benchmark reference (Corollary 2)
XV4_CS_INTERVALS = {"free": "[L, U_free]", "nhb": "[L, U_nhb]", "paired_free": "[L, U_free]",
                    "paired_nhb": "[L, U_nhb]", "point_nhb": "[J_bench_term, J_bench_mean]",
                    "point_free": "[J_bench_term, J_bench_term + p_bench v_range]"}
XV4_CS_MONO_TOL = 1e-12      # slack of the monotone-step assertion (plus the row-sum rounding of the policy)
XV4_CS_PAIR_TOL = 1e-10      # slack of 'paired >= unpaired' (both sides are sums of ~600 rounded products)
XV4_CS_COLUMNS = ["cell", "family", "draw", "policy", "method", "variant", "hp", "method_family", "scale", "set_kind",
                  "J_true", "J_bench_mean", "J_bench_term", "p_bench", "L", "U_free", "U_proj", "U_nhb",
                  "proj_is_same", "covered_free", "covered_nhb", "iter_lower", "iter_upper", "iter_upper_proj",
                  "converged_lower", "converged_upper", "converged_upper_proj", "rows_used", "rows_unrestricted"]
XV4_CS_FLOAT_COLUMNS = ["scale", "J_true", "J_bench_mean", "J_bench_term", "p_bench", "L", "U_free", "U_proj",
                        "U_nhb"]


# ---------- the inner problem: extreme of q.U over an L1 ball intersected with a simplex ----------
def x_cs_pack(q: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Packed rows of a non-negative [n, S] matrix: (vals [n, m], cols [n, m]) hold the positive entries of each row
    in column order, padded with zeros (column 0); m = max(1, largest row support)."""
    q = np.atleast_2d(np.asarray(q, dtype=np.float64))
    assert q.ndim == 2 and (q >= 0).all(), "rows must be non-negative"
    nz = q > 0
    cnt = nz.sum(axis=1)
    m = max(1, int(cnt.max(initial=0)))
    rows, cols = np.nonzero(nz)
    pos = np.arange(len(rows)) - (np.cumsum(cnt) - cnt)[rows]
    vals_p, cols_p = np.zeros((q.shape[0], m)), np.zeros((q.shape[0], m), dtype=np.int64)
    vals_p[rows, pos] = q[rows, cols]
    cols_p[rows, pos] = cols
    return vals_p, cols_p


def _x_cs_extreme_packed(vals: np.ndarray, cols: np.ndarray, r: np.ndarray, U: np.ndarray, support: np.ndarray,
                         sense: str, check: bool = True) -> np.ndarray:
    """min ('min') or max ('max') of q.U over {q in simplex(support) : ||q - qhat||_1 <= r} for every packed row.

    vals / cols: [n, m] (or [1, m], shared by all rows) packed qhat (x_cs_pack), inside the support; r: [n] radii
    (inf = unrestricted); U: [S] (shared) or [n, S]; support: [S] or [n, S] boolean. Exact greedy: move
    delta = min(r/2, 1 - qhat[i*]) onto i*, the support state with the lowest U (highest for 'max'), and remove it
    from the other states with the highest U first (lowest for 'max'); r = inf puts all mass on i*."""
    assert sense in ("min", "max"), sense
    sign = 1.0 if sense == "min" else -1.0
    r = np.asarray(r, dtype=np.float64).reshape(-1)
    n = len(r)
    vals = np.broadcast_to(np.asarray(vals, dtype=np.float64), (n, np.shape(vals)[-1]))
    cols = np.broadcast_to(np.asarray(cols, dtype=np.int64), (n, np.shape(cols)[-1]))
    W = sign * np.asarray(U, dtype=np.float64)
    sup = np.asarray(support, dtype=bool)
    if W.ndim == 1 and sup.ndim == 1:                      # one value vector and one support for every row
        Wm = np.where(sup, W, np.inf)
        istar = int(Wm.argmin())
        wstar = np.full(n, Wm[istar])
        Wc, hit = W[cols], cols == istar
        sup_c = sup[cols] if check else None
    else:
        W2 = np.broadcast_to(W, (n, W.shape[-1]))
        sup2 = np.broadcast_to(sup, (n, sup.shape[-1]))
        Wm = np.where(sup2, W2, np.inf)
        istar = Wm.argmin(axis=1)
        wstar = Wm[np.arange(n), istar]
        Wc, hit = np.take_along_axis(W2, cols, axis=1), cols == istar[:, None]
        sup_c = np.take_along_axis(sup2, cols, axis=1) if check else None
    assert np.isfinite(wstar).all(), "every row needs a non-empty support and finite values"
    fin = np.isfinite(r)
    if check:
        assert (r >= 0).all() and (vals >= 0).all(), "radii and qhat must be non-negative"
        assert not ((vals > 0) & ~sup_c).any(), "qhat must lie inside the support"
        assert np.allclose(vals[fin].sum(axis=1), 1.0, atol=1e-9), "a row with a finite radius needs a distribution"
    out = wstar.copy()                                     # unrestricted rows: all mass on i*
    if fin.any():
        f = slice(None) if fin.all() else fin
        v, w, h = vals[f], Wc[f], hit[f]
        qstar = np.where(h, v, 0.0).sum(axis=1)
        delta = np.clip(np.minimum(r[f] / 2.0, 1.0 - qstar), 0.0, None)
        rem = np.where(h, 0.0, v)
        order = np.argsort(-w, axis=1, kind="stable")     # highest (signed) value first
        rem_s, w_s = np.take_along_axis(rem, order, axis=1), np.take_along_axis(w, order, axis=1)
        take = np.clip(delta[:, None] - (np.cumsum(rem_s, axis=1) - rem_s), 0.0, rem_s)
        out[f] = ((rem_s - take) * w_s).sum(axis=1) + (qstar + delta) * wstar[f]
    return sign * out


def x_cs_l1_extreme(qhat: np.ndarray, r, U: np.ndarray, support_mask: np.ndarray, sense: str) -> np.ndarray:
    """min / max of q.U over B_1(qhat, r) ∩ simplex(support_mask), one value per row (vectorised greedy).

    qhat: [S] or [n, S] rows inside the support (a row with r = inf may be all zero: N = 0); r: scalar or [n]
    (inf = unrestricted); U: [S] or [n, S]; support_mask: [S] or [n, S]. A single qhat row is shared by the n rows of U.
    Used for the true rows and for d0."""
    q = np.atleast_2d(np.asarray(qhat, dtype=np.float64))
    U = np.asarray(U, dtype=np.float64)
    n = max(q.shape[0], U.shape[0] if U.ndim == 2 else 1, np.size(r),
            np.shape(support_mask)[0] if np.ndim(support_mask) == 2 else 1)
    assert q.shape[0] in (1, n) and np.size(r) in (1, n), (q.shape, np.shape(r), U.shape)
    vals, cols = x_cs_pack(q)
    rr = np.broadcast_to(np.asarray(r, dtype=np.float64).reshape(-1), (n,))
    return _x_cs_extreme_packed(vals, cols, rr, U, support_mask, sense)


# ---------- counts, radii and the monotone robust iteration ----------
def x_cs_counts(true_env: FiniteEnv, hosp: Dict[str, np.ndarray]) -> Dict[str, Any]:
    """Raw hospital counts on the true state space: packed empirical rows qhat of every (s, a) over the informative
    steps (every step without a known hazard; the steps that do not enter the timeout state with one), N(s, a), the
    empirical d0 of the episodes' first states, and the structure of a true row (support of q, hazard)."""
    S, A, live = true_env.S, true_env.A, np.asarray(true_env.live, dtype=bool)
    s, a, ns = (np.asarray(hosp[k], dtype=np.int64) for k in ("state", "action", "next_state"))
    assert len(s) > 0 and live[s].all(), "logged states must be live"
    h = true_env.meta["censoring_hazard"]
    support = np.ones(S, dtype=bool)                       # possible outcomes of q
    t, r_t = None, 0.0
    if h is not None:
        h, t = float(h), int(true_env.meta["timeout_state"])
        assert 0.0 < h < 1.0 and t in true_env.terminals, (h, t)
        assert np.all(true_env.P[live, :, t] == h), "true rows must be h e_timeout + (1 - h) q (II.3b)"
        support[t] = False
        r_t = float(true_env.entry_reward[t])
    inf = np.ones(len(s), dtype=bool) if h is None else ns != t
    key, cnt = np.unique((s[inf] * A + a[inf]) * S + ns[inf], return_counts=True)
    row, col = key // S, key % S
    assert support[col].all(), "an informative step left the support of q"
    N = np.bincount(row, weights=cnt, minlength=S * A).astype(np.float64)
    nnz = np.bincount(row, minlength=S * A)
    m = max(1, int(nnz.max(initial=0)))
    pos = np.arange(len(row)) - (np.cumsum(nnz) - nnz)[row]   # np.unique sorts by (row, col)
    vals, cols = np.zeros((S * A, m)), np.zeros((S * A, m), dtype=np.int64)
    vals[row, pos] = cnt / N[row]
    cols[row, pos] = col
    first = np.asarray(hosp["timestep"]) == 0
    n_ep = int(first.sum())
    assert n_ep == len(np.unique(hosp["episode_id"])), "every episode has exactly one first step"
    d0hat = np.bincount(s[first], minlength=S).astype(np.float64) / n_ep
    lo, hi = true_env.value_bounds
    n_live = int(live.sum())
    return {"S": int(S), "A": int(A), "live": live.copy(), "R": np.asarray(true_env.entry_reward, dtype=np.float64),
            "hazard": h, "timeout": t, "R_timeout": r_t, "support": support, "vals": vals, "cols": cols, "N": N,
            "d0hat": d0hat, "n_ep": n_ep, "v_min": float(lo), "v_max": float(hi), "K": n_live * int(A),
            "m": int(support.sum()), "m0": n_live, "steps": int(len(s)), "informative_steps": int(inf.sum()),
            "max_row_support": int(nnz.max(initial=0))}


def x_cs_radii(model: Dict[str, Any], scale: float, delta: float) -> Tuple[np.ndarray, float]:
    """Radii of the row sets of every (s, a) (Weissman L1 bound, anytime over N, union over the K live pairs, delta/2)
    and of the d0 set (delta/2), times `scale`; inf where N = 0 or where the radius reaches 2 (whole simplex)."""
    assert scale > 0.0 and 0.0 < delta < 1.0, (scale, delta)
    N = np.asarray(model["N"], dtype=np.float64)
    d_rows, d_d0 = delta / 2.0, delta / 2.0
    r = np.full(N.shape, np.inf)
    pos = N >= 1.0
    n = N[pos]
    r[pos] = scale * np.sqrt((2.0 / n) * (model["m"] * math.log(2.0) + np.log(model["K"] * n * (n + 1.0) / d_rows)))
    r[r >= 2.0] = np.inf
    n0 = int(model["n_ep"])
    r0 = (scale * math.sqrt((2.0 / n0) * (model["m0"] * math.log(2.0) + math.log(1.0 / d_d0))) if n0 >= 1
          else math.inf)
    return r, (r0 if r0 < 2.0 else math.inf)


def x_cs_robust_vector(model: Dict[str, Any], r: np.ndarray, pi: np.ndarray, sense: str, tol: float,
                       max_iter: int) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Monotone robust iteration of one true-space policy over the row sets with radii r (every (s, a)).

    'min': L_0 = v_min on live states, L_{k+1}(s) = sum_a pi(a|s) [h R_t + (1 - h) min_q q.(R + L_k)] (no h term
    without a known hazard); 'max': U_0 = v_max and max. Only the pairs with pi(a|s) > 0 enter. Returns the state
    vector (0 on absorbing states) and the iteration report. Every raw step must be monotone (asserted, slack
    XV4_CS_MONO_TOL plus the row-sum rounding of pi); the returned sequence is the exact monotone envelope."""
    assert sense in ("min", "max"), sense
    S, A, live = model["S"], model["A"], model["live"]
    pi = np.asarray(pi, dtype=np.float64)
    assert pi.shape == (S, A), pi.shape
    ss, aa = np.nonzero((pi > 0.0) & live[:, None])
    assert np.array_equal(np.unique(ss), np.flatnonzero(live)), "every live state needs an action with pi > 0"
    rr = np.asarray(r, dtype=np.float64)[ss * A + aa]
    o = np.argsort(~np.isfinite(rr), kind="stable")        # restricted rows first, then the unrestricted ones
    ss, aa, rr = ss[o], aa[o], rr[o]
    rows, w, nf = ss * A + aa, pi[ss, aa], int(np.isfinite(rr).sum())
    vals, cols = model["vals"][rows[:nf]], model["cols"][rows[:nf]]
    width = max(1, int((vals > 0).sum(axis=1).max(initial=0)))     # packed rows are left-aligned (x_cs_counts)
    vals, cols = vals[:, :width], cols[:, :width]
    pad_v, pad_c = np.zeros((1, 1)), np.zeros((1, 1), dtype=np.int64)  # unrestricted rows need no qhat
    R, sup, h = model["R"], model["support"], model["hazard"]
    _x_cs_extreme_packed(vals, cols, rr[:nf], R, sup, sense, check=True)     # validate the packed rows once
    sign = 1.0 if sense == "min" else -1.0
    V = np.where(live, model["v_min"] if sense == "min" else model["v_max"], 0.0)
    dev = float(np.abs(pi[live].sum(axis=1) - 1.0).max())
    mono_tol = XV4_CS_MONO_TOL + dev * max(abs(model["v_min"]), abs(model["v_max"]))
    worst, change, converged, it = 0.0, math.inf, False, 0
    for it in range(1, int(max_iter) + 1):
        X = R + V
        x = np.concatenate([_x_cs_extreme_packed(vals, cols, rr[:nf], X, sup, sense, check=False),
                            _x_cs_extreme_packed(pad_v, pad_c, rr[nf:], X, sup, sense, check=False)])
        if h is not None:
            x = h * model["R_timeout"] + (1.0 - h) * x
        new = np.where(live, np.bincount(ss, weights=w * x, minlength=S), 0.0)
        back = float((sign * (V - new))[live].max())    # > 0: a step against the monotone direction
        worst = max(worst, back)
        assert back <= mono_tol, f"robust {sense} iterate is not monotone at iteration {it} ({back:.3g})"
        new = np.maximum(new, V) if sense == "min" else np.minimum(new, V)
        change = float(np.abs(new - V).max())
        V = new
        if change <= tol:
            converged = True
            break
    return V, {"iterations": int(it), "converged": bool(converged), "last_change": change,
               "monotone_slack": worst, "rows": int(len(rows)), "rows_unrestricted": int((~np.isfinite(rr)).sum())}


# ---------- claims, coverage and the per-draw driver ----------
def x_cs_pair_stats(C: np.ndarray, u: np.ndarray, jt: np.ndarray) -> Dict[str, Any]:
    """Pairs, claims and errors (truth reversed or tied) of one claim matrix C[i, j] = 'i beats j', over all pairs and
    by type: AA (neither policy can leave the support), AU, UU, and sens = AU + UU (completion-sensitive)."""
    u = np.asarray(u, dtype=bool)
    jt = np.asarray(jt, dtype=np.float64)
    off = ~np.eye(len(u), dtype=bool)
    nu = u[:, None].astype(np.int64) + u[None, :].astype(np.int64)
    wrong = ~_x_v4_gt(jt, jt)
    C = np.asarray(C, dtype=bool) & off
    out: Dict[str, Any] = {}
    for sfx, tm in (("", off), ("_AA", off & (nu == 0)), ("_AU", off & (nu == 1)), ("_UU", off & (nu == 2)),
                    ("_sens", off & (nu >= 1))):
        out[f"pairs{sfx}"] = int(tm.sum() // 2)
        out[f"claims{sfx}"] = int((C & tm).sum())
        out[f"errors{sfx}"] = int((C & tm & wrong).sum())
    out["any_wrong"] = bool(out["errors"] > 0)
    return out


def x_cs_interval_stats(lo: np.ndarray, up: np.ndarray, jt: np.ndarray) -> Dict[str, Any]:
    """Coverage of J_true by the intervals [lo, up] of one policy set (tolerance XV4_TOL) and their mean width."""
    lo, up, jt = (np.asarray(x, dtype=np.float64) for x in (lo, up, jt))
    lo_ok, up_ok = lo <= jt + XV4_TOL, jt <= up + XV4_TOL
    cov = lo_ok & up_ok
    return {"n_policies": int(len(jt)), "n_covered": int(cov.sum()), "coverage": float(cov.mean()),
            "all_covered": bool(cov.all()), "lower_misses": int((~lo_ok).sum()), "upper_misses": int((~up_ok).sum()),
            "mean_width": float((up - lo).mean())}


def x_cs_identity(spec: dict, inst: CellInstance) -> dict:
    return {"kind": "xenv_v4_confidence", "draw": x_v4_draw_identity(spec, inst), "code": XV4_CS_CODE_HASH,
            "delta": float(XV4CFG.cs_delta), "scales": [float(x) for x in XV4CFG.cs_scales],
            "tol": float(XV4CFG.cs_tol), "max_iter": int(XV4CFG.cs_max_iter), "columns": XV4_CS_COLUMNS,
            "policies": "default (is_default) records of x_collect_records"}


def x_cs_check_rows(df: pd.DataFrame, v_min: float, v_max: float) -> Dict[str, Any]:
    """Invariants of one draw's confidence-set rows (asserted; re-run on cached tables) and a small report."""
    assert not df.duplicated(["policy", "scale"]).any(), "duplicated (policy, scale) confidence-set rows"
    L, Uf, Up, Un = (df[c].to_numpy(dtype=float) for c in ("L", "U_free", "U_proj", "U_nhb"))
    assert np.isfinite(np.c_[L, Uf, Up, Un]).all(), "non-finite confidence-set bound"
    gap = float((L - Uf).max(initial=-np.inf))
    assert gap <= XV4_TOL, f"L > U_free by {gap:.3g}"
    # exact in memory; XV4_TOL only guards a cached table against a float formatting change of the csv round trip
    assert float(np.abs(Un - np.minimum(Uf, Up)).max(initial=0.0)) <= XV4_TOL, "U_nhb must be min(U_free, U_proj)"
    below = float((v_min - L).max(initial=-np.inf))
    above = float((np.r_[Uf, Up] - v_max).max(initial=-np.inf))
    assert below <= XV4_TOL and above <= XV4_TOL, f"bound outside [v_min, v_max] ({below:.3g}, {above:.3g})"
    conv = np.stack([xs_as_bool(df[c]).to_numpy(dtype=bool)
                     for c in ("converged_lower", "converged_upper", "converged_upper_proj")], axis=1)
    it = df[["iter_lower", "iter_upper", "iter_upper_proj"]].to_numpy(dtype=np.int64)
    return {"rows": int(len(df)), "max_lower_minus_upper": gap,
            "max_nhb_minus_free": float((Un - Uf).max(initial=-np.inf)), "max_below_v_min": below,
            "max_above_v_max": above, "nonconverged_rows": int((~conv.all(axis=1)).sum()),
            "max_iterations": int(it.max(initial=0))}


def run_v4_cs_draw(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
    """Confidence-set bounds and certificates of the default policies of one draw (cached under v4cs/). The meta
    holds the per-scale x test rows under 'tests'."""
    cell, family, draw = spec["cell"], spec["family"], int(draw)
    assert family in ("gumbel", "garnet"), family
    t0 = time.perf_counter()
    inst = x_v4_instance(spec, draw)
    identity = x_cs_identity(spec, inst)
    key, ident_hash = "v4cs/" + x_cache_key(cell, draw), obj_hash(identity)[:20]
    build_s = time.perf_counter() - t0
    hit, meta_hit = XCACHE.get(key, identity), XCACHE.get(key + "/meta", identity)
    if hit is not None and meta_hit is not None:
        del inst
        assert list(hit.columns) == XV4_CS_COLUMNS, f"{key}: cached columns differ from XV4_CS_COLUMNS"
        meta = json.loads(str(meta_hit["meta_json"].iloc[0]))
        meta.update(cache="loaded", session_build_s=build_s,
                    checks=x_cs_check_rows(hit, meta["v_min"], meta["v_max"]))
        gc.collect()
        return hit, meta
    if (hit is None) != (meta_hit is None):
        print(f"  {key}: incomplete cache entry (results or meta missing) - recomputing this draw")
    XCACHE.require_allowed(key)
    learn = inst.learn_env
    t1 = time.perf_counter()
    records, keys = x_v4_default_records(inst)
    t2 = time.perf_counter()
    true_env, _, hosp, B = x_v4_rebuild_benchmark(spec, draw, inst)
    tgt = {lab: (env, sem, lift) for lab, env, sem, lift in inst.targets}
    T_true, sem_true, lift_true = tgt["true"]
    assert sem_true == "given" and lift_true is not None and T_true.S == true_env.S, cell
    model = x_cs_counts(true_env, hosp)
    del hosp
    v_min, v_max, live = model["v_min"], model["v_max"], model["live"]
    ev: Dict[bytes, Dict[str, Any]] = {}
    for rec, k in zip(records, keys):
        if k in ev:
            continue
        pi_true, pi_proj = B["lift"](rec["pi"]), B["lift"](x_project(rec["pi"], learn.mask, learn.live))
        assert np.array_equal(pi_true, lift_true(rec["pi"])), f"{rec['label']}: rebuilt lift != instance lift"
        e_true = x_v4_eval(T_true, pi_true, "given")
        e_mean, e_term = x_v4_eval(learn, rec["pi"], "given"), x_v4_eval(learn, rec["pi"], "terminate")
        bad = [(lab, e["status"]) for lab, e in (("true", e_true), ("bench_mean", e_mean), ("bench_term", e_term))
               if e["status"] != "ok"]
        assert not bad, f"{cell} draw {draw}: {rec['label']} is not proper on {bad}"
        ev[k] = {"J_true": e_true["return"], "J_bench_mean": e_mean["return"], "J_bench_term": e_term["return"],
                 "p_bench": e_mean["prob_any_violation"], "pi_true": pi_true, "pi_proj": pi_proj,
                 "proj_is_same": bool(np.array_equal(pi_true, pi_proj))}
    pols = [ev[k] for k in keys]
    n = len(pols)
    jt, jm, jterm, pb = (np.array([p[c] for p in pols]) for c in ("J_true", "J_bench_mean", "J_bench_term", "p_bench"))
    u = pb > XV4_PTOL
    used = np.zeros(model["S"] * model["A"], dtype=bool)
    for p in ev.values():
        for pi in (p["pi_true"], p["pi_proj"]):
            ss, aa = np.nonzero((pi > 0.0) & live[:, None])
            used[ss * model["A"] + aa] = True
    t3 = time.perf_counter()
    base = {"cell": cell, "family": family, "draw": draw}
    rows, tests, scales = [], [], {}
    for scale in XV4CFG.cs_scales:
        t_s = time.perf_counter()
        scale = float(scale)
        kind = "valid" if scale == 1.0 else "heuristic"
        r, r0 = x_cs_radii(model, scale, XV4CFG.cs_delta)
        memo: Dict[Tuple[bytes, str], Tuple[np.ndarray, Dict[str, Any]]] = {}
        got: Dict[str, List[Tuple[np.ndarray, Dict[str, Any]]]] = {"lower": [], "upper": [], "proj": []}
        for p in pols:
            for name, pi, sense in (("lower", p["pi_true"], "min"), ("upper", p["pi_true"], "max"),
                                    ("proj", p["pi_proj"], "max")):
                mk = (_x_pi_key(pi), sense)
                if mk not in memo:
                    memo[mk] = x_cs_robust_vector(model, r, pi, sense, XV4CFG.cs_tol, XV4CFG.cs_max_iter)
                got[name].append(memo[mk])
        Lv, Ufv, Upv = (np.stack([v for v, _ in got[nm]]) for nm in ("lower", "upper", "proj"))
        Unv = np.minimum(Ufv, Upv)
        L = x_cs_l1_extreme(model["d0hat"], r0, Lv, live, "min")
        Uf = x_cs_l1_extreme(model["d0hat"], r0, Ufv, live, "max")
        Up = x_cs_l1_extreme(model["d0hat"], r0, Upv, live, "max")
        Un = np.minimum(Uf, Up)
        PF = np.stack([x_cs_l1_extreme(model["d0hat"], r0, Lv[i][None, :] - Ufv, live, "min") for i in range(n)])
        PN = np.stack([x_cs_l1_extreme(model["d0hat"], r0, Lv[i][None, :] - Unv, live, "min") for i in range(n)])
        gap = float((L - Uf).max())
        assert gap <= XV4_TOL and (Un <= Uf).all(), f"{cell} draw {draw} scale {scale}: L > U_free by {gap:.3g}"
        assert (L >= v_min - XV4_TOL).all() and (Uf <= v_max + XV4_TOL).all(), "bound outside [v_min, v_max]"
        pair_slack = min(float((PF - (L[:, None] - Uf[None, :])).min()), float((PN - (L[:, None] - Un[None, :])).min()))
        assert pair_slack >= -XV4_CS_PAIR_TOL, f"a paired test is weaker than the unpaired one ({pair_slack:.3g})"
        claims = {"free": _x_v4_gt(L, Uf), "nhb": _x_v4_gt(L, Un), "paired_free": PF > XV4_TOL,
                  "paired_nhb": PN > XV4_TOL}
        ivs = {"free": (L, Uf), "nhb": (L, Un), "paired_free": (L, Uf), "paired_nhb": (L, Un)}
        ru = r[used]
        rad = {"r0": float(r0), "used_pairs": int(used.sum()),
               "radius_median": float(np.median(ru)) if len(ru) else float("nan"),
               "radius_median_finite": float(np.median(ru[np.isfinite(ru)])) if np.isfinite(ru).any() else float("nan"),
               "share_unrestricted": float((~np.isfinite(ru)).mean()) if len(ru) else float("nan")}
        for name in XV4_CS_TESTS:
            tests.append({**base, "scale": scale, "set_kind": kind, "test": name, "interval": XV4_CS_INTERVALS[name],
                          **x_cs_interval_stats(*ivs[name], jt), **x_cs_pair_stats(claims[name], u, jt), **rad})
        cov_f, cov_n = (L <= jt + XV4_TOL) & (jt <= Uf + XV4_TOL), (L <= jt + XV4_TOL) & (jt <= Un + XV4_TOL)
        for i, (rec, p) in enumerate(zip(records, pols)):
            (_, il), (_, iu), (_, ip) = got["lower"][i], got["upper"][i], got["proj"][i]
            d = rec["desc"]
            rows.append({**base, "policy": d["policy"], "method": d["method"], "variant": d["variant"], "hp": d["hp"],
                         "method_family": d["method_family"], "scale": scale, "set_kind": kind, "J_true": jt[i],
                         "J_bench_mean": jm[i], "J_bench_term": jterm[i], "p_bench": pb[i], "L": L[i], "U_free": Uf[i],
                         "U_proj": Up[i], "U_nhb": Un[i], "proj_is_same": p["proj_is_same"],
                         "covered_free": bool(cov_f[i]), "covered_nhb": bool(cov_n[i]),
                         "iter_lower": il["iterations"], "iter_upper": iu["iterations"],
                         "iter_upper_proj": ip["iterations"], "converged_lower": il["converged"],
                         "converged_upper": iu["converged"], "converged_upper_proj": ip["converged"],
                         "rows_used": il["rows"], "rows_unrestricted": il["rows_unrestricted"]})
        infos = [i_ for _, i_ in memo.values()]
        scales[str(scale)] = {**rad, "set_kind": kind, "robust_runs": len(memo),
                              "max_iterations": int(max(i_["iterations"] for i_ in infos)),
                              "mean_iterations": float(np.mean([i_["iterations"] for i_ in infos])),
                              "nonconverged_runs": int(sum(not i_["converged"] for i_ in infos)),
                              "max_monotone_slack": float(max(i_["monotone_slack"] for i_ in infos)),
                              "min_paired_minus_unpaired": pair_slack, "seconds": time.perf_counter() - t_s}
        del memo, got, Lv, Ufv, Upv, Unv, PF, PN
    v_range = v_max - v_min
    ref = {x["type"]: x for x in x_v4_pair_claims(jterm, jm, pb, v_range, jt)}
    for name, kind_ref, up in (("point_nhb", "cert_nhb", jm), ("point_free", "cert_free", jterm + pb * v_range)):
        st = x_cs_pair_stats(_x_v4_gt(jterm, up), u, jt)
        for ty, sfx in (("all", ""), ("AA", "_AA"), ("AU", "_AU"), ("UU", "_UU")):
            assert (st[f"claims{sfx}"], st[f"errors{sfx}"]) == (ref[ty][f"{kind_ref}_claims"],
                                                                 ref[ty][f"{kind_ref}_errors"]), (name, ty)
        tests.append({**base, "scale": float("nan"), "set_kind": "point", "test": name,
                      "interval": XV4_CS_INTERVALS[name], **x_cs_interval_stats(jterm, up, jt), **st,
                      "r0": float("nan"), "used_pairs": int(used.sum()), "radius_median": float("nan"),
                      "radius_median_finite": float("nan"), "share_unrestricted": float("nan")})
    t4 = time.perf_counter()
    df = pd.DataFrame(rows, columns=XV4_CS_COLUMNS)
    checks = x_cs_check_rows(df, v_min, v_max)
    meta = _x_compact({
        "cell": cell, "family": family, "draw": draw, "identity_hash": ident_hash, "code_hash": XV4_CS_CODE_HASH[:16],
        "cache": "computed", "v_min": v_min, "v_max": v_max, "hazard": model["hazard"], "n_policies": n,
        "unique_policies": len(ev), "true_states": model["S"], "true_live_states": model["m0"],
        "counts": {"steps": model["steps"], "informative_steps": model["informative_steps"],
                   "hospital_episodes": model["n_ep"], "logged_pairs": int((model["N"] > 0).sum()),
                   "max_row_support": model["max_row_support"], "outcomes_m": model["m"], "union_pairs_K": model["K"]},
        "scales": scales,
        "max_monotone_slack": float(max(v["max_monotone_slack"] for v in scales.values())),
        "min_paired_minus_unpaired": float(min(v["min_paired_minus_unpaired"] for v in scales.values())),
        "nonconverged_runs": int(sum(v["nonconverged_runs"] for v in scales.values())), "checks": checks,
        "timing": {"build_s": build_s, "methods_s": t2 - t1, "rebuild_eval_s": t3 - t2, "cs_s": t4 - t3,
                   "total_s": time.perf_counter() - t0}})
    meta["tests"] = tests                     # kept whole (not compacted): the per-draw x scale x test rows
    XCACHE.put(key + "/meta", identity, pd.DataFrame({"meta_json": [json.dumps(meta, default=_json_default)]}))
    XCACHE.put(key, identity, df)
    back = XCACHE.get(key, identity)
    assert back is not None and list(back.columns) == XV4_CS_COLUMNS and len(back) == len(df), f"{key}: cache write"
    for c in XV4_CS_FLOAT_COLUMNS:
        assert np.allclose(back[c].to_numpy(dtype=float), df[c].to_numpy(dtype=float), rtol=1e-14, atol=0.0,
                           equal_nan=True), f"{key}: cache round trip changed column {c}"
    del inst, records, ev, pols, model, df, rows
    gc.collect()
    return back, meta


# Objects whose source changes what a confidence-set draw computes (the row checks are re-run on cached tables).
XV4_CS_CODE = [x_cs_pack, _x_cs_extreme_packed, x_cs_l1_extreme, x_cs_counts, x_cs_radii, x_cs_robust_vector,
               x_cs_pair_stats, x_cs_interval_stats, x_cs_identity, run_v4_cs_draw]
XV4_CS_CODE_HASH = obj_hash({"common": XV4_COMMON_CODE_HASH, "code": code_hash(*XV4_CS_CODE),
                             "constants": {"tests": list(XV4_CS_TESTS), "point_tests": list(XV4_CS_POINT_TESTS),
                                           "intervals": XV4_CS_INTERVALS, "mono_tol": XV4_CS_MONO_TOL,
                                           "pair_tol": XV4_CS_PAIR_TOL, "columns": XV4_CS_COLUMNS,
                                           "float_columns": XV4_CS_FLOAT_COLUMNS}})


# ---------- self-test of the inner problem, summaries and the cell runner (analysis only, not hashed) ----------
def _x_cs_lp_reference(qhat: np.ndarray, r: float, U: np.ndarray, sup: np.ndarray, sense: str) -> float:
    """The inner problem as a linear program: variables (q, t), |q - qhat| <= t, sum t <= r, q in simplex(sup)."""
    S = len(qhat)
    c = np.r_[(1.0 if sense == "min" else -1.0) * U, np.zeros(S)]
    I = np.eye(S)
    A_ub = np.r_[np.c_[I, -I], np.c_[-I, -I]]
    b_ub = np.r_[qhat, -qhat]
    if np.isfinite(r):
        A_ub = np.r_[A_ub, np.r_[np.zeros(S), np.ones(S)][None, :]]
        b_ub = np.r_[b_ub, r]
    bounds = [(0.0, 1.0 if s_ else 0.0) for s_ in sup] + [(0.0, None)] * S
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=np.r_[np.ones(S), np.zeros(S)][None, :], b_eq=[1.0], bounds=bounds,
                  method="highs")
    assert res.status == 0, res.message
    return float(U @ res.x[:S])


def x_cs_selftest(n_instances: int = 200) -> Dict[str, Any]:
    """x_cs_l1_extreme against scipy's linprog on random rows: both senses, shared and per-row U and supports, tied
    values, radii tiny / moderate / >= 2 / inf, and all-zero rows with r = inf (N = 0)."""
    rng = np_rng("xenv", "v4", "cs-selftest")
    worst, rows = 0.0, 0
    for k in range(int(n_instances)):
        S, n, shared = int(rng.integers(2, 9)), int(rng.integers(1, 5)), k % 3 == 0
        sup = rng.random((n, S)) < 0.7
        j = rng.integers(0, S, n)
        if shared:                                         # one support and one U for every row
            sup[:], j[:] = sup[0], j[0]
        sup[np.arange(n), j] = True
        q = rng.dirichlet(np.ones(S), size=n) * (rng.random((n, S)) < 0.6) * sup
        q[np.arange(n), j] += q.sum(axis=1) == 0           # an empty draw puts its mass on the forced support state
        q /= q.sum(axis=1, keepdims=True)
        r = rng.choice(np.array([1e-4, 0.05, 0.3, 0.9, 1.7, 2.5, np.inf]), size=n)
        q[(r == np.inf) & (rng.random(n) < 0.3)] = 0.0    # unlogged rows (N = 0) are unrestricted
        U = rng.integers(-2, 3, size=(n, S)).astype(np.float64) if k % 2 else rng.normal(size=(n, S))
        for sense in ("min", "max"):
            got = (x_cs_l1_extreme(q, r, U[0], sup[0], sense) if shared
                   else x_cs_l1_extreme(q, r, U, sup, sense))
            for i in range(n):
                ref = _x_cs_lp_reference(q[i], float(r[i]), U[0] if shared else U[i], sup[i], sense)
                worst = max(worst, abs(float(got[i]) - ref))
                rows += 1
    return {"instances": int(n_instances), "rows_compared": rows, "max_abs_diff_vs_linprog": worst,
            "tolerance": 1e-8, "passes": bool(worst <= 1e-8)}


_XCS_SUM_COLUMNS = ("n_policies", "n_covered", "lower_misses", "upper_misses", "pairs", "claims", "errors",
                    "pairs_AA", "claims_AA", "errors_AA", "pairs_AU", "claims_AU", "errors_AU", "pairs_UU",
                    "claims_UU", "errors_UU", "pairs_sens", "claims_sens", "errors_sens")


def x_cs_summary_table(draws: pd.DataFrame, cells: Sequence[str]) -> pd.DataFrame:
    """Per cell, family pool and pooled, x scale x test: claims, share, errors and error rate (all pairs and the
    completion-sensitive pairs AU + UU), draws with a wrong claim or an uncovered policy, coverage and mean width."""
    if draws is None or not len(draws):
        return pd.DataFrame()
    fam_of = draws.drop_duplicates("cell").set_index("cell")["family"].to_dict()
    present = [c for c in cells if c in fam_of]
    units = [(c, [c]) for c in present]                    # family pools and 'pooled:all' always exist
    units += [(f"pooled:{f}", [c for c in present if fam_of[c] == f]) for f in sorted(set(fam_of.values()))]
    units.append(("pooled:all", present))
    out = []
    for name, cs in units:
        d = draws[draws["cell"].isin(cs)]
        for (scale, test), g in d.groupby(["scale", "test"], dropna=False, sort=False):
            s = g[list(_XCS_SUM_COLUMNS)].sum()
            out.append({"unit": name, "n_cells": len(cs), "scale": scale, "set_kind": g["set_kind"].iloc[0],
                        "test": test, "interval": g["interval"].iloc[0], "draws": int(len(g)),
                        **{c: int(s[c]) for c in _XCS_SUM_COLUMNS},
                        "share": _xv3_rate(s["claims"], s["pairs"]), "error_rate": _xv3_rate(s["errors"], s["claims"]),
                        "share_sens": _xv3_rate(s["claims_sens"], s["pairs_sens"]),
                        "error_rate_sens": _xv3_rate(s["errors_sens"], s["claims_sens"]),
                        "draws_any_wrong": int(xs_as_bool(g["any_wrong"]).sum()),
                        "draws_any_uncovered": int((~xs_as_bool(g["all_covered"])).sum()),
                        "coverage": _xv3_rate(s["n_covered"], s["n_policies"]),
                        "mean_width": float(np.average(g["mean_width"].to_numpy(float),
                                                       weights=g["n_policies"].to_numpy(float))),
                        "radius_median": float(g["radius_median"].median()),
                        "radius_median_finite": float(g["radius_median_finite"].median()),
                        "share_unrestricted": float(g["share_unrestricted"].mean()), "r0_mean": float(g["r0"].mean())})
    return pd.DataFrame(out)


def x_cs_run_checks(pol: pd.DataFrame, meta: pd.DataFrame) -> Dict[str, Any]:
    """Pooled invariant report of a run (every per-draw assertion has passed when this is called)."""
    def col(c: str, f: Callable, default):
        return float(f(pd.to_numeric(meta[c]).to_numpy(dtype=float))) if c in meta and len(meta) else default

    out = {"policy_rows": int(len(pol)),
           "max_lower_minus_upper": float((pol["L"] - pol["U_free"]).max()) if len(pol) else None,
           "max_nhb_minus_free": float((pol["U_nhb"] - pol["U_free"]).max()) if len(pol) else None,
           "max_monotone_slack": col("max_monotone_slack", np.max, None),
           "min_paired_minus_unpaired": col("min_paired_minus_unpaired", np.min, None),
           "nonconverged_runs": int(col("nonconverged_runs", np.sum, 0)),
           "nonconverged_policy_rows": int((~(xs_as_bool(pol["converged_lower"]) & xs_as_bool(pol["converged_upper"])
                                              & xs_as_bool(pol["converged_upper_proj"]))).sum()) if len(pol) else 0,
           "max_iterations": int(pol[["iter_lower", "iter_upper", "iter_upper_proj"]].to_numpy().max(initial=0))}
    out["passes"] = bool(len(pol) > 0 and out["max_lower_minus_upper"] <= XV4_TOL and out["max_nhb_minus_free"] <= 0.0
                         and out["max_monotone_slack"] is not None and out["min_paired_minus_unpaired"] is not None
                         and out["min_paired_minus_unpaired"] >= -XV4_CS_PAIR_TOL)
    return out


def x_cs_run(cells: Sequence[dict]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """Every draw of every confidence-set cell (run_v4_cs_draw via x_v4_run_draws): policy rows, per-draw test rows,
    flattened draw meta, run summary."""
    tests: List[dict] = []

    def one(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
        df, meta = run_v4_cs_draw(spec, draw)
        meta = dict(meta)
        tests.extend(meta.pop("tests"))
        return df, meta

    frames, meta_df, summary = x_v4_run_draws(cells, one, "II.9c confidence sets")
    pol = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=XV4_CS_COLUMNS)
    return pol, pd.DataFrame(tests), meta_df, summary


XV4_CS_POLICIES = pd.DataFrame(columns=XV4_CS_COLUMNS)
XV4_CS_DRAWS = pd.DataFrame()
XV4_CS_TABLE = pd.DataFrame()
XV4_CS_META = pd.DataFrame()             # flattened per-draw meta (memory only; the release sums 'timing.total_s')
XV4_CS_SUMMARY: Dict[str, Any] = {"status": "not run (xenv_v4_confidence off)", "code_hash": XV4_CS_CODE_HASH,
                                  "checks": {}, "elapsed_s": 0.0, "draws_complete": None}
print("II.9c code hash:", XV4_CS_CODE_HASH[:16])

#### II.9c run

Runs the self-test of the inner problem (it must pass), then every draw of every enabled confidence-set cell with
the progress and ETA of II.6a, and writes the four outputs listed above (the flattened draw meta stays in
memory as `XV4_CS_META`). A configured cell that is not enabled (family switch off, or not in the design) is
skipped and listed in `v4_cs_summary.json`. The headline block of the json gives, per family pool and scale, the
certified share and error rate of every test on all pairs and on the completion-sensitive pairs, beside the
point-benchmark certificates. The pooled invariant checks are asserted after the json is written.

In [ ]:
# ============================================================
# II.9c Confidence-set certificates: run
# ============================================================
EXECLOG.mark("II.9c Confidence-set certificates: run")
if EXPERIMENTS["xenv_v4_confidence"]:
    _t0 = time.perf_counter()
    _selftest = x_cs_selftest()
    print("II.9c inner-problem self-test:", _selftest)
    assert _selftest["passes"], f"x_cs_l1_extreme disagrees with linprog: {_selftest}"
    _cells, _skipped = x_v4_cells(XV4CFG.cs_cells)
    if _skipped:
        print("NOTE: confidence-set cells not enabled (family switch off or not in the design), skipped:", _skipped)
    if _cells:
        XV4_CS_POLICIES, XV4_CS_DRAWS, XV4_CS_META, _run = x_cs_run(_cells)
        XV4_CS_TABLE = x_cs_summary_table(XV4_CS_DRAWS, [str(s["cell"]) for s in _cells])
        XV4_CS_POLICIES.to_csv(XENV_TABLE_DIR / "v4_cs_policies.csv.gz", index=False, compression="gzip")
        XV4_CS_DRAWS.to_csv(XENV_TABLE_DIR / "v4_cs_draws.csv", index=False)
        XV4_CS_TABLE.to_csv(XENV_TABLE_DIR / "v4_cs_summary.csv", index=False)
        _checks = {**x_cs_run_checks(XV4_CS_POLICIES, XV4_CS_META), "test_rows": int(len(XV4_CS_DRAWS))}
        _hcols = ["unit", "scale", "set_kind", "test", "draws", "claims", "share", "errors", "error_rate", "share_sens",
                  "error_rate_sens", "draws_any_wrong", "draws_any_uncovered", "coverage", "mean_width"]
        _head = (XV4_CS_TABLE[XV4_CS_TABLE["unit"].astype(str).str.startswith("pooled:")][_hcols]
                 if len(XV4_CS_TABLE) else pd.DataFrame(columns=_hcols))
        _need = sum(x_v4_draws(s) for s in _cells)
        XV4_CS_SUMMARY = {
            "status": "run", "code_hash": XV4_CS_CODE_HASH,
            "config": {"delta": XV4CFG.cs_delta, "scales": list(XV4CFG.cs_scales), "tol": XV4CFG.cs_tol,
                       "max_iter": XV4CFG.cs_max_iter, "valid_scale": 1.0,
                       "note": "only scale 1.0 is covered by the union bound; smaller scales are heuristic"},
            "selftest": _selftest, "cells": [s["cell"] for s in _cells], "cells_skipped": _skipped,
            **{k: _run[k] for k in ("draws", "draws_computed", "draws_loaded", "draws_per_cell")},
            "draws_complete": bool(_run["draws"] == _need), "checks": _checks,
            "headline": _head.to_dict(orient="records"),
            "outputs": ["xenv/v4_cs_policies.csv.gz", "xenv/v4_cs_draws.csv", "xenv/v4_cs_summary.csv",
                        "xenv/v4_cs_summary.json"],
            "elapsed_s": time.perf_counter() - _t0}
        display(_head.drop(columns=["set_kind", "claims", "errors", "draws_any_uncovered"]).round(4))
        del _checks, _hcols, _head, _need, _run
    else:
        XV4_CS_SUMMARY = {"status": "not run: no configured confidence-set cell is enabled", "cells_skipped": _skipped,
                          "code_hash": XV4_CS_CODE_HASH, "selftest": _selftest, "checks": {}, "draws_complete": None,
                          "elapsed_s": time.perf_counter() - _t0}
    del _t0, _selftest, _cells, _skipped
save_json(XV4_CS_SUMMARY, XENV_TABLE_DIR / "v4_cs_summary.json")
print(json.dumps({k: v for k, v in XV4_CS_SUMMARY.items() if k not in ("headline", "draws_per_cell")}, indent=1,
                 default=_json_default))
# after the json is written, so that a failure can be inspected there (as the II.9 gate message says)
assert XV4_CS_SUMMARY.get("status") != "run" or XV4_CS_SUMMARY["checks"]["passes"], \
    f"II.9c invariant checks failed: {XV4_CS_SUMMARY['checks']} - see xenv/v4_cs_summary.json"

### II.9d Bootstrap with the construction re-estimated in every replicate

The certificates of Corollary 2 hold for the **given** supported rows. A benchmark's rows, mask, decision states and
d0 are estimated from hospital data, so its brackets also carry estimation error. II.8g measured part of that error
with B = 10 replicates, a **fixed** mask and fixed decision states, and scored coverage only over policies with
p > 0. This subsection repeats the bootstrap with B = `XV4CFG.boot_replicates` (200) replicates, in two variants:

* `fixed`: the II.8g model (`x_boot_model`: the supported rows and d0 re-estimated from the resampled steps, with the
  original mask, decision states, censoring and redirects), on the same random streams as II.8g;
* `reestimated`: the **whole construction** is rerun on every resampled hospital data set (`x_benchmark_from_data`:
  informative steps, count mask at τ, dead-state removal, trap censoring, dangling redirects, empirical d0,
  fallback action). The mask and the decision states can change from one replicate to the next.

**Cells:** `XV4CFG.boot_cells` (resolved by `x_v4_cells`; a configured cell whose family switch is off is skipped and
listed). The default set is Gumbel `best` with n = 10 000, τ = 20 and with n = 2 000, τ = 5, and Garnet h ∈ {0, 1}
at ε = 0.05. At most `XV4CFG.max_draws` draws per cell; default policies only (`is_default`), as in II.8g.

**Per (cell, draw).**
1. Rebuild the instance (`x_v4_instance`), regenerate the default policies (`x_v4_default_records`) and the hospital
   episodes and benchmark B (`x_v4_rebuild_benchmark`, which asserts that B equals the instance's benchmark).
2. Step map `smap = x_boot_step_map(true_env, hosp, τ, B)` (II.8g). Unit weights must reproduce the benchmark:
   `x_boot_model` with weights 1 equals `learn.P` and `learn.d0` (1e-14), and the resampled data set with every
   multiplicity 1 is the hospital data set, whose construction equals B (identity, index, decision states and
   fallback action). Both are asserted.
3. For b = 0, …, B − 1: episode multiplicities `mult = x_boot_weights(n_ep, np_rng("xenv", "bootstrap", cell, draw,
   b))`, the II.8g stream, so the first 10 `fixed` replicates are the II.8g replicates (II.8g default
   `resample_d0 = True`).
   * `fixed`: `x_boot_model(learn, smap, mult[episode], True)`; every default policy is evaluated exactly under the
     `given` (mean) rule and the `terminate` rule: J and p = P(any unsupported decision).
   * `reestimated`: `x_v4_boot_dataset` repeats episode e `mult[e]` times with new consecutive `episode_id`s, keeping
     the step order, `timestep`, `cutoff`, `terminated` and every other array field; the data set is validated with
     `x_validate_dataset`. Then `Bb = x_benchmark_from_data(true_env, hosp_b, τ, f"{name}|boot{b}")` with the
     default dead-state rule. A construction that fails (`AssertionError` / `ValueError`) is recorded as a `failed`
     replicate with its message, and the draw continues. Policies are mapped through the true state space:
     π_true = `B["lift"](π)`, π_b = π_true[`Bb["index"]`] (validated), and evaluated on `Bb["bench"]` under both
     rules; p refers to the replicate's own mask.
4. Point values on B (replicate −1) and the truth: J_true on the `true` target (T_masked, `given`, lifted) and the
   true-rows terminate value on `true_rows_terminate`.

**Intervals** (per policy and variant, over the **proper** replicates: not failed and `ok` under both rules; linear
quantiles at `XV4CFG.boot_quantiles` = (2.5 %, 97.5 %), which configure the analysis only):

| Interval | Lower end | Upper end |
|---|---|---|
| `nhb` | q_lo(J_term^b) | q_hi(J_mean^b) |
| `free` (no NHB) | q_lo(J_term^b) | q_hi(J_term^b + p^b (v_max − v_min)) |
| `minmax` | min J_term^b | max J_mean^b |
| `point` variant | J_bench_term (both types) | J_bench_mean (`nhb`); J_bench_term + p (v_max − v_min) (`free`) |

The interval table also carries the number of replicates, failed replicates and proper replicates of each policy,
the share of proper replicates with p^b > 0, the point values, p on the point benchmark and the truth.

**Invariants asserted per draw** (re-run on cached tables): the point benchmark and the truth are proper for every
policy; every (variant, replicate) holds the whole policy set once; failed replicates are whole replicates of the
`reestimated` variant; on every benchmark model (point, fixed, re-estimated) the two rules share p (1e-9) and
0 ≤ J_mean − J_term ≤ p (v_max − v_min) (Theorem 1 part iii, 1e-9). **Counted per draw:** the true-rows bracket
J_rows_term ≤ J_true ≤ J_rows_term + p (v_max − v_min) (Theorem 1 on the true rows); as for the theorem counts of
II.8i and II.9b, a violation fails the summary checks and the analysis cell raises after writing its files.
**Reported, not asserted:** the agreement of the first `fixed` replicates (and of replicate −1) with the II.8g rows of
the same draw (`XV3_BOOT_RESULTS`, else `xenv/v3_bootstrap_results.csv.gz`): same streams and model, but BLAS
rounding can differ across machines.

**Cache.** Key `v4boot/<cell>/draw<k>` (and `/meta`). The identity is the draw identity (`x_v4_draw_identity`), the
code hash of this subsection (its functions, the II.8g functions it calls, and `XV4_COMMON_CODE_HASH`), B, the
variants, `resample_d0` and the stream names. The raw replicate rows live **only in the cache**, in compact columns
(`variant`, `replicate`, `policy`, `status_given`, `status_term`, `J_given`, `J_term`, `p_given`, `p_term`; about
20 000 rows per draw). The table returned per draw is the interval table, computed from the rows read back.

| In memory | Content |
|---|---|
| `XV4_BOOT_IV` | one row per draw × default policy × variant (`point`, `fixed`, `reestimated`; II.9d analyses add `v3boot`) |
| `XV4_BOOT_META` | per draw: replicate counts, failures and messages, construction variability of the re-estimated replicates (decision states gained / lost, mask rows changed, dead and trap states, d0 support), fixed-model fallback pairs and traps, unit-weight checks, invariant slacks, II.8g agreement, timings |

The files are written by the analysis cell below.

**Expected cost on Colab.** These are estimates, not measurements. Per draw: an instance rebuild and one
`run_xmethods` fit (4–16 s, as in II.6), then per replicate one fixed-mask model and one full construction, each
followed by 2 × ~40 exact evaluations of the distinct default policies (≈ 32 000 evaluations per draw at B = 200).
With the II.6 / II.8g Colab timings (about 3–8 ms per Gumbel benchmark evaluation, 1–3 ms on Garnet; a Gumbel
construction from about 200 000 steps at n = 10 000 in about 0.2–0.5 s), that is about 3–6 min per Gumbel draw and
1–2 min per Garnet draw, i.e. roughly 4–8 h for the default 4 cells × 30 draws (lite profile: about 1.5–3 h; smoke: a
few minutes). Every draw is cached, so a disconnect loses at most the draw in progress; a cached draw reloads in
about 3–5 s.

In [ ]:
# ============================================================
# II.9d Bootstrap with the construction re-estimated per replicate
# ============================================================
EXECLOG.mark("II.9 Second-revision experiments: bootstrap with re-estimated construction")
import traceback

XV4_BOOT_POINT = XV3_BOOT_POINT                  # replicate label of the point benchmark and of the truth (-1)
XV4_BOOT_VARIANTS = ("fixed", "reestimated")     # bootstrap variants this subsection can compute
XV4_BOOT_VARIANT_ORDER = ("point", "v3boot", "fixed", "reestimated")   # row order of the analyses
XV4_BOOT_INTERVALS = ("nhb", "free", "minmax")
XV4_BOOT_STATUSES = (*XV3_EVAL_STATUSES, "failed")   # 'failed': the re-estimated construction raised
XV4_BOOT_RAW_COLUMNS = ["variant", "replicate", "policy", "status_given", "status_term", "J_given", "J_term",
                        "p_given", "p_term"]
XV4_BOOT_IV_COLUMNS = ["cell", "family", "draw", "policy", "method", "policy_variant", "method_family",
                       "boot_variant", "replicates", "n_failed", "n_proper", "n_nonok", "true", "p_true",
                       "true_rows_term", "point_term", "point_mean", "p_bench", "p_boot_pos_share",
                       "lo_nhb", "up_nhb", "lo_free", "up_free", "lo_minmax", "up_minmax"]
assert XV4CFG.boot_variants and set(XV4CFG.boot_variants) <= set(XV4_BOOT_VARIANTS), XV4CFG.boot_variants
assert int(XV4CFG.boot_replicates) >= 1 and 0.0 <= XV4CFG.boot_quantiles[0] < XV4CFG.boot_quantiles[1] <= 1.0


# ---------- resampled hospital data and re-estimated construction ----------
def x_v4_boot_dataset(hosp: Dict[str, np.ndarray], mult: np.ndarray) -> Dict[str, np.ndarray]:
    """Hospital data set of one bootstrap resample: episode e is repeated mult[e] times (episodes in id order), with
    new consecutive episode ids. Step order, timestep, cutoff, terminated and every other array field are carried."""
    ep = np.asarray(hosp["episode_id"], dtype=np.int64)
    n = len(ep)
    assert n > 0 and all(isinstance(v, np.ndarray) and len(v) == n for v in hosp.values()), \
        "every hospital field must be an array with one entry per step"
    starts = np.flatnonzero(np.r_[True, ep[1:] != ep[:-1]])
    n_ep = len(starts)
    assert np.array_equal(ep[starts], np.arange(n_ep)), "hospital episode ids must be 0..n-1, one block each, in order"
    m = np.asarray(mult, dtype=np.float64)
    assert m.shape == (n_ep,) and np.all(m >= 0) and np.all(m == np.round(m)), "multiplicities must be counts"
    lengths = np.diff(np.r_[starts, n])
    src = np.repeat(np.arange(n_ep), m.astype(np.int64))          # original episode of every new episode
    L = lengths[src]
    new_ep = np.repeat(np.arange(len(src)), L)
    offset = np.arange(int(L.sum())) - np.repeat(np.cumsum(L) - L, L)
    rows = starts[src][new_ep] + offset
    out = {k: v[rows] for k, v in hosp.items()}
    out["episode_id"] = new_ep.astype(hosp["episode_id"].dtype)
    return out


def x_v4_boot_construct(true_env: FiniteEnv, hosp_b: Dict[str, np.ndarray], tau: int, name: str,
                        empty_rule: Optional[str] = None) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """(x_benchmark_from_data on one resampled data set, None), or (None, message) when the construction fails with
    an AssertionError / ValueError. The benchmark name in the message is replaced by '<name>' so messages group;
    messages are cut to 150 characters so that the draw meta keeps them as a JSON list (_x_compact limit)."""
    try:
        return x_benchmark_from_data(true_env, hosp_b, tau, name, empty_rule=empty_rule), None
    except (AssertionError, ValueError) as exc:
        tb = traceback.extract_tb(exc.__traceback__)
        where = f" [{tb[-1].name}:{tb[-1].lineno}]" if tb else ""
        return None, f"{type(exc).__name__}{where}: {str(exc).replace(name, '<name>')}"[:150]


def _x_v4_boot_eval(env: FiniteEnv, pis: Dict[bytes, np.ndarray]) -> Dict[bytes, Tuple[dict, dict]]:
    """Exact evaluation of every distinct policy on `env` under the given (mean) and the terminate rule."""
    return {k: (x_v4_eval(env, pi, "given"), x_v4_eval(env, pi, "terminate")) for k, pi in pis.items()}


def _x_v4_boot_rows(variant: str, replicate: int, labels: Sequence[str], keys: Sequence[bytes],
                    ev: Optional[Dict[bytes, Tuple[dict, dict]]]) -> List[dict]:
    """Compact raw rows of one (variant, replicate); ev=None marks a failed replicate."""
    fail = ({"status": "failed"}, {"status": "failed"})
    out = []
    for lab, k in zip(labels, keys):
        g, t = fail if ev is None else ev[k]
        out.append({"variant": variant, "replicate": int(replicate), "policy": lab, "status_given": g["status"],
                    "status_term": t["status"], "J_given": float(g.get("return", np.nan)),
                    "J_term": float(t.get("return", np.nan)), "p_given": float(g.get("prob_any_violation", np.nan)),
                    "p_term": float(t.get("prob_any_violation", np.nan))})
    return out


def x_v4_boot_construction_stats(Bb: Dict[str, Any], B: Dict[str, Any]) -> Dict[str, float]:
    """How the re-estimated construction Bb differs from the benchmark B (decision states, mask rows, censoring)."""
    rep = Bb["report"]
    V, V0 = np.asarray(Bb["decision_states_true"]), np.asarray(B["decision_states_true"])
    common = np.intersect1d(V, V0)
    changed = ~(Bb["mask_true"][common] == B["mask_true"][common]).all(axis=1)
    return {"decision_states": int(rep["decision_states"]), "decision_states_new": int(len(np.setdiff1d(V, V0))),
            "decision_states_lost": int(len(np.setdiff1d(V0, V))), "mask_rows_changed": int(changed.sum()),
            "mask_rows_changed_share": float(changed.mean()) if len(common) else float("nan"),
            "admissible_pairs": int(rep["admissible_pairs"]),
            "mean_admissible_actions": float(rep["mean_admissible_actions"]),
            "dead_states_removed": int(rep["dead_states_removed"]),
            "trap_states_censored": int(rep["trap_states_censored"]),
            "censored_d0_mass": float(rep["censored_d0_mass"]), "d0_support_states": int((Bb["bench"].d0 > 0).sum()),
            "fallback_action_changed": int(int(rep["fallback_action"]) != int(B["fallback_action"]))}


def _x_v4_boot_dist(stats_: Sequence[dict]) -> Dict[str, Dict[str, Optional[float]]]:
    """min / median / mean / max over replicates of every statistic in `stats_` (None if no finite value)."""
    out: Dict[str, Dict[str, Optional[float]]] = {}
    for k in (stats_[0] if stats_ else {}):
        v = np.asarray([s[k] for s in stats_], dtype=np.float64)
        v = v[np.isfinite(v)]
        out[k] = ({"min": float(v.min()), "median": float(np.median(v)), "mean": float(v.mean()), "max": float(v.max())}
                  if len(v) else {"min": None, "median": None, "mean": None, "max": None})
    return out


def x_v4_boot_unit_check(true_env: FiniteEnv, hosp: Dict[str, np.ndarray], tau: int, B: Dict[str, Any],
                         learn: FiniteEnv, smap: Dict[str, Any]) -> Dict[str, Any]:
    """Unit weights reproduce the benchmark in both variants (asserted): x_boot_model with weights 1 gives learn.P and
    learn.d0, and the data set of unit multiplicities is `hosp`, whose construction equals B."""
    P1, d01, _ = x_boot_model(learn, smap, np.ones(len(smap["flat"])), True)
    out = {"max_abs_diff_P": float(np.abs(P1 - learn.P).max()), "max_abs_diff_d0": float(np.abs(d01 - learn.d0).max())}
    assert max(out.values()) <= 1e-14, f"{learn.name}: unit weights do not reproduce the benchmark {out}"
    del P1, d01
    n_ep = int(np.asarray(hosp["episode_id"]).max()) + 1
    h1 = x_v4_boot_dataset(hosp, np.ones(n_ep))
    assert set(h1) == set(hosp) and all(h1[k].dtype == hosp[k].dtype and np.array_equal(h1[k], hosp[k])
                                        for k in hosp), "unit multiplicities must give the hospital data set back"
    B1 = x_benchmark_from_data(true_env, h1, tau, learn.name)
    same = (obj_hash(B1["bench"].identity()) == obj_hash(B["bench"].identity())
            and np.array_equal(B1["index"], B["index"])
            and np.array_equal(B1["decision_states_true"], B["decision_states_true"])
            and int(B1["fallback_action"]) == int(B["fallback_action"]))
    assert same, f"{learn.name}: the construction on the unit-multiplicity data set differs from the benchmark"
    out["construction_identity"] = True
    return out


def x_v4_boot_identity(spec: dict, inst: CellInstance) -> dict:
    return {"kind": "xenv_v4_bootstrap", "draw": x_v4_draw_identity(spec, inst), "code": XV4_BOOT_CODE_HASH,
            "replicates": int(XV4CFG.boot_replicates), "variants": list(XV4CFG.boot_variants), "resample_d0": True,
            "rng": ["xenv", "bootstrap", "<cell>", "<draw>", "<b>"], "columns": XV4_BOOT_RAW_COLUMNS,
            "policies": "default (is_default) records of x_v4_default_records"}


# ---------- checks and intervals (pure pandas; re-run on cached tables) ----------
def x_v4_boot_check_rows(df: pd.DataFrame, v_range: float,
                         expect: Optional[Tuple[int, Sequence[str]]] = None) -> Dict[str, Any]:
    """Invariants of one draw's raw replicate rows (see markdown); asserted, and a small report is returned.
    expect = (B, variants): every variant holds exactly the replicates 0..B-1 (a complete table)."""
    sg, stt = df["status_given"].astype(str), df["status_term"].astype(str)
    var = df["variant"].astype(str)
    bad = sorted((set(sg) | set(stt)) - set(XV4_BOOT_STATUSES))
    assert not bad, f"unexpected bootstrap status values {bad}"
    assert set(var) <= {"point", "truth", *XV4_BOOT_VARIANTS}, sorted(set(var))
    dup = df.duplicated(["variant", "replicate", "policy"])
    assert not dup.any(), f"{int(dup.sum())} duplicated (variant, replicate, policy) rows"
    rep = df["replicate"].astype(np.int64)
    one = var.isin(["point", "truth"])
    assert (rep[one] == XV4_BOOT_POINT).all() and (rep[~one] >= 0).all(), "replicate labels"
    assert {"point", "truth"} <= set(var), "the point benchmark and truth rows (replicate -1) are missing"
    if expect is not None:
        n_rep, variants = int(expect[0]), [str(v) for v in expect[1]]
        assert set(var) - {"point", "truth"} == set(variants), (sorted(set(var)), variants)
        for v in variants:
            assert np.array_equal(np.sort(rep[var == v].unique()), np.arange(n_rep)), \
                f"variant {v}: the replicates are not exactly 0..{n_rep - 1}"
    assert ((sg[one] == "ok") & (stt[one] == "ok")).all(), \
        "a default policy is not proper on the point benchmark or on the truth"
    pols = set(df.loc[var == "point", "policy"])
    n_pol = df.groupby([var, rep])["policy"].nunique()
    assert pols and set(df["policy"]) == pols and (n_pol == len(pols)).all(), \
        "every (variant, replicate) must hold the whole default policy set"
    fg, ft = sg == "failed", stt == "failed"
    assert (fg == ft).all() and not (fg & (var != "reestimated")).any(), "failed rows outside re-estimated replicates"
    whole = fg.groupby([var, rep]).agg(["all", "any"])
    assert (whole["all"] == whole["any"]).all(), "a failed replicate must fail for every policy"
    bench = var.isin(["point", *XV4_BOOT_VARIANTS]) & (sg == "ok") & (stt == "ok")
    jm, jt = df.loc[bench, "J_given"].to_numpy(float), df.loc[bench, "J_term"].to_numpy(float)
    pm, ptm = df.loc[bench, "p_given"].to_numpy(float), df.loc[bench, "p_term"].to_numpy(float)
    width = jm - jt
    dp = float(np.abs(pm - ptm).max(initial=0.0))
    slack_lo = float((-width).max(initial=-np.inf))
    slack_hi = float((width - ptm * float(v_range)).max(initial=-np.inf))
    assert dp <= XV4_TOL, f"the two rules disagree on P(any unsupported decision) by {dp:.3g}"
    assert slack_lo <= XV4_TOL, f"J_mean < J_term on a bootstrap model by {slack_lo:.3g} (Theorem 1 part iii)"
    assert slack_hi <= XV4_TOL, f"J_mean - J_term exceeds p (v_max - v_min) by {slack_hi:.3g} (Theorem 1 part iii)"
    tr = df[var == "truth"]
    jtrue, jrt, ptr = (tr[c].to_numpy(float) for c in ("J_given", "J_term", "p_term"))
    bracket = (jrt > jtrue + XV4_TOL) | (jtrue > jrt + ptr * float(v_range) + XV4_TOL)
    nonok = df[~(fg | ((sg == "ok") & (stt == "ok")))].groupby("variant").size()
    return {"rows": int(len(df)), "policies": int(len(pols)), "pairs_checked": int(bench.sum()),
            "replicates": {str(v): int(rep[var == v].nunique()) for v in XV4_BOOT_VARIANTS if (var == v).any()},
            "failed_replicates": int(rep[fg].nunique()), "nonok_rows": int(nonok.sum()),
            "nonok_by_variant": {str(k): int(v) for k, v in nonok.items()}, "max_rule_p_difference": dp,
            "max_negative_width": slack_lo, "max_width_minus_bound": slack_hi,
            "truth_rows_bracket_violations": int(bracket.sum())}


def x_v4_boot_intervals(raw: pd.DataFrame, v_range: float, q: Tuple[float, float]) -> pd.DataFrame:
    """Per policy x variant of one draw: truth, point values, replicate counts and the nhb / free / minmax intervals
    over the proper replicates (not failed, 'ok' under both rules; linear quantiles as in II.8i). The 'point' rows hold
    the point intervals [J_term, J_mean] and [J_term, J_term + p v_range] (minmax undefined)."""
    var = raw["variant"].astype(str)
    pt = raw[var == "point"].set_index("policy")
    pol = pt.index
    assert len(pol) and pol.is_unique, "one point row per policy"
    tr = raw[var == "truth"].set_index("policy").reindex(pol)
    vr = float(v_range)
    base = pd.DataFrame({"policy": pol.to_numpy(), "true": tr["J_given"].to_numpy(float),
                         "p_true": tr["p_given"].to_numpy(float),
                         "true_rows_term": np.where(tr["status_term"].astype(str).to_numpy() == "ok",
                                                    tr["J_term"].to_numpy(float), np.nan),
                         "point_term": pt["J_term"].to_numpy(float), "point_mean": pt["J_given"].to_numpy(float),
                         "p_bench": pt["p_given"].to_numpy(float)})
    nan = np.full(len(pol), np.nan)
    frames = [base.assign(boot_variant="point", replicates=0, n_failed=0, n_proper=0, n_nonok=0, p_boot_pos_share=nan,
                          lo_nhb=base["point_term"], up_nhb=base["point_mean"], lo_free=base["point_term"],
                          up_free=base["point_term"] + base["p_bench"] * vr, lo_minmax=nan, up_minmax=nan)]
    others = [v for v in XV4_BOOT_VARIANT_ORDER if v != "point" and (var == v).any()]
    assert set(var) <= {"point", "truth", *others}, sorted(set(var))
    for v in others:
        sub = raw[var == v]
        reps = np.sort(sub["replicate"].astype(np.int64).unique())

        def mat(col: str) -> pd.DataFrame:
            return sub.pivot(index="policy", columns="replicate", values=col).reindex(index=pol, columns=reps)

        sg, stt = mat("status_given"), mat("status_term")
        ok = ((sg == "ok") & (stt == "ok")).to_numpy()
        n_failed = int((sg == "failed").all(axis=0).sum())
        Jg = np.where(ok, mat("J_given").to_numpy(float), np.nan)
        Jt = np.where(ok, mat("J_term").to_numpy(float), np.nan)
        Pg = np.where(ok, mat("p_given").to_numpy(float), np.nan)
        n_ok = ok.sum(axis=1)
        has = n_ok > 0
        res = {c: nan.copy() for c in ("lo_nhb", "up_nhb", "up_free", "lo_minmax", "up_minmax", "p_boot_pos_share")}
        if has.any():
            res["lo_nhb"][has] = np.nanquantile(Jt[has], float(q[0]), axis=1)
            res["up_nhb"][has] = np.nanquantile(Jg[has], float(q[1]), axis=1)
            res["up_free"][has] = np.nanquantile(Jt[has] + Pg[has] * vr, float(q[1]), axis=1)
            res["lo_minmax"][has] = np.nanmin(Jt[has], axis=1)
            res["up_minmax"][has] = np.nanmax(Jg[has], axis=1)
            res["p_boot_pos_share"][has] = (np.nan_to_num(Pg[has], nan=0.0) > XV4_PTOL).sum(axis=1) / n_ok[has]
        frames.append(base.assign(boot_variant=v, replicates=len(reps), n_failed=n_failed, n_proper=n_ok,
                                  n_nonok=len(reps) - n_failed - n_ok, lo_free=res["lo_nhb"], **res))
    return pd.concat(frames, ignore_index=True)


# ---------- II.8g rows of the same draws (comparison only; never cached) ----------
def x_v4_boot_from_v3(boot: pd.DataFrame) -> Dict[Tuple[str, int], pd.DataFrame]:
    """II.8g rows (XV3_BOOT_COLUMNS, one row per target) in this subsection's raw format, by (cell, draw): 'point'
    and 'truth' at replicate -1 (no true-rows terminate value: status 'absent') and 'v3boot' for the replicates."""
    if boot is None or not len(boot):
        return {}
    b = boot[boot["target"].astype(str).isin(["bench_mean", "bench_terminate", "true"])].copy()
    b["draw"] = pd.to_numeric(b["draw"]).astype(np.int64)
    b["replicate"] = pd.to_numeric(b["replicate"]).astype(np.int64)
    k = ["cell", "draw", "replicate", "policy"]
    cols = {"status": "status", "return": "J", "prob_any_violation": "p"}

    def part(target: str, sfx: str) -> pd.DataFrame:
        d = b.loc[b["target"] == target, k + list(cols)]
        return d.rename(columns={c: f"{n}_{sfx}" for c, n in cols.items()})

    bench = part("bench_mean", "given").merge(part("bench_terminate", "term"), on=k, how="outer", validate="one_to_one")
    bench["variant"] = np.where(bench["replicate"] == XV3_BOOT_POINT, "point", "v3boot")
    tr = part("true", "given").assign(variant="truth", status_term="absent", J_term=np.nan, p_term=np.nan)
    allr = pd.concat([bench, tr], ignore_index=True)
    for c in ("status_given", "status_term"):
        allr[c] = allr[c].fillna("missing").astype(str)
    return {(str(c), int(d)): g[XV4_BOOT_RAW_COLUMNS].reset_index(drop=True)
            for (c, d), g in allr.groupby(["cell", "draw"], sort=False)}


def x_v4_boot_v3_reference(cells: Sequence[str]) -> Tuple[Dict[Tuple[str, int], pd.DataFrame], str]:
    """II.8g rows of `cells` by (cell, draw): XV3_BOOT_RESULTS, or else the saved xenv/v3_bootstrap_results.csv.gz."""
    src = globals().get("XV3_BOOT_RESULTS")
    origin = "XV3_BOOT_RESULTS"
    if src is None or not len(src):
        path = XENV_TABLE_DIR / "v3_bootstrap_results.csv.gz"
        if not path.exists():
            return {}, "not available (II.8g not run in this session and no saved table)"
        src = pd.read_csv(path, float_precision="round_trip",
                          dtype={"cell": str, "policy": str, "target": str, "status": str})
        origin = f"xenv/{path.name}"
    return x_v4_boot_from_v3(src[src["cell"].astype(str).isin([str(c) for c in cells])]), origin


def x_v4_boot_v3_match(raw: pd.DataFrame, ref: Optional[pd.DataFrame]) -> Dict[str, Any]:
    """Replicate -1 and the first `fixed` replicates against the II.8g rows of the same draw (same streams and model;
    equal up to BLAS rounding). Reported, never asserted."""
    if ref is None or not len(ref):
        return {"status": "not compared (no II.8g rows for this draw)", "match": None}
    is_fixed = raw["variant"] == "fixed"
    n_fixed = int(raw.loc[is_fixed, "replicate"].max()) + 1 if is_fixed.any() else 0
    mine = raw[raw["variant"].isin(["point", "truth", "fixed"])].replace({"variant": {"fixed": "v3boot"}})
    r = ref[(ref["variant"] != "v3boot") | (ref["replicate"] < n_fixed)]
    m = r.merge(mine, on=["variant", "replicate", "policy"], how="left", suffixes=("_v3", ""), validate="one_to_one")
    have = m["status_given"].notna()
    with_term = have & (m["variant"] != "truth")
    st = have & ((m["status_given_v3"] != m["status_given"]) | (with_term & (m["status_term_v3"] != m["status_term"])))
    okg = have & (m["status_given_v3"] == "ok") & (m["status_given"] == "ok")
    okt = with_term & (m["status_term_v3"] == "ok") & (m["status_term"] == "ok")
    dJ = pd.concat([(m["J_given_v3"] - m["J_given"]).abs()[okg], (m["J_term_v3"] - m["J_term"]).abs()[okt]])
    dP = pd.concat([(m["p_given_v3"] - m["p_given"]).abs()[okg], (m["p_term_v3"] - m["p_term"]).abs()[okt]])
    var_j = np.concatenate([m.loc[okg, "variant"].to_numpy(), m.loc[okt, "variant"].to_numpy()])
    out = {"status": "compared", "rows": int(len(m)), "missing_in_v4": int((~have).sum()),
           "replicates_compared": int(r.loc[r["variant"] == "v3boot", "replicate"].nunique()),
           "status_mismatches": int(st.sum()), "max_abs_diff_return": float(dJ.max()) if len(dJ) else None,
           "max_abs_diff_p": float(dP.max()) if len(dP) else None,
           "max_abs_diff_return_by_variant": {str(v): float(dJ.to_numpy()[var_j == v].max())
                                              for v in sorted(set(var_j))}}
    out["match"] = bool(out["missing_in_v4"] == 0 and out["status_mismatches"] == 0
                        and (out["max_abs_diff_return"] or 0.0) <= XV4_TOL)
    return out


_XV4_BOOT_V3REF: Dict[Tuple[str, int], pd.DataFrame] = {}


def x_v4_boot_finish(spec: dict, draw: int, raw: pd.DataFrame, meta: dict) -> Tuple[pd.DataFrame, dict]:
    """Interval table of one draw (from the raw rows read back from the cache) and the session meta."""
    meta = dict(meta)
    desc = meta.pop("policy_desc")
    iv = x_v4_boot_intervals(raw, float(meta["v_range"]), XV4CFG.boot_quantiles)
    d = pd.DataFrame.from_dict(desc, orient="index").reindex(iv["policy"].to_numpy())
    assert d[["method", "policy_variant", "method_family"]].notna().all().all(), "policy descriptions missing"
    for c in ("method", "policy_variant", "method_family"):
        iv[c] = d[c].to_numpy()
    iv["cell"], iv["family"], iv["draw"] = str(spec["cell"]), str(spec["family"]), int(draw)
    meta["v3_match"] = x_v4_boot_v3_match(raw, _XV4_BOOT_V3REF.get((str(spec["cell"]), int(draw))))
    return iv[XV4_BOOT_IV_COLUMNS].reset_index(drop=True), meta


# ---------- one (cell, draw) ----------
def run_v4_boot_draw(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
    """Fixed-mask and re-estimated bootstrap of the default policies of one draw (raw rows cached under v4boot/);
    returns the per policy x variant intervals and the draw meta."""
    cell, family, draw = spec["cell"], spec["family"], int(draw)
    assert family in ("gumbel", "garnet"), family
    t0 = time.perf_counter()
    inst = x_v4_instance(spec, draw)
    assert (inst.cell, inst.family, int(inst.draw)) == (cell, family, draw), (inst.cell, inst.family, inst.draw)
    identity = x_v4_boot_identity(spec, inst)
    key, ident_hash = "v4boot/" + x_cache_key(cell, draw), obj_hash(identity)[:20]
    build_s = time.perf_counter() - t0
    hit, meta_hit = XCACHE.get(key, identity), XCACHE.get(key + "/meta", identity)
    if hit is not None and meta_hit is not None:
        del inst
        assert list(hit.columns) == XV4_BOOT_RAW_COLUMNS, f"{key}: cached columns differ from XV4_BOOT_RAW_COLUMNS"
        meta = json.loads(str(meta_hit["meta_json"].iloc[0]))
        meta.update(cache="loaded", session_build_s=build_s,
                    checks=x_v4_boot_check_rows(hit, meta["v_range"], (meta["replicates"], meta["variants"])))
        gc.collect()
        return x_v4_boot_finish(spec, draw, hit, meta)
    if (hit is None) != (meta_hit is None):
        print(f"  {key}: incomplete cache entry (results or meta missing) - recomputing this draw")
    XCACHE.require_allowed(key)
    learn = inst.learn_env
    t1 = time.perf_counter()
    records, keys = x_v4_default_records(inst)
    labels = [r["desc"]["policy"] for r in records]
    assert len(set(labels)) == len(labels), "duplicate default policy labels"
    pis = {k: r["pi"] for r, k in zip(records, keys)}
    t2 = time.perf_counter()
    true_env, _, hosp, B = x_v4_rebuild_benchmark(spec, draw, inst)
    smap = x_boot_step_map(true_env, hosp, spec["tau"], B)
    eps_id = smap["episode"]
    n_ep = int(eps_id.max()) + 1
    assert np.array_equal(np.unique(eps_id), np.arange(n_ep)), "hospital episode ids must be 0..n-1"
    repro = x_v4_boot_unit_check(true_env, hosp, spec["tau"], B, learn, smap)
    lifted = {k: B["lift"](pi) for k, pi in pis.items()}
    tgt = {lab: (env, sem, lift) for lab, env, sem, lift in inst.targets}
    assert {"true", "true_rows_terminate"} <= set(tgt) and tgt["true"][1] == "given" \
        and tgt["true_rows_terminate"][1] == "terminate", sorted(tgt)
    assert np.array_equal(tgt["true"][2](pis[keys[0]]), lifted[keys[0]]), "instance lift and rebuilt lift differ"
    lo, hi = learn.value_bounds
    t3 = time.perf_counter()
    rows = _x_v4_boot_rows("point", XV4_BOOT_POINT, labels, keys, _x_v4_boot_eval(learn, pis))
    (e_t, s_t, f_t), (e_r, s_r, f_r) = tgt["true"], tgt["true_rows_terminate"]
    ev_true = {k: (x_v4_eval(e_t, f_t(pi), s_t), x_v4_eval(e_r, f_r(pi), s_r)) for k, pi in pis.items()}
    rows += _x_v4_boot_rows("truth", XV4_BOOT_POINT, labels, keys, ev_true)
    del ev_true
    t4 = time.perf_counter()
    nB, variants = int(XV4CFG.boot_replicates), tuple(XV4CFG.boot_variants)
    fixed_stats: List[dict] = []
    re_stats: List[dict] = []
    failed: List[Tuple[int, str]] = []
    t_fixed = t_re = 0.0
    for b in trange(nB, desc=f"    {cell} draw {draw}: B = {nB}", leave=False, disable=not SHOW_PROGRESS):
        mult = x_boot_weights(n_ep, np_rng("xenv", "bootstrap", cell, draw, b))
        if "fixed" in variants:
            ta = time.perf_counter()
            P_b, d0_b, st_b = x_boot_model(learn, smap, mult[eps_id], True)
            env_b = learn.variant(f"{learn.name}|boot{b}", P=P_b, d0=d0_b, kind="benchmark", bootstrap_replicate=b)
            del P_b
            st_b["trap_states_given"] = x_trap_states(env_b.P, env_b.live, np.ones((env_b.S, env_b.A), dtype=bool))
            st_b["trap_states_terminate"] = x_trap_states(env_b.P, env_b.live, env_b.mask)
            rb = _x_v4_boot_rows("fixed", b, labels, keys, _x_v4_boot_eval(env_b, pis))
            st_b["nonok_rows"] = int(sum(r["status_given"] != "ok" or r["status_term"] != "ok" for r in rb))
            rows += rb
            fixed_stats.append(st_b)
            del env_b, rb
            t_fixed += time.perf_counter() - ta
        if "reestimated" in variants:
            ta = time.perf_counter()
            hosp_b = x_v4_boot_dataset(hosp, mult)
            x_validate_dataset(hosp_b, true_env)
            Bb, err = x_v4_boot_construct(true_env, hosp_b, spec["tau"], f"{learn.name}|boot{b}")
            del hosp_b
            if Bb is None:
                failed.append((b, str(err)))
                rows += _x_v4_boot_rows("reestimated", b, labels, keys, None)
            else:
                env_b, idx = Bb["bench"], np.asarray(Bb["index"], dtype=np.int64)
                pis_b = {k: validate_policy(p_true[idx], f"lifted policy on {env_b.name}", shape=(env_b.S, env_b.A))
                         for k, p_true in lifted.items()}
                rb = _x_v4_boot_rows("reestimated", b, labels, keys, _x_v4_boot_eval(env_b, pis_b))
                st_b = x_v4_boot_construction_stats(Bb, B)
                st_b["nonok_rows"] = int(sum(r["status_given"] != "ok" or r["status_term"] != "ok" for r in rb))
                re_stats.append(st_b)
                rows += rb
                del env_b, pis_b, rb
            del Bb
            t_re += time.perf_counter() - ta
    t5 = time.perf_counter()
    df = pd.DataFrame(rows, columns=XV4_BOOT_RAW_COLUMNS)
    checks = x_v4_boot_check_rows(df, hi - lo, (nB, variants))
    msgs = Counter(m for _, m in failed)
    meta = _x_compact({
        "cell": cell, "family": family, "draw": draw, "identity_hash": ident_hash, "code_hash": XV4_BOOT_CODE_HASH[:16],
        "cache": "computed", "replicates": nB, "variants": list(variants), "resample_d0": True,
        "v_range": float(hi - lo), "n_policies": len(records), "unique_policies": len(pis),
        "policy_desc": {r["desc"]["policy"]: {"method": r["desc"]["method"], "policy_variant": r["desc"]["variant"],
                                              "method_family": r["desc"]["method_family"]} for r in records},
        "hospital_episodes": n_ep, "step_map": {k: v for k, v in smap.items() if not isinstance(v, np.ndarray)},
        "benchmark": {"decision_states": int(len(B["decision_states_true"])),
                      "admissible_pairs": int(B["report"]["admissible_pairs"]),
                      "empty_rule": B["report"]["empty_rule"]},
        "reproduction": repro,
        "fixed": {"replicates": len(fixed_stats), **_x_v4_boot_dist(fixed_stats),
                  "fallback_pairs_total": int(sum(s["fallback_pairs"] for s in fixed_stats)),
                  "trap_states_given_max": int(max([s["trap_states_given"] for s in fixed_stats], default=0))},
        "reestimated": {"replicates": len(re_stats) + len(failed), "failed": len(failed), **_x_v4_boot_dist(re_stats)},
        "failed_replicates": [b for b, _ in failed],
        "failure_messages": [[m, int(n)] for m, n in msgs.most_common(5)],
        "checks": checks,
        "timing": {"build_s": build_s, "methods_s": t2 - t1, "rebuild_s": t3 - t2, "point_eval_s": t4 - t3,
                   "fixed_s": t_fixed, "reestimated_s": t_re, "bootstrap_s": t5 - t4,
                   "total_s": time.perf_counter() - t0}})
    XCACHE.put(key + "/meta", identity, pd.DataFrame({"meta_json": [json.dumps(meta, default=_json_default)]}))
    XCACHE.put(key, identity, df)
    back = XCACHE.get(key, identity)
    assert back is not None and list(back.columns) == XV4_BOOT_RAW_COLUMNS and len(back) == len(df), \
        f"{key}: cache write"
    for c in ("J_given", "J_term", "p_given", "p_term"):
        assert np.allclose(back[c].to_numpy(dtype=float), df[c].to_numpy(dtype=float), rtol=1e-14, atol=0.0,
                           equal_nan=True), f"{key}: cache round trip changed column {c}"
    for c in ("variant", "policy", "status_given", "status_term", "replicate"):
        assert (back[c].astype(str).to_numpy() == df[c].astype(str).to_numpy()).all(), f"{key}: cache round trip {c}"
    del inst, records, pis, lifted, rows, df, smap, hosp, B, true_env
    gc.collect()
    return x_v4_boot_finish(spec, draw, back, meta)


def x_v4_boot_selftest(spec: dict) -> Dict[str, Any]:
    """Cheap asserted checks on draw 0 of `spec`: a resampled data set is a valid hospital data set with the right
    multiplicities, a failing construction is caught and reported, and a failed replicate gives 'failed' rows."""
    inst = x_v4_instance(spec, 0)
    true_env, _, hosp, _ = x_v4_rebuild_benchmark(spec, 0, inst)
    del inst
    ep = np.asarray(hosp["episode_id"], dtype=np.int64)
    n_ep = int(ep.max()) + 1
    mult = x_boot_weights(n_ep, np_rng("xenv", "v4", "boot-selftest", str(spec["cell"])))
    hb = x_v4_boot_dataset(hosp, mult)
    val = x_validate_dataset(hb, true_env)
    S, A = true_env.S, true_env.A
    want = np.bincount(hosp["state"] * A + hosp["action"], weights=mult[ep], minlength=S * A)
    assert val["episodes"] == n_ep and val["transitions"] == int(mult[ep].sum()), (val["episodes"], n_ep)
    assert np.array_equal(np.bincount(hb["state"] * A + hb["action"], minlength=S * A), want), "pair multiplicities"
    T = int(hosp["timestep"].max()) + 1
    assert np.array_equal(np.bincount(hb["timestep"], minlength=T),
                          np.bincount(hosp["timestep"], weights=mult[ep], minlength=T)), "timestep multiplicities"
    Bf, err = x_v4_boot_construct(true_env, hosp, 10 ** 9, "v4-boot-selftest", empty_rule="icu")
    assert Bf is None and err, "a construction without any supported pair must fail and be caught"
    fr = _x_v4_boot_rows("reestimated", 0, ["p"], [b"k"], None)[0]
    assert fr["status_given"] == fr["status_term"] == "failed" and np.isnan(fr["J_given"]), fr
    # the checks and intervals of a draw with a failed replicate (toy rows: replicate 0 of 'reestimated' failed)
    ok = {"status": "ok", "return": 0.25, "prob_any_violation": 0.0}
    one = {b"k": (ok, ok)}
    toy = pd.DataFrame([r for v, b, e in (("point", XV4_BOOT_POINT, one), ("truth", XV4_BOOT_POINT, one),
                                          ("fixed", 0, one), ("fixed", 1, one), ("reestimated", 0, None),
                                          ("reestimated", 1, one))
                        for r in _x_v4_boot_rows(v, b, ["p"], [b"k"], e)], columns=XV4_BOOT_RAW_COLUMNS)
    chk = x_v4_boot_check_rows(toy, 1.0, (2, XV4_BOOT_VARIANTS))
    tiv = x_v4_boot_intervals(toy, 1.0, XV4CFG.boot_quantiles).set_index("boot_variant")
    assert chk["failed_replicates"] == 1 and chk["nonok_rows"] == 0, chk
    assert (tiv.loc["reestimated", "n_failed"], tiv.loc["reestimated", "n_proper"]) == (1, 1), tiv
    assert np.isclose(tiv.loc["reestimated", "lo_nhb"], 0.25) and tiv.loc["fixed", "n_failed"] == 0, tiv
    return {"cell": spec["cell"], "episodes": n_ep, "resampled_transitions": int(val["transitions"]),
            "hospital_transitions": int(len(ep)), "failure_path_message": err}


# Objects whose source changes what a draw computes or stores (the checks and intervals are re-run on cached rows).
XV4_BOOT_CODE = [x_v4_boot_dataset, x_v4_boot_construct, _x_v4_boot_eval, _x_v4_boot_rows,
                 x_v4_boot_construction_stats, _x_v4_boot_dist, x_v4_boot_unit_check, x_v4_boot_identity,
                 run_v4_boot_draw, _x_v3_eval, x_boot_step_map, x_boot_weights, x_boot_model]
XV4_BOOT_CODE_HASH = obj_hash({"common": XV4_COMMON_CODE_HASH, "code": code_hash(*XV4_BOOT_CODE),
                               "constants": {"point": XV4_BOOT_POINT, "statuses": list(XV4_BOOT_STATUSES),
                                             "variants": list(XV4_BOOT_VARIANTS), "columns": XV4_BOOT_RAW_COLUMNS}})

XV4_BOOT_IV = pd.DataFrame(columns=XV4_BOOT_IV_COLUMNS)
XV4_BOOT_META = pd.DataFrame()
XV4_BOOT_CELLS: List[dict] = []
XV4_BOOT_SUMMARY: Dict[str, Any] = {"status": "not run (xenv_v4_bootstrap off)", "code_hash": XV4_BOOT_CODE_HASH,
                                    "draws_complete": None}
if EXPERIMENTS["xenv_v4_bootstrap"]:
    _t_boot = time.perf_counter()
    XV4_BOOT_CELLS, _xv4b_skipped = x_v4_cells(XV4CFG.boot_cells)
    _xv4b_skipped += [s["cell"] for s in XV4_BOOT_CELLS if s.get("construction", "best") != "best"]
    XV4_BOOT_CELLS = [s for s in XV4_BOOT_CELLS if s.get("construction", "best") == "best"]
    if _xv4b_skipped:
        print("NOTE: II.9d bootstrap cells not enabled (family switch off or not in the design), skipped:",
              _xv4b_skipped)
    if XV4_BOOT_CELLS:
        _XV4_BOOT_V3REF, _xv4b_v3src = x_v4_boot_v3_reference([s["cell"] for s in XV4_BOOT_CELLS])
        print(f"II.9d: II.8g rows for the comparison: {_xv4b_v3src} ({len(_XV4_BOOT_V3REF)} draws)")
        _xv4b_self = x_v4_boot_selftest(XV4_BOOT_CELLS[0])
        print(f"II.9d bootstrap: B = {XV4CFG.boot_replicates}, variants {list(XV4CFG.boot_variants)}")
        _xv4b_frames, XV4_BOOT_META, _xv4b_run = x_v4_run_draws(XV4_BOOT_CELLS, run_v4_boot_draw, "II.9d bootstrap")
        XV4_BOOT_IV = (pd.concat(_xv4b_frames, ignore_index=True) if _xv4b_frames
                       else pd.DataFrame(columns=XV4_BOOT_IV_COLUMNS))
        _xv4b_counts = {s["cell"]: int((XV4_BOOT_META["cell"] == s["cell"]).sum()) if len(XV4_BOOT_META) else 0
                        for s in XV4_BOOT_CELLS}
        XV4_BOOT_SUMMARY = {"status": "run", "code_hash": XV4_BOOT_CODE_HASH, **_xv4b_run,
                            "run_elapsed_s": float(time.perf_counter() - _t_boot), "cells_skipped": _xv4b_skipped,
                            "replicates": int(XV4CFG.boot_replicates), "variants": list(XV4CFG.boot_variants),
                            "quantiles": list(XV4CFG.boot_quantiles), "selftest": _xv4b_self,
                            "v3_reference": _xv4b_v3src, "draws_per_cell_present": _xv4b_counts,
                            "draws_complete": all(_xv4b_counts[s["cell"]] == x_v4_draws(s) for s in XV4_BOOT_CELLS)}
        del _xv4b_frames, _xv4b_run, _xv4b_counts, _xv4b_self, _xv4b_v3src
    else:
        XV4_BOOT_SUMMARY = {"status": "not run: no configured II.9d bootstrap cell is enabled",
                            "cells_skipped": _xv4b_skipped, "draws_complete": None, "code_hash": XV4_BOOT_CODE_HASH}
    del _xv4b_skipped, _t_boot
    gc.collect()
print("II.9d bootstrap:", XV4_BOOT_SUMMARY["status"], "|", len(XV4_BOOT_IV), "interval rows | code hash",
      XV4_BOOT_CODE_HASH[:16])

### II.9d Analyses: coverage over all default policies, certificates, family-wise errors, matched baselines

The analyses use the interval table of the run above (`XV4_BOOT_IV`). For the comparison with II.8g, the II.8g rows
of the same draws (B = 10, fixed mask; `XV3_BOOT_RESULTS` or its saved table) are turned into intervals with the
**same definitions** (variant `v3boot`; proper replicates = `ok` under both rules) and added to the table. A policy
enters `v3boot` only if its II.8g replicate −1 values (J_bench_term, J_bench_mean, J_true) equal this run's to 1e-9:
otherwise II.8g regenerated another policy (a near-tie broken differently on another machine), and the policy is
left out and counted (`v3boot_policies` in the summary). Variants:
`point` (the point benchmark), `v3boot` (II.8g, where available), `fixed` and `reestimated` (B = 200); interval
types `nhb`, `free` and `minmax` (the `point` variant has no `minmax`). Policies whose interval is undefined (no
proper replicate) are counted and left out of that interval's statistics.

* **Coverage** of J_true: lo − 1e-9 ≤ J_true ≤ up + 1e-9, over **all** default policies, and separately over the
  policies with p > 1e-12 and p = 0 on the point benchmark; lower / upper end holding; mean width.
* **Certificates** of an interval type: i is certified above j if lo_i > up_j + 1e-9. A claim is an error if
  J_true(i) ≤ J_true(j) + 1e-9 (ties are errors) and a reversal if J_true(i) < J_true(j) − 1e-9. Shares are over
  unordered pairs, overall and over completion-sensitive pairs (at least one policy with p > 0 on the point
  benchmark: types AU + UU of II.9b).
* **Family-wise errors per draw:** draws with at least one wrong certificate, and draws in which every policy is
  covered.
* **Matched-coverage margin baseline:** per draw, the k largest gaps of the point J_mean, J_term and midpoint values
  (`x_v4_margin_claims`), with k = the number of certified pairs of that interval type in that draw.
* Pools: per cell, per family (`pooled:gumbel`, `pooled:garnet`) and `pooled:all`.

Coverage and error rates are results: reported, never asserted. The summary also reports the invariant slacks of the
run, the unit-weight checks, the true-rows bracket count, the failed re-estimated replicates (count, draws, most
frequent messages), the variability of the re-estimated construction, and the agreement of the fixed replicates with
II.8g. If `checks.passes` is false (a theorem count or identity failed), the cell raises after writing its files.

| Output | Content |
|---|---|
| `XV4_BOOT_IV` · `xenv/v4_boot_intervals.csv.gz` | one row per draw × default policy × variant: truth, point values, p, replicate counts, the three intervals |
| `XV4_BOOT_DRAWS` · `xenv/v4_boot_draws.csv` | one row per draw × variant × interval type: coverage counts, certified pairs and errors (all and completion-sensitive), family-wise flags, margin baselines |
| `XV4_BOOT_COVERAGE` · `xenv/v4_boot_coverage.csv` | per cell and pooled × variant × interval type: coverage (all, p > 0, p = 0), widths, certified share and error rate, draws with any wrong certificate / all covered, margin baselines |
| `XV4_BOOT_META` · `xenv/v4_boot_meta.csv` | per draw meta of the run (flattened), timings |
| `XV4_BOOT_SUMMARY` · `xenv/v4_boot_summary.json` | status, code hash, checks, timing, draws complete, failures, II.8g agreement, headline numbers, outputs |

**Expected cost on Colab:** under a minute (pure pandas over at most a few hundred draws).

In [ ]:
# ============================================================
# II.9d Bootstrap analyses
# ============================================================
EXECLOG.mark("II.9 Second-revision experiments: bootstrap analyses")


def x_v4_boot_pair_stats(lo: np.ndarray, up: np.ndarray, p: np.ndarray, jt: np.ndarray, pm: np.ndarray,
                         pt: np.ndarray) -> Dict[str, Any]:
    """Coverage, certified comparisons lo_i > up_j and the matched-coverage margin baselines (k = certified pairs,
    on the point J_mean pm / J_term pt / midpoint) of one interval type over the policies of one draw."""
    lo, up, p, jt, pm, pt = (np.asarray(x, dtype=float) for x in (lo, up, p, jt, pm, pt))
    n = len(lo)
    off = ~np.eye(n, dtype=bool)
    u = p > XV4_PTOL
    cs = off & (u[:, None] | u[None, :])
    C = _x_v4_gt(lo, up) & off
    ordered = _x_v4_gt(jt, jt)
    wrong, rev = ~ordered, ordered.T
    lo_ok, up_ok = jt >= lo - XV4_TOL, jt <= up + XV4_TOL
    cov = lo_ok & up_ok
    k = int(C.sum())
    out = {"policies": n, "pairs": n * (n - 1) // 2, "pairs_cs": int(cs.sum() // 2), "certified": k,
           "certified_cs": int((C & cs).sum()), "errors": int((C & wrong).sum()),
           "errors_cs": int((C & cs & wrong).sum()), "reversals": int((C & rev).sum()), "covered": int(cov.sum()),
           "lower_fail": int((~lo_ok).sum()), "upper_fail": int((~up_ok).sum()), "n_ppos": int(u.sum()),
           "covered_ppos": int((cov & u).sum()), "n_pzero": int((~u).sum()), "covered_pzero": int((cov & ~u).sum()),
           "width_sum": float((up - lo).sum()), "width_sum_ppos": float((up - lo)[u].sum()),
           "any_wrong": bool((C & wrong).any()), "all_covered": bool(cov.all())}
    for nm, v in (("mean", pm), ("term", pt), ("mid", (pm + pt) / 2.0)):
        M = x_v4_margin_claims(v, k)
        out[f"margin_{nm}_claims"], out[f"margin_{nm}_errors"] = int(M.sum()), int((M & wrong).sum())
        out[f"margin_{nm}_any_wrong"] = bool((M & wrong).any())
    return out


def x_v4_boot_draw_stats(iv: pd.DataFrame) -> pd.DataFrame:
    """One row per (cell, draw, variant, interval type) with x_v4_boot_pair_stats over the defined intervals."""
    rows = []
    for (c, dr, var), d in iv.groupby(["cell", "draw", "boot_variant"], sort=False):
        for it in XV4_BOOT_INTERVALS:
            lc, uc = f"lo_{it}", f"up_{it}"
            ok = d[[lc, uc, "true", "p_bench", "point_mean", "point_term"]].notna().all(axis=1)
            if not ok.any():
                continue
            dd = d[ok]
            rows.append({"cell": c, "family": d["family"].iloc[0], "draw": int(dr), "boot_variant": var,
                         "interval": it, "policies_without_interval": int((~ok).sum()),
                         "median_proper_replicates": float(dd["n_proper"].median()),
                         **x_v4_boot_pair_stats(*(dd[col].to_numpy(float) for col in
                                                  (lc, uc, "p_bench", "true", "point_mean", "point_term")))})
    return pd.DataFrame(rows)


def x_v4_boot_coverage(D: pd.DataFrame, cells: Sequence[str]) -> pd.DataFrame:
    """Per cell, per family pool and pooled, x variant x interval type: sums of the draw statistics."""
    if D.empty:
        return pd.DataFrame()
    fams = sorted(set(D["family"]))
    units = [(c, [c]) for c in cells if c in set(D["cell"])]
    units += [(f"pooled:{f}", sorted(set(D.loc[D["family"] == f, "cell"]))) for f in fams]
    if len(fams) > 1:
        units.append(("pooled:all", sorted(set(D["cell"]))))
    rows = []
    for name, cs in units:
        du = D[D["cell"].isin(cs)]
        for var in XV4_BOOT_VARIANT_ORDER:
            for it in XV4_BOOT_INTERVALS:
                d = du[(du["boot_variant"] == var) & (du["interval"] == it)]
                if d.empty:
                    continue
                s = d.drop(columns=["cell", "family", "draw", "boot_variant", "interval"]).sum(numeric_only=True)
                nd, aw, ac = len(d), int(d["any_wrong"].astype(bool).sum()), int(d["all_covered"].astype(bool).sum())
                r = {"cell": name, "n_cells": len(cs), "boot_variant": var, "interval": it, "draws": nd,
                     "policy_rows": int(s["policies"]),
                     "policies_without_interval": int(s["policies_without_interval"]),
                     "n_ppos": int(s["n_ppos"]), "n_pzero": int(s["n_pzero"]),
                     "coverage_all": _xv3_rate(s["covered"], s["policies"]),
                     "coverage_ppos": _xv3_rate(s["covered_ppos"], s["n_ppos"]),
                     "coverage_pzero": _xv3_rate(s["covered_pzero"], s["n_pzero"]),
                     "lower_holds": _xv3_rate(s["policies"] - s["lower_fail"], s["policies"]),
                     "upper_holds": _xv3_rate(s["policies"] - s["upper_fail"], s["policies"]),
                     "mean_width": _xv3_rate(s["width_sum"], s["policies"]),
                     "mean_width_ppos": _xv3_rate(s["width_sum_ppos"], s["n_ppos"]),
                     "pairs": int(s["pairs"]), "certified": int(s["certified"]),
                     "certified_share": _xv3_rate(s["certified"], s["pairs"]), "pairs_cs": int(s["pairs_cs"]),
                     "certified_cs": int(s["certified_cs"]),
                     "certified_share_cs": _xv3_rate(s["certified_cs"], s["pairs_cs"]),
                     "errors": int(s["errors"]), "error_rate": _xv3_rate(s["errors"], s["certified"]),
                     "errors_cs": int(s["errors_cs"]), "error_rate_cs": _xv3_rate(s["errors_cs"], s["certified_cs"]),
                     "reversals": int(s["reversals"]), "draws_any_wrong": aw,
                     "draws_any_wrong_share": _xv3_rate(aw, nd),
                     "draws_all_covered": ac, "draws_all_covered_share": _xv3_rate(ac, nd)}
                for nm in ("mean", "term", "mid"):
                    mw = int(d[f"margin_{nm}_any_wrong"].astype(bool).sum())
                    r.update({f"margin_{nm}_claims": int(s[f"margin_{nm}_claims"]),
                              f"margin_{nm}_errors": int(s[f"margin_{nm}_errors"]),
                              f"margin_{nm}_error_rate": _xv3_rate(s[f"margin_{nm}_errors"], s[f"margin_{nm}_claims"]),
                              f"margin_{nm}_draws_any_wrong": mw})
                rows.append(r)
    return pd.DataFrame(rows)


def x_v4_boot_v3_intervals(iv: pd.DataFrame, meta: pd.DataFrame, ref: Dict[Tuple[str, int], pd.DataFrame]
                           ) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """(II.8g intervals, counts): the fixed-mask B = XV3BOOT.replicates intervals of the draws in `iv`, with this
    subsection's definitions (variant 'v3boot'). The policy descriptions, the truth, the point values and p are those
    of this run's point rows of the same draw (so every variant of a draw is scored against identical values).

    A policy is kept only if its II.8g replicate -1 values (J_bench_term, J_bench_mean, J_true) equal this run's to
    XV4_TOL, i.e. II.8g regenerated the same policy: a near-tie broken differently on another machine (II.8g's
    point check) gives another policy, whose interval must not be scored against this run's truth. The other
    policies, and policies present on one side only, are left out and counted."""
    rep = {"draws": 0, "policies": 0, "policies_value_mismatch": 0, "policies_missing_in_v3": 0,
           "policies_not_in_v4": 0}
    if iv.empty or not ref or meta.empty:
        return pd.DataFrame(columns=XV4_BOOT_IV_COLUMNS), rep
    vr = {(str(c), int(d)): float(v) for c, d, v in zip(meta["cell"], meta["draw"], meta["v_range"])}
    same = ["true", "p_true", "true_rows_term", "point_term", "point_mean", "p_bench"]
    chk = ["point_term", "point_mean", "true"]
    frames = []
    for (c, dr), g in iv[iv["boot_variant"] == "point"].groupby(["cell", "draw"], sort=False):
        r = ref.get((str(c), int(dr)))
        if r is None or not (r["variant"] == "v3boot").any():
            continue
        one = x_v4_boot_intervals(r, vr[(str(c), int(dr))], XV4CFG.boot_quantiles)
        v3pt, mine = one[one["boot_variant"] == "point"].set_index("policy"), g.set_index("policy")
        both = mine.index.intersection(v3pt.index)
        dv = np.abs(v3pt.loc[both, chk].to_numpy(float) - mine.loc[both, chk].to_numpy(float))
        keep = both[(dv <= XV4_TOL).all(axis=1)]              # NaN (a missing II.8g value) is not kept
        rep["policies_missing_in_v3"] += int(len(mine.index.difference(v3pt.index)))
        rep["policies_not_in_v4"] += int(len(v3pt.index.difference(mine.index)))
        rep["policies_value_mismatch"] += int(len(both) - len(keep))
        one = one[(one["boot_variant"] == "v3boot") & one["policy"].isin(keep)].drop(columns=same).merge(
            g[["policy", "method", "policy_variant", "method_family", *same]], on="policy", how="inner",
            validate="one_to_one")
        if one.empty:
            continue
        one["cell"], one["family"], one["draw"] = c, g["family"].iloc[0], int(dr)
        frames.append(one[XV4_BOOT_IV_COLUMNS])
        rep["draws"] += 1
        rep["policies"] += int(len(one))
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=XV4_BOOT_IV_COLUMNS)
    return out, rep


def _x_v4_boot_col(meta: pd.DataFrame, col: str, how: str):
    """max / sum / all of a flattened meta column (None when the column is absent)."""
    if meta.empty or col not in meta.columns:
        return None
    v = meta[col]
    if how == "all":
        return bool(v.astype(bool).all())
    v = pd.to_numeric(v, errors="coerce")
    return float(v.max()) if how == "max" else float(v.sum())


def x_v4_boot_run_checks(meta: pd.DataFrame) -> Dict[str, Any]:
    """Invariant slacks and unit-weight checks over the draws of the run (asserted per draw), and the true-rows bracket
    J_rows_term <= J_true <= J_rows_term + p (v_max - v_min) (Theorem 1 on the true rows; counted per draw, and a
    violation fails `passes`, as II.8i / II.9b count their theorem facts)."""
    out = {"max_rule_p_difference": _x_v4_boot_col(meta, "checks.max_rule_p_difference", "max"),
           "max_negative_width": _x_v4_boot_col(meta, "checks.max_negative_width", "max"),
           "max_width_minus_bound": _x_v4_boot_col(meta, "checks.max_width_minus_bound", "max"),
           "unit_weights_max_abs_diff_P": _x_v4_boot_col(meta, "reproduction.max_abs_diff_P", "max"),
           "unit_weights_max_abs_diff_d0": _x_v4_boot_col(meta, "reproduction.max_abs_diff_d0", "max"),
           "unit_weights_construction_identity": _x_v4_boot_col(meta, "reproduction.construction_identity", "all"),
           "truth_rows_bracket_violations": _x_v4_boot_col(meta, "checks.truth_rows_bracket_violations", "sum")}
    theorem = all((out[k] or 0.0) <= XV4_TOL
                  for k in ("max_rule_p_difference", "max_negative_width", "max_width_minus_bound"))
    unit = all((out[k] or 0.0) <= 1e-14 for k in ("unit_weights_max_abs_diff_P", "unit_weights_max_abs_diff_d0"))
    out["passes"] = (bool(len(meta)) and theorem and unit and out["unit_weights_construction_identity"] is not False
                     and not out["truth_rows_bracket_violations"])
    return out


def x_v4_boot_failures(meta: pd.DataFrame) -> Dict[str, Any]:
    """Failed re-estimated replicates: totals, draws affected, per cell and the most frequent messages."""
    if meta.empty or "reestimated.failed" not in meta.columns:
        return {"total": 0, "draws_with_failure": 0}
    f = pd.to_numeric(meta["reestimated.failed"], errors="coerce").fillna(0).astype(int)
    n = pd.to_numeric(meta["reestimated.replicates"], errors="coerce").fillna(0).astype(int)
    msgs: Counter = Counter()
    unparsed = 0
    for txt in meta.get("failure_messages", pd.Series(dtype=object)).dropna():
        try:      # a JSON list; _x_compact replaces a list longer than its limit by a hashed summary string
            pairs = json.loads(str(txt))
        except json.JSONDecodeError:
            unparsed += 1
            continue
        for m, c in pairs:
            msgs[str(m)] += int(c)
    return {"total": int(f.sum()), "replicates": int(n.sum()), "share": _xv3_rate(int(f.sum()), int(n.sum())),
            "draws_with_failure": int((f > 0).sum()),
            "by_cell": {str(c): int(v) for c, v in f.groupby(meta["cell"]).sum().items()},
            "messages": [[m, int(c)] for m, c in msgs.most_common(10)], "draws_messages_unparsed": unparsed}


def x_v4_boot_construction_summary(meta: pd.DataFrame) -> Dict[str, Dict[str, Optional[float]]]:
    """Per cell: the mean over draws of the per-draw median of each re-estimated construction statistic."""
    stats_ = ("decision_states_new", "decision_states_lost", "mask_rows_changed_share", "admissible_pairs",
              "dead_states_removed", "trap_states_censored", "fallback_action_changed")
    out: Dict[str, Dict[str, Optional[float]]] = {}
    for c, g in (meta.groupby("cell", sort=False) if len(meta) and "cell" in meta.columns else []):
        out[str(c)] = {s: (float(pd.to_numeric(g[f"reestimated.{s}.median"], errors="coerce").mean())
                           if f"reestimated.{s}.median" in g.columns else None) for s in stats_}
    return out


def x_v4_boot_v3_summary(meta: pd.DataFrame) -> Dict[str, Any]:
    """Agreement of the fixed replicates (and replicate -1) with the II.8g rows, over the compared draws."""
    if meta.empty or "v3_match.status" not in meta.columns:
        return {"draws_compared": 0}
    m = meta[meta["v3_match.status"] == "compared"]
    if m.empty:
        return {"draws_compared": 0}
    return {"draws_compared": int(len(m)),
            "rows": int(pd.to_numeric(m["v3_match.rows"]).sum()),
            "missing_in_v4": int(pd.to_numeric(m["v3_match.missing_in_v4"]).sum()),
            "status_mismatches": int(pd.to_numeric(m["v3_match.status_mismatches"]).sum()),
            "max_abs_diff_return": _x_v4_boot_col(m, "v3_match.max_abs_diff_return", "max"),
            "max_abs_diff_p": _x_v4_boot_col(m, "v3_match.max_abs_diff_p", "max"),
            "draws_matching": int(m["v3_match.match"].astype(bool).sum()), "tolerance": XV4_TOL}


def x_v4_boot_headline(cov: pd.DataFrame) -> Dict[str, Any]:
    """Headline numbers of the widest pool, per variant x interval type."""
    if cov.empty:
        return {}
    pool = next((p for p in ("pooled:all", "pooled:gumbel", "pooled:garnet") if (cov["cell"] == p).any()),
                str(cov["cell"].iloc[0]))
    keep = ("coverage_all", "coverage_ppos", "coverage_pzero", "certified_share", "certified_share_cs", "error_rate",
            "error_rate_cs", "draws_any_wrong_share", "draws_all_covered_share", "margin_mid_error_rate")
    return {"pool": pool, **{f"{r['boot_variant']}|{r['interval']}": {k: r[k] for k in keep}
                             for _, r in cov[cov["cell"] == pool].iterrows()}}


XV4_BOOT_DRAWS = pd.DataFrame()
XV4_BOOT_COVERAGE = pd.DataFrame()
_xv4b_out = {"intervals": "xenv/v4_boot_intervals.csv.gz", "draws": "xenv/v4_boot_draws.csv",
             "coverage": "xenv/v4_boot_coverage.csv", "meta": "xenv/v4_boot_meta.csv",
             "summary": "xenv/v4_boot_summary.json"}
if XV4_BOOT_SUMMARY.get("status") == "run":
    _t_an = time.perf_counter()
    _xv4b_v3iv, _xv4b_v3rep = x_v4_boot_v3_intervals(XV4_BOOT_IV, XV4_BOOT_META, _XV4_BOOT_V3REF)
    if _xv4b_v3rep["policies_value_mismatch"] or _xv4b_v3rep["policies_missing_in_v3"]:
        print("NOTE: II.8g policies left out of the v3boot intervals (replicate -1 values differ from this run's, or "
              "missing):", _xv4b_v3rep)
    if len(_xv4b_v3iv):
        XV4_BOOT_IV = pd.concat([XV4_BOOT_IV[XV4_BOOT_IV["boot_variant"] != "v3boot"], _xv4b_v3iv], ignore_index=True)
    XV4_BOOT_DRAWS = x_v4_boot_draw_stats(XV4_BOOT_IV)
    XV4_BOOT_COVERAGE = x_v4_boot_coverage(XV4_BOOT_DRAWS, [s["cell"] for s in XV4_BOOT_CELLS])
    XV4_BOOT_IV.to_csv(XENV_TABLE_DIR / "v4_boot_intervals.csv.gz", index=False, compression="gzip")
    XV4_BOOT_DRAWS.to_csv(XENV_TABLE_DIR / "v4_boot_draws.csv", index=False)
    XV4_BOOT_COVERAGE.to_csv(XENV_TABLE_DIR / "v4_boot_coverage.csv", index=False)
    XV4_BOOT_META.to_csv(XENV_TABLE_DIR / "v4_boot_meta.csv", index=False)
    _xv4b_checks = x_v4_boot_run_checks(XV4_BOOT_META)
    XV4_BOOT_SUMMARY.update({
        "checks": _xv4b_checks, "max_rule_p_difference": _xv4b_checks["max_rule_p_difference"],
        "max_width_minus_bound": _xv4b_checks["max_width_minus_bound"],
        "failed_replicates": x_v4_boot_failures(XV4_BOOT_META),
        "construction_variability": x_v4_boot_construction_summary(XV4_BOOT_META),
        "v3_match": x_v4_boot_v3_summary(XV4_BOOT_META),
        "v3boot_draws": int(_xv4b_v3rep["draws"]), "v3boot_policies": _xv4b_v3rep,
        "headline": x_v4_boot_headline(XV4_BOOT_COVERAGE),
        "interval_rows": int(len(XV4_BOOT_IV)), "draw_rows": int(len(XV4_BOOT_DRAWS)),
        "analysis_elapsed_s": float(time.perf_counter() - _t_an), "outputs": _xv4b_out})
    XV4_BOOT_SUMMARY["elapsed_s"] = float(XV4_BOOT_SUMMARY["run_elapsed_s"] + XV4_BOOT_SUMMARY["analysis_elapsed_s"])
    if XV4_BOOT_SUMMARY["v3_match"].get("draws_compared") and \
            XV4_BOOT_SUMMARY["v3_match"].get("draws_matching") != XV4_BOOT_SUMMARY["v3_match"]["draws_compared"]:
        print("NOTE: fixed-mask replicates differ from the II.8g rows on some draws (reported, not asserted):",
              {k: XV4_BOOT_SUMMARY["v3_match"][k] for k in ("status_mismatches", "max_abs_diff_return")})
    if len(XV4_BOOT_COVERAGE):
        display(XV4_BOOT_COVERAGE[XV4_BOOT_COVERAGE["cell"].str.startswith("pooled:")][
            ["cell", "boot_variant", "interval", "draws", "coverage_all", "coverage_ppos", "coverage_pzero",
             "certified_share", "certified_share_cs", "error_rate", "draws_any_wrong_share",
             "draws_all_covered_share"]].round(4))
    del _t_an, _xv4b_v3iv, _xv4b_v3rep, _xv4b_checks
else:
    XV4_BOOT_SUMMARY.setdefault("checks", {})
    XV4_BOOT_SUMMARY.setdefault("elapsed_s", 0.0)
save_json(XV4_BOOT_SUMMARY, XENV_TABLE_DIR / "v4_boot_summary.json")
print(json.dumps({k: v for k, v in XV4_BOOT_SUMMARY.items()
                  if k not in ("draws_per_cell", "construction_variability", "headline")}, indent=1,
                 default=_json_default))
del _xv4b_out
gc.collect()
if XV4_BOOT_SUMMARY.get("checks", {}).get("passes") is False:
    raise AssertionError(f"II.9d checks failed: {XV4_BOOT_SUMMARY['checks']} (xenv/v4_boot_summary.json)")

### II.9e Sensitivity cells: timeout hazard, dead-state rule, physician support and state abstraction

The II.6 benchmarks fix three construction choices: the Gumbel timeout hazard η = 0.05 (the evaluation horizon,
20 decisions on average), the ICU-Sepsis dead-state rule `icu` (a decision state without an action reaching τ is
removed and its inflow dropped) and an ε-optimal physician whose count masks are near-singleton. II.8 added a
benchmark on aggregated observation cells. This section varies these choices in new cells that run the **full II.6
pipeline**: the same learners on user data of the same size, the same records (behaviour policy and the projection
of every native record), exact evaluation on every target, the reference rows, the strict per-draw invariant checks
(`x_check_draw_rows`) and the result columns `XRESULT_COLUMNS`. The lift of benchmark policies is varied in II.9f.

**Cells** (`x_v4_sens_build_cells`, from `XV4CFG`). A base cell that is not enabled is skipped and listed; in
smoke mode `x_v4_cells` substitutes the smoke cell of the same construction, and a cell that would then repeat
another is skipped.

* **timeout**: `<base>|eta<η>` for each (base, η) in `timeout_cells`, the base II.6 spec with `timeout_hazard` = η.
  The exact Gumbel system is enumerated at η (`x_v4_true_system`; `_x_v3_gumbel_env` caches one system per
  hazard, so a hazard never reuses the system of another), and the physician softens the optimum of that system.
  The random streams are named by the new cell.
* **deadrule**: `<base>|dead-argmax` for each base in `deadrule_cells` (Gumbel and Garnet), the base spec with
  `empty_rule = "argmax"`: a decision state without an action reaching τ keeps its most frequent action instead of
  being removed. The streams are named by the **base** cell (`stream_cell`), so the hospital episodes and the user
  and split streams are those of the base cell. Only the rule differs (paired design).
* **support**: `<support_base>|v4eps<ε>` for ε in `support_eps`, the base spec with physician ε. The physician is
  `_x_v3_gumbel_physician(variant, η, ε, "optimal")`. A softer physician logs more actions per state, so more pairs
  reach τ. The streams are named by the new cell.
* **abstraction**: `gumbel|agg-<kind>|n<n>|tau<τ>|eps<ε>` for kind in `aggregations`, with (n, τ, ε) =
  `aggregation_base`. This is the II.8c design (true full-state system, full-state physician, benchmark estimated
  on cells φ(s)) with a new φ from `x_v4_gumbel_aggregation`:
  * `glucose`: φ(s) = (hr, sbp, o2, abx, vaso, vent, diabetic). Glucose is hidden and diabetes visible
    (3·3·2·2·2·2·2 = 288 cells, 224 of them with a live state). Glucose then confounds: the physician sees it,
    and it changes death and discharge.
  * `kmeans<K>`: K cells from k-means over the observation features of the 606 true live states: (hr, sbp, o2,
    glucose, abx, vaso, vent), each level divided by (levels − 1), the scaling of the true environment's
    features. Diabetes is excluded, so it is a hidden confounder as in `agg-obs`. A state and its diabetic twin
    have the same features and always share a cell, so there are 303 distinct points and K ≤ 303. The clustering
    is deterministic: k-means++ seeding from `np_rng("xenv", "v4", "kmeans", K)`, then Lloyd iterations until the
    assignment repeats. Squared distances are computed elementwise (no BLAS, so the cells do not depend on the
    machine), ties go to the lowest cluster index, an empty cluster is re-seeded at the farthest point, and the
    cells are numbered by their first live state.

  `construct_aggregated_benchmark_v4` repeats `construct_aggregated_benchmark` step by step, with φ given as an
  argument. It reuses the II.8 functions unchanged: the population-limit model `x_aggregated_env`, the mapped
  hospital data `x_aggregate_dataset`, the estimate `x_benchmark_from_data` and the lift π(a | φ(s)). The targets
  are `bench_mean`, `bench_terminate` and `true` (lifted through φ), as in the II.8 aggregated cells. The reference
  cell is the II.8 cell `gumbel|agg-obs|n<n>|tau<τ>|eps<ε>` when it is enabled. In smoke mode, whose II.8 cells
  use other sizes, the abstraction cells take the sizes of the smoke `agg-obs` cell and pair with it.

**Per (cell, draw).**
1. `build_v4_instance(spec, draw)` mirrors the `best` branch of `build_cell_instance`. The hospital, user and split
   streams are `("xenv", purpose, spec.get("stream_cell", cell), draw)`. The hospital data are validated
   (`x_validate_dataset`), the benchmark is `x_benchmark_from_data(..., empty_rule=spec.get("empty_rule"))`, and
   the instance has the five targets, `true_optimum_in_mask_fraction` for Gumbel and the same meta. Abstraction
   cells take the aggregated construction above, as `build_v3_instance` does.
2. `run_v4_cell_draw` mirrors `run_v3_cell_draw`: the identity `{"kind", "draw": x_v4_draw_identity(spec, inst),
   "code": XV4_SENS_CODE_HASH}` is looked up in `XCACHE` under `v4sens/<cell>/draw<k>` (and `/meta`). On a miss,
   if `ALLOW_TRAINING`, the draw runs `run_xmethods` on the training episodes, `x_collect_records`, the FQE
   selection model, `x_evaluate_targets` and `x_check_draw_rows` (strict). The table returned is always the one
   read back from the cache, with a round-trip check.

`XV4_SENS_CODE_HASH` covers the source of every function of this section on the path of a draw and the module
constants they read. It folds in `XV4_COMMON_CODE_HASH`, and through it the Part II and II.8 code hashes.

**Self-test** (asserted; a few seconds).
1. Take the base spec of the first enabled sensitivity cell of each family, with `stream_cell` = its own name, the
   hazard unchanged and the default dead-state rule (`XCFG.benchmark_empty_rule`, `icu`). `build_v4_instance(spec,
   0)` must have the identity, behaviour policy and dev split of `build_cell_instance(spec, 0)`.
2. The factor aggregation must reproduce II.8's `obs` map, and `construct_aggregated_benchmark_v4` with that map
   must reproduce `construct_aggregated_benchmark(…, "obs", …)`: benchmark identity, pulled-back mask, decision
   states and lift.
3. Every abstraction of the full design (`glucose`, `kmeans50`, `kmeans150`) is computed twice; the two maps must
   be identical, with non-empty cells. This also exercises the k-means path in smoke mode, where only the first
   abstraction runs as a cell.

In [ ]:
# ============================================================
# II.9e Sensitivity cells: cells, abstractions, instances, per-draw driver, self-test
# ============================================================
EXECLOG.mark("II.9e Sensitivity cells: definitions and self-test")

XV4_SENS_KINDS = ("timeout", "deadrule", "support", "abstraction")
XV4_AGG_FACTORS = {"glucose": ("hr", "sbp", "o2", "abx", "vaso", "vent", "diabetic")}   # factor maps (glucose hidden)
XV4_KMEANS_FEATURES = ("hr", "sbp", "o2", "glucose", "abx", "vaso", "vent")              # diabetes hidden
XV4_KMEANS_MAX_ITER = 1000                     # Lloyd iterations (+ empty-cluster re-seeds); convergence is asserted


def x_v4_sens_build_cells() -> Tuple[List[dict], Dict[str, Optional[str]], List[str]]:
    """(specs, reference cell of each spec, skipped names) of the II.9e cells (see markdown). The reference cell is
    used by the analyses only and never enters a spec, so a missing II.8 reference does not change any identity."""
    cells: List[dict] = []
    refs: Dict[str, Optional[str]] = {}
    skipped: List[str] = []

    def base_of(name: str, families: Sequence[str]) -> Optional[dict]:
        found, _ = x_v4_cells([name], families=families)
        b = found[0] if found else None
        return b if b is not None and "v3_kind" not in b and b.get("construction", "best") == "best" else None

    def add(spec: dict, ref: Optional[str], configured: str) -> None:
        if any(c["cell"] == spec["cell"] for c in cells):
            skipped.append(f"{configured} (would repeat {spec['cell']} after the smoke-mode substitution)")
            return
        assert spec["v4_kind"] in XV4_SENS_KINDS, spec["v4_kind"]
        cells.append(spec)
        refs[spec["cell"]] = ref

    for name, eta in XV4CFG.timeout_cells:
        assert 0.0 < float(eta) < 1.0, eta
        b = base_of(name, ("gumbel",))
        if b is None:
            skipped.append(f"{name}|eta{float(eta)}")
            continue
        add({**b, "cell": f"{b['cell']}|eta{float(eta)}", "timeout_hazard": float(eta), "v4_kind": "timeout"},
            str(b["cell"]), f"{name}|eta{float(eta)}")
    for name in XV4CFG.deadrule_cells:
        b = base_of(name, ("gumbel", "garnet"))
        if b is None or str(XCFG.benchmark_empty_rule) == "argmax":    # 'argmax' everywhere: the cell is its base
            skipped.append(f"{name}|dead-argmax")
            continue
        add({**b, "cell": f"{b['cell']}|dead-argmax", "empty_rule": "argmax", "stream_cell": str(b["cell"]),
             "v4_kind": "deadrule"}, str(b["cell"]), f"{name}|dead-argmax")
    b = base_of(XV4CFG.support_base, ("gumbel",))
    for eps in XV4CFG.support_eps:
        assert 0.0 < float(eps) < 1.0, eps
        if b is None:
            skipped.append(f"{XV4CFG.support_base}|v4eps{float(eps)}")
            continue
        add({**b, "cell": f"{b['cell']}|v4eps{float(eps)}", "eps": float(eps), "v4_kind": "support"}, str(b["cell"]),
            f"{XV4CFG.support_base}|v4eps{float(eps)}")
    n, tau, eps = XV4CFG.aggregation_base
    v3 = {str(s["cell"]): s for s in (globals().get("X_V3_CELLS") or [])}
    ref = f"gumbel|agg-obs|n{int(n)}|tau{int(tau)}|eps{float(eps)}"
    if ref not in v3:   # reference not enabled: none (listed as missing by the analyses)
        ref = None
        if RUN_MODE == "smoke":   # the smoke designs use other sizes: the agg-obs cell of the optimal physician
            ref = next((c for c, s in v3.items() if s.get("aggregation") == "obs" and s.get("physician") == "optimal"),
                       None)
            if ref is not None:   # and its sizes, as x_v4_cells substitutes the smoke cell of a construction
                n, tau, eps = v3[ref]["n_hosp"], v3[ref]["tau"], v3[ref]["eps"]
    for kind in XV4CFG.aggregations:
        kind = str(kind)
        assert kind in XV4_AGG_FACTORS or (kind.startswith("kmeans") and kind[6:].isdigit()), kind
        cell = f"gumbel|agg-{kind}|n{int(n)}|tau{int(tau)}|eps{float(eps)}"
        add({"cell": cell, "family": "gumbel", "v4_kind": "abstraction", "construction": f"agg-{kind}",
             "aggregation": kind, "n_hosp": int(n), "tau": int(tau), "eps": float(eps), "physician": "optimal",
             "draws": int(XCFG.gumbel_draws), "variant": XCFG.gumbel_variant,
             "timeout_hazard": float(XCFG.gumbel_timeout_hazard), "user_episodes": int(XCFG.gumbel_user_episodes)},
            ref, cell)
    names = [c["cell"] for c in cells]
    assert len(names) == len(set(names)), f"duplicate II.9e cell names: {names}"
    clash = sorted(set(names) & set(x_v4_all_specs()))
    assert not clash, f"II.9e cells must not reuse II.6 / II.8 cell names: {clash}"
    return cells, refs, skipped


# ---------- state abstractions of the exact Gumbel system ----------
def _x_v4_gumbel_live_layout(env: FiniteEnv) -> Tuple[Dict[str, Any], np.ndarray, int]:
    """(gumbel_state_space(), live full indices, number of live states) after the layout checks of II.8c."""
    space = gumbel_state_space()
    live_full = space["live_full"]
    n_live = int(env.meta["n_live"])
    assert env.kind == "gumbel" and n_live == len(live_full) and env.S == n_live + 3, env.name
    death, discharge, timeout = n_live, n_live + 1, n_live + 2
    assert (env.terminals, env.sink, env.goal, int(env.meta["timeout_state"])) == \
        ((death, discharge, timeout), death, discharge, timeout), "unexpected absorbing layout"
    return space, live_full, n_live


def _x_v4_factor_aggregation(env: FiniteEnv, label: str, factors: Sequence[str]) -> Dict[str, Any]:
    """x_gumbel_aggregation (II.8c) for any set of Gumbel factors (first factor most significant): phi maps a live
    state to the rank of its factor tuple among the tuples of live states; absorbing states follow in true order."""
    space, live_full, n_live = _x_v4_gumbel_live_layout(env)
    factors = tuple(str(f) for f in factors)
    assert factors and len(set(factors)) == len(factors) and set(factors) <= set(GumbelSim.FACTORS), factors
    fidx = [GumbelSim.FACTORS.index(f) for f in factors]
    radix = np.array([GumbelSim.LEVELS[i] for i in fidx], dtype=np.int64)
    strides = np.ones(len(radix), dtype=np.int64)
    for j in range(len(radix) - 2, -1, -1):
        strides[j] = strides[j + 1] * radix[j + 1]
    full = np.array([space["full"][f] for f in live_full], dtype=np.int64)
    cell = full[:, fidx] @ strides
    keys = np.unique(cell)
    n_agg = int(len(keys))
    phi = np.empty(env.S, dtype=np.int64)
    phi[:n_live] = np.searchsorted(keys, cell)
    phi[n_live:] = n_agg + np.arange(3)
    levels = (keys[:, None] // strides[None, :]) % radix[None, :]
    members = np.bincount(phi[:n_live], minlength=n_agg)
    assert members.min() >= 1 and int(members.sum()) == n_live
    return {"aggregation": str(label), "phi": phi, "n_live": n_agg, "S": n_agg + 3, "cells_total": int(radix.prod()),
            "factors": factors, "radix": radix, "keys": keys, "levels": levels, "members": members,
            "report": {"method": "factors", "factors": list(factors)}}


def x_v4_kmeans(X: np.ndarray, K: int, rng: np.random.Generator,
                max_iter: int = XV4_KMEANS_MAX_ITER) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    """Deterministic k-means of the rows of X into K non-empty clusters: (labels [n], centres [K, d], info).

    k-means++ seeding from `rng`, then Lloyd iterations until the assignment repeats. Squared Euclidean distances are
    computed elementwise (no BLAS, so the result does not depend on the machine); ties go to the lowest cluster index;
    an empty cluster is re-seeded at the point farthest from its centre among clusters with more than one member
    (lowest row index on ties), and the next assignment is always followed by a centre update, so the result is a
    fixed point of Lloyd's map (centres = member means). Clusters are numbered by their first row, so the labels do
    not depend on the seeding order."""
    X = np.asarray(X, dtype=np.float64)
    n, K = int(len(X)), int(K)
    n_distinct = int(len(np.unique(X, axis=0)))
    assert X.ndim == 2 and np.isfinite(X).all() and 1 <= K <= n_distinct, (X.shape, K, n_distinct)

    def sqdist(C: np.ndarray) -> np.ndarray:
        return ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)

    C = np.empty((K, X.shape[1]))
    C[0] = X[int(rng.integers(0, n))]
    d2 = sqdist(C[:1])[:, 0]
    for k in range(1, K):                         # k-means++: P(row) proportional to its squared distance
        cum = np.cumsum(d2)
        assert cum[-1] > 0.0, "k-means++: fewer distinct rows than clusters"
        i = int(np.searchsorted(cum, rng.random() * cum[-1], side="right"))
        i = min(i, int(np.flatnonzero(d2 > 0.0)[-1]))          # rounding at the top end never picks a chosen row
        C[k] = X[i]
        d2 = np.minimum(d2, sqdist(C[k:k + 1])[:, 0])
    labels = np.full(n, -1, dtype=np.int64)
    reseeds, converged, it = 0, False, 0
    for it in range(1, int(max_iter) + 1):
        D = sqdist(C)
        new = D.argmin(axis=1)                                   # ties -> lowest cluster index
        counts = np.bincount(new, minlength=K)
        if (counts == 0).any():
            own = np.where(counts[new] > 1, D[np.arange(n), new], -1.0)
            assert own.max() > 0.0, "k-means: no row can re-seed an empty cluster"
            C[int(np.flatnonzero(counts == 0)[0])] = X[int(own.argmax())]
            reseeds += 1
            labels = np.full(n, -1, dtype=np.int64)       # the next assignment is accepted and every centre updated
            continue
        if np.array_equal(new, labels):
            converged = True
            break
        labels = new
        for j in range(X.shape[1]):
            C[:, j] = np.bincount(labels, weights=X[:, j], minlength=K) / counts
    assert converged, f"k-means (K={K}) did not converge in {max_iter} iterations"
    first = np.array([int(np.flatnonzero(labels == k)[0]) for k in range(K)])
    order = np.argsort(first, kind="stable")
    rank = np.empty(K, dtype=np.int64)
    rank[order] = np.arange(K)
    labels, C = rank[labels], C[order]
    info = {"K": K, "rows": n, "distinct_rows": n_distinct, "iterations": int(it), "reseeds": int(reseeds),
            "converged": bool(converged), "inertia": float(((X - C[labels]) ** 2).sum())}
    return labels, C, info


def x_v4_gumbel_aggregation(env: FiniteEnv, kind: str) -> Dict[str, Any]:
    """Index map phi of the exact Gumbel env for an II.9e abstraction, in the dict layout of x_gumbel_aggregation.

    'glucose': the factor map XV4_AGG_FACTORS['glucose'] (glucose hidden, diabetes visible). 'kmeansK': x_v4_kmeans
    of the scaled observation features XV4_KMEANS_FEATURES of the true live states into K cells; the cell 'levels' are
    the cluster centres with radix 1, so x_aggregated_env uses the centres as the cell features."""
    kind = str(kind)
    if kind in XV4_AGG_FACTORS:
        return _x_v4_factor_aggregation(env, kind, XV4_AGG_FACTORS[kind])
    assert kind.startswith("kmeans") and kind[6:].isdigit(), f"unknown II.9e aggregation {kind!r}"
    K = int(kind[6:])
    space, live_full, n_live = _x_v4_gumbel_live_layout(env)
    fidx = [GumbelSim.FACTORS.index(f) for f in XV4_KMEANS_FEATURES]
    scale = np.array([GumbelSim.LEVELS[i] - 1 for i in fidx], dtype=np.float64)
    X = np.array([[space["full"][f][i] for i in fidx] for f in live_full], dtype=np.float64) / scale
    labels, centres, info = x_v4_kmeans(X, K, np_rng("xenv", "v4", "kmeans", K))
    phi = np.empty(env.S, dtype=np.int64)
    phi[:n_live] = labels
    phi[n_live:] = K + np.arange(3)
    members = np.bincount(labels, minlength=K)
    assert members.min() >= 1 and int(members.sum()) == n_live
    return {"aggregation": kind, "phi": phi, "n_live": K, "S": K + 3, "cells_total": K,
            "factors": XV4_KMEANS_FEATURES, "radix": np.ones(len(fidx), dtype=np.int64), "keys": np.arange(K),
            "levels": centres, "members": members,
            "report": {"method": "kmeans", "features": list(XV4_KMEANS_FEATURES),
                       "feature_scaling": "level / (levels - 1)", **info}}


_X_V4_AGG: Dict[Tuple[str, str], Dict[str, Any]] = {}


def _x_v4_aggregation(env: FiniteEnv, kind: str) -> Dict[str, Any]:
    """x_v4_gumbel_aggregation, cached per (system, kind) within the session."""
    key = (str(env.name), str(kind))
    if key not in _X_V4_AGG:
        _X_V4_AGG[key] = x_v4_gumbel_aggregation(env, kind)
    return _X_V4_AGG[key]


def construct_aggregated_benchmark_v4(true_env: FiniteEnv, behavior: np.ndarray, agg: Dict[str, Any], n_hosp: int,
                                      tau: int, rng: np.random.Generator, name: str) -> Dict[str, Any]:
    """construct_aggregated_benchmark (II.8c) with the aggregation `agg` (x_v4_gumbel_aggregation or
    x_gumbel_aggregation) given instead of its name: the same steps, report keys and outputs (II.8 functions reused)."""
    kind = str(agg["aggregation"])
    phi = agg["phi"]
    agg_env, agg_check = x_aggregated_env(true_env, agg, behavior, f"{true_env.name}|agg-{kind}")
    hosp = x_sample_dataset(true_env, behavior, int(n_hosp), XCFG.max_steps, rng)
    hosp_report = x_validate_dataset(hosp, true_env)
    hosp_agg = x_aggregate_dataset(hosp, phi, agg_env)
    agg_report = x_validate_dataset(hosp_agg, agg_env)
    out = x_benchmark_from_data(agg_env, hosp_agg, tau, name)
    lift_cells = out["lift"]
    S, A = true_env.S, true_env.A

    def lift(pi_bench: np.ndarray) -> np.ndarray:
        """Benchmark policy -> true full-state policy pi(a | phi(s)) (absorbing rows uniform)."""
        return lift_cells(pi_bench)[phi]

    mask_true = out["mask_true"][phi]
    assert mask_true.shape == (S, A) and mask_true[~true_env.live].all()
    cells = out["decision_states_true"]                     # cell indices of the benchmark's decision states
    in_cells = true_env.live & np.isin(phi, cells)
    logged = np.zeros(S, dtype=bool)
    logged[np.unique(hosp["state"])] = True
    rep = out["report"]
    rep.update({"construction": f"agg-{kind}", "aggregation": kind,
                "aggregated_cells_total": int(agg["cells_total"]), "aggregated_live_cells": int(agg["n_live"]),
                "members_per_live_cell_max": int(agg["members"].max()),
                "full_live_states": int(true_env.live.sum()), "full_states_logged": int(logged.sum()),
                "full_live_states_in_decision_cells": int(in_cells.sum()),
                "full_logged_states_in_decision_cells": int((in_cells & logged).sum()),
                "full_d0_mass_in_decision_cells": float(true_env.d0[in_cells].sum()),
                **{f"aggregation_check_{k}": v for k, v in agg_check.items()},
                **{f"phi_{k}": v for k, v in (agg.get("report") or {}).items()},
                "hospital_validation": hosp_report, "aggregated_hospital_validation": agg_report})
    del hosp, hosp_agg
    return {"bench": out["bench"], "bench_behavior": out["bench_behavior"], "index": out["index"],
            "mask_true": mask_true, "lift": lift, "fallback_action": out["fallback_action"], "report": rep,
            "decision_states_true": np.flatnonzero(in_cells & logged), "decision_cells": cells, "phi": phi,
            "agg_env": agg_env}


# ---------- instances and the per-draw driver ----------
def build_v4_instance(spec: dict, draw: int) -> CellInstance:
    """Deterministic (cell, draw) instance of a Section II.9e cell: the 'best' branch of build_cell_instance with the
    streams named by spec['stream_cell'] (default: the cell), the dead-state rule spec['empty_rule'] (default:
    XCFG.benchmark_empty_rule) and the spec's own hazard and physician eps; abstraction cells ('aggregation' in the
    spec) take the aggregated construction of build_v3_instance with the II.9e phi."""
    t0 = time.perf_counter()
    cell, family, draw = spec["cell"], spec["family"], int(draw)
    stream = str(spec.get("stream_cell", cell))
    aggregated = "aggregation" in spec
    assert draw >= 0 and family in ("gumbel", "garnet") and "v3_kind" not in spec, (draw, family, spec.get("v3_kind"))
    assert aggregated or spec.get("construction", "best") == "best", spec.get("construction")
    rng_u = np_rng("xenv", "user", stream, draw)
    true_env, true_behavior = x_v4_true_system_draw(spec, draw)
    if family == "gumbel":
        assert float(true_env.meta["timeout_hazard"]) == float(spec["timeout_hazard"]), (true_env.name, spec["cell"])
    bname = f"{cell}|d{draw}|bench"
    if aggregated:
        assert family == "gumbel" and spec.get("empty_rule") is None, (family, spec.get("empty_rule"))
        B = construct_aggregated_benchmark_v4(true_env, true_behavior, _x_v4_aggregation(true_env, spec["aggregation"]),
                                              spec["n_hosp"], spec["tau"], np_rng("xenv", "hospital", stream, draw),
                                              bname)
    else:   # construct_benchmark with the stream cell and the dead-state rule of the spec
        hosp = x_v4_hospital(spec, draw, true_env, true_behavior, stream_cell=stream)
        hosp_report = x_validate_dataset(hosp, true_env)
        B = x_benchmark_from_data(true_env, hosp, spec["tau"], bname, empty_rule=spec.get("empty_rule"))
        B["report"]["hospital_validation"] = hosp_report
        del hosp
    learn_env, behavior, lift = B["bench"], B["bench_behavior"], B["lift"]
    data = x_sample_dataset(learn_env, behavior, spec["user_episodes"], XCFG.max_steps, rng_u)
    T_masked, T_meanimp = x_true_targets(true_env, B["mask_true"])
    targets = [("bench_mean", learn_env, "given", None), ("bench_terminate", learn_env, "terminate", None),
               ("true", T_masked, "given", lift)]
    if not aggregated:
        targets += [("true_rows_mean", T_meanimp, "given", lift), ("true_rows_terminate", T_masked, "terminate", lift)]
    del T_meanimp
    if family == "gumbel":   # is the true optimal action supported? (decision states of the construction)
        opt_act, Vt = _x_v3_gumbel_opt_action(spec["variant"], spec["timeout_hazard"]), B["decision_states_true"]
        B["report"]["true_optimum_in_mask_fraction"] = (float(B["mask_true"][Vt, opt_act[Vt]].mean()) if len(Vt)
                                                        else float("nan"))
    deploy = "true"
    construction = {k: v for k, v in B["report"].items() if k != "hospital_validation"}
    hosp_validation = B["report"]["hospital_validation"]
    source = f"x_sample_dataset({spec.get('construction', 'best')} benchmark, clinician estimate)"
    del B
    labels = [t[0] for t in targets]
    assert len(labels) == len(set(labels)) and deploy in labels, labels
    for lab, env, sem, lift_fn in targets:
        assert sem in X_SEMANTICS and (lift_fn is not None or env.S == learn_env.S), (lab, env.S, learn_env.S)
    user_validation = x_validate_dataset(data, learn_env)
    train, dev = x_split_episodes(data, XCFG.dev_fraction, np_rng("xenv", "split", stream, draw))
    meta = {"spec": {k: (list(v) if isinstance(v, tuple) else v) for k, v in spec.items()}, "user_data_source": source,
            "stream_cell": stream, "user_validation": user_validation, "hospital_validation": hosp_validation,
            "construction": construction, "train_transitions": int(len(train["state"])),
            "dev_transitions": int(len(dev["state"])), "build_s": time.perf_counter() - t0}
    return CellInstance(cell=cell, family=family, draw=draw, learn_env=learn_env, behavior=behavior, data=data,
                        train=train, dev=dev, targets=targets, deploy_target=deploy, meta=meta)


def x_v4_sens_identity(spec: dict, inst: CellInstance) -> dict:
    """Cache identity of one II.9e draw: the II.6 draw identity (instance, behaviour, dev split, XCFG, spec, driver
    constants) and this section's code hash."""
    return {"kind": "xenv_v4_sensitivity", "draw": x_v4_draw_identity(spec, inst), "code": XV4_SENS_CODE_HASH,
            "section": "II.9e"}


def run_v4_cell_draw(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
    """All results of one draw of one Section II.9e cell (the steps and checks of run_cell_draw; cached in v4sens/)."""
    cell, family = spec["cell"], spec["family"]
    assert family in ("gumbel", "garnet") and spec.get("v4_kind") in XV4_SENS_KINDS, (family, spec.get("v4_kind"))
    t0 = time.perf_counter()
    inst = build_v4_instance(spec, draw)
    assert (inst.cell, inst.family, int(inst.draw)) == (cell, family, int(draw)), (inst.cell, inst.family, inst.draw)
    identity = x_v4_sens_identity(spec, inst)
    key, ident_hash = "v4sens/" + x_cache_key(cell, draw), obj_hash(identity)[:20]
    strict = True
    build_s = time.perf_counter() - t0
    hit, meta_hit = XCACHE.get(key, identity), XCACHE.get(key + "/meta", identity)
    if hit is not None and meta_hit is not None:
        del inst
        assert list(hit.columns) == XRESULT_COLUMNS, f"{key}: cached columns differ from XRESULT_COLUMNS"
        meta = json.loads(str(meta_hit["meta_json"].iloc[0]))
        meta.update(cache="loaded", session_build_s=build_s, checks=x_check_draw_rows(hit, strict))
        gc.collect()
        return hit, meta
    if (hit is None) != (meta_hit is None):
        print(f"  {key}: incomplete cache entry (results or meta missing) - recomputing this draw")
    XCACHE.require_allowed(key)
    learn = inst.learn_env
    t1 = time.perf_counter()
    raw = run_xmethods(LearnerView.from_env(learn), inst.train, f"{cell}|draw{int(draw)}")
    records = x_collect_records(raw, inst.behavior, learn)
    del raw
    t2 = time.perf_counter()
    fqe = x_fqe_env(inst.dev, inst.train, learn, name=f"fqe|{cell}|draw{int(draw)}")
    t3 = time.perf_counter()
    rows, ev_stats = x_evaluate_targets(inst, records, fqe)
    df = pd.DataFrame(rows, columns=XRESULT_COLUMNS)
    t4 = time.perf_counter()
    checks = x_check_draw_rows(df, strict)
    t5 = time.perf_counter()
    nonconv = [r["label"] for r in records if r["desc"]["converged"] is False]
    lm = learn.mask[learn.live]
    meta = _x_compact({
        "cell": cell, "family": family, "v4_kind": spec["v4_kind"], "draw": int(draw),
        "stream_cell": str(spec.get("stream_cell", cell)), "deploy_target": inst.deploy_target,
        "identity_hash": ident_hash, "code_hash": XV4_SENS_CODE_HASH[:16], "cache": "computed",
        "learn_env": {"name": learn.name, "S": learn.S, "A": learn.A, "live": int(learn.live.sum()),
                      "admissible_pairs": int(lm.sum()), "mean_admissible_actions": float(lm.sum(axis=1).mean())},
        "data": {"train_transitions": int(len(inst.train["state"])),
                 "train_episodes": int(np.unique(inst.train["episode_id"]).size),
                 "dev_transitions": int(len(inst.dev["state"])),
                 "dev_episodes": int(np.unique(inst.dev["episode_id"]).size)},
        "targets": [t[0] for t in inst.targets],
        "records": {"total": len(records), **dict(Counter(r["variant"] for r in records)),
                    "nonconverged": len(nonconv), "nonconverged_labels": nonconv},
        "evaluation": ev_stats, "fqe_rows": fqe.meta.get("fqe_rows", {}), "checks": checks,
        "timing": {"build_s": build_s, "methods_s": t2 - t1, "fqe_build_s": t3 - t2, "eval_s": t4 - t3,
                   "check_s": t5 - t4, "total_s": t5 - t0},
        "instance": inst.meta})
    XCACHE.put(key + "/meta", identity, pd.DataFrame({"meta_json": [json.dumps(meta, default=_json_default)]}))
    XCACHE.put(key, identity, df)
    back = XCACHE.get(key, identity)
    assert back is not None and list(back.columns) == XRESULT_COLUMNS and len(back) == len(df), f"{key}: cache write"
    for c in _XMETRIC_SOURCE:
        assert np.allclose(back[c].to_numpy(dtype=float), df[c].to_numpy(dtype=float), rtol=1e-14, atol=0.0,
                           equal_nan=True), f"{key}: cache round trip changed column {c}"
    assert (back["status"].astype(str).to_numpy() == df["status"].to_numpy()).all(), f"{key}: cache round trip"
    del inst, fqe, records, rows, df
    gc.collect()
    return back, meta


# Objects whose source changes what an II.9e draw computes (the checks are re-run on cached tables instead).
XV4_SENS_CODE = [x_v4_sens_build_cells, _x_v4_gumbel_live_layout, _x_v4_factor_aggregation, x_v4_kmeans,
                 x_v4_gumbel_aggregation, _x_v4_aggregation, construct_aggregated_benchmark_v4, build_v4_instance,
                 x_v4_sens_identity, run_v4_cell_draw]
XV4_SENS_CODE_HASH = obj_hash({"common": XV4_COMMON_CODE_HASH, "code": code_hash(*XV4_SENS_CODE),
                               "constants": {"kinds": list(XV4_SENS_KINDS),
                                             "agg_factors": {k: list(v) for k, v in XV4_AGG_FACTORS.items()},
                                             "kmeans_features": list(XV4_KMEANS_FEATURES),
                                             "kmeans_max_iter": XV4_KMEANS_MAX_ITER, "columns": XRESULT_COLUMNS}})


def x_v4_sens_selftest(cells: Sequence[dict], refs: Dict[str, Optional[str]]) -> Dict[str, Any]:
    """Identity self-test of II.9e (see markdown); every comparison is asserted and reported."""
    t0 = time.perf_counter()
    out: Dict[str, Any] = {"instances": {}, "aggregation": {}}
    by_cell = {str(s["cell"]): s for s in X_CELLS}
    for s in cells:
        base = by_cell.get(str(refs.get(str(s["cell"]))))
        if base is None or base["family"] in {v["family"] for v in out["instances"].values()}:
            continue
        s4 = {**base, "stream_cell": str(base["cell"]), "empty_rule": str(XCFG.benchmark_empty_rule)}
        a, b = build_v4_instance(s4, 0), build_cell_instance(base, 0)
        same = {"instance_identity": obj_hash(a.identity()) == obj_hash(b.identity()),
                "behaviour": bool(np.array_equal(a.behavior, b.behavior)),
                "dev_split": array_dict_hash(a.dev) == array_dict_hash(b.dev)}
        assert all(same.values()), f"build_v4_instance differs from build_cell_instance on {base['cell']}: {same}"
        out["instances"][str(base["cell"])] = {"family": base["family"], "for_cell": str(s["cell"]), **same}
        del a, b
    agg_cells = [s for s in cells if s["v4_kind"] == "abstraction"]
    if agg_cells:
        s = agg_cells[0]
        env, beh = x_v4_true_system(s)
        obs = x_gumbel_aggregation(env, "obs")
        mine = _x_v4_factor_aggregation(env, "obs", ("hr", "sbp", "o2", "glucose", "abx", "vaso", "vent"))
        same_map = (all(np.array_equal(obs[k], mine[k]) for k in ("phi", "keys", "levels", "radix", "members"))
                    and all(obs[k] == mine[k] for k in ("n_live", "S", "cells_total")))
        assert same_map, "the II.9e factor aggregation does not reproduce II.8's 'obs' map"
        n_hosp, tau = int(s["n_hosp"]), int(s["tau"])
        B3 = construct_aggregated_benchmark(env, beh, "obs", n_hosp, tau, np_rng("xenv", "v4", "selftest", "agg-obs"),
                                           "v4-selftest|agg-obs")
        B4 = construct_aggregated_benchmark_v4(env, beh, obs, n_hosp, tau, np_rng("xenv", "v4", "selftest", "agg-obs"),
                                               "v4-selftest|agg-obs")
        pi = x_uniform_admissible(B3["bench"].mask, B3["bench"].live)
        same = {"factor_map": bool(same_map),
                "bench_identity": obj_hash(B3["bench"].identity()) == obj_hash(B4["bench"].identity()),
                "mask_true": bool(np.array_equal(B3["mask_true"], B4["mask_true"])),
                "decision_states_true": bool(np.array_equal(B3["decision_states_true"], B4["decision_states_true"])),
                "lift": bool(np.array_equal(B3["lift"](pi), B4["lift"](pi)))}
        assert all(same.values()), f"construct_aggregated_benchmark_v4 does not reproduce II.8c on 'obs': {same}"
        out["aggregation"]["obs_reproduces_ii8c"] = {"n_hosp": n_hosp, "tau": tau, **same}
        del B3, B4
        for kind in dict.fromkeys([*XV4CFG.aggregations, *XV4Config().aggregations]):   # full design, also in smoke
            g1, g2 = x_v4_gumbel_aggregation(env, kind), x_v4_gumbel_aggregation(env, kind)
            assert np.array_equal(g1["phi"], g2["phi"]) and np.array_equal(g1["levels"], g2["levels"]), kind
            out["aggregation"][str(kind)] = {"cells": int(g1["n_live"]), "cells_total": int(g1["cells_total"]),
                                             "members_min": int(g1["members"].min()),
                                             "members_max": int(g1["members"].max()),
                                             **{k: v for k, v in g1["report"].items()
                                                if k in ("iterations", "reseeds", "inertia")}}
    gc.collect()
    out.update(passes=True, seconds=time.perf_counter() - t0)
    return out


XV4_SENS_CELLS: List[dict] = []
XV4_SENS_REFS: Dict[str, Optional[str]] = {}
XV4_SENS_SKIPPED: List[str] = []
XV4_SENS_SELFTEST: Dict[str, Any] = {"status": "not run (xenv_v4_sensitivity off)"}
if EXPERIMENTS["xenv_v4_sensitivity"]:
    XV4_SENS_CELLS, XV4_SENS_REFS, XV4_SENS_SKIPPED = x_v4_sens_build_cells()
    print("II.9e cells:", [(c["cell"], c["v4_kind"], x_v4_draws(c)) for c in XV4_SENS_CELLS])
    if XV4_SENS_SKIPPED:
        print("NOTE: II.9e cells skipped (base cell not enabled, or a repeat in smoke mode):", XV4_SENS_SKIPPED)
    XV4_SENS_SELFTEST = x_v4_sens_selftest(XV4_SENS_CELLS, XV4_SENS_REFS)
    print(f"II.9e self-test passed in {XV4_SENS_SELFTEST['seconds']:.1f} s:",
          json.dumps({k: v for k, v in XV4_SENS_SELFTEST.items() if k not in ("passes", "seconds")}, indent=1,
                     default=_json_default))
print("II.9e code hash:", XV4_SENS_CODE_HASH[:16])

#### II.9e Run and paired analyses

Every draw of every II.9e cell runs through `run_v4_cell_draw` (`x_v4_run_draws`, with the progress and ETA of
II.6a; at most `XV4CFG.max_draws` draws per cell). Each new cell is then analysed beside its **reference cell**: the
base II.6 cell from `XRESULTS` for the timeout, dead-rule and support cells, and the II.8 `agg-obs` cell from
`XRESULTS_V3`, when present, for the abstraction cells. The reference rows are restricted to the same draw indices.
`XRESULTS` and `XRESULTS_V3` are used when they are in memory; otherwise `xenv_results_all.csv.gz` and
`v3_results_all.csv.gz` (and the draw-meta files) are read from `XENV_TABLE_DIR`. Only the default policies enter
the reported statistics, as in II.8g; the theorem counts below use every policy of the new cells.

* **Bracket coverage** among the policy-draws with p > 0, where p = P(any unsupported decision) on `bench_mean`
  > 1e-12, computed with the functions of II.8i. The semantic bracket (`sem_*`) is J(true rows + terminate) ≤
  J_true ≤ J(true rows + mean rule), whose lower end is a theorem (so `sem_*` coverage is the upper end's); the
  benchmark bracket (`bench_*`) is J_true ∈ [J_bench_term, J_bench_mean]. Abstraction cells have no true-rows
  targets, so they have only `bench_*`. II.8i bracketed every policy, so its rates are not directly comparable
  with these; the base rows here are recomputed on the default policies, so new and base are.
* **Certificates** per draw (`x_v4_pair_claims`, v_range = v_max − v_min of the family), on two kinds of rows:
  * the true supported rows (`cc_true_rows_*`): J_term = `true_rows_terminate`, J_W = `true_rows_mean`, p from
    `true_rows_mean`;
  * the benchmark (`cc_bench_*`).

  The kinds are the NHB certificate J_term(i) > J_W(j), the NHB-free certificate J_term(i) > J_term(j) + p_j
  v_range, the NHB-only claims and the single-rule point claims. Each kind reports its share of the pairs and of the
  completion-sensitive pairs (types AU + UU), its errors against J_true (ties are errors), and the draws with at
  least one wrong certificate.
* **Rank fidelity**: the per-draw Kendall τ_b of `bench_mean` and of `bench_terminate` against `true`
  (`xs_rank_fidelity_draws` of II.7), and τ(mean) − τ(terminate) with a draw-bootstrap interval. The change new −
  reference also gets a bootstrap interval (`XCFG.bootstrap` resamples). It is paired by draw for the dead-rule
  cells, which share the hospital data; otherwise the draws are resampled independently within each cell.
* **Construction statistics** (`constr_*`: means over draws of the construction report):
  * decision states, coverage of the true live states, admissible pairs;
  * mean admissible actions and the single-admissible fraction (the support of the softer physicians);
  * dead states removed and empty-row fallbacks (the dead-state rule);
  * censored d0 mass, and the share of decision states whose true optimal action is supported;
  * for the abstractions: cells, live cells, decision cells and the d0 mass on decision cells.

`v4_sens_summary.csv` has three rows per new cell: `side` = `new`, `base` (the reference cell) and `new - base`
(the numeric differences and the τ intervals); a cell whose reference is missing has the `new` row only. It has
the same rows per pool of at least two cells of one kind and family (`pooled:<kind>|<family>`, no τ intervals).

**Invariants asserted.**
* Per draw, the II.6 checks (`x_check_draw_rows`, strict projection identity), on computed and on cached tables.
* Across draws, `x_check_results`.
* On the new cells, the bracketing theorem counts of II.8i: no lower-bound violation on the true rows, no negative
  width, zero width without unsupported decisions, width ≤ p·v_range, and values in [v_min, v_max].

Coverage, certificate errors and rank fidelity are reported, never asserted.

| Output | Content |
|---|---|
| `XRESULTS_V4SENS` · `xenv/v4_sens_results_all.csv.gz` | one row per draw × policy × target (`XRESULT_COLUMNS`) |
| `XDRAW_META_V4SENS` · `xenv/v4_sens_draw_meta.csv` | per draw: construction and validation reports, timings |
| `XV4_SENS_TABLE` · `xenv/v4_sens_summary.csv` | per new cell and pool: new, reference and difference rows |
| `XV4_SENS_CLAIMS` · `xenv/v4_sens_certified_draws.csv` | per cell × draw × targets × pair type: claims, errors |
| `xenv/v4_sens_rank_fidelity.csv`, `XV4_SENS_RANK` · `xenv/v4_sens_rank_fidelity_draws.csv` | rank fidelity |
| `XV4_SENS_SUMMARY` · `xenv/v4_sens_summary.json` | status, code hash, cells, self-test, checks, deltas, timing |

**Expected cost on Colab (CPU-bound).** These are estimates, not measurements. On Colab, II.6 Gumbel `best` draws
took 10–16 s and Garnet draws about 4 s (`xenv_draw_meta.csv`); II.8 aggregated draws were estimated at 10–12 s.
The full design has 17 cells (4 timeout, 8 dead-rule of which 2 Garnet, 2 support, 3 abstraction) × 30 draws:
* 360 Gumbel `best` draws, about 80 min; the softer physicians may take somewhat longer;
* 60 Garnet draws, about 5 min;
* 90 abstraction draws, about 17 min;
* about a minute per new timeout hazard for the exact enumeration and its physician;
* under a minute for the self-test and a few minutes for the analyses (the reference rows are read from disk
  when they are not in memory).

That is about **1.5–2 h** in all. The lite profile (10 draws per cell) takes about a third of that. A cached draw
reloads in about 3 s, so a re-run costs about 25 min plus the analyses.

In [ ]:
# ============================================================
# II.9e Sensitivity cells: run and paired analyses
# ============================================================
EXECLOG.mark("II.9e Sensitivity cells: run and analyses")

XV4_SENS_CLAIM_KINDS = ("cert_nhb", "cert_free", "cert_nhb_only", "point_mean", "point_term")
XV4_SENS_CONSTRUCTION_KEYS = ("decision_states", "coverage_true_live", "admissible_pairs", "mean_admissible_actions",
                              "single_admissible_fraction", "dead_states_removed", "empty_row_fallbacks",
                              "censored_d0_mass", "true_d0_mass_on_decision_states", "true_optimum_in_mask_fraction",
                              "inadmissible_logged_fraction", "aggregated_cells_total", "aggregated_live_cells",
                              "full_live_states_in_decision_cells", "full_d0_mass_in_decision_cells",
                              "phi_inertia")
XV4_SENS_HEADLINE = ("bench_coverage_violating", "sem_upper_coverage_violating", "cc_bench_cert_nhb_share",
                     "cc_bench_cert_nhb_error_rate", "cc_bench_cert_free_share", "cc_bench_cert_free_error_rate",
                     "cc_true_rows_cert_nhb_share", "cc_true_rows_cert_nhb_error_rate", "tau_mean_delta",
                     "tau_terminate_delta", "tau_diff_delta", "constr_mean_admissible_actions",
                     "constr_single_admissible_fraction")


def _xv4_sens_table(global_name: str, file_name: str, results: bool) -> Tuple[Optional[pd.DataFrame], str]:
    """A Part II table from memory (global) when non-empty, else from XENV_TABLE_DIR; (frame or None, source)."""
    df = globals().get(global_name)
    if df is not None and len(df):
        return df, global_name
    p = XENV_TABLE_DIR / file_name
    if not p.exists():
        return None, "missing"
    if results:
        return pd.read_csv(p, float_precision="round_trip",
                           dtype={"deploy_target": str, "target": str, "target_env": str}), str(p)
    return pd.read_csv(p), str(p)


def xv4_sens_reference(draws_of: Dict[str, int]) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, str]]:
    """Result rows and draw meta of the reference cells (draws 0..draws_of[cell] - 1): II.6 (XRESULTS / XDRAW_META)
    and II.8 (XRESULTS_V3 / XDRAW_META_V3), from memory or from their files, and the source of each cell."""
    res, meta, source = [], [], {}
    for rg, mg, rf, mf in (("XRESULTS", "XDRAW_META", "xenv_results_all.csv.gz", "xenv_draw_meta.csv"),
                           ("XRESULTS_V3", "XDRAW_META_V3", "v3_results_all.csv.gz", "v3_draw_meta.csv")):
        if not draws_of:
            break
        r, src = _xv4_sens_table(rg, rf, True)
        if r is None:
            continue
        r = r[r["cell"].astype(str).isin(list(draws_of))]
        r = r[pd.to_numeric(r["draw"]).astype(np.int64) < r["cell"].astype(str).map(draws_of).astype(np.int64)]
        if not len(r):
            continue
        res.append(r)
        source.update({str(c): src for c in r["cell"].unique()})
        m, _ = _xv4_sens_table(mg, mf, False)
        if m is not None and len(m) and {"cell", "draw"} <= set(m.columns):
            m = m[m["cell"].astype(str).isin(list(draws_of))]
            meta.append(m[pd.to_numeric(m["draw"]).astype(np.int64)
                          < m["cell"].astype(str).map(draws_of).astype(np.int64)])
    return (pd.concat(res, ignore_index=True) if res else pd.DataFrame(columns=XRESULT_COLUMNS),
            pd.concat(meta, ignore_index=True) if meta else pd.DataFrame(), source)


def xv4_sens_claims_draws(Wd: pd.DataFrame) -> pd.DataFrame:
    """Per (cell, draw, targets, pair type): certificates and point claims of the default policies of one draw
    (x_v4_pair_claims, v_range = v_max - v_min of the family), scored against J_true. 'true_rows': J_term / J_W on
    the true supported rows (p from true_rows_mean); 'bench': the estimated benchmark (p from bench_mean). A draw
    with an improper evaluation on the targets is left out (counted by the caller)."""
    rows = []
    for (c, dr), d in Wd.groupby(["cell", "draw"], sort=False):
        fam = str(d["family"].iloc[0])
        v_range = float(XV3_VRANGE[fam][1] - XV3_VRANGE[fam][0])
        for tf, (lc, uc, pc) in (("true_rows", ("true_rows_terminate", "true_rows_mean", "p_true")),
                                 ("bench", ("bench_terminate", "bench_mean", "p_bench"))):
            if d[[lc, uc, pc, "true"]].isna().any().any():
                continue
            for r in x_v4_pair_claims(d[lc].to_numpy(dtype=float), d[uc].to_numpy(dtype=float),
                                      d[pc].to_numpy(dtype=float), v_range, d["true"].to_numpy(dtype=float)):
                rows.append({"cell": c, "draw": int(dr), "family": fam, "kind": str(d["v3_kind"].iloc[0]),
                             "targets": tf, **r})
    return pd.DataFrame(rows)


def xv4_sens_claims_summary(C: pd.DataFrame) -> Dict[str, Any]:
    """Claims, shares (of all pairs and of completion-sensitive AU + UU pairs), errors, error rates and draws with a
    wrong certificate, per targets, pooled over the rows of C (xv4_sens_claims_draws)."""
    out: Dict[str, Any] = {}
    for tf in ("true_rows", "bench"):
        d = C[C["targets"] == tf] if len(C) else C
        if d.empty:
            continue
        a, cs = d[d["type"] == "all"], d[d["type"].isin(["AU", "UU"])]
        pairs, pairs_cs = int(a["pairs"].sum()), int(cs["pairs"].sum())
        p = f"cc_{tf}_"
        out.update({p + "draws": int(len(a)), p + "pairs": pairs, p + "pairs_completion_sensitive": pairs_cs})
        for k in XV4_SENS_CLAIM_KINDS:
            n, e = int(a[f"{k}_claims"].sum()), int(a[f"{k}_errors"].sum())
            out.update({p + f"{k}_claims": n, p + f"{k}_share": _xv3_rate(n, pairs), p + f"{k}_errors": e,
                        p + f"{k}_error_rate": _xv3_rate(e, n),
                        p + f"{k}_share_completion_sensitive": _xv3_rate(int(cs[f"{k}_claims"].sum()), pairs_cs)})
            if k.startswith("cert_"):
                out[p + f"draws_any_wrong_{k}"] = int((a[f"{k}_errors"] > 0).sum())
    return out


def xv4_sens_rank_summary(cells: Sequence[str], rank_tab: pd.DataFrame, rank_draws: pd.DataFrame) -> Dict[str, Any]:
    """Kendall tau-b of the ranking by bench_mean / bench_terminate against true, and their difference: the II.8i
    table (mean, bootstrap interval, n) for one cell; the mean over the draws of all cells for a pool."""
    labels = (("mean", "tau_mean"), ("terminate", "tau_terminate"), ("mean - terminate", "tau_diff"))
    out: Dict[str, Any] = {}
    if len(cells) == 1:
        t = rank_tab[rank_tab["cell"] == cells[0]].set_index("completion") if len(rank_tab) else rank_tab
        for comp, lab in labels:
            if len(t) and comp in t.index:
                for sfx in ("mean", "ci_low", "ci_high", "n"):
                    out[f"{lab}_{sfx}"] = float(t.loc[comp, f"kendall_tau_b_{sfx}"])
        return out
    d = rank_draws[rank_draws["cell"].isin(list(cells)) & rank_draws["completion"].isin(["mean", "terminate"])] \
        if len(rank_draws) else rank_draws
    if d.empty:
        return out
    t = d.pivot_table(index=["cell", "draw"], columns="completion", values="kendall_tau_b", aggfunc="first")
    if {"mean", "terminate"} <= set(t.columns):
        t["mean - terminate"] = t["mean"] - t["terminate"]
    for comp, lab in labels:
        if comp in t.columns:
            v = t[comp].dropna()
            out[f"{lab}_mean"], out[f"{lab}_n"] = (float(v.mean()) if len(v) else np.nan), float(len(v))
    return out


def _xv4_mean_gap(parts: List[np.ndarray]) -> np.ndarray:
    """Bootstrap statistic: mean of the first stratum minus mean of the second (independent draws)."""
    return parts[0][..., 0].mean(axis=1) - parts[1][..., 0].mean(axis=1)


def xv4_sens_tau_delta(rank_draws: pd.DataFrame, cell: str, ref: str, paired: bool, n_boot: int) -> Dict[str, Any]:
    """Kendall tau-b of `cell` minus that of its reference cell, per completion and for tau(mean) - tau(terminate),
    with a percentile bootstrap interval over draws: paired by draw when both cells share the hospital draws,
    otherwise draws resampled independently within each cell."""
    out: Dict[str, Any] = {"delta_paired": bool(paired)}
    d = rank_draws[rank_draws["cell"].isin([cell, ref]) & rank_draws["completion"].isin(["mean", "terminate"])] \
        if len(rank_draws) else rank_draws
    if d.empty:
        return out
    t = d.pivot_table(index=["cell", "draw"], columns="completion", values="kendall_tau_b", aggfunc="first")
    if not {cell, ref} <= set(t.index.get_level_values("cell")):
        return out
    if {"mean", "terminate"} <= set(t.columns):
        t["diff"] = t["mean"] - t["terminate"]
    for col, lab in (("mean", "tau_mean"), ("terminate", "tau_terminate"), ("diff", "tau_diff")):
        if col not in t.columns:
            continue
        a, b = t.xs(cell, level="cell")[col].dropna(), t.xs(ref, level="cell")[col].dropna()
        if paired:
            j = a.index.intersection(b.index)
            v = (a[j] - b[j]).to_numpy(dtype=np.float64)
            point, reps = xs_stratified_bootstrap({cell: v[:, None]} if len(v) else {}, xs_stat_mean(),
                                                  ("v4sens", cell, lab), n_boot)
            n = len(v)
        elif len(a) and len(b):
            point, reps = xs_stratified_bootstrap({"new": a.to_numpy(dtype=np.float64)[:, None],
                                                   "base": b.to_numpy(dtype=np.float64)[:, None]},
                                                  _xv4_mean_gap, ("v4sens", cell, lab), n_boot)
            n = min(len(a), len(b))
        else:
            continue
        lo, hi = xs_ci(reps)
        out.update({f"{lab}_delta": float(point), f"{lab}_delta_ci_low": lo, f"{lab}_delta_ci_high": hi,
                    f"{lab}_delta_n": int(n)})
    return out


def xv4_sens_construction(cells: Sequence[str], meta: pd.DataFrame) -> Dict[str, Any]:
    """Means over draws of the construction report (flattened draw meta 'instance.construction.<key>')."""
    if meta is None or not len(meta) or "cell" not in meta.columns:
        return {}
    m = meta[meta["cell"].astype(str).isin(list(cells))]
    out: Dict[str, Any] = {}
    for k in XV4_SENS_CONSTRUCTION_KEYS:
        col = f"instance.construction.{k}"
        if col in m.columns:
            v = pd.to_numeric(m[col], errors="coerce").dropna()
            if len(v):
                out[f"constr_{k}"] = float(v.mean())
    return out


def xv4_sens_unit(cells: Sequence[str], Wd: pd.DataFrame, C: pd.DataFrame, rank_tab: pd.DataFrame,
                  rank_draws: pd.DataFrame, meta: pd.DataFrame) -> Dict[str, Any]:
    """All analyses of one unit (one cell or a pool of cells): bracketing, certificates, rank fidelity, construction."""
    d = Wd[Wd["cell"].isin(list(cells))]
    out: Dict[str, Any] = {"n_cells": len(cells), "n_draws": int(d[["cell", "draw"]].drop_duplicates().shape[0]),
                           "default_policies_per_draw": (float(d.groupby(["cell", "draw"]).size().median())
                                                         if len(d) else np.nan)}
    out.update({f"sem_{k}": v for k, v in xv3_bracket_semantics(d).items()})
    out.update({f"bench_{k}": v for k, v in xv3_bracket_benchmark(d).items()})
    out.update(xv4_sens_claims_summary(C[C["cell"].isin(list(cells))] if len(C) else C))
    out.update(xv4_sens_rank_summary(list(cells), rank_tab, rank_draws))
    out.update(xv4_sens_construction(list(cells), meta))
    return out


def _xv4_is_num(x: Any) -> bool:
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def xv4_sens_analyses(cells: Sequence[dict], refs: Dict[str, Optional[str]], res_new: pd.DataFrame,
                      meta_new: pd.DataFrame, n_boot: int
                      ) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame], Dict[str, Any]]:
    """Section II.9e analyses: (summary table, per-draw frames, checks) for the new cells and their references."""
    names = [str(s["cell"]) for s in cells]
    draws_of: Dict[str, int] = {}
    for s in cells:
        r = refs.get(str(s["cell"]))
        if r:
            draws_of[r] = max(draws_of.get(r, 0), x_v4_draws(s))
    ref_res, ref_meta, source = xv4_sens_reference(draws_of)
    present = [c for c in draws_of if c in set(ref_res["cell"].astype(str))]
    kind_of = {**{str(s["cell"]): str(s["v4_kind"]) for s in cells}, **{c: "reference" for c in present}}
    _, frame_checks = xv3_bracket_frame(res_new, kind_of)
    all_res = pd.concat([res_new, ref_res], ignore_index=True) if len(ref_res) else res_new
    W, _ = xv3_bracket_frame(all_res, kind_of)
    Wd = W[W["is_default"]]
    C = xv4_sens_claims_draws(Wd)
    rank_draws, rank_tab = xv3_rank_fidelity(all_res, names + present, n_boot)
    meta_all = pd.concat([meta_new, ref_meta], ignore_index=True) if len(ref_meta) else meta_new
    del all_res, W
    rows: List[dict] = []

    def emit(head: dict, new_cells: List[str], ref_cells: List[str], tau: Optional[dict]) -> None:
        new = xv4_sens_unit(new_cells, Wd, C, rank_tab, rank_draws, meta_all)
        rows.append({**head, "side": "new", "unit": ";".join(new_cells), **new})
        if not ref_cells:
            return
        base = xv4_sens_unit(ref_cells, Wd, C, rank_tab, rank_draws, meta_all)
        rows.append({**head, "side": "base", "unit": ";".join(ref_cells), **base})
        delta = {k: new[k] - base[k] for k in new if k in base and k not in ("n_cells", "n_draws")
                 and not k.endswith(("_ci_low", "_ci_high")) and _xv4_is_num(new[k]) and _xv4_is_num(base[k])}
        rows.append({**head, "side": "new - base", "unit": f"{';'.join(new_cells)} - {';'.join(ref_cells)}", **delta,
                     **(tau or {})})

    for s in cells:
        c, r = str(s["cell"]), refs.get(str(s["cell"]))
        paired = bool(r) and str(s.get("stream_cell", c)) == r
        head = {"row_type": "cell", "cell": c, "v4_kind": s["v4_kind"], "family": s["family"], "reference": r or "",
                "reference_source": source.get(str(r), "missing") if r else "none", "paired": paired}
        ok = r in present
        emit(head, [c], [r] if ok else [], xv4_sens_tau_delta(rank_draws, c, r, paired, n_boot) if ok else None)
    pools: Dict[Tuple[str, str], List[str]] = {}
    for s in cells:
        pools.setdefault((str(s["v4_kind"]), str(s["family"])), []).append(str(s["cell"]))
    for (kind, fam), cs in pools.items():
        if len(cs) < 2:
            continue
        rc = sorted({str(refs[c]) for c in cs if refs.get(c) in present})
        head = {"row_type": "pool", "cell": f"pooled:{kind}|{fam}", "v4_kind": kind, "family": fam,
                "reference": ";".join(rc), "reference_source": ";".join(sorted({source[c] for c in rc})),
                "paired": all(spec.get("stream_cell") == refs.get(str(spec["cell"])) for spec in cells
                              if str(spec["cell"]) in cs)}
        emit(head, cs, rc, None)
    table = pd.DataFrame(rows)
    nd = Wd[Wd["cell"].isin(names)][["cell", "draw"]].drop_duplicates()
    have = (C[C["cell"].isin(names) & (C["type"] == "all")].groupby("targets").size() if len(C)   # one row per draw
            else pd.Series(dtype=int))
    tc = {k: int(frame_checks.get(k, 0)) for k in ("values_outside_value_range", "lower_bound_violations_true_rows",
                                                   "negative_width_true_rows", "negative_width_bench",
                                                   "nonzero_width_admissible", "width_above_bound")}
    checks = {"bracket_frame": frame_checks, "theorem_counts": tc, "theorem_passes": all(v == 0 for v in tc.values()),
              "reference_cells": present, "reference_cells_missing": [c for c in draws_of if c not in present],
              "reference_source": source, "new_cell_draws": int(len(nd)),
              "certificate_draws_by_targets": {str(k): int(v) for k, v in have.items()}}
    return table, {"claims_draws": C, "rank_draws": rank_draws, "rank_table": rank_tab}, checks


XRESULTS_V4SENS = pd.DataFrame(columns=XRESULT_COLUMNS)
XDRAW_META_V4SENS = pd.DataFrame()
XV4_SENS_TABLE = pd.DataFrame()
XV4_SENS_CLAIMS = pd.DataFrame()
XV4_SENS_RANK = pd.DataFrame()
XV4_SENS_SUMMARY: Dict[str, Any] = {"status": "not run (xenv_v4_sensitivity off)", "code_hash": XV4_SENS_CODE_HASH,
                                    "checks": {}, "elapsed_s": 0.0, "draws_complete": None}
if EXPERIMENTS["xenv_v4_sensitivity"] and not XV4_SENS_CELLS:
    XV4_SENS_SUMMARY = {"status": "not run: no configured II.9e cell is enabled", "code_hash": XV4_SENS_CODE_HASH,
                        "cells_skipped": XV4_SENS_SKIPPED, "checks": {"selftest": XV4_SENS_SELFTEST}, "elapsed_s": 0.0,
                        "draws_complete": None}
elif EXPERIMENTS["xenv_v4_sensitivity"]:
    _t0 = time.perf_counter()
    _frames, XDRAW_META_V4SENS, _run = x_v4_run_draws(XV4_SENS_CELLS, run_v4_cell_draw, "II.9e",
                                                      time_key=lambda s: f"{s['family']}|{s['v4_kind']}")
    XRESULTS_V4SENS = pd.concat(_frames, ignore_index=True) if _frames else pd.DataFrame(columns=XRESULT_COLUMNS)
    del _frames
    gc.collect()
    assert list(XRESULTS_V4SENS.columns) == XRESULT_COLUMNS
    _cross = x_check_results(XRESULTS_V4SENS)
    XRESULTS_V4SENS.to_csv(XENV_TABLE_DIR / "v4_sens_results_all.csv.gz", index=False, compression="gzip")
    XDRAW_META_V4SENS.to_csv(XENV_TABLE_DIR / "v4_sens_draw_meta.csv", index=False)
    XV4_SENS_TABLE, _sframes, _schecks = xv4_sens_analyses(XV4_SENS_CELLS, XV4_SENS_REFS, XRESULTS_V4SENS,
                                                           XDRAW_META_V4SENS, int(XCFG.bootstrap))
    XV4_SENS_CLAIMS, XV4_SENS_RANK = _sframes["claims_draws"], _sframes["rank_draws"]
    _outputs = {"v4_sens_summary.csv": XV4_SENS_TABLE, "v4_sens_certified_draws.csv": XV4_SENS_CLAIMS,
                "v4_sens_rank_fidelity.csv": _sframes["rank_table"], "v4_sens_rank_fidelity_draws.csv": XV4_SENS_RANK}
    for _name, _df in _outputs.items():
        _df.to_csv(XENV_TABLE_DIR / _name, index=False)
    _have = XRESULTS_V4SENS.groupby("cell")["draw"].nunique().to_dict() if len(XRESULTS_V4SENS) else {}
    _per = {str(s["cell"]): {"v4_kind": s["v4_kind"], "draws": int(_have.get(s["cell"], 0)),
                             "configured_draws": x_v4_draws(s), "reference": XV4_SENS_REFS.get(str(s["cell"]))}
            for s in XV4_SENS_CELLS}
    _dm = {}
    for _c in ("checks.max_slack_vs_optimum", "checks.max_slack_vs_admissible_optimum",
               "checks.projection_identity_worst"):
        _dm[f"max_{_c.split('.', 1)[1]}"] = (float(XDRAW_META_V4SENS[_c].max()) if _c in XDRAW_META_V4SENS else None)
    _dm["nonconverged_records"] = (int(XDRAW_META_V4SENS["records.nonconverged"].sum())
                                   if "records.nonconverged" in XDRAW_META_V4SENS else 0)
    _dl = XV4_SENS_TABLE[(XV4_SENS_TABLE["side"] == "new - base") & (XV4_SENS_TABLE["row_type"] == "cell")] \
        if len(XV4_SENS_TABLE) else XV4_SENS_TABLE
    XV4_SENS_SUMMARY = {
        "status": "run", "code_hash": XV4_SENS_CODE_HASH, "cells": [s["cell"] for s in XV4_SENS_CELLS],
        "cells_skipped": XV4_SENS_SKIPPED, "draws_per_cell": _per,
        "draws_complete": bool(_per) and all(v["draws"] == v["configured_draws"] for v in _per.values()),
        "driver": {k: v for k, v in _run.items() if k != "draws_per_cell"},
        "checks": {"selftest": XV4_SENS_SELFTEST, "cross_draw": _cross, "draw_checks": _dm, **_schecks,
                   "passes": bool(XV4_SENS_SELFTEST.get("passes")) and bool(_schecks["theorem_passes"])},
        "deltas": {str(r["cell"]): {k: float(r[k]) for k in XV4_SENS_HEADLINE if k in r and pd.notna(r[k])}
                   for r in _dl.to_dict("records")},
        "outputs": ["xenv/v4_sens_results_all.csv.gz", "xenv/v4_sens_draw_meta.csv",
                    *[f"xenv/{k}" for k in _outputs], "xenv/v4_sens_summary.json"],
        "elapsed_s": time.perf_counter() - _t0}
    if len(XV4_SENS_TABLE):
        _cols = [c for c in ("cell", "side", "n_draws", "sem_upper_coverage_violating", "bench_coverage_violating",
                             "cc_bench_cert_nhb_share", "cc_bench_cert_nhb_error_rate", "cc_bench_cert_free_share",
                             "cc_bench_cert_free_error_rate", "tau_mean_mean", "tau_terminate_mean", "tau_diff_mean",
                             "constr_mean_admissible_actions", "constr_single_admissible_fraction")
                 if c in XV4_SENS_TABLE.columns]
        display(XV4_SENS_TABLE[_cols].round(4))
        del _cols
    del _t0, _run, _cross, _sframes, _schecks, _outputs, _name, _df, _have, _per, _dm, _c, _dl
    gc.collect()
save_json(XV4_SENS_SUMMARY, XENV_TABLE_DIR / "v4_sens_summary.json")
print(json.dumps({k: v for k, v in XV4_SENS_SUMMARY.items() if k not in ("draws_per_cell", "checks", "deltas")},
                 indent=1, default=_json_default))
print("II.9e complete:", len(XRESULTS_V4SENS), "result rows,", len(XDRAW_META_V4SENS), "draws.")
if XV4_SENS_SUMMARY["status"] == "run" and not XV4_SENS_SUMMARY["checks"]["passes"]:
    raise AssertionError(f"II.9e theorem checks failed: {XV4_SENS_SUMMARY['checks']['theorem_counts']} "
                         "(xenv/v4_sens_summary.json)")

### II.9f Lift sensitivity: other lifts of the benchmark policies, and the range of J_true over all lifts

A benchmark policy is defined on the benchmark's decision states V only. To score it on the true system (targets
`true`, `true_rows_mean`, `true_rows_terminate`), Section II.3d **lifts** it to the true state space: the policy's own
rows on V, and the hospital's most frequent action (`fallback_action`) on every other true live state. That rule is
a convention of the testbed, not part of the benchmark, and the results may depend on it.
This section re-scores the default policies of every lift cell (`XV4CFG.lift_cells`: the six II.6 Gumbel `best`
cells and the eight Garnet cells) under other lifts, and computes the range of J_true over **all** lifts.

**Lifts** (`XV4CFG.lift_kinds`). On V every lift copies the benchmark policy; outside V (the true live states that
the construction censored or that the hospital never logged) the rows are:

| Lift | Rows outside V |
|---|---|
| `fallback` | the II.3d lift `B["lift"]` (the hospital's most frequent action); asserted equal to the instance's `true` lift on the first default policy |
| `physician` | the true behaviour policy that generated the hospital data |
| `uniform` | uniform over all actions |
| `worst` / `best` | the lift that minimises / maximises J_true (`XV4CFG.lift_bounds`; target `true` only) |

**Range over all lifts.** With the policy fixed on V, the lifts are exactly the policies of a variant of the
`true` target in which every action at a state of V has the policy-mixed row Σ_a π(a|s) P(·|s, a) and every action
is admissible (semantics `given`). The largest J_true over all lifts is the optimum of that variant, J_hi
(`x_optimal(variant, "given")`; a deterministic stationary lift attains it), and the smallest is J_lo = −(optimum of
the same variant with negated entry rewards). The negated variant is a valid `FiniteEnv`: rewards stay on absorbing
states, the sink and goal labels are kept (they matter only for the terminate rule, which is not used), and the
evaluator's value range follows the negated rewards. Value iteration needs every policy to absorb: the true Gumbel
and Garnet systems have no trap (asserted in II.3), and a policy-mixed row has the union support of its actions'
rows, so the variant has no trap either (asserted per policy). The greedy lift of each end is evaluated exactly on
`true` itself and must reproduce the solver's exact value (1e-9). A lift copies the benchmark policy on V, so J
under any lift depends only on the policy's rows on V; evaluations and range solves are memoised by those rows.

**Per (cell, draw).**
1. Rebuild the instance (`x_v4_instance`), regenerate the default policies (`x_v4_default_records`), and rebuild the
   hospital episodes and the benchmark (`x_v4_rebuild_benchmark`, benchmark identity asserted). V =
   `B["decision_states_true"]`, the first |V| benchmark states (asserted).
2. Assert that the rebuilt `B["lift"]` equals the instance's `true` lift on the first default policy.
3. Evaluate every default policy exactly on `bench_mean` and `bench_terminate` (lift `benchmark`), and under each
   lift on the three true-type targets of the instance (their environments and semantics).
4. With `lift_bounds`: J_lo and J_hi on `true` (above), stored as the lifts `worst` and `best`.

**Invariants asserted per draw** (re-run on cached tables): every evaluation is proper (the benchmark and the true
system have no trap); on the benchmark the two rules share p = P(any unsupported decision) and
0 ≤ J_mean − J_term ≤ p·(v_max − v_min); under every lift the three true-type targets share p, J_true ≥
J_true_rows_terminate (Theorem 1(i)) and 0 ≤ J_true_rows_mean − J_true_rows_terminate ≤ p·(v_max − v_min)
(Theorem 1(iii)); J_worst ≤ J_true(lift) ≤ J_best for every lift (tolerance 1e-8, the DP / exact-value gap that
`x_optimal` allows). Coverage and certification rates under a lift are outcomes and are only reported.

**Analyses** (default policies, tolerance 1e-9, ties count as errors), per cell, family pool and all cells, for
each lift:
* semantic bracket coverage J_true ≤ J_true_rows_mean among policies with p > 0 on the true rows (the lower end is
  a theorem; its violations are counted), and benchmark bracket coverage J_true ∈ [J_bench_terminate,
  J_bench_mean] among policies with p > 0 on the benchmark and overall;
* certificates (`x_v4_pair_claims`) on the true-rows targets and on the benchmark, scored against J_true under the
  lift: NHB and NHB-free claims and errors, overall and on completion-sensitive pairs (AU + UU), single-rule point
  claims, draws with any wrong certificate. Benchmark claims do not change with the lift; their errors can;
* rank fidelity: per-draw Kendall τ-b and top-1 agreement of `bench_mean` and `bench_terminate` against J_true
  under the lift (`xs_rank_agreement`), and τ(mean) − τ(terminate) with a draw-stratified bootstrap interval;
* the change of J_true against the `fallback` lift.

One more row per unit (lift `range`) summarises the bounds: the width J_hi − J_lo, the gaps between the fallback
lift and each end, the share of policies whose benchmark interval contains the whole range, and the benchmark NHB /
NHB-free certified claims that hold under **every** lift (J_lo(i) > J_hi(j)) or fail under every lift
(J_hi(i) ≤ J_lo(j), ties count as failures); the rest are lift-dependent. Both conditions are applied with a margin
of twice the range tolerance (2·10⁻⁸) beyond the tie tolerance, so the solver slack of the two ends cannot move a
claim into the wrong class. Each policy's range is taken over its own lifts, so, up to that margin, the two
conditions are exact when the two policies may be lifted independently; for one lift rule shared by both
policies, "holds" and "fails" stay sufficient conditions and some lift-dependent claims may in fact be robust.

Results are cached in `XCACHE` under `v4lift/<cell>/draw<k>` (and `/meta`). The identity is the regenerated draw
identity (`x_v4_draw_identity`), the code hash of this section (which folds in `XV4_COMMON_CODE_HASH`), the lift
kinds and `lift_bounds`. As a check that the regenerated policies are the II.6 ones, the `fallback` and benchmark
rows are compared with `XRESULTS` / `XRESULTS_V3` (`x_boot_point_check`; reported, not asserted).

| Output | Content |
|---|---|
| `xenv/v4_lift_results.csv.gz` | one row per draw × default policy × lift × target: lifts `fallback` / `physician` / `uniform` on `true`, `true_rows_mean`, `true_rows_terminate`; `worst` / `best` on `true`; `benchmark` on `bench_mean` / `bench_terminate` |
| `xenv/v4_lift_summary.csv` | per cell, family pool and all cells × lift: coverage, certificates, rank fidelity, change against `fallback`; `range` rows: width of the range and benchmark claims that hold or fail under every lift |
| `xenv/v4_lift_summary.json` | status, code hash, cells run and skipped, cache use, invariant maxima, the comparison with II.6, headline numbers |

**Expected cost on Colab** (estimates, scaled from local timings of the same array sizes). A Gumbel draw costs an
instance rebuild and one `run_xmethods` fit (6–12 s, as in II.8g), about 400 exact evaluations on the 606-state
true system (about 10 s), and, with `lift_bounds`, two value-iteration solves per distinct default policy (about 40)
on the 609 × 8 × 609 variant, each about 500 sweeps at 2–3 ms: 1.5–2.5 min per draw, so 4.5–7.5 h for the 180
Gumbel draws of the paper design. A Garnet draw takes 8–15 s (about 1 h for 240 draws). The `lite` profile (10
draws per cell) takes about a third. `lift_bounds` is off by default in paper mode (a Gumbel draw then takes about
20–25 s: about 1 h for 180 Gumbel draws, 2 h in all); enable it for a subset of cells to get the range. A cached draw
reloads in about 3–5 s (instance rebuild and the checks). The smoke run (two draws of one Gumbel and two Garnet
cells, all three lifts and the bounds) still solves the full 609-state Gumbel variant: about 3–5 min.

In [ ]:
# ============================================================
# II.9f Lift sensitivity
# ============================================================
EXECLOG.mark("II.9f Lift sensitivity")

XV4_LIFT_TARGETS = ("true", "true_rows_mean", "true_rows_terminate")   # the true-type targets of an estimated draw
XV4_LIFT_BENCH = "benchmark"         # lift label of the benchmark rows (bench_mean / bench_terminate need no lift)
XV4_LIFT_ENDS = ("worst", "best")    # lift labels of the range ends of J_true over all lifts (target 'true' only)
XV4_LIFT_METRICS = ("return", "prob_any_violation", "expected_violations", "avg_episode_length")
XV4_LIFT_COLUMNS = ["cell", "family", "draw", "policy", "method", "variant", "hp", "is_default", "method_family",
                    "lift", "target", "semantics", "status", *XV4_LIFT_METRICS]
XV4_LIFT_BOUND_TOL = 1e-8            # slack of J_worst <= J_lift <= J_best: the DP / exact gap x_optimal allows
XV4_LIFT_RANGE_CHECKS = ("lift_below_worst", "lift_above_best", "range_negative_width")   # checked at BOUND_TOL
XV4_LIFT_PAIR_TARGETS = {"true_rows": ("true_rows_terminate", "true_rows_mean", "p_true"),
                         "bench": ("bench_terminate", "bench_mean", "p_bench")}


def x_v4_lift_map(V: np.ndarray, Sb: int, rows: np.ndarray, live: np.ndarray,
                  name: str) -> Callable[[np.ndarray], np.ndarray]:
    """Lift of benchmark policies ([Sb, A]) to the true state space: the policy's rows on the decision states V (the
    benchmark states 0..|V|-1), `rows` ([S, A]) on every other true live state, uniform rows on absorbing states (as
    the II.3d lift)."""
    V = np.asarray(V, dtype=np.int64)
    S, A = np.shape(rows)
    outside = np.asarray(live, dtype=bool).copy()
    outside[V] = False
    base = np.full((S, A), 1.0 / A)
    base[outside] = validate_policy(rows, f"{name} rows", shape=(S, A))[outside]

    def lift(pi_bench: np.ndarray) -> np.ndarray:
        pb = validate_policy(pi_bench, f"{name} policy", shape=(int(Sb), A))
        out = base.copy()
        out[V] = pb[:len(V)]
        return out

    return lift


def x_v4_lift_maps(B: Dict[str, Any], true_env: FiniteEnv, true_behavior: np.ndarray,
                   kinds: Sequence[str]) -> Dict[str, Callable[[np.ndarray], np.ndarray]]:
    """The lifts `kinds` of benchmark B: 'fallback' is B['lift']; 'physician' / 'uniform' copy the policy on V and use
    the true behaviour rows / uniform rows on every other true live state."""
    S, A, Sb = true_env.S, true_env.A, int(B["bench"].S)
    V = np.asarray(B["decision_states_true"], dtype=np.int64)
    name = B["bench"].name
    out: Dict[str, Callable[[np.ndarray], np.ndarray]] = {}
    for kind in kinds:
        if kind == "fallback":
            out[kind] = B["lift"]
        elif kind == "physician":
            out[kind] = x_v4_lift_map(V, Sb, true_behavior, true_env.live, f"{name}|lift-physician")
        elif kind == "uniform":
            out[kind] = x_v4_lift_map(V, Sb, np.full((S, A), 1.0 / A), true_env.live, f"{name}|lift-uniform")
        else:
            raise ValueError(f"unknown lift kind {kind!r}")
    return out


def x_v4_lift_bounds(env: FiniteEnv, pi_on_V: np.ndarray, V: np.ndarray,
                     name: str) -> Dict[str, Tuple[np.ndarray, float, Dict[str, Any]]]:
    """Range of J (semantics 'given') of `env` over all lifts of one policy: exact optimal control outside the decision
    states V with the policy's rows `pi_on_V` ([|V|, A]) fixed on V.

    In the variant every action at a state of V has the policy-mixed row sum_a pi(a|s) P(.|s, a) and every action is
    admissible, so its policies are the lifts. 'best': x_optimal of the variant; 'worst': minus the optimum of the
    variant with negated entry rewards. Returns per end (greedy action of every state, end value, solver report). Every
    state of V ties across actions by construction, so the report counts near ties outside V only."""
    V = np.asarray(V, dtype=np.int64)
    pin = validate_policy(pi_on_V, f"{name} rows on V", shape=(len(V), env.A))
    P = env.P.copy()
    P[V] = np.einsum("sa,san->sn", pin, env.P[V], optimize=True)[:, None, :]
    full = np.ones((env.S, env.A), dtype=bool)
    assert x_trap_states(P, env.live, full) == 0, f"{name}: a lift is improper, but the true system has no trap"
    out: Dict[str, Tuple[np.ndarray, float, Dict[str, Any]]] = {}
    for end, sign in (("best", 1.0), ("worst", -1.0)):
        var = FiniteEnv(name=f"{name}|lift-{end}", kind="variant", P=P,
                        entry_reward=np.where(env.live, 0.0, sign * env.entry_reward), d0=env.d0,
                        terminals=env.terminals, sink=env.sink, goal=env.goal, mask=full,
                        meta={"base": env.name, "lift_range_end": end})
        pi_opt, _, info = x_optimal(var, "given")
        ties_out = int(info["near_tie_live_states"]) - (len(V) if env.A > 1 else 0)
        out[end] = (pi_opt.argmax(axis=1), sign * float(info["exact_return_of_greedy"]),
                    {"iterations": int(info["iterations"]), "abs_gap": float(info["abs_gap"]),
                     "near_tie_outside_V": max(ties_out, 0)})
        del var, pi_opt
    del P
    return out


def x_v4_lift_identity(spec: dict, inst: CellInstance) -> dict:
    return {"kind": "xenv_v4_lift", "draw": x_v4_draw_identity(spec, inst), "code": XV4_LIFT_CODE_HASH,
            "lift_kinds": list(XV4CFG.lift_kinds), "lift_bounds": bool(XV4CFG.lift_bounds),
            "columns": XV4_LIFT_COLUMNS, "policies": "default (is_default) records of x_collect_records"}


def x_v4_lift_check_rows(df: pd.DataFrame, v_range: float) -> Dict[str, Any]:
    """Invariants of one draw's lift rows (see markdown); asserted, and a small report of the slacks is returned."""
    bad = sorted(set(df["status"].astype(str)) - {"ok"})
    assert not bad, f"statuses {bad}: the benchmark and the true system are proper for every policy"
    dup = df.duplicated(["policy", "lift", "target"])
    assert not dup.any(), f"{int(dup.sum())} duplicated (policy, lift, target) lift rows"
    w = df.set_index(["policy", "lift", "target"])
    J, Pv = w["return"].unstack("target"), w["prob_any_violation"].unstack("target")
    pols = list(pd.unique(df["policy"]))
    lifts = list(pd.unique(df["lift"]))
    kinds = [x for x in lifts if x not in (XV4_LIFT_BENCH, *XV4_LIFT_ENDS)]
    assert XV4_LIFT_BENCH in lifts and kinds, f"lift rows without benchmark rows or lifts: {lifts}"

    def block(lift: str, targets: Sequence[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
        j = J.xs(lift, level="lift").reindex(index=pols, columns=list(targets))
        p = Pv.xs(lift, level="lift").reindex(index=pols, columns=list(targets))
        assert j.notna().all().all(), f"lift {lift}: missing (policy, target) rows"
        return j, p

    def slacks(lo: pd.Series, up: pd.Series, p: pd.Series) -> Tuple[float, float]:
        width = (up - lo).to_numpy(dtype=float)
        over = width - p.to_numpy(dtype=float) * float(v_range)
        return float((-width).max(initial=-np.inf)), float(over.max(initial=-np.inf))

    jb, pb = block(XV4_LIFT_BENCH, ("bench_mean", "bench_terminate"))
    rep = {"rows": int(len(df)), "policies": len(pols), "lifts": kinds,
           "bench_rule_p_difference": float((pb["bench_mean"] - pb["bench_terminate"]).abs().max())}
    rep["bench_negative_width"], rep["bench_width_minus_bound"] = slacks(jb["bench_terminate"], jb["bench_mean"],
                                                                        pb["bench_terminate"])
    dp, lower, neg, over = [], [], [], []
    for kind in kinds:
        j, p = block(kind, XV4_LIFT_TARGETS)
        dp.append(float(max((p["true"] - p["true_rows_mean"]).abs().max(),
                            (p["true"] - p["true_rows_terminate"]).abs().max())))
        lower.append(float((j["true_rows_terminate"] - j["true"]).max()))
        n_, o_ = slacks(j["true_rows_terminate"], j["true_rows_mean"], p["true_rows_mean"])
        neg.append(n_)
        over.append(o_)
    rep.update({"target_p_difference": max(dp), "lower_bound_shortfall": max(lower), "rows_negative_width": max(neg),
                "rows_width_minus_bound": max(over)})
    for k in ("bench_rule_p_difference", "target_p_difference"):
        assert rep[k] <= XV4_TOL, f"{k} = {rep[k]:.3g}: the completions must share P(any unsupported decision)"
    for k in ("bench_negative_width", "bench_width_minus_bound", "lower_bound_shortfall", "rows_negative_width",
              "rows_width_minus_bound"):
        assert rep[k] <= XV4_TOL, f"{k} = {rep[k]:.3g} (Theorem 1 parts i / iii)"
    rep["has_range"] = all(e in lifts for e in XV4_LIFT_ENDS)
    if rep["has_range"]:
        lo = block("worst", ("true",))[0]["true"]
        hi = block("best", ("true",))[0]["true"]
        below = [float((lo - block(k, ("true",))[0]["true"]).max()) for k in kinds]
        above = [float((block(k, ("true",))[0]["true"] - hi).max()) for k in kinds]
        rep.update({"lift_below_worst": max(below), "lift_above_best": max(above),
                    "range_negative_width": float((lo - hi).max()), "range_width_max": float((hi - lo).max())})
        for k in XV4_LIFT_RANGE_CHECKS:
            assert rep[k] <= XV4_LIFT_BOUND_TOL, f"{k} = {rep[k]:.3g}: the range must contain every lift"
    return rep


def run_v4_lift_draw(spec: dict, draw: int) -> Tuple[pd.DataFrame, dict]:
    """Default policies of one draw under every lift of XV4CFG.lift_kinds on the true-type targets, the benchmark
    values, and the range of J_true over all lifts (cached under v4lift/)."""
    cell, family, draw = spec["cell"], spec["family"], int(draw)
    assert family in ("gumbel", "garnet"), family
    t0 = time.perf_counter()
    inst = x_v4_instance(spec, draw)
    identity = x_v4_lift_identity(spec, inst)
    key, ident_hash = "v4lift/" + x_cache_key(cell, draw), obj_hash(identity)[:20]
    build_s = time.perf_counter() - t0
    hit, meta_hit = XCACHE.get(key, identity), XCACHE.get(key + "/meta", identity)
    if hit is not None and meta_hit is not None:
        del inst
        assert list(hit.columns) == XV4_LIFT_COLUMNS, f"{key}: cached columns differ from XV4_LIFT_COLUMNS"
        meta = json.loads(str(meta_hit["meta_json"].iloc[0]))
        meta.update(cache="loaded", session_build_s=build_s, checks=x_v4_lift_check_rows(hit, meta["v_range"]))
        gc.collect()
        return hit, meta
    if (hit is None) != (meta_hit is None):
        print(f"  {key}: incomplete cache entry (results or meta missing) - recomputing this draw")
    XCACHE.require_allowed(key)
    learn = inst.learn_env
    t1 = time.perf_counter()
    records, keys = x_v4_default_records(inst)
    t2 = time.perf_counter()
    true_env, beh, hosp, B = x_v4_rebuild_benchmark(spec, draw, inst)
    del hosp
    V = np.asarray(B["decision_states_true"], dtype=np.int64)
    nV = len(V)
    assert np.array_equal(np.asarray(B["index"], dtype=np.int64)[:nV], V), "benchmark index must start with V"
    outside = true_env.live.copy()
    outside[V] = False
    tgt = {lab: (env, sem, lift) for lab, env, sem, lift in inst.targets}
    missing = [t for t in (*XV4_LIFT_TARGETS, "bench_mean", "bench_terminate") if t not in tgt]
    assert not missing, f"{cell}: lift sensitivity needs the benchmark and true-type targets; missing {missing}"
    assert all(tgt[t][2] is not None and tgt[t][0].S == true_env.S for t in XV4_LIFT_TARGETS), cell
    assert tgt["true"][1] == "given", tgt["true"][1]
    pi0 = records[0]["pi"]
    assert np.array_equal(B["lift"](pi0), tgt["true"][2](pi0)), \
        f"{cell} draw {draw}: the rebuilt fallback lift differs from the instance's 'true' lift"
    lifts = x_v4_lift_maps(B, true_env, beh, XV4CFG.lift_kinds)
    on_V = [_x_pi_key(np.ascontiguousarray(r["pi"][:nV])) for r in records]   # J under a lift depends on these rows
    lo, hi = learn.value_bounds
    base = {"cell": cell, "family": family, "draw": draw}
    rows: List[dict] = []

    def add_rows(lift: str, target: str, semantics: str, evs: Dict[bytes, dict], rkeys: Sequence[bytes]) -> None:
        for r, k in zip(records, rkeys):
            d, ev = r["desc"], evs[k]
            rows.append({**base, "policy": d["policy"], "method": d["method"], "variant": d["variant"], "hp": d["hp"],
                         "is_default": True, "method_family": d["method_family"], "lift": lift, "target": target,
                         "semantics": semantics, "status": ev["status"],
                         **{c: ev.get(c, np.nan) for c in XV4_LIFT_METRICS}})

    for target in ("bench_mean", "bench_terminate"):
        env, sem, _ = tgt[target]
        memo: Dict[bytes, dict] = {}
        for r, k in zip(records, keys):
            if k not in memo:
                memo[k] = x_v4_eval(env, r["pi"], sem)
        add_rows(XV4_LIFT_BENCH, target, sem, memo, keys)
    n_eval = 0
    for kind, lift in lifts.items():
        for target in XV4_LIFT_TARGETS:
            env, sem, _ = tgt[target]
            memo = {}
            for r, k in zip(records, on_V):
                if k not in memo:
                    memo[k] = x_v4_eval(env, lift(r["pi"]), sem)
                    n_eval += 1
            add_rows(kind, target, sem, memo, on_V)
    t3 = time.perf_counter()
    bound_stats: List[dict] = []
    if XV4CFG.lift_bounds:
        env, sem, _ = tgt["true"]
        ends: Dict[str, Dict[bytes, dict]] = {e: {} for e in XV4_LIFT_ENDS}
        for r, k in zip(records, on_V):
            if k in ends["best"]:
                continue
            res = x_v4_lift_bounds(env, r["pi"][:nV], V, f"{learn.name}|{r['desc']['policy']}")
            for end in XV4_LIFT_ENDS:
                act, value, info = res[end]
                lift_end = x_v4_lift_map(V, learn.S, one_hot_policy(act, env.A), env.live, f"{learn.name}|lift-{end}")
                ev = x_v4_eval(env, lift_end(r["pi"]), sem)
                gap = abs(float(ev.get("return", np.nan)) - value)
                assert ev["status"] == "ok" and gap <= XV4_TOL * max(1.0, abs(value)), \
                    f"{cell} draw {draw} {r['desc']['policy']}: the {end} lift on 'true' gives {ev} vs {value}"
                ends[end][k] = ev
                bound_stats.append({"end": end, "value_gap": gap, **info})
            del res
        for end in XV4_LIFT_ENDS:
            add_rows(end, "true", sem, ends[end], on_V)
    t4 = time.perf_counter()
    df = pd.DataFrame(rows, columns=XV4_LIFT_COLUMNS)
    checks = x_v4_lift_check_rows(df, hi - lo)
    bs = pd.DataFrame(bound_stats)
    meta = _x_compact({
        "cell": cell, "family": family, "draw": draw, "identity_hash": ident_hash, "code_hash": XV4_LIFT_CODE_HASH[:16],
        "cache": "computed", "v_range": float(hi - lo), "lift_kinds": list(lifts),
        "lift_bounds": bool(XV4CFG.lift_bounds), "n_policies": len(records), "unique_policies": len(set(keys)),
        "unique_rows_on_V": len(set(on_V)), "true_space_evaluations": n_eval, "decision_states": int(nV),
        "outside_live_states": int(outside.sum()), "true_d0_mass_outside": float(true_env.d0[outside].sum()),
        "fallback_action": int(B["fallback_action"]),
        "bounds": ({"solves": int(len(bs)), "vi_iterations_max": int(bs["iterations"].max()),
                    "vi_iterations_mean": float(bs["iterations"].mean()), "dp_gap_max": float(bs["abs_gap"].max()),
                    "value_gap_max": float(bs["value_gap"].max()),
                    "near_tie_outside_V_max": int(bs["near_tie_outside_V"].max())} if len(bs) else {"solves": 0}),
        "checks": checks,
        "timing": {"build_s": build_s, "methods_s": t2 - t1, "eval_s": t3 - t2, "bounds_s": t4 - t3,
                   "total_s": time.perf_counter() - t0}})
    XCACHE.put(key + "/meta", identity, pd.DataFrame({"meta_json": [json.dumps(meta, default=_json_default)]}))
    XCACHE.put(key, identity, df)
    back = XCACHE.get(key, identity)
    assert back is not None and list(back.columns) == XV4_LIFT_COLUMNS and len(back) == len(df), f"{key}: cache write"
    for c in XV4_LIFT_METRICS:
        assert np.allclose(back[c].to_numpy(dtype=float), df[c].to_numpy(dtype=float), rtol=1e-14, atol=0.0,
                           equal_nan=True), f"{key}: cache round trip changed column {c}"
    for c in ("status", "lift", "target", "policy"):
        assert (back[c].astype(str).to_numpy() == df[c].astype(str).to_numpy()).all(), f"{key}: cache round trip {c}"
    del inst, df, B, lifts
    gc.collect()
    return back, meta


# Objects whose source changes what a lift draw computes or stores (the checks are re-run on cached tables);
# _x_v3_eval (II.8g) is the evaluation wrapper behind x_v4_eval, whose keys are the stored metric columns.
XV4_LIFT_CODE = [x_v4_lift_map, x_v4_lift_maps, x_v4_lift_bounds, x_v4_lift_identity, run_v4_lift_draw, _x_v3_eval]
XV4_LIFT_CODE_HASH = obj_hash({"common": XV4_COMMON_CODE_HASH, "code": code_hash(*XV4_LIFT_CODE),
                               "constants": {"targets": list(XV4_LIFT_TARGETS), "bench": XV4_LIFT_BENCH,
                                             "ends": list(XV4_LIFT_ENDS), "metrics": list(XV4_LIFT_METRICS),
                                             "columns": XV4_LIFT_COLUMNS, "bound_tol": XV4_LIFT_BOUND_TOL}})


# ---------- analyses (pure pandas on the stacked rows) ----------
_XV4_LIFT_FLAGS = ["viol", "viol_true", "sem_lo_ok", "sem_up_ok", "b_lo_ok", "b_up_ok", "b_in", "width",
                   "bench_width", "bound_width", "bench_covers_range", "change_vs_fallback"]
XV4_LIFT_FRAME_COLUMNS = ["cell", "family", "draw", "policy", "lift", "v_range", *XV4_LIFT_TARGETS, "p_true",
                          "bench_mean", "bench_terminate", "p_bench", "j_lo", "j_hi", "true_fallback", *_XV4_LIFT_FLAGS]


def x_v4_lift_frame(res: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Per (cell, draw, policy, lift): J_true and the true-rows bracket under the lift (p from true_rows_mean), the
    benchmark values (p from bench_mean), the range ends J_lo / J_hi over all lifts (NaN without bounds), J_true under
    the fallback lift, and the coverage flags."""
    if res is None or not len(res):
        return pd.DataFrame(columns=XV4_LIFT_FRAME_COLUMNS)
    r = res.copy()
    r["draw"] = pd.to_numeric(r["draw"]).astype(np.int64)
    k = ["cell", "draw", "policy"]
    w = r.set_index(k + ["lift", "target"])
    J, Pv = w["return"], w["prob_any_violation"]
    lifts = list(pd.unique(r["lift"]))
    kinds = [x for x in lifts if x not in (XV4_LIFT_BENCH, *XV4_LIFT_ENDS)]
    jb = J.xs(XV4_LIFT_BENCH, level="lift").unstack("target")
    side = pd.DataFrame({"bench_mean": jb["bench_mean"], "bench_terminate": jb["bench_terminate"],
                         "p_bench": Pv.xs(XV4_LIFT_BENCH, level="lift").unstack("target")["bench_mean"]})

    def true_of(lift: str) -> pd.Series:
        if lift not in lifts:
            return pd.Series(np.nan, index=side.index)
        return J.xs(lift, level="lift").xs("true", level="target").reindex(side.index)

    side["j_lo"], side["j_hi"], side["true_fallback"] = true_of("worst"), true_of("best"), true_of("fallback")
    side = side.join(r.drop_duplicates(k).set_index(k)[["family"]])
    parts = []
    for kind in kinds:
        jk = J.xs(kind, level="lift").unstack("target").reindex(side.index)
        pk = Pv.xs(kind, level="lift").unstack("target").reindex(side.index)
        parts.append(side.assign(lift=kind, **{t: jk[t] for t in XV4_LIFT_TARGETS}, p_true=pk["true_rows_mean"]))
    W = pd.concat(parts).reset_index()
    m = meta.assign(draw=pd.to_numeric(meta["draw"]).astype(np.int64)).set_index(["cell", "draw"])["v_range"]
    W["v_range"] = m.reindex(pd.MultiIndex.from_frame(W[["cell", "draw"]])).to_numpy(dtype=float)
    assert W["v_range"].notna().all(), "draw meta without v_range"
    W["viol"] = W["p_bench"] > XV4_PTOL
    W["viol_true"] = W["p_true"] > XV4_PTOL
    W["sem_lo_ok"] = W["true"] >= W["true_rows_terminate"] - XV4_TOL
    W["sem_up_ok"] = W["true"] <= W["true_rows_mean"] + XV4_TOL
    W["b_lo_ok"] = W["true"] >= W["bench_terminate"] - XV4_TOL
    W["b_up_ok"] = W["true"] <= W["bench_mean"] + XV4_TOL
    W["b_in"] = W["b_lo_ok"] & W["b_up_ok"]
    W["width"] = W["true_rows_mean"] - W["true_rows_terminate"]
    W["bench_width"] = W["bench_mean"] - W["bench_terminate"]
    W["bound_width"] = W["j_hi"] - W["j_lo"]
    W["bench_covers_range"] = (W["bench_terminate"] <= W["j_lo"] + XV4_TOL) & (W["j_hi"] <= W["bench_mean"] + XV4_TOL)
    W["change_vs_fallback"] = W["true"] - W["true_fallback"]
    return W[XV4_LIFT_FRAME_COLUMNS]


def x_v4_lift_draw_stats(W: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """(pairs, ranks, range) per draw. pairs: x_v4_pair_claims per (cell, draw, lift, targets) with J_true under the
    lift as truth; ranks: Kendall tau-b / top-1 agreement of each benchmark rule against J_true under the lift; range:
    benchmark NHB / NHB-free claims that hold (J_lo(i) > J_hi(j)) or fail (J_hi(i) <= J_lo(j)) under every lift, both
    with a margin of 2 * XV4_LIFT_BOUND_TOL beyond the tie tolerance XV4_TOL."""
    pair_rows, rank_rows, range_rows = [], [], []
    for (c, dr, lift), d in W.groupby(["cell", "draw", "lift"], sort=False):
        unit = {"cell": c, "family": d["family"].iloc[0], "draw": int(dr), "lift": lift}
        vr, jt = float(d["v_range"].iloc[0]), d["true"].to_numpy(dtype=float)
        for tf, (lc, uc, pc) in XV4_LIFT_PAIR_TARGETS.items():
            for r in x_v4_pair_claims(d[lc].to_numpy(dtype=float), d[uc].to_numpy(dtype=float),
                                      d[pc].to_numpy(dtype=float), vr, jt):
                pair_rows.append({**unit, "targets": tf, **r})
        for rule in ("bench_mean", "bench_terminate"):
            ra = xs_rank_agreement(d[rule].to_numpy(dtype=float), jt, np.full(len(d), np.nan))
            rank_rows.append({**unit, "ranked_by": rule, "evaluated_on": "true",
                              **{m: ra[m] for m in ("n_policies", "kendall_tau_b", "top1_agreement",
                                                    "tau_benchmark_constant", "tau_undefined_constant_values")}})
    one = W.drop_duplicates(["cell", "draw", "policy"])
    for (c, dr), d in one.groupby(["cell", "draw"], sort=False):
        if d[["j_lo", "j_hi"]].isna().any().any():
            continue
        n, vr = len(d), float(d["v_range"].iloc[0])
        off = ~np.eye(n, dtype=bool)
        blo, bup, pb = (d[x].to_numpy(dtype=float) for x in ("bench_terminate", "bench_mean", "p_bench"))
        jlo, jhi = d["j_lo"].to_numpy(dtype=float), d["j_hi"].to_numpy(dtype=float)
        # each end is the exact value of an attained lift, within the solver slack XV4_LIFT_BOUND_TOL of the extreme:
        # a margin of 2 * BOUND_TOL around the tie tolerance keeps 'hold' (correct under every lift) and 'fail'
        # (wrong, ties included, under every lift) valid whatever that slack does; the rest are lift-dependent
        hold = jlo[:, None] - jhi[None, :] > XV4_TOL + 2.0 * XV4_LIFT_BOUND_TOL
        fail = jhi[:, None] - jlo[None, :] <= XV4_TOL - 2.0 * XV4_LIFT_BOUND_TOL
        u = pb > XV4_PTOL
        scopes = {"all": off, "cs": off & (u[:, None] | u[None, :])}
        r = {"cell": c, "family": d["family"].iloc[0], "draw": int(dr), "n_policies": n}
        for nm, C in (("cert_nhb", _x_v4_gt(blo, bup)), ("cert_free", _x_v4_gt(blo, blo + pb * vr))):
            for sc, msk in scopes.items():
                Cm = C & msk
                r[f"{nm}_{sc}_claims"] = int(Cm.sum())
                r[f"{nm}_{sc}_hold_every_lift"] = int((Cm & hold).sum())
                r[f"{nm}_{sc}_wrong_every_lift"] = int((Cm & fail).sum())
        range_rows.append(r)
    return pd.DataFrame(pair_rows), pd.DataFrame(rank_rows), pd.DataFrame(range_rows)


def x_v4_lift_summary(W: pd.DataFrame, D: pd.DataFrame, R: pd.DataFrame, G: pd.DataFrame, cells: Sequence[str],
                      n_boot: int) -> pd.DataFrame:
    """Per unit (cell, family pool, all cells) x lift: coverage, certificates, rank fidelity and the change against
    the fallback lift; plus one 'range' row per unit with the bounds statistics."""
    if W.empty:
        return pd.DataFrame()
    present = set(W["cell"])
    units = [(c, [c]) for c in cells if c in present]
    fams = sorted(set(W["family"]))
    units += [(f"pooled:{f}", [c for c in cells if c in set(W.loc[W["family"] == f, "cell"])]) for f in fams]
    if len(fams) > 1:
        units.append(("pooled:all", [c for c in cells if c in present]))
    kinds = list(pd.unique(W["lift"]))
    rows = []
    for name, cs in units:
        for lift in kinds:
            w = W[W["cell"].isin(cs) & (W["lift"] == lift)]
            if w.empty:
                continue
            v, vt = w[w["viol"]], w[w["viol_true"]]
            ch = w["change_vs_fallback"].dropna().abs()
            row = {"unit": name, "n_cells": len(cs), "lift": lift,
                   "n_draws": int(w[["cell", "draw"]].drop_duplicates().shape[0]), "n_policy_draws": int(len(w)),
                   "n_violating_bench": int(len(v)), "n_violating_true": int(len(vt)),
                   "sem_lower_violations": int((~w["sem_lo_ok"]).sum()),
                   "sem_upper_coverage_all": _xv3_rate(w["sem_up_ok"].sum(), len(w)),
                   "sem_upper_coverage_violating": _xv3_rate(vt["sem_up_ok"].sum(), len(vt)),
                   "sem_width_mean_violating": float(vt["width"].mean()) if len(vt) else np.nan,
                   "bench_coverage_all": _xv3_rate(w["b_in"].sum(), len(w)),
                   "bench_coverage_violating": _xv3_rate(v["b_in"].sum(), len(v)),
                   "bench_lower_holds_violating": _xv3_rate(v["b_lo_ok"].sum(), len(v)),
                   "bench_upper_holds_violating": _xv3_rate(v["b_up_ok"].sum(), len(v)),
                   "true_change_vs_fallback_mean_abs": float(ch.mean()) if len(ch) else np.nan,
                   "true_change_vs_fallback_max_abs": float(ch.max()) if len(ch) else np.nan,
                   "true_changed_vs_fallback_share": _xv3_rate((ch > XV4_TOL).sum(), len(ch))}
            d = D[D["cell"].isin(cs) & (D["lift"] == lift)]
            for tf in XV4_LIFT_PAIR_TARGETS:
                dt = d[d["targets"] == tf]
                a, s = dt[dt["type"] == "all"], dt[dt["type"].isin(["AU", "UU"])]
                pa, ps = int(a["pairs"].sum()), int(s["pairs"].sum())
                row[f"{tf}_pairs"], row[f"{tf}_cs_pairs"] = pa, ps
                for kind in ("cert_nhb", "cert_free", "point_mean", "point_term"):
                    n, e = int(a[f"{kind}_claims"].sum()), int(a[f"{kind}_errors"].sum())
                    row.update({f"{tf}_{kind}_claims": n, f"{tf}_{kind}_share": _xv3_rate(n, pa),
                                f"{tf}_{kind}_errors": e, f"{tf}_{kind}_error_rate": _xv3_rate(e, n)})
                for kind in ("cert_nhb", "cert_free"):
                    n, e = int(s[f"{kind}_claims"].sum()), int(s[f"{kind}_errors"].sum())
                    row.update({f"{tf}_{kind}_cs_claims": n, f"{tf}_{kind}_cs_share": _xv3_rate(n, ps),
                                f"{tf}_{kind}_cs_errors": e, f"{tf}_{kind}_cs_error_rate": _xv3_rate(e, n),
                                f"{tf}_draws_any_wrong_{kind}": int((a[f"{kind}_errors"] > 0).sum())})
            rk = R[R["cell"].isin(cs) & (R["lift"] == lift)]
            for rule in ("bench_mean", "bench_terminate"):
                t = rk[rk["ranked_by"] == rule]
                row[f"tau_{rule}_mean"] = float(t["kendall_tau_b"].mean()) if len(t) else np.nan
                row[f"top1_{rule}_mean"] = float(t["top1_agreement"].mean()) if len(t) else np.nan
            piv = rk.pivot_table(index=["cell", "draw"], columns="ranked_by", values="kendall_tau_b", aggfunc="first")
            dlt = ((piv["bench_mean"] - piv["bench_terminate"]).dropna()
                   if {"bench_mean", "bench_terminate"} <= set(piv.columns) else pd.Series(dtype=float))
            groups = ({c: g.to_numpy(dtype=np.float64)[:, None] for c, g in dlt.groupby(level="cell")}
                      if len(dlt) else {})
            point, reps = xs_stratified_bootstrap(groups, xs_stat_mean(), ("v4lift", lift, "tau difference"), n_boot)
            row["tau_diff_mean"] = point
            row["tau_diff_ci_low"], row["tau_diff_ci_high"] = xs_ci(reps)
            row["tau_diff_n"] = int(len(dlt))
            rows.append(row)
        one = W[W["cell"].isin(cs)].drop_duplicates(["cell", "draw", "policy"])
        one = one[one["j_lo"].notna() & one["j_hi"].notna()]
        if one.empty:
            continue
        v = one[one["viol"]]
        row = {"unit": name, "n_cells": len(cs), "lift": "range",
               "n_draws": int(one[["cell", "draw"]].drop_duplicates().shape[0]), "n_policy_draws": int(len(one)),
               "n_violating_bench": int(len(v)), "bound_width_mean": float(one["bound_width"].mean()),
               "bound_width_median": float(one["bound_width"].median()),
               "bound_width_max": float(one["bound_width"].max()),
               "bound_width_mean_violating": float(v["bound_width"].mean()) if len(v) else np.nan,
               "bound_width_zero_share": _xv3_rate((one["bound_width"] <= XV4_TOL).sum(), len(one)),
               "fallback_gap_to_best_mean": float((one["j_hi"] - one["true_fallback"]).mean()),
               "fallback_gap_to_worst_mean": float((one["true_fallback"] - one["j_lo"]).mean()),
               "bench_covers_range_share": _xv3_rate(one["bench_covers_range"].sum(), len(one)),
               "bench_covers_range_share_violating": _xv3_rate(v["bench_covers_range"].sum(), len(v))}
        g = G[G["cell"].isin(cs)] if len(G) else G
        for nm in ("cert_nhb", "cert_free"):
            for sc in ("all", "cs"):
                n = int(g[f"{nm}_{sc}_claims"].sum()) if len(g) else 0
                h = int(g[f"{nm}_{sc}_hold_every_lift"].sum()) if len(g) else 0
                f_ = int(g[f"{nm}_{sc}_wrong_every_lift"].sum()) if len(g) else 0
                row.update({f"bench_{nm}_{sc}_claims": n, f"bench_{nm}_{sc}_hold_every_lift": h,
                            f"bench_{nm}_{sc}_hold_every_lift_share": _xv3_rate(h, n),
                            f"bench_{nm}_{sc}_wrong_every_lift": f_,
                            f"bench_{nm}_{sc}_wrong_every_lift_share": _xv3_rate(f_, n),
                            f"bench_{nm}_{sc}_lift_dependent": n - h - f_})
            if len(g):   # draws with a certified benchmark claim that some lift could make wrong
                row[f"bench_draws_any_{nm}_not_robust"] = int(
                    (g[f"{nm}_all_claims"] > g[f"{nm}_all_hold_every_lift"]).sum())
        rows.append(row)
    return pd.DataFrame(rows)


XV4_LIFT_HEADLINE_COLUMNS = ("n_draws", "n_policy_draws", "sem_upper_coverage_violating", "bench_coverage_violating",
                             "true_rows_cert_nhb_error_rate", "true_rows_cert_free_error_rate",
                             "bench_cert_nhb_error_rate", "bench_cert_free_error_rate", "bench_cert_nhb_cs_error_rate",
                             "bench_cert_free_cs_error_rate", "tau_bench_mean_mean", "tau_bench_terminate_mean",
                             "tau_diff_mean", "tau_diff_ci_low", "tau_diff_ci_high",
                             "true_change_vs_fallback_mean_abs", "bound_width_mean", "bound_width_mean_violating",
                             "bench_covers_range_share", "bench_cert_nhb_all_hold_every_lift_share",
                             "bench_cert_free_all_hold_every_lift_share", "bench_cert_nhb_all_wrong_every_lift_share",
                             "bench_cert_free_all_wrong_every_lift_share")


def x_v4_lift_headline(summary: pd.DataFrame) -> Dict[str, Any]:
    """Headline numbers per lift of the widest pooled unit (pooled:all, else the family pool, else the only cell)."""
    if summary is None or summary.empty:
        return {}
    units = list(pd.unique(summary["unit"]))
    unit = "pooled:all" if "pooled:all" in units else next((u for u in units if u.startswith("pooled:")), units[0])
    s = summary[summary["unit"] == unit].set_index("lift")
    out: Dict[str, Any] = {"unit": unit}
    for lift, r in s.iterrows():
        out[str(lift)] = {c: r[c] for c in XV4_LIFT_HEADLINE_COLUMNS if c in r.index and pd.notna(r[c])}
    return out


def x_v4_lift_point_check(res: pd.DataFrame) -> Dict[str, Any]:
    """Fallback-lift and benchmark rows of the regenerated default policies against the II.6 / II.8 rows of the same
    draws (x_boot_point_check; reported, not asserted)."""
    pt = res[res["lift"].isin(["fallback", XV4_LIFT_BENCH])].assign(replicate=XV3_BOOT_POINT)
    cells = set(pt["cell"])
    xr = [x[x["cell"].isin(cells)] for x in (globals().get("XRESULTS"), globals().get("XRESULTS_V3"))
          if x is not None and len(x)]
    xr = [x for x in xr if len(x)]
    return x_boot_point_check(pt, pd.concat(xr, ignore_index=True) if xr else None)


XV4_LIFT_CHECK_KEYS = ("bench_rule_p_difference", "bench_negative_width", "bench_width_minus_bound",
                       "target_p_difference", "lower_bound_shortfall", "rows_negative_width", "rows_width_minus_bound",
                       "lift_below_worst", "lift_above_best", "range_negative_width", "range_width_max")


def x_v4_lift_checks(meta: pd.DataFrame) -> Dict[str, Any]:
    """Largest per-draw slack of every asserted invariant (each was asserted per draw), the solver statistics, and
    `passes` (every slack within its tolerance: XV4_TOL, or XV4_LIFT_BOUND_TOL for the range checks)."""
    out: Dict[str, Any] = {"draws_checked": int(len(meta))}
    for k in XV4_LIFT_CHECK_KEYS:
        col = f"checks.{k}"
        out[f"max_{k}"] = float(pd.to_numeric(meta[col], errors="coerce").max()) if col in meta else None
    for k in ("bounds.vi_iterations_max", "bounds.dp_gap_max", "bounds.value_gap_max",
              "bounds.near_tie_outside_V_max"):
        out[f"max_{k.split('.', 1)[1]}"] = float(pd.to_numeric(meta[k], errors="coerce").max()) if k in meta else None
    out["bound_solves"] = (int(pd.to_numeric(meta["bounds.solves"], errors="coerce").sum())
                           if "bounds.solves" in meta else 0)
    out["tolerances"] = {"value": XV4_TOL, "p": XV4_PTOL, "bounds": XV4_LIFT_BOUND_TOL}
    slack = {k: out[f"max_{k}"] for k in XV4_LIFT_CHECK_KEYS if k != "range_width_max"}
    out["passes"] = bool(len(meta)) and all(
        v is None or not np.isfinite(v) or v <= (XV4_LIFT_BOUND_TOL if k in XV4_LIFT_RANGE_CHECKS else XV4_TOL)
        for k, v in slack.items())
    return out


XV4_LIFT_ANALYSIS_CODE = [x_v4_lift_check_rows, x_v4_lift_frame, x_v4_lift_draw_stats, x_v4_lift_summary,
                          x_v4_lift_headline, x_v4_lift_point_check, x_v4_lift_checks]

XV4_LIFT_FRAMES = ("results", "draw_meta", "policies", "pairs", "ranks", "range", "summary")
XV4_LIFT: Dict[str, pd.DataFrame] = {k: pd.DataFrame() for k in XV4_LIFT_FRAMES}
XV4_LIFT["results"] = pd.DataFrame(columns=XV4_LIFT_COLUMNS)
XV4_LIFT_CELLS: List[dict] = []
XV4_LIFT_SUMMARY: Dict[str, Any] = {"status": "not run (xenv_v4_lift off)", "code_hash": XV4_LIFT_CODE_HASH,
                                    "checks": {}, "elapsed_s": 0.0, "draws_complete": None}
if EXPERIMENTS["xenv_v4_lift"]:
    _t0 = time.perf_counter()
    XV4_LIFT_CELLS, _skipped = x_v4_cells(XV4CFG.lift_cells)
    if _skipped:
        print("NOTE: lift cells not enabled (family switch off or not in the design), skipped:", _skipped)
    if XV4_LIFT_CELLS:
        _frames, _meta, _run = x_v4_run_draws(XV4_LIFT_CELLS, run_v4_lift_draw, "II.9f lift")
        _res = pd.concat(_frames, ignore_index=True) if _frames else pd.DataFrame(columns=XV4_LIFT_COLUMNS)
        del _frames
        _cells = [str(s["cell"]) for s in XV4_LIFT_CELLS]
        _W = x_v4_lift_frame(_res, _meta)
        _D, _R, _G = x_v4_lift_draw_stats(_W)
        _summ = x_v4_lift_summary(_W, _D, _R, _G, _cells, int(XCFG.bootstrap))
        XV4_LIFT = {"results": _res, "draw_meta": _meta, "policies": _W, "pairs": _D, "ranks": _R, "range": _G,
                    "summary": _summ}
        _res.to_csv(XENV_TABLE_DIR / "v4_lift_results.csv.gz", index=False, compression="gzip")
        _summ.to_csv(XENV_TABLE_DIR / "v4_lift_summary.csv", index=False)
        _n_meta = _meta["cell"].value_counts() if len(_meta) else pd.Series(dtype=int)
        XV4_LIFT_SUMMARY = {
            "status": "run", "code_hash": XV4_LIFT_CODE_HASH, **_run, "cells_skipped": _skipped,
            "draws_complete": all(int(_n_meta.get(s["cell"], 0)) == x_v4_draws(s) for s in XV4_LIFT_CELLS),
            "lift_kinds": list(XV4CFG.lift_kinds), "lift_bounds": bool(XV4CFG.lift_bounds), "rows": int(len(_res)),
            "analysis_code_hash": code_hash(*XV4_LIFT_ANALYSIS_CODE)[:16], "checks": x_v4_lift_checks(_meta),
            "point_check": x_v4_lift_point_check(_res), "headline": x_v4_lift_headline(_summ),
            "outputs": ["xenv/v4_lift_results.csv.gz", "xenv/v4_lift_summary.csv", "xenv/v4_lift_summary.json"],
            "elapsed_s": time.perf_counter() - _t0}
        if XV4_LIFT_SUMMARY["point_check"].get("match") is False:
            print("NOTE: regenerated fallback / benchmark rows differ from XRESULTS on",
                  XV4_LIFT_SUMMARY["point_check"]["rows_above_tol"], "rows (max",
                  XV4_LIFT_SUMMARY["point_check"]["max_abs_diff"], ") - see xenv/v4_lift_summary.json.")
        del _res, _meta, _run, _cells, _W, _D, _R, _G, _summ, _n_meta
    else:
        XV4_LIFT_SUMMARY = {"status": "not run: no configured lift cell is enabled", "code_hash": XV4_LIFT_CODE_HASH,
                            "cells_skipped": _skipped, "checks": {}, "draws_complete": None,
                            "elapsed_s": time.perf_counter() - _t0}
    del _t0, _skipped
save_json(XV4_LIFT_SUMMARY, XENV_TABLE_DIR / "v4_lift_summary.json")
print(json.dumps({k: v for k, v in XV4_LIFT_SUMMARY.items() if k not in ("draws_per_cell", "headline")}, indent=1,
                 default=_json_default))
if len(XV4_LIFT["summary"]):
    _cols = [c for c in ("unit", "lift", "n_draws", "sem_upper_coverage_violating", "bench_coverage_violating",
                         "true_rows_cert_nhb_error_rate", "bench_cert_nhb_error_rate", "bench_cert_free_error_rate",
                         "tau_bench_mean_mean", "tau_bench_terminate_mean", "true_change_vs_fallback_mean_abs",
                         "bound_width_mean", "bench_cert_nhb_all_hold_every_lift_share",
                         "bench_cert_free_all_hold_every_lift_share") if c in XV4_LIFT["summary"].columns]
    display(XV4_LIFT["summary"][_cols].round(4))
    del _cols

### II.9g Clean rerun of Part II into an empty cache

Every Part II number so far was produced through the identity-addressed cache `XCACHE`. Draws computed in different
Colab sessions were joined by their identities; the v2 session, for example, reloaded all 500 II.6 draws from the
cache (`xenv_driver_summary.json`). A run without caches is the strongest replay check. This section reruns the Part II
draws from scratch into a new, **empty** cache directory and compares every rerun table row by row with the cached
table of the same draw. A match shows that each cached result is what the current code computes from scratch: no
stale entry under a matching identity, no state that the identity misses, and a deterministic pipeline.

**Scope** (`XV4CFG.clean_scope`; `max_draws` caps both):

| Scope | Draws rerun |
|---|---|
| `all` (paper + full) | every draw of every enabled II.6 cell (`X_CELLS`, family switches as in II.6a; `run_cell_draw`) and, with `clean_include_v3`, of every II.8 cell (`X_V3_CELLS`; `run_v3_cell_draw`). With `clean_include_v3_boot` also the II.8g bootstrap draws (`run_v3_boot_draw`). That is off by default: II.8g already regenerated the default policies, hospital data and benchmarks of its draws without reading the II.6 cache, and compared its replicate −1 values with the II.6 rows (`x_boot_point_check`). |
| `sample` (lite profile and smoke mode) | the first `clean_sample_draws` draws of each of these cells |

Part I (GPU training of the critics, policy networks and diffusion models) is out of scope. Its clean-training
command is a fresh `ROOT` or `FORCE_RETRAIN` (Section 1), and its artifact replay is checked in Section 32. Only the
CPU-bound exact computation of Part II is rerun here.

**Steps.**
1. The rerun needs `ALLOW_TRAINING = True` (asserted in `x_v4_clean_run`). In artifact-replay mode
   (`ALLOW_TRAINING = False`) every draw would have to be computed, so the rerun is not started and the section
   records `not run`.
2. Create a new directory `XV4CFG.clean_cache_dir/run-<UTC stamp>-<pid>` on the Colab local disk. It must not be
   inside `ROOT` and must not exist yet (asserted). Rebind the global `XCACHE` to `XCache(<that directory>)` inside
   `try / finally` and restore it afterwards. Every II.6 / II.8 function reads `XCACHE` at call time, so the rerun
   reads and writes only the clean cache and the main cache is never written. The in-memory memo of reference optima
   (`_XREF_MEMO`) holds results, so it is emptied for the run and restored afterwards. The deterministic construction
   caches are kept: the exactly enumerated Gumbel systems with their physician policies and true optimal actions,
   the Garnet systems and the diffusion schedules. They are deterministic inputs of a draw, not results: the systems
   and the physicians enter the draw identity through the instance hashes, and the true optimal actions enter only a
   construction statistic of the draw meta.
3. Per draw, rerun the draw. It must report `cache = computed`, since the clean cache starts empty (asserted). A
   rerun that raises (an invariant of the draw fails, say) is recorded as a failing draw with the message and has no
   rerun rows; the check goes on with the next draw. Then take the cached table of the same draw: its rows in
   `XRESULTS`, `XRESULTS_V3` or `XV3_BOOT_RESULTS` (the tables read back from the main cache in II.6a, II.8f and
   II.8g). If those lack the draw, read the main-cache entry under the identity of the regenerated draw
   (`x_v4_instance` and the draw identity); if that read raises, the draw has no cached table (recorded with the
   message). Before the rerun, a self-check reads draw 0 of the first cell of each section both ways; the two tables
   must be identical (reported as `reference_selfcheck`, not asserted).
4. **Rows.** Key on (cell, draw, policy, target), plus the replicate for II.8g. Count the rows missing on either side
   and the status mismatches. For every numeric column, report the max |Δ| over the matched rows and the number of
   rows in which exactly one side is NaN. Count the rows with |Δ| > `clean_tol`·max(1, |cached|) in any column.
   Compare the label columns (method, hp, solver, converged, error, …) as text, and record whether each row is
   exactly equal.
5. **Certified counts.** On both tables, compute the certified pairs of `x_v4_pair_claims` per draw, over the default
   non-reference policies that are `ok` under both rules. Count NHB (`cert_nhb`) and NHB-free (`cert_free`) pairs on
   the true rows (`true_rows_terminate` / `true_rows_mean`, p from `true_rows_mean`) and on the benchmark
   (`bench_terminate` / `bench_mean`, p from `bench_mean`), with v_max − v_min from `XV3_VRANGE`. II.8g tables are
   scored per replicate on the benchmark rules: replicate −1 as `bench` and the B replicates pooled as `boot_bench`.
6. Delete the clean cache directory (`shutil.rmtree`), also after a failure, unless the environment variable
   `SEPSIS_KEEP_CLEAN_CACHE=1` is set.

**Pass criterion.** It is printed and stored as `XV4_CLEAN["pass"]` with its four parts. It is never asserted here;
the Section 34 gate uses it. The criterion: every |Δ| ≤ `clean_tol`·max(1, |cached|) with no NaN-pattern change;
identical statuses; identical certified counts; and no row missing on either side, so every draw must have a cached
table and a rerun that did not raise. Label columns (solver, converged, error text, …) are compared but do not enter
the criterion. Bitwise equality, with every value and label identical, is reported separately as
`bitwise_identical`. Another BLAS or CPU can change the last bits of a linear solve without changing any reported
number.

| Output | Content |
|---|---|
| `XV4_CLEAN` · `xenv/v4_clean_check.json` | status, code hash, scope and configuration, the pass flag and its four criteria, `bitwise_identical`, pooled checks (draws, draws whose rerun raised, rows, missing rows, status and label mismatches, max \|Δ\| per column, rows above tolerance, certified-count differences), the failing draws, the reference self-check, per-section driver summaries, `elapsed_s` and `draws_complete` |
| `XV4_CLEAN_CELLS` · `xenv/v4_clean_diffs.csv` | per section × cell and pooled (`all`): draws (passing, exact, without a cached table, rerun raised), rows, missing rows, status mismatches, max \|Δ\| and NaN changes per numeric column, rows above tolerance, label mismatches, certified counts of both tables and their differences, rerun time |
| `XV4_CLEAN_DRAWS` | the same per draw (in memory only) |

**Expected cost on Colab (CPU-bound).** These are estimates, not measurements of this section. A rerun costs what
computing its draws cost, plus well under a second per draw to write and read back the clean cache entry and to
compare the two tables. The compute times recorded when the II.6 draws were computed (`timing.total_s` in
`xenv_draw_meta.csv`) sum to about 107 min for the 500 draws: about 50 s per ICU-Sepsis draw (42 min), 14 s per
Gumbel draw (48 min) and 4 s per Garnet draw (17 min). The 27.5 min of the II.6 driver in the v2 session
(`xenv_driver_summary.json`) was the reload of cached draws, not their computation. The 180 II.8 draws take about
35–40 min (II.8f). Paper + full, scope `all`: about 2.5 h in all; the optional II.8g bootstrap adds 20–25 min. Lite
(scope `sample`, 3 draws of each of the 16 II.6 and 6 II.8 cells): about 15 min. Smoke: 1–2 min. The self-check adds
one instance rebuild per section (seconds). Before the rerun starts, the cell prints the expected time from the
compute times recorded in the cached draw meta (`xenv_draw_meta.csv`, `v3_draw_meta.csv`, `v3_bootstrap_meta.csv`),
and an ETA after every cell and section. The clean cache holds a few tens of MB on the local disk. The rerun is not
resumable: after a disconnect it starts again in a new directory.

In [ ]:
# ============================================================
# II.9g Clean rerun of Part II into an empty cache
# ============================================================
EXECLOG.mark("II.9 Second-revision experiments: clean rerun")
import tempfile

XV4_CLEAN_RESULT_KEYS = ("cell", "draw", "policy", "target")
XV4_CLEAN_BOOT_KEYS = ("cell", "draw", "policy", "replicate", "target")
XV4_CLEAN_BOOT_NUMERIC = ("return", "prob_any_violation", "expected_violations", "avg_episode_length")
# (name, lower rule, upper rule) of the certified-count comparison; p is read from the upper (mean-rule) target
XV4_CLEAN_RULES = (("true_rows", "true_rows_terminate", "true_rows_mean"), ("bench", "bench_terminate", "bench_mean"))
XV4_CLEAN_KEEP_ENV = "SEPSIS_KEEP_CLEAN_CACHE"     # "1" keeps the clean cache directory after the run
assert set(XV4_CLEAN_RESULT_KEYS) <= set(XRESULT_COLUMNS)
assert set(XV4_CLEAN_BOOT_KEYS) | set(XV4_CLEAN_BOOT_NUMERIC) <= set(XV3_BOOT_COLUMNS)


def x_v4_clean_root() -> Path:
    """A new directory for the clean cache on local disk: XV4CFG.clean_cache_dir / run-<UTC stamp>-<pid> (outside
    Colab, where /content does not exist, the same name under the local temporary directory); never inside ROOT."""
    base = Path(XV4CFG.clean_cache_dir)
    if not IN_COLAB and not base.parent.exists():
        base = Path(tempfile.gettempdir()) / base.name
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    root = (base / f"run-{stamp}-{os.getpid()}").resolve()
    run_root = Path(ROOT).resolve()
    assert root != run_root and run_root not in root.parents, f"{root}: the clean cache must not be inside ROOT"
    assert not root.exists(), f"{root} exists already: the clean cache must start empty"
    return root


def x_v4_clean_jobs() -> Tuple[List[dict], List[str]]:
    """The reruns of the clean check, one job per section (II.6, II.8, optionally II.8g): runner, cache-key prefix,
    identity function, the enabled cells with the draws in scope, the cached reference rows with their (cell, draw)
    index and the recorded draw meta. Returns the jobs and the cells left out (with the reason)."""
    scope = str(XV4CFG.clean_scope)
    assert scope in ("all", "sample"), f"XV4CFG.clean_scope must be 'all' or 'sample', not {scope!r}"
    g = globals()
    v3_cells = list(g.get("X_V3_CELLS") or [])
    plan = [("II.6", [s for s in X_CELLS if EXPERIMENTS[f"xenv_{s['family']}"]], run_cell_draw, "", x_v4_draw_identity,
             "XRESULTS", "XDRAW_META", XV4_CLEAN_RESULT_KEYS, tuple(_XMETRIC_SOURCE), XRESULT_COLUMNS)]
    skipped = [f"II.6 {s['cell']} (family switch off)" for s in X_CELLS if not EXPERIMENTS[f"xenv_{s['family']}"]]
    if XV4CFG.clean_include_v3:
        plan.append(("II.8", v3_cells, run_v3_cell_draw, "v3/", x_v4_draw_identity, "XRESULTS_V3", "XDRAW_META_V3",
                     XV4_CLEAN_RESULT_KEYS, tuple(_XMETRIC_SOURCE), XRESULT_COLUMNS))
    else:
        skipped += [f"II.8 {s['cell']} (clean_include_v3 off)" for s in v3_cells]
    if XV4CFG.clean_include_v3_boot:
        boot_cells = list(g.get("XV3_BOOT_CELLS") or [])
        if not boot_cells:
            skipped.append("II.8g bootstrap (xenv_v3_bootstrap off or no bootstrap cell enabled)")
        plan.append(("II.8g", boot_cells, run_v3_boot_draw, "v3boot/", x_boot_identity, "XV3_BOOT_RESULTS",
                     "XV3_BOOT_META", XV4_CLEAN_BOOT_KEYS, XV4_CLEAN_BOOT_NUMERIC, XV3_BOOT_COLUMNS))

    def n_draws(spec: dict) -> int:
        n = x_v4_draws(spec)
        return n if scope == "all" else min(n, int(XV4CFG.clean_sample_draws))

    jobs = []
    for section, specs, runner, prefix, ident, ref_name, meta_name, keys, numeric, columns in plan:
        ref = g.get(ref_name)
        ref = pd.DataFrame(columns=list(columns)) if ref is None else ref
        assert set(columns) <= set(ref.columns), f"{ref_name} lacks columns {sorted(set(columns) - set(ref.columns))}"
        index = ({} if not len(ref) else ref.groupby([ref["cell"].astype(str),
                                                      pd.to_numeric(ref["draw"]).astype(np.int64)], sort=False).indices)
        meta = g.get(meta_name)
        jobs.append({"section": section, "runner": runner, "prefix": prefix, "identity": ident, "keys": tuple(keys),
                     "numeric": tuple(numeric), "columns": list(columns), "reference_name": ref_name,
                     "reference": ref, "reference_index": index, "meta": pd.DataFrame() if meta is None else meta,
                     "specs": {str(s["cell"]): s for s in specs},
                     # "draws" enters no identity (x_draw_identity drops it); the runner gets the original spec
                     "cells": [{**s, "draws": n_draws(s)} for s in specs]})
    return jobs, skipped


def x_v4_clean_estimate(jobs: Sequence[dict]) -> Tuple[Optional[float], int]:
    """Expected rerun seconds of the draws in scope, from the per-cell mean of the compute times recorded in the cached
    draw meta ('timing.total_s', kept from the session that computed each draw), and the draws without a record."""
    total, unknown, known = 0.0, 0, False
    for job in jobs:
        m = job["meta"]
        per = (pd.to_numeric(m["timing.total_s"], errors="coerce").groupby(m["cell"].astype(str)).mean()
               if len(m) and {"cell", "timing.total_s"} <= set(m.columns) else pd.Series(dtype=np.float64))
        for s in job["cells"]:
            t = float(per.get(str(s["cell"]), np.nan))
            if np.isfinite(t):
                total, known = total + t * int(s["draws"]), True
            else:
                unknown += int(s["draws"])
    return (total if known else None), unknown


def x_v4_clean_reference(job: dict, spec: dict, draw: int, main: XCache,
                         memory: bool = True) -> Tuple[Optional[pd.DataFrame], str]:
    """The cached table of one draw: its rows in the in-memory results (read back from the main cache in II.6a / II.8f
    / II.8g), else the main-cache entry under the identity of the regenerated draw; (None, 'missing') if neither.
    memory=False reads the main cache directly (the self-check)."""
    pos = job["reference_index"].get((str(spec["cell"]), int(draw))) if memory else None
    if pos is not None:
        return job["reference"].iloc[pos][job["columns"]].reset_index(drop=True), "memory"
    inst = x_v4_instance(spec, draw)
    hit = main.get(job["prefix"] + x_cache_key(spec["cell"], draw), job["identity"](spec, inst))
    del inst
    return (None, "missing") if hit is None else (hit[job["columns"]], "main_cache")


def _x_v4_clean_keyed(df: pd.DataFrame, keys: Sequence[str]) -> pd.DataFrame:
    """Copy with normalised key columns (integers for draw / replicate, text otherwise); keys must be unique."""
    out = df.copy()
    for k in keys:
        out[k] = pd.to_numeric(out[k]).astype(np.int64) if k in ("draw", "replicate") else out[k].astype(str)
    dup = out.duplicated(list(keys))
    assert not dup.any(), f"{int(dup.sum())} duplicated {tuple(keys)} rows"
    return out


def _x_v4_clean_text(s: pd.Series) -> np.ndarray:
    """Label values as text, missing values as '' (a CSV round trip turns empty cells into NaN)."""
    return s.astype(object).where(s.notna(), "").map(str).to_numpy()


def x_v4_clean_compare(new: Optional[pd.DataFrame], ref: Optional[pd.DataFrame], keys: Sequence[str],
                       numeric: Sequence[str], tol: float) -> Dict[str, Any]:
    """Row-by-row comparison of a rerun table with the cached table of the same draw (see markdown, step 4). A
    missing side (None: no cached table, or a rerun that raised) counts every row of the other side as missing."""
    keys, numeric = list(keys), list(numeric)
    out: Dict[str, Any] = {"rows_rerun": 0 if new is None else int(len(new)),
                           "rows_reference": 0 if ref is None else int(len(ref))}
    if new is None or ref is None:
        out.update({"rows_matched": 0, "missing_in_rerun": out["rows_reference"],
                    "missing_in_reference": out["rows_rerun"],
                    "status_mismatches": 0, **{f"max_abs_diff[{c}]": np.nan for c in numeric},
                    **{f"nan_mismatches[{c}]": 0 for c in numeric}, "rows_above_tol": 0, "rows_exact": 0,
                    "label_mismatches": 0, "label_mismatch_columns": ""})
        return out
    assert list(new.columns) == list(ref.columns), "the rerun and the cached table have different columns"
    labels = [c for c in new.columns if c not in keys and c not in numeric and c != "status"]
    m = _x_v4_clean_keyed(new, keys).merge(_x_v4_clean_keyed(ref, keys), on=keys, how="outer",
                                           suffixes=("_rerun", "_ref"), indicator=True, validate="one_to_one")
    b = m[(m["_merge"] == "both").to_numpy()]
    exact = _x_v4_clean_text(b["status_rerun"]) == _x_v4_clean_text(b["status_ref"])
    out.update({"rows_matched": int(len(b)), "missing_in_rerun": int((m["_merge"] == "right_only").sum()),
                "missing_in_reference": int((m["_merge"] == "left_only").sum()),
                "status_mismatches": int((~exact).sum())})
    above = np.zeros(len(b), dtype=bool)
    for c in numeric:
        x = pd.to_numeric(b[f"{c}_rerun"], errors="coerce").to_numpy(dtype=np.float64)
        y = pd.to_numeric(b[f"{c}_ref"], errors="coerce").to_numpy(dtype=np.float64)
        nx, ny = np.isnan(x), np.isnan(y)
        d = np.where(nx | ny, 0.0, np.abs(x - y))
        out[f"max_abs_diff[{c}]"] = float(d.max(initial=0.0))
        out[f"nan_mismatches[{c}]"] = int((nx != ny).sum())
        scale = np.maximum(1.0, np.where(ny, 0.0, np.abs(y)))          # tolerance absolute below 1, relative above
        above |= (d > tol * scale) | (nx != ny)
        exact &= (x == y) | (nx & ny)
    bad, n_bad = [], 0
    for c in labels:
        neq = _x_v4_clean_text(b[f"{c}_rerun"]) != _x_v4_clean_text(b[f"{c}_ref"])
        if neq.any():
            bad.append(c)
            n_bad += int(neq.sum())
        exact &= ~neq
    out.update({"rows_above_tol": int(above.sum()), "rows_exact": int(exact.sum()), "label_mismatches": n_bad,
                "label_mismatch_columns": ";".join(bad)})
    return out


def x_v4_clean_certified(df: pd.DataFrame, family: str) -> Dict[str, int]:
    """Certified pairs of one draw's table (x_v4_pair_claims, type 'all'): NHB (cert_nhb) and NHB-free (cert_free)
    counts over the default non-reference policies that are ok under both rules, on the true rows and on the benchmark.
    Bootstrap tables are scored per replicate: replicate -1 under the rule name, the replicates pooled under
    'boot_<name>'. Keys '<name>.policies', '<name>.cert_nhb', '<name>.cert_free'."""
    v_lo, v_hi = XV3_VRANGE[family]
    d = df[xs_as_bool(df["is_default"]).to_numpy()]
    if "is_reference" in d.columns:
        d = d[~xs_as_bool(d["is_reference"]).to_numpy()]
    d = d.assign(target=d["target"].astype(str), status=d["status"].astype(str))
    rep = (pd.to_numeric(d["replicate"]).astype(np.int64).to_numpy() if "replicate" in d.columns
           else np.full(len(d), XV3_BOOT_POINT, dtype=np.int64))
    out: Dict[str, int] = {}
    for r in np.unique(rep):
        dr = d[rep == r]
        have = set(dr["target"])
        for name, lo_t, up_t in XV4_CLEAN_RULES:
            if not {lo_t, up_t} <= have:
                continue
            key = name if int(r) == XV3_BOOT_POINT else f"boot_{name}"
            sub = dr[dr["target"].isin([lo_t, up_t]) & (dr["status"] == "ok")].set_index(["policy", "target"])
            J = sub["return"].astype(np.float64).unstack("target").reindex(columns=[lo_t, up_t]).dropna()
            p = sub["prob_any_violation"].astype(np.float64).unstack("target").reindex(index=J.index, columns=[up_t])
            claims = {"cert_nhb_claims": 0, "cert_free_claims": 0}
            if len(J):
                claims = next(c for c in x_v4_pair_claims(J[lo_t].to_numpy(np.float64), J[up_t].to_numpy(np.float64),
                                                          p[up_t].to_numpy(np.float64), v_hi - v_lo)
                              if c["type"] == "all")
            for k, v in (("policies", len(J)), ("cert_nhb", claims["cert_nhb_claims"]),
                         ("cert_free", claims["cert_free_claims"])):
                out[f"{key}.{k}"] = out.get(f"{key}.{k}", 0) + int(v)
    return out


def x_v4_clean_selfcheck(job: dict, main: XCache) -> Dict[str, Any]:
    """Draw 0 of the first cell of a section read both ways: the in-memory table and the main-cache entry under the
    identity of the regenerated draw. They are the same table read back, so they must be identical (reported). This
    also exercises the main-cache path of x_v4_clean_reference in every run mode."""
    spec = job["specs"][str(job["cells"][0]["cell"])]
    out: Dict[str, Any] = {"cell": str(spec["cell"]), "draw": 0}
    if int(job["cells"][0]["draws"]) < 1 or (str(spec["cell"]), 0) not in job["reference_index"]:
        return {**out, "status": "not checked (draw 0 has no in-memory table)", "match": None}
    mem, _ = x_v4_clean_reference(job, spec, 0, main)
    try:
        hit, _ = x_v4_clean_reference(job, spec, 0, main, memory=False)
    except MemoryError:
        raise
    except Exception as exc:          # the rerun of the same draw meets the same error and records it
        return {**out, "status": f"failed: {type(exc).__name__}: {str(exc)[:200]}", "match": False}
    if hit is None:
        return {**out, "status": "no main-cache entry under the regenerated identity", "match": False}
    cmp = x_v4_clean_compare(hit, mem, job["keys"], job["numeric"], 0.0)
    match = cmp["rows_exact"] == cmp["rows_matched"] == cmp["rows_rerun"] == cmp["rows_reference"]
    return {**out, "status": "checked", "match": bool(match),
            **{k: cmp[k] for k in ("rows_reference", "rows_matched", "rows_exact", "status_mismatches",
                                   "rows_above_tol", "label_mismatch_columns")}}


def x_v4_clean_draw(job: dict, spec: dict, draw: int, main: XCache) -> Tuple[pd.DataFrame, dict]:
    """Rerun one draw into the clean cache (the rebound global XCACHE) and compare it with its cached table.
    A rerun that raises is recorded as a failing draw without rerun rows (see markdown, step 3).
    Returns one comparison row and a meta for x_v4_run_draws."""
    cell, draw = str(spec["cell"]), int(draw)
    orig = job["specs"][cell]
    assert XCACHE is not main, "the clean rerun must run with XCACHE rebound to the clean cache"
    t0 = time.perf_counter()
    new, error = None, ""
    try:
        new, meta = job["runner"](orig, draw)
    except MemoryError:
        raise
    except Exception as exc:          # a clean-check outcome, not a crash: recorded, and the next draw runs
        error = f"{type(exc).__name__}: {str(exc)[:300]}"
        print(f"  WARNING: II.9g {job['section']} {cell} draw {draw}: the rerun raised {error}")
    if new is not None:
        assert meta["cache"] == "computed", \
            f"{cell} draw {draw}: {meta['cache']!r} although the clean cache started empty"
        assert list(new.columns) == job["columns"], f"{cell} draw {draw}: rerun columns differ"
    t1 = time.perf_counter()
    ref, source, ref_error = None, "missing", ""
    try:
        ref, source = x_v4_clean_reference(job, orig, draw, main)
    except MemoryError:
        raise
    except Exception as exc:          # the main-cache path rebuilds the instance; a failure means no cached table
        ref_error = f"{type(exc).__name__}: {str(exc)[:300]}"
        print(f"  WARNING: II.9g {job['section']} {cell} draw {draw}: the cached table could not be read: {ref_error}")
    t2 = time.perf_counter()
    cmp = x_v4_clean_compare(new, ref, job["keys"], job["numeric"], float(XV4CFG.clean_tol))
    cn = {} if new is None else x_v4_clean_certified(new, orig["family"])
    cr = {} if ref is None else x_v4_clean_certified(ref, orig["family"])
    cert = {**{f"{k}.rerun": v for k, v in cn.items()}, **{f"{k}.reference": v for k, v in cr.items()}}
    cdiff = np.nan
    if new is not None and ref is not None:
        cdiff = float(sum(abs(cn.get(k, 0) - cr.get(k, 0)) for k in set(cn) | set(cr) if not k.endswith(".policies")))
    reasons = [f"the rerun raised {error}"] if error else []
    if ref is None:
        reasons.append("no cached table" + (f" ({ref_error})" if ref_error else ""))
    elif new is not None:
        reasons += [f"{cmp[k]} {txt}" for k, txt in (("missing_in_rerun", "cached rows missing in the rerun"),
                                                     ("missing_in_reference", "rerun rows missing in the cache"),
                                                     ("status_mismatches", "status mismatches"),
                                                     ("rows_above_tol", "rows with |diff| > clean_tol or a NaN change"))
                    if cmp[k]]
        if cdiff != 0:
            reasons.append(f"certified counts differ by {cdiff:g}")
    passes = not reasons
    exact = passes and cmp["rows_exact"] == cmp["rows_matched"]
    row = {"section": job["section"], "cell": cell, "family": orig["family"], "draw": draw, "reference": source,
           "passes": passes, "exact": exact, "failure": "; ".join(reasons), "rerun_error": error,
           "reference_error": ref_error, **cmp,
           "certified_count_differences": cdiff, **cert, "rerun_s": t1 - t0, "reference_s": t2 - t1,
           "compare_s": time.perf_counter() - t2}
    del new, ref
    return pd.DataFrame([row]), {"cache": "computed", "section": job["section"], "cell": cell, "draw": draw,
                                 "reference": source, "passes": passes, "rerun_error": error,
                                 "timing": {"rerun_s": t1 - t0, "reference_s": t2 - t1, "compare_s": row["compare_s"],
                                            "total_s": time.perf_counter() - t0}}


def x_v4_clean_run(jobs: List[dict], root: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Rerun the draws of every job into the empty cache `root` and compare each with its cached table.

    The global XCACHE is rebound to XCache(root) inside try / finally and restored afterwards; the memo of reference
    optima is emptied for the run and restored. The clean cache is deleted afterwards (also after a failure) unless
    SEPSIS_KEEP_CLEAN_CACHE=1. Returns the per-draw comparison and a run report."""
    global XCACHE
    assert ALLOW_TRAINING, "the clean rerun computes every draw: it needs ALLOW_TRAINING = True"
    main, root = XCACHE, Path(root)
    assert not root.exists(), f"{root}: the clean cache must start empty"
    assert root.resolve() != Path(main.root).resolve() and Path(main.root).resolve() not in root.resolve().parents, \
        f"{root}: the clean cache must not be inside the main cache {main.root}"
    keep = os.environ.get(XV4_CLEAN_KEEP_ENV, "").strip() == "1"
    todo = [j for j in jobs if j["cells"]]
    expected, unknown = x_v4_clean_estimate(todo)
    print(f"II.9g clean rerun ({XV4CFG.clean_scope} scope): {sum(len(j['cells']) for j in todo)} cells, "
          f"{sum(int(s['draws']) for j in todo for s in j['cells'])} draws, into {root}")
    print(f"  expected time from the compute times recorded in the cached draw meta: {_x_fmt_s(expected)}"
          + (f" (+ {unknown} draws without a recorded time)" if unknown else ""))
    selfcheck = {j["section"]: x_v4_clean_selfcheck(j, main) for j in todo}     # reads only (main cache, memory)
    if any(v["match"] is False for v in selfcheck.values()):
        print("NOTE: II.9g in-memory and main-cache tables differ (reference self-check):",
              {k: v for k, v in selfcheck.items() if v["match"] is False})
    frames: List[pd.DataFrame] = []
    sections: Dict[str, Any] = {}
    ref_memo = dict(_XREF_MEMO)
    t_all = time.perf_counter()
    obs_s, exp_s = 0.0, 0.0
    clean: Optional[XCache] = None
    try:
        clean = XCache(root)
        XCACHE = clean
        _XREF_MEMO.clear()
        for ji, job in enumerate(todo, 1):
            fr, _, summ = x_v4_run_draws(job["cells"], lambda s, d, job=job: x_v4_clean_draw(job, s, d, main),
                                         f"II.9g {job['section']}")
            frames += fr
            sections[job["section"]] = summ
            done, _ = x_v4_clean_estimate([job])
            obs_s, exp_s = obs_s + float(summ["elapsed_s"]), exp_s + (done or 0.0)
            rest, _ = x_v4_clean_estimate(todo[ji:])
            eta = None if rest is None else rest * (obs_s / exp_s if exp_s > 0 else 1.0)
            print(f"II.9g {job['section']}: {summ['draws']} draws rerun and compared in "
                  f"{_x_fmt_s(summ['elapsed_s'])} | total elapsed {_x_fmt_s(time.perf_counter() - t_all)} | "
                  f"ETA remaining sections {_x_fmt_s(eta) if ji < len(todo) else '0 s'}")
    finally:
        XCACHE = main
        _XREF_MEMO.clear()
        _XREF_MEMO.update(ref_memo)
        written = 0 if clean is None else len(clean.written)
        if keep:
            print(f"II.9g: clean cache kept at {root} ({XV4_CLEAN_KEEP_ENV}=1)")
        else:
            shutil.rmtree(root, ignore_errors=True)
            try:
                root.parent.rmdir()              # the base directory too, if no other run is kept there
            except OSError:
                pass
    assert XCACHE is main
    draws = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return draws, {"cache_dir": str(root), "cache_kept": keep, "cache_deleted": not keep and not root.exists(),
                   "cache_entries_written": written, "expected_s": expected, "draws_without_recorded_time": unknown,
                   "reference_selfcheck": selfcheck, "sections": sections, "run_s": time.perf_counter() - t_all}


def x_v4_clean_cells(draws: pd.DataFrame) -> pd.DataFrame:
    """Per (section, cell) and pooled ('all'): draws, rows, max |diff| per numeric column, NaN changes, mismatches,
    rows above tolerance, certified counts of both tables and their differences, rerun time."""
    if draws.empty:
        return pd.DataFrame()
    sums = ["rows_rerun", "rows_reference", "rows_matched", "missing_in_rerun", "missing_in_reference",
            "status_mismatches", "rows_above_tol", "rows_exact", "label_mismatches",
            *[c for c in draws.columns if c.startswith("nan_mismatches[")], "rerun_s", "reference_s", "compare_s"]
    maxes = [c for c in draws.columns if c.startswith("max_abs_diff[")]
    certs = [c for c in draws.columns if c.endswith((".rerun", ".reference"))]
    rows = []
    for (sec, cell), g in [*draws.groupby(["section", "cell"], sort=False), (("all", "all"), draws)]:
        cd = pd.to_numeric(g["certified_count_differences"], errors="coerce")
        r = {"section": sec, "cell": cell, "draws": int(len(g)), "draws_passing": int(g["passes"].astype(bool).sum()),
             "draws_exact": int(g["exact"].astype(bool).sum()),
             "draws_without_reference": int((g["reference"] == "missing").sum()),
             "draws_from_main_cache": int((g["reference"] == "main_cache").sum()),
             "draws_rerun_failed": int((g["rerun_error"].fillna("").astype(str) != "").sum())}
        r.update({c: g[c].sum() for c in sums})
        r.update({c: (float(g[c].max()) if g[c].notna().any() else np.nan) for c in maxes})
        r["certified_count_differences"] = float(cd.sum(min_count=1))
        r["draws_with_certified_count_difference"] = int((cd > 0).sum())
        r.update({c: g[c].sum(min_count=1) for c in certs})
        r["label_mismatch_columns"] = ";".join(sorted({x for v in g["label_mismatch_columns"].fillna("")
                                                        for x in str(v).split(";") if x}))
        rows.append(r)
    return pd.DataFrame(rows)


def x_v4_clean_summary(draws: pd.DataFrame, cells: pd.DataFrame, jobs: Sequence[dict], skipped: Sequence[str],
                       run: Dict[str, Any], elapsed_s: float) -> Dict[str, Any]:
    """XV4_CLEAN: pass flag and criteria, bitwise flag, pooled checks, failing draws, driver summaries, timing."""
    tol = float(XV4CFG.clean_tol)
    planned = int(sum(int(s["draws"]) for j in jobs for s in j["cells"]))
    p = cells[cells["section"] == "all"].iloc[0]
    criteria = {"every_max_abs_diff_within_tol": bool(p["rows_above_tol"] == 0),
                "statuses_identical": bool(p["status_mismatches"] == 0),
                "certified_counts_identical": bool(p["certified_count_differences"] == 0),
                "no_missing_rows": bool(p["missing_in_rerun"] == 0 and p["missing_in_reference"] == 0
                                        and p["draws_without_reference"] == 0)}
    maxes = {c[len("max_abs_diff["):-1]: float(p[c]) for c in cells.columns
             if c.startswith("max_abs_diff[") and pd.notna(p[c])}
    checks = {k: (p[k].item() if hasattr(p[k], "item") else p[k]) for k in
              ("draws", "draws_passing", "draws_exact", "draws_without_reference", "draws_from_main_cache",
               "draws_rerun_failed", "rows_rerun", "rows_reference", "rows_matched", "missing_in_rerun",
               "missing_in_reference", "status_mismatches", "rows_above_tol", "rows_exact", "label_mismatches",
               "label_mismatch_columns", "certified_count_differences", "draws_with_certified_count_difference")}
    selfcheck = {k: v["match"] for k, v in run.get("reference_selfcheck", {}).items()}
    fail = draws.loc[~draws["passes"].astype(bool), ["section", "cell", "draw", "failure"]]
    return {"status": "run", "code_hash": XV4_CLEAN_CODE_HASH,
            "config": {k: v for k, v in asdict(XV4CFG).items() if k.startswith("clean_") or k == "max_draws"},
            "tolerance": tol, "pass": bool(all(criteria.values()) and len(draws) > 0),
            "bitwise_identical": bool(criteria["no_missing_rows"] and p["draws_exact"] == p["draws"]),
            "criteria": criteria, "checks": {**checks, "draws_planned": planned, "max_abs_diff": maxes,
                                             "reference_selfcheck_match": selfcheck},
            "cells": {f"{r['section']}|{r['cell']}": bool(r["draws_passing"] == r["draws"])
                      for _, r in cells[cells["section"] != "all"].iterrows()},
            "failing_draws": fail.head(50).to_dict("records"), "failing_draws_total": int(len(fail)),
            "cells_skipped": list(skipped), **run, "elapsed_s": float(elapsed_s),
            "draws_complete": bool(len(draws) == planned and planned > 0),
            "files": {"check": "xenv/v4_clean_check.json", "diffs": "xenv/v4_clean_diffs.csv"}}


XV4_CLEAN_CODE = [x_v4_clean_root, x_v4_clean_jobs, x_v4_clean_estimate, x_v4_clean_reference, _x_v4_clean_keyed,
                  _x_v4_clean_text, x_v4_clean_compare, x_v4_clean_certified, x_v4_clean_selfcheck, x_v4_clean_draw,
                  x_v4_clean_run, x_v4_clean_cells, x_v4_clean_summary]
XV4_CLEAN_CODE_HASH = obj_hash({"common": XV4_COMMON_CODE_HASH, "code": code_hash(*XV4_CLEAN_CODE),
                                "constants": {"result_keys": XV4_CLEAN_RESULT_KEYS, "boot_keys": XV4_CLEAN_BOOT_KEYS,
                                              "boot_numeric": XV4_CLEAN_BOOT_NUMERIC, "rules": XV4_CLEAN_RULES,
                                              "v_range": XV3_VRANGE, "boot_point": XV3_BOOT_POINT,
                                              "keep_env": XV4_CLEAN_KEEP_ENV, "result_columns": XRESULT_COLUMNS,
                                              "metrics": list(_XMETRIC_SOURCE), "boot_columns": XV3_BOOT_COLUMNS}})

XV4_CLEAN_DRAWS = pd.DataFrame()
XV4_CLEAN_CELLS = pd.DataFrame()
XV4_CLEAN: Dict[str, Any] = {"status": "not run (xenv_v4_clean_check off)", "code_hash": XV4_CLEAN_CODE_HASH,
                             "pass": None, "bitwise_identical": None, "criteria": {}, "checks": {}, "elapsed_s": 0.0,
                             "draws_complete": None}
if EXPERIMENTS["xenv_v4_clean_check"] and not ALLOW_TRAINING:
    XV4_CLEAN["status"] = "not run (ALLOW_TRAINING is False: the clean rerun computes every draw)"
    print("NOTE: II.9g clean rerun skipped: ALLOW_TRAINING is False (artifact-replay mode).")
elif EXPERIMENTS["xenv_v4_clean_check"]:
    _t0 = time.perf_counter()
    _jobs, _skipped = x_v4_clean_jobs()
    if _skipped:
        print("NOTE: II.9g leaves out:", _skipped)
    if sum(int(s["draws"]) for j in _jobs for s in j["cells"]) > 0:
        XV4_CLEAN_DRAWS, _run = x_v4_clean_run(_jobs, x_v4_clean_root())
        XV4_CLEAN_CELLS = x_v4_clean_cells(XV4_CLEAN_DRAWS)
        XV4_CLEAN = x_v4_clean_summary(XV4_CLEAN_DRAWS, XV4_CLEAN_CELLS, _jobs, _skipped, _run,
                                       time.perf_counter() - _t0)
        XV4_CLEAN_CELLS.to_csv(XENV_TABLE_DIR / "v4_clean_diffs.csv", index=False)
        _cols = [c for c in ("section", "cell", "draws", "draws_passing", "draws_exact", "draws_without_reference",
                             "draws_rerun_failed", "rows_rerun", "missing_in_rerun", "missing_in_reference",
                             "status_mismatches",
                             "rows_above_tol", "max_abs_diff[return]", "max_abs_diff[avg_episode_length]",
                             "certified_count_differences", "rerun_s") if c in XV4_CLEAN_CELLS.columns]
        display(XV4_CLEAN_CELLS[_cols])
        del _run, _cols
    else:
        XV4_CLEAN.update(status="not run (no Part II cell is enabled for the clean rerun)", cells_skipped=_skipped)
    del _jobs, _skipped, _t0
    gc.collect()
save_json(XV4_CLEAN, XENV_TABLE_DIR / "v4_clean_check.json")
print(json.dumps({k: v for k, v in XV4_CLEAN.items() if k not in ("failing_draws", "sections", "cells", "config")},
                 indent=1, default=_json_default))
if XV4_CLEAN.get("pass") is not None:
    print(f"II.9g pass criterion: every max |diff| <= clean_tol = {XV4CFG.clean_tol:g} with no NaN change, identical "
          f"statuses, identical certified counts (NHB and NHB-free; true rows and bench) and no row missing on either "
          f"side (every draw reran and has a cached table) -> {'PASS' if XV4_CLEAN['pass'] else 'FAIL'} "
          f"{XV4_CLEAN['criteria']}; bitwise identical (reported "
          f"separately): {XV4_CLEAN['bitwise_identical']}")
    if not XV4_CLEAN["pass"]:
        print(f"  {XV4_CLEAN['failing_draws_total']} failing draws, e.g.", XV4_CLEAN["failing_draws"][:5])

### II.9h Overview of the further experiments: status, headline numbers, checks and figures

This cell collects the outputs of II.9b–II.9g. It computes nothing new about policies. For every subsection whose
switch is on it reads the summary (`XV4_ROBUST_SUMMARY`, `XV4_CS_SUMMARY`, `XV4_BOOT_SUMMARY`, `XV4_SENS_SUMMARY`,
`XV4_LIFT_SUMMARY`, `XV4_CLEAN`) from memory, or from its json file in `xenv/` when the global does not exist (the
cell was not executed in this session). A subsection whose switch is off is not read at all, so a summary left on
disk by an earlier session is never mixed in. The tables it needs (`XV4_CS_TABLE`, `XV4_CS_POLICIES`,
`XV4_BOOT_COVERAGE`, `XV4_BOOT_IV`, `XV4_SENS_TABLE`) are read only for a subsection that ran (switch on, status
`run`), and from the same place as its summary: from memory when the summary is in memory, else from `xenv/`. A
subsection that is switched off, did not run, or has no (readable) summary is named in the status
(`run | missing: … | switched off: …`), and its headline block and figure say why they are empty. Nothing here
fails because a subsection is missing; a file cut by a disconnect is reported as unreadable. `draws_complete` is
true when every required subsection ran with all its draws; a subsection is required when its switch is on, except
II.9g in artifact-replay mode (`ALLOW_TRAINING = False`), as in the Section 34 gate.

**Headline numbers** (`xenv/v4_overview.json`; the pooled units of each subsection):
* II.9b: NHB and NHB-free certified share and error rate per pool and target set, with the draw-clustered intervals
  of the error rates, also on the completion-sensitive pairs (AU + UU); the median breakdown fraction κ* of the
  NHB-only claims; the II.8g intervals over all default policies;
* II.9c: per test and scale, and for the point-benchmark certificates of the same draws: claims, certified share,
  error rate (all and completion-sensitive pairs), draws with a wrong claim or an uncovered policy, coverage and mean
  width;
* II.9d: per variant (point, II.8g B = 10, fixed and re-estimated B = 200) and interval type: coverage over all
  default policies, over p > 0 and over p = 0, certified share and error rate, family-wise draw shares, and the error
  rate of the midpoint margin rule with as many claims (matched-coverage baseline);
* II.9e: the differences new − base of every sensitivity cell and pool (coverage, certificates, Kendall τ, support
  of the physician), with the kind, the pairing and the draws of the new cell, and the new cells that have no
  reference cell;
* II.9f: the range of J_true over all lifts (width, share of benchmark intervals that contain it, benchmark
  certificates that hold under every lift) and the error rates under each lift;
* II.9g: the pass flag of the clean rerun, its four criteria and bitwise identity.

**Checks** (`xenv/v4_stats_checks.json`). For every subsection: switch, status, whether its draws are complete, and
its invariant flag. The flags are the subsections' own: II.9b theorem and identity counts; II.9c self-test of the
inner problem and bound order; II.9d per-replicate invariants (Theorem 1(iii)), unit weights and the true-rows
bracket; II.9e self-test and theorem counts; II.9f its pass flag and the slack maxima of its asserted invariants
within their tolerances (checked again here). II.9g has no invariant; its pass flag is reported, and the Section 34
gate uses it. Two orderings are recounted here from the stored tables, since both hold by construction:

* every confidence-set row: finite bounds, L ≤ U_free and U_nhb = min(U_free, U_proj) ≤ U_free;
* every bootstrap interval (Theorem 1(iii) on every replicate, and monotone quantiles): lo ≤ up for the NHB
  interval; up_free ≥ up_nhb; lo_free = lo_nhb; the min / max interval contains the quantile interval; the ends are
  defined together (the point interval has no min / max ends). The tolerance is 4·1e-9: the per-replicate slacks
  (each ≤ 1e-9) add up in up_free ≥ up_nhb.

A non-zero count (a table without its columns counts too), or a subsection flag that is false, sets `passes` to
false. The cell then raises after writing its outputs, as II.8i does. Coverage and error rates are outcomes:
reported, never checked.

**Figures** (style of `fig_v3_bootstrap`: horizontal bars per cell; a cross at 0 marks a value that is undefined or
not available, e.g. an error rate without any certified pair, or a cell without II.8g draws):

* `fig_v4_confidence` (rows: NHB-free and NHB certificates): for every II.9c cell, the certified share and the
  share of certified pairs that are wrong. Bars show the paired confidence-set test (`paired_free` / `paired_nhb`)
  at every scale (scale 1 valid, smaller scales heuristic) beside the point-benchmark certificate (`point_free` /
  `point_nhb`) of the same policies;
* `fig_v4_bootstrap` (rows: NHB interval, NHB-free interval): for every II.9d cell, the coverage of J_true over all
  default policies, the certified share and the certification error of the point interval, II.8g (B = 10, fixed
  mask), and the fixed and re-estimated bootstraps (B = 200).

| Output | Content |
|---|---|
| `XV4_STATS` | overview, checks, per-subsection status, the small tables read, figure status, the current outputs |
| `xenv/v4_overview.json` | status (what is missing), per-subsection status, headline numbers, the paths of every II.9 output with `exists` and `current` (written by a subsection that ran, or by this cell) |
| `xenv/v4_stats_checks.json` | per-subsection status, completion and invariant flags, recounted orderings, inputs, figure status, `passes` |
| `figures/xenv/fig_v4_confidence.*`, `figures/xenv/fig_v4_bootstrap.*` | the two figures above |

**Expected cost on Colab** (estimate, not a measurement): under a minute. The summaries and tables are small.
Reading `v4_cs_policies.csv.gz` and `v4_boot_intervals.csv.gz` (a few 10 000 rows each) from disk, when they are
not in memory, adds a few seconds.

In [ ]:
# ============================================================
# II.9h Second-revision overview, checks and figures
# ============================================================
EXECLOG.mark("II.9h Second-revision experiments: overview, checks and figures")

# subsection -> (switch, summary global, summary file in xenv/)
XV4H_SUBSECTIONS = {"II.9b": ("xenv_v4_revision_robust", "XV4_ROBUST_SUMMARY", "v4_robust_summary.json"),
                    "II.9c": ("xenv_v4_confidence", "XV4_CS_SUMMARY", "v4_cs_summary.json"),
                    "II.9d": ("xenv_v4_bootstrap", "XV4_BOOT_SUMMARY", "v4_boot_summary.json"),
                    "II.9e": ("xenv_v4_sensitivity", "XV4_SENS_SUMMARY", "v4_sens_summary.json"),
                    "II.9f": ("xenv_v4_lift", "XV4_LIFT_SUMMARY", "v4_lift_summary.json"),
                    "II.9g": ("xenv_v4_clean_check", "XV4_CLEAN", "v4_clean_check.json")}
assert {v[0] for v in XV4H_SUBSECTIONS.values()} == set(XV4_SWITCHES), "II.9h must cover every II.9 switch"
# tables read for the checks and figures: key -> (subsection, global, file in xenv/)
XV4H_TABLES = {"cs_table": ("II.9c", "XV4_CS_TABLE", "v4_cs_summary.csv"),
               "cs_policies": ("II.9c", "XV4_CS_POLICIES", "v4_cs_policies.csv.gz"),
               "boot_coverage": ("II.9d", "XV4_BOOT_COVERAGE", "v4_boot_coverage.csv"),
               "boot_intervals": ("II.9d", "XV4_BOOT_IV", "v4_boot_intervals.csv.gz"),
               "sens_table": ("II.9e", "XV4_SENS_TABLE", "v4_sens_summary.csv")}
XV4H_KEEP_TABLES = ("cs_table", "boot_coverage", "sens_table")     # kept in XV4_STATS (the others are large)
XV4H_RECOUNT_TABLES = {"II.9c": "cs_policies", "II.9d": "boot_intervals"}   # input of the recounted orderings
# Every II.9 output, relative to tables/ ('xenv/...') or to the figures ('figures/xenv/...'), by subsection.
XV4H_OUTPUTS = {
    "II.9a": ("xenv/v4_config.json",),
    "II.9b": ("xenv/v4_robust_pairs.csv.gz", "xenv/v4_robust_summary.csv", "xenv/v4_robust_gapbins.csv",
              "xenv/v4_robust_kappa.csv", "xenv/v4_v3boot_allpolicy.csv", "xenv/v4_robust_summary.json"),
    "II.9c": ("xenv/v4_cs_policies.csv.gz", "xenv/v4_cs_draws.csv", "xenv/v4_cs_summary.csv",
              "xenv/v4_cs_summary.json"),
    "II.9d": ("xenv/v4_boot_intervals.csv.gz", "xenv/v4_boot_draws.csv", "xenv/v4_boot_coverage.csv",
              "xenv/v4_boot_meta.csv", "xenv/v4_boot_summary.json"),
    "II.9e": ("xenv/v4_sens_results_all.csv.gz", "xenv/v4_sens_draw_meta.csv", "xenv/v4_sens_summary.csv",
              "xenv/v4_sens_certified_draws.csv", "xenv/v4_sens_rank_fidelity.csv",
              "xenv/v4_sens_rank_fidelity_draws.csv", "xenv/v4_sens_summary.json"),
    "II.9f": ("xenv/v4_lift_results.csv.gz", "xenv/v4_lift_summary.csv", "xenv/v4_lift_summary.json"),
    "II.9g": ("xenv/v4_clean_check.json", "xenv/v4_clean_diffs.csv"),
    "II.9h": ("xenv/v4_overview.json", "xenv/v4_stats_checks.json", "figures/xenv/fig_v4_confidence.png",
              "figures/xenv/fig_v4_confidence.pdf", "figures/xenv/fig_v4_bootstrap.png",
              "figures/xenv/fig_v4_bootstrap.pdf")}
# Figure rows: (label, confidence-set test, point-benchmark test) and (interval type, label); bootstrap variants.
XV4H_CS_FIG = (("NHB-free certificate", "paired_free", "point_free"), ("NHB certificate", "paired_nhb", "point_nhb"))
XV4H_BOOT_FIG_INTERVALS = (("nhb", "NHB interval"), ("free", "NHB-free interval"))
XV4H_BOOT_FIG_VARIANTS = ("point", "v3boot", "fixed", "reestimated")
# Headline columns per subsection.
XV4H_ROBUST_KEYS = ("draws", "pairs", "cert_nhb_share", "cert_nhb_error_rate", "cert_nhb_error_rate_ci_low",
                    "cert_nhb_error_rate_ci_high", "cert_free_share", "cert_free_error_rate",
                    "cert_free_error_rate_ci_low", "cert_free_error_rate_ci_high", "cert_nhb_cs_share",
                    "cert_nhb_cs_error_rate", "cert_free_cs_share", "cert_free_cs_error_rate")
XV4H_CS_KEYS = ("draws", "claims", "share", "errors", "error_rate", "share_sens", "error_rate_sens",
                "draws_any_wrong", "draws_any_uncovered", "coverage", "mean_width")
XV4H_BOOT_KEYS = ("draws", "coverage_all", "coverage_ppos", "coverage_pzero", "mean_width", "certified_share",
                  "error_rate", "certified_share_cs", "error_rate_cs", "draws_any_wrong_share",
                  "draws_all_covered_share", "margin_mid_error_rate")
XV4H_SENS_KEYS = ("bench_coverage_violating", "sem_upper_coverage_violating", "cc_bench_cert_nhb_share",
                  "cc_bench_cert_nhb_error_rate", "cc_bench_cert_free_share", "cc_bench_cert_free_error_rate",
                  "cc_true_rows_cert_nhb_share", "cc_true_rows_cert_nhb_error_rate", "cc_true_rows_cert_free_share",
                  "cc_true_rows_cert_free_error_rate", "tau_mean_mean", "tau_terminate_mean", "tau_diff_mean",
                  "tau_mean_delta", "tau_mean_delta_ci_low", "tau_mean_delta_ci_high", "tau_terminate_delta",
                  "tau_terminate_delta_ci_low", "tau_terminate_delta_ci_high", "tau_diff_delta",
                  "tau_diff_delta_ci_low", "tau_diff_delta_ci_high", "constr_mean_admissible_actions",
                  "constr_single_admissible_fraction")
XV4H_CLEAN_CHECKS = ("draws", "draws_planned", "draws_passing", "draws_exact", "draws_without_reference",
                     "rows_matched", "missing_in_rerun", "missing_in_reference", "status_mismatches", "rows_above_tol",
                     "certified_count_differences", "max_abs_diff")
# II.9f invariant slacks (x_v4_lift_checks 'max_<key>'): value tolerance, and the range tolerance of its 'tolerances'.
XV4H_LIFT_SLACKS = ("bench_rule_p_difference", "bench_negative_width", "bench_width_minus_bound",
                    "target_p_difference", "lower_bound_shortfall", "rows_negative_width", "rows_width_minus_bound")
XV4H_LIFT_RANGE_SLACKS = ("lift_below_worst", "lift_above_best", "range_negative_width")
# Tolerance of the recounted II.9d orderings: each replicate satisfies Theorem 1(iii) and the shared p to XV4_TOL
# (asserted in II.8g / II.9d), and up_free >= up_nhb composes these slacks with v_range <= 2.
XV4H_ORDER_TOL = 4 * XV4_TOL


def _xv4h_path(rel: str, table_dir: Path, fig_dir: Path) -> Path:
    """Absolute path of an output listed relative to tables/ ('xenv/...') or to the figures ('figures/xenv/...')."""
    if rel.startswith("figures/xenv/"):
        return Path(fig_dir) / rel[len("figures/xenv/"):]
    assert rel.startswith("xenv/"), rel
    return Path(table_dir) / rel[len("xenv/"):]


def xv4h_summary(name: str, table_dir: Path) -> Tuple[Dict[str, Any], str]:
    """Summary of one subsection: the global when it exists, else its json file in table_dir; ({}, 'missing') when
    neither, and a status that names the file when it cannot be read (e.g. a write cut by a disconnect)."""
    _, glob, fname = XV4H_SUBSECTIONS[name]
    s = globals().get(glob)
    if isinstance(s, dict) and s:
        return s, "memory"
    p = Path(table_dir) / fname
    if not p.exists():
        return {}, "missing"
    try:
        s = json.loads(p.read_text(encoding="utf-8"))
    except XCACHE_READ_ERRORS as exc:
        return {"status": f"missing: xenv/{fname} unreadable ({type(exc).__name__})"}, f"xenv/{fname} (unreadable)"
    if not isinstance(s, dict):
        return {"status": f"missing: xenv/{fname} is not a summary"}, f"xenv/{fname} (not a summary)"
    return s, f"xenv/{fname}"


def xv4h_table(global_name: str, file_name: str, ran: bool, in_memory: bool, table_dir: Path
               ) -> Tuple[pd.DataFrame, str]:
    """A table of a subsection that ran, from the same place as its summary: the global when the summary is in memory
    (never a file of another session beside a summary of this one), else its file in table_dir (exact float round
    trip). An empty frame when the subsection did not run or the table is missing or unreadable."""
    if not ran:
        return pd.DataFrame(), "not used (subsection not run)"
    if in_memory:
        df = globals().get(global_name)
        if isinstance(df, pd.DataFrame):
            return df, (global_name if len(df) else f"{global_name} (empty)")
        return pd.DataFrame(), f"missing: {global_name} is not in memory (its cell did not run in this session)"
    p = Path(table_dir) / file_name
    if not p.exists():
        return pd.DataFrame(), "missing"
    try:
        return pd.read_csv(p, float_precision="round_trip", low_memory=False), f"xenv/{file_name}"
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), f"xenv/{file_name} (empty)"
    except XCACHE_READ_ERRORS as exc:                   # e.g. a gzip cut by a disconnect
        return pd.DataFrame(), f"xenv/{file_name} unreadable ({type(exc).__name__})"


def _xv4h_all(flags: Sequence[Optional[bool]]) -> Optional[bool]:
    """Combined flag: None if no flag is set, else whether every set flag is true."""
    f = [bool(x) for x in flags if x is not None]
    return all(f) if f else None


def _xv4h_num(x: Any) -> Optional[float]:
    """A finite float, or None (missing, null in json, NaN)."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) else None


# ---------- invariants (recounted orderings and the subsections' own flags) ----------
def xv4h_cs_order_counts(pol: pd.DataFrame) -> Dict[str, int]:
    """Orderings of the confidence-set rows of II.9c (all must be 0): finite bounds, L <= U_free, U_nhb <= U_free,
    U_nhb = min(U_free, U_proj); tolerance XV4_TOL. A table without these columns counts as one failure."""
    if pol is None or not len(pol):
        return {}
    cols = ("L", "U_free", "U_proj", "U_nhb")
    if not set(cols) <= set(pol.columns):
        return {"cs_table_missing_columns": len(set(cols) - set(pol.columns))}
    L, Uf, Up, Un = (pd.to_numeric(pol[c], errors="coerce").to_numpy(dtype=float) for c in cols)
    with np.errstate(invalid="ignore"):
        return {"cs_nonfinite_bounds": int((~np.isfinite(np.c_[L, Uf, Up, Un])).any(axis=1).sum()),
                "cs_lower_above_free_upper": int((L > Uf + XV4_TOL).sum()),
                "cs_nhb_above_free_upper": int((Un > Uf + XV4_TOL).sum()),
                "cs_nhb_not_min_of_free_and_proj": int((np.abs(Un - np.minimum(Uf, Up)) > XV4_TOL).sum())}


def xv4h_boot_order_counts(iv: pd.DataFrame) -> Dict[str, int]:
    """Orderings of the II.9d interval table (all must be 0), tolerance XV4H_ORDER_TOL: lo <= up (NHB),
    up_free >= up_nhb, lo_free = lo_nhb, the min / max interval containing the quantile interval, and the ends
    defined together (min / max ends too, except on the point rows, which have none). A table without these columns
    counts as one failure."""
    if iv is None or not len(iv):
        return {}
    cols = ("boot_variant", "lo_nhb", "up_nhb", "lo_free", "up_free", "lo_minmax", "up_minmax")
    if not set(cols) <= set(iv.columns):
        return {"boot_table_missing_columns": len(set(cols) - set(iv.columns))}
    g = {c: pd.to_numeric(iv[c], errors="coerce").to_numpy(dtype=float) for c in cols[1:]}
    nan = {c: np.isnan(v) for c, v in g.items()}
    boot = iv["boot_variant"].astype(str).to_numpy() != "point"
    tol = XV4H_ORDER_TOL
    with np.errstate(invalid="ignore"):
        return {"boot_nhb_lower_above_upper": int((g["lo_nhb"] > g["up_nhb"] + tol).sum()),
                "boot_free_upper_below_nhb_upper": int((g["up_free"] < g["up_nhb"] - tol).sum()),
                "boot_free_lower_differs": int((np.abs(g["lo_free"] - g["lo_nhb"]) > tol).sum()),
                "boot_minmax_inside_quantile": int(((g["lo_minmax"] > g["lo_nhb"] + tol)
                                                    | (g["up_minmax"] < g["up_nhb"] - tol)).sum()),
                "boot_ends_defined_apart": int(((nan["lo_nhb"] != nan["up_nhb"]) | (nan["lo_free"] != nan["lo_nhb"])
                                                | (nan["up_free"] != nan["up_nhb"])
                                                | (boot & ((nan["lo_minmax"] != nan["lo_nhb"])
                                                           | (nan["up_minmax"] != nan["up_nhb"])))).sum())}


def xv4h_invariants(name: str, s: Dict[str, Any], order_counts: Dict[str, int]
                    ) -> Tuple[Optional[bool], Dict[str, Any]]:
    """Invariant flag of one subsection that ran (None when it reports none) and its evidence: the subsection's own
    flags and the orderings recounted here ('cs_*' for II.9c, 'boot_*' for II.9d). Coverage and error rates never
    enter."""
    chk = s.get("checks") or {}
    if name == "II.9b":
        return chk.get("passes"), {"theorem_counts": chk.get("theorem_counts"),
                                   "identity_counts": chk.get("identity_counts")}
    if name in ("II.9c", "II.9d"):
        own = {k: v for k, v in order_counts.items() if k.startswith("cs_" if name == "II.9c" else "boot_")}
        st = s.get("selftest") if name == "II.9c" else None
        st_ok = st.get("passes") if isinstance(st, dict) else None
        flag = _xv4h_all([chk.get("passes"), st_ok, (all(v == 0 for v in own.values()) if own else None)])
        return flag, {"checks": {k: v for k, v in chk.items() if not isinstance(v, (dict, list))},
                      "selftest_passes": st_ok, "recounted": own}
    if name == "II.9e":
        return chk.get("passes"), {"theorem_counts": chk.get("theorem_counts"),
                                   "selftest_passes": (chk.get("selftest") or {}).get("passes"),
                                   "draw_checks": chk.get("draw_checks")}
    if name == "II.9f":
        tol_b = _xv4h_num((chk.get("tolerances") or {}).get("bounds")) or XV4_TOL
        slack = {k: _xv4h_num(chk.get(f"max_{k}")) for k in (*XV4H_LIFT_SLACKS, *XV4H_LIFT_RANGE_SLACKS)}
        bad = [k for k, v in slack.items() if v is not None and v > (tol_b if k in XV4H_LIFT_RANGE_SLACKS else XV4_TOL)]
        have = any(v is not None for v in slack.values())
        return _xv4h_all([chk.get("passes"), (not bad) if have else None]), {
            "passes": chk.get("passes"), "max_slacks": slack, "above_tolerance": bad, "range_tolerance": tol_b}
    assert name == "II.9g", name
    return None, {"clean_pass": s.get("pass"), "note": "no invariant: the pass flag is reported and gated separately"}


# ---------- headline numbers ----------
def xv4h_headline_robust(s: Dict[str, Any]) -> Dict[str, Any]:
    """II.9b: certificates per pool and target set, the median kappa* of the NHB-only claims, and the II.8g intervals
    over all default policies (pooled units)."""
    hd, kp = s.get("headline") or {}, s.get("kappa") or {}
    return {"pools": {u: {tf: {k: r.get(k) for k in XV4H_ROBUST_KEYS if k in r} for tf, r in g.items()}
                      for u, g in hd.items()},
            "kappa_star_median_nhb_only": {
                u: {tf: {"nhb_only_claims": r.get("nhb_only_claims"),
                         "kappa_star_q50": (r.get("kappa_star_quantiles") or {}).get("kappa_star_q50")}
                    for tf, r in g.items()} for u, g in kp.items()},
            "v3boot_allpolicy": s.get("v3boot") or {}}


def xv4h_headline_cs(table: pd.DataFrame, s: Dict[str, Any]) -> Dict[str, Any]:
    """II.9c: per pooled unit (the cell itself without a pool) x test x scale ('point' for the point-benchmark
    certificates): claims, certified share, error rate (all and completion-sensitive pairs), wrong-claim and
    uncovered-policy draws, coverage and mean width."""
    recs = table.to_dict("records") if len(table) else list(s.get("headline") or [])
    out: Dict[str, Any] = {"config": s.get("config") or {}}
    units = list(dict.fromkeys(str(r["unit"]) for r in recs))
    pools = [u for u in units if u.startswith("pooled:")] or units
    for r in recs:
        if str(r["unit"]) not in pools:
            continue
        sc = _xv4h_num(r.get("scale"))
        lab = "point" if sc is None else f"scale {sc:g}"
        out.setdefault(str(r["unit"]), {})[f"{r['test']}|{lab}"] = {k: r.get(k) for k in XV4H_CS_KEYS if k in r}
    return out


def xv4h_headline_boot(cov: pd.DataFrame, s: Dict[str, Any]) -> Dict[str, Any]:
    """II.9d: per pooled unit x variant x interval type: coverage over all / p > 0 / p = 0 default policies, width,
    certificates, family-wise draw shares and the matched midpoint margin; B of both bootstraps and the failed
    re-estimated replicates."""
    v3 = globals().get("XV3BOOT")
    out: Dict[str, Any] = {"replicates": s.get("replicates"), "replicates_v3boot": getattr(v3, "replicates", None),
                           "quantiles": s.get("quantiles"),
                           "failed_replicates": {k: (s.get("failed_replicates") or {}).get(k)
                                                 for k in ("total", "replicates", "share", "draws_with_failure")}}
    if not len(cov):
        out["summary_headline"] = s.get("headline") or {}
        return out
    units = [str(u) for u in pd.unique(cov["cell"])]
    pools = [u for u in units if u.startswith("pooled:")] or units
    for r in cov[cov["cell"].astype(str).isin(pools)].to_dict("records"):
        out.setdefault(str(r["cell"]), {})[f"{r['boot_variant']}|{r['interval']}"] = {
            k: r[k] for k in XV4H_BOOT_KEYS if k in r}
    return out


def xv4h_headline_sens(table: pd.DataFrame, s: Dict[str, Any]) -> Dict[str, Any]:
    """II.9e: the differences new - base per sensitivity cell and per pool (rows 'new - base' of the II.9e table, with
    the kind, the pairing and the draws of the new side; the summary's per-cell deltas when the table is not
    available), and the new cells without a reference cell (no difference)."""
    if not len(table) or not {"row_type", "side", "cell"} <= set(table.columns):
        return {"cells": dict(s.get("deltas") or {}), "pools": {}, "without_reference": None}
    d = table[table["side"] == "new - base"]
    new = table[table["side"] == "new"]
    n_new = ({(str(r["row_type"]), str(r["cell"])): (None if _xv4h_num(r["n_draws"]) is None else int(r["n_draws"]))
              for r in new.to_dict("records")} if "n_draws" in new.columns else {})
    pick = {rt: {str(r["cell"]): {"v4_kind": r.get("v4_kind"), "reference": r.get("reference"),
                                  "paired": r.get("paired"), "n_draws": n_new.get((rt, str(r["cell"]))),
                                  **{k: r[k] for k in XV4H_SENS_KEYS if k in r and pd.notna(r[k])}}
                 for r in d[d["row_type"] == rt].to_dict("records")} for rt in ("cell", "pool")}
    alone = [str(c) for c in new.loc[new["row_type"] == "cell", "cell"] if str(c) not in pick["cell"]]
    return {"cells": pick["cell"], "pools": pick["pool"], "without_reference": alone}


def xv4h_headline_lift(s: Dict[str, Any]) -> Dict[str, Any]:
    """II.9f: the headline of its widest pool per lift, including the 'range' row (bounds over all lifts)."""
    return {"lift_kinds": s.get("lift_kinds"), "lift_bounds": s.get("lift_bounds"), **(s.get("headline") or {})}


def xv4h_headline_clean(s: Dict[str, Any]) -> Dict[str, Any]:
    """II.9g: pass flag, criteria, bitwise identity and the pooled comparison counts."""
    chk = s.get("checks") or {}
    return {**{k: s.get(k) for k in ("pass", "bitwise_identical", "criteria", "tolerance", "failing_draws_total")},
            "scope": (s.get("config") or {}).get("clean_scope"), **{k: chk.get(k) for k in XV4H_CLEAN_CHECKS}}


# ---------- figures (style of xs_fig_v3_bootstrap) ----------
def _xv4h_bars(ax, n_rows: int, series: Sequence[Tuple[str, np.ndarray, str]]) -> bool:
    """Grouped horizontal bars in percent: one group per row, one bar per series (label, values in [0, 1], colour).
    An undefined value (NaN) gets a muted cross just right of the axis instead of a bar. Returns whether a value was
    undefined."""
    y = np.arange(n_rows)
    k = max(len(series), 1)
    h = 0.8 / k
    undefined = False
    for i, (lab, v, col) in enumerate(series):
        v = 100.0 * np.asarray(v, dtype=float)
        yi = y - 0.4 + h * (i + 0.5)
        ax.barh(yi, np.nan_to_num(v), height=0.9 * h, color=col, label=lab)
        bad = ~np.isfinite(v)
        if bad.any():
            undefined = True
            ax.scatter(np.full(int(bad.sum()), 0.015), yi[bad], marker="x", s=16, color=XS_MUTED, linewidth=1.0,
                       zorder=3, transform=ax.get_yaxis_transform())   # x in axes fraction, y in data
    return undefined


def _xv4h_pct_limit(series: Sequence[Tuple[str, np.ndarray, str]]) -> float:
    """Right x limit (percent) of an error-rate panel: 5 % at least, else 10 % above the largest value."""
    v = np.concatenate([100.0 * np.asarray(s[1], dtype=float) for s in series]) if series else np.zeros(0)
    v = v[np.isfinite(v)]
    return max(5.0, 1.1 * float(v.max())) if len(v) else 5.0


def _xv4h_finish(fig, axes: np.ndarray, order: Sequence[str], undefined: bool, ncol: int) -> None:
    """Cell labels on the shared y axis (run order, top to bottom) and one legend under the figure, so that it never
    covers data or an axis label."""
    y = np.arange(len(order))
    for ax in axes[:, 0]:
        ax.set_yticks(y)
        ax.set_yticklabels([xs_cell_label(c) for c in order], fontsize=7)
    axes[0, 0].invert_yaxis()
    handles, labels = axes[-1, 0].get_legend_handles_labels()
    if undefined:
        handles.append(plt.Line2D([], [], marker="x", linestyle="none", markersize=4, color=XS_MUTED))
        labels.append("undefined or not available (e.g. no certified pair)")
    fig.legend(handles, labels, fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(0.08, 0.0), ncol=ncol)


def xs_fig_v4_confidence(table: pd.DataFrame, cells: Sequence[str], fig_dir: Path) -> str:
    """Per II.9c cell: certified share and certification error of the paired confidence-set tests at every scale,
    beside the point-benchmark certificates of the same policies (rows: NHB-free / NHB certificates)."""
    t = table[table["unit"].astype(str).isin([str(c) for c in cells])] if len(table) else table
    if t.empty:
        return "not produced: no II.9c confidence-set results"
    order = [str(c) for c in cells if str(c) in set(t["unit"].astype(str))]
    unit, test = t["unit"].astype(str), t["test"].astype(str)
    scale = pd.to_numeric(t["scale"], errors="coerce")
    scales = sorted({float(x) for x in scale.dropna()}, reverse=True)
    kinds = [k for k in XV4H_CS_FIG if test.isin(k[1:]).any()]
    if not kinds:
        return "not produced: the II.9c table has no paired confidence-set or point-benchmark test"

    def values(name: str, sc: Optional[float], col: str) -> np.ndarray:
        m = (test == name) & (scale.isna() if sc is None else np.isclose(scale, sc))
        if col not in t.columns or not m.any():
            return np.full(len(order), np.nan)
        return pd.to_numeric(t.loc[m, col], errors="coerce").set_axis(unit[m]).reindex(order).to_numpy(dtype=float)

    fig, axes = plt.subplots(len(kinds), 2, figsize=(12, len(kinds) * (0.9 + 0.5 * len(order)) + 0.4), squeeze=False,
                             sharey=True)
    undefined = False
    for r, (lab, cs_test, pt_test) in enumerate(kinds):
        for j, (col, xlab, what) in enumerate((("share", "certified pairs (%)", "certified share"),
                                               ("error_rate", "certified pairs that are wrong (%)",
                                                "certification error"))):
            series = [("point benchmark (same policies)", values(pt_test, None, col), XS_COLORS[0])]
            series += [(f"confidence set, scale {s:g} ({'valid' if s == 1.0 else 'heuristic'})",
                        values(cs_test, s, col), XS_COLORS[(2 + i) % len(XS_COLORS)]) for i, s in enumerate(scales)]
            ax = axes[r, j]
            undefined |= _xv4h_bars(ax, len(order), series)
            ax.set_xlim(0, 100 if col == "share" else _xv4h_pct_limit(series))
            ax.set_xlabel(xlab)
            ax.set_title(f"({'abcdefgh'[2 * r + j]}) {lab}: {what}", loc="left", color=XS_INK, fontsize=9)
    _xv4h_finish(fig, axes, order, undefined, ncol=3)
    return _xs_save(fig, fig_dir, "fig_v4_confidence")


def xs_fig_v4_bootstrap(cov: pd.DataFrame, cells: Sequence[str], fig_dir: Path, b_v3: Optional[int],
                        b_v4: Optional[int]) -> str:
    """Per II.9d cell: coverage of J_true over all default policies, certified share and certification error of the
    point interval, II.8g (B = 10, fixed mask) and the fixed and re-estimated bootstraps (rows: NHB / NHB-free)."""
    c = cov[cov["cell"].astype(str).isin([str(x) for x in cells])] if len(cov) else cov
    if c.empty:
        return "not produced: no II.9d bootstrap coverage rows"
    order = [str(x) for x in cells if str(x) in set(c["cell"].astype(str))]
    rows = [(it, lab) for it, lab in XV4H_BOOT_FIG_INTERVALS if (c["interval"] == it).any()]
    variants = [v for v in XV4H_BOOT_FIG_VARIANTS if (c["boot_variant"] == v).any()]
    if not rows or not variants:
        return "not produced: the II.9d coverage table has no NHB / NHB-free rows"
    b3, b4 = (f", B = {n}" if n else "" for n in (b_v3, b_v4))
    labels = {"point": "point benchmark", "v3boot": f"II.8g, fixed mask{b3}", "fixed": f"fixed mask{b4}",
              "reestimated": f"re-estimated construction{b4}"}
    panels = (("coverage_all", "coverage of J_true, all default policies (%)", "coverage"),
              ("certified_share", "certified pairs (%)", "certified share"),
              ("error_rate", "certified pairs that are wrong (%)", "certification error"))

    def values(variant: str, interval: str, col: str) -> np.ndarray:
        m = (c["boot_variant"] == variant) & (c["interval"] == interval)
        if col not in c.columns or not m.any():
            return np.full(len(order), np.nan)
        return (pd.to_numeric(c.loc[m, col], errors="coerce").set_axis(c.loc[m, "cell"].astype(str)).reindex(order)
                .to_numpy(dtype=float))

    fig, axes = plt.subplots(len(rows), 3, figsize=(12, len(rows) * (0.9 + 0.5 * len(order)) + 0.4), squeeze=False,
                             sharey=True)
    undefined = False
    for r, (it, lab) in enumerate(rows):
        for j, (col, xlab, what) in enumerate(panels):
            series = [(labels[v], values(v, it, col), XS_COLORS[XV4H_BOOT_FIG_VARIANTS.index(v)]) for v in variants]
            ax = axes[r, j]
            undefined |= _xv4h_bars(ax, len(order), series)
            ax.set_xlim(0, _xv4h_pct_limit(series) if col == "error_rate" else 100)
            ax.set_xlabel(xlab)
            ax.set_title(f"({'abcdefghi'[3 * r + j]}) {lab}: {what}", loc="left", color=XS_INK, fontsize=9)
    _xv4h_finish(fig, axes, order, undefined, ncol=3)
    return _xs_save(fig, fig_dir, "fig_v4_bootstrap")


# ---------- run ----------
def _xv4h_cells(s: Dict[str, Any], table: pd.DataFrame, col: str) -> List[str]:
    """Cells of a subsection in its run order (summary 'cells'), else in table order (pooled units left out)."""
    cells = [str(c) for c in (s.get("cells") or [])]
    if not cells and len(table) and col in table.columns:
        cells = [str(u) for u in pd.unique(table[col].astype(str)) if not str(u).startswith("pooled:")]
    return cells


def xv4h_run(table_dir: Path, fig_dir: Path) -> Dict[str, Any]:
    """Section II.9h end to end: status, headline numbers, checks, figures and the two json files (inputs read from and
    outputs written to table_dir / fig_dir). Returns XV4_STATS."""
    t0 = time.perf_counter()
    table_dir, fig_dir = Path(table_dir), Path(fig_dir)
    summ: Dict[str, Dict[str, Any]] = {}
    subs: Dict[str, Dict[str, Any]] = {}
    for name, (switch, _, _) in XV4H_SUBSECTIONS.items():
        on = bool(EXPERIMENTS[switch])
        # a switched-off subsection is not read, so a summary left by an earlier session is never mixed in
        s, src = xv4h_summary(name, table_dir) if on else ({}, "not read (switched off)")
        summ[name] = s
        st = (str(s.get("status", "missing: no summary in memory or on disk")) if on
              else f"switched off ({switch} = False)")
        # II.9g is not required in artifact-replay mode (ALLOW_TRAINING False), as in the Section 34 gate
        subs[name] = {"switch": switch, "enabled": on, "required": on and (name != "II.9g" or bool(ALLOW_TRAINING)),
                      "summary_source": src, "status": st, "ran": on and st == "run",
                      "draws_complete": s.get("draws_complete"), "code_hash": s.get("code_hash"),
                      "elapsed_s": s.get("elapsed_s")}
    tables: Dict[str, pd.DataFrame] = {}
    inputs: Dict[str, Any] = {"summaries": {k: v["summary_source"] for k, v in subs.items()}, "tables": {}}
    for key, (sub, glob, fname) in XV4H_TABLES.items():
        tables[key], src = xv4h_table(glob, fname, subs[sub]["ran"], subs[sub]["summary_source"] == "memory",
                                      table_dir)
        inputs["tables"][key] = {"source": src, "rows": int(len(tables[key]))}
    order_counts = {**xv4h_cs_order_counts(tables["cs_policies"]), **xv4h_boot_order_counts(tables["boot_intervals"])}
    for name, s in subs.items():
        if s["ran"]:
            flag, detail = xv4h_invariants(name, summ[name], order_counts)
            if name in XV4H_RECOUNT_TABLES:
                detail["recount_input"] = inputs["tables"][XV4H_RECOUNT_TABLES[name]]
        else:
            flag, detail = None, {"note": "not evaluated: the subsection did not run"}
        s.update(invariants_pass=None if flag is None else bool(flag), invariants=detail)
    missing = {k: v["status"] for k, v in subs.items() if v["enabled"] and not v["ran"]}
    off = [k for k, v in subs.items() if not v["enabled"]]
    parts = ["run"]
    if missing:
        parts.append("missing: " + "; ".join(f"{k} {st}" for k, st in missing.items()))
    if off:
        parts.append("switched off: " + ", ".join(off))
    status = " | ".join(parts)
    required = [v for v in subs.values() if v["required"]]
    draws_complete = None if not required else all(v["ran"] and v["draws_complete"] is True for v in required)
    makers: Dict[str, Callable[[], Dict[str, Any]]] = {
        "II.9b": lambda: xv4h_headline_robust(summ["II.9b"]),
        "II.9c": lambda: xv4h_headline_cs(tables["cs_table"], summ["II.9c"]),
        "II.9d": lambda: xv4h_headline_boot(tables["boot_coverage"], summ["II.9d"]),
        "II.9e": lambda: xv4h_headline_sens(tables["sens_table"], summ["II.9e"]),
        "II.9f": lambda: xv4h_headline_lift(summ["II.9f"]), "II.9g": lambda: xv4h_headline_clean(summ["II.9g"])}
    headline = {k: (fn() if subs[k]["ran"] else {"status": f"not available: {subs[k]['status']}"})
                for k, fn in makers.items()}
    figures: Dict[str, str] = {}
    if subs["II.9c"]["ran"]:
        figures["fig_v4_confidence"] = xs_fig_v4_confidence(
            tables["cs_table"], _xv4h_cells(summ["II.9c"], tables["cs_table"], "unit"), fig_dir)
    else:
        figures["fig_v4_confidence"] = f"not produced: II.9c {subs['II.9c']['status']}"
    if subs["II.9d"]["ran"]:
        v3 = globals().get("XV3BOOT")
        figures["fig_v4_bootstrap"] = xs_fig_v4_bootstrap(
            tables["boot_coverage"], _xv4h_cells(summ["II.9d"], tables["boot_coverage"], "cell"), fig_dir,
            getattr(v3, "replicates", None), summ["II.9d"].get("replicates"))
    else:
        figures["fig_v4_bootstrap"] = f"not produced: II.9d {subs['II.9d']['status']}"
    inv = {k: v["invariants_pass"] for k, v in subs.items()}
    passes = bool(all(f is not False for f in inv.values()) and all(n == 0 for n in order_counts.values()))
    checks = {"status": status, "code_hash": XV4_STATS_CODE_HASH, "passes": passes, "invariants_pass": inv,
              "order_counts": order_counts, "draws_complete": draws_complete, "missing": missing,
              "switched_off": off, "subsections": subs, "inputs": inputs, "figures": figures,
              "tolerances": {"value": XV4_TOL, "p": XV4_PTOL, "boot_order": XV4H_ORDER_TOL},
              "elapsed_s": time.perf_counter() - t0}
    save_json(checks, table_dir / "v4_stats_checks.json")
    written = {"xenv/v4_overview.json", "xenv/v4_stats_checks.json"}
    # 'current': written by a subsection that ran (II.9a always), or by this cell; a file of another session is not
    current = {"II.9a": True, **{k: v["ran"] for k, v in subs.items()}}
    outputs = {sec: [{"path": rel, "exists": bool(rel in written or _xv4h_path(rel, table_dir, fig_dir).exists()),
                      "current": bool((rel in written or figures.get(Path(rel).stem) == "written")
                                      if sec == "II.9h" else current[sec])} for rel in rels]
               for sec, rels in XV4H_OUTPUTS.items()}
    keys = ("switch", "enabled", "required", "status", "summary_source", "ran", "draws_complete", "invariants_pass",
            "code_hash", "elapsed_s")
    overview = {"status": status, "code_hash": XV4_STATS_CODE_HASH,
                "subsections": {k: {c: v[c] for c in keys} for k, v in subs.items()}, "missing": missing,
                "switched_off": off, "draws_complete": draws_complete, "checks_passes": passes, "headline": headline,
                "figures": figures, "outputs": outputs, "elapsed_s": time.perf_counter() - t0}
    save_json(overview, table_dir / "v4_overview.json")
    return {"overview": overview, "checks": checks, "subsections": subs, "figures": figures,
            "tables": {k: tables[k] for k in XV4H_KEEP_TABLES},
            "outputs": [o["path"] for v in outputs.values() for o in v if o["exists"] and o["current"]]}


# Objects whose source changes what II.9h reports, and the module constants they read (by value).
XV4_STATS_CODE = [_xv4h_path, xv4h_summary, xv4h_table, _xv4h_all, _xv4h_num, xv4h_cs_order_counts,
                  xv4h_boot_order_counts, xv4h_invariants, xv4h_headline_robust, xv4h_headline_cs, xv4h_headline_boot,
                  xv4h_headline_sens, xv4h_headline_lift, xv4h_headline_clean, _xv4h_bars, _xv4h_pct_limit,
                  _xv4h_finish, xs_fig_v4_confidence, xs_fig_v4_bootstrap, _xv4h_cells, xv4h_run]
XV4_STATS_CODE_HASH = obj_hash({
    "common": XV4_COMMON_CODE_HASH, "code": code_hash(*XV4_STATS_CODE),
    "constants": {"subsections": {k: list(v) for k, v in XV4H_SUBSECTIONS.items()},
                  "tables": {k: list(v) for k, v in XV4H_TABLES.items()}, "keep_tables": list(XV4H_KEEP_TABLES),
                  "recount_tables": XV4H_RECOUNT_TABLES, "outputs": {k: list(v) for k, v in XV4H_OUTPUTS.items()},
                  "cs_fig": [list(x) for x in XV4H_CS_FIG],
                  "boot_fig_intervals": [list(x) for x in XV4H_BOOT_FIG_INTERVALS],
                  "boot_fig_variants": list(XV4H_BOOT_FIG_VARIANTS), "robust_keys": list(XV4H_ROBUST_KEYS),
                  "cs_keys": list(XV4H_CS_KEYS), "boot_keys": list(XV4H_BOOT_KEYS), "sens_keys": list(XV4H_SENS_KEYS),
                  "clean_checks": list(XV4H_CLEAN_CHECKS), "lift_slacks": list(XV4H_LIFT_SLACKS),
                  "lift_range_slacks": list(XV4H_LIFT_RANGE_SLACKS), "order_tol": XV4H_ORDER_TOL,
                  "read_errors": [e.__name__ for e in XCACHE_READ_ERRORS]}})

XV4_STATS = xv4h_run(XENV_TABLE_DIR, XENV_FIG_DIR)
print("II.9h code hash:", XV4_STATS_CODE_HASH[:16], "| status:", XV4_STATS["overview"]["status"])
display(pd.DataFrame([{"subsection": k, **{c: v[c] for c in ("switch", "enabled", "required", "status",
                                                              "summary_source", "draws_complete", "invariants_pass")}}
                      for k, v in XV4_STATS["subsections"].items()]))
_t = XV4_STATS["tables"]["cs_table"]
if len(_t):
    display(_t[_t["unit"].astype(str).str.startswith("pooled:") | (_t["unit"].nunique() == 1)][
        [c for c in ("unit", "scale", "test", "draws", "share", "error_rate", "share_sens", "error_rate_sens",
                     "draws_any_wrong", "coverage") if c in _t.columns]].round(4))
_t = XV4_STATS["tables"]["boot_coverage"]
if len(_t):
    display(_t[_t["cell"].astype(str).str.startswith("pooled:") & _t["interval"].isin(["nhb", "free"])][
        [c for c in ("cell", "boot_variant", "interval", "draws", "coverage_all", "coverage_ppos", "coverage_pzero",
                     "certified_share", "error_rate") if c in _t.columns]].round(4))
del _t
print("II.9g clean check:", json.dumps(XV4_STATS["overview"]["headline"]["II.9g"], default=_json_default))
print("Figures:", XV4_STATS["figures"], "| draws complete:", XV4_STATS["overview"]["draws_complete"],
      "| checks pass:", XV4_STATS["checks"]["passes"])
if not XV4_STATS["checks"]["passes"]:
    raise AssertionError(f"II.9 invariant checks failed: {XV4_STATS['checks']['invariants_pass']}, recounted "
                         f"{XV4_STATS['checks']['order_counts']} (xenv/v4_stats_checks.json)")

## 32. Artifact replay: cross-session comparison, saved policy files, CPU checkpoint inference (audit C04)

Three levels, reported separately so that *replay* is never confused with *fresh retraining*:
1. **Cross-session replay** — the previous session's `exact_results_all.csv` and policy-file hashes were
   snapshotted (Section 7) *before* anything was overwritten; they are compared with this session's results.
   With an unchanged ROOT this is a genuine replay check; after `FORCE_RETRAIN` it measures fresh-retraining
   differences (expected up to the hardware tolerance).
2. **File round-trip** — every saved policy file of the paper-table groups is reloaded, hash-checked,
   re-validated and re-evaluated.
3. **Checkpoint inference on CPU** — main-study critic and diffusion checkpoints are rebuilt on CPU using the
   normaliser stored **inside** each checkpoint; re-extracted actions are compared with the exported policies.
   Guidance tables for the diffusion replay come from the CPU-replayed critic B and tabular CQL.

In [ ]:
# ============================================================
# 32. Artifact replay
# ============================================================
EXECLOG.mark("32. Artifact replay")
refresh_results()
finalize_projection_checks(verbose=False)
REPLAY_GROUPS = {"main", "reference", "inference_ablation", "legacy", "critic_ablation", "soft_sensitivity",
                 "schedule_ablation"}
LATEST_FILES = {}
for rec in POLICY_FILES:
    LATEST_FILES[(rec["policy"], rec["seed"])] = rec
res_index = RESULTS[RESULTS["status"] == "ok"].set_index(["name", "seed", "rule"])

# (1) cross-session replay
cross_rows, hash_mismatch = [], []
prev = PREV_SESSION["exact_results"]
if prev is not None and len(prev):
    prev_ok = prev[prev["status"] == "ok"].drop_duplicates(["name", "seed", "rule"], keep="last").set_index(["name", "seed", "rule"])
    common_idx = res_index.index.intersection(prev_ok.index)
    for key in common_idx:
        cur, old = res_index.loc[key], prev_ok.loc[key]
        cross_rows.append({"policy": key[0], "seed": key[1], "rule": key[2],
                           **{f"abs_diff_{m}": abs(float(cur[m]) - float(old[m]))
                              for m in ("return", "inadmissible_rate", "avg_episode_length")}})
    hash_mismatch = [r["file"] for r in LATEST_FILES.values()
                     if r["file"] in PREV_SESSION["policy_hashes"] and PREV_SESSION["policy_hashes"][r["file"]] != r["policy_sha256"]]
CROSS = pd.DataFrame(cross_rows)
CROSS.to_csv(TABLE_DIR / "cross_session_replay.csv", index=False)
CROSS_OK = (None if prev is None or CROSS.empty else
            bool(not hash_mismatch and CROSS.filter(like="abs_diff").max().max() <= 1e-10))

# (2) file round-trip
replay_rows = []
for (name, seed), rec in LATEST_FILES.items():
    if REGISTRY[name].group not in REPLAY_GROUPS:
        continue
    with np.load(ROOT / rec["file"]) as z:
        pi = validate_policy(z["policy"], f"replay {name}")
    hash_ok = array_hash(pi) == rec["policy_sha256"]
    for rule in RULES:
        if (name, seed, rule) not in res_index.index:
            continue
        m = EVALUATOR.evaluate(pi, rule)
        saved = res_index.loc[(name, seed, rule)]
        replay_rows.append({"policy": name, "seed": seed, "rule": rule, "file": rec["file"], "file_hash_matches": hash_ok,
                            "abs_diff_return": abs(m["return"] - float(saved["return"])),
                            "abs_diff_inadmissible_rate": abs(m["inadmissible_rate"] - float(saved["inadmissible_rate"])),
                            "abs_diff_length": abs(m["avg_episode_length"] - float(saved["avg_episode_length"]))})
POLICY_REEVAL = pd.DataFrame(replay_rows)
POLICY_REEVAL.to_csv(TABLE_DIR / "policy_reevaluation.csv", index=False)
REPLAY_SUMMARY = {
    "file_round_trip_rows": int(len(POLICY_REEVAL)),
    "all_file_hashes_match": bool(POLICY_REEVAL["file_hash_matches"].all()) if len(POLICY_REEVAL) else None,
    "max_abs_discrepancy": float(POLICY_REEVAL[["abs_diff_return", "abs_diff_inadmissible_rate",
                                                "abs_diff_length"]].max().max()) if len(POLICY_REEVAL) else None,
    "cross_session_rows": int(len(CROSS)),
    "cross_session_policy_hash_mismatches": (len(hash_mismatch) if prev is not None else None),
    "cross_session_max_abs_diff": (float(CROSS.filter(like="abs_diff").max().max()) if len(CROSS) else None),
    "cross_session_ok": CROSS_OK,
    "force_retrain": FORCE_RETRAIN,
}

# Projection return / length differences (C01 closure: per-seed equality under mean; projected equality across rules).
okx = RESULTS[RESULTS["status"] == "ok"].set_index(["name", "seed", "rule"])
pr_rows = []
for child in [n for n in REGISTRY if n.endswith(" [projected]")]:
    par = REGISTRY[child].parent
    for seed in REGISTRY[child].by_seed:
        ks = [(par, seed, "mean"), (par, seed, "terminate"), (child, seed, "mean"), (child, seed, "terminate")]
        if not all(k in okx.index for k in ks):
            continue
        om, ot, pm_, pt_ = (okx.loc[k] for k in ks)
        pr_rows.append({"policy": par, "seed": seed, "group": REGISTRY[child].group,
                        "dJ_orig_mean_minus_proj_mean": om["return"] - pm_["return"],
                        "dJ_proj_mean_minus_proj_terminate": pm_["return"] - pt_["return"],
                        "dL_orig_mean_minus_proj_mean": om["avg_episode_length"] - pm_["avg_episode_length"],
                        "dL_proj_mean_minus_proj_terminate": pm_["avg_episode_length"] - pt_["avg_episode_length"],
                        "dJ_native_mean_minus_native_terminate": om["return"] - ot["return"],
                        "dL_native_mean_minus_native_terminate": om["avg_episode_length"] - ot["avg_episode_length"]})
PROJ_DIFFS = pd.DataFrame(pr_rows)
PROJ_DIFFS.to_csv(TABLE_DIR / "projection_return_differences.csv", index=False)
if len(PROJ_DIFFS) and MEAN_RULE_IDENTITY_HOLDS:
    _cols = [c for c in PROJ_DIFFS.columns if c.startswith("d") and "native" not in c]
    assert PROJ_DIFFS[_cols].abs().max().max() <= 1e-9, "Projected returns/lengths are not mean-equivalent."

# (3) CPU checkpoint inference
cpu = torch.device("cpu")
ckpt_rows = []
CPU_Q = {}


def _x_from_normalizer(nm: dict) -> np.ndarray:
    return ((BENCH.state_features[:, nm["keep"]] - np.asarray(nm["mean"], dtype=np.float32))
            / np.asarray(nm["std"], dtype=np.float32)).astype(np.float32)


for (key, bname, seed), res in list(CRITICS.items()):
    if bname != "main" or key not in SPEC_BY_KEY:
        continue
    spec = res["spec"]
    payload = STORE.load_final(res["label"], critic_identity(spec, seed, MAIN), map_location="cpu", respect_force=False)
    if payload is None:
        ckpt_rows.append({"model": spec.display, "seed": seed, "status": "missing checkpoint"})
        continue
    nm = payload["meta"]["normalizer"]
    x = _x_from_normalizer(nm)
    net = TwinDuelingCQL(len(nm["keep"]), A_N, CFG.hidden_dim)
    net.load_state_dict(payload["model"])
    q1, q2 = critic_tables(net, x, device=cpu)
    CPU_Q[(key, seed)] = np.minimum(q1, q2)
    if spec.display not in REGISTRY or seed not in REGISTRY[spec.display].by_seed:
        continue
    pi = greedy_policy(np.minimum(q1, q2), masked=spec.masked_extraction)
    ref = REGISTRY[spec.display].by_seed[seed]
    ckpt_rows.append({"model": spec.display, "seed": seed, "status": "ok",
                      "trained_on": payload.get("device_type"), "replayed_on": "cpu",
                      "normalizer_matches_bundle": bool(np.allclose(x, MAIN.all_x, atol=1e-6)),
                      "max_abs_q_diff_cpu_vs_session": float(max(np.abs(q1 - res["q1"]).max(), np.abs(q2 - res["q2"]).max())),
                      "action_agreement_live_states": float((pi[LIVE].argmax(1) == ref[LIVE].argmax(1)).mean())})
for seed in CFG.seeds:
    key = ("main", seed, CFG.diffusion_T)
    if key not in DIFF_PRIORS or PRIMARY_DIFFUSION not in REGISTRY or seed not in REGISTRY[PRIMARY_DIFFUSION].by_seed:
        continue
    payload = STORE.load_final(DIFF_PRIORS[key]["label"], diffusion_identity(MAIN, seed, CFG.diffusion_T),
                               map_location="cpu", respect_force=False)
    if payload is None or (PREREG["primary_hard_critic"], seed) not in CPU_Q:
        continue
    nm = payload["meta"]["normalizer"]
    x_ck = _x_from_normalizer(nm)
    den = build_denoiser(x_ck.shape[1])
    den.load_state_dict(payload["model"])
    dif = CategoricalDiffusion(A_N, CFG.diffusion_T, CFG.diffusion_beta_start, CFG.diffusion_beta_end)
    cache = ReverseCache(den, dif, x_ck, device=cpu)
    G_B = critic_ensemble(CPU_Q[(PREREG["primary_hard_critic"], seed)], get_tab("cql", MAIN, seed)["q"])
    pi, _ = guided_reverse_policy(cache, G_B, G_PRIMARY)
    ref = REGISTRY[PRIMARY_DIFFUSION].by_seed[seed]
    ckpt_rows.append({"model": PRIMARY_DIFFUSION, "seed": seed, "status": "ok",
                      "trained_on": payload.get("device_type"), "replayed_on": "cpu",
                      "normalizer_matches_bundle": bool(np.allclose(x_ck, MAIN.all_x, atol=1e-6)),
                      "action_agreement_live_states": float((pi[LIVE].argmax(1) == ref[LIVE].argmax(1)).mean())})
    del cache
CHECKPOINT_REPLAY = pd.DataFrame(ckpt_rows)
CHECKPOINT_REPLAY.to_csv(TABLE_DIR / "checkpoint_replay_cpu.csv", index=False)
REPLAY_SUMMARY["checkpoint_replays"] = int(len(CHECKPOINT_REPLAY))
REPLAY_SUMMARY["min_action_agreement"] = (float(CHECKPOINT_REPLAY["action_agreement_live_states"].min())
                                          if len(CHECKPOINT_REPLAY) and "action_agreement_live_states" in CHECKPOINT_REPLAY else None)
print(json.dumps(REPLAY_SUMMARY, indent=2))
numerical = json.loads((TABLE_DIR / "numerical_audit.json").read_text())
numerical.update({"policy_replay": REPLAY_SUMMARY, "projection_identity_verified": PROJECTION_IDENTITY_VERIFIED,
                  "mean_rule_identity": MEAN_RULE_IDENTITY})
save_json(numerical, TABLE_DIR / "numerical_audit.json")
if REPLAY_SUMMARY["max_abs_discrepancy"] is not None:
    assert REPLAY_SUMMARY["max_abs_discrepancy"] <= 1e-12, "Replayed policy evaluation differs from the session results."
if CROSS_OK is False:
    print("NOTE: results differ from the previous session in this ROOT (see cross_session_replay.csv). This is expected "
          "after FORCE_RETRAIN or a code change; otherwise investigate.")
if REPLAY_SUMMARY["min_action_agreement"] is not None and REPLAY_SUMMARY["min_action_agreement"] < 1.0:
    print("NOTE: CPU checkpoint inference changed some greedy actions (near-ties between CPU and GPU arithmetic); "
          "see checkpoint_replay_cpu.csv.")

## 33. Release package: code, environment lock, manifests, table-to-command map, compute summary (audit C04)

The inventory with sizes and SHA-256 and the release archive are written **after the Section 34 gate** (the cell
at the end of Section 34), so that they contain this run's audit closure checklist and reproducibility manifest.
That cell also re-exports the code snapshot (and its hashes in `input_sha256.json`), so the archived notebook and
cell sources include the gate and closure output; the export below is kept as a fallback.

The archive contains **the code** (the executed notebook is exported automatically on Colab, plus the
executed cell sources), per-seed results, policy files, checkpoint metadata (and weights if
`RELEASE_INCLUDE_WEIGHTS`), dataset manifests always and dataset arrays if `RELEASE_INCLUDE_DATASET` (check the
benchmark's terms before redistribution; manifests and content hashes allow deterministic regeneration), the
environment lock, a table-to-command map covering every generated table and figure (Part I and the Part II files
under `tables/xenv/` and `figures/xenv/`), the Part II per-(cell, draw) result cache (`xenv_cache/`, so Part II
can be replayed with `ALLOW_TRAINING = False`), and an inventory. `compute_summary.csv` sums the logged wall time
of every executed cell per section, per session and per part, with each session's GPU (reproducibility-checklist item
"compute resources"). The execution-log totals count only cells that finished (a cell interrupted by a disconnect,
e.g. during the long II.6a driver cell, is not logged), so a "part II draws" row adds the per-draw driver compute
stored with every draw (`XDRAW_META["timing.total_s"]`, computed in this or an earlier session); the additional
experiments of Section II.8 add "part II v3 draws" (`XDRAW_META_V3`) and "part II v3 bootstrap draws"
(`XV3_BOOT_META`) rows in the same way. Every II.8 output (`tables/xenv/v3_*`, `figures/xenv/fig_v3_*`) is in the
table-to-command map, and the II.8 code hashes are in `input_sha256.json`. The further experiments of
Section II.9 add "part II v4 ..." rows for their cached draws (confidence sets when their draw meta is in memory as
`XV4_CS_META`, bootstrap `XV4_BOOT_META`, sensitivity cells `XDRAW_META_V4SENS`, lifts `XV4_LIFT["draw_meta"]`) and
for the clean rerun (`XV4_CLEAN_DRAWS`); every II.9 output (`tables/xenv/v4_*`, `figures/xenv/fig_v4_*`) is in the
map, and the II.9 code and configuration hashes are in `input_sha256.json`. 

In [ ]:
# ============================================================
# 33. Release package
# ============================================================
EXECLOG.mark("33. Release package")


def compute_summary(log_dir: Path, current_gpu: Optional[str] = None) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Logged cell wall time per (session, section), per section, per session, per part and in total.

    Sources: `execution_log.jsonl` (one record per executed cell, appended after the cell finishes, so the
    release cells themselves are not included) and the per-session `runtime_*.json` (GPU, compute profile).
    Every logged execution counts, including re-runs. Unparseable lines (a write cut by a disconnect) and cells
    without timing are counted and reported, never silently dropped."""
    recs, bad_lines = [], 0
    log_path = Path(log_dir) / "execution_log.jsonl"
    if log_path.exists():
        for line in log_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                recs.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1
    log = pd.DataFrame(recs).reindex(columns=["session_start", "section", "success", "elapsed_s"])
    log["elapsed_s"] = pd.to_numeric(log["elapsed_s"], errors="coerce")
    sessions: Dict[str, dict] = {}
    for f in sorted(Path(log_dir).glob("runtime_*.json")):
        info = json.loads(f.read_text(encoding="utf-8"))
        sessions[str(info.get("session_start_utc"))] = info
    log["gpu"] = [sessions.get(str(s), {}).get("gpu") for s in log["session_start"]]
    log["compute_profile"] = [sessions.get(str(s), {}).get("compute_profile") for s in log["session_start"]]
    log["part"] = ["Part II" if str(sec).startswith("II.") else "Part I" for sec in log["section"]]
    rows = []

    def _row(level: str, sub: pd.DataFrame, **keys) -> None:
        e = sub["elapsed_s"]
        rows.append({"level": level, **keys, "cells": int(len(sub)), "failed_cells": int(sub["success"].eq(False).sum()),
                     "cells_without_timing": int(e.isna().sum()), "elapsed_s": float(e.sum()),
                     "elapsed_h": float(e.sum()) / 3600.0})

    for (sess, sec), sub in log.groupby(["session_start", "section"], sort=False):
        _row("session x section", sub, session_start=sess, section=sec, part=sub["part"].iloc[0],
             gpu=sub["gpu"].iloc[0], compute_profile=sub["compute_profile"].iloc[0])
    for sec, sub in log.groupby("section", sort=False):
        _row("section", sub, section=sec, part=sub["part"].iloc[0])
    for sess, sub in log.groupby("session_start", sort=False):
        _row("session", sub, session_start=sess, gpu=sub["gpu"].iloc[0], compute_profile=sub["compute_profile"].iloc[0])
    for part, sub in log.groupby("part", sort=False):
        _row("part", sub, part=part)
    gpus = sorted({str(g) for g in log["gpu"].dropna()})
    _row("total", log, gpu="; ".join(gpus) if gpus else None)
    out = pd.DataFrame(rows)
    summary = {"total_hours": float(log["elapsed_s"].sum()) / 3600.0, "sessions": int(log["session_start"].nunique()),
               "cells_logged": int(len(log)), "cells_without_timing": int(log["elapsed_s"].isna().sum()),
               "unparseable_log_lines": int(bad_lines), "gpus": gpus, "current_gpu": current_gpu,
               "hours_by_part": {p: float(g["elapsed_s"].sum()) / 3600.0 for p, g in log.groupby("part")},
               "scope": "cells logged before Section 33 in this ROOT (all sessions, re-runs included)"}
    return out, summary


COMPUTE_SUMMARY, COMPUTE_TOTALS = compute_summary(LOG_DIR, GPU_NAME)
# Part II driver compute per draw (stored in each draw's cached meta, so it survives interrupted cells and is
# re-emitted for draws loaded from the cache); the execution log counts finished cells only.
_xdm = globals().get("XDRAW_META")
_xdm = pd.DataFrame() if _xdm is None else _xdm
_xdraw_s = (pd.to_numeric(_xdm["timing.total_s"], errors="coerce") if "timing.total_s" in _xdm.columns
            else pd.Series(dtype=np.float64))
COMPUTE_SUMMARY = pd.concat([COMPUTE_SUMMARY, pd.DataFrame([{
    "level": "part II draws", "part": "Part II", "cells": int(len(_xdraw_s)), "failed_cells": 0,
    "cells_without_timing": int(_xdraw_s.isna().sum()), "elapsed_s": float(_xdraw_s.sum()),
    "elapsed_h": float(_xdraw_s.sum()) / 3600.0}])], ignore_index=True)
COMPUTE_TOTALS["part2_draw_compute_h"] = float(_xdraw_s.sum()) / 3600.0
COMPUTE_TOTALS["part2_draws_with_timing"] = int(_xdraw_s.notna().sum())
# Section II.8 (revision experiments): per-draw compute of the new cells and of the bootstrap ensembles.
for _lvl, _key, _name in (("part II v3 draws", "part2_v3", "XDRAW_META_V3"),
                          ("part II v3 bootstrap draws", "part2_v3_bootstrap", "XV3_BOOT_META")):
    _m = globals().get(_name)
    _m = pd.DataFrame() if _m is None else _m
    _ts = (pd.to_numeric(_m["timing.total_s"], errors="coerce") if "timing.total_s" in _m.columns
           else pd.Series(dtype=np.float64))
    COMPUTE_SUMMARY = pd.concat([COMPUTE_SUMMARY, pd.DataFrame([{
        "level": _lvl, "part": "Part II", "cells": int(len(_ts)), "failed_cells": 0,
        "cells_without_timing": int(_ts.isna().sum()), "elapsed_s": float(_ts.sum()),
        "elapsed_h": float(_ts.sum()) / 3600.0}])], ignore_index=True)
    COMPUTE_TOTALS[f"{_key}_draw_compute_h"] = float(_ts.sum()) / 3600.0
    COMPUTE_TOTALS[f"{_key}_draws_with_timing"] = int(_ts.notna().sum())
# Section II.9: per-draw compute of the cached draws of II.9c-II.9f (draw meta 'timing.total_s') and
# of the clean rerun of II.9g (rerun, reference and comparison seconds per draw).
_xv4_lift_meta = (globals().get("XV4_LIFT") or {}).get("draw_meta")
for _lvl, _key, _m, _cols in (
        ("part II v4 confidence-set draws", "part2_v4_confidence", globals().get("XV4_CS_META"), ("timing.total_s",)),
        ("part II v4 bootstrap draws", "part2_v4_bootstrap", globals().get("XV4_BOOT_META"), ("timing.total_s",)),
        ("part II v4 sensitivity draws", "part2_v4_sensitivity", globals().get("XDRAW_META_V4SENS"),
         ("timing.total_s",)),
        ("part II v4 lift draws", "part2_v4_lift", _xv4_lift_meta, ("timing.total_s",)),
        ("part II v4 clean-rerun draws", "part2_v4_clean", globals().get("XV4_CLEAN_DRAWS"),
         ("rerun_s", "reference_s", "compare_s"))):
    _m = pd.DataFrame() if _m is None else _m
    _ts = (sum(pd.to_numeric(_m[c], errors="coerce") for c in _cols) if set(_cols) <= set(_m.columns)
           else pd.Series(dtype=np.float64))
    COMPUTE_SUMMARY = pd.concat([COMPUTE_SUMMARY, pd.DataFrame([{
        "level": _lvl, "part": "Part II", "cells": int(len(_ts)), "failed_cells": 0,
        "cells_without_timing": int(_ts.isna().sum()), "elapsed_s": float(_ts.sum()),
        "elapsed_h": float(_ts.sum()) / 3600.0}])], ignore_index=True)
    COMPUTE_TOTALS[f"{_key}_draw_compute_h"] = float(_ts.sum()) / 3600.0
    COMPUTE_TOTALS[f"{_key}_draws_with_timing"] = int(_ts.notna().sum())
COMPUTE_SUMMARY.to_csv(TABLE_DIR / "compute_summary.csv", index=False)
_xv4_hours = {k[len("part2_v4_"):-len("_draw_compute_h")]: round(v, 2) for k, v in COMPUTE_TOTALS.items()
              if k.startswith("part2_v4_") and k.endswith("_draw_compute_h")}
print(f"Logged compute: {COMPUTE_TOTALS['total_hours']:.2f} h over {COMPUTE_TOTALS['sessions']} session(s); "
      f"by part {({k: round(v, 2) for k, v in COMPUTE_TOTALS['hours_by_part'].items()})}; GPUs {COMPUTE_TOTALS['gpus']}; "
      f"Part II driver compute over {COMPUTE_TOTALS['part2_draws_with_timing']} draws (any session): "
      f"{COMPUTE_TOTALS['part2_draw_compute_h']:.2f} h; II.8 draws {COMPUTE_TOTALS['part2_v3_draw_compute_h']:.2f} h, "
      f"II.8 bootstrap draws {COMPUTE_TOTALS['part2_v3_bootstrap_draw_compute_h']:.2f} h; II.9 draws (h) {_xv4_hours}")
if COMPUTE_TOTALS["unparseable_log_lines"] or COMPUTE_TOTALS["cells_without_timing"]:
    print(f"NOTE: {COMPUTE_TOTALS['unparseable_log_lines']} unparseable execution-log lines and "
          f"{COMPUTE_TOTALS['cells_without_timing']} cells without timing are excluded from the hour totals.")

TABLE_TO_COMMAND = [
    ("Table 1 dataset properties", "Sections 5-6", "validate_dataset, corrected support audit", "eda_dataset_summary.csv; corrected_support_audit.json; dataset_manifest_main.json; table1_reconciliation.csv"),
    ("Table 2 feature representation", "Sections 4, 7", "feature provenance", "feature_provenance_timing.csv; benchmark_construction_provenance.json"),
    ("Table 3 hyperparameters", "Section 1", "Config / apply_profile / PREREG", "fresh_config.json; preregistration.json; experiment_switches.json; run_profile.json"),
    ("Table 4 main comparison", "Sections 19-20", "evaluate_registered / wide_table", "table_main_policy_comparison.csv; exact_results_summary.csv; exact_results_all.csv"),
    ("Table 5 statistics", "Section 30", "compare / holm_adjust", "statistics_primary.json; statistics_secondary_family.csv; statistics_historical_table5_one_sided.csv; paired_differences.csv; replication_planning.csv"),
    ("Table 6 inference ablations", "Sections 18, 22", "run_diffusion_suite / ablation_table", "table6_inference_ablations_primary_critic.csv; table6_inference_ablations_legacy_critic.csv; post_training_controls_all.csv"),
    ("Table 7 guidance sweep", "Sections 18, 22", "run_diffusion_suite", "table7_guidance_sweep.csv"),
    ("Table 8 per-seed returns", "Section 20", "pivot of RESULTS", "table_per_seed_returns.csv"),
    ("C01 projection controls", "Sections 19, 21, 32", "add_projection_controls", "projection_equivalence_checks.csv; projection_return_differences.csv; legacy_projection_controls_summary.csv"),
    ("C02 critic stability / constraint placement", "Sections 16, 23", "run_critics(ABLATION_SPECS) / PLACEMENT_CELLS", "critic_stability_ablation.csv; critic_stability_summary.csv; critic_stability_summary_main_all_seeds.csv; critic_stability_gate.json; critic_diagnostics.csv; legacy_critic_diagnostics.csv; table_constraint_placement.csv"),
    ("C03 reproducibility", "Section 13", "reproducibility + regulariser tests", "reproducibility_tests.json; regularizer_gradient_tests.json"),
    ("M02 support estimation", "Section 25", "estimated_support_mask", "support_estimation_mask_comparison.csv; support_estimation_policy_results.csv"),
    ("M04 selection protocol", "Section 26", "empirical_mdp_evaluator", "selection_candidates_fqe_vs_exact.csv; selection_fqe_selected.csv; selection_locked_fresh_evaluation.csv; selection_protocol.json; tuning_budget.csv"),
    ("M05 soft constraint", "Section 24", "soft analysis", "soft_constraint_analysis.csv; greedy_margin_tie_report.csv"),
    ("M06 diffusion necessity and cost", "Sections 17-18, 22", "policy nets / prior x critic / cost", "table_diffusion_schedule_length.csv; cost_accounting.csv; cost_accounting_components.csv; cost_accounting_component_summary.csv; diffusion_policy_construction_cost.csv; diffusion_schedule_audit.json; diffusion_reverse_support_*.json"),
    ("M07 robustness", "Sections 27-28, 30", "replication / regimes / stress / subgroups", "statistics_hierarchical_replication.csv; statistics_hierarchical_paired_differences.csv; replication_results.csv; data_regime_datasets.csv; data_regime_results.csv; data_regime_size_summary.csv; stress_test_results.csv; stress_test_ranking_persistence.csv; subgroup_diagnostics.csv"),
    ("M08 representation", "Sections 4, 29", "feature provenance / ablation", "representation_ablation.csv"),
    ("M09 evaluator validation", "Sections 3, 8, 14, 20", "toy tests / env semantics / Monte Carlo", "numerical_audit.json; monte_carlo_crosscheck.csv; benchmark_reference_sanity.csv; benchmark_semantics_audit.json; benchmark_audit.json"),
    ("M10 episode semantics", "Section 5", "validate_dataset + short-cutoff test", "dataset_manifests_all.csv"),
    ("Replay", "Sections 21, 32", "artifact replay", "cross_session_replay.csv; policy_reevaluation.csv; checkpoint_replay_cpu.csv; legacy_policy_reevaluation.csv; legacy_replay.json"),
    ("EDA tables and figures", "Section 6", "EDA", "eda_*.csv; eda_transitions_enriched.csv.gz; figures/eda/*"),
    ("Benchmark support figure", "Section 4", "benchmark audit", "figures/benchmark_support_set_sizes.png"),
    ("Figures 2-3", "Section 6", "EDA", "figures/paper_fig2_dataset_distributions.*; figures/paper_fig3_severity.*"),
    ("Figures 7-10", "Section 31", "figures", "figures/paper_fig7_*; figures/paper_fig8_*; figures/paper_fig9_*; figures/paper_fig10_*"),
    ("New diagnostic figures", "Section 31", "figures", "figures/fig_c01_projection_controls.*; figures/fig_c02_critic_scale_drift.*; figures/fig_m03_hierarchical_replication.*; figures/fig_m04_fqe_vs_exact.*; figures/fig_m07_stress_ranking.*"),
    ("Gates, manifests and provenance", "Sections 1, 30, 33, 34", "support gate / closure / inventory", "fresh_custom_model_performance_gate.json; audit_closure_checklist.csv; fresh_reproducibility_manifest.json; input_sha256.json; table_to_command_map.csv; checkpoint_metadata.csv"),
    ("Compute resources (checklist)", "Section 33", "compute_summary over logs/execution_log.jsonl", "compute_summary.csv"),
    # ---- Part II: cross-environment exact study (Sections II.1-II.7); paths relative to tables/ and figures/ ----
    ("Part II configuration and preregistration", "Section II.1", "XConfig / apply_xrun_mode / PREREG_PART2", "xenv/xenv_config.json; xenv/xenv_preregistration.json"),
    ("Part II environment suite and adapter checks", "Section II.3", "x_icu_env / gumbel_true_env / gumbel_exactness_check / garnet_true_env / construct_benchmark / x_environment_suite_table", "xenv/icu_adapter_check.json; xenv/gumbel_exactness_check.json; xenv/gumbel_simulator_report.json; xenv/gumbel_state_decode.csv; xenv/xenv_environment_suite.csv; data/xenv_gumbel_true_*.npz"),
    ("Part II method catalogue and self-test", "Section II.4", "run_xmethods / X_METHOD_CATALOG / x_methods_selftest", "xenv/xenv_method_catalog.csv; xenv/xenv_methods_selftest.json"),
    ("Propositions P1-P7", "Section II.5", "proposition checks", "xenv/proposition_checks.csv; xenv/proposition_summary.csv"),
    ("Exact constrained frontiers", "Section II.5", "x_cmdp_lp over XCFG.frontier_kappas", "xenv/exact_frontiers.csv"),
    ("Part II raw results and draw metadata", "Section II.6", "run_cell_draw", "xenv/xenv_results_all.csv.gz; xenv/xenv_draw_meta.csv; xenv/xenv_driver_summary.json"),
    ("Exact tabular learners vs Part I SGD learners", "Section II.6", "tabular cross-check", "xenv/xenv_tabular_crosscheck.csv"),
    ("Main-track table: cross-environment performance", "Section II.7", "normalised scores / stratified-bootstrap IQM / profiles", "xenv/xenv_normalized_scores.csv; xenv/table_xenv_main.csv; xenv/table_xenv_main_long.csv; xenv/table_performance_profiles.csv; xenv/xenv_poi_matrix.csv; xenv/xenv_stats_checks.json"),
    ("Main-track table: benchmark fidelity", "Section II.7", "Kendall tau-b / top-1 / regret / error decomposition", "xenv/table_rank_fidelity.csv; xenv/table_rank_fidelity_draws.csv; xenv/table_error_decomposition.csv; xenv/table_error_decomposition_draws.csv"),
    ("Part II preregistered tests H1-H3", "Section II.7", "sign-flip / Holm / TOST / permutation Spearman", "xenv/xenv_prereg_tests.json; xenv/xenv_h2_family.csv"),
    ("Hyper-parameter selection (default / FQE / oracle)", "Section II.7", "hp selection", "xenv/table_hp_selection.csv"),
    ("Part II figures", "Sections II.5-II.7", "matplotlib figures", "figures/xenv/fig_xenv_frontiers.*; figures/xenv/fig_xenv_rank_fidelity.*; figures/xenv/fig_xenv_harm_phase.*; figures/xenv/fig_xenv_iqm.*; figures/xenv/fig_xenv_profiles.*; figures/xenv/fig_xenv_poi.*; figures/xenv/fig_xenv_error_decomposition.*"),
    # ---- Part II revision experiments (Section II.8, REV-1 ... REV-3) ----
    ("Revision II.8: configuration, physicians, environment suite", "Sections II.8a-II.8e", "XV3Config / gumbel_physician_report / x_v3_build_cells / x_v3_environment_suite_table", "xenv/v3_config.json; xenv/v3_physician_report.csv; xenv/v3_environment_suite.csv; xenv/v3_environment_checks.json"),
    ("Revision II.8: raw results of the new cells (REV-1)", "Section II.8f", "x_v3_run_cells / run_v3_cell_draw", "xenv/v3_results_all.csv.gz; xenv/v3_draw_meta.csv; xenv/v3_driver_summary.json"),
    ("Revision II.8: bootstrap-ensemble intervals, raw (REV-2)", "Section II.8g", "x_v3_boot_run / run_v3_boot_draw / x_boot_step_map / x_boot_model", "xenv/v3_bootstrap_config.json; xenv/v3_bootstrap_results.csv.gz; xenv/v3_bootstrap_meta.csv; xenv/v3_bootstrap_summary.json"),
    ("Revision II.8: ICU-Sepsis harm curves J(lambda) (REV-3)", "Section II.8h", "x_harm_env / x_harm_curves / x_harm_summary", "xenv/v3_icu_harm_curves.csv; xenv/v3_icu_harm_summary.csv; xenv/v3_icu_harm_checks.json"),
    ("Revision II.8: bracketing coverage, certified comparisons, rank fidelity, bootstrap coverage", "Section II.8i", "xv3_stats_run", "xenv/v3_bracketing.csv; xenv/v3_bracketing_draws.csv; xenv/v3_certified.csv; xenv/v3_certified_draws.csv; xenv/v3_rank_fidelity.csv; xenv/v3_rank_fidelity_draws.csv; xenv/v3_bootstrap_intervals.csv; xenv/v3_bootstrap_coverage.csv; xenv/v3_stats_checks.json"),
    ("Revision II.8 figures", "Sections II.8h-II.8i", "matplotlib figures", "figures/xenv/fig_v3_bracketing.*; figures/xenv/fig_v3_bootstrap.*; figures/xenv/fig_v3_icu_harm.*"),
    # ---- Part II second-revision experiments (Section II.9, REV-4 ... REV-9) ----
    ("Revision II.9: configuration", "Section II.9a", "XV4Config / apply_xv4_run_mode", "xenv/v4_config.json"),
    ("Revision II.9: certificates without NHB, pair types, abstention, hidden-benefit sensitivity kappa, II.8g intervals over all policies (REV-4)", "Section II.9b", "xv4r_run / x_v4_pair_claims / xv4r_kappa_claims / xv4r_boot_table", "xenv/v4_robust_pairs.csv.gz; xenv/v4_robust_summary.csv; xenv/v4_robust_gapbins.csv; xenv/v4_robust_kappa.csv; xenv/v4_v3boot_allpolicy.csv; xenv/v4_robust_summary.json"),
    ("Revision II.9: confidence-set certificates on the true state space (REV-5)", "Section II.9c", "run_v4_cs_draw / x_cs_robust_vector / x_cs_l1_extreme / x_cs_summary_table", "xenv/v4_cs_policies.csv.gz; xenv/v4_cs_draws.csv; xenv/v4_cs_summary.csv; xenv/v4_cs_summary.json"),
    ("Revision II.9: bootstrap B = 200, fixed mask and re-estimated construction (REV-6)", "Section II.9d", "run_v4_boot_draw / x_v4_boot_construct / x_v4_boot_intervals / x_v4_boot_coverage", "xenv/v4_boot_intervals.csv.gz; xenv/v4_boot_draws.csv; xenv/v4_boot_coverage.csv; xenv/v4_boot_meta.csv; xenv/v4_boot_summary.json"),
    ("Revision II.9: sensitivity cells - timeout hazard, dead-state rule, softer physicians, state abstractions (REV-7)", "Section II.9e", "build_v4_instance / run_v4_cell_draw / xv4_sens_analyses", "xenv/v4_sens_results_all.csv.gz; xenv/v4_sens_draw_meta.csv; xenv/v4_sens_summary.csv; xenv/v4_sens_certified_draws.csv; xenv/v4_sens_rank_fidelity.csv; xenv/v4_sens_rank_fidelity_draws.csv; xenv/v4_sens_summary.json"),
    ("Revision II.9: lift sensitivity and the range of J_true over all lifts (REV-8)", "Section II.9f", "run_v4_lift_draw / x_v4_lift_bounds / x_v4_lift_summary", "xenv/v4_lift_results.csv.gz; xenv/v4_lift_summary.csv; xenv/v4_lift_summary.json"),
    ("Revision II.9: clean rerun of Part II into an empty cache (REV-9)", "Section II.9g", "x_v4_clean_run / x_v4_clean_compare / x_v4_clean_certified", "xenv/v4_clean_check.json; xenv/v4_clean_diffs.csv"),
    ("Revision II.9: overview, headline numbers and checks", "Section II.9h", "xv4h_run", "xenv/v4_overview.json; xenv/v4_stats_checks.json"),
    ("Revision II.9 figures", "Section II.9h", "matplotlib figures", "figures/xenv/fig_v4_confidence.*; figures/xenv/fig_v4_bootstrap.*"),
]
TABLE_MAP = pd.DataFrame(TABLE_TO_COMMAND, columns=["manuscript_item", "notebook_section", "code", "output_files"])
TABLE_MAP.to_csv(TABLE_DIR / "table_to_command_map.csv", index=False)

# Dataset manifests for every generated dataset (M10 closure), independent of the release switch.
man_rows = []
for mf in sorted(DATA_DIR.glob("*.manifest.json")):
    d = json.loads(mf.read_text())
    man_rows.append({"file": d["file"], "spec_hash": d["spec_hash"], "content_sha256": d["content_sha256"],
                     "file_sha256": d.get("file_sha256"),
                     **{f"spec_{k}": v for k, v in d["spec"].items() if k not in ("benchmark",)}, **d["report"]})
DATASET_MANIFESTS = pd.DataFrame(man_rows)
DATASET_MANIFESTS.to_csv(TABLE_DIR / "dataset_manifests_all.csv", index=False)
display(DATASET_MANIFESTS[[c for c in ["file", "transitions", "episodes", "terminated_episodes", "censored_episodes",
                                       "inadmissible_logged_fraction"] if c in DATASET_MANIFESTS]])

# Every generated table / figure must be mapped to its section and code.
import fnmatch
_patterns = [p_.strip() for v in TABLE_MAP["output_files"] for p_ in v.split(";")]
_unmapped = []
for f in sorted(list(TABLE_DIR.rglob("*")) + list(FIG_DIR.rglob("*"))):
    if not f.is_file():
        continue
    rel_t = str(f.relative_to(TABLE_DIR)) if TABLE_DIR in f.parents else None
    rel_f = "figures/" + str(f.relative_to(FIG_DIR)) if FIG_DIR in f.parents else None
    if not any(fnmatch.fnmatch(x, pat) for pat in _patterns for x in (rel_t, rel_f) if x):
        _unmapped.append(str(f.relative_to(ROOT)))
pd.DataFrame({"unmapped_output": _unmapped}).to_csv(RELEASE_DIR / "unmapped_outputs.csv", index=False)
if _unmapped:
    print(f"NOTE: {len(_unmapped)} generated files are not listed in the table-to-command map (release/unmapped_outputs.csv).")

meta_rows = []
for f in CKPT_DIR.rglob("*.identity.json"):
    d = json.loads(f.read_text())
    pt_file = f.with_name(d["file"])
    meta_rows.append({"label": d["label"], "identity_hash": d["identity_hash"], "saved_utc": d["saved_utc"],
                      "trained_on": d.get("device_type"), "gpu": d.get("gpu"),
                      "written_this_session": d["identity_hash"] in STORE.session_hashes,
                      "loaded_from_earlier_session": d["identity_hash"] in STORE.loaded_hashes,
                      "file": str(pt_file.relative_to(ROOT)), "bytes": pt_file.stat().st_size if pt_file.exists() else None})
CHECKPOINT_META = pd.DataFrame(meta_rows)
CHECKPOINT_META.to_csv(TABLE_DIR / "checkpoint_metadata.csv", index=False)

def export_release_code(code_dir: Path) -> None:
    """Executed notebook (Colab export) and executed cell sources into code_dir; a failed export keeps older files."""
    code_dir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import _message
        nb_json = _message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]
        (code_dir / "notebook_executed.ipynb").write_text(json.dumps(nb_json), encoding="utf-8")
        print("Executed notebook exported to the release.")
    except Exception as exc:
        print("Could not export the notebook automatically (download it via File -> Download):", type(exc).__name__)
    try:
        _cells = get_ipython().user_ns.get("In", [])[1:]  # noqa: F821
        (code_dir / "executed_cells.py").write_text("\n\n# ---- cell ----\n".join(_cells), encoding="utf-8")
    except Exception:
        pass


if EXPERIMENTS["release_package"]:
    code_dir = RELEASE_DIR / "code"
    export_release_code(code_dir)   # re-exported in 34b after the gate; this copy is the fallback
    lock = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    (RELEASE_DIR / "environment_lock.txt").write_text(lock, encoding="utf-8")
    save_json(RUNTIME_INFO, RELEASE_DIR / "runtime.json")
    TABLE_MAP.to_csv(RELEASE_DIR / "table_to_command_map.csv", index=False)
    registry_frame().to_csv(RELEASE_DIR / "policy_registry.csv", index=False)
    pd.DataFrame(list(LATEST_FILES.values())).to_csv(RELEASE_DIR / "policy_files.csv", index=False)
    CHECKPOINT_META.to_csv(RELEASE_DIR / "checkpoint_metadata.csv", index=False)
    inputs = {
        "benchmark_assets_sha256": BENCH.asset_hashes,
        "benchmark_identity": BENCH.identity,
        "construction_scripts": PROVENANCE.get("fetched_sha256", PROVENANCE_REF["sha256"]),
        "datasets": {r["file"]: {"content_sha256": r["content_sha256"], "file_sha256": r["file_sha256"]} for r in man_rows},
        "config_hash": obj_hash(asdict(CFG)),
        "preregistration_hash": obj_hash(PREREG),
        "run_profile": RUN_PROFILE,
        "xenv_config_hash": obj_hash(asdict(XCFG)) if "XCFG" in globals() else None,
        "xenv_preregistration_hash": obj_hash(PREREG_PART2) if "PREREG_PART2" in globals() else None,
        "xenv_code_hash": globals().get("XENV_CODE_HASH"),
        "xenv_v3_code_hashes": {"cells": globals().get("XENV_V3_CODE_HASH"),
                                "bootstrap": globals().get("XV3_BOOT_CODE_HASH"),
                                "icu_harm": globals().get("XV3_HARM_CODE_HASH")},
        "xenv_v3_config_hashes": {"cells": obj_hash(asdict(XV3CFG)) if "XV3CFG" in globals() else None,
                                  "bootstrap": obj_hash(asdict(XV3BOOT)) if "XV3BOOT" in globals() else None},
        "xenv_v4_code_hashes": {"common": globals().get("XV4_COMMON_CODE_HASH"),
                                "robust": globals().get("XV4_ROBUST_CODE_HASH"),
                                "confidence": globals().get("XV4_CS_CODE_HASH"),
                                "bootstrap": globals().get("XV4_BOOT_CODE_HASH"),
                                "sensitivity": globals().get("XV4_SENS_CODE_HASH"),
                                "lift": globals().get("XV4_LIFT_CODE_HASH"),
                                "clean_check": globals().get("XV4_CLEAN_CODE_HASH"),
                                "stats": globals().get("XV4_STATS_CODE_HASH")},
        "xenv_v4_config_hashes": {"v4": obj_hash(asdict(XV4CFG)) if "XV4CFG" in globals() else None},
        "code_hashes": {"dataset": DATASET_CODE_HASH, "critics": CRITIC_CODE_HASH, "policy_nets": POLICYNET_CODE_HASH,
                        "diffusion": DIFFUSION_CODE_HASH,
                        "evaluator": code_hash(ExactMDPEvaluator, validate_policy, project_to_support)},
        "release_code_sha256": {p_.name: sha256_file(p_) for p_ in code_dir.glob("*") if p_.is_file()},
        "legacy_inputs": {str(p_.relative_to(LEGACY_ROOT)): sha256_file(p_) for p_ in
                          ([LEGACY_ROOT / "data" / "offline_expert_10000eps_seed2026.npz", LEGACY_ROOT / "tables" / "exact_results_all.csv"]
                           if LEGACY_ROOT.exists() else []) if p_.exists()},
    }
    save_json(inputs, TABLE_DIR / "input_sha256.json")
    save_json(inputs, RELEASE_DIR / "input_sha256.json")
    (RELEASE_DIR / "README_release.md").write_text(
        "# Release contents\n\n"
        f"Run mode: {RUN_MODE}; created {utc_now()}.\n\n"
        "* release/code/ — executed notebook (Colab export) and executed cell sources\n"
        "* tables/ — every CSV/JSON behind the manuscript tables (see table_to_command_map.csv)\n"
        "* figures/ — PNG + PDF figures\n* policies/ — every evaluated policy (npz) per group and seed\n"
        "* checkpoints/ — identity-addressed checkpoints (+ .identity.json metadata)\n"
        "* data/ — dataset manifests (content SHA-256) and, if included, the frozen datasets; Part II exact Gumbel "
        "simulator matrices (xenv_gumbel_true_*.npz)\n* logs/ — training and execution logs\n"
        "* tables/xenv/, figures/xenv/ — Part II (cross-environment exact study) tables and figures; v3_* and fig_v3_* are "
        "the revision experiments of Section II.8 (REV-1 ... REV-3), v4_* and fig_v4_* the second-revision experiments "
        "of Section II.9 (REV-4 ... REV-9; status and headline numbers in tables/xenv/v4_overview.json)\n"
        "* xenv_cache/ — identity-addressed Part II results per (cell, draw); replay with ALLOW_TRAINING=False\n"
        "* tables/compute_summary.csv — logged wall time per section / session / part, with the GPU\n"
        "* environment_lock.txt, runtime.json, artifact_inventory.csv, input_sha256.json\n\n"
        f"Compute profile: {COMPUTE_PROFILE} ({len(CFG.seeds)} main seeds; {len(CFG.replication_dataset_seeds)} x "
        f"{len(CFG.replication_train_seeds)} replication).\n\n"
        "Clean training: a fresh ROOT (or FORCE_RETRAIN=True; delete checkpoints/force_retrain_token.json to start a new forced run).\n"
        "Artifact replay: same ROOT, ALLOW_TRAINING=False. Checkpoints are device-independent in identity; CPU replay of GPU "
        "checkpoints is supported and labelled (near-tie greedy actions may differ).\n",
        encoding="utf-8")
    print("Release files written; the code re-export, the inventory and the archive follow the Section 34 gate.")

## 34. Final integrity gate and audit closure checklist

Structural invariants are hard assertions. Scientific checks are reported with pass/fail so that a
legitimate negative finding never crashes the notebook. The checklist maps every audit ID to its evidence;
items that concern only the paper text are marked "manuscript-only".

**Part II gate** (applies to whichever `xenv_*` switches are on): every proposition check must pass
(`PROPOSITIONS_OK` is not `False`), and so must every exact-frontier cross-check (`X_FRONTIERS_OK`, which also
covers frontiers run without `xenv_theory`); the ICU-Sepsis adapter check (`X_ICU_ADAPTER_CHECK`, run whenever any
`xenv_*` switch of Sections II.3–II.8 is on; Section II.9 does not use the adapter) and the Gumbel exactness check
(`GUMBEL_EXACTNESS`) must pass when they ran; every Part II
result row has status `ok` or `improper` (improper = non-absorbing under γ = 1, counted and reported per family and
target, never hidden); `ok` rows satisfy |J| ≤ 1 (all entry rewards lie in [−1, 1]),
a pooled violation rate in [0, 1], P(any violation) ≤ E[violations], and equality of the two under termination.
Closure rows **MT-1 … MT-5** record the Part II additions: multi-environment ground truth, verified
propositions, the extended baseline suite, benchmark fidelity (H1–H3), and statistics at scale (10 seeds, 8 × 2
replication, the configured number of Part II draws per cell, a Part II design without deviations from
`PREREG_PART2['design']`). MT-5 fails by design for the lite profile and smoke mode, which are not paper scale. The
last cell of this section re-exports the code snapshot and writes the release inventory (sizes, SHA-256) and the
archive, after the closure checklist and the reproducibility manifest exist.

**II.8 gate (Section II.8)**, switch-aware: it never fails because a switch is off. When an II.8 cell switch is
on, the II.8 environment checks must pass and `XRESULTS_V3` must satisfy the row invariants of the Part II gate.
The theorem counts of II.8i must be zero: no lower-bound violation on the true rows, non-negative widths, zero
width for admissible policies, width ≤ p·(v_max − v_min), and values inside [v_min, v_max]. With
`xenv_v3_bootstrap`, the per-draw bootstrap invariants (Theorem part iii on every replicate) must hold. With
`xenv_v3_icu_harm`, the harm-curve identities must hold: endpoints equal to the reported mean / terminate values,
J_term ≤ J_λ ≤ J_mean, monotone in λ, and certified pairs ordered at every λ. Coverage and certification rates are
scientific outcomes: they are reported, never gated. Closure rows **REV-1 … REV-3** record the additional
experiments: REV-1 the II.8 cells (physicians that are not near-optimal, aggregated-state benchmarks with diabetes
hidden, random-row Garnet) with their bracketing, certified-comparison and rank-fidelity analyses; REV-2 the
bootstrap-ensemble intervals; REV-3 the ICU-Sepsis harm curves and certified comparisons.

**II.9 gate (Section II.9)**, switch-aware in the same way. It gates only on invariants, on the completion
of the enabled subsections and on the clean check's pass flag; coverage and error rates are never gated. Hard
assertions: every invariant flag that II.9h collected is not false (II.9b theorem and identity counts, II.9c
self-test of the inner problem and bound order, II.9d per-replicate invariants and unit weights, II.9e self-test and
theorem counts, II.9f invariant slacks) and the orderings II.9h recounted are zero; with II.9c, L ≤ U_free; with
II.9d, Theorem 1(iii) on every replicate; with II.9e, its draw and theorem checks. Closure rows **REV-4 … REV-9**
record each subsection:
switch on, status `run`, draws complete and invariants holding (REV-9 also needs the clean rerun to pass; it is not
run in artifact-replay mode, `ALLOW_TRAINING = False`). REV-4 is the robustness analysis of the stored results
(certificates without NHB, pair types, κ, II.8g intervals over all policies), REV-5 the confidence-set certificates,
REV-6 the bootstrap with the construction re-estimated, REV-7 the sensitivity cells, REV-8 the lift sensitivity and
REV-9 the clean rerun.

In [ ]:
# ============================================================
# 34. Final integrity gate
# ============================================================
EXECLOG.mark("34. Final integrity gate")
refresh_results()
finalize_projection_checks(verbose=False)
for name, e in REGISTRY.items():
    for seed, pi in e.by_seed.items():
        validate_policy(pi, f"{name} seed={seed}")
STRUCTURAL_USES = {"extraction", "training+extraction", "projection", "construction", "inference (every reverse step)",
                   "inference (final step only)"}
support_failures = []
for name, e in REGISTRY.items():
    if "released benchmark mask" in e.mask_source and "dataset-estimated" not in e.mask_source and e.mask_use in STRUCTURAL_USES:
        worst = max(inadmissible_mass(pi).max() for pi in e.by_seed.values())
        if worst > 1e-12:
            support_failures.append((name, worst))
assert not support_failures, f"Masked / projected policies with inadmissible mass: {support_failures[:5]}"
core = RESULTS[RESULTS["group"].isin(["main", "reference"])]
improper_core = core[core["status"] != "ok"][["name", "seed", "rule", "status"]]
if RUN_MODE == "paper":
    assert improper_core.empty, improper_core
elif len(improper_core):
    print("Smoke mode: some barely-trained main policies are improper (expected with tiny budgets):\n", improper_core)
ok = RESULTS[RESULTS["status"] == "ok"]
assert ok["return"].between(-1e-10, 1 + 1e-10).all()
assert ok["inadmissible_rate"].between(-1e-12, 1 + 1e-12).all()
assert (ok["prob_any_violation"] <= ok["expected_violations"] + 1e-9).all(), "P(any violation) must not exceed E[count]."
term_ok = ok[ok["rule"] == "terminate"]
assert np.allclose(term_ok["prob_any_violation"], term_ok["expected_violations"], atol=1e-9), \
    "Under termination the expected violation count must equal the first-violation probability."
assert DATASET_REPORT["transitions"] == len(OFFLINE["state"]) and DATASET_REPORT["episodes"] == CFG.dataset_episodes
assert 0.0 <= SUPPORT_AUDIT["admissible_pair_coverage"] <= 1.0

# ---------- Part II gate (cross-environment exact study) ----------
XENV_SWITCH_STATE = {k: bool(v) for k, v in EXPERIMENTS.items() if k.startswith("xenv_")}
XENV_RUN_FAMILIES = [f for f in ("icu", "gumbel", "garnet") if XENV_SWITCH_STATE.get(f"xenv_{f}")]


def xenv_pass_flag(check: Optional[dict], name: str) -> Optional[bool]:
    """Boolean pass flag ('passes') of a Part II self-check dict; None when the check did not run (missing, or the
    placeholder {"status": "not run"} that Section II.3 leaves when its switch is off)."""
    if check is None:
        return None
    if "passes" not in check and "passed" not in check and str(check.get("status", "")).startswith("not run"):
        return None
    flag = check.get("passes", check.get("passed"))
    if not isinstance(flag, (bool, np.bool_)):
        raise KeyError(f"{name} has no boolean 'passes' flag; keys: {sorted(check)[:25]}")
    return bool(flag)


PROPOSITIONS_STATUS = globals().get("PROPOSITIONS_OK")
if XENV_SWITCH_STATE["xenv_theory"]:
    assert "PROPOSITIONS_OK" in globals(), "xenv_theory is on but Section II.5 did not run."
assert PROPOSITIONS_STATUS is not False, "A Part II proposition check failed - see xenv/proposition_checks.csv."
ICU_ADAPTER_OK = xenv_pass_flag(globals().get("X_ICU_ADAPTER_CHECK"), "X_ICU_ADAPTER_CHECK")
GUMBEL_EXACT_OK = xenv_pass_flag(globals().get("GUMBEL_EXACTNESS"), "GUMBEL_EXACTNESS")
assert ICU_ADAPTER_OK == globals().get("X_ICU_ADAPTER_OK"), "X_ICU_ADAPTER_CHECK and X_ICU_ADAPTER_OK disagree."
assert GUMBEL_EXACT_OK == globals().get("GUMBEL_EXACTNESS_OK"), "GUMBEL_EXACTNESS and GUMBEL_EXACTNESS_OK disagree."
assert ICU_ADAPTER_OK is not False, "ICU-Sepsis FiniteEnv adapter differs from the Part I evaluator (xenv/icu_adapter_check.json)."
assert GUMBEL_EXACT_OK is not False, "Enumerated Gumbel rows disagree with the simulator step (xenv/gumbel_exactness_check.json)."
if any(v for k, v in XENV_SWITCH_STATE.items() if not k.startswith("xenv_v4_")):   # II.9 never uses the adapter
    assert ICU_ADAPTER_OK is not None, "a Part II switch is on but the ICU-Sepsis adapter check (Section II.3) did not run."
if XENV_SWITCH_STATE["xenv_gumbel"]:
    assert GUMBEL_EXACT_OK is not None, "xenv_gumbel is on but the Gumbel exactness check did not run."
FRONTIERS_STATUS = globals().get("X_FRONTIERS_OK")
if XENV_SWITCH_STATE["xenv_frontiers"]:
    assert "X_FRONTIERS_OK" in globals(), "xenv_frontiers is on but Section II.5 did not run."
assert FRONTIERS_STATUS is not False, "An exact-frontier cross-check failed - see xenv/exact_frontiers.csv (column passes)."

XRES_GATE = globals().get("XRESULTS")
XRES_GATE = pd.DataFrame() if XRES_GATE is None else XRES_GATE
XIMPROPER = pd.DataFrame(columns=["family", "target", "rows", "improper"])
XDRAWS = pd.DataFrame(columns=["family", "cell", "draws", "configured_draws"])
if XENV_RUN_FAMILIES:
    assert len(XRES_GATE), f"Part II switches {XENV_RUN_FAMILIES} are on but XRESULTS is missing or empty."
    assert bool(globals().get("XENV_DONE", False)), "Part II statistics (Section II.7) did not complete (XENV_DONE)."
if len(XRES_GATE):
    _xneed = {"cell", "family", "draw", "policy", "method_family", "target", "semantics", "status", "return",
              "inadmissible_rate", "expected_violations", "prob_any_violation"}
    assert _xneed <= set(XRES_GATE.columns), f"XRESULTS lacks columns {sorted(_xneed - set(XRES_GATE.columns))}"
    _xbad = sorted(set(XRES_GATE["status"].astype(str)) - {"ok", "improper"})
    assert not _xbad, f"Part II rows with unexpected status: {_xbad}"
    _xok = XRES_GATE[XRES_GATE["status"] == "ok"]
    _xret = _xok["return"].to_numpy(dtype=float)
    assert np.isfinite(_xret).all() and (np.abs(_xret) <= 1 + 1e-9).all(), "Part II return outside [-1, 1]."
    assert _xok["inadmissible_rate"].between(-1e-12, 1 + 1e-12).all(), "Part II violation rate outside [0, 1]."
    assert (_xok["prob_any_violation"] <= _xok["expected_violations"] + 1e-9).all(), "Part II: P(any violation) > E[count]."
    _xterm = _xok[_xok["semantics"] == "terminate"]
    assert np.allclose(_xterm["prob_any_violation"], _xterm["expected_violations"], atol=1e-9), \
        "Part II: under termination the expected violation count must equal the first-violation probability."
    XIMPROPER = (XRES_GATE.assign(improper=XRES_GATE["status"].eq("improper"))
                 .groupby(["family", "target"])["improper"].agg(rows="size", improper="sum").reset_index())
    _xcfg_draws = {f: int(getattr(XCFG, f"{f}_draws")) for f in ("icu", "gumbel", "garnet")}
    XDRAWS = (XRES_GATE.groupby(["family", "cell"])["draw"].nunique().rename("draws").reset_index()
              .assign(configured_draws=lambda d: d["family"].map(_xcfg_draws)))
    print("Part II improper rows (family x target):")
    display(XIMPROPER)
XDRAWS_COMPLETE = (bool((XDRAWS["draws"] == XDRAWS["configured_draws"]).all()
                        and set(XENV_RUN_FAMILIES) <= set(XDRAWS["family"])) if XENV_RUN_FAMILIES else None)
GUMBEL_PROVENANCE_STATUS = (globals().get("GUMBEL_REPORT") or {}).get("provenance", {}).get("status")
XGATE = {"switches": XENV_SWITCH_STATE, "families_run": XENV_RUN_FAMILIES, "propositions_ok": PROPOSITIONS_STATUS,
         "frontiers_ok": FRONTIERS_STATUS, "icu_adapter_ok": ICU_ADAPTER_OK, "gumbel_exactness_ok": GUMBEL_EXACT_OK,
         "gumbel_provenance": GUMBEL_PROVENANCE_STATUS,
         "result_rows": int(len(XRES_GATE)), "improper_rows": int(XIMPROPER["improper"].sum()) if len(XIMPROPER) else 0,
         "improper_by_family_target": XIMPROPER.to_dict(orient="records"),
         "draws_per_cell": XDRAWS.to_dict(orient="records"), "draws_complete": XDRAWS_COMPLETE,
         "xenv_done": globals().get("XENV_DONE")}
print(json.dumps({k: v for k, v in XGATE.items() if k not in ("improper_by_family_target", "draws_per_cell")},
                 indent=2, default=_json_default))

# ---------- Revision gate (Section II.8; switch-aware, never fails because a switch is off) ----------
XV3_SWITCH_STATE = {k: bool(EXPERIMENTS.get(k, False)) for k in globals().get("XV3_ALL_SWITCHES", ())}
_xv3_cells_on = bool(globals().get("X_V3_CELLS")) and any(
    XV3_SWITCH_STATE.get(k, False) for k in ("xenv_v3_physicians", "xenv_v3_aggregated", "xenv_v3_garnet_random"))
_xv3_env_ok = xenv_pass_flag(globals().get("X_V3_ENV_CHECK"), "X_V3_ENV_CHECK")
XRES_V3_GATE = globals().get("XRESULTS_V3")
XRES_V3_GATE = pd.DataFrame() if XRES_V3_GATE is None else XRES_V3_GATE
if _xv3_cells_on:
    assert _xv3_env_ok is True, "II.8 environment checks did not pass (xenv/v3_environment_checks.json)."
    assert len(XRES_V3_GATE), "II.8 cell switches are on but XRESULTS_V3 is missing or empty (Section II.8f)."
if len(XRES_V3_GATE):
    _v3bad = sorted(set(XRES_V3_GATE["status"].astype(str)) - {"ok", "improper"})
    assert not _v3bad, f"II.8 rows with unexpected status: {_v3bad}"
    _v3ok = XRES_V3_GATE[XRES_V3_GATE["status"] == "ok"]
    _v3ret = _v3ok["return"].to_numpy(dtype=float)
    assert np.isfinite(_v3ret).all() and (np.abs(_v3ret) <= 1 + 1e-9).all(), "II.8 return outside [-1, 1]."
    assert _v3ok["inadmissible_rate"].between(-1e-12, 1 + 1e-12).all(), "II.8 violation rate outside [0, 1]."
    assert (_v3ok["prob_any_violation"] <= _v3ok["expected_violations"] + 1e-9).all(), "II.8: P(any violation) > E[count]."
    _v3term = _v3ok[_v3ok["semantics"] == "terminate"]
    assert np.allclose(_v3term["prob_any_violation"], _v3term["expected_violations"], atol=1e-9), \
        "II.8: under termination the expected violation count must equal the first-violation probability."
    del _v3bad, _v3ok, _v3ret, _v3term
_v3drv = globals().get("XV3_DRIVER_SUMMARY") or {}
_v3st = globals().get("XV3_STATS") or {}
_v3chk = _v3st.get("checks") or {}
_v3done = globals().get("XV3_STATS_DONE") or {}
_v3boot = globals().get("XV3_BOOT_SUMMARY") or {}
_v3harm = globals().get("XV3_HARM_CHECK") or {}
_v3boot_ran = _v3boot.get("status") == "run"
assert _v3chk.get("passes") is not False, f"II.8i theorem checks failed: {_v3chk.get('theorem_counts')}"
if _v3boot_ran:
    assert (_v3boot.get("max_width_minus_bound") or 0.0) <= 1e-9 and (_v3boot.get("max_rule_p_difference") or 0.0) <= 1e-9, \
        "II.8g bootstrap invariants (Theorem part iii on every replicate) failed - see xenv/v3_bootstrap_summary.json."
if XV3_SWITCH_STATE.get("xenv_v3_icu_harm"):
    assert _v3harm.get("passes") is True, "II.8h harm-curve identities failed (xenv/v3_icu_harm_checks.json)."
_v3pc = (_v3boot.get("point_check") or {}).get("match")
REV1_OK = (None if not _xv3_cells_on else
           bool(_xv3_env_ok and len(XRES_V3_GATE) and _v3drv.get("draws_complete") and _v3done.get("cells")
                and _v3chk.get("passes")))
REV2_OK = (None if not (XV3_SWITCH_STATE.get("xenv_v3_bootstrap") and _v3boot_ran) else
           bool(_v3boot.get("draws_complete") and _v3done.get("bootstrap") and _v3chk.get("passes")))
REV3_OK = None if not XV3_SWITCH_STATE.get("xenv_v3_icu_harm") else bool(_v3harm.get("passes") is True)
XV3GATE = {"switches": XV3_SWITCH_STATE, "environment_checks_ok": _xv3_env_ok, "result_rows": int(len(XRES_V3_GATE)),
           "code_hash": globals().get("XENV_V3_CODE_HASH"), "draws_complete": _v3drv.get("draws_complete"),
           "draws_per_cell": _v3drv.get("draws_per_cell"),
           "improper_rows": int(XRES_V3_GATE["status"].ne("ok").sum()) if len(XRES_V3_GATE) else 0,
           "stats_done": _v3done, "theorem_counts": _v3chk.get("theorem_counts"), "stats_passes": _v3chk.get("passes"),
           "bootstrap": {k: _v3boot.get(k) for k in ("status", "cells", "cells_skipped", "draws", "draws_complete",
                                                     "fallback_pairs_total", "draws_with_trap_replicate",
                                                     "max_width_minus_bound", "code_hash")},
           "bootstrap_policies_match_xresults": _v3pc,
           "icu_harm": {k: _v3harm.get(k) for k in ("status", "passes", "part1_policies", "part2_policies",
                                                    "endpoints_part1_vs_RESULTS", "endpoints_part2_vs_XRESULTS")},
           "rev_checks": {"REV-1": REV1_OK, "REV-2": REV2_OK, "REV-3": REV3_OK}}
print("Revision gate (II.8):", json.dumps({k: XV3GATE[k] for k in ("switches", "result_rows", "draws_complete",
                                                                     "stats_passes", "rev_checks")},
                                          default=_json_default))

# ---------- Second-revision gate (Section II.9): invariants, completion of enabled switches, clean-check pass ----------
XV4_SWITCH_STATE = {k: bool(EXPERIMENTS.get(k, False)) for k in globals().get("XV4_SWITCHES", ())}
# closure row -> (II.9 subsection, switch, summary global)
XV4_REV_ROWS = {"REV-4": ("II.9b", "xenv_v4_revision_robust", "XV4_ROBUST_SUMMARY"),
                "REV-5": ("II.9c", "xenv_v4_confidence", "XV4_CS_SUMMARY"),
                "REV-6": ("II.9d", "xenv_v4_bootstrap", "XV4_BOOT_SUMMARY"),
                "REV-7": ("II.9e", "xenv_v4_sensitivity", "XV4_SENS_SUMMARY"),
                "REV-8": ("II.9f", "xenv_v4_lift", "XV4_LIFT_SUMMARY"),
                "REV-9": ("II.9g", "xenv_v4_clean_check", "XV4_CLEAN")}
_v4st = globals().get("XV4_STATS") or {}
_v4sub = _v4st.get("subsections") or {}
_v4sum = {rev: globals().get(g) or {} for rev, (_, _, g) in XV4_REV_ROWS.items()}
_v4bad = {k: v.get("invariants") for k, v in _v4sub.items() if v.get("invariants_pass") is False}
assert not _v4bad, f"II.9 invariant checks failed: {_v4bad} - see xenv/v4_stats_checks.json"
assert not any((_v4st.get("checks") or {}).get("order_counts", {}).values()), \
    f"II.9h recounted orderings failed: {_v4st['checks']['order_counts']} - see xenv/v4_stats_checks.json"
if _v4sum["REV-5"].get("status") == "run":
    assert ((_v4sum["REV-5"].get("checks") or {}).get("max_lower_minus_upper") or 0.0) <= 1e-9, \
        "II.9c confidence-set bounds out of order (L > U_free) - see xenv/v4_cs_summary.json."
if _v4sum["REV-6"].get("status") == "run":
    assert (_v4sum["REV-6"].get("max_width_minus_bound") or 0.0) <= 1e-9 \
        and (_v4sum["REV-6"].get("max_rule_p_difference") or 0.0) <= 1e-9, \
        "II.9d bootstrap invariants (Theorem 1 part iii on every replicate) failed - see xenv/v4_boot_summary.json."
if _v4sum["REV-7"].get("status") == "run":
    assert (_v4sum["REV-7"].get("checks") or {}).get("passes") is not False, \
        "II.9e sensitivity-cell checks (self-test, theorem counts) failed - see xenv/v4_sens_summary.json."


def xv4_rev_flag(rev: str) -> Optional[bool]:
    """Closure flag of one II.9 subsection: None when its switch is off (II.9g also in artifact-replay mode);
    otherwise status 'run', draws complete and invariants not failed in II.9h, and for II.9g the clean-check pass
    flag. Coverage and error rates never enter."""
    sub, switch, _ = XV4_REV_ROWS[rev]
    if not XV4_SWITCH_STATE.get(switch) or (sub == "II.9g" and not ALLOW_TRAINING):
        return None
    s, info = _v4sum[rev], _v4sub.get(sub) or {}
    ok = (s.get("status") == "run" and s.get("draws_complete") is True and info.get("ran") is True
          and info.get("invariants_pass") is not False)
    return bool(ok and (sub != "II.9g" or s.get("pass") is True))


XV4_REV = {rev: xv4_rev_flag(rev) for rev in XV4_REV_ROWS}
XV4GATE = {"switches": XV4_SWITCH_STATE, "stats_status": (_v4st.get("overview") or {}).get("status"),
           "stats_passes": (_v4st.get("checks") or {}).get("passes"),
           "subsections": {sub: {k: (_v4sub.get(sub) or {}).get(k) for k in ("enabled", "status", "ran",
                                                                            "draws_complete", "invariants_pass",
                                                                            "code_hash")}
                           for sub, _, _ in XV4_REV_ROWS.values()},
           "clean_check": {k: _v4sum["REV-9"].get(k) for k in ("status", "pass", "bitwise_identical", "criteria",
                                                                "failing_draws_total")},
           "stats_code_hash": globals().get("XV4_STATS_CODE_HASH"), "rev_checks": XV4_REV}
print("Second-revision gate (II.9):", json.dumps({k: XV4GATE[k] for k in ("switches", "stats_passes", "rev_checks")},
                                                 default=_json_default))

legacy_ok = LEGACY.get("max_abs_discrepancy")
repro_ok = (REPRO_TESTS.get("critic_training", {}).get("passes_declared_tolerance") and
            REPRO_TESTS.get("diffusion_training", {}).get("passes_declared_tolerance")) if REPRO_TESTS else None


def status(flag, yes="check passed", no="check FAILED - inspect evidence", none="not run (experiment switched off)"):
    return none if flag is None else (yes if flag else no)


CLOSURE = [
    ("C01", "Mean-equivalent projection + masked BC/IQL + uniform-admissible", status(PROJECTION_IDENTITY_VERIFIED), "projection_equivalence_checks.csv; table_main_policy_comparison.csv; legacy_projection_controls_summary.csv"),
    ("C02", "Hard AM-CQL repaired designs, TD-row/loss/alpha/placement ablations, diagnostics",
     status(CRITIC_STABILITY_GATE.get("stable_on_all_seeds") if CRITIC_STABILITY_GATE else None,
            "primary critic empirically stable on all main seeds", "primary critic NOT stable on all seeds - see gate"),
     "critic_stability_gate.json; critic_stability_summary*.csv; table_constraint_placement.csv; critic_diagnostics.csv"),
    ("C03", "Seeding, RNG streams, run identity, resumable checkpoints, cache invalidation", status(repro_ok), "reproducibility_tests.json"),
    ("C04", "All tables generated here; replay; release with code and hashes",
     status(REPLAY_SUMMARY.get("cross_session_ok"),
            "cross-session replay matches the previous session",
            "cross-session differences (expected after FORCE_RETRAIN / code change) - inspect",
            none="no previous session in this ROOT: only the in-session file round-trip was checked (re-run with ALLOW_TRAINING=False for a replay)"),
     "cross_session_replay.csv; policy_reevaluation.csv; checkpoint_replay_cpu.csv; release/"),
    ("M01", "Corrected admissible support coverage", status(0.0 <= SUPPORT_AUDIT["admissible_pair_coverage"] <= 1.0), "corrected_support_audit.json"),
    ("M02", "Released vs estimated support; threshold verified", status(PROVENANCE["checks"].get("threshold_rule_count_le_threshold_removed") if isinstance(PROVENANCE["checks"].get("threshold_rule_count_le_threshold_removed"), bool) else None, none="offline: verify threshold manually"), "benchmark_construction_provenance.json; support_estimation_*.csv"),
    ("M03", "Prereg, two-sided tests, Holm over distinct hypotheses, hierarchical analysis", status("primary" in STATS, "analyses produced"), "statistics_*.csv/json; replication_planning.csv"),
    ("M04", "Selection protocol with FQE validation and locked fresh evaluation", status(bool(SELECTION) if EXPERIMENTS["selection_protocol"] else None, "protocol run"), "selection_*.csv; tuning_budget.csv"),
    ("M05", "Soft training vs extraction alignment", status(bool(len(SOFT_ANALYSIS)) if EXPERIMENTS["main_neural_critics"] else None, "analysis produced"), "soft_constraint_analysis.csv"),
    ("M06", "Direct categorical alternatives, placement ablations, cost", status(bool(len(TABLE6_PRIMARY)) if EXPERIMENTS["main_diffusion"] else None, "analysis produced"), "table6_*.csv; cost_accounting.csv"),
    ("M07", "Replication, regimes, stress tests, subgroups", status(bool(len(SUBGROUPS)), "analysis produced"), "replication_results.csv; statistics_hierarchical_replication.csv; data_regime_size_summary.csv; stress_*.csv; subgroup_diagnostics.csv"),
    ("M08", "Feature provenance/timing table and representation ablation", status(True, "table produced; treatment columns define the action and are clustering features"), "feature_provenance_timing.csv; representation_ablation.csv"),
    ("M09", "Strict evaluator, invariants, env semantics, Monte Carlo", status(bool(ENV_SEMANTICS.get("terminate_all_consistent", True)) and all(MC_CHECK["within_4_se"]) if EXPERIMENTS["monte_carlo_crosscheck"] and len(MC_ROWS) else bool(EVALUATOR_TOY_TESTS)), "numerical_audit.json; monte_carlo_crosscheck.csv"),
    ("M10", "Episode semantics + short-cutoff test", status(CUTOFF_TEST["cutoff_episodes"] > 0), "dataset_manifest_main.json; dataset_manifests_all.csv"),
    ("M11", "Manuscript rewrite and consistency pass", "manuscript-only (see Section 35)", "-"),
    ("M12", "Related work around support-constrained methods", "manuscript-only (tabular BCQ baseline added here)", "-"),
    ("m01-m03", "Citations, merged feature sections, notation/precision", "manuscript-only", "-"),
    ("m04", "Pearson vs Spearman labelled", "closed in code", "benchmark_audit.json; eda_sofa_admissibility_association.csv"),
    ("m05", "Pinned benchmark, recorded runtime, version-safe boxplot", "closed in code", "release/environment_lock.txt"),
    ("m06", "Benchmark-support terminology", "closed in code", "-"),
    ("m07", "Execution metadata and logging", "closed in code", "logs/execution_log.jsonl; logs/runtime_*.json"),
    ("m08", "Venue packaging / authorship", "manuscript-only", "-"),
    ("Legacy", "Original artifacts replayed",
     status(((legacy_ok is not None and legacy_ok <= 1e-10) if (EXPERIMENTS["legacy_artifact_replay"] and LEGACY.get("found")
             and (LEGACY_ROOT / "tables" / "exact_results_all.csv").exists()) else None),
            none="not run (switched off, original directory or its exact_results_all.csv not found)"), "legacy_replay.json"),
]

# ---------- Main-track closure rows (Part II and statistics at scale) ----------
_xfam = set(XRES_GATE["family"]) if len(XRES_GATE) else set()
_ground_truth_on = [f for f in ("gumbel", "garnet") if XENV_SWITCH_STATE.get(f"xenv_{f}")]
_gumbel_prov_bad = str(GUMBEL_PROVENANCE_STATUS or "").startswith(("MISMATCH", "MISSING"))
MT1_OK = (None if not _ground_truth_on else bool(set(_ground_truth_on) <= _xfam and ICU_ADAPTER_OK is not False
                                                  and (GUMBEL_EXACT_OK is True or "gumbel" not in _ground_truth_on)
                                                  and not _gumbel_prov_bad))
XENV_REQUIRED_METHOD_FAMILIES = {"behaviour", "value", "pessimistic", "constrained", "guided-prior", "diffusion"}
_xmf = set(XRES_GATE["method_family"].dropna()) if len(XRES_GATE) else set()
MT3_OK = None if not XENV_RUN_FAMILIES else bool(XENV_REQUIRED_METHOD_FAMILIES <= _xmf)
_fidelity_files = [XENV_TABLE_DIR / n for n in ("xenv_prereg_tests.json", "table_rank_fidelity.csv", "table_error_decomposition.csv")]
_xst = globals().get("XSTATS") or {}
_xpt = _xst.get("prereg_tests") or {}
_rank_fid = _xst.get("rank_fidelity")
MT4_OK = None if not _ground_truth_on else bool(
    all(f_.exists() for f_ in _fidelity_files)                      # written by this session's II.7 (xs_run)
    and (not XENV_SWITCH_STATE.get("xenv_gumbel") or (_xpt.get("H1") or {}).get("status") == "run")
    and (not XENV_SWITCH_STATE.get("xenv_garnet") or (_xpt.get("H3") or {}).get("status") == "run")
    and _rank_fid is not None and len(_rank_fid) > 0)
_primary_n = (STATS.get("primary") or {}).get("n_pairs")
_hier_n = int(HIERARCHICAL["n_datasets"].min()) if len(HIERARCHICAL) else None
MT5_CHECKS = {
    "paper_scale_design": bool(RUN_PROFILE["paper_scale"] and len(CFG.seeds) >= 10
                               and len(CFG.replication_dataset_seeds) >= 8 and len(CFG.replication_train_seeds) >= 2),
    "primary_pairs_ge_10": None if _primary_n is None else bool(_primary_n >= 10),
    "replication_datasets_ge_8": None if _hier_n is None else bool(_hier_n >= 8),
    "part2_draws_complete": XDRAWS_COMPLETE,
    "all_part2_families_run": {"icu", "gumbel", "garnet"} <= set(XENV_RUN_FAMILIES),
    "all_cells_present": bool(set(str(c_["cell"]) for c_ in (globals().get("X_CELLS") or []))
                              <= set(XDRAWS["cell"].astype(str))),
    "part2_design_as_preregistered": (("design_deviations" in _xpt and not _xpt["design_deviations"])
                                      if XENV_RUN_FAMILIES else None),   # a missing report does not establish it
}
MT5_OK = all(v is True for v in MT5_CHECKS.values())   # a component that did not run does not establish scale
_mt5_text = (f"{len(CFG.seeds)} main seeds (primary pairs: {_primary_n}); {len(CFG.replication_dataset_seeds)} x "
             f"{len(CFG.replication_train_seeds)} replication (estimation; datasets analysed: {_hier_n}); Part II draws "
             f"complete: {XDRAWS_COMPLETE}; all Part II families run: {MT5_CHECKS['all_part2_families_run']}; all cells "
             f"present: {MT5_CHECKS['all_cells_present']}; Part II design as preregistered: "
             f"{MT5_CHECKS['part2_design_as_preregistered']}; profile={COMPUTE_PROFILE}, mode={RUN_MODE}")
CLOSURE += [
    ("MT-1", "Multi-environment ground truth: exact Gumbel-max simulator + Garnet harm family + ICU-Sepsis adapter",
     status(MT1_OK, f"environments built, adapter and exactness checks passed; Gumbel provenance: {GUMBEL_PROVENANCE_STATUS}",
            f"missing environment, failed check or Gumbel provenance {GUMBEL_PROVENANCE_STATUS} - inspect"),
     "xenv/icu_adapter_check.json; xenv/gumbel_exactness_check.json; xenv/xenv_environment_suite.csv; data/xenv_gumbel_true_*.npz"),
    ("MT-2", "Propositions P1-P7 verified numerically (+ exact constrained frontiers)",
     status(PROPOSITIONS_STATUS if XENV_SWITCH_STATE["xenv_theory"] else None, "all proposition checks passed"),
     "xenv/proposition_checks.csv; xenv/proposition_summary.csv; xenv/exact_frontiers.csv"),
    ("MT-3", "Baseline suite: value, pessimistic (VI-LCB, MBS-QI, SPIBB), constrained (Lagrangian, CMDP-LP), guided-prior (SfBC, IDQL, FISOR-style), exact diffusion",
     status(MT3_OK, "all method families evaluated", f"method families missing: {sorted(XENV_REQUIRED_METHOD_FAMILIES - _xmf)}"),
     "xenv/xenv_method_catalog.csv; xenv/table_xenv_main.csv; xenv/xenv_h2_family.csv"),
    ("MT-4", "Benchmark fidelity H1-H3: rank fidelity, error decomposition, harm moderation",
     status(MT4_OK, "analyses produced (H1 / H3 run, rank-fidelity rows present)",
            "fidelity outputs missing or H1 / H3 not run - inspect Section II.7"),
     "xenv/table_rank_fidelity.csv; xenv/table_error_decomposition.csv; xenv/xenv_prereg_tests.json"),
    ("MT-5", "Statistics at scale: 10 main seeds, 8 x 2 replication (estimation, not testing), configured Part II draws",
     status(MT5_OK, "paper scale: " + _mt5_text, "NOT paper scale: " + _mt5_text),
     "run_profile.json; statistics_primary.json; statistics_hierarchical_replication.csv; xenv/xenv_draw_meta.csv"),
]
# ---------- Revision closure rows (Section II.8) ----------
_rev2_none = ("not run (xenv_v3_bootstrap off)" if not XV3_SWITCH_STATE.get("xenv_v3_bootstrap")
              else f"not run: {_v3boot.get('status', 'no summary')}")
CLOSURE += [
    ("REV-1", "Revision cells (II.8): confounded and antibiotic-withholding Gumbel physicians, aggregated-state "
              "benchmarks with diabetes hidden, Garnet with random unsupported rows; bracketing coverage, certified "
              "comparisons and rank fidelity on them",
     status(REV1_OK, f"cells run, draws complete, environment and theorem checks passed "
                     f"({XV3GATE['result_rows']} rows)",
            "II.8 draws incomplete, analyses missing or a check failed - inspect xenv/v3_driver_summary.json and "
            "xenv/v3_stats_checks.json",
            none="not run (II.8 cell switches off)"),
     "xenv/v3_environment_suite.csv; xenv/v3_results_all.csv.gz; xenv/v3_bracketing.csv; xenv/v3_certified.csv; "
     "xenv/v3_rank_fidelity.csv; figures/xenv/fig_v3_bracketing.*"),
    ("REV-2", "Estimation-aware intervals: bootstrap ensembles of the hospital data (B = "
              f"{XV3BOOT.replicates if 'XV3BOOT' in globals() else '?'}) on II.6 cells; coverage and certified comparisons",
     status(REV2_OK, f"bootstrap run on {len(_v3boot.get('cells') or [])} cells, draws complete, invariants hold; "
                     f"regenerated policies match XRESULTS: {_v3pc}",
            "bootstrap draws incomplete or analyses missing - inspect xenv/v3_bootstrap_summary.json",
            none=_rev2_none),
     "xenv/v3_bootstrap_results.csv.gz; xenv/v3_bootstrap_summary.json; xenv/v3_bootstrap_coverage.csv; "
     "xenv/v3_bootstrap_intervals.csv; figures/xenv/fig_v3_bootstrap.*"),
    ("REV-3", "ICU-Sepsis harm curves J(lambda) on the released rows (Part I main + reference policies, seed 0; Part II "
              "ICU draw-0 default policies): ranking switches and certified share",
     status(REV3_OK, "curves produced; endpoint, bracketing, monotonicity and certified-order identities hold",
            "a harm-curve identity failed - inspect xenv/v3_icu_harm_checks.json",
            none="not run (xenv_v3_icu_harm off)"),
     "xenv/v3_icu_harm_curves.csv; xenv/v3_icu_harm_summary.csv; xenv/v3_icu_harm_checks.json; figures/xenv/fig_v3_icu_harm.*"),
]
# ---------- Second-revision closure rows (Section II.9) ----------


def _xv4_none(rev: str) -> str:
    sub, switch, _ = XV4_REV_ROWS[rev]
    if sub == "II.9g" and XV4_SWITCH_STATE.get(switch) and not ALLOW_TRAINING:
        return "not run (artifact replay: ALLOW_TRAINING is False)"
    return f"not run ({switch} off)"


def _xv4_fail(rev: str, file: str) -> str:
    sub = XV4_REV_ROWS[rev][0]
    st = (_v4sub.get(sub) or {}).get("status", _v4sum[rev].get("status", "no summary"))
    return (f"{sub} status '{st}', draws incomplete or a check failed - inspect xenv/{file} and "
            "xenv/v4_stats_checks.json")


CLOSURE += [
    ("REV-4", "Certificates without NHB (Corollary 2), pair types, distinct policies, abstention at matched coverage, "
              "hidden-benefit sensitivity kappa and the completion-sensitive headline on every Part II cell; II.8g "
              "intervals over all default policies",
     status(XV4_REV["REV-4"], "analysis run on every stored draw; theorem and identity checks passed",
            _xv4_fail("REV-4", "v4_robust_summary.json"), none=_xv4_none("REV-4")),
     "xenv/v4_robust_summary.csv; xenv/v4_robust_kappa.csv; xenv/v4_robust_gapbins.csv; xenv/v4_v3boot_allpolicy.csv; "
     "xenv/v4_robust_summary.json; xenv/v4_overview.json"),
    ("REV-5", "Confidence-set certificates on the true state space: L1 sets for the true rows and d0 from the raw "
              "hospital counts, valid for all claims of a draw at once at scale 1",
     status(XV4_REV["REV-5"], "draws complete; inner-problem self-test and bound order passed",
            _xv4_fail("REV-5", "v4_cs_summary.json"), none=_xv4_none("REV-5")),
     "xenv/v4_cs_summary.csv; xenv/v4_cs_draws.csv; xenv/v4_cs_policies.csv.gz; xenv/v4_cs_summary.json; "
     "figures/xenv/fig_v4_confidence.*"),
    ("REV-6", "Bootstrap with B = 200 that re-estimates the whole construction in every replicate, beside the fixed "
              "mask; coverage over all default policies",
     status(XV4_REV["REV-6"], "draws complete; per-replicate invariants and unit-weight identities hold",
            _xv4_fail("REV-6", "v4_boot_summary.json"), none=_xv4_none("REV-6")),
     "xenv/v4_boot_coverage.csv; xenv/v4_boot_draws.csv; xenv/v4_boot_intervals.csv.gz; xenv/v4_boot_summary.json; "
     "figures/xenv/fig_v4_bootstrap.*"),
    ("REV-7", "Construction sensitivity: Gumbel timeout hazard, the argmax dead-state rule on the same hospital data, "
              "softer physicians (wider support) and other state abstractions, each paired with its base cell",
     status(XV4_REV["REV-7"], "cells run, draws complete; self-test, draw checks and theorem counts passed",
            _xv4_fail("REV-7", "v4_sens_summary.json"), none=_xv4_none("REV-7")),
     "xenv/v4_sens_summary.csv; xenv/v4_sens_certified_draws.csv; xenv/v4_sens_rank_fidelity.csv; "
     "xenv/v4_sens_results_all.csv.gz; xenv/v4_sens_summary.json"),
    ("REV-8", "Lift sensitivity: physician and uniform lifts beside the fallback lift, and the range of J_true over "
              "all lifts",
     status(XV4_REV["REV-8"], "draws complete; Theorem 1 and range invariants hold under every lift",
            _xv4_fail("REV-8", "v4_lift_summary.json"), none=_xv4_none("REV-8")),
     "xenv/v4_lift_summary.csv; xenv/v4_lift_results.csv.gz; xenv/v4_lift_summary.json"),
    ("REV-9", "Clean rerun of the Part II draws into an empty cache, compared row by row with the cached results",
     status(XV4_REV["REV-9"], "rerun matches the cache (max |diff| <= clean_tol, identical statuses and certified "
                              f"counts, no missing row); bitwise identical: {_v4sum['REV-9'].get('bitwise_identical')}",
            _xv4_fail("REV-9", "v4_clean_check.json") + f" (pass flag: {_v4sum['REV-9'].get('pass')})",
            none=_xv4_none("REV-9")),
     "xenv/v4_clean_check.json; xenv/v4_clean_diffs.csv"),
]
CLOSURE_DF = pd.DataFrame(CLOSURE, columns=["audit_id", "item", "status", "evidence"])
CLOSURE_DF.to_csv(TABLE_DIR / "audit_closure_checklist.csv", index=False)
display(CLOSURE_DF)

manifest = {
    "implementation": "revised-audit-complete",
    "run_mode": RUN_MODE,
    "root": str(ROOT),
    "benchmark": BENCH.identity,
    "dataset": OFFLINE_MANIFEST["file"],
    "dataset_content_sha256": OFFLINE_MANIFEST["content_sha256"],
    "policies_registered": len(REGISTRY),
    "policy_seed_pairs_evaluated": int(RESULTS[["name", "seed"]].drop_duplicates().shape[0]),
    "seeds": list(CFG.seeds),
    "compute_profile": COMPUTE_PROFILE,
    "run_profile": RUN_PROFILE,
    "compute_logged": COMPUTE_TOTALS,
    "runtime": RUNTIME_INFO,
    "experiments": EXPERIMENTS,
    "prereg_hash": obj_hash(PREREG),
    "checkpoints_written_this_session": int(CHECKPOINT_META["written_this_session"].sum()) if len(CHECKPOINT_META) else 0,
    "checkpoints_loaded_from_earlier_sessions": int(CHECKPOINT_META["loaded_from_earlier_session"].sum()) if len(CHECKPOINT_META) else 0,
    "xenv": {
        **XGATE,
        "config": asdict(XCFG) if "XCFG" in globals() else None,
        "prereg_hash": obj_hash(PREREG_PART2) if "PREREG_PART2" in globals() else None,
        "code_hash": globals().get("XENV_CODE_HASH"),
        "cells": int(XRES_GATE["cell"].nunique()) if len(XRES_GATE) else 0,
        "cell_draws_evaluated": int(XRES_GATE[["cell", "draw"]].drop_duplicates().shape[0]) if len(XRES_GATE) else 0,
        "xcache_written_this_session": len(XCACHE.written) if "XCACHE" in globals() else 0,
        "xcache_loaded_from_earlier_sessions": len(XCACHE.loaded) if "XCACHE" in globals() else 0,
        "mt_checks": {"MT-1": MT1_OK, "MT-2": PROPOSITIONS_STATUS, "MT-3": MT3_OK, "MT-4": MT4_OK, "MT-5": MT5_OK,
                      "MT-5 components": MT5_CHECKS},
        "v3": XV3GATE,
        "v4": XV4GATE,
    },
    "finished_utc": utc_now(),
}
save_json(manifest, TABLE_DIR / "fresh_reproducibility_manifest.json")
print("All structural integrity checks passed. Results root:", ROOT)

In [ ]:
# ============================================================
# 34b. Code re-export, release inventory and archive (after the gate, so the closure checklist, the manifest and
#     the executed gate cells are included)
# ============================================================
if EXPERIMENTS["release_package"]:
    # Re-export the code snapshot now that the gate and closure cells have run, and re-hash it.
    export_release_code(code_dir)
    inputs["release_code_sha256"] = {p_.name: sha256_file(p_) for p_ in code_dir.glob("*") if p_.is_file()}
    save_json(inputs, TABLE_DIR / "input_sha256.json")
    save_json(inputs, RELEASE_DIR / "input_sha256.json")
    inv_path = RELEASE_DIR / "artifact_inventory.csv"
    for old_zip in RELEASE_DIR.glob("release_*.zip"):
        old_zip.unlink()
    inventory = []
    for sub in ["tables", "figures", "logs", "policies", "release", "data", "checkpoints", "xenv_cache"]:
        if not (ROOT / sub).exists():
            continue
        for f in sorted((ROOT / sub).rglob("*")):
            if not f.is_file() or f == inv_path or f.name.endswith(".zip") or f.name.startswith("resume_") or f.name.endswith(".tmp"):
                continue
            if sub == "checkpoints" and not RELEASE_INCLUDE_WEIGHTS and f.suffix == ".pt":
                continue
            if sub == "data" and not RELEASE_INCLUDE_DATASET and f.name.startswith("offline_") and f.suffix == ".npz":
                continue
            inventory.append({"path": str(f.relative_to(ROOT)), "bytes": f.stat().st_size, "sha256": sha256_file(f)})
    INVENTORY = pd.DataFrame(inventory)
    INVENTORY.to_csv(inv_path, index=False)
    zip_path = RELEASE_DIR / f"release_{RUN_MODE}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for rec in inventory:
            zf.write(ROOT / rec["path"], arcname=rec["path"])
        zf.write(inv_path, arcname="release/artifact_inventory.csv")
    print(f"Release archive: {zip_path} ({zip_path.stat().st_size / 2 ** 20:.1f} MB, {len(inventory)} files)")

## 35. Paper hand-off

This section lists which output files of the notebook's own contribution map exist after the run. The map
has groups C1–C7 and, where this notebook registers them, the II.8 groups REV-1 to REV-3 and the II.9 groups
REV-4 to REV-9. **Nothing in this section is a result**, and the list is not the set of inputs of the paper:
it omits files the paper reads (for example `exact_results_all.csv` and `xenv/xenv_results_all.csv.gz`),
and its figures are the notebook's own. In the final run II.9d, II.9f and II.9g were switched off, so their
files are listed as missing, which is expected. The paper's tables, figures and numbers are rebuilt by
`paper_scripts/` from the final run's exports in `results/` and the printed outputs of its record in
`run_record/`; the App. B.5 counts come from `paper_scripts/verify_bracket.py`. The supplementary
`TABLE_MAP.md` maps each table and figure to its script and input files.

In [ ]:
# ============================================================
# 35. Paper hand-off: which evidence files exist
# ============================================================
EXECLOG.mark("35. Paper hand-off")
HANDOFF_EVIDENCE = {
    "C1 propositions": ["xenv/proposition_checks.csv", "xenv/proposition_summary.csv"],
    "C2 frontiers": ["xenv/exact_frontiers.csv", "figures/xenv/fig_xenv_frontiers.pdf"],
    "C3 ground-truth testbed": ["xenv/gumbel_exactness_check.json", "xenv/icu_adapter_check.json",
                                "xenv/xenv_environment_suite.csv"],
    "C4 benchmark validity": ["xenv/table_rank_fidelity.csv", "xenv/table_error_decomposition.csv",
                              "xenv/xenv_prereg_tests.json", "figures/xenv/fig_xenv_rank_fidelity.pdf",
                              "figures/xenv/fig_xenv_harm_phase.pdf", "figures/xenv/fig_xenv_error_decomposition.pdf"],
    "C5 methods study": ["xenv/table_xenv_main.csv", "xenv/xenv_h2_family.csv", "xenv/xenv_normalized_scores.csv",
                         "xenv/table_performance_profiles.csv", "xenv/table_hp_selection.csv", "xenv/xenv_method_catalog.csv",
                         "figures/xenv/fig_xenv_iqm.pdf", "figures/xenv/fig_xenv_profiles.pdf", "figures/xenv/fig_xenv_poi.pdf"],
    "C6 ICU-Sepsis case study": ["table_main_policy_comparison.csv", "statistics_primary.json",
                                 "statistics_secondary_family.csv", "statistics_hierarchical_replication.csv",
                                 "projection_equivalence_checks.csv", "critic_stability_summary.csv",
                                 "regularizer_gradient_tests.json"],
    "C7 reproducibility": ["table_to_command_map.csv", "audit_closure_checklist.csv", "fresh_reproducibility_manifest.json",
                           "input_sha256.json", "compute_summary.csv", "run_profile.json"],
    "REV-1 II.8 cells: bracketing, certified comparisons, rank fidelity": [
        "xenv/v3_environment_suite.csv", "xenv/v3_results_all.csv.gz", "xenv/v3_bracketing.csv", "xenv/v3_certified.csv",
        "xenv/v3_rank_fidelity.csv", "xenv/v3_stats_checks.json", "figures/xenv/fig_v3_bracketing.pdf"],
    "REV-2 bootstrap-ensemble intervals": ["xenv/v3_bootstrap_summary.json", "xenv/v3_bootstrap_intervals.csv",
                                           "xenv/v3_bootstrap_coverage.csv", "figures/xenv/fig_v3_bootstrap.pdf"],
    "REV-3 ICU-Sepsis harm curves": ["xenv/v3_icu_harm_curves.csv", "xenv/v3_icu_harm_summary.csv",
                                     "xenv/v3_icu_harm_checks.json", "figures/xenv/fig_v3_icu_harm.pdf"],
    "REV-4 certificates without NHB, pair types, kappa, II.8g over all policies": [
        "xenv/v4_robust_summary.csv", "xenv/v4_robust_kappa.csv", "xenv/v4_robust_gapbins.csv",
        "xenv/v4_v3boot_allpolicy.csv", "xenv/v4_robust_summary.json"],
    "REV-5 confidence-set certificates": ["xenv/v4_cs_summary.csv", "xenv/v4_cs_draws.csv", "xenv/v4_cs_summary.json",
                                          "figures/xenv/fig_v4_confidence.pdf"],
    "REV-6 bootstrap B = 200, re-estimated construction": ["xenv/v4_boot_coverage.csv", "xenv/v4_boot_draws.csv",
                                                           "xenv/v4_boot_summary.json",
                                                           "figures/xenv/fig_v4_bootstrap.pdf"],
    "REV-7 sensitivity cells": ["xenv/v4_sens_summary.csv", "xenv/v4_sens_certified_draws.csv",
                                "xenv/v4_sens_rank_fidelity.csv", "xenv/v4_sens_summary.json"],
    "REV-8 lift sensitivity": ["xenv/v4_lift_summary.csv", "xenv/v4_lift_results.csv.gz", "xenv/v4_lift_summary.json"],
    "REV-9 clean rerun": ["xenv/v4_clean_check.json", "xenv/v4_clean_diffs.csv"],
    "II.9 overview and checks": ["xenv/v4_overview.json", "xenv/v4_stats_checks.json"],
}


def handoff_path(rel: str) -> Path:
    return FIG_DIR / rel[len("figures/"):] if rel.startswith("figures/") else TABLE_DIR / rel


HANDOFF_STATUS = pd.DataFrame([{"contribution": c, "file": f, "exists": handoff_path(f).exists()}
                               for c, files in HANDOFF_EVIDENCE.items() for f in files])
display(HANDOFF_STATUS.groupby("contribution", sort=False)["exists"].agg(present="sum", expected="size").reset_index())
_handoff_missing = HANDOFF_STATUS.loc[~HANDOFF_STATUS["exists"], "file"].tolist()
print("Missing evidence files:", _handoff_missing or "none")
print(f"Paper scale (closure MT-5): {MT5_OK} | profile={COMPUTE_PROFILE}, mode={RUN_MODE} | "
      f"PREREG status: {PREREG['status'][:40]!r} | PREREG_PART2 status: {PREREG_PART2['status'][:40]!r}")
if not MT5_OK:
    print("NOTE: this run is not paper scale - do not report its numbers as the paper's results (see audit_closure_checklist.csv).")